In [106]:
from sympy import symbols, pprint, collect, factor
from sympy import *

x = []
name = "x"
for i in range(0,21):
    v = symbols(name+str(i))
    x.append(v)


In [86]:
import numpy as np
import re
from qiskit import QuantumCircuit, QuantumRegister, Aer, execute
from qiskit.opflow import PauliOp, SummedOp, I
from qiskit.quantum_info import Pauli
from qiskit.circuit import ParameterVector
from qiskit.algorithms import QAOA
from qiskit.algorithms.optimizers import COBYLA

In [17]:
from sympy import symbols, expand, preorder_traversal, Symbol, Mul, Add

def substitute_with_global_binary_symbols(expr, bit_length, base_name="b"):
    """
    Replace each variable in expr with a symbolic binary representation.
    x_i → b_{3i+1} + 2*b_{3i+2} + 4*b_{3i+3}, etc.
    """
    vars = sorted(expr.free_symbols, key=lambda x: str(x))  # consistent ordering
    subs = {}

    for idx, v in enumerate(vars):
        bin_vars = [symbols(f"{base_name}_{bit_index}") for bit_index in range(idx * bit_length + 1, (idx + 1) * bit_length + 1)]
        binary_expr = sum((2**i) * bin_vars[i] for i in range(bit_length))
        subs[v] = binary_expr

    return expand(expr.subs(subs))


def remove_variable_exponents(expr):
    """
    Replace all instances of x**n (n > 1) with x in a symbolic expression.
    """
    replacements = {}

    for subexpr in preorder_traversal(expr):
        if subexpr.is_Pow:
            base, exp = subexpr.args
            if isinstance(base, Symbol) and exp.is_Number and exp > 1:
                replacements[subexpr] = base

    return expr.xreplace(replacements)

def substitute_with_spin_variables(expr, prefix_original='b', prefix_new='z'):
    """
    Substitute each variable x_i in expr with (1 - z_i)/2, where z_i is a new symbolic variable.
    """
    subs = {}

    for var in expr.free_symbols:
        name = str(var)
        if name.startswith(prefix_original):
            index = name[len(prefix_original):]
            z = symbols(f"{prefix_new}{index}")
            subs[var] = (1 - z) / 2

    return simplify(expand(expr.subs(subs)))

In [22]:
def parse_hamiltonian_expr(expr):
    terms = sympify(expr, evaluate=False).as_ordered_terms()
    parsed = []

    for term in terms:
        coeff = term
        indices = []

        if isinstance(term, Symbol):
            coeff = 1
            indices = [int(str(term).split("_")[1]) - 1]
        else:
            factors = term.as_ordered_factors()
            coeff = 1
            indices = []
            for factor in factors:
                if factor.is_Number:
                    coeff *= float(factor)
                elif isinstance(factor, Symbol) and str(factor).startswith("z_"):
                    qubit_idx = int(str(factor).split("_")[1]) - 1
                    indices.append(qubit_idx)
                else:
                    raise ValueError(f"Unsupported factor: {factor}")

        # If no qubit indices, assign [0] instead of []
        if len(indices) == 0:
            indices = [0]

        parsed.append((int(coeff), sorted(indices)))

    return parsed

In [92]:
from sympy import IndexedBase, Symbol
import re

def convert_symbols_to_indexed(expr, prefix="z"):
    """
    Converts symbols like z_1, z_2, ... in a SymPy expression to indexed form z[1], z[2], ...

    Args:
        expr (sympy expression): The SymPy expression with symbols like z_1.
        prefix (str): The variable name prefix (e.g., 'z').

    Returns:
        sympy expression: The expression with replaced indexed variables.
    """
    indexed_array = IndexedBase(prefix)
    replacements = {}

    for sym in expr.free_symbols:
        if isinstance(sym, Symbol):
            match = re.fullmatch(rf"{prefix}_(\d+)", sym.name)
            if match:
                index = int(match.group(1))
                replacements[sym] = indexed_array[index]

    return expr.subs(replacements)



In [327]:
def bitstring_cost(bitstring, hamiltonian):
    """
    Calculate the cost (energy) of a bitstring using the Hamiltonian.
    bitstring: str, e.g. '010011001' (Qiskit bit order, rightmost is qubit 0)
    """
    z = np.array([1 if b == '0' else -1 for b in bitstring[::-1]])  # reverse to qubit order
    cost = 0
    for term in hamiltonian.oplist:
        coeff = term.coeff.real
        label = term.primitive.to_label()
        indices = [i for i, p in enumerate(label) if p == 'Z']
        product = np.prod(z[indices]) if indices else 1
        cost += coeff * product
    return cost

def find_best_bitstrings(circuit, hamiltonian, shots=2048, top_k=5):
    backend = Aer.get_backend('qasm_simulator')
    job = execute(circuit, backend=backend, shots=shots)
    counts = job.result().get_counts()

    scored_bitstrings = []
    for bitstring, count in counts.items():
        cost = bitstring_cost(bitstring, hamiltonian)
        scored_bitstrings.append((bitstring, cost, count))

    # Sort by cost (lowest first), then by count (highest first)
    scored_bitstrings.sort(key=lambda x: (x[1], -x[2]))

    print(f"Top {top_k} bitstrings:")
    for bs, cost, count in scored_bitstrings[:top_k]:
        print(f"Bitstring: {bs}, Cost: {cost:.4f}, Count: {count}")

    return scored_bitstrings[:top_k]


In [132]:
# Build Hamiltonian from your terms (extend the list as needed)
def build_cost_hamiltonian_1(num_qubits,terms):

    zero_op = PauliOp(Pauli('I' * num_qubits)) * 0
    hamiltonian = SummedOp([zero_op])

    for coeff, qubits_idx in terms:
        pauli_label = ['I'] * num_qubits
        for i in qubits_idx:
            pauli_label[i] = 'Z'
        pauli_str = ''.join(pauli_label)
        hamiltonian += PauliOp(Pauli(pauli_str)) * coeff

    return hamiltonian

# Multi-controlled RZ helper
def apply_multi_controlled_rz(qc, angle, controls, target):
    qc.h(target)
    from qiskit.circuit.library import MCXGate
    mcx = MCXGate(len(controls))
    qc.append(mcx, controls + [target])
    qc.rz(angle, target)
    qc.append(mcx, controls + [target])
    qc.h(target)

# Apply cost unitary e^{-i gamma H}
def apply_cost_unitary(qc, hamiltonian, gamma):
    for term in hamiltonian.oplist:
        coeff = term.coeff.real
        label = term.primitive.to_label()
        qubits_in_term = [i for i, p in enumerate(label) if p == 'Z']

        if len(qubits_in_term) == 0:
            continue
        if len(qubits_in_term) == 1:
            qc.rz(2 * gamma * coeff, qubits_in_term[0])
            continue

        target = qubits_in_term[-1]
        controls = qubits_in_term[:-1]
        apply_multi_controlled_rz(qc, 2 * gamma * coeff, controls, target)

# Apply mixer unitary (RX rotations)
def apply_mixer_unitary(qc, beta):
    for q in range(qc.num_qubits):
        qc.rx(2 * beta, q)

# Build full QAOA circuit with p layers
def build_qaoa_circuit(num_qubits, hamiltonian, p, gammas, betas):
    qr = QuantumRegister(num_qubits)
    qc = QuantumCircuit(qr)

    # Initialize to uniform superposition
    qc.h(range(num_qubits))

    for layer in range(p):
        apply_cost_unitary(qc, hamiltonian, gammas[layer])
        apply_mixer_unitary(qc, betas[layer])

    qc.measure_all()
    return qc

# Expectation evaluation function
def expectation_from_counts(counts, hamiltonian):
    """
    Compute expectation <H> from measurement counts.
    """
    expect = 0
    shots = sum(counts.values())
    for bitstring, count in counts.items():
        z = np.array([1 if b=='0' else -1 for b in bitstring[::-1]])  # Qiskit reverses bit order
        val = 0
        for term in hamiltonian.oplist:
            coeff = term.coeff.real
            label = term.primitive.to_label()
            indices = [i for i, p in enumerate(label) if p == 'Z']
            product = np.prod(z[indices]) if indices else 1
            val += coeff * product
        expect += val * count / shots
    return expect

# Main QAOA optimization
def qaoa(num_qubits, hamiltonian, p=1, shots=1024):

    # Parameters
    gammas = ParameterVector('gamma', p)
    betas = ParameterVector('beta', p)

    backend = Aer.get_backend('qasm_simulator')

    def objective(x):
        # Build circuit with current parameters
        qc = build_qaoa_circuit(num_qubits, hamiltonian, p, x[:p], x[p:])
        job = execute(qc, backend=backend, shots=shots)
        counts = job.result().get_counts()
        return expectation_from_counts(counts, hamiltonian)

    # Initial guess
    x0 = np.array([0.1]*(2*p))

    optimizer = COBYLA(maxiter=100)
    opt_result = optimizer.optimize(num_vars=2*p, objective_function=objective, initial_point=x0)

    print(f"Optimal parameters (gammas, betas): {opt_result[0]}")
    print(f"Minimum expectation value: {opt_result[1]}")

    # Final circuit with optimized params
    final_qc = build_qaoa_circuit(num_qubits, hamiltonian, p, opt_result[0][:p], opt_result[0][p:])
    return final_qc, opt_result

In [273]:
def bitstring_to_pm1(LIST,H):
    Z = []
    for i in range(len(LIST)):
        bitstring = LIST[i][0]
        reversed_bits = bitstring[::-1]
        z = []
        z.append(0)
        for j, bit in enumerate(reversed_bits):
            #print(bit)
            z.append(1 - 2*int(bit))
        Z.append(z)
    #print(Z)

#def Evaluations(Z,H):
    Cost = []
    for i in range(len(Z)):
        Cost.append(f"Bitstring: {LIST[0]}, Evaluated cost: {H(Z[i])}")
    return(Cost)


# Catalan's $x^a-y^b=1$ for $a, b > 1, x, y > 0$ Case $a=2, b=3$ has a solution which is $x = 3, y= 2.$

In [285]:
def D_Cat(x,y):
    return(x**2-y**3-1)

For numbers between 0 and 15, which has binary length of 4:

In [286]:
paso1_D_Cat_less_15 = substitute_with_global_binary_symbols(D_Cat(x[1],x[2])**2, 4, base_name="b")
paso2_D_Cat_less_15 = remove_variable_exponents(paso1_D_Cat_less_15)
paso3_D_Cat_less_15 = substitute_with_spin_variables(paso2_D_Cat_less_15)
#print(paso3_D_Cat_less_15)

In [287]:
def evaluate_hamiltonian_less_15(z):
    return(96*z[1]*z[2]*z[3]*z[4] - 180*z[1]*z[2]*z[3] - 360*z[1]*z[2]*z[4] + 12*z[1]*z[2]*z[5]*z[6]*z[7] + 24*z[1]*z[2]*z[5]*z[6]*z[8] - 45*z[1]*z[2]*z[5]*z[6] + 48*z[1]*z[2]*z[5]*z[7]*z[8] - 90*z[1]*z[2]*z[5]*z[7] - 180*z[1]*z[2]*z[5]*z[8] + 232*z[1]*z[2]*z[5] + 96*z[1]*z[2]*z[6]*z[7]*z[8] - 180*z[1]*z[2]*z[6]*z[7] - 360*z[1]*z[2]*z[6]*z[8] + 461*z[1]*z[2]*z[6] - 720*z[1]*z[2]*z[7]*z[8] + 898*z[1]*z[2]*z[7] + 1604*z[1]*z[2]*z[8] - 1342*z[1]*z[2] - 720*z[1]*z[3]*z[4] + 24*z[1]*z[3]*z[5]*z[6]*z[7] + 48*z[1]*z[3]*z[5]*z[6]*z[8] - 90*z[1]*z[3]*z[5]*z[6] + 96*z[1]*z[3]*z[5]*z[7]*z[8] - 180*z[1]*z[3]*z[5]*z[7] - 360*z[1]*z[3]*z[5]*z[8] + 464*z[1]*z[3]*z[5] + 192*z[1]*z[3]*z[6]*z[7]*z[8] - 360*z[1]*z[3]*z[6]*z[7] - 720*z[1]*z[3]*z[6]*z[8] + 922*z[1]*z[3]*z[6] - 1440*z[1]*z[3]*z[7]*z[8] + 1796*z[1]*z[3]*z[7] + 3208*z[1]*z[3]*z[8] - 2708*z[1]*z[3] + 48*z[1]*z[4]*z[5]*z[6]*z[7] + 96*z[1]*z[4]*z[5]*z[6]*z[8] - 180*z[1]*z[4]*z[5]*z[6] + 192*z[1]*z[4]*z[5]*z[7]*z[8] - 360*z[1]*z[4]*z[5]*z[7] - 720*z[1]*z[4]*z[5]*z[8] + 928*z[1]*z[4]*z[5] + 384*z[1]*z[4]*z[6]*z[7]*z[8] - 720*z[1]*z[4]*z[6]*z[7] - 1440*z[1]*z[4]*z[6]*z[8] + 1844*z[1]*z[4]*z[6] - 2880*z[1]*z[4]*z[7]*z[8] + 3592*z[1]*z[4]*z[7] + 6416*z[1]*z[4]*z[8] - 5608*z[1]*z[4] - 90*z[1]*z[5]*z[6]*z[7] - 180*z[1]*z[5]*z[6]*z[8] + 675*z[1]*z[5]*z[6]/2 - 360*z[1]*z[5]*z[7]*z[8] + 675*z[1]*z[5]*z[7] + 1350*z[1]*z[5]*z[8] - 1740*z[1]*z[5] - 720*z[1]*z[6]*z[7]*z[8] + 1350*z[1]*z[6]*z[7] + 2700*z[1]*z[6]*z[8] - 6915*z[1]*z[6]/2 + 5400*z[1]*z[7]*z[8] - 6735*z[1]*z[7] - 12030*z[1]*z[8] + 23445*z[1]/2 - 1440*z[2]*z[3]*z[4] + 48*z[2]*z[3]*z[5]*z[6]*z[7] + 96*z[2]*z[3]*z[5]*z[6]*z[8] - 180*z[2]*z[3]*z[5]*z[6] + 192*z[2]*z[3]*z[5]*z[7]*z[8] - 360*z[2]*z[3]*z[5]*z[7] - 720*z[2]*z[3]*z[5]*z[8] + 928*z[2]*z[3]*z[5] + 384*z[2]*z[3]*z[6]*z[7]*z[8] - 720*z[2]*z[3]*z[6]*z[7] - 1440*z[2]*z[3]*z[6]*z[8] + 1844*z[2]*z[3]*z[6] - 2880*z[2]*z[3]*z[7]*z[8] + 3592*z[2]*z[3]*z[7] + 6416*z[2]*z[3]*z[8] - 5428*z[2]*z[3] + 96*z[2]*z[4]*z[5]*z[6]*z[7] + 192*z[2]*z[4]*z[5]*z[6]*z[8] - 360*z[2]*z[4]*z[5]*z[6] + 384*z[2]*z[4]*z[5]*z[7]*z[8] - 720*z[2]*z[4]*z[5]*z[7] - 1440*z[2]*z[4]*z[5]*z[8] + 1856*z[2]*z[4]*z[5] + 768*z[2]*z[4]*z[6]*z[7]*z[8] - 1440*z[2]*z[4]*z[6]*z[7] - 2880*z[2]*z[4]*z[6]*z[8] + 3688*z[2]*z[4]*z[6] - 5760*z[2]*z[4]*z[7]*z[8] + 7184*z[2]*z[4]*z[7] + 12832*z[2]*z[4]*z[8] - 11240*z[2]*z[4] - 180*z[2]*z[5]*z[6]*z[7] - 360*z[2]*z[5]*z[6]*z[8] + 675*z[2]*z[5]*z[6] - 720*z[2]*z[5]*z[7]*z[8] + 1350*z[2]*z[5]*z[7] + 2700*z[2]*z[5]*z[8] - 3480*z[2]*z[5] - 1440*z[2]*z[6]*z[7]*z[8] + 2700*z[2]*z[6]*z[7] + 5400*z[2]*z[6]*z[8] - 6915*z[2]*z[6] + 10800*z[2]*z[7]*z[8] - 13470*z[2]*z[7] - 24060*z[2]*z[8] + 23490*z[2] + 192*z[3]*z[4]*z[5]*z[6]*z[7] + 384*z[3]*z[4]*z[5]*z[6]*z[8] - 720*z[3]*z[4]*z[5]*z[6] + 768*z[3]*z[4]*z[5]*z[7]*z[8] - 1440*z[3]*z[4]*z[5]*z[7] - 2880*z[3]*z[4]*z[5]*z[8] + 3712*z[3]*z[4]*z[5] + 1536*z[3]*z[4]*z[6]*z[7]*z[8] - 2880*z[3]*z[4]*z[6]*z[7] - 5760*z[3]*z[4]*z[6]*z[8] + 7376*z[3]*z[4]*z[6] - 11520*z[3]*z[4]*z[7]*z[8] + 14368*z[3]*z[4]*z[7] + 25664*z[3]*z[4]*z[8] - 22672*z[3]*z[4] - 360*z[3]*z[5]*z[6]*z[7] - 720*z[3]*z[5]*z[6]*z[8] + 1350*z[3]*z[5]*z[6] - 1440*z[3]*z[5]*z[7]*z[8] + 2700*z[3]*z[5]*z[7] + 5400*z[3]*z[5]*z[8] - 6960*z[3]*z[5] - 2880*z[3]*z[6]*z[7]*z[8] + 5400*z[3]*z[6]*z[7] + 10800*z[3]*z[6]*z[8] - 13830*z[3]*z[6] + 21600*z[3]*z[7]*z[8] - 26940*z[3]*z[7] - 48120*z[3]*z[8] + 47340*z[3] - 720*z[4]*z[5]*z[6]*z[7] - 1440*z[4]*z[5]*z[6]*z[8] + 2700*z[4]*z[5]*z[6] - 2880*z[4]*z[5]*z[7]*z[8] + 5400*z[4]*z[5]*z[7] + 10800*z[4]*z[5]*z[8] - 13920*z[4]*z[5] - 5760*z[4]*z[6]*z[7]*z[8] + 10800*z[4]*z[6]*z[7] + 21600*z[4]*z[6]*z[8] - 27660*z[4]*z[6] + 43200*z[4]*z[7]*z[8] - 53880*z[4]*z[7] - 96240*z[4]*z[8] + 97560*z[4] + 91200*z[5]*z[6]*z[7]*z[8] - 97632*z[5]*z[6]*z[7] - 152064*z[5]*z[6]*z[8] + 315947*z[5]*z[6]/2 - 282528*z[5]*z[7]*z[8] + 289547*z[5]*z[7] + 402454*z[5]*z[8] - 817749*z[5]/2 - 554256*z[6]*z[7]*z[8] + 565804*z[6]*z[7] + 781208*z[6]*z[8] - 1583307*z[6]/2 + 1381456*z[7]*z[8] - 1390743*z[7] - 1759374*z[8] + 1778473)

In [288]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 8  # 
p = 1  # QAOA depth

hamiltonian_less_15 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_Cat_less_15))
final_circuit_less_15, result_less_15 = qaoa(num_qubits, hamiltonian_less_15, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [1.10006567 1.10007542]
Minimum expectation value: -158535.775390625


In [289]:
top_solutions_less_15 = find_best_bitstrings(final_circuit_less_15, hamiltonian_less_15)

Top 5 bitstrings:
Bitstring: 00010001, Cost: -3556945.0000, Count: 21
Bitstring: 00100011, Cost: -3556945.0000, Count: 2
Bitstring: 00000001, Cost: -3556945.0000, Count: 1
Bitstring: 00110101, Cost: -3556937.0000, Count: 2
Bitstring: 01011011, Cost: -3556921.0000, Count: 62


In [290]:
bitstring_to_pm1(top_solutions_less_15,evaluate_hamiltonian_less_15)

["Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 1.0",
 "Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 0.0",
 "Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 0.0",
 "Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 9.0",
 "Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 25.0"]

# Diophantine equation $x^2+y^2=3$ which has no integer solutions 

In [233]:
def D_equation(x,y):
    return(x**2+y**2-3)

For numbers between 0 and 15, which has binary length of 4:

In [300]:
paso1_D_equation_less_15_2 = substitute_with_global_binary_symbols(D_equation(x[1],x[2])**2, 4, base_name="b")
paso2_D_equation_less_15_2 = remove_variable_exponents(paso1_D_equation_less_15_2)
paso3_D_equation_less_15_2 = substitute_with_spin_variables(paso2_D_equation_less_15_2)

In [298]:
def evaluate_hamiltonian_less_15_2(z):
    return(96*z[1]*z[2]*z[3]*z[4] - 180*z[1]*z[2]*z[3] - 360*z[1]*z[2]*z[4] + 2*z[1]*z[2]*z[5]*z[6] + 4*z[1]*z[2]*z[5]*z[7] + 8*z[1]*z[2]*z[5]*z[8] - 15*z[1]*z[2]*z[5] + 8*z[1]*z[2]*z[6]*z[7] + 16*z[1]*z[2]*z[6]*z[8] - 30*z[1]*z[2]*z[6] + 32*z[1]*z[2]*z[7]*z[8] - 60*z[1]*z[2]*z[7] - 120*z[1]*z[2]*z[8] + 609*z[1]*z[2] - 720*z[1]*z[3]*z[4] + 4*z[1]*z[3]*z[5]*z[6] + 8*z[1]*z[3]*z[5]*z[7] + 16*z[1]*z[3]*z[5]*z[8] - 30*z[1]*z[3]*z[5] + 16*z[1]*z[3]*z[6]*z[7] + 32*z[1]*z[3]*z[6]*z[8] - 60*z[1]*z[3]*z[6] + 64*z[1]*z[3]*z[7]*z[8] - 120*z[1]*z[3]*z[7] - 240*z[1]*z[3]*z[8] + 1194*z[1]*z[3] + 8*z[1]*z[4]*z[5]*z[6] + 16*z[1]*z[4]*z[5]*z[7] + 32*z[1]*z[4]*z[5]*z[8] - 60*z[1]*z[4]*z[5] + 32*z[1]*z[4]*z[6]*z[7] + 64*z[1]*z[4]*z[6]*z[8] - 120*z[1]*z[4]*z[6] + 128*z[1]*z[4]*z[7]*z[8] - 240*z[1]*z[4]*z[7] - 480*z[1]*z[4]*z[8] + 2196*z[1]*z[4] - 15*z[1]*z[5]*z[6] - 30*z[1]*z[5]*z[7] - 60*z[1]*z[5]*z[8] + 225*z[1]*z[5]/2 - 60*z[1]*z[6]*z[7] - 120*z[1]*z[6]*z[8] + 225*z[1]*z[6] - 240*z[1]*z[7]*z[8] + 450*z[1]*z[7] + 900*z[1]*z[8] - 2910*z[1] - 1440*z[2]*z[3]*z[4] + 8*z[2]*z[3]*z[5]*z[6] + 16*z[2]*z[3]*z[5]*z[7] + 32*z[2]*z[3]*z[5]*z[8] - 60*z[2]*z[3]*z[5] + 32*z[2]*z[3]*z[6]*z[7] + 64*z[2]*z[3]*z[6]*z[8] - 120*z[2]*z[3]*z[6] + 128*z[2]*z[3]*z[7]*z[8] - 240*z[2]*z[3]*z[7] - 480*z[2]*z[3]*z[8] + 2376*z[2]*z[3] + 16*z[2]*z[4]*z[5]*z[6] + 32*z[2]*z[4]*z[5]*z[7] + 64*z[2]*z[4]*z[5]*z[8] - 120*z[2]*z[4]*z[5] + 64*z[2]*z[4]*z[6]*z[7] + 128*z[2]*z[4]*z[6]*z[8] - 240*z[2]*z[4]*z[6] + 256*z[2]*z[4]*z[7]*z[8] - 480*z[2]*z[4]*z[7] - 960*z[2]*z[4]*z[8] + 4368*z[2]*z[4] - 30*z[2]*z[5]*z[6] - 60*z[2]*z[5]*z[7] - 120*z[2]*z[5]*z[8] + 225*z[2]*z[5] - 120*z[2]*z[6]*z[7] - 240*z[2]*z[6]*z[8] + 450*z[2]*z[6] - 480*z[2]*z[7]*z[8] + 900*z[2]*z[7] + 1800*z[2]*z[8] - 5775*z[2] + 32*z[3]*z[4]*z[5]*z[6] + 64*z[3]*z[4]*z[5]*z[7] + 128*z[3]*z[4]*z[5]*z[8] - 240*z[3]*z[4]*z[5] + 128*z[3]*z[4]*z[6]*z[7] + 256*z[3]*z[4]*z[6]*z[8] - 480*z[3]*z[4]*z[6] + 512*z[3]*z[4]*z[7]*z[8] - 960*z[3]*z[4]*z[7] - 1920*z[3]*z[4]*z[8] + 8544*z[3]*z[4] - 60*z[3]*z[5]*z[6] - 120*z[3]*z[5]*z[7] - 240*z[3]*z[5]*z[8] + 450*z[3]*z[5] - 240*z[3]*z[6]*z[7] - 480*z[3]*z[6]*z[8] + 900*z[3]*z[6] - 960*z[3]*z[7]*z[8] + 1800*z[3]*z[7] + 3600*z[3]*z[8] - 11190*z[3] - 120*z[4]*z[5]*z[6] - 240*z[4]*z[5]*z[7] - 480*z[4]*z[5]*z[8] + 900*z[4]*z[5] - 480*z[4]*z[6]*z[7] - 960*z[4]*z[6]*z[8] + 1800*z[4]*z[6] - 1920*z[4]*z[7]*z[8] + 3600*z[4]*z[7] + 7200*z[4]*z[8] - 19500*z[4] + 96*z[5]*z[6]*z[7]*z[8] - 180*z[5]*z[6]*z[7] - 360*z[5]*z[6]*z[8] + 609*z[5]*z[6] - 720*z[5]*z[7]*z[8] + 1194*z[5]*z[7] + 2196*z[5]*z[8] - 2910*z[5] - 1440*z[6]*z[7]*z[8] + 2376*z[6]*z[7] + 4368*z[6]*z[8] - 5775*z[6] + 8544*z[7]*z[8] - 11190*z[7] - 19500*z[8] + 66761/2)

In [301]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 8  # 
p = 1  # QAOA depth

hamiltonian_less_15_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_equation_less_15_2))
final_circuit_less_15_2, result_less_15_2 = qaoa(num_qubits, hamiltonian_less_15_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.10009991 1.09999578]
Minimum expectation value: 14175.4453125


In [302]:
top_solutions_less_15_2 = find_best_bitstrings(final_circuit_less_15_2, hamiltonian_less_15_2)

Top 5 bitstrings:
Bitstring: 00010001, Cost: -66760.0000, Count: 1
Bitstring: 00000001, Cost: -66756.0000, Count: 8
Bitstring: 00100001, Cost: -66756.0000, Count: 1
Bitstring: 00000011, Cost: -66724.0000, Count: 2
Bitstring: 00110001, Cost: -66712.0000, Count: 2


In [ ]:
bitstring_to_pm1(top_solutions_less_15_2, evaluate_hamiltonian_less_15_2)

For numbers between 0 and 127, which has binary length of 7

In [303]:
paso1_D_equation_less_127_2 = substitute_with_global_binary_symbols(D_equation(x[1],x[2])**2, 7, base_name="b")
paso2_D_equation_less_127_2 = remove_variable_exponents(paso1_D_equation_less_127_2)
paso3_D_equation_less_127_2 = substitute_with_spin_variables(paso2_D_equation_less_127_2)

In [304]:
def evaluate_hamiltonian_less_127_2(z):
    return(32*z[1]*z[10]*z[11]*z[2] + 64*z[1]*z[10]*z[11]*z[3] + 128*z[1]*z[10]*z[11]*z[4] + 256*z[1]*z[10]*z[11]*z[5] + 512*z[1]*z[10]*z[11]*z[6] + 1024*z[1]*z[10]*z[11]*z[7] - 2032*z[1]*z[10]*z[11] + 64*z[1]*z[10]*z[12]*z[2] + 128*z[1]*z[10]*z[12]*z[3] + 256*z[1]*z[10]*z[12]*z[4] + 512*z[1]*z[10]*z[12]*z[5] + 1024*z[1]*z[10]*z[12]*z[6] + 2048*z[1]*z[10]*z[12]*z[7] - 4064*z[1]*z[10]*z[12] + 128*z[1]*z[10]*z[13]*z[2] + 256*z[1]*z[10]*z[13]*z[3] + 512*z[1]*z[10]*z[13]*z[4] + 1024*z[1]*z[10]*z[13]*z[5] + 2048*z[1]*z[10]*z[13]*z[6] + 4096*z[1]*z[10]*z[13]*z[7] - 8128*z[1]*z[10]*z[13] + 256*z[1]*z[10]*z[14]*z[2] + 512*z[1]*z[10]*z[14]*z[3] + 1024*z[1]*z[10]*z[14]*z[4] + 2048*z[1]*z[10]*z[14]*z[5] + 4096*z[1]*z[10]*z[14]*z[6] + 8192*z[1]*z[10]*z[14]*z[7] - 16256*z[1]*z[10]*z[14] + 4*z[1]*z[10]*z[2]*z[8] + 8*z[1]*z[10]*z[2]*z[9] - 508*z[1]*z[10]*z[2] + 8*z[1]*z[10]*z[3]*z[8] + 16*z[1]*z[10]*z[3]*z[9] - 1016*z[1]*z[10]*z[3] + 16*z[1]*z[10]*z[4]*z[8] + 32*z[1]*z[10]*z[4]*z[9] - 2032*z[1]*z[10]*z[4] + 32*z[1]*z[10]*z[5]*z[8] + 64*z[1]*z[10]*z[5]*z[9] - 4064*z[1]*z[10]*z[5] + 64*z[1]*z[10]*z[6]*z[8] + 128*z[1]*z[10]*z[6]*z[9] - 8128*z[1]*z[10]*z[6] + 128*z[1]*z[10]*z[7]*z[8] + 256*z[1]*z[10]*z[7]*z[9] - 16256*z[1]*z[10]*z[7] - 254*z[1]*z[10]*z[8] - 508*z[1]*z[10]*z[9] + 32258*z[1]*z[10] + 128*z[1]*z[11]*z[12]*z[2] + 256*z[1]*z[11]*z[12]*z[3] + 512*z[1]*z[11]*z[12]*z[4] + 1024*z[1]*z[11]*z[12]*z[5] + 2048*z[1]*z[11]*z[12]*z[6] + 4096*z[1]*z[11]*z[12]*z[7] - 8128*z[1]*z[11]*z[12] + 256*z[1]*z[11]*z[13]*z[2] + 512*z[1]*z[11]*z[13]*z[3] + 1024*z[1]*z[11]*z[13]*z[4] + 2048*z[1]*z[11]*z[13]*z[5] + 4096*z[1]*z[11]*z[13]*z[6] + 8192*z[1]*z[11]*z[13]*z[7] - 16256*z[1]*z[11]*z[13] + 512*z[1]*z[11]*z[14]*z[2] + 1024*z[1]*z[11]*z[14]*z[3] + 2048*z[1]*z[11]*z[14]*z[4] + 4096*z[1]*z[11]*z[14]*z[5] + 8192*z[1]*z[11]*z[14]*z[6] + 16384*z[1]*z[11]*z[14]*z[7] - 32512*z[1]*z[11]*z[14] + 8*z[1]*z[11]*z[2]*z[8] + 16*z[1]*z[11]*z[2]*z[9] - 1016*z[1]*z[11]*z[2] + 16*z[1]*z[11]*z[3]*z[8] + 32*z[1]*z[11]*z[3]*z[9] - 2032*z[1]*z[11]*z[3] + 32*z[1]*z[11]*z[4]*z[8] + 64*z[1]*z[11]*z[4]*z[9] - 4064*z[1]*z[11]*z[4] + 64*z[1]*z[11]*z[5]*z[8] + 128*z[1]*z[11]*z[5]*z[9] - 8128*z[1]*z[11]*z[5] + 128*z[1]*z[11]*z[6]*z[8] + 256*z[1]*z[11]*z[6]*z[9] - 16256*z[1]*z[11]*z[6] + 256*z[1]*z[11]*z[7]*z[8] + 512*z[1]*z[11]*z[7]*z[9] - 32512*z[1]*z[11]*z[7] - 508*z[1]*z[11]*z[8] - 1016*z[1]*z[11]*z[9] + 64516*z[1]*z[11] + 512*z[1]*z[12]*z[13]*z[2] + 1024*z[1]*z[12]*z[13]*z[3] + 2048*z[1]*z[12]*z[13]*z[4] + 4096*z[1]*z[12]*z[13]*z[5] + 8192*z[1]*z[12]*z[13]*z[6] + 16384*z[1]*z[12]*z[13]*z[7] - 32512*z[1]*z[12]*z[13] + 1024*z[1]*z[12]*z[14]*z[2] + 2048*z[1]*z[12]*z[14]*z[3] + 4096*z[1]*z[12]*z[14]*z[4] + 8192*z[1]*z[12]*z[14]*z[5] + 16384*z[1]*z[12]*z[14]*z[6] + 32768*z[1]*z[12]*z[14]*z[7] - 65024*z[1]*z[12]*z[14] + 16*z[1]*z[12]*z[2]*z[8] + 32*z[1]*z[12]*z[2]*z[9] - 2032*z[1]*z[12]*z[2] + 32*z[1]*z[12]*z[3]*z[8] + 64*z[1]*z[12]*z[3]*z[9] - 4064*z[1]*z[12]*z[3] + 64*z[1]*z[12]*z[4]*z[8] + 128*z[1]*z[12]*z[4]*z[9] - 8128*z[1]*z[12]*z[4] + 128*z[1]*z[12]*z[5]*z[8] + 256*z[1]*z[12]*z[5]*z[9] - 16256*z[1]*z[12]*z[5] + 256*z[1]*z[12]*z[6]*z[8] + 512*z[1]*z[12]*z[6]*z[9] - 32512*z[1]*z[12]*z[6] + 512*z[1]*z[12]*z[7]*z[8] + 1024*z[1]*z[12]*z[7]*z[9] - 65024*z[1]*z[12]*z[7] - 1016*z[1]*z[12]*z[8] - 2032*z[1]*z[12]*z[9] + 129032*z[1]*z[12] + 2048*z[1]*z[13]*z[14]*z[2] + 4096*z[1]*z[13]*z[14]*z[3] + 8192*z[1]*z[13]*z[14]*z[4] + 16384*z[1]*z[13]*z[14]*z[5] + 32768*z[1]*z[13]*z[14]*z[6] + 65536*z[1]*z[13]*z[14]*z[7] - 130048*z[1]*z[13]*z[14] + 32*z[1]*z[13]*z[2]*z[8] + 64*z[1]*z[13]*z[2]*z[9] - 4064*z[1]*z[13]*z[2] + 64*z[1]*z[13]*z[3]*z[8] + 128*z[1]*z[13]*z[3]*z[9] - 8128*z[1]*z[13]*z[3] + 128*z[1]*z[13]*z[4]*z[8] + 256*z[1]*z[13]*z[4]*z[9] - 16256*z[1]*z[13]*z[4] + 256*z[1]*z[13]*z[5]*z[8] + 512*z[1]*z[13]*z[5]*z[9] - 32512*z[1]*z[13]*z[5] + 512*z[1]*z[13]*z[6]*z[8] + 1024*z[1]*z[13]*z[6]*z[9] - 65024*z[1]*z[13]*z[6] + 1024*z[1]*z[13]*z[7]*z[8] + 2048*z[1]*z[13]*z[7]*z[9] - 130048*z[1]*z[13]*z[7] - 2032*z[1]*z[13]*z[8] - 4064*z[1]*z[13]*z[9] + 258064*z[1]*z[13] + 64*z[1]*z[14]*z[2]*z[8] + 128*z[1]*z[14]*z[2]*z[9] - 8128*z[1]*z[14]*z[2] + 128*z[1]*z[14]*z[3]*z[8] + 256*z[1]*z[14]*z[3]*z[9] - 16256*z[1]*z[14]*z[3] + 256*z[1]*z[14]*z[4]*z[8] + 512*z[1]*z[14]*z[4]*z[9] - 32512*z[1]*z[14]*z[4] + 512*z[1]*z[14]*z[5]*z[8] + 1024*z[1]*z[14]*z[5]*z[9] - 65024*z[1]*z[14]*z[5] + 1024*z[1]*z[14]*z[6]*z[8] + 2048*z[1]*z[14]*z[6]*z[9] - 130048*z[1]*z[14]*z[6] + 2048*z[1]*z[14]*z[7]*z[8] + 4096*z[1]*z[14]*z[7]*z[9] - 260096*z[1]*z[14]*z[7] - 4064*z[1]*z[14]*z[8] - 8128*z[1]*z[14]*z[9] + 516128*z[1]*z[14] + 96*z[1]*z[2]*z[3]*z[4] + 192*z[1]*z[2]*z[3]*z[5] + 384*z[1]*z[2]*z[3]*z[6] + 768*z[1]*z[2]*z[3]*z[7] - 1524*z[1]*z[2]*z[3] + 384*z[1]*z[2]*z[4]*z[5] + 768*z[1]*z[2]*z[4]*z[6] + 1536*z[1]*z[2]*z[4]*z[7] - 3048*z[1]*z[2]*z[4] + 1536*z[1]*z[2]*z[5]*z[6] + 3072*z[1]*z[2]*z[5]*z[7] - 6096*z[1]*z[2]*z[5] + 6144*z[1]*z[2]*z[6]*z[7] - 12192*z[1]*z[2]*z[6] - 24384*z[1]*z[2]*z[7] + 2*z[1]*z[2]*z[8]*z[9] - 127*z[1]*z[2]*z[8] - 254*z[1]*z[2]*z[9] + 43169*z[1]*z[2] + 768*z[1]*z[3]*z[4]*z[5] + 1536*z[1]*z[3]*z[4]*z[6] + 3072*z[1]*z[3]*z[4]*z[7] - 6096*z[1]*z[3]*z[4] + 3072*z[1]*z[3]*z[5]*z[6] + 6144*z[1]*z[3]*z[5]*z[7] - 12192*z[1]*z[3]*z[5] + 12288*z[1]*z[3]*z[6]*z[7] - 24384*z[1]*z[3]*z[6] - 48768*z[1]*z[3]*z[7] + 4*z[1]*z[3]*z[8]*z[9] - 254*z[1]*z[3]*z[8] - 508*z[1]*z[3]*z[9] + 86314*z[1]*z[3] + 6144*z[1]*z[4]*z[5]*z[6] + 12288*z[1]*z[4]*z[5]*z[7] - 24384*z[1]*z[4]*z[5] + 24576*z[1]*z[4]*z[6]*z[7] - 48768*z[1]*z[4]*z[6] - 97536*z[1]*z[4]*z[7] + 8*z[1]*z[4]*z[8]*z[9] - 508*z[1]*z[4]*z[8] - 1016*z[1]*z[4]*z[9] + 172436*z[1]*z[4] + 49152*z[1]*z[5]*z[6]*z[7] - 97536*z[1]*z[5]*z[6] - 195072*z[1]*z[5]*z[7] + 16*z[1]*z[5]*z[8]*z[9] - 1016*z[1]*z[5]*z[8] - 2032*z[1]*z[5]*z[9] + 343336*z[1]*z[5] - 390144*z[1]*z[6]*z[7] + 32*z[1]*z[6]*z[8]*z[9] - 2032*z[1]*z[6]*z[8] - 4064*z[1]*z[6]*z[9] + 674384*z[1]*z[6] + 64*z[1]*z[7]*z[8]*z[9] - 4064*z[1]*z[7]*z[8] - 8128*z[1]*z[7]*z[9] + 1250464*z[1]*z[7] - 127*z[1]*z[8]*z[9] + 16129*z[1]*z[8]/2 + 16129*z[1]*z[9] - 1717294*z[1] + 24576*z[10]*z[11]*z[12]*z[13] + 49152*z[10]*z[11]*z[12]*z[14] + 768*z[10]*z[11]*z[12]*z[8] + 1536*z[10]*z[11]*z[12]*z[9] - 97536*z[10]*z[11]*z[12] + 98304*z[10]*z[11]*z[13]*z[14] + 1536*z[10]*z[11]*z[13]*z[8] + 3072*z[10]*z[11]*z[13]*z[9] - 195072*z[10]*z[11]*z[13] + 3072*z[10]*z[11]*z[14]*z[8] + 6144*z[10]*z[11]*z[14]*z[9] - 390144*z[10]*z[11]*z[14] + 128*z[10]*z[11]*z[2]*z[3] + 256*z[10]*z[11]*z[2]*z[4] + 512*z[10]*z[11]*z[2]*z[5] + 1024*z[10]*z[11]*z[2]*z[6] + 2048*z[10]*z[11]*z[2]*z[7] - 4064*z[10]*z[11]*z[2] + 512*z[10]*z[11]*z[3]*z[4] + 1024*z[10]*z[11]*z[3]*z[5] + 2048*z[10]*z[11]*z[3]*z[6] + 4096*z[10]*z[11]*z[3]*z[7] - 8128*z[10]*z[11]*z[3] + 2048*z[10]*z[11]*z[4]*z[5] + 4096*z[10]*z[11]*z[4]*z[6] + 8192*z[10]*z[11]*z[4]*z[7] - 16256*z[10]*z[11]*z[4] + 8192*z[10]*z[11]*z[5]*z[6] + 16384*z[10]*z[11]*z[5]*z[7] - 32512*z[10]*z[11]*z[5] + 32768*z[10]*z[11]*z[6]*z[7] - 65024*z[10]*z[11]*z[6] - 130048*z[10]*z[11]*z[7] + 96*z[10]*z[11]*z[8]*z[9] - 6096*z[10]*z[11]*z[8] - 12192*z[10]*z[11]*z[9] + 689504*z[10]*z[11] + 196608*z[10]*z[12]*z[13]*z[14] + 3072*z[10]*z[12]*z[13]*z[8] + 6144*z[10]*z[12]*z[13]*z[9] - 390144*z[10]*z[12]*z[13] + 6144*z[10]*z[12]*z[14]*z[8] + 12288*z[10]*z[12]*z[14]*z[9] - 780288*z[10]*z[12]*z[14] + 256*z[10]*z[12]*z[2]*z[3] + 512*z[10]*z[12]*z[2]*z[4] + 1024*z[10]*z[12]*z[2]*z[5] + 2048*z[10]*z[12]*z[2]*z[6] + 4096*z[10]*z[12]*z[2]*z[7] - 8128*z[10]*z[12]*z[2] + 1024*z[10]*z[12]*z[3]*z[4] + 2048*z[10]*z[12]*z[3]*z[5] + 4096*z[10]*z[12]*z[3]*z[6] + 8192*z[10]*z[12]*z[3]*z[7] - 16256*z[10]*z[12]*z[3] + 4096*z[10]*z[12]*z[4]*z[5] + 8192*z[10]*z[12]*z[4]*z[6] + 16384*z[10]*z[12]*z[4]*z[7] - 32512*z[10]*z[12]*z[4] + 16384*z[10]*z[12]*z[5]*z[6] + 32768*z[10]*z[12]*z[5]*z[7] - 65024*z[10]*z[12]*z[5] + 65536*z[10]*z[12]*z[6]*z[7] - 130048*z[10]*z[12]*z[6] - 260096*z[10]*z[12]*z[7] + 192*z[10]*z[12]*z[8]*z[9] - 12192*z[10]*z[12]*z[8] - 24384*z[10]*z[12]*z[9] + 1372864*z[10]*z[12] + 12288*z[10]*z[13]*z[14]*z[8] + 24576*z[10]*z[13]*z[14]*z[9] - 1560576*z[10]*z[13]*z[14] + 512*z[10]*z[13]*z[2]*z[3] + 1024*z[10]*z[13]*z[2]*z[4] + 2048*z[10]*z[13]*z[2]*z[5] + 4096*z[10]*z[13]*z[2]*z[6] + 8192*z[10]*z[13]*z[2]*z[7] - 16256*z[10]*z[13]*z[2] + 2048*z[10]*z[13]*z[3]*z[4] + 4096*z[10]*z[13]*z[3]*z[5] + 8192*z[10]*z[13]*z[3]*z[6] + 16384*z[10]*z[13]*z[3]*z[7] - 32512*z[10]*z[13]*z[3] + 8192*z[10]*z[13]*z[4]*z[5] + 16384*z[10]*z[13]*z[4]*z[6] + 32768*z[10]*z[13]*z[4]*z[7] - 65024*z[10]*z[13]*z[4] + 32768*z[10]*z[13]*z[5]*z[6] + 65536*z[10]*z[13]*z[5]*z[7] - 130048*z[10]*z[13]*z[5] + 131072*z[10]*z[13]*z[6]*z[7] - 260096*z[10]*z[13]*z[6] - 520192*z[10]*z[13]*z[7] + 384*z[10]*z[13]*z[8]*z[9] - 24384*z[10]*z[13]*z[8] - 48768*z[10]*z[13]*z[9] + 2696576*z[10]*z[13] + 1024*z[10]*z[14]*z[2]*z[3] + 2048*z[10]*z[14]*z[2]*z[4] + 4096*z[10]*z[14]*z[2]*z[5] + 8192*z[10]*z[14]*z[2]*z[6] + 16384*z[10]*z[14]*z[2]*z[7] - 32512*z[10]*z[14]*z[2] + 4096*z[10]*z[14]*z[3]*z[4] + 8192*z[10]*z[14]*z[3]*z[5] + 16384*z[10]*z[14]*z[3]*z[6] + 32768*z[10]*z[14]*z[3]*z[7] - 65024*z[10]*z[14]*z[3] + 16384*z[10]*z[14]*z[4]*z[5] + 32768*z[10]*z[14]*z[4]*z[6] + 65536*z[10]*z[14]*z[4]*z[7] - 130048*z[10]*z[14]*z[4] + 65536*z[10]*z[14]*z[5]*z[6] + 131072*z[10]*z[14]*z[5]*z[7] - 260096*z[10]*z[14]*z[5] + 262144*z[10]*z[14]*z[6]*z[7] - 520192*z[10]*z[14]*z[6] - 1040384*z[10]*z[14]*z[7] + 768*z[10]*z[14]*z[8]*z[9] - 48768*z[10]*z[14]*z[8] - 97536*z[10]*z[14]*z[9] + 4999936*z[10]*z[14] + 16*z[10]*z[2]*z[3]*z[8] + 32*z[10]*z[2]*z[3]*z[9] - 2032*z[10]*z[2]*z[3] + 32*z[10]*z[2]*z[4]*z[8] + 64*z[10]*z[2]*z[4]*z[9] - 4064*z[10]*z[2]*z[4] + 64*z[10]*z[2]*z[5]*z[8] + 128*z[10]*z[2]*z[5]*z[9] - 8128*z[10]*z[2]*z[5] + 128*z[10]*z[2]*z[6]*z[8] + 256*z[10]*z[2]*z[6]*z[9] - 16256*z[10]*z[2]*z[6] + 256*z[10]*z[2]*z[7]*z[8] + 512*z[10]*z[2]*z[7]*z[9] - 32512*z[10]*z[2]*z[7] - 508*z[10]*z[2]*z[8] - 1016*z[10]*z[2]*z[9] + 64516*z[10]*z[2] + 64*z[10]*z[3]*z[4]*z[8] + 128*z[10]*z[3]*z[4]*z[9] - 8128*z[10]*z[3]*z[4] + 128*z[10]*z[3]*z[5]*z[8] + 256*z[10]*z[3]*z[5]*z[9] - 16256*z[10]*z[3]*z[5] + 256*z[10]*z[3]*z[6]*z[8] + 512*z[10]*z[3]*z[6]*z[9] - 32512*z[10]*z[3]*z[6] + 512*z[10]*z[3]*z[7]*z[8] + 1024*z[10]*z[3]*z[7]*z[9] - 65024*z[10]*z[3]*z[7] - 1016*z[10]*z[3]*z[8] - 2032*z[10]*z[3]*z[9] + 129032*z[10]*z[3] + 256*z[10]*z[4]*z[5]*z[8] + 512*z[10]*z[4]*z[5]*z[9] - 32512*z[10]*z[4]*z[5] + 512*z[10]*z[4]*z[6]*z[8] + 1024*z[10]*z[4]*z[6]*z[9] - 65024*z[10]*z[4]*z[6] + 1024*z[10]*z[4]*z[7]*z[8] + 2048*z[10]*z[4]*z[7]*z[9] - 130048*z[10]*z[4]*z[7] - 2032*z[10]*z[4]*z[8] - 4064*z[10]*z[4]*z[9] + 258064*z[10]*z[4] + 1024*z[10]*z[5]*z[6]*z[8] + 2048*z[10]*z[5]*z[6]*z[9] - 130048*z[10]*z[5]*z[6] + 2048*z[10]*z[5]*z[7]*z[8] + 4096*z[10]*z[5]*z[7]*z[9] - 260096*z[10]*z[5]*z[7] - 4064*z[10]*z[5]*z[8] - 8128*z[10]*z[5]*z[9] + 516128*z[10]*z[5] + 4096*z[10]*z[6]*z[7]*z[8] + 8192*z[10]*z[6]*z[7]*z[9] - 520192*z[10]*z[6]*z[7] - 8128*z[10]*z[6]*z[8] - 16256*z[10]*z[6]*z[9] + 1032256*z[10]*z[6] - 16256*z[10]*z[7]*z[8] - 32512*z[10]*z[7]*z[9] + 2064512*z[10]*z[7] - 1524*z[10]*z[8]*z[9] + 86314*z[10]*z[8] + 172616*z[10]*z[9] - 6865366*z[10] + 393216*z[11]*z[12]*z[13]*z[14] + 6144*z[11]*z[12]*z[13]*z[8] + 12288*z[11]*z[12]*z[13]*z[9] - 780288*z[11]*z[12]*z[13] + 12288*z[11]*z[12]*z[14]*z[8] + 24576*z[11]*z[12]*z[14]*z[9] - 1560576*z[11]*z[12]*z[14] + 512*z[11]*z[12]*z[2]*z[3] + 1024*z[11]*z[12]*z[2]*z[4] + 2048*z[11]*z[12]*z[2]*z[5] + 4096*z[11]*z[12]*z[2]*z[6] + 8192*z[11]*z[12]*z[2]*z[7] - 16256*z[11]*z[12]*z[2] + 2048*z[11]*z[12]*z[3]*z[4] + 4096*z[11]*z[12]*z[3]*z[5] + 8192*z[11]*z[12]*z[3]*z[6] + 16384*z[11]*z[12]*z[3]*z[7] - 32512*z[11]*z[12]*z[3] + 8192*z[11]*z[12]*z[4]*z[5] + 16384*z[11]*z[12]*z[4]*z[6] + 32768*z[11]*z[12]*z[4]*z[7] - 65024*z[11]*z[12]*z[4] + 32768*z[11]*z[12]*z[5]*z[6] + 65536*z[11]*z[12]*z[5]*z[7] - 130048*z[11]*z[12]*z[5] + 131072*z[11]*z[12]*z[6]*z[7] - 260096*z[11]*z[12]*z[6] - 520192*z[11]*z[12]*z[7] + 384*z[11]*z[12]*z[8]*z[9] - 24384*z[11]*z[12]*z[8] - 48768*z[11]*z[12]*z[9] + 2742656*z[11]*z[12] + 24576*z[11]*z[13]*z[14]*z[8] + 49152*z[11]*z[13]*z[14]*z[9] - 3121152*z[11]*z[13]*z[14] + 1024*z[11]*z[13]*z[2]*z[3] + 2048*z[11]*z[13]*z[2]*z[4] + 4096*z[11]*z[13]*z[2]*z[5] + 8192*z[11]*z[13]*z[2]*z[6] + 16384*z[11]*z[13]*z[2]*z[7] - 32512*z[11]*z[13]*z[2] + 4096*z[11]*z[13]*z[3]*z[4] + 8192*z[11]*z[13]*z[3]*z[5] + 16384*z[11]*z[13]*z[3]*z[6] + 32768*z[11]*z[13]*z[3]*z[7] - 65024*z[11]*z[13]*z[3] + 16384*z[11]*z[13]*z[4]*z[5] + 32768*z[11]*z[13]*z[4]*z[6] + 65536*z[11]*z[13]*z[4]*z[7] - 130048*z[11]*z[13]*z[4] + 65536*z[11]*z[13]*z[5]*z[6] + 131072*z[11]*z[13]*z[5]*z[7] - 260096*z[11]*z[13]*z[5] + 262144*z[11]*z[13]*z[6]*z[7] - 520192*z[11]*z[13]*z[6] - 1040384*z[11]*z[13]*z[7] + 768*z[11]*z[13]*z[8]*z[9] - 48768*z[11]*z[13]*z[8] - 97536*z[11]*z[13]*z[9] + 5387008*z[11]*z[13] + 2048*z[11]*z[14]*z[2]*z[3] + 4096*z[11]*z[14]*z[2]*z[4] + 8192*z[11]*z[14]*z[2]*z[5] + 16384*z[11]*z[14]*z[2]*z[6] + 32768*z[11]*z[14]*z[2]*z[7] - 65024*z[11]*z[14]*z[2] + 8192*z[11]*z[14]*z[3]*z[4] + 16384*z[11]*z[14]*z[3]*z[5] + 32768*z[11]*z[14]*z[3]*z[6] + 65536*z[11]*z[14]*z[3]*z[7] - 130048*z[11]*z[14]*z[3] + 32768*z[11]*z[14]*z[4]*z[5] + 65536*z[11]*z[14]*z[4]*z[6] + 131072*z[11]*z[14]*z[4]*z[7] - 260096*z[11]*z[14]*z[4] + 131072*z[11]*z[14]*z[5]*z[6] + 262144*z[11]*z[14]*z[5]*z[7] - 520192*z[11]*z[14]*z[5] + 524288*z[11]*z[14]*z[6]*z[7] - 1040384*z[11]*z[14]*z[6] - 2080768*z[11]*z[14]*z[7] + 1536*z[11]*z[14]*z[8]*z[9] - 97536*z[11]*z[14]*z[8] - 195072*z[11]*z[14]*z[9] + 9987584*z[11]*z[14] + 32*z[11]*z[2]*z[3]*z[8] + 64*z[11]*z[2]*z[3]*z[9] - 4064*z[11]*z[2]*z[3] + 64*z[11]*z[2]*z[4]*z[8] + 128*z[11]*z[2]*z[4]*z[9] - 8128*z[11]*z[2]*z[4] + 128*z[11]*z[2]*z[5]*z[8] + 256*z[11]*z[2]*z[5]*z[9] - 16256*z[11]*z[2]*z[5] + 256*z[11]*z[2]*z[6]*z[8] + 512*z[11]*z[2]*z[6]*z[9] - 32512*z[11]*z[2]*z[6] + 512*z[11]*z[2]*z[7]*z[8] + 1024*z[11]*z[2]*z[7]*z[9] - 65024*z[11]*z[2]*z[7] - 1016*z[11]*z[2]*z[8] - 2032*z[11]*z[2]*z[9] + 129032*z[11]*z[2] + 128*z[11]*z[3]*z[4]*z[8] + 256*z[11]*z[3]*z[4]*z[9] - 16256*z[11]*z[3]*z[4] + 256*z[11]*z[3]*z[5]*z[8] + 512*z[11]*z[3]*z[5]*z[9] - 32512*z[11]*z[3]*z[5] + 512*z[11]*z[3]*z[6]*z[8] + 1024*z[11]*z[3]*z[6]*z[9] - 65024*z[11]*z[3]*z[6] + 1024*z[11]*z[3]*z[7]*z[8] + 2048*z[11]*z[3]*z[7]*z[9] - 130048*z[11]*z[3]*z[7] - 2032*z[11]*z[3]*z[8] - 4064*z[11]*z[3]*z[9] + 258064*z[11]*z[3] + 512*z[11]*z[4]*z[5]*z[8] + 1024*z[11]*z[4]*z[5]*z[9] - 65024*z[11]*z[4]*z[5] + 1024*z[11]*z[4]*z[6]*z[8] + 2048*z[11]*z[4]*z[6]*z[9] - 130048*z[11]*z[4]*z[6] + 2048*z[11]*z[4]*z[7]*z[8] + 4096*z[11]*z[4]*z[7]*z[9] - 260096*z[11]*z[4]*z[7] - 4064*z[11]*z[4]*z[8] - 8128*z[11]*z[4]*z[9] + 516128*z[11]*z[4] + 2048*z[11]*z[5]*z[6]*z[8] + 4096*z[11]*z[5]*z[6]*z[9] - 260096*z[11]*z[5]*z[6] + 4096*z[11]*z[5]*z[7]*z[8] + 8192*z[11]*z[5]*z[7]*z[9] - 520192*z[11]*z[5]*z[7] - 8128*z[11]*z[5]*z[8] - 16256*z[11]*z[5]*z[9] + 1032256*z[11]*z[5] + 8192*z[11]*z[6]*z[7]*z[8] + 16384*z[11]*z[6]*z[7]*z[9] - 1040384*z[11]*z[6]*z[7] - 16256*z[11]*z[6]*z[8] - 32512*z[11]*z[6]*z[9] + 2064512*z[11]*z[6] - 32512*z[11]*z[7]*z[8] - 65024*z[11]*z[7]*z[9] + 4129024*z[11]*z[7] - 3048*z[11]*z[8]*z[9] + 172436*z[11]*z[8] + 344848*z[11]*z[9] - 13706348*z[11] + 49152*z[12]*z[13]*z[14]*z[8] + 98304*z[12]*z[13]*z[14]*z[9] - 6242304*z[12]*z[13]*z[14] + 2048*z[12]*z[13]*z[2]*z[3] + 4096*z[12]*z[13]*z[2]*z[4] + 8192*z[12]*z[13]*z[2]*z[5] + 16384*z[12]*z[13]*z[2]*z[6] + 32768*z[12]*z[13]*z[2]*z[7] - 65024*z[12]*z[13]*z[2] + 8192*z[12]*z[13]*z[3]*z[4] + 16384*z[12]*z[13]*z[3]*z[5] + 32768*z[12]*z[13]*z[3]*z[6] + 65536*z[12]*z[13]*z[3]*z[7] - 130048*z[12]*z[13]*z[3] + 32768*z[12]*z[13]*z[4]*z[5] + 65536*z[12]*z[13]*z[4]*z[6] + 131072*z[12]*z[13]*z[4]*z[7] - 260096*z[12]*z[13]*z[4] + 131072*z[12]*z[13]*z[5]*z[6] + 262144*z[12]*z[13]*z[5]*z[7] - 520192*z[12]*z[13]*z[5] + 524288*z[12]*z[13]*z[6]*z[7] - 1040384*z[12]*z[13]*z[6] - 2080768*z[12]*z[13]*z[7] + 1536*z[12]*z[13]*z[8]*z[9] - 97536*z[12]*z[13]*z[8] - 195072*z[12]*z[13]*z[9] + 10724864*z[12]*z[13] + 4096*z[12]*z[14]*z[2]*z[3] + 8192*z[12]*z[14]*z[2]*z[4] + 16384*z[12]*z[14]*z[2]*z[5] + 32768*z[12]*z[14]*z[2]*z[6] + 65536*z[12]*z[14]*z[2]*z[7] - 130048*z[12]*z[14]*z[2] + 16384*z[12]*z[14]*z[3]*z[4] + 32768*z[12]*z[14]*z[3]*z[5] + 65536*z[12]*z[14]*z[3]*z[6] + 131072*z[12]*z[14]*z[3]*z[7] - 260096*z[12]*z[14]*z[3] + 65536*z[12]*z[14]*z[4]*z[5] + 131072*z[12]*z[14]*z[4]*z[6] + 262144*z[12]*z[14]*z[4]*z[7] - 520192*z[12]*z[14]*z[4] + 262144*z[12]*z[14]*z[5]*z[6] + 524288*z[12]*z[14]*z[5]*z[7] - 1040384*z[12]*z[14]*z[5] + 1048576*z[12]*z[14]*z[6]*z[7] - 2080768*z[12]*z[14]*z[6] - 4161536*z[12]*z[14]*z[7] + 3072*z[12]*z[14]*z[8]*z[9] - 195072*z[12]*z[14]*z[8] - 390144*z[12]*z[14]*z[9] + 19876864*z[12]*z[14] + 64*z[12]*z[2]*z[3]*z[8] + 128*z[12]*z[2]*z[3]*z[9] - 8128*z[12]*z[2]*z[3] + 128*z[12]*z[2]*z[4]*z[8] + 256*z[12]*z[2]*z[4]*z[9] - 16256*z[12]*z[2]*z[4] + 256*z[12]*z[2]*z[5]*z[8] + 512*z[12]*z[2]*z[5]*z[9] - 32512*z[12]*z[2]*z[5] + 512*z[12]*z[2]*z[6]*z[8] + 1024*z[12]*z[2]*z[6]*z[9] - 65024*z[12]*z[2]*z[6] + 1024*z[12]*z[2]*z[7]*z[8] + 2048*z[12]*z[2]*z[7]*z[9] - 130048*z[12]*z[2]*z[7] - 2032*z[12]*z[2]*z[8] - 4064*z[12]*z[2]*z[9] + 258064*z[12]*z[2] + 256*z[12]*z[3]*z[4]*z[8] + 512*z[12]*z[3]*z[4]*z[9] - 32512*z[12]*z[3]*z[4] + 512*z[12]*z[3]*z[5]*z[8] + 1024*z[12]*z[3]*z[5]*z[9] - 65024*z[12]*z[3]*z[5] + 1024*z[12]*z[3]*z[6]*z[8] + 2048*z[12]*z[3]*z[6]*z[9] - 130048*z[12]*z[3]*z[6] + 2048*z[12]*z[3]*z[7]*z[8] + 4096*z[12]*z[3]*z[7]*z[9] - 260096*z[12]*z[3]*z[7] - 4064*z[12]*z[3]*z[8] - 8128*z[12]*z[3]*z[9] + 516128*z[12]*z[3] + 1024*z[12]*z[4]*z[5]*z[8] + 2048*z[12]*z[4]*z[5]*z[9] - 130048*z[12]*z[4]*z[5] + 2048*z[12]*z[4]*z[6]*z[8] + 4096*z[12]*z[4]*z[6]*z[9] - 260096*z[12]*z[4]*z[6] + 4096*z[12]*z[4]*z[7]*z[8] + 8192*z[12]*z[4]*z[7]*z[9] - 520192*z[12]*z[4]*z[7] - 8128*z[12]*z[4]*z[8] - 16256*z[12]*z[4]*z[9] + 1032256*z[12]*z[4] + 4096*z[12]*z[5]*z[6]*z[8] + 8192*z[12]*z[5]*z[6]*z[9] - 520192*z[12]*z[5]*z[6] + 8192*z[12]*z[5]*z[7]*z[8] + 16384*z[12]*z[5]*z[7]*z[9] - 1040384*z[12]*z[5]*z[7] - 16256*z[12]*z[5]*z[8] - 32512*z[12]*z[5]*z[9] + 2064512*z[12]*z[5] + 16384*z[12]*z[6]*z[7]*z[8] + 32768*z[12]*z[6]*z[7]*z[9] - 2080768*z[12]*z[6]*z[7] - 32512*z[12]*z[6]*z[8] - 65024*z[12]*z[6]*z[9] + 4129024*z[12]*z[6] - 65024*z[12]*z[7]*z[8] - 130048*z[12]*z[7]*z[9] + 8258048*z[12]*z[7] - 6096*z[12]*z[8]*z[9] + 343336*z[12]*z[8] + 686624*z[12]*z[9] - 27217624*z[12] + 8192*z[13]*z[14]*z[2]*z[3] + 16384*z[13]*z[14]*z[2]*z[4] + 32768*z[13]*z[14]*z[2]*z[5] + 65536*z[13]*z[14]*z[2]*z[6] + 131072*z[13]*z[14]*z[2]*z[7] - 260096*z[13]*z[14]*z[2] + 32768*z[13]*z[14]*z[3]*z[4] + 65536*z[13]*z[14]*z[3]*z[5] + 131072*z[13]*z[14]*z[3]*z[6] + 262144*z[13]*z[14]*z[3]*z[7] - 520192*z[13]*z[14]*z[3] + 131072*z[13]*z[14]*z[4]*z[5] + 262144*z[13]*z[14]*z[4]*z[6] + 524288*z[13]*z[14]*z[4]*z[7] - 1040384*z[13]*z[14]*z[4] + 524288*z[13]*z[14]*z[5]*z[6] + 1048576*z[13]*z[14]*z[5]*z[7] - 2080768*z[13]*z[14]*z[5] + 2097152*z[13]*z[14]*z[6]*z[7] - 4161536*z[13]*z[14]*z[6] - 8323072*z[13]*z[14]*z[7] + 6144*z[13]*z[14]*z[8]*z[9] - 390144*z[13]*z[14]*z[8] - 780288*z[13]*z[14]*z[9] + 38967296*z[13]*z[14] + 128*z[13]*z[2]*z[3]*z[8] + 256*z[13]*z[2]*z[3]*z[9] - 16256*z[13]*z[2]*z[3] + 256*z[13]*z[2]*z[4]*z[8] + 512*z[13]*z[2]*z[4]*z[9] - 32512*z[13]*z[2]*z[4] + 512*z[13]*z[2]*z[5]*z[8] + 1024*z[13]*z[2]*z[5]*z[9] - 65024*z[13]*z[2]*z[5] + 1024*z[13]*z[2]*z[6]*z[8] + 2048*z[13]*z[2]*z[6]*z[9] - 130048*z[13]*z[2]*z[6] + 2048*z[13]*z[2]*z[7]*z[8] + 4096*z[13]*z[2]*z[7]*z[9] - 260096*z[13]*z[2]*z[7] - 4064*z[13]*z[2]*z[8] - 8128*z[13]*z[2]*z[9] + 516128*z[13]*z[2] + 512*z[13]*z[3]*z[4]*z[8] + 1024*z[13]*z[3]*z[4]*z[9] - 65024*z[13]*z[3]*z[4] + 1024*z[13]*z[3]*z[5]*z[8] + 2048*z[13]*z[3]*z[5]*z[9] - 130048*z[13]*z[3]*z[5] + 2048*z[13]*z[3]*z[6]*z[8] + 4096*z[13]*z[3]*z[6]*z[9] - 260096*z[13]*z[3]*z[6] + 4096*z[13]*z[3]*z[7]*z[8] + 8192*z[13]*z[3]*z[7]*z[9] - 520192*z[13]*z[3]*z[7] - 8128*z[13]*z[3]*z[8] - 16256*z[13]*z[3]*z[9] + 1032256*z[13]*z[3] + 2048*z[13]*z[4]*z[5]*z[8] + 4096*z[13]*z[4]*z[5]*z[9] - 260096*z[13]*z[4]*z[5] + 4096*z[13]*z[4]*z[6]*z[8] + 8192*z[13]*z[4]*z[6]*z[9] - 520192*z[13]*z[4]*z[6] + 8192*z[13]*z[4]*z[7]*z[8] + 16384*z[13]*z[4]*z[7]*z[9] - 1040384*z[13]*z[4]*z[7] - 16256*z[13]*z[4]*z[8] - 32512*z[13]*z[4]*z[9] + 2064512*z[13]*z[4] + 8192*z[13]*z[5]*z[6]*z[8] + 16384*z[13]*z[5]*z[6]*z[9] - 1040384*z[13]*z[5]*z[6] + 16384*z[13]*z[5]*z[7]*z[8] + 32768*z[13]*z[5]*z[7]*z[9] - 2080768*z[13]*z[5]*z[7] - 32512*z[13]*z[5]*z[8] - 65024*z[13]*z[5]*z[9] + 4129024*z[13]*z[5] + 32768*z[13]*z[6]*z[7]*z[8] + 65536*z[13]*z[6]*z[7]*z[9] - 4161536*z[13]*z[6]*z[7] - 65024*z[13]*z[6]*z[8] - 130048*z[13]*z[6]*z[9] + 8258048*z[13]*z[6] - 130048*z[13]*z[7]*z[8] - 260096*z[13]*z[7]*z[9] + 16516096*z[13]*z[7] - 12192*z[13]*z[8]*z[9] + 674384*z[13]*z[8] + 1348672*z[13]*z[9] - 52874672*z[13] + 256*z[14]*z[2]*z[3]*z[8] + 512*z[14]*z[2]*z[3]*z[9] - 32512*z[14]*z[2]*z[3] + 512*z[14]*z[2]*z[4]*z[8] + 1024*z[14]*z[2]*z[4]*z[9] - 65024*z[14]*z[2]*z[4] + 1024*z[14]*z[2]*z[5]*z[8] + 2048*z[14]*z[2]*z[5]*z[9] - 130048*z[14]*z[2]*z[5] + 2048*z[14]*z[2]*z[6]*z[8] + 4096*z[14]*z[2]*z[6]*z[9] - 260096*z[14]*z[2]*z[6] + 4096*z[14]*z[2]*z[7]*z[8] + 8192*z[14]*z[2]*z[7]*z[9] - 520192*z[14]*z[2]*z[7] - 8128*z[14]*z[2]*z[8] - 16256*z[14]*z[2]*z[9] + 1032256*z[14]*z[2] + 1024*z[14]*z[3]*z[4]*z[8] + 2048*z[14]*z[3]*z[4]*z[9] - 130048*z[14]*z[3]*z[4] + 2048*z[14]*z[3]*z[5]*z[8] + 4096*z[14]*z[3]*z[5]*z[9] - 260096*z[14]*z[3]*z[5] + 4096*z[14]*z[3]*z[6]*z[8] + 8192*z[14]*z[3]*z[6]*z[9] - 520192*z[14]*z[3]*z[6] + 8192*z[14]*z[3]*z[7]*z[8] + 16384*z[14]*z[3]*z[7]*z[9] - 1040384*z[14]*z[3]*z[7] - 16256*z[14]*z[3]*z[8] - 32512*z[14]*z[3]*z[9] + 2064512*z[14]*z[3] + 4096*z[14]*z[4]*z[5]*z[8] + 8192*z[14]*z[4]*z[5]*z[9] - 520192*z[14]*z[4]*z[5] + 8192*z[14]*z[4]*z[6]*z[8] + 16384*z[14]*z[4]*z[6]*z[9] - 1040384*z[14]*z[4]*z[6] + 16384*z[14]*z[4]*z[7]*z[8] + 32768*z[14]*z[4]*z[7]*z[9] - 2080768*z[14]*z[4]*z[7] - 32512*z[14]*z[4]*z[8] - 65024*z[14]*z[4]*z[9] + 4129024*z[14]*z[4] + 16384*z[14]*z[5]*z[6]*z[8] + 32768*z[14]*z[5]*z[6]*z[9] - 2080768*z[14]*z[5]*z[6] + 32768*z[14]*z[5]*z[7]*z[8] + 65536*z[14]*z[5]*z[7]*z[9] - 4161536*z[14]*z[5]*z[7] - 65024*z[14]*z[5]*z[8] - 130048*z[14]*z[5]*z[9] + 8258048*z[14]*z[5] + 65536*z[14]*z[6]*z[7]*z[8] + 131072*z[14]*z[6]*z[7]*z[9] - 8323072*z[14]*z[6]*z[7] - 130048*z[14]*z[6]*z[8] - 260096*z[14]*z[6]*z[9] + 16516096*z[14]*z[6] - 260096*z[14]*z[7]*z[8] - 520192*z[14]*z[7]*z[9] + 33032192*z[14]*z[7] - 24384*z[14]*z[8]*z[9] + 1250464*z[14]*z[8] + 2500736*z[14]*z[9] - 93264736*z[14] + 1536*z[2]*z[3]*z[4]*z[5] + 3072*z[2]*z[3]*z[4]*z[6] + 6144*z[2]*z[3]*z[4]*z[7] - 12192*z[2]*z[3]*z[4] + 6144*z[2]*z[3]*z[5]*z[6] + 12288*z[2]*z[3]*z[5]*z[7] - 24384*z[2]*z[3]*z[5] + 24576*z[2]*z[3]*z[6]*z[7] - 48768*z[2]*z[3]*z[6] - 97536*z[2]*z[3]*z[7] + 8*z[2]*z[3]*z[8]*z[9] - 508*z[2]*z[3]*z[8] - 1016*z[2]*z[3]*z[9] + 172616*z[2]*z[3] + 12288*z[2]*z[4]*z[5]*z[6] + 24576*z[2]*z[4]*z[5]*z[7] - 48768*z[2]*z[4]*z[5] + 49152*z[2]*z[4]*z[6]*z[7] - 97536*z[2]*z[4]*z[6] - 195072*z[2]*z[4]*z[7] + 16*z[2]*z[4]*z[8]*z[9] - 1016*z[2]*z[4]*z[8] - 2032*z[2]*z[4]*z[9] + 344848*z[2]*z[4] + 98304*z[2]*z[5]*z[6]*z[7] - 195072*z[2]*z[5]*z[6] - 390144*z[2]*z[5]*z[7] + 32*z[2]*z[5]*z[8]*z[9] - 2032*z[2]*z[5]*z[8] - 4064*z[2]*z[5]*z[9] + 686624*z[2]*z[5] - 780288*z[2]*z[6]*z[7] + 64*z[2]*z[6]*z[8]*z[9] - 4064*z[2]*z[6]*z[8] - 8128*z[2]*z[6]*z[9] + 1348672*z[2]*z[6] + 128*z[2]*z[7]*z[8]*z[9] - 8128*z[2]*z[7]*z[8] - 16256*z[2]*z[7]*z[9] + 2500736*z[2]*z[7] - 254*z[2]*z[8]*z[9] + 16129*z[2]*z[8] + 32258*z[2]*z[9] - 3434207*z[2] + 24576*z[3]*z[4]*z[5]*z[6] + 49152*z[3]*z[4]*z[5]*z[7] - 97536*z[3]*z[4]*z[5] + 98304*z[3]*z[4]*z[6]*z[7] - 195072*z[3]*z[4]*z[6] - 390144*z[3]*z[4]*z[7] + 32*z[3]*z[4]*z[8]*z[9] - 2032*z[3]*z[4]*z[8] - 4064*z[3]*z[4]*z[9] + 689504*z[3]*z[4] + 196608*z[3]*z[5]*z[6]*z[7] - 390144*z[3]*z[5]*z[6] - 780288*z[3]*z[5]*z[7] + 64*z[3]*z[5]*z[8]*z[9] - 4064*z[3]*z[5]*z[8] - 8128*z[3]*z[5]*z[9] + 1372864*z[3]*z[5] - 1560576*z[3]*z[6]*z[7] + 128*z[3]*z[6]*z[8]*z[9] - 8128*z[3]*z[6]*z[8] - 16256*z[3]*z[6]*z[9] + 2696576*z[3]*z[6] + 256*z[3]*z[7]*z[8]*z[9] - 16256*z[3]*z[7]*z[8] - 32512*z[3]*z[7]*z[9] + 4999936*z[3]*z[7] - 508*z[3]*z[8]*z[9] + 32258*z[3]*z[8] + 64516*z[3]*z[9] - 6865366*z[3] + 393216*z[4]*z[5]*z[6]*z[7] - 780288*z[4]*z[5]*z[6] - 1560576*z[4]*z[5]*z[7] + 128*z[4]*z[5]*z[8]*z[9] - 8128*z[4]*z[5]*z[8] - 16256*z[4]*z[5]*z[9] + 2742656*z[4]*z[5] - 3121152*z[4]*z[6]*z[7] + 256*z[4]*z[6]*z[8]*z[9] - 16256*z[4]*z[6]*z[8] - 32512*z[4]*z[6]*z[9] + 5387008*z[4]*z[6] + 512*z[4]*z[7]*z[8]*z[9] - 32512*z[4]*z[7]*z[8] - 65024*z[4]*z[7]*z[9] + 9987584*z[4]*z[7] - 1016*z[4]*z[8]*z[9] + 64516*z[4]*z[8] + 129032*z[4]*z[9] - 13706348*z[4] - 6242304*z[5]*z[6]*z[7] + 512*z[5]*z[6]*z[8]*z[9] - 32512*z[5]*z[6]*z[8] - 65024*z[5]*z[6]*z[9] + 10724864*z[5]*z[6] + 1024*z[5]*z[7]*z[8]*z[9] - 65024*z[5]*z[7]*z[8] - 130048*z[5]*z[7]*z[9] + 19876864*z[5]*z[7] - 2032*z[5]*z[8]*z[9] + 129032*z[5]*z[8] + 258064*z[5]*z[9] - 27217624*z[5] + 2048*z[6]*z[7]*z[8]*z[9] - 130048*z[6]*z[7]*z[8] - 260096*z[6]*z[7]*z[9] + 38967296*z[6]*z[7] - 4064*z[6]*z[8]*z[9] + 258064*z[6]*z[8] + 516128*z[6]*z[9] - 52874672*z[6] - 8128*z[7]*z[8]*z[9] + 516128*z[7]*z[8] + 1032256*z[7]*z[9] - 93264736*z[7] + 43169*z[8]*z[9] - 1717294*z[8] - 3434207*z[9] + 326978409/2)

In [305]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 14  # 
p = 1  # QAOA depth

hamiltonian_less_127_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_equation_less_127_2))
final_circuit_less_127_2, result_less_127_2 = qaoa(num_qubits, hamiltonian_less_127_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.05331899 0.81324159]
Minimum expectation value: 21327126.140625


In [306]:
top_solutions_less_127_2 = find_best_bitstrings(final_circuit_less_127_2, hamiltonian_less_127_2)

Top 5 bitstrings:
Bitstring: 00000000000101, Cost: -326977924.0000, Count: 1
Bitstring: 00000100000101, Cost: -326977732.0000, Count: 1
Bitstring: 00010000000101, Cost: -326971012.0000, Count: 1
Bitstring: 00001010001001, Cost: -326967800.0000, Count: 1
Bitstring: 00000000001101, Cost: -326950852.0000, Count: 1


In [307]:
bitstring_to_pm1(top_solutions_less_127_2, evaluate_hamiltonian_less_127_2)

["Bitstring: ('00000000000101', -326977924.0, 1), Evaluated cost: 484.0",
 "Bitstring: ('00000000000101', -326977924.0, 1), Evaluated cost: 676.0",
 "Bitstring: ('00000000000101', -326977924.0, 1), Evaluated cost: 7396.0",
 "Bitstring: ('00000000000101', -326977924.0, 1), Evaluated cost: 10609.0",
 "Bitstring: ('00000000000101', -326977924.0, 1), Evaluated cost: 27556.0"]

For numbers between 0 and 255, which has binary length of 8

In [308]:
paso1_D_equation_less_255_2 = substitute_with_global_binary_symbols(D_equation(x[1],x[2])**2, 8, base_name="b")
paso2_D_equation_less_255_2 = remove_variable_exponents(paso1_D_equation_less_255_2)
paso3_D_equation_less_255_2 = substitute_with_spin_variables(paso2_D_equation_less_255_2)

In [309]:
def evaluate_hamiltonian_less_250_2(z):
    return(8*z[1]*z[10]*z[11]*z[2] + 16*z[1]*z[10]*z[11]*z[3] + 32*z[1]*z[10]*z[11]*z[4] + 64*z[1]*z[10]*z[11]*z[5] + 128*z[1]*z[10]*z[11]*z[6] + 256*z[1]*z[10]*z[11]*z[7] + 512*z[1]*z[10]*z[11]*z[8] - 1020*z[1]*z[10]*z[11] + 16*z[1]*z[10]*z[12]*z[2] + 32*z[1]*z[10]*z[12]*z[3] + 64*z[1]*z[10]*z[12]*z[4] + 128*z[1]*z[10]*z[12]*z[5] + 256*z[1]*z[10]*z[12]*z[6] + 512*z[1]*z[10]*z[12]*z[7] + 1024*z[1]*z[10]*z[12]*z[8] - 2040*z[1]*z[10]*z[12] + 32*z[1]*z[10]*z[13]*z[2] + 64*z[1]*z[10]*z[13]*z[3] + 128*z[1]*z[10]*z[13]*z[4] + 256*z[1]*z[10]*z[13]*z[5] + 512*z[1]*z[10]*z[13]*z[6] + 1024*z[1]*z[10]*z[13]*z[7] + 2048*z[1]*z[10]*z[13]*z[8] - 4080*z[1]*z[10]*z[13] + 64*z[1]*z[10]*z[14]*z[2] + 128*z[1]*z[10]*z[14]*z[3] + 256*z[1]*z[10]*z[14]*z[4] + 512*z[1]*z[10]*z[14]*z[5] + 1024*z[1]*z[10]*z[14]*z[6] + 2048*z[1]*z[10]*z[14]*z[7] + 4096*z[1]*z[10]*z[14]*z[8] - 8160*z[1]*z[10]*z[14] + 128*z[1]*z[10]*z[15]*z[2] + 256*z[1]*z[10]*z[15]*z[3] + 512*z[1]*z[10]*z[15]*z[4] + 1024*z[1]*z[10]*z[15]*z[5] + 2048*z[1]*z[10]*z[15]*z[6] + 4096*z[1]*z[10]*z[15]*z[7] + 8192*z[1]*z[10]*z[15]*z[8] - 16320*z[1]*z[10]*z[15] + 256*z[1]*z[10]*z[16]*z[2] + 512*z[1]*z[10]*z[16]*z[3] + 1024*z[1]*z[10]*z[16]*z[4] + 2048*z[1]*z[10]*z[16]*z[5] + 4096*z[1]*z[10]*z[16]*z[6] + 8192*z[1]*z[10]*z[16]*z[7] + 16384*z[1]*z[10]*z[16]*z[8] - 32640*z[1]*z[10]*z[16] + 2*z[1]*z[10]*z[2]*z[9] - 510*z[1]*z[10]*z[2] + 4*z[1]*z[10]*z[3]*z[9] - 1020*z[1]*z[10]*z[3] + 8*z[1]*z[10]*z[4]*z[9] - 2040*z[1]*z[10]*z[4] + 16*z[1]*z[10]*z[5]*z[9] - 4080*z[1]*z[10]*z[5] + 32*z[1]*z[10]*z[6]*z[9] - 8160*z[1]*z[10]*z[6] + 64*z[1]*z[10]*z[7]*z[9] - 16320*z[1]*z[10]*z[7] + 128*z[1]*z[10]*z[8]*z[9] - 32640*z[1]*z[10]*z[8] - 255*z[1]*z[10]*z[9] + 65025*z[1]*z[10] + 32*z[1]*z[11]*z[12]*z[2] + 64*z[1]*z[11]*z[12]*z[3] + 128*z[1]*z[11]*z[12]*z[4] + 256*z[1]*z[11]*z[12]*z[5] + 512*z[1]*z[11]*z[12]*z[6] + 1024*z[1]*z[11]*z[12]*z[7] + 2048*z[1]*z[11]*z[12]*z[8] - 4080*z[1]*z[11]*z[12] + 64*z[1]*z[11]*z[13]*z[2] + 128*z[1]*z[11]*z[13]*z[3] + 256*z[1]*z[11]*z[13]*z[4] + 512*z[1]*z[11]*z[13]*z[5] + 1024*z[1]*z[11]*z[13]*z[6] + 2048*z[1]*z[11]*z[13]*z[7] + 4096*z[1]*z[11]*z[13]*z[8] - 8160*z[1]*z[11]*z[13] + 128*z[1]*z[11]*z[14]*z[2] + 256*z[1]*z[11]*z[14]*z[3] + 512*z[1]*z[11]*z[14]*z[4] + 1024*z[1]*z[11]*z[14]*z[5] + 2048*z[1]*z[11]*z[14]*z[6] + 4096*z[1]*z[11]*z[14]*z[7] + 8192*z[1]*z[11]*z[14]*z[8] - 16320*z[1]*z[11]*z[14] + 256*z[1]*z[11]*z[15]*z[2] + 512*z[1]*z[11]*z[15]*z[3] + 1024*z[1]*z[11]*z[15]*z[4] + 2048*z[1]*z[11]*z[15]*z[5] + 4096*z[1]*z[11]*z[15]*z[6] + 8192*z[1]*z[11]*z[15]*z[7] + 16384*z[1]*z[11]*z[15]*z[8] - 32640*z[1]*z[11]*z[15] + 512*z[1]*z[11]*z[16]*z[2] + 1024*z[1]*z[11]*z[16]*z[3] + 2048*z[1]*z[11]*z[16]*z[4] + 4096*z[1]*z[11]*z[16]*z[5] + 8192*z[1]*z[11]*z[16]*z[6] + 16384*z[1]*z[11]*z[16]*z[7] + 32768*z[1]*z[11]*z[16]*z[8] - 65280*z[1]*z[11]*z[16] + 4*z[1]*z[11]*z[2]*z[9] - 1020*z[1]*z[11]*z[2] + 8*z[1]*z[11]*z[3]*z[9] - 2040*z[1]*z[11]*z[3] + 16*z[1]*z[11]*z[4]*z[9] - 4080*z[1]*z[11]*z[4] + 32*z[1]*z[11]*z[5]*z[9] - 8160*z[1]*z[11]*z[5] + 64*z[1]*z[11]*z[6]*z[9] - 16320*z[1]*z[11]*z[6] + 128*z[1]*z[11]*z[7]*z[9] - 32640*z[1]*z[11]*z[7] + 256*z[1]*z[11]*z[8]*z[9] - 65280*z[1]*z[11]*z[8] - 510*z[1]*z[11]*z[9] + 130050*z[1]*z[11] + 128*z[1]*z[12]*z[13]*z[2] + 256*z[1]*z[12]*z[13]*z[3] + 512*z[1]*z[12]*z[13]*z[4] + 1024*z[1]*z[12]*z[13]*z[5] + 2048*z[1]*z[12]*z[13]*z[6] + 4096*z[1]*z[12]*z[13]*z[7] + 8192*z[1]*z[12]*z[13]*z[8] - 16320*z[1]*z[12]*z[13] + 256*z[1]*z[12]*z[14]*z[2] + 512*z[1]*z[12]*z[14]*z[3] + 1024*z[1]*z[12]*z[14]*z[4] + 2048*z[1]*z[12]*z[14]*z[5] + 4096*z[1]*z[12]*z[14]*z[6] + 8192*z[1]*z[12]*z[14]*z[7] + 16384*z[1]*z[12]*z[14]*z[8] - 32640*z[1]*z[12]*z[14] + 512*z[1]*z[12]*z[15]*z[2] + 1024*z[1]*z[12]*z[15]*z[3] + 2048*z[1]*z[12]*z[15]*z[4] + 4096*z[1]*z[12]*z[15]*z[5] + 8192*z[1]*z[12]*z[15]*z[6] + 16384*z[1]*z[12]*z[15]*z[7] + 32768*z[1]*z[12]*z[15]*z[8] - 65280*z[1]*z[12]*z[15] + 1024*z[1]*z[12]*z[16]*z[2] + 2048*z[1]*z[12]*z[16]*z[3] + 4096*z[1]*z[12]*z[16]*z[4] + 8192*z[1]*z[12]*z[16]*z[5] + 16384*z[1]*z[12]*z[16]*z[6] + 32768*z[1]*z[12]*z[16]*z[7] + 65536*z[1]*z[12]*z[16]*z[8] - 130560*z[1]*z[12]*z[16] + 8*z[1]*z[12]*z[2]*z[9] - 2040*z[1]*z[12]*z[2] + 16*z[1]*z[12]*z[3]*z[9] - 4080*z[1]*z[12]*z[3] + 32*z[1]*z[12]*z[4]*z[9] - 8160*z[1]*z[12]*z[4] + 64*z[1]*z[12]*z[5]*z[9] - 16320*z[1]*z[12]*z[5] + 128*z[1]*z[12]*z[6]*z[9] - 32640*z[1]*z[12]*z[6] + 256*z[1]*z[12]*z[7]*z[9] - 65280*z[1]*z[12]*z[7] + 512*z[1]*z[12]*z[8]*z[9] - 130560*z[1]*z[12]*z[8] - 1020*z[1]*z[12]*z[9] + 260100*z[1]*z[12] + 512*z[1]*z[13]*z[14]*z[2] + 1024*z[1]*z[13]*z[14]*z[3] + 2048*z[1]*z[13]*z[14]*z[4] + 4096*z[1]*z[13]*z[14]*z[5] + 8192*z[1]*z[13]*z[14]*z[6] + 16384*z[1]*z[13]*z[14]*z[7] + 32768*z[1]*z[13]*z[14]*z[8] - 65280*z[1]*z[13]*z[14] + 1024*z[1]*z[13]*z[15]*z[2] + 2048*z[1]*z[13]*z[15]*z[3] + 4096*z[1]*z[13]*z[15]*z[4] + 8192*z[1]*z[13]*z[15]*z[5] + 16384*z[1]*z[13]*z[15]*z[6] + 32768*z[1]*z[13]*z[15]*z[7] + 65536*z[1]*z[13]*z[15]*z[8] - 130560*z[1]*z[13]*z[15] + 2048*z[1]*z[13]*z[16]*z[2] + 4096*z[1]*z[13]*z[16]*z[3] + 8192*z[1]*z[13]*z[16]*z[4] + 16384*z[1]*z[13]*z[16]*z[5] + 32768*z[1]*z[13]*z[16]*z[6] + 65536*z[1]*z[13]*z[16]*z[7] + 131072*z[1]*z[13]*z[16]*z[8] - 261120*z[1]*z[13]*z[16] + 16*z[1]*z[13]*z[2]*z[9] - 4080*z[1]*z[13]*z[2] + 32*z[1]*z[13]*z[3]*z[9] - 8160*z[1]*z[13]*z[3] + 64*z[1]*z[13]*z[4]*z[9] - 16320*z[1]*z[13]*z[4] + 128*z[1]*z[13]*z[5]*z[9] - 32640*z[1]*z[13]*z[5] + 256*z[1]*z[13]*z[6]*z[9] - 65280*z[1]*z[13]*z[6] + 512*z[1]*z[13]*z[7]*z[9] - 130560*z[1]*z[13]*z[7] + 1024*z[1]*z[13]*z[8]*z[9] - 261120*z[1]*z[13]*z[8] - 2040*z[1]*z[13]*z[9] + 520200*z[1]*z[13] + 2048*z[1]*z[14]*z[15]*z[2] + 4096*z[1]*z[14]*z[15]*z[3] + 8192*z[1]*z[14]*z[15]*z[4] + 16384*z[1]*z[14]*z[15]*z[5] + 32768*z[1]*z[14]*z[15]*z[6] + 65536*z[1]*z[14]*z[15]*z[7] + 131072*z[1]*z[14]*z[15]*z[8] - 261120*z[1]*z[14]*z[15] + 4096*z[1]*z[14]*z[16]*z[2] + 8192*z[1]*z[14]*z[16]*z[3] + 16384*z[1]*z[14]*z[16]*z[4] + 32768*z[1]*z[14]*z[16]*z[5] + 65536*z[1]*z[14]*z[16]*z[6] + 131072*z[1]*z[14]*z[16]*z[7] + 262144*z[1]*z[14]*z[16]*z[8] - 522240*z[1]*z[14]*z[16] + 32*z[1]*z[14]*z[2]*z[9] - 8160*z[1]*z[14]*z[2] + 64*z[1]*z[14]*z[3]*z[9] - 16320*z[1]*z[14]*z[3] + 128*z[1]*z[14]*z[4]*z[9] - 32640*z[1]*z[14]*z[4] + 256*z[1]*z[14]*z[5]*z[9] - 65280*z[1]*z[14]*z[5] + 512*z[1]*z[14]*z[6]*z[9] - 130560*z[1]*z[14]*z[6] + 1024*z[1]*z[14]*z[7]*z[9] - 261120*z[1]*z[14]*z[7] + 2048*z[1]*z[14]*z[8]*z[9] - 522240*z[1]*z[14]*z[8] - 4080*z[1]*z[14]*z[9] + 1040400*z[1]*z[14] + 8192*z[1]*z[15]*z[16]*z[2] + 16384*z[1]*z[15]*z[16]*z[3] + 32768*z[1]*z[15]*z[16]*z[4] + 65536*z[1]*z[15]*z[16]*z[5] + 131072*z[1]*z[15]*z[16]*z[6] + 262144*z[1]*z[15]*z[16]*z[7] + 524288*z[1]*z[15]*z[16]*z[8] - 1044480*z[1]*z[15]*z[16] + 64*z[1]*z[15]*z[2]*z[9] - 16320*z[1]*z[15]*z[2] + 128*z[1]*z[15]*z[3]*z[9] - 32640*z[1]*z[15]*z[3] + 256*z[1]*z[15]*z[4]*z[9] - 65280*z[1]*z[15]*z[4] + 512*z[1]*z[15]*z[5]*z[9] - 130560*z[1]*z[15]*z[5] + 1024*z[1]*z[15]*z[6]*z[9] - 261120*z[1]*z[15]*z[6] + 2048*z[1]*z[15]*z[7]*z[9] - 522240*z[1]*z[15]*z[7] + 4096*z[1]*z[15]*z[8]*z[9] - 1044480*z[1]*z[15]*z[8] - 8160*z[1]*z[15]*z[9] + 2080800*z[1]*z[15] + 128*z[1]*z[16]*z[2]*z[9] - 32640*z[1]*z[16]*z[2] + 256*z[1]*z[16]*z[3]*z[9] - 65280*z[1]*z[16]*z[3] + 512*z[1]*z[16]*z[4]*z[9] - 130560*z[1]*z[16]*z[4] + 1024*z[1]*z[16]*z[5]*z[9] - 261120*z[1]*z[16]*z[5] + 2048*z[1]*z[16]*z[6]*z[9] - 522240*z[1]*z[16]*z[6] + 4096*z[1]*z[16]*z[7]*z[9] - 1044480*z[1]*z[16]*z[7] + 8192*z[1]*z[16]*z[8]*z[9] - 2088960*z[1]*z[16]*z[8] - 16320*z[1]*z[16]*z[9] + 4161600*z[1]*z[16] + 96*z[1]*z[2]*z[3]*z[4] + 192*z[1]*z[2]*z[3]*z[5] + 384*z[1]*z[2]*z[3]*z[6] + 768*z[1]*z[2]*z[3]*z[7] + 1536*z[1]*z[2]*z[3]*z[8] - 3060*z[1]*z[2]*z[3] + 384*z[1]*z[2]*z[4]*z[5] + 768*z[1]*z[2]*z[4]*z[6] + 1536*z[1]*z[2]*z[4]*z[7] + 3072*z[1]*z[2]*z[4]*z[8] - 6120*z[1]*z[2]*z[4] + 1536*z[1]*z[2]*z[5]*z[6] + 3072*z[1]*z[2]*z[5]*z[7] + 6144*z[1]*z[2]*z[5]*z[8] - 12240*z[1]*z[2]*z[5] + 6144*z[1]*z[2]*z[6]*z[7] + 12288*z[1]*z[2]*z[6]*z[8] - 24480*z[1]*z[2]*z[6] + 24576*z[1]*z[2]*z[7]*z[8] - 48960*z[1]*z[2]*z[7] - 97920*z[1]*z[2]*z[8] - 255*z[1]*z[2]*z[9] + 173729*z[1]*z[2] + 768*z[1]*z[3]*z[4]*z[5] + 1536*z[1]*z[3]*z[4]*z[6] + 3072*z[1]*z[3]*z[4]*z[7] + 6144*z[1]*z[3]*z[4]*z[8] - 12240*z[1]*z[3]*z[4] + 3072*z[1]*z[3]*z[5]*z[6] + 6144*z[1]*z[3]*z[5]*z[7] + 12288*z[1]*z[3]*z[5]*z[8] - 24480*z[1]*z[3]*z[5] + 12288*z[1]*z[3]*z[6]*z[7] + 24576*z[1]*z[3]*z[6]*z[8] - 48960*z[1]*z[3]*z[6] + 49152*z[1]*z[3]*z[7]*z[8] - 97920*z[1]*z[3]*z[7] - 195840*z[1]*z[3]*z[8] - 510*z[1]*z[3]*z[9] + 347434*z[1]*z[3] + 6144*z[1]*z[4]*z[5]*z[6] + 12288*z[1]*z[4]*z[5]*z[7] + 24576*z[1]*z[4]*z[5]*z[8] - 48960*z[1]*z[4]*z[5] + 24576*z[1]*z[4]*z[6]*z[7] + 49152*z[1]*z[4]*z[6]*z[8] - 97920*z[1]*z[4]*z[6] + 98304*z[1]*z[4]*z[7]*z[8] - 195840*z[1]*z[4]*z[7] - 391680*z[1]*z[4]*z[8] - 1020*z[1]*z[4]*z[9] + 694676*z[1]*z[4] + 49152*z[1]*z[5]*z[6]*z[7] + 98304*z[1]*z[5]*z[6]*z[8] - 195840*z[1]*z[5]*z[6] + 196608*z[1]*z[5]*z[7]*z[8] - 391680*z[1]*z[5]*z[7] - 783360*z[1]*z[5]*z[8] - 2040*z[1]*z[5]*z[9] + 1387816*z[1]*z[5] + 393216*z[1]*z[6]*z[7]*z[8] - 783360*z[1]*z[6]*z[7] - 1566720*z[1]*z[6]*z[8] - 4080*z[1]*z[6]*z[9] + 2763344*z[1]*z[6] - 3133440*z[1]*z[7]*z[8] - 8160*z[1]*z[7]*z[9] + 5428384*z[1]*z[7] - 16320*z[1]*z[8]*z[9] + 10070336*z[1]*z[8] + 65025*z[1]*z[9]/2 - 13860270*z[1] + 1536*z[10]*z[11]*z[12]*z[13] + 3072*z[10]*z[11]*z[12]*z[14] + 6144*z[10]*z[11]*z[12]*z[15] + 12288*z[10]*z[11]*z[12]*z[16] + 96*z[10]*z[11]*z[12]*z[9] - 24480*z[10]*z[11]*z[12] + 6144*z[10]*z[11]*z[13]*z[14] + 12288*z[10]*z[11]*z[13]*z[15] + 24576*z[10]*z[11]*z[13]*z[16] + 192*z[10]*z[11]*z[13]*z[9] - 48960*z[10]*z[11]*z[13] + 24576*z[10]*z[11]*z[14]*z[15] + 49152*z[10]*z[11]*z[14]*z[16] + 384*z[10]*z[11]*z[14]*z[9] - 97920*z[10]*z[11]*z[14] + 98304*z[10]*z[11]*z[15]*z[16] + 768*z[10]*z[11]*z[15]*z[9] - 195840*z[10]*z[11]*z[15] + 1536*z[10]*z[11]*z[16]*z[9] - 391680*z[10]*z[11]*z[16] + 32*z[10]*z[11]*z[2]*z[3] + 64*z[10]*z[11]*z[2]*z[4] + 128*z[10]*z[11]*z[2]*z[5] + 256*z[10]*z[11]*z[2]*z[6] + 512*z[10]*z[11]*z[2]*z[7] + 1024*z[10]*z[11]*z[2]*z[8] - 2040*z[10]*z[11]*z[2] + 128*z[10]*z[11]*z[3]*z[4] + 256*z[10]*z[11]*z[3]*z[5] + 512*z[10]*z[11]*z[3]*z[6] + 1024*z[10]*z[11]*z[3]*z[7] + 2048*z[10]*z[11]*z[3]*z[8] - 4080*z[10]*z[11]*z[3] + 512*z[10]*z[11]*z[4]*z[5] + 1024*z[10]*z[11]*z[4]*z[6] + 2048*z[10]*z[11]*z[4]*z[7] + 4096*z[10]*z[11]*z[4]*z[8] - 8160*z[10]*z[11]*z[4] + 2048*z[10]*z[11]*z[5]*z[6] + 4096*z[10]*z[11]*z[5]*z[7] + 8192*z[10]*z[11]*z[5]*z[8] - 16320*z[10]*z[11]*z[5] + 8192*z[10]*z[11]*z[6]*z[7] + 16384*z[10]*z[11]*z[6]*z[8] - 32640*z[10]*z[11]*z[6] + 32768*z[10]*z[11]*z[7]*z[8] - 65280*z[10]*z[11]*z[7] - 130560*z[10]*z[11]*z[8] - 3060*z[10]*z[11]*z[9] + 694856*z[10]*z[11] + 12288*z[10]*z[12]*z[13]*z[14] + 24576*z[10]*z[12]*z[13]*z[15] + 49152*z[10]*z[12]*z[13]*z[16] + 384*z[10]*z[12]*z[13]*z[9] - 97920*z[10]*z[12]*z[13] + 49152*z[10]*z[12]*z[14]*z[15] + 98304*z[10]*z[12]*z[14]*z[16] + 768*z[10]*z[12]*z[14]*z[9] - 195840*z[10]*z[12]*z[14] + 196608*z[10]*z[12]*z[15]*z[16] + 1536*z[10]*z[12]*z[15]*z[9] - 391680*z[10]*z[12]*z[15] + 3072*z[10]*z[12]*z[16]*z[9] - 783360*z[10]*z[12]*z[16] + 64*z[10]*z[12]*z[2]*z[3] + 128*z[10]*z[12]*z[2]*z[4] + 256*z[10]*z[12]*z[2]*z[5] + 512*z[10]*z[12]*z[2]*z[6] + 1024*z[10]*z[12]*z[2]*z[7] + 2048*z[10]*z[12]*z[2]*z[8] - 4080*z[10]*z[12]*z[2] + 256*z[10]*z[12]*z[3]*z[4] + 512*z[10]*z[12]*z[3]*z[5] + 1024*z[10]*z[12]*z[3]*z[6] + 2048*z[10]*z[12]*z[3]*z[7] + 4096*z[10]*z[12]*z[3]*z[8] - 8160*z[10]*z[12]*z[3] + 1024*z[10]*z[12]*z[4]*z[5] + 2048*z[10]*z[12]*z[4]*z[6] + 4096*z[10]*z[12]*z[4]*z[7] + 8192*z[10]*z[12]*z[4]*z[8] - 16320*z[10]*z[12]*z[4] + 4096*z[10]*z[12]*z[5]*z[6] + 8192*z[10]*z[12]*z[5]*z[7] + 16384*z[10]*z[12]*z[5]*z[8] - 32640*z[10]*z[12]*z[5] + 16384*z[10]*z[12]*z[6]*z[7] + 32768*z[10]*z[12]*z[6]*z[8] - 65280*z[10]*z[12]*z[6] + 65536*z[10]*z[12]*z[7]*z[8] - 130560*z[10]*z[12]*z[7] - 261120*z[10]*z[12]*z[8] - 6120*z[10]*z[12]*z[9] + 1389328*z[10]*z[12] + 98304*z[10]*z[13]*z[14]*z[15] + 196608*z[10]*z[13]*z[14]*z[16] + 1536*z[10]*z[13]*z[14]*z[9] - 391680*z[10]*z[13]*z[14] + 393216*z[10]*z[13]*z[15]*z[16] + 3072*z[10]*z[13]*z[15]*z[9] - 783360*z[10]*z[13]*z[15] + 6144*z[10]*z[13]*z[16]*z[9] - 1566720*z[10]*z[13]*z[16] + 128*z[10]*z[13]*z[2]*z[3] + 256*z[10]*z[13]*z[2]*z[4] + 512*z[10]*z[13]*z[2]*z[5] + 1024*z[10]*z[13]*z[2]*z[6] + 2048*z[10]*z[13]*z[2]*z[7] + 4096*z[10]*z[13]*z[2]*z[8] - 8160*z[10]*z[13]*z[2] + 512*z[10]*z[13]*z[3]*z[4] + 1024*z[10]*z[13]*z[3]*z[5] + 2048*z[10]*z[13]*z[3]*z[6] + 4096*z[10]*z[13]*z[3]*z[7] + 8192*z[10]*z[13]*z[3]*z[8] - 16320*z[10]*z[13]*z[3] + 2048*z[10]*z[13]*z[4]*z[5] + 4096*z[10]*z[13]*z[4]*z[6] + 8192*z[10]*z[13]*z[4]*z[7] + 16384*z[10]*z[13]*z[4]*z[8] - 32640*z[10]*z[13]*z[4] + 8192*z[10]*z[13]*z[5]*z[6] + 16384*z[10]*z[13]*z[5]*z[7] + 32768*z[10]*z[13]*z[5]*z[8] - 65280*z[10]*z[13]*z[5] + 32768*z[10]*z[13]*z[6]*z[7] + 65536*z[10]*z[13]*z[6]*z[8] - 130560*z[10]*z[13]*z[6] + 131072*z[10]*z[13]*z[7]*z[8] - 261120*z[10]*z[13]*z[7] - 522240*z[10]*z[13]*z[8] - 12240*z[10]*z[13]*z[9] + 2775584*z[10]*z[13] + 786432*z[10]*z[14]*z[15]*z[16] + 6144*z[10]*z[14]*z[15]*z[9] - 1566720*z[10]*z[14]*z[15] + 12288*z[10]*z[14]*z[16]*z[9] - 3133440*z[10]*z[14]*z[16] + 256*z[10]*z[14]*z[2]*z[3] + 512*z[10]*z[14]*z[2]*z[4] + 1024*z[10]*z[14]*z[2]*z[5] + 2048*z[10]*z[14]*z[2]*z[6] + 4096*z[10]*z[14]*z[2]*z[7] + 8192*z[10]*z[14]*z[2]*z[8] - 16320*z[10]*z[14]*z[2] + 1024*z[10]*z[14]*z[3]*z[4] + 2048*z[10]*z[14]*z[3]*z[5] + 4096*z[10]*z[14]*z[3]*z[6] + 8192*z[10]*z[14]*z[3]*z[7] + 16384*z[10]*z[14]*z[3]*z[8] - 32640*z[10]*z[14]*z[3] + 4096*z[10]*z[14]*z[4]*z[5] + 8192*z[10]*z[14]*z[4]*z[6] + 16384*z[10]*z[14]*z[4]*z[7] + 32768*z[10]*z[14]*z[4]*z[8] - 65280*z[10]*z[14]*z[4] + 16384*z[10]*z[14]*z[5]*z[6] + 32768*z[10]*z[14]*z[5]*z[7] + 65536*z[10]*z[14]*z[5]*z[8] - 130560*z[10]*z[14]*z[5] + 65536*z[10]*z[14]*z[6]*z[7] + 131072*z[10]*z[14]*z[6]*z[8] - 261120*z[10]*z[14]*z[6] + 262144*z[10]*z[14]*z[7]*z[8] - 522240*z[10]*z[14]*z[7] - 1044480*z[10]*z[14]*z[8] - 24480*z[10]*z[14]*z[9] + 5526592*z[10]*z[14] + 24576*z[10]*z[15]*z[16]*z[9] - 6266880*z[10]*z[15]*z[16] + 512*z[10]*z[15]*z[2]*z[3] + 1024*z[10]*z[15]*z[2]*z[4] + 2048*z[10]*z[15]*z[2]*z[5] + 4096*z[10]*z[15]*z[2]*z[6] + 8192*z[10]*z[15]*z[2]*z[7] + 16384*z[10]*z[15]*z[2]*z[8] - 32640*z[10]*z[15]*z[2] + 2048*z[10]*z[15]*z[3]*z[4] + 4096*z[10]*z[15]*z[3]*z[5] + 8192*z[10]*z[15]*z[3]*z[6] + 16384*z[10]*z[15]*z[3]*z[7] + 32768*z[10]*z[15]*z[3]*z[8] - 65280*z[10]*z[15]*z[3] + 8192*z[10]*z[15]*z[4]*z[5] + 16384*z[10]*z[15]*z[4]*z[6] + 32768*z[10]*z[15]*z[4]*z[7] + 65536*z[10]*z[15]*z[4]*z[8] - 130560*z[10]*z[15]*z[4] + 32768*z[10]*z[15]*z[5]*z[6] + 65536*z[10]*z[15]*z[5]*z[7] + 131072*z[10]*z[15]*z[5]*z[8] - 261120*z[10]*z[15]*z[5] + 131072*z[10]*z[15]*z[6]*z[7] + 262144*z[10]*z[15]*z[6]*z[8] - 522240*z[10]*z[15]*z[6] + 524288*z[10]*z[15]*z[7]*z[8] - 1044480*z[10]*z[15]*z[7] - 2088960*z[10]*z[15]*z[8] - 48960*z[10]*z[15]*z[9] + 10856576*z[10]*z[15] + 1024*z[10]*z[16]*z[2]*z[3] + 2048*z[10]*z[16]*z[2]*z[4] + 4096*z[10]*z[16]*z[2]*z[5] + 8192*z[10]*z[16]*z[2]*z[6] + 16384*z[10]*z[16]*z[2]*z[7] + 32768*z[10]*z[16]*z[2]*z[8] - 65280*z[10]*z[16]*z[2] + 4096*z[10]*z[16]*z[3]*z[4] + 8192*z[10]*z[16]*z[3]*z[5] + 16384*z[10]*z[16]*z[3]*z[6] + 32768*z[10]*z[16]*z[3]*z[7] + 65536*z[10]*z[16]*z[3]*z[8] - 130560*z[10]*z[16]*z[3] + 16384*z[10]*z[16]*z[4]*z[5] + 32768*z[10]*z[16]*z[4]*z[6] + 65536*z[10]*z[16]*z[4]*z[7] + 131072*z[10]*z[16]*z[4]*z[8] - 261120*z[10]*z[16]*z[4] + 65536*z[10]*z[16]*z[5]*z[6] + 131072*z[10]*z[16]*z[5]*z[7] + 262144*z[10]*z[16]*z[5]*z[8] - 522240*z[10]*z[16]*z[5] + 262144*z[10]*z[16]*z[6]*z[7] + 524288*z[10]*z[16]*z[6]*z[8] - 1044480*z[10]*z[16]*z[6] + 1048576*z[10]*z[16]*z[7]*z[8] - 2088960*z[10]*z[16]*z[7] - 4177920*z[10]*z[16]*z[8] - 97920*z[10]*z[16]*z[9] + 20140288*z[10]*z[16] + 8*z[10]*z[2]*z[3]*z[9] - 2040*z[10]*z[2]*z[3] + 16*z[10]*z[2]*z[4]*z[9] - 4080*z[10]*z[2]*z[4] + 32*z[10]*z[2]*z[5]*z[9] - 8160*z[10]*z[2]*z[5] + 64*z[10]*z[2]*z[6]*z[9] - 16320*z[10]*z[2]*z[6] + 128*z[10]*z[2]*z[7]*z[9] - 32640*z[10]*z[2]*z[7] + 256*z[10]*z[2]*z[8]*z[9] - 65280*z[10]*z[2]*z[8] - 510*z[10]*z[2]*z[9] + 130050*z[10]*z[2] + 32*z[10]*z[3]*z[4]*z[9] - 8160*z[10]*z[3]*z[4] + 64*z[10]*z[3]*z[5]*z[9] - 16320*z[10]*z[3]*z[5] + 128*z[10]*z[3]*z[6]*z[9] - 32640*z[10]*z[3]*z[6] + 256*z[10]*z[3]*z[7]*z[9] - 65280*z[10]*z[3]*z[7] + 512*z[10]*z[3]*z[8]*z[9] - 130560*z[10]*z[3]*z[8] - 1020*z[10]*z[3]*z[9] + 260100*z[10]*z[3] + 128*z[10]*z[4]*z[5]*z[9] - 32640*z[10]*z[4]*z[5] + 256*z[10]*z[4]*z[6]*z[9] - 65280*z[10]*z[4]*z[6] + 512*z[10]*z[4]*z[7]*z[9] - 130560*z[10]*z[4]*z[7] + 1024*z[10]*z[4]*z[8]*z[9] - 261120*z[10]*z[4]*z[8] - 2040*z[10]*z[4]*z[9] + 520200*z[10]*z[4] + 512*z[10]*z[5]*z[6]*z[9] - 130560*z[10]*z[5]*z[6] + 1024*z[10]*z[5]*z[7]*z[9] - 261120*z[10]*z[5]*z[7] + 2048*z[10]*z[5]*z[8]*z[9] - 522240*z[10]*z[5]*z[8] - 4080*z[10]*z[5]*z[9] + 1040400*z[10]*z[5] + 2048*z[10]*z[6]*z[7]*z[9] - 522240*z[10]*z[6]*z[7] + 4096*z[10]*z[6]*z[8]*z[9] - 1044480*z[10]*z[6]*z[8] - 8160*z[10]*z[6]*z[9] + 2080800*z[10]*z[6] + 8192*z[10]*z[7]*z[8]*z[9] - 2088960*z[10]*z[7]*z[8] - 16320*z[10]*z[7]*z[9] + 4161600*z[10]*z[7] - 32640*z[10]*z[8]*z[9] + 8323200*z[10]*z[8] + 173729*z[10]*z[9] - 27719775*z[10] + 24576*z[11]*z[12]*z[13]*z[14] + 49152*z[11]*z[12]*z[13]*z[15] + 98304*z[11]*z[12]*z[13]*z[16] + 768*z[11]*z[12]*z[13]*z[9] - 195840*z[11]*z[12]*z[13] + 98304*z[11]*z[12]*z[14]*z[15] + 196608*z[11]*z[12]*z[14]*z[16] + 1536*z[11]*z[12]*z[14]*z[9] - 391680*z[11]*z[12]*z[14] + 393216*z[11]*z[12]*z[15]*z[16] + 3072*z[11]*z[12]*z[15]*z[9] - 783360*z[11]*z[12]*z[15] + 6144*z[11]*z[12]*z[16]*z[9] - 1566720*z[11]*z[12]*z[16] + 128*z[11]*z[12]*z[2]*z[3] + 256*z[11]*z[12]*z[2]*z[4] + 512*z[11]*z[12]*z[2]*z[5] + 1024*z[11]*z[12]*z[2]*z[6] + 2048*z[11]*z[12]*z[2]*z[7] + 4096*z[11]*z[12]*z[2]*z[8] - 8160*z[11]*z[12]*z[2] + 512*z[11]*z[12]*z[3]*z[4] + 1024*z[11]*z[12]*z[3]*z[5] + 2048*z[11]*z[12]*z[3]*z[6] + 4096*z[11]*z[12]*z[3]*z[7] + 8192*z[11]*z[12]*z[3]*z[8] - 16320*z[11]*z[12]*z[3] + 2048*z[11]*z[12]*z[4]*z[5] + 4096*z[11]*z[12]*z[4]*z[6] + 8192*z[11]*z[12]*z[4]*z[7] + 16384*z[11]*z[12]*z[4]*z[8] - 32640*z[11]*z[12]*z[4] + 8192*z[11]*z[12]*z[5]*z[6] + 16384*z[11]*z[12]*z[5]*z[7] + 32768*z[11]*z[12]*z[5]*z[8] - 65280*z[11]*z[12]*z[5] + 32768*z[11]*z[12]*z[6]*z[7] + 65536*z[11]*z[12]*z[6]*z[8] - 130560*z[11]*z[12]*z[6] + 131072*z[11]*z[12]*z[7]*z[8] - 261120*z[11]*z[12]*z[7] - 522240*z[11]*z[12]*z[8] - 12240*z[11]*z[12]*z[9] + 2778464*z[11]*z[12] + 196608*z[11]*z[13]*z[14]*z[15] + 393216*z[11]*z[13]*z[14]*z[16] + 3072*z[11]*z[13]*z[14]*z[9] - 783360*z[11]*z[13]*z[14] + 786432*z[11]*z[13]*z[15]*z[16] + 6144*z[11]*z[13]*z[15]*z[9] - 1566720*z[11]*z[13]*z[15] + 12288*z[11]*z[13]*z[16]*z[9] - 3133440*z[11]*z[13]*z[16] + 256*z[11]*z[13]*z[2]*z[3] + 512*z[11]*z[13]*z[2]*z[4] + 1024*z[11]*z[13]*z[2]*z[5] + 2048*z[11]*z[13]*z[2]*z[6] + 4096*z[11]*z[13]*z[2]*z[7] + 8192*z[11]*z[13]*z[2]*z[8] - 16320*z[11]*z[13]*z[2] + 1024*z[11]*z[13]*z[3]*z[4] + 2048*z[11]*z[13]*z[3]*z[5] + 4096*z[11]*z[13]*z[3]*z[6] + 8192*z[11]*z[13]*z[3]*z[7] + 16384*z[11]*z[13]*z[3]*z[8] - 32640*z[11]*z[13]*z[3] + 4096*z[11]*z[13]*z[4]*z[5] + 8192*z[11]*z[13]*z[4]*z[6] + 16384*z[11]*z[13]*z[4]*z[7] + 32768*z[11]*z[13]*z[4]*z[8] - 65280*z[11]*z[13]*z[4] + 16384*z[11]*z[13]*z[5]*z[6] + 32768*z[11]*z[13]*z[5]*z[7] + 65536*z[11]*z[13]*z[5]*z[8] - 130560*z[11]*z[13]*z[5] + 65536*z[11]*z[13]*z[6]*z[7] + 131072*z[11]*z[13]*z[6]*z[8] - 261120*z[11]*z[13]*z[6] + 262144*z[11]*z[13]*z[7]*z[8] - 522240*z[11]*z[13]*z[7] - 1044480*z[11]*z[13]*z[8] - 24480*z[11]*z[13]*z[9] + 5550784*z[11]*z[13] + 1572864*z[11]*z[14]*z[15]*z[16] + 12288*z[11]*z[14]*z[15]*z[9] - 3133440*z[11]*z[14]*z[15] + 24576*z[11]*z[14]*z[16]*z[9] - 6266880*z[11]*z[14]*z[16] + 512*z[11]*z[14]*z[2]*z[3] + 1024*z[11]*z[14]*z[2]*z[4] + 2048*z[11]*z[14]*z[2]*z[5] + 4096*z[11]*z[14]*z[2]*z[6] + 8192*z[11]*z[14]*z[2]*z[7] + 16384*z[11]*z[14]*z[2]*z[8] - 32640*z[11]*z[14]*z[2] + 2048*z[11]*z[14]*z[3]*z[4] + 4096*z[11]*z[14]*z[3]*z[5] + 8192*z[11]*z[14]*z[3]*z[6] + 16384*z[11]*z[14]*z[3]*z[7] + 32768*z[11]*z[14]*z[3]*z[8] - 65280*z[11]*z[14]*z[3] + 8192*z[11]*z[14]*z[4]*z[5] + 16384*z[11]*z[14]*z[4]*z[6] + 32768*z[11]*z[14]*z[4]*z[7] + 65536*z[11]*z[14]*z[4]*z[8] - 130560*z[11]*z[14]*z[4] + 32768*z[11]*z[14]*z[5]*z[6] + 65536*z[11]*z[14]*z[5]*z[7] + 131072*z[11]*z[14]*z[5]*z[8] - 261120*z[11]*z[14]*z[5] + 131072*z[11]*z[14]*z[6]*z[7] + 262144*z[11]*z[14]*z[6]*z[8] - 522240*z[11]*z[14]*z[6] + 524288*z[11]*z[14]*z[7]*z[8] - 1044480*z[11]*z[14]*z[7] - 2088960*z[11]*z[14]*z[8] - 48960*z[11]*z[14]*z[9] + 11052416*z[11]*z[14] + 49152*z[11]*z[15]*z[16]*z[9] - 12533760*z[11]*z[15]*z[16] + 1024*z[11]*z[15]*z[2]*z[3] + 2048*z[11]*z[15]*z[2]*z[4] + 4096*z[11]*z[15]*z[2]*z[5] + 8192*z[11]*z[15]*z[2]*z[6] + 16384*z[11]*z[15]*z[2]*z[7] + 32768*z[11]*z[15]*z[2]*z[8] - 65280*z[11]*z[15]*z[2] + 4096*z[11]*z[15]*z[3]*z[4] + 8192*z[11]*z[15]*z[3]*z[5] + 16384*z[11]*z[15]*z[3]*z[6] + 32768*z[11]*z[15]*z[3]*z[7] + 65536*z[11]*z[15]*z[3]*z[8] - 130560*z[11]*z[15]*z[3] + 16384*z[11]*z[15]*z[4]*z[5] + 32768*z[11]*z[15]*z[4]*z[6] + 65536*z[11]*z[15]*z[4]*z[7] + 131072*z[11]*z[15]*z[4]*z[8] - 261120*z[11]*z[15]*z[4] + 65536*z[11]*z[15]*z[5]*z[6] + 131072*z[11]*z[15]*z[5]*z[7] + 262144*z[11]*z[15]*z[5]*z[8] - 522240*z[11]*z[15]*z[5] + 262144*z[11]*z[15]*z[6]*z[7] + 524288*z[11]*z[15]*z[6]*z[8] - 1044480*z[11]*z[15]*z[6] + 1048576*z[11]*z[15]*z[7]*z[8] - 2088960*z[11]*z[15]*z[7] - 4177920*z[11]*z[15]*z[8] - 97920*z[11]*z[15]*z[9] + 21711616*z[11]*z[15] + 2048*z[11]*z[16]*z[2]*z[3] + 4096*z[11]*z[16]*z[2]*z[4] + 8192*z[11]*z[16]*z[2]*z[5] + 16384*z[11]*z[16]*z[2]*z[6] + 32768*z[11]*z[16]*z[2]*z[7] + 65536*z[11]*z[16]*z[2]*z[8] - 130560*z[11]*z[16]*z[2] + 8192*z[11]*z[16]*z[3]*z[4] + 16384*z[11]*z[16]*z[3]*z[5] + 32768*z[11]*z[16]*z[3]*z[6] + 65536*z[11]*z[16]*z[3]*z[7] + 131072*z[11]*z[16]*z[3]*z[8] - 261120*z[11]*z[16]*z[3] + 32768*z[11]*z[16]*z[4]*z[5] + 65536*z[11]*z[16]*z[4]*z[6] + 131072*z[11]*z[16]*z[4]*z[7] + 262144*z[11]*z[16]*z[4]*z[8] - 522240*z[11]*z[16]*z[4] + 131072*z[11]*z[16]*z[5]*z[6] + 262144*z[11]*z[16]*z[5]*z[7] + 524288*z[11]*z[16]*z[5]*z[8] - 1044480*z[11]*z[16]*z[5] + 524288*z[11]*z[16]*z[6]*z[7] + 1048576*z[11]*z[16]*z[6]*z[8] - 2088960*z[11]*z[16]*z[6] + 2097152*z[11]*z[16]*z[7]*z[8] - 4177920*z[11]*z[16]*z[7] - 8355840*z[11]*z[16]*z[8] - 195840*z[11]*z[16]*z[9] + 40277504*z[11]*z[16] + 16*z[11]*z[2]*z[3]*z[9] - 4080*z[11]*z[2]*z[3] + 32*z[11]*z[2]*z[4]*z[9] - 8160*z[11]*z[2]*z[4] + 64*z[11]*z[2]*z[5]*z[9] - 16320*z[11]*z[2]*z[5] + 128*z[11]*z[2]*z[6]*z[9] - 32640*z[11]*z[2]*z[6] + 256*z[11]*z[2]*z[7]*z[9] - 65280*z[11]*z[2]*z[7] + 512*z[11]*z[2]*z[8]*z[9] - 130560*z[11]*z[2]*z[8] - 1020*z[11]*z[2]*z[9] + 260100*z[11]*z[2] + 64*z[11]*z[3]*z[4]*z[9] - 16320*z[11]*z[3]*z[4] + 128*z[11]*z[3]*z[5]*z[9] - 32640*z[11]*z[3]*z[5] + 256*z[11]*z[3]*z[6]*z[9] - 65280*z[11]*z[3]*z[6] + 512*z[11]*z[3]*z[7]*z[9] - 130560*z[11]*z[3]*z[7] + 1024*z[11]*z[3]*z[8]*z[9] - 261120*z[11]*z[3]*z[8] - 2040*z[11]*z[3]*z[9] + 520200*z[11]*z[3] + 256*z[11]*z[4]*z[5]*z[9] - 65280*z[11]*z[4]*z[5] + 512*z[11]*z[4]*z[6]*z[9] - 130560*z[11]*z[4]*z[6] + 1024*z[11]*z[4]*z[7]*z[9] - 261120*z[11]*z[4]*z[7] + 2048*z[11]*z[4]*z[8]*z[9] - 522240*z[11]*z[4]*z[8] - 4080*z[11]*z[4]*z[9] + 1040400*z[11]*z[4] + 1024*z[11]*z[5]*z[6]*z[9] - 261120*z[11]*z[5]*z[6] + 2048*z[11]*z[5]*z[7]*z[9] - 522240*z[11]*z[5]*z[7] + 4096*z[11]*z[5]*z[8]*z[9] - 1044480*z[11]*z[5]*z[8] - 8160*z[11]*z[5]*z[9] + 2080800*z[11]*z[5] + 4096*z[11]*z[6]*z[7]*z[9] - 1044480*z[11]*z[6]*z[7] + 8192*z[11]*z[6]*z[8]*z[9] - 2088960*z[11]*z[6]*z[8] - 16320*z[11]*z[6]*z[9] + 4161600*z[11]*z[6] + 16384*z[11]*z[7]*z[8]*z[9] - 4177920*z[11]*z[7]*z[8] - 32640*z[11]*z[7]*z[9] + 8323200*z[11]*z[7] - 65280*z[11]*z[8]*z[9] + 16646400*z[11]*z[8] + 347434*z[11]*z[9] - 55433430*z[11] + 393216*z[12]*z[13]*z[14]*z[15] + 786432*z[12]*z[13]*z[14]*z[16] + 6144*z[12]*z[13]*z[14]*z[9] - 1566720*z[12]*z[13]*z[14] + 1572864*z[12]*z[13]*z[15]*z[16] + 12288*z[12]*z[13]*z[15]*z[9] - 3133440*z[12]*z[13]*z[15] + 24576*z[12]*z[13]*z[16]*z[9] - 6266880*z[12]*z[13]*z[16] + 512*z[12]*z[13]*z[2]*z[3] + 1024*z[12]*z[13]*z[2]*z[4] + 2048*z[12]*z[13]*z[2]*z[5] + 4096*z[12]*z[13]*z[2]*z[6] + 8192*z[12]*z[13]*z[2]*z[7] + 16384*z[12]*z[13]*z[2]*z[8] - 32640*z[12]*z[13]*z[2] + 2048*z[12]*z[13]*z[3]*z[4] + 4096*z[12]*z[13]*z[3]*z[5] + 8192*z[12]*z[13]*z[3]*z[6] + 16384*z[12]*z[13]*z[3]*z[7] + 32768*z[12]*z[13]*z[3]*z[8] - 65280*z[12]*z[13]*z[3] + 8192*z[12]*z[13]*z[4]*z[5] + 16384*z[12]*z[13]*z[4]*z[6] + 32768*z[12]*z[13]*z[4]*z[7] + 65536*z[12]*z[13]*z[4]*z[8] - 130560*z[12]*z[13]*z[4] + 32768*z[12]*z[13]*z[5]*z[6] + 65536*z[12]*z[13]*z[5]*z[7] + 131072*z[12]*z[13]*z[5]*z[8] - 261120*z[12]*z[13]*z[5] + 131072*z[12]*z[13]*z[6]*z[7] + 262144*z[12]*z[13]*z[6]*z[8] - 522240*z[12]*z[13]*z[6] + 524288*z[12]*z[13]*z[7]*z[8] - 1044480*z[12]*z[13]*z[7] - 2088960*z[12]*z[13]*z[8] - 48960*z[12]*z[13]*z[9] + 11098496*z[12]*z[13] + 3145728*z[12]*z[14]*z[15]*z[16] + 24576*z[12]*z[14]*z[15]*z[9] - 6266880*z[12]*z[14]*z[15] + 49152*z[12]*z[14]*z[16]*z[9] - 12533760*z[12]*z[14]*z[16] + 1024*z[12]*z[14]*z[2]*z[3] + 2048*z[12]*z[14]*z[2]*z[4] + 4096*z[12]*z[14]*z[2]*z[5] + 8192*z[12]*z[14]*z[2]*z[6] + 16384*z[12]*z[14]*z[2]*z[7] + 32768*z[12]*z[14]*z[2]*z[8] - 65280*z[12]*z[14]*z[2] + 4096*z[12]*z[14]*z[3]*z[4] + 8192*z[12]*z[14]*z[3]*z[5] + 16384*z[12]*z[14]*z[3]*z[6] + 32768*z[12]*z[14]*z[3]*z[7] + 65536*z[12]*z[14]*z[3]*z[8] - 130560*z[12]*z[14]*z[3] + 16384*z[12]*z[14]*z[4]*z[5] + 32768*z[12]*z[14]*z[4]*z[6] + 65536*z[12]*z[14]*z[4]*z[7] + 131072*z[12]*z[14]*z[4]*z[8] - 261120*z[12]*z[14]*z[4] + 65536*z[12]*z[14]*z[5]*z[6] + 131072*z[12]*z[14]*z[5]*z[7] + 262144*z[12]*z[14]*z[5]*z[8] - 522240*z[12]*z[14]*z[5] + 262144*z[12]*z[14]*z[6]*z[7] + 524288*z[12]*z[14]*z[6]*z[8] - 1044480*z[12]*z[14]*z[6] + 1048576*z[12]*z[14]*z[7]*z[8] - 2088960*z[12]*z[14]*z[7] - 4177920*z[12]*z[14]*z[8] - 97920*z[12]*z[14]*z[9] + 22098688*z[12]*z[14] + 98304*z[12]*z[15]*z[16]*z[9] - 25067520*z[12]*z[15]*z[16] + 2048*z[12]*z[15]*z[2]*z[3] + 4096*z[12]*z[15]*z[2]*z[4] + 8192*z[12]*z[15]*z[2]*z[5] + 16384*z[12]*z[15]*z[2]*z[6] + 32768*z[12]*z[15]*z[2]*z[7] + 65536*z[12]*z[15]*z[2]*z[8] - 130560*z[12]*z[15]*z[2] + 8192*z[12]*z[15]*z[3]*z[4] + 16384*z[12]*z[15]*z[3]*z[5] + 32768*z[12]*z[15]*z[3]*z[6] + 65536*z[12]*z[15]*z[3]*z[7] + 131072*z[12]*z[15]*z[3]*z[8] - 261120*z[12]*z[15]*z[3] + 32768*z[12]*z[15]*z[4]*z[5] + 65536*z[12]*z[15]*z[4]*z[6] + 131072*z[12]*z[15]*z[4]*z[7] + 262144*z[12]*z[15]*z[4]*z[8] - 522240*z[12]*z[15]*z[4] + 131072*z[12]*z[15]*z[5]*z[6] + 262144*z[12]*z[15]*z[5]*z[7] + 524288*z[12]*z[15]*z[5]*z[8] - 1044480*z[12]*z[15]*z[5] + 524288*z[12]*z[15]*z[6]*z[7] + 1048576*z[12]*z[15]*z[6]*z[8] - 2088960*z[12]*z[15]*z[6] + 2097152*z[12]*z[15]*z[7]*z[8] - 4177920*z[12]*z[15]*z[7] - 8355840*z[12]*z[15]*z[8] - 195840*z[12]*z[15]*z[9] + 43410944*z[12]*z[15] + 4096*z[12]*z[16]*z[2]*z[3] + 8192*z[12]*z[16]*z[2]*z[4] + 16384*z[12]*z[16]*z[2]*z[5] + 32768*z[12]*z[16]*z[2]*z[6] + 65536*z[12]*z[16]*z[2]*z[7] + 131072*z[12]*z[16]*z[2]*z[8] - 261120*z[12]*z[16]*z[2] + 16384*z[12]*z[16]*z[3]*z[4] + 32768*z[12]*z[16]*z[3]*z[5] + 65536*z[12]*z[16]*z[3]*z[6] + 131072*z[12]*z[16]*z[3]*z[7] + 262144*z[12]*z[16]*z[3]*z[8] - 522240*z[12]*z[16]*z[3] + 65536*z[12]*z[16]*z[4]*z[5] + 131072*z[12]*z[16]*z[4]*z[6] + 262144*z[12]*z[16]*z[4]*z[7] + 524288*z[12]*z[16]*z[4]*z[8] - 1044480*z[12]*z[16]*z[4] + 262144*z[12]*z[16]*z[5]*z[6] + 524288*z[12]*z[16]*z[5]*z[7] + 1048576*z[12]*z[16]*z[5]*z[8] - 2088960*z[12]*z[16]*z[5] + 1048576*z[12]*z[16]*z[6]*z[7] + 2097152*z[12]*z[16]*z[6]*z[8] - 4177920*z[12]*z[16]*z[6] + 4194304*z[12]*z[16]*z[7]*z[8] - 8355840*z[12]*z[16]*z[7] - 16711680*z[12]*z[16]*z[8] - 391680*z[12]*z[16]*z[9] + 80530432*z[12]*z[16] + 32*z[12]*z[2]*z[3]*z[9] - 8160*z[12]*z[2]*z[3] + 64*z[12]*z[2]*z[4]*z[9] - 16320*z[12]*z[2]*z[4] + 128*z[12]*z[2]*z[5]*z[9] - 32640*z[12]*z[2]*z[5] + 256*z[12]*z[2]*z[6]*z[9] - 65280*z[12]*z[2]*z[6] + 512*z[12]*z[2]*z[7]*z[9] - 130560*z[12]*z[2]*z[7] + 1024*z[12]*z[2]*z[8]*z[9] - 261120*z[12]*z[2]*z[8] - 2040*z[12]*z[2]*z[9] + 520200*z[12]*z[2] + 128*z[12]*z[3]*z[4]*z[9] - 32640*z[12]*z[3]*z[4] + 256*z[12]*z[3]*z[5]*z[9] - 65280*z[12]*z[3]*z[5] + 512*z[12]*z[3]*z[6]*z[9] - 130560*z[12]*z[3]*z[6] + 1024*z[12]*z[3]*z[7]*z[9] - 261120*z[12]*z[3]*z[7] + 2048*z[12]*z[3]*z[8]*z[9] - 522240*z[12]*z[3]*z[8] - 4080*z[12]*z[3]*z[9] + 1040400*z[12]*z[3] + 512*z[12]*z[4]*z[5]*z[9] - 130560*z[12]*z[4]*z[5] + 1024*z[12]*z[4]*z[6]*z[9] - 261120*z[12]*z[4]*z[6] + 2048*z[12]*z[4]*z[7]*z[9] - 522240*z[12]*z[4]*z[7] + 4096*z[12]*z[4]*z[8]*z[9] - 1044480*z[12]*z[4]*z[8] - 8160*z[12]*z[4]*z[9] + 2080800*z[12]*z[4] + 2048*z[12]*z[5]*z[6]*z[9] - 522240*z[12]*z[5]*z[6] + 4096*z[12]*z[5]*z[7]*z[9] - 1044480*z[12]*z[5]*z[7] + 8192*z[12]*z[5]*z[8]*z[9] - 2088960*z[12]*z[5]*z[8] - 16320*z[12]*z[5]*z[9] + 4161600*z[12]*z[5] + 8192*z[12]*z[6]*z[7]*z[9] - 2088960*z[12]*z[6]*z[7] + 16384*z[12]*z[6]*z[8]*z[9] - 4177920*z[12]*z[6]*z[8] - 32640*z[12]*z[6]*z[9] + 8323200*z[12]*z[6] + 32768*z[12]*z[7]*z[8]*z[9] - 8355840*z[12]*z[7]*z[8] - 65280*z[12]*z[7]*z[9] + 16646400*z[12]*z[7] - 130560*z[12]*z[8]*z[9] + 33292800*z[12]*z[8] + 694676*z[12]*z[9] - 110817900*z[12] + 6291456*z[13]*z[14]*z[15]*z[16] + 49152*z[13]*z[14]*z[15]*z[9] - 12533760*z[13]*z[14]*z[15] + 98304*z[13]*z[14]*z[16]*z[9] - 25067520*z[13]*z[14]*z[16] + 2048*z[13]*z[14]*z[2]*z[3] + 4096*z[13]*z[14]*z[2]*z[4] + 8192*z[13]*z[14]*z[2]*z[5] + 16384*z[13]*z[14]*z[2]*z[6] + 32768*z[13]*z[14]*z[2]*z[7] + 65536*z[13]*z[14]*z[2]*z[8] - 130560*z[13]*z[14]*z[2] + 8192*z[13]*z[14]*z[3]*z[4] + 16384*z[13]*z[14]*z[3]*z[5] + 32768*z[13]*z[14]*z[3]*z[6] + 65536*z[13]*z[14]*z[3]*z[7] + 131072*z[13]*z[14]*z[3]*z[8] - 261120*z[13]*z[14]*z[3] + 32768*z[13]*z[14]*z[4]*z[5] + 65536*z[13]*z[14]*z[4]*z[6] + 131072*z[13]*z[14]*z[4]*z[7] + 262144*z[13]*z[14]*z[4]*z[8] - 522240*z[13]*z[14]*z[4] + 131072*z[13]*z[14]*z[5]*z[6] + 262144*z[13]*z[14]*z[5]*z[7] + 524288*z[13]*z[14]*z[5]*z[8] - 1044480*z[13]*z[14]*z[5] + 524288*z[13]*z[14]*z[6]*z[7] + 1048576*z[13]*z[14]*z[6]*z[8] - 2088960*z[13]*z[14]*z[6] + 2097152*z[13]*z[14]*z[7]*z[8] - 4177920*z[13]*z[14]*z[7] - 8355840*z[13]*z[14]*z[8] - 195840*z[13]*z[14]*z[9] + 44148224*z[13]*z[14] + 196608*z[13]*z[15]*z[16]*z[9] - 50135040*z[13]*z[15]*z[16] + 4096*z[13]*z[15]*z[2]*z[3] + 8192*z[13]*z[15]*z[2]*z[4] + 16384*z[13]*z[15]*z[2]*z[5] + 32768*z[13]*z[15]*z[2]*z[6] + 65536*z[13]*z[15]*z[2]*z[7] + 131072*z[13]*z[15]*z[2]*z[8] - 261120*z[13]*z[15]*z[2] + 16384*z[13]*z[15]*z[3]*z[4] + 32768*z[13]*z[15]*z[3]*z[5] + 65536*z[13]*z[15]*z[3]*z[6] + 131072*z[13]*z[15]*z[3]*z[7] + 262144*z[13]*z[15]*z[3]*z[8] - 522240*z[13]*z[15]*z[3] + 65536*z[13]*z[15]*z[4]*z[5] + 131072*z[13]*z[15]*z[4]*z[6] + 262144*z[13]*z[15]*z[4]*z[7] + 524288*z[13]*z[15]*z[4]*z[8] - 1044480*z[13]*z[15]*z[4] + 262144*z[13]*z[15]*z[5]*z[6] + 524288*z[13]*z[15]*z[5]*z[7] + 1048576*z[13]*z[15]*z[5]*z[8] - 2088960*z[13]*z[15]*z[5] + 1048576*z[13]*z[15]*z[6]*z[7] + 2097152*z[13]*z[15]*z[6]*z[8] - 4177920*z[13]*z[15]*z[6] + 4194304*z[13]*z[15]*z[7]*z[8] - 8355840*z[13]*z[15]*z[7] - 16711680*z[13]*z[15]*z[8] - 391680*z[13]*z[15]*z[9] + 86723584*z[13]*z[15] + 8192*z[13]*z[16]*z[2]*z[3] + 16384*z[13]*z[16]*z[2]*z[4] + 32768*z[13]*z[16]*z[2]*z[5] + 65536*z[13]*z[16]*z[2]*z[6] + 131072*z[13]*z[16]*z[2]*z[7] + 262144*z[13]*z[16]*z[2]*z[8] - 522240*z[13]*z[16]*z[2] + 32768*z[13]*z[16]*z[3]*z[4] + 65536*z[13]*z[16]*z[3]*z[5] + 131072*z[13]*z[16]*z[3]*z[6] + 262144*z[13]*z[16]*z[3]*z[7] + 524288*z[13]*z[16]*z[3]*z[8] - 1044480*z[13]*z[16]*z[3] + 131072*z[13]*z[16]*z[4]*z[5] + 262144*z[13]*z[16]*z[4]*z[6] + 524288*z[13]*z[16]*z[4]*z[7] + 1048576*z[13]*z[16]*z[4]*z[8] - 2088960*z[13]*z[16]*z[4] + 524288*z[13]*z[16]*z[5]*z[6] + 1048576*z[13]*z[16]*z[5]*z[7] + 2097152*z[13]*z[16]*z[5]*z[8] - 4177920*z[13]*z[16]*z[5] + 2097152*z[13]*z[16]*z[6]*z[7] + 4194304*z[13]*z[16]*z[6]*z[8] - 8355840*z[13]*z[16]*z[6] + 8388608*z[13]*z[16]*z[7]*z[8] - 16711680*z[13]*z[16]*z[7] - 33423360*z[13]*z[16]*z[8] - 783360*z[13]*z[16]*z[9] + 160864256*z[13]*z[16] + 64*z[13]*z[2]*z[3]*z[9] - 16320*z[13]*z[2]*z[3] + 128*z[13]*z[2]*z[4]*z[9] - 32640*z[13]*z[2]*z[4] + 256*z[13]*z[2]*z[5]*z[9] - 65280*z[13]*z[2]*z[5] + 512*z[13]*z[2]*z[6]*z[9] - 130560*z[13]*z[2]*z[6] + 1024*z[13]*z[2]*z[7]*z[9] - 261120*z[13]*z[2]*z[7] + 2048*z[13]*z[2]*z[8]*z[9] - 522240*z[13]*z[2]*z[8] - 4080*z[13]*z[2]*z[9] + 1040400*z[13]*z[2] + 256*z[13]*z[3]*z[4]*z[9] - 65280*z[13]*z[3]*z[4] + 512*z[13]*z[3]*z[5]*z[9] - 130560*z[13]*z[3]*z[5] + 1024*z[13]*z[3]*z[6]*z[9] - 261120*z[13]*z[3]*z[6] + 2048*z[13]*z[3]*z[7]*z[9] - 522240*z[13]*z[3]*z[7] + 4096*z[13]*z[3]*z[8]*z[9] - 1044480*z[13]*z[3]*z[8] - 8160*z[13]*z[3]*z[9] + 2080800*z[13]*z[3] + 1024*z[13]*z[4]*z[5]*z[9] - 261120*z[13]*z[4]*z[5] + 2048*z[13]*z[4]*z[6]*z[9] - 522240*z[13]*z[4]*z[6] + 4096*z[13]*z[4]*z[7]*z[9] - 1044480*z[13]*z[4]*z[7] + 8192*z[13]*z[4]*z[8]*z[9] - 2088960*z[13]*z[4]*z[8] - 16320*z[13]*z[4]*z[9] + 4161600*z[13]*z[4] + 4096*z[13]*z[5]*z[6]*z[9] - 1044480*z[13]*z[5]*z[6] + 8192*z[13]*z[5]*z[7]*z[9] - 2088960*z[13]*z[5]*z[7] + 16384*z[13]*z[5]*z[8]*z[9] - 4177920*z[13]*z[5]*z[8] - 32640*z[13]*z[5]*z[9] + 8323200*z[13]*z[5] + 16384*z[13]*z[6]*z[7]*z[9] - 4177920*z[13]*z[6]*z[7] + 32768*z[13]*z[6]*z[8]*z[9] - 8355840*z[13]*z[6]*z[8] - 65280*z[13]*z[6]*z[9] + 16646400*z[13]*z[6] + 65536*z[13]*z[7]*z[8]*z[9] - 16711680*z[13]*z[7]*z[8] - 130560*z[13]*z[7]*z[9] + 33292800*z[13]*z[7] - 261120*z[13]*z[8]*z[9] + 66585600*z[13]*z[8] + 1387816*z[13]*z[9] - 221244120*z[13] + 393216*z[14]*z[15]*z[16]*z[9] - 100270080*z[14]*z[15]*z[16] + 8192*z[14]*z[15]*z[2]*z[3] + 16384*z[14]*z[15]*z[2]*z[4] + 32768*z[14]*z[15]*z[2]*z[5] + 65536*z[14]*z[15]*z[2]*z[6] + 131072*z[14]*z[15]*z[2]*z[7] + 262144*z[14]*z[15]*z[2]*z[8] - 522240*z[14]*z[15]*z[2] + 32768*z[14]*z[15]*z[3]*z[4] + 65536*z[14]*z[15]*z[3]*z[5] + 131072*z[14]*z[15]*z[3]*z[6] + 262144*z[14]*z[15]*z[3]*z[7] + 524288*z[14]*z[15]*z[3]*z[8] - 1044480*z[14]*z[15]*z[3] + 131072*z[14]*z[15]*z[4]*z[5] + 262144*z[14]*z[15]*z[4]*z[6] + 524288*z[14]*z[15]*z[4]*z[7] + 1048576*z[14]*z[15]*z[4]*z[8] - 2088960*z[14]*z[15]*z[4] + 524288*z[14]*z[15]*z[5]*z[6] + 1048576*z[14]*z[15]*z[5]*z[7] + 2097152*z[14]*z[15]*z[5]*z[8] - 4177920*z[14]*z[15]*z[5] + 2097152*z[14]*z[15]*z[6]*z[7] + 4194304*z[14]*z[15]*z[6]*z[8] - 8355840*z[14]*z[15]*z[6] + 8388608*z[14]*z[15]*z[7]*z[8] - 16711680*z[14]*z[15]*z[7] - 33423360*z[14]*z[15]*z[8] - 783360*z[14]*z[15]*z[9] + 172660736*z[14]*z[15] + 16384*z[14]*z[16]*z[2]*z[3] + 32768*z[14]*z[16]*z[2]*z[4] + 65536*z[14]*z[16]*z[2]*z[5] + 131072*z[14]*z[16]*z[2]*z[6] + 262144*z[14]*z[16]*z[2]*z[7] + 524288*z[14]*z[16]*z[2]*z[8] - 1044480*z[14]*z[16]*z[2] + 65536*z[14]*z[16]*z[3]*z[4] + 131072*z[14]*z[16]*z[3]*z[5] + 262144*z[14]*z[16]*z[3]*z[6] + 524288*z[14]*z[16]*z[3]*z[7] + 1048576*z[14]*z[16]*z[3]*z[8] - 2088960*z[14]*z[16]*z[3] + 262144*z[14]*z[16]*z[4]*z[5] + 524288*z[14]*z[16]*z[4]*z[6] + 1048576*z[14]*z[16]*z[4]*z[7] + 2097152*z[14]*z[16]*z[4]*z[8] - 4177920*z[14]*z[16]*z[4] + 1048576*z[14]*z[16]*z[5]*z[6] + 2097152*z[14]*z[16]*z[5]*z[7] + 4194304*z[14]*z[16]*z[5]*z[8] - 8355840*z[14]*z[16]*z[5] + 4194304*z[14]*z[16]*z[6]*z[7] + 8388608*z[14]*z[16]*z[6]*z[8] - 16711680*z[14]*z[16]*z[6] + 16777216*z[14]*z[16]*z[7]*z[8] - 33423360*z[14]*z[16]*z[7] - 66846720*z[14]*z[16]*z[8] - 1566720*z[14]*z[16]*z[9] + 320155648*z[14]*z[16] + 128*z[14]*z[2]*z[3]*z[9] - 32640*z[14]*z[2]*z[3] + 256*z[14]*z[2]*z[4]*z[9] - 65280*z[14]*z[2]*z[4] + 512*z[14]*z[2]*z[5]*z[9] - 130560*z[14]*z[2]*z[5] + 1024*z[14]*z[2]*z[6]*z[9] - 261120*z[14]*z[2]*z[6] + 2048*z[14]*z[2]*z[7]*z[9] - 522240*z[14]*z[2]*z[7] + 4096*z[14]*z[2]*z[8]*z[9] - 1044480*z[14]*z[2]*z[8] - 8160*z[14]*z[2]*z[9] + 2080800*z[14]*z[2] + 512*z[14]*z[3]*z[4]*z[9] - 130560*z[14]*z[3]*z[4] + 1024*z[14]*z[3]*z[5]*z[9] - 261120*z[14]*z[3]*z[5] + 2048*z[14]*z[3]*z[6]*z[9] - 522240*z[14]*z[3]*z[6] + 4096*z[14]*z[3]*z[7]*z[9] - 1044480*z[14]*z[3]*z[7] + 8192*z[14]*z[3]*z[8]*z[9] - 2088960*z[14]*z[3]*z[8] - 16320*z[14]*z[3]*z[9] + 4161600*z[14]*z[3] + 2048*z[14]*z[4]*z[5]*z[9] - 522240*z[14]*z[4]*z[5] + 4096*z[14]*z[4]*z[6]*z[9] - 1044480*z[14]*z[4]*z[6] + 8192*z[14]*z[4]*z[7]*z[9] - 2088960*z[14]*z[4]*z[7] + 16384*z[14]*z[4]*z[8]*z[9] - 4177920*z[14]*z[4]*z[8] - 32640*z[14]*z[4]*z[9] + 8323200*z[14]*z[4] + 8192*z[14]*z[5]*z[6]*z[9] - 2088960*z[14]*z[5]*z[6] + 16384*z[14]*z[5]*z[7]*z[9] - 4177920*z[14]*z[5]*z[7] + 32768*z[14]*z[5]*z[8]*z[9] - 8355840*z[14]*z[5]*z[8] - 65280*z[14]*z[5]*z[9] + 16646400*z[14]*z[5] + 32768*z[14]*z[6]*z[7]*z[9] - 8355840*z[14]*z[6]*z[7] + 65536*z[14]*z[6]*z[8]*z[9] - 16711680*z[14]*z[6]*z[8] - 130560*z[14]*z[6]*z[9] + 33292800*z[14]*z[6] + 131072*z[14]*z[7]*z[8]*z[9] - 33423360*z[14]*z[7]*z[8] - 261120*z[14]*z[7]*z[9] + 66585600*z[14]*z[7] - 522240*z[14]*z[8]*z[9] + 133171200*z[14]*z[8] + 2763344*z[14]*z[9] - 439354800*z[14] + 32768*z[15]*z[16]*z[2]*z[3] + 65536*z[15]*z[16]*z[2]*z[4] + 131072*z[15]*z[16]*z[2]*z[5] + 262144*z[15]*z[16]*z[2]*z[6] + 524288*z[15]*z[16]*z[2]*z[7] + 1048576*z[15]*z[16]*z[2]*z[8] - 2088960*z[15]*z[16]*z[2] + 131072*z[15]*z[16]*z[3]*z[4] + 262144*z[15]*z[16]*z[3]*z[5] + 524288*z[15]*z[16]*z[3]*z[6] + 1048576*z[15]*z[16]*z[3]*z[7] + 2097152*z[15]*z[16]*z[3]*z[8] - 4177920*z[15]*z[16]*z[3] + 524288*z[15]*z[16]*z[4]*z[5] + 1048576*z[15]*z[16]*z[4]*z[6] + 2097152*z[15]*z[16]*z[4]*z[7] + 4194304*z[15]*z[16]*z[4]*z[8] - 8355840*z[15]*z[16]*z[4] + 2097152*z[15]*z[16]*z[5]*z[6] + 4194304*z[15]*z[16]*z[5]*z[7] + 8388608*z[15]*z[16]*z[5]*z[8] - 16711680*z[15]*z[16]*z[5] + 8388608*z[15]*z[16]*z[6]*z[7] + 16777216*z[15]*z[16]*z[6]*z[8] - 33423360*z[15]*z[16]*z[6] + 33554432*z[15]*z[16]*z[7]*z[8] - 66846720*z[15]*z[16]*z[7] - 133693440*z[15]*z[16]*z[8] - 3133440*z[15]*z[16]*z[9] + 627728384*z[15]*z[16] + 256*z[15]*z[2]*z[3]*z[9] - 65280*z[15]*z[2]*z[3] + 512*z[15]*z[2]*z[4]*z[9] - 130560*z[15]*z[2]*z[4] + 1024*z[15]*z[2]*z[5]*z[9] - 261120*z[15]*z[2]*z[5] + 2048*z[15]*z[2]*z[6]*z[9] - 522240*z[15]*z[2]*z[6] + 4096*z[15]*z[2]*z[7]*z[9] - 1044480*z[15]*z[2]*z[7] + 8192*z[15]*z[2]*z[8]*z[9] - 2088960*z[15]*z[2]*z[8] - 16320*z[15]*z[2]*z[9] + 4161600*z[15]*z[2] + 1024*z[15]*z[3]*z[4]*z[9] - 261120*z[15]*z[3]*z[4] + 2048*z[15]*z[3]*z[5]*z[9] - 522240*z[15]*z[3]*z[5] + 4096*z[15]*z[3]*z[6]*z[9] - 1044480*z[15]*z[3]*z[6] + 8192*z[15]*z[3]*z[7]*z[9] - 2088960*z[15]*z[3]*z[7] + 16384*z[15]*z[3]*z[8]*z[9] - 4177920*z[15]*z[3]*z[8] - 32640*z[15]*z[3]*z[9] + 8323200*z[15]*z[3] + 4096*z[15]*z[4]*z[5]*z[9] - 1044480*z[15]*z[4]*z[5] + 8192*z[15]*z[4]*z[6]*z[9] - 2088960*z[15]*z[4]*z[6] + 16384*z[15]*z[4]*z[7]*z[9] - 4177920*z[15]*z[4]*z[7] + 32768*z[15]*z[4]*z[8]*z[9] - 8355840*z[15]*z[4]*z[8] - 65280*z[15]*z[4]*z[9] + 16646400*z[15]*z[4] + 16384*z[15]*z[5]*z[6]*z[9] - 4177920*z[15]*z[5]*z[6] + 32768*z[15]*z[5]*z[7]*z[9] - 8355840*z[15]*z[5]*z[7] + 65536*z[15]*z[5]*z[8]*z[9] - 16711680*z[15]*z[5]*z[8] - 130560*z[15]*z[5]*z[9] + 33292800*z[15]*z[5] + 65536*z[15]*z[6]*z[7]*z[9] - 16711680*z[15]*z[6]*z[7] + 131072*z[15]*z[6]*z[8]*z[9] - 33423360*z[15]*z[6]*z[8] - 261120*z[15]*z[6]*z[9] + 66585600*z[15]*z[6] + 262144*z[15]*z[7]*z[8]*z[9] - 66846720*z[15]*z[7]*z[8] - 522240*z[15]*z[7]*z[9] + 133171200*z[15]*z[7] - 1044480*z[15]*z[8]*z[9] + 266342400*z[15]*z[8] + 5428384*z[15]*z[9] - 853642080*z[15] + 512*z[16]*z[2]*z[3]*z[9] - 130560*z[16]*z[2]*z[3] + 1024*z[16]*z[2]*z[4]*z[9] - 261120*z[16]*z[2]*z[4] + 2048*z[16]*z[2]*z[5]*z[9] - 522240*z[16]*z[2]*z[5] + 4096*z[16]*z[2]*z[6]*z[9] - 1044480*z[16]*z[2]*z[6] + 8192*z[16]*z[2]*z[7]*z[9] - 2088960*z[16]*z[2]*z[7] + 16384*z[16]*z[2]*z[8]*z[9] - 4177920*z[16]*z[2]*z[8] - 32640*z[16]*z[2]*z[9] + 8323200*z[16]*z[2] + 2048*z[16]*z[3]*z[4]*z[9] - 522240*z[16]*z[3]*z[4] + 4096*z[16]*z[3]*z[5]*z[9] - 1044480*z[16]*z[3]*z[5] + 8192*z[16]*z[3]*z[6]*z[9] - 2088960*z[16]*z[3]*z[6] + 16384*z[16]*z[3]*z[7]*z[9] - 4177920*z[16]*z[3]*z[7] + 32768*z[16]*z[3]*z[8]*z[9] - 8355840*z[16]*z[3]*z[8] - 65280*z[16]*z[3]*z[9] + 16646400*z[16]*z[3] + 8192*z[16]*z[4]*z[5]*z[9] - 2088960*z[16]*z[4]*z[5] + 16384*z[16]*z[4]*z[6]*z[9] - 4177920*z[16]*z[4]*z[6] + 32768*z[16]*z[4]*z[7]*z[9] - 8355840*z[16]*z[4]*z[7] + 65536*z[16]*z[4]*z[8]*z[9] - 16711680*z[16]*z[4]*z[8] - 130560*z[16]*z[4]*z[9] + 33292800*z[16]*z[4] + 32768*z[16]*z[5]*z[6]*z[9] - 8355840*z[16]*z[5]*z[6] + 65536*z[16]*z[5]*z[7]*z[9] - 16711680*z[16]*z[5]*z[7] + 131072*z[16]*z[5]*z[8]*z[9] - 33423360*z[16]*z[5]*z[8] - 261120*z[16]*z[5]*z[9] + 66585600*z[16]*z[5] + 131072*z[16]*z[6]*z[7]*z[9] - 33423360*z[16]*z[6]*z[7] + 262144*z[16]*z[6]*z[8]*z[9] - 66846720*z[16]*z[6]*z[8] - 522240*z[16]*z[6]*z[9] + 133171200*z[16]*z[6] + 524288*z[16]*z[7]*z[8]*z[9] - 133693440*z[16]*z[7]*z[8] - 1044480*z[16]*z[7]*z[9] + 266342400*z[16]*z[7] - 2088960*z[16]*z[8]*z[9] + 532684800*z[16]*z[8] + 10070336*z[16]*z[9] - 1506744000*z[16] + 1536*z[2]*z[3]*z[4]*z[5] + 3072*z[2]*z[3]*z[4]*z[6] + 6144*z[2]*z[3]*z[4]*z[7] + 12288*z[2]*z[3]*z[4]*z[8] - 24480*z[2]*z[3]*z[4] + 6144*z[2]*z[3]*z[5]*z[6] + 12288*z[2]*z[3]*z[5]*z[7] + 24576*z[2]*z[3]*z[5]*z[8] - 48960*z[2]*z[3]*z[5] + 24576*z[2]*z[3]*z[6]*z[7] + 49152*z[2]*z[3]*z[6]*z[8] - 97920*z[2]*z[3]*z[6] + 98304*z[2]*z[3]*z[7]*z[8] - 195840*z[2]*z[3]*z[7] - 391680*z[2]*z[3]*z[8] - 1020*z[2]*z[3]*z[9] + 694856*z[2]*z[3] + 12288*z[2]*z[4]*z[5]*z[6] + 24576*z[2]*z[4]*z[5]*z[7] + 49152*z[2]*z[4]*z[5]*z[8] - 97920*z[2]*z[4]*z[5] + 49152*z[2]*z[4]*z[6]*z[7] + 98304*z[2]*z[4]*z[6]*z[8] - 195840*z[2]*z[4]*z[6] + 196608*z[2]*z[4]*z[7]*z[8] - 391680*z[2]*z[4]*z[7] - 783360*z[2]*z[4]*z[8] - 2040*z[2]*z[4]*z[9] + 1389328*z[2]*z[4] + 98304*z[2]*z[5]*z[6]*z[7] + 196608*z[2]*z[5]*z[6]*z[8] - 391680*z[2]*z[5]*z[6] + 393216*z[2]*z[5]*z[7]*z[8] - 783360*z[2]*z[5]*z[7] - 1566720*z[2]*z[5]*z[8] - 4080*z[2]*z[5]*z[9] + 2775584*z[2]*z[5] + 786432*z[2]*z[6]*z[7]*z[8] - 1566720*z[2]*z[6]*z[7] - 3133440*z[2]*z[6]*z[8] - 8160*z[2]*z[6]*z[9] + 5526592*z[2]*z[6] - 6266880*z[2]*z[7]*z[8] - 16320*z[2]*z[7]*z[9] + 10856576*z[2]*z[7] - 32640*z[2]*z[8]*z[9] + 20140288*z[2]*z[8] + 65025*z[2]*z[9] - 27719775*z[2] + 24576*z[3]*z[4]*z[5]*z[6] + 49152*z[3]*z[4]*z[5]*z[7] + 98304*z[3]*z[4]*z[5]*z[8] - 195840*z[3]*z[4]*z[5] + 98304*z[3]*z[4]*z[6]*z[7] + 196608*z[3]*z[4]*z[6]*z[8] - 391680*z[3]*z[4]*z[6] + 393216*z[3]*z[4]*z[7]*z[8] - 783360*z[3]*z[4]*z[7] - 1566720*z[3]*z[4]*z[8] - 4080*z[3]*z[4]*z[9] + 2778464*z[3]*z[4] + 196608*z[3]*z[5]*z[6]*z[7] + 393216*z[3]*z[5]*z[6]*z[8] - 783360*z[3]*z[5]*z[6] + 786432*z[3]*z[5]*z[7]*z[8] - 1566720*z[3]*z[5]*z[7] - 3133440*z[3]*z[5]*z[8] - 8160*z[3]*z[5]*z[9] + 5550784*z[3]*z[5] + 1572864*z[3]*z[6]*z[7]*z[8] - 3133440*z[3]*z[6]*z[7] - 6266880*z[3]*z[6]*z[8] - 16320*z[3]*z[6]*z[9] + 11052416*z[3]*z[6] - 12533760*z[3]*z[7]*z[8] - 32640*z[3]*z[7]*z[9] + 21711616*z[3]*z[7] - 65280*z[3]*z[8]*z[9] + 40277504*z[3]*z[8] + 130050*z[3]*z[9] - 55433430*z[3] + 393216*z[4]*z[5]*z[6]*z[7] + 786432*z[4]*z[5]*z[6]*z[8] - 1566720*z[4]*z[5]*z[6] + 1572864*z[4]*z[5]*z[7]*z[8] - 3133440*z[4]*z[5]*z[7] - 6266880*z[4]*z[5]*z[8] - 16320*z[4]*z[5]*z[9] + 11098496*z[4]*z[5] + 3145728*z[4]*z[6]*z[7]*z[8] - 6266880*z[4]*z[6]*z[7] - 12533760*z[4]*z[6]*z[8] - 32640*z[4]*z[6]*z[9] + 22098688*z[4]*z[6] - 25067520*z[4]*z[7]*z[8] - 65280*z[4]*z[7]*z[9] + 43410944*z[4]*z[7] - 130560*z[4]*z[8]*z[9] + 80530432*z[4]*z[8] + 260100*z[4]*z[9] - 110817900*z[4] + 6291456*z[5]*z[6]*z[7]*z[8] - 12533760*z[5]*z[6]*z[7] - 25067520*z[5]*z[6]*z[8] - 65280*z[5]*z[6]*z[9] + 44148224*z[5]*z[6] - 50135040*z[5]*z[7]*z[8] - 130560*z[5]*z[7]*z[9] + 86723584*z[5]*z[7] - 261120*z[5]*z[8]*z[9] + 160864256*z[5]*z[8] + 520200*z[5]*z[9] - 221244120*z[5] - 100270080*z[6]*z[7]*z[8] - 261120*z[6]*z[7]*z[9] + 172660736*z[6]*z[7] - 522240*z[6]*z[8]*z[9] + 320155648*z[6]*z[8] + 1040400*z[6]*z[9] - 439354800*z[6] - 1044480*z[7]*z[8]*z[9] + 627728384*z[7]*z[8] + 2080800*z[7]*z[9] - 853642080*z[7] + 4161600*z[8]*z[9] - 1506744000*z[8] - 13860270*z[9] + 5288584809/2)

In [310]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 16  # 
p = 1  # QAOA depth

hamiltonian_less_255_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_equation_less_255_2))
final_circuit_less_255_2, result_less_255_2 = qaoa(num_qubits, hamiltonian_less_255_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [-1.10875547 -2.59077789]
Minimum expectation value: -842032613.8320312


In [311]:
top_solutions_less_255_2 = find_best_bitstrings(final_circuit_less_255_2, hamiltonian_less_255_2)

Top 5 bitstrings:
Bitstring: 0000010100000001, Cost: -5288584280.0000, Count: 1
Bitstring: 0000101000000001, Cost: -5288575204.0000, Count: 1
Bitstring: 0000010100001001, Cost: -5288574200.0000, Count: 1
Bitstring: 0000100000000111, Cost: -5288572708.0000, Count: 1
Bitstring: 0000101100000001, Cost: -5288570648.0000, Count: 1


In [312]:
bitstring_to_pm1(top_solutions_less_255_2, evaluate_hamiltonian_less_255_2)

["Bitstring: ('0000010100000001', -5288584280.0, 1), Evaluated cost: 529.0",
 "Bitstring: ('0000010100000001', -5288584280.0, 1), Evaluated cost: 9604.0",
 "Bitstring: ('0000010100000001', -5288584280.0, 1), Evaluated cost: 10609.0",
 "Bitstring: ('0000010100000001', -5288584280.0, 1), Evaluated cost: 12100.0",
 "Bitstring: ('0000010100000001', -5288584280.0, 1), Evaluated cost: 14161.0"]

# Erdös-Strauss Diophantine equation $4xyz = n(xy+xz+yz)$ Case $n=4$

In [337]:
def D_Erdös_Straus(x,y,z):
    return(4*x*y*z-4*(y*z+x*z+x*y))

In [343]:
paso1_D_Erdös_Straus_less_15_1 = substitute_with_global_binary_symbols(D_Erdös_Straus(x[1],x[2],x[3])**2, 4, base_name="b")
paso2_D_Erdös_Straus_less_15_1 = remove_variable_exponents(paso1_D_Erdös_Straus_less_15_1)
paso3_D_Erdös_Straus_less_15_1 = substitute_with_spin_variables(paso2_D_Erdös_Straus_less_15_1)

In [358]:
def evaluate_hamiltonian_D_Erdös_Straus_less_15(z):
    return(64*z[1]*z[10]*z[11]*z[2]*z[5]*z[6] + 128*z[1]*z[10]*z[11]*z[2]*z[5]*z[7] + 256*z[1]*z[10]*z[11]*z[2]*z[5]*z[8] - 416*z[1]*z[10]*z[11]*z[2]*z[5] + 256*z[1]*z[10]*z[11]*z[2]*z[6]*z[7] + 512*z[1]*z[10]*z[11]*z[2]*z[6]*z[8] - 832*z[1]*z[10]*z[11]*z[2]*z[6] + 1024*z[1]*z[10]*z[11]*z[2]*z[7]*z[8] - 1664*z[1]*z[10]*z[11]*z[2]*z[7] - 3328*z[1]*z[10]*z[11]*z[2]*z[8] + 4064*z[1]*z[10]*z[11]*z[2] + 128*z[1]*z[10]*z[11]*z[3]*z[5]*z[6] + 256*z[1]*z[10]*z[11]*z[3]*z[5]*z[7] + 512*z[1]*z[10]*z[11]*z[3]*z[5]*z[8] - 832*z[1]*z[10]*z[11]*z[3]*z[5] + 512*z[1]*z[10]*z[11]*z[3]*z[6]*z[7] + 1024*z[1]*z[10]*z[11]*z[3]*z[6]*z[8] - 1664*z[1]*z[10]*z[11]*z[3]*z[6] + 2048*z[1]*z[10]*z[11]*z[3]*z[7]*z[8] - 3328*z[1]*z[10]*z[11]*z[3]*z[7] - 6656*z[1]*z[10]*z[11]*z[3]*z[8] + 8128*z[1]*z[10]*z[11]*z[3] + 256*z[1]*z[10]*z[11]*z[4]*z[5]*z[6] + 512*z[1]*z[10]*z[11]*z[4]*z[5]*z[7] + 1024*z[1]*z[10]*z[11]*z[4]*z[5]*z[8] - 1664*z[1]*z[10]*z[11]*z[4]*z[5] + 1024*z[1]*z[10]*z[11]*z[4]*z[6]*z[7] + 2048*z[1]*z[10]*z[11]*z[4]*z[6]*z[8] - 3328*z[1]*z[10]*z[11]*z[4]*z[6] + 4096*z[1]*z[10]*z[11]*z[4]*z[7]*z[8] - 6656*z[1]*z[10]*z[11]*z[4]*z[7] - 13312*z[1]*z[10]*z[11]*z[4]*z[8] + 16256*z[1]*z[10]*z[11]*z[4] - 416*z[1]*z[10]*z[11]*z[5]*z[6] - 832*z[1]*z[10]*z[11]*z[5]*z[7] - 1664*z[1]*z[10]*z[11]*z[5]*z[8] + 2672*z[1]*z[10]*z[11]*z[5] - 1664*z[1]*z[10]*z[11]*z[6]*z[7] - 3328*z[1]*z[10]*z[11]*z[6]*z[8] + 5344*z[1]*z[10]*z[11]*z[6] - 6656*z[1]*z[10]*z[11]*z[7]*z[8] + 10688*z[1]*z[10]*z[11]*z[7] + 21376*z[1]*z[10]*z[11]*z[8] - 26000*z[1]*z[10]*z[11] + 128*z[1]*z[10]*z[12]*z[2]*z[5]*z[6] + 256*z[1]*z[10]*z[12]*z[2]*z[5]*z[7] + 512*z[1]*z[10]*z[12]*z[2]*z[5]*z[8] - 832*z[1]*z[10]*z[12]*z[2]*z[5] + 512*z[1]*z[10]*z[12]*z[2]*z[6]*z[7] + 1024*z[1]*z[10]*z[12]*z[2]*z[6]*z[8] - 1664*z[1]*z[10]*z[12]*z[2]*z[6] + 2048*z[1]*z[10]*z[12]*z[2]*z[7]*z[8] - 3328*z[1]*z[10]*z[12]*z[2]*z[7] - 6656*z[1]*z[10]*z[12]*z[2]*z[8] + 8128*z[1]*z[10]*z[12]*z[2] + 256*z[1]*z[10]*z[12]*z[3]*z[5]*z[6] + 512*z[1]*z[10]*z[12]*z[3]*z[5]*z[7] + 1024*z[1]*z[10]*z[12]*z[3]*z[5]*z[8] - 1664*z[1]*z[10]*z[12]*z[3]*z[5] + 1024*z[1]*z[10]*z[12]*z[3]*z[6]*z[7] + 2048*z[1]*z[10]*z[12]*z[3]*z[6]*z[8] - 3328*z[1]*z[10]*z[12]*z[3]*z[6] + 4096*z[1]*z[10]*z[12]*z[3]*z[7]*z[8] - 6656*z[1]*z[10]*z[12]*z[3]*z[7] - 13312*z[1]*z[10]*z[12]*z[3]*z[8] + 16256*z[1]*z[10]*z[12]*z[3] + 512*z[1]*z[10]*z[12]*z[4]*z[5]*z[6] + 1024*z[1]*z[10]*z[12]*z[4]*z[5]*z[7] + 2048*z[1]*z[10]*z[12]*z[4]*z[5]*z[8] - 3328*z[1]*z[10]*z[12]*z[4]*z[5] + 2048*z[1]*z[10]*z[12]*z[4]*z[6]*z[7] + 4096*z[1]*z[10]*z[12]*z[4]*z[6]*z[8] - 6656*z[1]*z[10]*z[12]*z[4]*z[6] + 8192*z[1]*z[10]*z[12]*z[4]*z[7]*z[8] - 13312*z[1]*z[10]*z[12]*z[4]*z[7] - 26624*z[1]*z[10]*z[12]*z[4]*z[8] + 32512*z[1]*z[10]*z[12]*z[4] - 832*z[1]*z[10]*z[12]*z[5]*z[6] - 1664*z[1]*z[10]*z[12]*z[5]*z[7] - 3328*z[1]*z[10]*z[12]*z[5]*z[8] + 5344*z[1]*z[10]*z[12]*z[5] - 3328*z[1]*z[10]*z[12]*z[6]*z[7] - 6656*z[1]*z[10]*z[12]*z[6]*z[8] + 10688*z[1]*z[10]*z[12]*z[6] - 13312*z[1]*z[10]*z[12]*z[7]*z[8] + 21376*z[1]*z[10]*z[12]*z[7] + 42752*z[1]*z[10]*z[12]*z[8] - 52000*z[1]*z[10]*z[12] + 16*z[1]*z[10]*z[2]*z[5]*z[6]*z[9] - 208*z[1]*z[10]*z[2]*z[5]*z[6] + 32*z[1]*z[10]*z[2]*z[5]*z[7]*z[9] - 416*z[1]*z[10]*z[2]*z[5]*z[7] + 64*z[1]*z[10]*z[2]*z[5]*z[8]*z[9] - 832*z[1]*z[10]*z[2]*z[5]*z[8] - 104*z[1]*z[10]*z[2]*z[5]*z[9] + 1336*z[1]*z[10]*z[2]*z[5] + 64*z[1]*z[10]*z[2]*z[6]*z[7]*z[9] - 832*z[1]*z[10]*z[2]*z[6]*z[7] + 128*z[1]*z[10]*z[2]*z[6]*z[8]*z[9] - 1664*z[1]*z[10]*z[2]*z[6]*z[8] - 208*z[1]*z[10]*z[2]*z[6]*z[9] + 2672*z[1]*z[10]*z[2]*z[6] + 256*z[1]*z[10]*z[2]*z[7]*z[8]*z[9] - 3328*z[1]*z[10]*z[2]*z[7]*z[8] - 416*z[1]*z[10]*z[2]*z[7]*z[9] + 5344*z[1]*z[10]*z[2]*z[7] - 832*z[1]*z[10]*z[2]*z[8]*z[9] + 10688*z[1]*z[10]*z[2]*z[8] + 1016*z[1]*z[10]*z[2]*z[9] - 13000*z[1]*z[10]*z[2] + 32*z[1]*z[10]*z[3]*z[5]*z[6]*z[9] - 416*z[1]*z[10]*z[3]*z[5]*z[6] + 64*z[1]*z[10]*z[3]*z[5]*z[7]*z[9] - 832*z[1]*z[10]*z[3]*z[5]*z[7] + 128*z[1]*z[10]*z[3]*z[5]*z[8]*z[9] - 1664*z[1]*z[10]*z[3]*z[5]*z[8] - 208*z[1]*z[10]*z[3]*z[5]*z[9] + 2672*z[1]*z[10]*z[3]*z[5] + 128*z[1]*z[10]*z[3]*z[6]*z[7]*z[9] - 1664*z[1]*z[10]*z[3]*z[6]*z[7] + 256*z[1]*z[10]*z[3]*z[6]*z[8]*z[9] - 3328*z[1]*z[10]*z[3]*z[6]*z[8] - 416*z[1]*z[10]*z[3]*z[6]*z[9] + 5344*z[1]*z[10]*z[3]*z[6] + 512*z[1]*z[10]*z[3]*z[7]*z[8]*z[9] - 6656*z[1]*z[10]*z[3]*z[7]*z[8] - 832*z[1]*z[10]*z[3]*z[7]*z[9] + 10688*z[1]*z[10]*z[3]*z[7] - 1664*z[1]*z[10]*z[3]*z[8]*z[9] + 21376*z[1]*z[10]*z[3]*z[8] + 2032*z[1]*z[10]*z[3]*z[9] - 26000*z[1]*z[10]*z[3] + 64*z[1]*z[10]*z[4]*z[5]*z[6]*z[9] - 832*z[1]*z[10]*z[4]*z[5]*z[6] + 128*z[1]*z[10]*z[4]*z[5]*z[7]*z[9] - 1664*z[1]*z[10]*z[4]*z[5]*z[7] + 256*z[1]*z[10]*z[4]*z[5]*z[8]*z[9] - 3328*z[1]*z[10]*z[4]*z[5]*z[8] - 416*z[1]*z[10]*z[4]*z[5]*z[9] + 5344*z[1]*z[10]*z[4]*z[5] + 256*z[1]*z[10]*z[4]*z[6]*z[7]*z[9] - 3328*z[1]*z[10]*z[4]*z[6]*z[7] + 512*z[1]*z[10]*z[4]*z[6]*z[8]*z[9] - 6656*z[1]*z[10]*z[4]*z[6]*z[8] - 832*z[1]*z[10]*z[4]*z[6]*z[9] + 10688*z[1]*z[10]*z[4]*z[6] + 1024*z[1]*z[10]*z[4]*z[7]*z[8]*z[9] - 13312*z[1]*z[10]*z[4]*z[7]*z[8] - 1664*z[1]*z[10]*z[4]*z[7]*z[9] + 21376*z[1]*z[10]*z[4]*z[7] - 3328*z[1]*z[10]*z[4]*z[8]*z[9] + 42752*z[1]*z[10]*z[4]*z[8] + 4064*z[1]*z[10]*z[4]*z[9] - 52000*z[1]*z[10]*z[4] - 104*z[1]*z[10]*z[5]*z[6]*z[9] + 1336*z[1]*z[10]*z[5]*z[6] - 208*z[1]*z[10]*z[5]*z[7]*z[9] + 2672*z[1]*z[10]*z[5]*z[7] - 416*z[1]*z[10]*z[5]*z[8]*z[9] + 5344*z[1]*z[10]*z[5]*z[8] + 668*z[1]*z[10]*z[5]*z[9] - 8460*z[1]*z[10]*z[5] - 416*z[1]*z[10]*z[6]*z[7]*z[9] + 5344*z[1]*z[10]*z[6]*z[7] - 832*z[1]*z[10]*z[6]*z[8]*z[9] + 10688*z[1]*z[10]*z[6]*z[8] + 1336*z[1]*z[10]*z[6]*z[9] - 16920*z[1]*z[10]*z[6] - 1664*z[1]*z[10]*z[7]*z[8]*z[9] + 21376*z[1]*z[10]*z[7]*z[8] + 2672*z[1]*z[10]*z[7]*z[9] - 33840*z[1]*z[10]*z[7] + 5344*z[1]*z[10]*z[8]*z[9] - 67680*z[1]*z[10]*z[8] - 6500*z[1]*z[10]*z[9] + 81940*z[1]*z[10] + 256*z[1]*z[11]*z[12]*z[2]*z[5]*z[6] + 512*z[1]*z[11]*z[12]*z[2]*z[5]*z[7] + 1024*z[1]*z[11]*z[12]*z[2]*z[5]*z[8] - 1664*z[1]*z[11]*z[12]*z[2]*z[5] + 1024*z[1]*z[11]*z[12]*z[2]*z[6]*z[7] + 2048*z[1]*z[11]*z[12]*z[2]*z[6]*z[8] - 3328*z[1]*z[11]*z[12]*z[2]*z[6] + 4096*z[1]*z[11]*z[12]*z[2]*z[7]*z[8] - 6656*z[1]*z[11]*z[12]*z[2]*z[7] - 13312*z[1]*z[11]*z[12]*z[2]*z[8] + 16256*z[1]*z[11]*z[12]*z[2] + 512*z[1]*z[11]*z[12]*z[3]*z[5]*z[6] + 1024*z[1]*z[11]*z[12]*z[3]*z[5]*z[7] + 2048*z[1]*z[11]*z[12]*z[3]*z[5]*z[8] - 3328*z[1]*z[11]*z[12]*z[3]*z[5] + 2048*z[1]*z[11]*z[12]*z[3]*z[6]*z[7] + 4096*z[1]*z[11]*z[12]*z[3]*z[6]*z[8] - 6656*z[1]*z[11]*z[12]*z[3]*z[6] + 8192*z[1]*z[11]*z[12]*z[3]*z[7]*z[8] - 13312*z[1]*z[11]*z[12]*z[3]*z[7] - 26624*z[1]*z[11]*z[12]*z[3]*z[8] + 32512*z[1]*z[11]*z[12]*z[3] + 1024*z[1]*z[11]*z[12]*z[4]*z[5]*z[6] + 2048*z[1]*z[11]*z[12]*z[4]*z[5]*z[7] + 4096*z[1]*z[11]*z[12]*z[4]*z[5]*z[8] - 6656*z[1]*z[11]*z[12]*z[4]*z[5] + 4096*z[1]*z[11]*z[12]*z[4]*z[6]*z[7] + 8192*z[1]*z[11]*z[12]*z[4]*z[6]*z[8] - 13312*z[1]*z[11]*z[12]*z[4]*z[6] + 16384*z[1]*z[11]*z[12]*z[4]*z[7]*z[8] - 26624*z[1]*z[11]*z[12]*z[4]*z[7] - 53248*z[1]*z[11]*z[12]*z[4]*z[8] + 65024*z[1]*z[11]*z[12]*z[4] - 1664*z[1]*z[11]*z[12]*z[5]*z[6] - 3328*z[1]*z[11]*z[12]*z[5]*z[7] - 6656*z[1]*z[11]*z[12]*z[5]*z[8] + 10688*z[1]*z[11]*z[12]*z[5] - 6656*z[1]*z[11]*z[12]*z[6]*z[7] - 13312*z[1]*z[11]*z[12]*z[6]*z[8] + 21376*z[1]*z[11]*z[12]*z[6] - 26624*z[1]*z[11]*z[12]*z[7]*z[8] + 42752*z[1]*z[11]*z[12]*z[7] + 85504*z[1]*z[11]*z[12]*z[8] - 104000*z[1]*z[11]*z[12] + 32*z[1]*z[11]*z[2]*z[5]*z[6]*z[9] - 416*z[1]*z[11]*z[2]*z[5]*z[6] + 64*z[1]*z[11]*z[2]*z[5]*z[7]*z[9] - 832*z[1]*z[11]*z[2]*z[5]*z[7] + 128*z[1]*z[11]*z[2]*z[5]*z[8]*z[9] - 1664*z[1]*z[11]*z[2]*z[5]*z[8] - 208*z[1]*z[11]*z[2]*z[5]*z[9] + 2672*z[1]*z[11]*z[2]*z[5] + 128*z[1]*z[11]*z[2]*z[6]*z[7]*z[9] - 1664*z[1]*z[11]*z[2]*z[6]*z[7] + 256*z[1]*z[11]*z[2]*z[6]*z[8]*z[9] - 3328*z[1]*z[11]*z[2]*z[6]*z[8] - 416*z[1]*z[11]*z[2]*z[6]*z[9] + 5344*z[1]*z[11]*z[2]*z[6] + 512*z[1]*z[11]*z[2]*z[7]*z[8]*z[9] - 6656*z[1]*z[11]*z[2]*z[7]*z[8] - 832*z[1]*z[11]*z[2]*z[7]*z[9] + 10688*z[1]*z[11]*z[2]*z[7] - 1664*z[1]*z[11]*z[2]*z[8]*z[9] + 21376*z[1]*z[11]*z[2]*z[8] + 2032*z[1]*z[11]*z[2]*z[9] - 26000*z[1]*z[11]*z[2] + 64*z[1]*z[11]*z[3]*z[5]*z[6]*z[9] - 832*z[1]*z[11]*z[3]*z[5]*z[6] + 128*z[1]*z[11]*z[3]*z[5]*z[7]*z[9] - 1664*z[1]*z[11]*z[3]*z[5]*z[7] + 256*z[1]*z[11]*z[3]*z[5]*z[8]*z[9] - 3328*z[1]*z[11]*z[3]*z[5]*z[8] - 416*z[1]*z[11]*z[3]*z[5]*z[9] + 5344*z[1]*z[11]*z[3]*z[5] + 256*z[1]*z[11]*z[3]*z[6]*z[7]*z[9] - 3328*z[1]*z[11]*z[3]*z[6]*z[7] + 512*z[1]*z[11]*z[3]*z[6]*z[8]*z[9] - 6656*z[1]*z[11]*z[3]*z[6]*z[8] - 832*z[1]*z[11]*z[3]*z[6]*z[9] + 10688*z[1]*z[11]*z[3]*z[6] + 1024*z[1]*z[11]*z[3]*z[7]*z[8]*z[9] - 13312*z[1]*z[11]*z[3]*z[7]*z[8] - 1664*z[1]*z[11]*z[3]*z[7]*z[9] + 21376*z[1]*z[11]*z[3]*z[7] - 3328*z[1]*z[11]*z[3]*z[8]*z[9] + 42752*z[1]*z[11]*z[3]*z[8] + 4064*z[1]*z[11]*z[3]*z[9] - 52000*z[1]*z[11]*z[3] + 128*z[1]*z[11]*z[4]*z[5]*z[6]*z[9] - 1664*z[1]*z[11]*z[4]*z[5]*z[6] + 256*z[1]*z[11]*z[4]*z[5]*z[7]*z[9] - 3328*z[1]*z[11]*z[4]*z[5]*z[7] + 512*z[1]*z[11]*z[4]*z[5]*z[8]*z[9] - 6656*z[1]*z[11]*z[4]*z[5]*z[8] - 832*z[1]*z[11]*z[4]*z[5]*z[9] + 10688*z[1]*z[11]*z[4]*z[5] + 512*z[1]*z[11]*z[4]*z[6]*z[7]*z[9] - 6656*z[1]*z[11]*z[4]*z[6]*z[7] + 1024*z[1]*z[11]*z[4]*z[6]*z[8]*z[9] - 13312*z[1]*z[11]*z[4]*z[6]*z[8] - 1664*z[1]*z[11]*z[4]*z[6]*z[9] + 21376*z[1]*z[11]*z[4]*z[6] + 2048*z[1]*z[11]*z[4]*z[7]*z[8]*z[9] - 26624*z[1]*z[11]*z[4]*z[7]*z[8] - 3328*z[1]*z[11]*z[4]*z[7]*z[9] + 42752*z[1]*z[11]*z[4]*z[7] - 6656*z[1]*z[11]*z[4]*z[8]*z[9] + 85504*z[1]*z[11]*z[4]*z[8] + 8128*z[1]*z[11]*z[4]*z[9] - 104000*z[1]*z[11]*z[4] - 208*z[1]*z[11]*z[5]*z[6]*z[9] + 2672*z[1]*z[11]*z[5]*z[6] - 416*z[1]*z[11]*z[5]*z[7]*z[9] + 5344*z[1]*z[11]*z[5]*z[7] - 832*z[1]*z[11]*z[5]*z[8]*z[9] + 10688*z[1]*z[11]*z[5]*z[8] + 1336*z[1]*z[11]*z[5]*z[9] - 16920*z[1]*z[11]*z[5] - 832*z[1]*z[11]*z[6]*z[7]*z[9] + 10688*z[1]*z[11]*z[6]*z[7] - 1664*z[1]*z[11]*z[6]*z[8]*z[9] + 21376*z[1]*z[11]*z[6]*z[8] + 2672*z[1]*z[11]*z[6]*z[9] - 33840*z[1]*z[11]*z[6] - 3328*z[1]*z[11]*z[7]*z[8]*z[9] + 42752*z[1]*z[11]*z[7]*z[8] + 5344*z[1]*z[11]*z[7]*z[9] - 67680*z[1]*z[11]*z[7] + 10688*z[1]*z[11]*z[8]*z[9] - 135360*z[1]*z[11]*z[8] - 13000*z[1]*z[11]*z[9] + 163880*z[1]*z[11] + 64*z[1]*z[12]*z[2]*z[5]*z[6]*z[9] - 832*z[1]*z[12]*z[2]*z[5]*z[6] + 128*z[1]*z[12]*z[2]*z[5]*z[7]*z[9] - 1664*z[1]*z[12]*z[2]*z[5]*z[7] + 256*z[1]*z[12]*z[2]*z[5]*z[8]*z[9] - 3328*z[1]*z[12]*z[2]*z[5]*z[8] - 416*z[1]*z[12]*z[2]*z[5]*z[9] + 5344*z[1]*z[12]*z[2]*z[5] + 256*z[1]*z[12]*z[2]*z[6]*z[7]*z[9] - 3328*z[1]*z[12]*z[2]*z[6]*z[7] + 512*z[1]*z[12]*z[2]*z[6]*z[8]*z[9] - 6656*z[1]*z[12]*z[2]*z[6]*z[8] - 832*z[1]*z[12]*z[2]*z[6]*z[9] + 10688*z[1]*z[12]*z[2]*z[6] + 1024*z[1]*z[12]*z[2]*z[7]*z[8]*z[9] - 13312*z[1]*z[12]*z[2]*z[7]*z[8] - 1664*z[1]*z[12]*z[2]*z[7]*z[9] + 21376*z[1]*z[12]*z[2]*z[7] - 3328*z[1]*z[12]*z[2]*z[8]*z[9] + 42752*z[1]*z[12]*z[2]*z[8] + 4064*z[1]*z[12]*z[2]*z[9] - 52000*z[1]*z[12]*z[2] + 128*z[1]*z[12]*z[3]*z[5]*z[6]*z[9] - 1664*z[1]*z[12]*z[3]*z[5]*z[6] + 256*z[1]*z[12]*z[3]*z[5]*z[7]*z[9] - 3328*z[1]*z[12]*z[3]*z[5]*z[7] + 512*z[1]*z[12]*z[3]*z[5]*z[8]*z[9] - 6656*z[1]*z[12]*z[3]*z[5]*z[8] - 832*z[1]*z[12]*z[3]*z[5]*z[9] + 10688*z[1]*z[12]*z[3]*z[5] + 512*z[1]*z[12]*z[3]*z[6]*z[7]*z[9] - 6656*z[1]*z[12]*z[3]*z[6]*z[7] + 1024*z[1]*z[12]*z[3]*z[6]*z[8]*z[9] - 13312*z[1]*z[12]*z[3]*z[6]*z[8] - 1664*z[1]*z[12]*z[3]*z[6]*z[9] + 21376*z[1]*z[12]*z[3]*z[6] + 2048*z[1]*z[12]*z[3]*z[7]*z[8]*z[9] - 26624*z[1]*z[12]*z[3]*z[7]*z[8] - 3328*z[1]*z[12]*z[3]*z[7]*z[9] + 42752*z[1]*z[12]*z[3]*z[7] - 6656*z[1]*z[12]*z[3]*z[8]*z[9] + 85504*z[1]*z[12]*z[3]*z[8] + 8128*z[1]*z[12]*z[3]*z[9] - 104000*z[1]*z[12]*z[3] + 256*z[1]*z[12]*z[4]*z[5]*z[6]*z[9] - 3328*z[1]*z[12]*z[4]*z[5]*z[6] + 512*z[1]*z[12]*z[4]*z[5]*z[7]*z[9] - 6656*z[1]*z[12]*z[4]*z[5]*z[7] + 1024*z[1]*z[12]*z[4]*z[5]*z[8]*z[9] - 13312*z[1]*z[12]*z[4]*z[5]*z[8] - 1664*z[1]*z[12]*z[4]*z[5]*z[9] + 21376*z[1]*z[12]*z[4]*z[5] + 1024*z[1]*z[12]*z[4]*z[6]*z[7]*z[9] - 13312*z[1]*z[12]*z[4]*z[6]*z[7] + 2048*z[1]*z[12]*z[4]*z[6]*z[8]*z[9] - 26624*z[1]*z[12]*z[4]*z[6]*z[8] - 3328*z[1]*z[12]*z[4]*z[6]*z[9] + 42752*z[1]*z[12]*z[4]*z[6] + 4096*z[1]*z[12]*z[4]*z[7]*z[8]*z[9] - 53248*z[1]*z[12]*z[4]*z[7]*z[8] - 6656*z[1]*z[12]*z[4]*z[7]*z[9] + 85504*z[1]*z[12]*z[4]*z[7] - 13312*z[1]*z[12]*z[4]*z[8]*z[9] + 171008*z[1]*z[12]*z[4]*z[8] + 16256*z[1]*z[12]*z[4]*z[9] - 208000*z[1]*z[12]*z[4] - 416*z[1]*z[12]*z[5]*z[6]*z[9] + 5344*z[1]*z[12]*z[5]*z[6] - 832*z[1]*z[12]*z[5]*z[7]*z[9] + 10688*z[1]*z[12]*z[5]*z[7] - 1664*z[1]*z[12]*z[5]*z[8]*z[9] + 21376*z[1]*z[12]*z[5]*z[8] + 2672*z[1]*z[12]*z[5]*z[9] - 33840*z[1]*z[12]*z[5] - 1664*z[1]*z[12]*z[6]*z[7]*z[9] + 21376*z[1]*z[12]*z[6]*z[7] - 3328*z[1]*z[12]*z[6]*z[8]*z[9] + 42752*z[1]*z[12]*z[6]*z[8] + 5344*z[1]*z[12]*z[6]*z[9] - 67680*z[1]*z[12]*z[6] - 6656*z[1]*z[12]*z[7]*z[8]*z[9] + 85504*z[1]*z[12]*z[7]*z[8] + 10688*z[1]*z[12]*z[7]*z[9] - 135360*z[1]*z[12]*z[7] + 21376*z[1]*z[12]*z[8]*z[9] - 270720*z[1]*z[12]*z[8] - 26000*z[1]*z[12]*z[9] + 327760*z[1]*z[12] - 104*z[1]*z[2]*z[5]*z[6]*z[9] + 1016*z[1]*z[2]*z[5]*z[6] - 208*z[1]*z[2]*z[5]*z[7]*z[9] + 2032*z[1]*z[2]*z[5]*z[7] - 416*z[1]*z[2]*z[5]*z[8]*z[9] + 4064*z[1]*z[2]*z[5]*z[8] + 668*z[1]*z[2]*z[5]*z[9] - 6500*z[1]*z[2]*z[5] - 416*z[1]*z[2]*z[6]*z[7]*z[9] + 4064*z[1]*z[2]*z[6]*z[7] - 832*z[1]*z[2]*z[6]*z[8]*z[9] + 8128*z[1]*z[2]*z[6]*z[8] + 1336*z[1]*z[2]*z[6]*z[9] - 13000*z[1]*z[2]*z[6] - 1664*z[1]*z[2]*z[7]*z[8]*z[9] + 16256*z[1]*z[2]*z[7]*z[8] + 2672*z[1]*z[2]*z[7]*z[9] - 26000*z[1]*z[2]*z[7] + 5344*z[1]*z[2]*z[8]*z[9] - 52000*z[1]*z[2]*z[8] - 6500*z[1]*z[2]*z[9] + 63180*z[1]*z[2] - 208*z[1]*z[3]*z[5]*z[6]*z[9] + 2032*z[1]*z[3]*z[5]*z[6] - 416*z[1]*z[3]*z[5]*z[7]*z[9] + 4064*z[1]*z[3]*z[5]*z[7] - 832*z[1]*z[3]*z[5]*z[8]*z[9] + 8128*z[1]*z[3]*z[5]*z[8] + 1336*z[1]*z[3]*z[5]*z[9] - 13000*z[1]*z[3]*z[5] - 832*z[1]*z[3]*z[6]*z[7]*z[9] + 8128*z[1]*z[3]*z[6]*z[7] - 1664*z[1]*z[3]*z[6]*z[8]*z[9] + 16256*z[1]*z[3]*z[6]*z[8] + 2672*z[1]*z[3]*z[6]*z[9] - 26000*z[1]*z[3]*z[6] - 3328*z[1]*z[3]*z[7]*z[8]*z[9] + 32512*z[1]*z[3]*z[7]*z[8] + 5344*z[1]*z[3]*z[7]*z[9] - 52000*z[1]*z[3]*z[7] + 10688*z[1]*z[3]*z[8]*z[9] - 104000*z[1]*z[3]*z[8] - 13000*z[1]*z[3]*z[9] + 126360*z[1]*z[3] - 416*z[1]*z[4]*z[5]*z[6]*z[9] + 4064*z[1]*z[4]*z[5]*z[6] - 832*z[1]*z[4]*z[5]*z[7]*z[9] + 8128*z[1]*z[4]*z[5]*z[7] - 1664*z[1]*z[4]*z[5]*z[8]*z[9] + 16256*z[1]*z[4]*z[5]*z[8] + 2672*z[1]*z[4]*z[5]*z[9] - 26000*z[1]*z[4]*z[5] - 1664*z[1]*z[4]*z[6]*z[7]*z[9] + 16256*z[1]*z[4]*z[6]*z[7] - 3328*z[1]*z[4]*z[6]*z[8]*z[9] + 32512*z[1]*z[4]*z[6]*z[8] + 5344*z[1]*z[4]*z[6]*z[9] - 52000*z[1]*z[4]*z[6] - 6656*z[1]*z[4]*z[7]*z[8]*z[9] + 65024*z[1]*z[4]*z[7]*z[8] + 10688*z[1]*z[4]*z[7]*z[9] - 104000*z[1]*z[4]*z[7] + 21376*z[1]*z[4]*z[8]*z[9] - 208000*z[1]*z[4]*z[8] - 26000*z[1]*z[4]*z[9] + 252720*z[1]*z[4] + 668*z[1]*z[5]*z[6]*z[9] - 6500*z[1]*z[5]*z[6] + 1336*z[1]*z[5]*z[7]*z[9] - 13000*z[1]*z[5]*z[7] + 2672*z[1]*z[5]*z[8]*z[9] - 26000*z[1]*z[5]*z[8] - 4230*z[1]*z[5]*z[9] + 40970*z[1]*z[5] + 2672*z[1]*z[6]*z[7]*z[9] - 26000*z[1]*z[6]*z[7] + 5344*z[1]*z[6]*z[8]*z[9] - 52000*z[1]*z[6]*z[8] - 8460*z[1]*z[6]*z[9] + 81940*z[1]*z[6] + 10688*z[1]*z[7]*z[8]*z[9] - 104000*z[1]*z[7]*z[8] - 16920*z[1]*z[7]*z[9] + 163880*z[1]*z[7] - 33840*z[1]*z[8]*z[9] + 327760*z[1]*z[8] + 40970*z[1]*z[9] - 396350*z[1] + 256*z[10]*z[11]*z[2]*z[3]*z[5]*z[6] + 512*z[10]*z[11]*z[2]*z[3]*z[5]*z[7] + 1024*z[10]*z[11]*z[2]*z[3]*z[5]*z[8] - 1664*z[10]*z[11]*z[2]*z[3]*z[5] + 1024*z[10]*z[11]*z[2]*z[3]*z[6]*z[7] + 2048*z[10]*z[11]*z[2]*z[3]*z[6]*z[8] - 3328*z[10]*z[11]*z[2]*z[3]*z[6] + 4096*z[10]*z[11]*z[2]*z[3]*z[7]*z[8] - 6656*z[10]*z[11]*z[2]*z[3]*z[7] - 13312*z[10]*z[11]*z[2]*z[3]*z[8] + 16256*z[10]*z[11]*z[2]*z[3] + 512*z[10]*z[11]*z[2]*z[4]*z[5]*z[6] + 1024*z[10]*z[11]*z[2]*z[4]*z[5]*z[7] + 2048*z[10]*z[11]*z[2]*z[4]*z[5]*z[8] - 3328*z[10]*z[11]*z[2]*z[4]*z[5] + 2048*z[10]*z[11]*z[2]*z[4]*z[6]*z[7] + 4096*z[10]*z[11]*z[2]*z[4]*z[6]*z[8] - 6656*z[10]*z[11]*z[2]*z[4]*z[6] + 8192*z[10]*z[11]*z[2]*z[4]*z[7]*z[8] - 13312*z[10]*z[11]*z[2]*z[4]*z[7] - 26624*z[10]*z[11]*z[2]*z[4]*z[8] + 32512*z[10]*z[11]*z[2]*z[4] - 832*z[10]*z[11]*z[2]*z[5]*z[6] - 1664*z[10]*z[11]*z[2]*z[5]*z[7] - 3328*z[10]*z[11]*z[2]*z[5]*z[8] + 5344*z[10]*z[11]*z[2]*z[5] - 3328*z[10]*z[11]*z[2]*z[6]*z[7] - 6656*z[10]*z[11]*z[2]*z[6]*z[8] + 10688*z[10]*z[11]*z[2]*z[6] - 13312*z[10]*z[11]*z[2]*z[7]*z[8] + 21376*z[10]*z[11]*z[2]*z[7] + 42752*z[10]*z[11]*z[2]*z[8] - 52000*z[10]*z[11]*z[2] + 1024*z[10]*z[11]*z[3]*z[4]*z[5]*z[6] + 2048*z[10]*z[11]*z[3]*z[4]*z[5]*z[7] + 4096*z[10]*z[11]*z[3]*z[4]*z[5]*z[8] - 6656*z[10]*z[11]*z[3]*z[4]*z[5] + 4096*z[10]*z[11]*z[3]*z[4]*z[6]*z[7] + 8192*z[10]*z[11]*z[3]*z[4]*z[6]*z[8] - 13312*z[10]*z[11]*z[3]*z[4]*z[6] + 16384*z[10]*z[11]*z[3]*z[4]*z[7]*z[8] - 26624*z[10]*z[11]*z[3]*z[4]*z[7] - 53248*z[10]*z[11]*z[3]*z[4]*z[8] + 65024*z[10]*z[11]*z[3]*z[4] - 1664*z[10]*z[11]*z[3]*z[5]*z[6] - 3328*z[10]*z[11]*z[3]*z[5]*z[7] - 6656*z[10]*z[11]*z[3]*z[5]*z[8] + 10688*z[10]*z[11]*z[3]*z[5] - 6656*z[10]*z[11]*z[3]*z[6]*z[7] - 13312*z[10]*z[11]*z[3]*z[6]*z[8] + 21376*z[10]*z[11]*z[3]*z[6] - 26624*z[10]*z[11]*z[3]*z[7]*z[8] + 42752*z[10]*z[11]*z[3]*z[7] + 85504*z[10]*z[11]*z[3]*z[8] - 104000*z[10]*z[11]*z[3] - 3328*z[10]*z[11]*z[4]*z[5]*z[6] - 6656*z[10]*z[11]*z[4]*z[5]*z[7] - 13312*z[10]*z[11]*z[4]*z[5]*z[8] + 21376*z[10]*z[11]*z[4]*z[5] - 13312*z[10]*z[11]*z[4]*z[6]*z[7] - 26624*z[10]*z[11]*z[4]*z[6]*z[8] + 42752*z[10]*z[11]*z[4]*z[6] - 53248*z[10]*z[11]*z[4]*z[7]*z[8] + 85504*z[10]*z[11]*z[4]*z[7] + 171008*z[10]*z[11]*z[4]*z[8] - 208000*z[10]*z[11]*z[4] + 4064*z[10]*z[11]*z[5]*z[6] + 8128*z[10]*z[11]*z[5]*z[7] + 16256*z[10]*z[11]*z[5]*z[8] - 26000*z[10]*z[11]*z[5] + 16256*z[10]*z[11]*z[6]*z[7] + 32512*z[10]*z[11]*z[6]*z[8] - 52000*z[10]*z[11]*z[6] + 65024*z[10]*z[11]*z[7]*z[8] - 104000*z[10]*z[11]*z[7] - 208000*z[10]*z[11]*z[8] + 252720*z[10]*z[11] + 512*z[10]*z[12]*z[2]*z[3]*z[5]*z[6] + 1024*z[10]*z[12]*z[2]*z[3]*z[5]*z[7] + 2048*z[10]*z[12]*z[2]*z[3]*z[5]*z[8] - 3328*z[10]*z[12]*z[2]*z[3]*z[5] + 2048*z[10]*z[12]*z[2]*z[3]*z[6]*z[7] + 4096*z[10]*z[12]*z[2]*z[3]*z[6]*z[8] - 6656*z[10]*z[12]*z[2]*z[3]*z[6] + 8192*z[10]*z[12]*z[2]*z[3]*z[7]*z[8] - 13312*z[10]*z[12]*z[2]*z[3]*z[7] - 26624*z[10]*z[12]*z[2]*z[3]*z[8] + 32512*z[10]*z[12]*z[2]*z[3] + 1024*z[10]*z[12]*z[2]*z[4]*z[5]*z[6] + 2048*z[10]*z[12]*z[2]*z[4]*z[5]*z[7] + 4096*z[10]*z[12]*z[2]*z[4]*z[5]*z[8] - 6656*z[10]*z[12]*z[2]*z[4]*z[5] + 4096*z[10]*z[12]*z[2]*z[4]*z[6]*z[7] + 8192*z[10]*z[12]*z[2]*z[4]*z[6]*z[8] - 13312*z[10]*z[12]*z[2]*z[4]*z[6] + 16384*z[10]*z[12]*z[2]*z[4]*z[7]*z[8] - 26624*z[10]*z[12]*z[2]*z[4]*z[7] - 53248*z[10]*z[12]*z[2]*z[4]*z[8] + 65024*z[10]*z[12]*z[2]*z[4] - 1664*z[10]*z[12]*z[2]*z[5]*z[6] - 3328*z[10]*z[12]*z[2]*z[5]*z[7] - 6656*z[10]*z[12]*z[2]*z[5]*z[8] + 10688*z[10]*z[12]*z[2]*z[5] - 6656*z[10]*z[12]*z[2]*z[6]*z[7] - 13312*z[10]*z[12]*z[2]*z[6]*z[8] + 21376*z[10]*z[12]*z[2]*z[6] - 26624*z[10]*z[12]*z[2]*z[7]*z[8] + 42752*z[10]*z[12]*z[2]*z[7] + 85504*z[10]*z[12]*z[2]*z[8] - 104000*z[10]*z[12]*z[2] + 2048*z[10]*z[12]*z[3]*z[4]*z[5]*z[6] + 4096*z[10]*z[12]*z[3]*z[4]*z[5]*z[7] + 8192*z[10]*z[12]*z[3]*z[4]*z[5]*z[8] - 13312*z[10]*z[12]*z[3]*z[4]*z[5] + 8192*z[10]*z[12]*z[3]*z[4]*z[6]*z[7] + 16384*z[10]*z[12]*z[3]*z[4]*z[6]*z[8] - 26624*z[10]*z[12]*z[3]*z[4]*z[6] + 32768*z[10]*z[12]*z[3]*z[4]*z[7]*z[8] - 53248*z[10]*z[12]*z[3]*z[4]*z[7] - 106496*z[10]*z[12]*z[3]*z[4]*z[8] + 130048*z[10]*z[12]*z[3]*z[4] - 3328*z[10]*z[12]*z[3]*z[5]*z[6] - 6656*z[10]*z[12]*z[3]*z[5]*z[7] - 13312*z[10]*z[12]*z[3]*z[5]*z[8] + 21376*z[10]*z[12]*z[3]*z[5] - 13312*z[10]*z[12]*z[3]*z[6]*z[7] - 26624*z[10]*z[12]*z[3]*z[6]*z[8] + 42752*z[10]*z[12]*z[3]*z[6] - 53248*z[10]*z[12]*z[3]*z[7]*z[8] + 85504*z[10]*z[12]*z[3]*z[7] + 171008*z[10]*z[12]*z[3]*z[8] - 208000*z[10]*z[12]*z[3] - 6656*z[10]*z[12]*z[4]*z[5]*z[6] - 13312*z[10]*z[12]*z[4]*z[5]*z[7] - 26624*z[10]*z[12]*z[4]*z[5]*z[8] + 42752*z[10]*z[12]*z[4]*z[5] - 26624*z[10]*z[12]*z[4]*z[6]*z[7] - 53248*z[10]*z[12]*z[4]*z[6]*z[8] + 85504*z[10]*z[12]*z[4]*z[6] - 106496*z[10]*z[12]*z[4]*z[7]*z[8] + 171008*z[10]*z[12]*z[4]*z[7] + 342016*z[10]*z[12]*z[4]*z[8] - 416000*z[10]*z[12]*z[4] + 8128*z[10]*z[12]*z[5]*z[6] + 16256*z[10]*z[12]*z[5]*z[7] + 32512*z[10]*z[12]*z[5]*z[8] - 52000*z[10]*z[12]*z[5] + 32512*z[10]*z[12]*z[6]*z[7] + 65024*z[10]*z[12]*z[6]*z[8] - 104000*z[10]*z[12]*z[6] + 130048*z[10]*z[12]*z[7]*z[8] - 208000*z[10]*z[12]*z[7] - 416000*z[10]*z[12]*z[8] + 505440*z[10]*z[12] + 64*z[10]*z[2]*z[3]*z[5]*z[6]*z[9] - 832*z[10]*z[2]*z[3]*z[5]*z[6] + 128*z[10]*z[2]*z[3]*z[5]*z[7]*z[9] - 1664*z[10]*z[2]*z[3]*z[5]*z[7] + 256*z[10]*z[2]*z[3]*z[5]*z[8]*z[9] - 3328*z[10]*z[2]*z[3]*z[5]*z[8] - 416*z[10]*z[2]*z[3]*z[5]*z[9] + 5344*z[10]*z[2]*z[3]*z[5] + 256*z[10]*z[2]*z[3]*z[6]*z[7]*z[9] - 3328*z[10]*z[2]*z[3]*z[6]*z[7] + 512*z[10]*z[2]*z[3]*z[6]*z[8]*z[9] - 6656*z[10]*z[2]*z[3]*z[6]*z[8] - 832*z[10]*z[2]*z[3]*z[6]*z[9] + 10688*z[10]*z[2]*z[3]*z[6] + 1024*z[10]*z[2]*z[3]*z[7]*z[8]*z[9] - 13312*z[10]*z[2]*z[3]*z[7]*z[8] - 1664*z[10]*z[2]*z[3]*z[7]*z[9] + 21376*z[10]*z[2]*z[3]*z[7] - 3328*z[10]*z[2]*z[3]*z[8]*z[9] + 42752*z[10]*z[2]*z[3]*z[8] + 4064*z[10]*z[2]*z[3]*z[9] - 52000*z[10]*z[2]*z[3] + 128*z[10]*z[2]*z[4]*z[5]*z[6]*z[9] - 1664*z[10]*z[2]*z[4]*z[5]*z[6] + 256*z[10]*z[2]*z[4]*z[5]*z[7]*z[9] - 3328*z[10]*z[2]*z[4]*z[5]*z[7] + 512*z[10]*z[2]*z[4]*z[5]*z[8]*z[9] - 6656*z[10]*z[2]*z[4]*z[5]*z[8] - 832*z[10]*z[2]*z[4]*z[5]*z[9] + 10688*z[10]*z[2]*z[4]*z[5] + 512*z[10]*z[2]*z[4]*z[6]*z[7]*z[9] - 6656*z[10]*z[2]*z[4]*z[6]*z[7] + 1024*z[10]*z[2]*z[4]*z[6]*z[8]*z[9] - 13312*z[10]*z[2]*z[4]*z[6]*z[8] - 1664*z[10]*z[2]*z[4]*z[6]*z[9] + 21376*z[10]*z[2]*z[4]*z[6] + 2048*z[10]*z[2]*z[4]*z[7]*z[8]*z[9] - 26624*z[10]*z[2]*z[4]*z[7]*z[8] - 3328*z[10]*z[2]*z[4]*z[7]*z[9] + 42752*z[10]*z[2]*z[4]*z[7] - 6656*z[10]*z[2]*z[4]*z[8]*z[9] + 85504*z[10]*z[2]*z[4]*z[8] + 8128*z[10]*z[2]*z[4]*z[9] - 104000*z[10]*z[2]*z[4] - 208*z[10]*z[2]*z[5]*z[6]*z[9] + 2672*z[10]*z[2]*z[5]*z[6] - 416*z[10]*z[2]*z[5]*z[7]*z[9] + 5344*z[10]*z[2]*z[5]*z[7] - 832*z[10]*z[2]*z[5]*z[8]*z[9] + 10688*z[10]*z[2]*z[5]*z[8] + 1336*z[10]*z[2]*z[5]*z[9] - 16920*z[10]*z[2]*z[5] - 832*z[10]*z[2]*z[6]*z[7]*z[9] + 10688*z[10]*z[2]*z[6]*z[7] - 1664*z[10]*z[2]*z[6]*z[8]*z[9] + 21376*z[10]*z[2]*z[6]*z[8] + 2672*z[10]*z[2]*z[6]*z[9] - 33840*z[10]*z[2]*z[6] - 3328*z[10]*z[2]*z[7]*z[8]*z[9] + 42752*z[10]*z[2]*z[7]*z[8] + 5344*z[10]*z[2]*z[7]*z[9] - 67680*z[10]*z[2]*z[7] + 10688*z[10]*z[2]*z[8]*z[9] - 135360*z[10]*z[2]*z[8] - 13000*z[10]*z[2]*z[9] + 163880*z[10]*z[2] + 256*z[10]*z[3]*z[4]*z[5]*z[6]*z[9] - 3328*z[10]*z[3]*z[4]*z[5]*z[6] + 512*z[10]*z[3]*z[4]*z[5]*z[7]*z[9] - 6656*z[10]*z[3]*z[4]*z[5]*z[7] + 1024*z[10]*z[3]*z[4]*z[5]*z[8]*z[9] - 13312*z[10]*z[3]*z[4]*z[5]*z[8] - 1664*z[10]*z[3]*z[4]*z[5]*z[9] + 21376*z[10]*z[3]*z[4]*z[5] + 1024*z[10]*z[3]*z[4]*z[6]*z[7]*z[9] - 13312*z[10]*z[3]*z[4]*z[6]*z[7] + 2048*z[10]*z[3]*z[4]*z[6]*z[8]*z[9] - 26624*z[10]*z[3]*z[4]*z[6]*z[8] - 3328*z[10]*z[3]*z[4]*z[6]*z[9] + 42752*z[10]*z[3]*z[4]*z[6] + 4096*z[10]*z[3]*z[4]*z[7]*z[8]*z[9] - 53248*z[10]*z[3]*z[4]*z[7]*z[8] - 6656*z[10]*z[3]*z[4]*z[7]*z[9] + 85504*z[10]*z[3]*z[4]*z[7] - 13312*z[10]*z[3]*z[4]*z[8]*z[9] + 171008*z[10]*z[3]*z[4]*z[8] + 16256*z[10]*z[3]*z[4]*z[9] - 208000*z[10]*z[3]*z[4] - 416*z[10]*z[3]*z[5]*z[6]*z[9] + 5344*z[10]*z[3]*z[5]*z[6] - 832*z[10]*z[3]*z[5]*z[7]*z[9] + 10688*z[10]*z[3]*z[5]*z[7] - 1664*z[10]*z[3]*z[5]*z[8]*z[9] + 21376*z[10]*z[3]*z[5]*z[8] + 2672*z[10]*z[3]*z[5]*z[9] - 33840*z[10]*z[3]*z[5] - 1664*z[10]*z[3]*z[6]*z[7]*z[9] + 21376*z[10]*z[3]*z[6]*z[7] - 3328*z[10]*z[3]*z[6]*z[8]*z[9] + 42752*z[10]*z[3]*z[6]*z[8] + 5344*z[10]*z[3]*z[6]*z[9] - 67680*z[10]*z[3]*z[6] - 6656*z[10]*z[3]*z[7]*z[8]*z[9] + 85504*z[10]*z[3]*z[7]*z[8] + 10688*z[10]*z[3]*z[7]*z[9] - 135360*z[10]*z[3]*z[7] + 21376*z[10]*z[3]*z[8]*z[9] - 270720*z[10]*z[3]*z[8] - 26000*z[10]*z[3]*z[9] + 327760*z[10]*z[3] - 832*z[10]*z[4]*z[5]*z[6]*z[9] + 10688*z[10]*z[4]*z[5]*z[6] - 1664*z[10]*z[4]*z[5]*z[7]*z[9] + 21376*z[10]*z[4]*z[5]*z[7] - 3328*z[10]*z[4]*z[5]*z[8]*z[9] + 42752*z[10]*z[4]*z[5]*z[8] + 5344*z[10]*z[4]*z[5]*z[9] - 67680*z[10]*z[4]*z[5] - 3328*z[10]*z[4]*z[6]*z[7]*z[9] + 42752*z[10]*z[4]*z[6]*z[7] - 6656*z[10]*z[4]*z[6]*z[8]*z[9] + 85504*z[10]*z[4]*z[6]*z[8] + 10688*z[10]*z[4]*z[6]*z[9] - 135360*z[10]*z[4]*z[6] - 13312*z[10]*z[4]*z[7]*z[8]*z[9] + 171008*z[10]*z[4]*z[7]*z[8] + 21376*z[10]*z[4]*z[7]*z[9] - 270720*z[10]*z[4]*z[7] + 42752*z[10]*z[4]*z[8]*z[9] - 541440*z[10]*z[4]*z[8] - 52000*z[10]*z[4]*z[9] + 655520*z[10]*z[4] + 1016*z[10]*z[5]*z[6]*z[9] - 13000*z[10]*z[5]*z[6] + 2032*z[10]*z[5]*z[7]*z[9] - 26000*z[10]*z[5]*z[7] + 4064*z[10]*z[5]*z[8]*z[9] - 52000*z[10]*z[5]*z[8] - 6500*z[10]*z[5]*z[9] + 81940*z[10]*z[5] + 4064*z[10]*z[6]*z[7]*z[9] - 52000*z[10]*z[6]*z[7] + 8128*z[10]*z[6]*z[8]*z[9] - 104000*z[10]*z[6]*z[8] - 13000*z[10]*z[6]*z[9] + 163880*z[10]*z[6] + 16256*z[10]*z[7]*z[8]*z[9] - 208000*z[10]*z[7]*z[8] - 26000*z[10]*z[7]*z[9] + 327760*z[10]*z[7] - 52000*z[10]*z[8]*z[9] + 655520*z[10]*z[8] + 63180*z[10]*z[9] - 792700*z[10] + 1024*z[11]*z[12]*z[2]*z[3]*z[5]*z[6] + 2048*z[11]*z[12]*z[2]*z[3]*z[5]*z[7] + 4096*z[11]*z[12]*z[2]*z[3]*z[5]*z[8] - 6656*z[11]*z[12]*z[2]*z[3]*z[5] + 4096*z[11]*z[12]*z[2]*z[3]*z[6]*z[7] + 8192*z[11]*z[12]*z[2]*z[3]*z[6]*z[8] - 13312*z[11]*z[12]*z[2]*z[3]*z[6] + 16384*z[11]*z[12]*z[2]*z[3]*z[7]*z[8] - 26624*z[11]*z[12]*z[2]*z[3]*z[7] - 53248*z[11]*z[12]*z[2]*z[3]*z[8] + 65024*z[11]*z[12]*z[2]*z[3] + 2048*z[11]*z[12]*z[2]*z[4]*z[5]*z[6] + 4096*z[11]*z[12]*z[2]*z[4]*z[5]*z[7] + 8192*z[11]*z[12]*z[2]*z[4]*z[5]*z[8] - 13312*z[11]*z[12]*z[2]*z[4]*z[5] + 8192*z[11]*z[12]*z[2]*z[4]*z[6]*z[7] + 16384*z[11]*z[12]*z[2]*z[4]*z[6]*z[8] - 26624*z[11]*z[12]*z[2]*z[4]*z[6] + 32768*z[11]*z[12]*z[2]*z[4]*z[7]*z[8] - 53248*z[11]*z[12]*z[2]*z[4]*z[7] - 106496*z[11]*z[12]*z[2]*z[4]*z[8] + 130048*z[11]*z[12]*z[2]*z[4] - 3328*z[11]*z[12]*z[2]*z[5]*z[6] - 6656*z[11]*z[12]*z[2]*z[5]*z[7] - 13312*z[11]*z[12]*z[2]*z[5]*z[8] + 21376*z[11]*z[12]*z[2]*z[5] - 13312*z[11]*z[12]*z[2]*z[6]*z[7] - 26624*z[11]*z[12]*z[2]*z[6]*z[8] + 42752*z[11]*z[12]*z[2]*z[6] - 53248*z[11]*z[12]*z[2]*z[7]*z[8] + 85504*z[11]*z[12]*z[2]*z[7] + 171008*z[11]*z[12]*z[2]*z[8] - 208000*z[11]*z[12]*z[2] + 4096*z[11]*z[12]*z[3]*z[4]*z[5]*z[6] + 8192*z[11]*z[12]*z[3]*z[4]*z[5]*z[7] + 16384*z[11]*z[12]*z[3]*z[4]*z[5]*z[8] - 26624*z[11]*z[12]*z[3]*z[4]*z[5] + 16384*z[11]*z[12]*z[3]*z[4]*z[6]*z[7] + 32768*z[11]*z[12]*z[3]*z[4]*z[6]*z[8] - 53248*z[11]*z[12]*z[3]*z[4]*z[6] + 65536*z[11]*z[12]*z[3]*z[4]*z[7]*z[8] - 106496*z[11]*z[12]*z[3]*z[4]*z[7] - 212992*z[11]*z[12]*z[3]*z[4]*z[8] + 260096*z[11]*z[12]*z[3]*z[4] - 6656*z[11]*z[12]*z[3]*z[5]*z[6] - 13312*z[11]*z[12]*z[3]*z[5]*z[7] - 26624*z[11]*z[12]*z[3]*z[5]*z[8] + 42752*z[11]*z[12]*z[3]*z[5] - 26624*z[11]*z[12]*z[3]*z[6]*z[7] - 53248*z[11]*z[12]*z[3]*z[6]*z[8] + 85504*z[11]*z[12]*z[3]*z[6] - 106496*z[11]*z[12]*z[3]*z[7]*z[8] + 171008*z[11]*z[12]*z[3]*z[7] + 342016*z[11]*z[12]*z[3]*z[8] - 416000*z[11]*z[12]*z[3] - 13312*z[11]*z[12]*z[4]*z[5]*z[6] - 26624*z[11]*z[12]*z[4]*z[5]*z[7] - 53248*z[11]*z[12]*z[4]*z[5]*z[8] + 85504*z[11]*z[12]*z[4]*z[5] - 53248*z[11]*z[12]*z[4]*z[6]*z[7] - 106496*z[11]*z[12]*z[4]*z[6]*z[8] + 171008*z[11]*z[12]*z[4]*z[6] - 212992*z[11]*z[12]*z[4]*z[7]*z[8] + 342016*z[11]*z[12]*z[4]*z[7] + 684032*z[11]*z[12]*z[4]*z[8] - 832000*z[11]*z[12]*z[4] + 16256*z[11]*z[12]*z[5]*z[6] + 32512*z[11]*z[12]*z[5]*z[7] + 65024*z[11]*z[12]*z[5]*z[8] - 104000*z[11]*z[12]*z[5] + 65024*z[11]*z[12]*z[6]*z[7] + 130048*z[11]*z[12]*z[6]*z[8] - 208000*z[11]*z[12]*z[6] + 260096*z[11]*z[12]*z[7]*z[8] - 416000*z[11]*z[12]*z[7] - 832000*z[11]*z[12]*z[8] + 1010880*z[11]*z[12] + 128*z[11]*z[2]*z[3]*z[5]*z[6]*z[9] - 1664*z[11]*z[2]*z[3]*z[5]*z[6] + 256*z[11]*z[2]*z[3]*z[5]*z[7]*z[9] - 3328*z[11]*z[2]*z[3]*z[5]*z[7] + 512*z[11]*z[2]*z[3]*z[5]*z[8]*z[9] - 6656*z[11]*z[2]*z[3]*z[5]*z[8] - 832*z[11]*z[2]*z[3]*z[5]*z[9] + 10688*z[11]*z[2]*z[3]*z[5] + 512*z[11]*z[2]*z[3]*z[6]*z[7]*z[9] - 6656*z[11]*z[2]*z[3]*z[6]*z[7] + 1024*z[11]*z[2]*z[3]*z[6]*z[8]*z[9] - 13312*z[11]*z[2]*z[3]*z[6]*z[8] - 1664*z[11]*z[2]*z[3]*z[6]*z[9] + 21376*z[11]*z[2]*z[3]*z[6] + 2048*z[11]*z[2]*z[3]*z[7]*z[8]*z[9] - 26624*z[11]*z[2]*z[3]*z[7]*z[8] - 3328*z[11]*z[2]*z[3]*z[7]*z[9] + 42752*z[11]*z[2]*z[3]*z[7] - 6656*z[11]*z[2]*z[3]*z[8]*z[9] + 85504*z[11]*z[2]*z[3]*z[8] + 8128*z[11]*z[2]*z[3]*z[9] - 104000*z[11]*z[2]*z[3] + 256*z[11]*z[2]*z[4]*z[5]*z[6]*z[9] - 3328*z[11]*z[2]*z[4]*z[5]*z[6] + 512*z[11]*z[2]*z[4]*z[5]*z[7]*z[9] - 6656*z[11]*z[2]*z[4]*z[5]*z[7] + 1024*z[11]*z[2]*z[4]*z[5]*z[8]*z[9] - 13312*z[11]*z[2]*z[4]*z[5]*z[8] - 1664*z[11]*z[2]*z[4]*z[5]*z[9] + 21376*z[11]*z[2]*z[4]*z[5] + 1024*z[11]*z[2]*z[4]*z[6]*z[7]*z[9] - 13312*z[11]*z[2]*z[4]*z[6]*z[7] + 2048*z[11]*z[2]*z[4]*z[6]*z[8]*z[9] - 26624*z[11]*z[2]*z[4]*z[6]*z[8] - 3328*z[11]*z[2]*z[4]*z[6]*z[9] + 42752*z[11]*z[2]*z[4]*z[6] + 4096*z[11]*z[2]*z[4]*z[7]*z[8]*z[9] - 53248*z[11]*z[2]*z[4]*z[7]*z[8] - 6656*z[11]*z[2]*z[4]*z[7]*z[9] + 85504*z[11]*z[2]*z[4]*z[7] - 13312*z[11]*z[2]*z[4]*z[8]*z[9] + 171008*z[11]*z[2]*z[4]*z[8] + 16256*z[11]*z[2]*z[4]*z[9] - 208000*z[11]*z[2]*z[4] - 416*z[11]*z[2]*z[5]*z[6]*z[9] + 5344*z[11]*z[2]*z[5]*z[6] - 832*z[11]*z[2]*z[5]*z[7]*z[9] + 10688*z[11]*z[2]*z[5]*z[7] - 1664*z[11]*z[2]*z[5]*z[8]*z[9] + 21376*z[11]*z[2]*z[5]*z[8] + 2672*z[11]*z[2]*z[5]*z[9] - 33840*z[11]*z[2]*z[5] - 1664*z[11]*z[2]*z[6]*z[7]*z[9] + 21376*z[11]*z[2]*z[6]*z[7] - 3328*z[11]*z[2]*z[6]*z[8]*z[9] + 42752*z[11]*z[2]*z[6]*z[8] + 5344*z[11]*z[2]*z[6]*z[9] - 67680*z[11]*z[2]*z[6] - 6656*z[11]*z[2]*z[7]*z[8]*z[9] + 85504*z[11]*z[2]*z[7]*z[8] + 10688*z[11]*z[2]*z[7]*z[9] - 135360*z[11]*z[2]*z[7] + 21376*z[11]*z[2]*z[8]*z[9] - 270720*z[11]*z[2]*z[8] - 26000*z[11]*z[2]*z[9] + 327760*z[11]*z[2] + 512*z[11]*z[3]*z[4]*z[5]*z[6]*z[9] - 6656*z[11]*z[3]*z[4]*z[5]*z[6] + 1024*z[11]*z[3]*z[4]*z[5]*z[7]*z[9] - 13312*z[11]*z[3]*z[4]*z[5]*z[7] + 2048*z[11]*z[3]*z[4]*z[5]*z[8]*z[9] - 26624*z[11]*z[3]*z[4]*z[5]*z[8] - 3328*z[11]*z[3]*z[4]*z[5]*z[9] + 42752*z[11]*z[3]*z[4]*z[5] + 2048*z[11]*z[3]*z[4]*z[6]*z[7]*z[9] - 26624*z[11]*z[3]*z[4]*z[6]*z[7] + 4096*z[11]*z[3]*z[4]*z[6]*z[8]*z[9] - 53248*z[11]*z[3]*z[4]*z[6]*z[8] - 6656*z[11]*z[3]*z[4]*z[6]*z[9] + 85504*z[11]*z[3]*z[4]*z[6] + 8192*z[11]*z[3]*z[4]*z[7]*z[8]*z[9] - 106496*z[11]*z[3]*z[4]*z[7]*z[8] - 13312*z[11]*z[3]*z[4]*z[7]*z[9] + 171008*z[11]*z[3]*z[4]*z[7] - 26624*z[11]*z[3]*z[4]*z[8]*z[9] + 342016*z[11]*z[3]*z[4]*z[8] + 32512*z[11]*z[3]*z[4]*z[9] - 416000*z[11]*z[3]*z[4] - 832*z[11]*z[3]*z[5]*z[6]*z[9] + 10688*z[11]*z[3]*z[5]*z[6] - 1664*z[11]*z[3]*z[5]*z[7]*z[9] + 21376*z[11]*z[3]*z[5]*z[7] - 3328*z[11]*z[3]*z[5]*z[8]*z[9] + 42752*z[11]*z[3]*z[5]*z[8] + 5344*z[11]*z[3]*z[5]*z[9] - 67680*z[11]*z[3]*z[5] - 3328*z[11]*z[3]*z[6]*z[7]*z[9] + 42752*z[11]*z[3]*z[6]*z[7] - 6656*z[11]*z[3]*z[6]*z[8]*z[9] + 85504*z[11]*z[3]*z[6]*z[8] + 10688*z[11]*z[3]*z[6]*z[9] - 135360*z[11]*z[3]*z[6] - 13312*z[11]*z[3]*z[7]*z[8]*z[9] + 171008*z[11]*z[3]*z[7]*z[8] + 21376*z[11]*z[3]*z[7]*z[9] - 270720*z[11]*z[3]*z[7] + 42752*z[11]*z[3]*z[8]*z[9] - 541440*z[11]*z[3]*z[8] - 52000*z[11]*z[3]*z[9] + 655520*z[11]*z[3] - 1664*z[11]*z[4]*z[5]*z[6]*z[9] + 21376*z[11]*z[4]*z[5]*z[6] - 3328*z[11]*z[4]*z[5]*z[7]*z[9] + 42752*z[11]*z[4]*z[5]*z[7] - 6656*z[11]*z[4]*z[5]*z[8]*z[9] + 85504*z[11]*z[4]*z[5]*z[8] + 10688*z[11]*z[4]*z[5]*z[9] - 135360*z[11]*z[4]*z[5] - 6656*z[11]*z[4]*z[6]*z[7]*z[9] + 85504*z[11]*z[4]*z[6]*z[7] - 13312*z[11]*z[4]*z[6]*z[8]*z[9] + 171008*z[11]*z[4]*z[6]*z[8] + 21376*z[11]*z[4]*z[6]*z[9] - 270720*z[11]*z[4]*z[6] - 26624*z[11]*z[4]*z[7]*z[8]*z[9] + 342016*z[11]*z[4]*z[7]*z[8] + 42752*z[11]*z[4]*z[7]*z[9] - 541440*z[11]*z[4]*z[7] + 85504*z[11]*z[4]*z[8]*z[9] - 1082880*z[11]*z[4]*z[8] - 104000*z[11]*z[4]*z[9] + 1311040*z[11]*z[4] + 2032*z[11]*z[5]*z[6]*z[9] - 26000*z[11]*z[5]*z[6] + 4064*z[11]*z[5]*z[7]*z[9] - 52000*z[11]*z[5]*z[7] + 8128*z[11]*z[5]*z[8]*z[9] - 104000*z[11]*z[5]*z[8] - 13000*z[11]*z[5]*z[9] + 163880*z[11]*z[5] + 8128*z[11]*z[6]*z[7]*z[9] - 104000*z[11]*z[6]*z[7] + 16256*z[11]*z[6]*z[8]*z[9] - 208000*z[11]*z[6]*z[8] - 26000*z[11]*z[6]*z[9] + 327760*z[11]*z[6] + 32512*z[11]*z[7]*z[8]*z[9] - 416000*z[11]*z[7]*z[8] - 52000*z[11]*z[7]*z[9] + 655520*z[11]*z[7] - 104000*z[11]*z[8]*z[9] + 1311040*z[11]*z[8] + 126360*z[11]*z[9] - 1585400*z[11] + 256*z[12]*z[2]*z[3]*z[5]*z[6]*z[9] - 3328*z[12]*z[2]*z[3]*z[5]*z[6] + 512*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] - 6656*z[12]*z[2]*z[3]*z[5]*z[7] + 1024*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] - 13312*z[12]*z[2]*z[3]*z[5]*z[8] - 1664*z[12]*z[2]*z[3]*z[5]*z[9] + 21376*z[12]*z[2]*z[3]*z[5] + 1024*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] - 13312*z[12]*z[2]*z[3]*z[6]*z[7] + 2048*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] - 26624*z[12]*z[2]*z[3]*z[6]*z[8] - 3328*z[12]*z[2]*z[3]*z[6]*z[9] + 42752*z[12]*z[2]*z[3]*z[6] + 4096*z[12]*z[2]*z[3]*z[7]*z[8]*z[9] - 53248*z[12]*z[2]*z[3]*z[7]*z[8] - 6656*z[12]*z[2]*z[3]*z[7]*z[9] + 85504*z[12]*z[2]*z[3]*z[7] - 13312*z[12]*z[2]*z[3]*z[8]*z[9] + 171008*z[12]*z[2]*z[3]*z[8] + 16256*z[12]*z[2]*z[3]*z[9] - 208000*z[12]*z[2]*z[3] + 512*z[12]*z[2]*z[4]*z[5]*z[6]*z[9] - 6656*z[12]*z[2]*z[4]*z[5]*z[6] + 1024*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] - 13312*z[12]*z[2]*z[4]*z[5]*z[7] + 2048*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] - 26624*z[12]*z[2]*z[4]*z[5]*z[8] - 3328*z[12]*z[2]*z[4]*z[5]*z[9] + 42752*z[12]*z[2]*z[4]*z[5] + 2048*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] - 26624*z[12]*z[2]*z[4]*z[6]*z[7] + 4096*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] - 53248*z[12]*z[2]*z[4]*z[6]*z[8] - 6656*z[12]*z[2]*z[4]*z[6]*z[9] + 85504*z[12]*z[2]*z[4]*z[6] + 8192*z[12]*z[2]*z[4]*z[7]*z[8]*z[9] - 106496*z[12]*z[2]*z[4]*z[7]*z[8] - 13312*z[12]*z[2]*z[4]*z[7]*z[9] + 171008*z[12]*z[2]*z[4]*z[7] - 26624*z[12]*z[2]*z[4]*z[8]*z[9] + 342016*z[12]*z[2]*z[4]*z[8] + 32512*z[12]*z[2]*z[4]*z[9] - 416000*z[12]*z[2]*z[4] - 832*z[12]*z[2]*z[5]*z[6]*z[9] + 10688*z[12]*z[2]*z[5]*z[6] - 1664*z[12]*z[2]*z[5]*z[7]*z[9] + 21376*z[12]*z[2]*z[5]*z[7] - 3328*z[12]*z[2]*z[5]*z[8]*z[9] + 42752*z[12]*z[2]*z[5]*z[8] + 5344*z[12]*z[2]*z[5]*z[9] - 67680*z[12]*z[2]*z[5] - 3328*z[12]*z[2]*z[6]*z[7]*z[9] + 42752*z[12]*z[2]*z[6]*z[7] - 6656*z[12]*z[2]*z[6]*z[8]*z[9] + 85504*z[12]*z[2]*z[6]*z[8] + 10688*z[12]*z[2]*z[6]*z[9] - 135360*z[12]*z[2]*z[6] - 13312*z[12]*z[2]*z[7]*z[8]*z[9] + 171008*z[12]*z[2]*z[7]*z[8] + 21376*z[12]*z[2]*z[7]*z[9] - 270720*z[12]*z[2]*z[7] + 42752*z[12]*z[2]*z[8]*z[9] - 541440*z[12]*z[2]*z[8] - 52000*z[12]*z[2]*z[9] + 655520*z[12]*z[2] + 1024*z[12]*z[3]*z[4]*z[5]*z[6]*z[9] - 13312*z[12]*z[3]*z[4]*z[5]*z[6] + 2048*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] - 26624*z[12]*z[3]*z[4]*z[5]*z[7] + 4096*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] - 53248*z[12]*z[3]*z[4]*z[5]*z[8] - 6656*z[12]*z[3]*z[4]*z[5]*z[9] + 85504*z[12]*z[3]*z[4]*z[5] + 4096*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] - 53248*z[12]*z[3]*z[4]*z[6]*z[7] + 8192*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] - 106496*z[12]*z[3]*z[4]*z[6]*z[8] - 13312*z[12]*z[3]*z[4]*z[6]*z[9] + 171008*z[12]*z[3]*z[4]*z[6] + 16384*z[12]*z[3]*z[4]*z[7]*z[8]*z[9] - 212992*z[12]*z[3]*z[4]*z[7]*z[8] - 26624*z[12]*z[3]*z[4]*z[7]*z[9] + 342016*z[12]*z[3]*z[4]*z[7] - 53248*z[12]*z[3]*z[4]*z[8]*z[9] + 684032*z[12]*z[3]*z[4]*z[8] + 65024*z[12]*z[3]*z[4]*z[9] - 832000*z[12]*z[3]*z[4] - 1664*z[12]*z[3]*z[5]*z[6]*z[9] + 21376*z[12]*z[3]*z[5]*z[6] - 3328*z[12]*z[3]*z[5]*z[7]*z[9] + 42752*z[12]*z[3]*z[5]*z[7] - 6656*z[12]*z[3]*z[5]*z[8]*z[9] + 85504*z[12]*z[3]*z[5]*z[8] + 10688*z[12]*z[3]*z[5]*z[9] - 135360*z[12]*z[3]*z[5] - 6656*z[12]*z[3]*z[6]*z[7]*z[9] + 85504*z[12]*z[3]*z[6]*z[7] - 13312*z[12]*z[3]*z[6]*z[8]*z[9] + 171008*z[12]*z[3]*z[6]*z[8] + 21376*z[12]*z[3]*z[6]*z[9] - 270720*z[12]*z[3]*z[6] - 26624*z[12]*z[3]*z[7]*z[8]*z[9] + 342016*z[12]*z[3]*z[7]*z[8] + 42752*z[12]*z[3]*z[7]*z[9] - 541440*z[12]*z[3]*z[7] + 85504*z[12]*z[3]*z[8]*z[9] - 1082880*z[12]*z[3]*z[8] - 104000*z[12]*z[3]*z[9] + 1311040*z[12]*z[3] - 3328*z[12]*z[4]*z[5]*z[6]*z[9] + 42752*z[12]*z[4]*z[5]*z[6] - 6656*z[12]*z[4]*z[5]*z[7]*z[9] + 85504*z[12]*z[4]*z[5]*z[7] - 13312*z[12]*z[4]*z[5]*z[8]*z[9] + 171008*z[12]*z[4]*z[5]*z[8] + 21376*z[12]*z[4]*z[5]*z[9] - 270720*z[12]*z[4]*z[5] - 13312*z[12]*z[4]*z[6]*z[7]*z[9] + 171008*z[12]*z[4]*z[6]*z[7] - 26624*z[12]*z[4]*z[6]*z[8]*z[9] + 342016*z[12]*z[4]*z[6]*z[8] + 42752*z[12]*z[4]*z[6]*z[9] - 541440*z[12]*z[4]*z[6] - 53248*z[12]*z[4]*z[7]*z[8]*z[9] + 684032*z[12]*z[4]*z[7]*z[8] + 85504*z[12]*z[4]*z[7]*z[9] - 1082880*z[12]*z[4]*z[7] + 171008*z[12]*z[4]*z[8]*z[9] - 2165760*z[12]*z[4]*z[8] - 208000*z[12]*z[4]*z[9] + 2622080*z[12]*z[4] + 4064*z[12]*z[5]*z[6]*z[9] - 52000*z[12]*z[5]*z[6] + 8128*z[12]*z[5]*z[7]*z[9] - 104000*z[12]*z[5]*z[7] + 16256*z[12]*z[5]*z[8]*z[9] - 208000*z[12]*z[5]*z[8] - 26000*z[12]*z[5]*z[9] + 327760*z[12]*z[5] + 16256*z[12]*z[6]*z[7]*z[9] - 208000*z[12]*z[6]*z[7] + 32512*z[12]*z[6]*z[8]*z[9] - 416000*z[12]*z[6]*z[8] - 52000*z[12]*z[6]*z[9] + 655520*z[12]*z[6] + 65024*z[12]*z[7]*z[8]*z[9] - 832000*z[12]*z[7]*z[8] - 104000*z[12]*z[7]*z[9] + 1311040*z[12]*z[7] - 208000*z[12]*z[8]*z[9] + 2622080*z[12]*z[8] + 252720*z[12]*z[9] - 3170800*z[12] - 416*z[2]*z[3]*z[5]*z[6]*z[9] + 4064*z[2]*z[3]*z[5]*z[6] - 832*z[2]*z[3]*z[5]*z[7]*z[9] + 8128*z[2]*z[3]*z[5]*z[7] - 1664*z[2]*z[3]*z[5]*z[8]*z[9] + 16256*z[2]*z[3]*z[5]*z[8] + 2672*z[2]*z[3]*z[5]*z[9] - 26000*z[2]*z[3]*z[5] - 1664*z[2]*z[3]*z[6]*z[7]*z[9] + 16256*z[2]*z[3]*z[6]*z[7] - 3328*z[2]*z[3]*z[6]*z[8]*z[9] + 32512*z[2]*z[3]*z[6]*z[8] + 5344*z[2]*z[3]*z[6]*z[9] - 52000*z[2]*z[3]*z[6] - 6656*z[2]*z[3]*z[7]*z[8]*z[9] + 65024*z[2]*z[3]*z[7]*z[8] + 10688*z[2]*z[3]*z[7]*z[9] - 104000*z[2]*z[3]*z[7] + 21376*z[2]*z[3]*z[8]*z[9] - 208000*z[2]*z[3]*z[8] - 26000*z[2]*z[3]*z[9] + 252720*z[2]*z[3] - 832*z[2]*z[4]*z[5]*z[6]*z[9] + 8128*z[2]*z[4]*z[5]*z[6] - 1664*z[2]*z[4]*z[5]*z[7]*z[9] + 16256*z[2]*z[4]*z[5]*z[7] - 3328*z[2]*z[4]*z[5]*z[8]*z[9] + 32512*z[2]*z[4]*z[5]*z[8] + 5344*z[2]*z[4]*z[5]*z[9] - 52000*z[2]*z[4]*z[5] - 3328*z[2]*z[4]*z[6]*z[7]*z[9] + 32512*z[2]*z[4]*z[6]*z[7] - 6656*z[2]*z[4]*z[6]*z[8]*z[9] + 65024*z[2]*z[4]*z[6]*z[8] + 10688*z[2]*z[4]*z[6]*z[9] - 104000*z[2]*z[4]*z[6] - 13312*z[2]*z[4]*z[7]*z[8]*z[9] + 130048*z[2]*z[4]*z[7]*z[8] + 21376*z[2]*z[4]*z[7]*z[9] - 208000*z[2]*z[4]*z[7] + 42752*z[2]*z[4]*z[8]*z[9] - 416000*z[2]*z[4]*z[8] - 52000*z[2]*z[4]*z[9] + 505440*z[2]*z[4] + 1336*z[2]*z[5]*z[6]*z[9] - 13000*z[2]*z[5]*z[6] + 2672*z[2]*z[5]*z[7]*z[9] - 26000*z[2]*z[5]*z[7] + 5344*z[2]*z[5]*z[8]*z[9] - 52000*z[2]*z[5]*z[8] - 8460*z[2]*z[5]*z[9] + 81940*z[2]*z[5] + 5344*z[2]*z[6]*z[7]*z[9] - 52000*z[2]*z[6]*z[7] + 10688*z[2]*z[6]*z[8]*z[9] - 104000*z[2]*z[6]*z[8] - 16920*z[2]*z[6]*z[9] + 163880*z[2]*z[6] + 21376*z[2]*z[7]*z[8]*z[9] - 208000*z[2]*z[7]*z[8] - 33840*z[2]*z[7]*z[9] + 327760*z[2]*z[7] - 67680*z[2]*z[8]*z[9] + 655520*z[2]*z[8] + 81940*z[2]*z[9] - 792700*z[2] - 1664*z[3]*z[4]*z[5]*z[6]*z[9] + 16256*z[3]*z[4]*z[5]*z[6] - 3328*z[3]*z[4]*z[5]*z[7]*z[9] + 32512*z[3]*z[4]*z[5]*z[7] - 6656*z[3]*z[4]*z[5]*z[8]*z[9] + 65024*z[3]*z[4]*z[5]*z[8] + 10688*z[3]*z[4]*z[5]*z[9] - 104000*z[3]*z[4]*z[5] - 6656*z[3]*z[4]*z[6]*z[7]*z[9] + 65024*z[3]*z[4]*z[6]*z[7] - 13312*z[3]*z[4]*z[6]*z[8]*z[9] + 130048*z[3]*z[4]*z[6]*z[8] + 21376*z[3]*z[4]*z[6]*z[9] - 208000*z[3]*z[4]*z[6] - 26624*z[3]*z[4]*z[7]*z[8]*z[9] + 260096*z[3]*z[4]*z[7]*z[8] + 42752*z[3]*z[4]*z[7]*z[9] - 416000*z[3]*z[4]*z[7] + 85504*z[3]*z[4]*z[8]*z[9] - 832000*z[3]*z[4]*z[8] - 104000*z[3]*z[4]*z[9] + 1010880*z[3]*z[4] + 2672*z[3]*z[5]*z[6]*z[9] - 26000*z[3]*z[5]*z[6] + 5344*z[3]*z[5]*z[7]*z[9] - 52000*z[3]*z[5]*z[7] + 10688*z[3]*z[5]*z[8]*z[9] - 104000*z[3]*z[5]*z[8] - 16920*z[3]*z[5]*z[9] + 163880*z[3]*z[5] + 10688*z[3]*z[6]*z[7]*z[9] - 104000*z[3]*z[6]*z[7] + 21376*z[3]*z[6]*z[8]*z[9] - 208000*z[3]*z[6]*z[8] - 33840*z[3]*z[6]*z[9] + 327760*z[3]*z[6] + 42752*z[3]*z[7]*z[8]*z[9] - 416000*z[3]*z[7]*z[8] - 67680*z[3]*z[7]*z[9] + 655520*z[3]*z[7] - 135360*z[3]*z[8]*z[9] + 1311040*z[3]*z[8] + 163880*z[3]*z[9] - 1585400*z[3] + 5344*z[4]*z[5]*z[6]*z[9] - 52000*z[4]*z[5]*z[6] + 10688*z[4]*z[5]*z[7]*z[9] - 104000*z[4]*z[5]*z[7] + 21376*z[4]*z[5]*z[8]*z[9] - 208000*z[4]*z[5]*z[8] - 33840*z[4]*z[5]*z[9] + 327760*z[4]*z[5] + 21376*z[4]*z[6]*z[7]*z[9] - 208000*z[4]*z[6]*z[7] + 42752*z[4]*z[6]*z[8]*z[9] - 416000*z[4]*z[6]*z[8] - 67680*z[4]*z[6]*z[9] + 655520*z[4]*z[6] + 85504*z[4]*z[7]*z[8]*z[9] - 832000*z[4]*z[7]*z[8] - 135360*z[4]*z[7]*z[9] + 1311040*z[4]*z[7] - 270720*z[4]*z[8]*z[9] + 2622080*z[4]*z[8] + 327760*z[4]*z[9] - 3170800*z[4] - 6500*z[5]*z[6]*z[9] + 63180*z[5]*z[6] - 13000*z[5]*z[7]*z[9] + 126360*z[5]*z[7] - 26000*z[5]*z[8]*z[9] + 252720*z[5]*z[8] + 40970*z[5]*z[9] - 396350*z[5] - 26000*z[6]*z[7]*z[9] + 252720*z[6]*z[7] - 52000*z[6]*z[8]*z[9] + 505440*z[6]*z[8] + 81940*z[6]*z[9] - 792700*z[6] - 104000*z[7]*z[8]*z[9] + 1010880*z[7]*z[8] + 163880*z[7]*z[9] - 1585400*z[7] + 327760*z[8]*z[9] - 3170800*z[8] - 396350*z[9] + 3830050)

In [351]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 12  # 
p = 1  # QAOA depth

hamiltonian_less_15_1_Erdös = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_Erdös_Straus_less_15_1))
final_circuit_less_15_1_Erdös, result_less_15_1_Erdös = qaoa(num_qubits, hamiltonian_less_15_1_Erdös, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.60064947 1.07226294]
Minimum expectation value: -2862746.21484375


In [357]:
top_solutions_less_15_1_Erdös = find_best_bitstrings(final_circuit_less_15_1_Erdös, hamiltonian_less_15_1_Erdös)

Top 5 bitstrings:
Bitstring: 011000100011, Cost: -7660100.0000, Count: 3
Bitstring: 000000001101, Cost: -7660100.0000, Count: 2
Bitstring: 001001100011, Cost: -7660100.0000, Count: 2
Bitstring: 000000001011, Cost: -7660100.0000, Count: 2
Bitstring: 000000000001, Cost: -7660100.0000, Count: 1


In [359]:
bitstring_to_pm1(top_solutions_less_15_1_Erdös, evaluate_hamiltonian_D_Erdös_Straus_less_15)

["Bitstring: ('011000100011', -7660100.0, 3), Evaluated cost: 0",
 "Bitstring: ('011000100011', -7660100.0, 3), Evaluated cost: 0",
 "Bitstring: ('011000100011', -7660100.0, 3), Evaluated cost: 0",
 "Bitstring: ('011000100011', -7660100.0, 3), Evaluated cost: 0",
 "Bitstring: ('011000100011', -7660100.0, 3), Evaluated cost: 0"]

# Pell's Equation $x^2-ny^2= 1$ Case $n=7$

In [353]:
def D_Pells_Equation(x,y):
    return(x**2-7*y**2-1)

In [354]:
paso1_D_Pells_Equation_less_15_1 = substitute_with_global_binary_symbols(D_Pells_Equation(x[1],x[2])**2, 4, base_name="b")
paso2_D_Pells_Equation_less_15_1 = remove_variable_exponents(paso1_D_Pells_Equation_less_15_1)
paso3_D_Pells_Equation_less_15_1 = substitute_with_spin_variables(paso2_D_Pells_Equation_less_15_1)

In [361]:
def evaluate_hamiltonian_D_Erdös_Pells_Equation_15(z):
    return(96*z[1]*z[2]*z[3]*z[4] - 180*z[1]*z[2]*z[3] - 360*z[1]*z[2]*z[4] - 14*z[1]*z[2]*z[5]*z[6] - 28*z[1]*z[2]*z[5]*z[7] - 56*z[1]*z[2]*z[5]*z[8] + 105*z[1]*z[2]*z[5] - 56*z[1]*z[2]*z[6]*z[7] - 112*z[1]*z[2]*z[6]*z[8] + 210*z[1]*z[2]*z[6] - 224*z[1]*z[2]*z[7]*z[8] + 420*z[1]*z[2]*z[7] + 840*z[1]*z[2]*z[8] - 627*z[1]*z[2] - 720*z[1]*z[3]*z[4] - 28*z[1]*z[3]*z[5]*z[6] - 56*z[1]*z[3]*z[5]*z[7] - 112*z[1]*z[3]*z[5]*z[8] + 210*z[1]*z[3]*z[5] - 112*z[1]*z[3]*z[6]*z[7] - 224*z[1]*z[3]*z[6]*z[8] + 420*z[1]*z[3]*z[6] - 448*z[1]*z[3]*z[7]*z[8] + 840*z[1]*z[3]*z[7] + 1680*z[1]*z[3]*z[8] - 1278*z[1]*z[3] - 56*z[1]*z[4]*z[5]*z[6] - 112*z[1]*z[4]*z[5]*z[7] - 224*z[1]*z[4]*z[5]*z[8] + 420*z[1]*z[4]*z[5] - 224*z[1]*z[4]*z[6]*z[7] - 448*z[1]*z[4]*z[6]*z[8] + 840*z[1]*z[4]*z[6] - 896*z[1]*z[4]*z[7]*z[8] + 1680*z[1]*z[4]*z[7] + 3360*z[1]*z[4]*z[8] - 2748*z[1]*z[4] + 105*z[1]*z[5]*z[6] + 210*z[1]*z[5]*z[7] + 420*z[1]*z[5]*z[8] - 1575*z[1]*z[5]/2 + 420*z[1]*z[6]*z[7] + 840*z[1]*z[6]*z[8] - 1575*z[1]*z[6] + 1680*z[1]*z[7]*z[8] - 3150*z[1]*z[7] - 6300*z[1]*z[8] + 6360*z[1] - 1440*z[2]*z[3]*z[4] - 56*z[2]*z[3]*z[5]*z[6] - 112*z[2]*z[3]*z[5]*z[7] - 224*z[2]*z[3]*z[5]*z[8] + 420*z[2]*z[3]*z[5] - 224*z[2]*z[3]*z[6]*z[7] - 448*z[2]*z[3]*z[6]*z[8] + 840*z[2]*z[3]*z[6] - 896*z[2]*z[3]*z[7]*z[8] + 1680*z[2]*z[3]*z[7] + 3360*z[2]*z[3]*z[8] - 2568*z[2]*z[3] - 112*z[2]*z[4]*z[5]*z[6] - 224*z[2]*z[4]*z[5]*z[7] - 448*z[2]*z[4]*z[5]*z[8] + 840*z[2]*z[4]*z[5] - 448*z[2]*z[4]*z[6]*z[7] - 896*z[2]*z[4]*z[6]*z[8] + 1680*z[2]*z[4]*z[6] - 1792*z[2]*z[4]*z[7]*z[8] + 3360*z[2]*z[4]*z[7] + 6720*z[2]*z[4]*z[8] - 5520*z[2]*z[4] + 210*z[2]*z[5]*z[6] + 420*z[2]*z[5]*z[7] + 840*z[2]*z[5]*z[8] - 1575*z[2]*z[5] + 840*z[2]*z[6]*z[7] + 1680*z[2]*z[6]*z[8] - 3150*z[2]*z[6] + 3360*z[2]*z[7]*z[8] - 6300*z[2]*z[7] - 12600*z[2]*z[8] + 12765*z[2] - 224*z[3]*z[4]*z[5]*z[6] - 448*z[3]*z[4]*z[5]*z[7] - 896*z[3]*z[4]*z[5]*z[8] + 1680*z[3]*z[4]*z[5] - 896*z[3]*z[4]*z[6]*z[7] - 1792*z[3]*z[4]*z[6]*z[8] + 3360*z[3]*z[4]*z[6] - 3584*z[3]*z[4]*z[7]*z[8] + 6720*z[3]*z[4]*z[7] + 13440*z[3]*z[4]*z[8] - 11232*z[3]*z[4] + 420*z[3]*z[5]*z[6] + 840*z[3]*z[5]*z[7] + 1680*z[3]*z[5]*z[8] - 3150*z[3]*z[5] + 1680*z[3]*z[6]*z[7] + 3360*z[3]*z[6]*z[8] - 6300*z[3]*z[6] + 6720*z[3]*z[7]*z[8] - 12600*z[3]*z[7] - 25200*z[3]*z[8] + 25890*z[3] + 840*z[4]*z[5]*z[6] + 1680*z[4]*z[5]*z[7] + 3360*z[4]*z[5]*z[8] - 6300*z[4]*z[5] + 3360*z[4]*z[6]*z[7] + 6720*z[4]*z[6]*z[8] - 12600*z[4]*z[6] + 13440*z[4]*z[7]*z[8] - 25200*z[4]*z[7] - 50400*z[4]*z[8] + 54660*z[4] + 4704*z[5]*z[6]*z[7]*z[8] - 8820*z[5]*z[6]*z[7] - 17640*z[5]*z[6]*z[8] + 21469*z[5]*z[6] - 35280*z[5]*z[7]*z[8] + 41762*z[5]*z[7] + 74116*z[5]*z[8] - 79800*z[5] - 70560*z[6]*z[7]*z[8] + 82936*z[6]*z[7] + 147056*z[6]*z[8] - 157395*z[6] + 284704*z[7]*z[8] - 297150*z[7] - 453180*z[8] + 948137/2)

In [371]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 8  # 
p = 1  # QAOA depth

hamiltonian_less_15_1_Pells = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_Pells_Equation_less_15_1))
final_circuit_less_15_1_Pells, result_less_15_1_Erdös = qaoa(num_qubits, hamiltonian_less_15_1_Pells, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [1.03952716 1.22076106]
Minimum expectation value: -107363.853515625


In [372]:
top_solutions_less_15_1_Pells = find_best_bitstrings(final_circuit_less_15_1_Pells, hamiltonian_less_15_1_Pells)

Top 5 bitstrings:
Bitstring: 00000001, Cost: -948137.0000, Count: 9
Bitstring: 00010011, Cost: -948135.0000, Count: 1
Bitstring: 00100101, Cost: -948121.0000, Count: 6
Bitstring: 00010001, Cost: -948087.0000, Count: 25
Bitstring: 00000011, Cost: -948073.0000, Count: 10


In [373]:
bitstring_to_pm1(top_solutions_less_15_1_Pells, evaluate_hamiltonian_D_Erdös_Pells_Equation_15)

["Bitstring: ('00000001', -948137.0, 9), Evaluated cost: 0.0",
 "Bitstring: ('00000001', -948137.0, 9), Evaluated cost: 1.0",
 "Bitstring: ('00000001', -948137.0, 9), Evaluated cost: 16.0",
 "Bitstring: ('00000001', -948137.0, 9), Evaluated cost: 49.0",
 "Bitstring: ('00000001', -948137.0, 9), Evaluated cost: 64.0"]

# Diophantine equation $w^3+x^3 = y^3 + z^3 $ The smallest nontrivial solution in positive integers is $12^3 + 13 = 93 + 10^3 = 1729$

In [407]:
def D_Hardy_Ramanujan_Number(w,x,y,z):
    return(w**3+x**3-y**3-z**3)

In [409]:
paso1_D_Hardy_Ramanujan_Number_less_15_1 = substitute_with_global_binary_symbols(D_Hardy_Ramanujan_Number(x[1],x[2],x[3],x[4])**2, 4, base_name="b")
paso2_D_Hardy_Ramanujan_Number_less_15_1 = remove_variable_exponents(paso1_D_Hardy_Ramanujan_Number_less_15_1)
paso3_D_Hardy_Ramanujan_Number_less_15_1 = substitute_with_spin_variables(paso2_D_Hardy_Ramanujan_Number_less_15_1)

In [414]:
print(paso3_D_Hardy_Ramanujan_Number_less_15_1)

-576*z_1*z_10*z_11*z_12*z_2*z_3 - 1152*z_1*z_10*z_11*z_12*z_2*z_4 + 2160*z_1*z_10*z_11*z_12*z_2 - 2304*z_1*z_10*z_11*z_12*z_3*z_4 + 4320*z_1*z_10*z_11*z_12*z_3 + 8640*z_1*z_10*z_11*z_12*z_4 - 11136*z_1*z_10*z_11*z_12 - 72*z_1*z_10*z_11*z_2*z_3*z_9 + 1080*z_1*z_10*z_11*z_2*z_3 - 144*z_1*z_10*z_11*z_2*z_4*z_9 + 2160*z_1*z_10*z_11*z_2*z_4 + 270*z_1*z_10*z_11*z_2*z_9 - 4050*z_1*z_10*z_11*z_2 - 288*z_1*z_10*z_11*z_3*z_4*z_9 + 4320*z_1*z_10*z_11*z_3*z_4 + 540*z_1*z_10*z_11*z_3*z_9 - 8100*z_1*z_10*z_11*z_3 + 1080*z_1*z_10*z_11*z_4*z_9 - 16200*z_1*z_10*z_11*z_4 - 1392*z_1*z_10*z_11*z_9 + 20880*z_1*z_10*z_11 - 144*z_1*z_10*z_12*z_2*z_3*z_9 + 2160*z_1*z_10*z_12*z_2*z_3 - 288*z_1*z_10*z_12*z_2*z_4*z_9 + 4320*z_1*z_10*z_12*z_2*z_4 + 540*z_1*z_10*z_12*z_2*z_9 - 8100*z_1*z_10*z_12*z_2 - 576*z_1*z_10*z_12*z_3*z_4*z_9 + 8640*z_1*z_10*z_12*z_3*z_4 + 1080*z_1*z_10*z_12*z_3*z_9 - 16200*z_1*z_10*z_12*z_3 + 2160*z_1*z_10*z_12*z_4*z_9 - 32400*z_1*z_10*z_12*z_4 - 2784*z_1*z_10*z_12*z_9 + 41760*z_1*z_10*z_12 

In [411]:
top_solutions_less_15_1_D_Hardy_Ramanujan_Number = find_best_bitstrings(final_circuit_less_15_1_D_Hardy_Ramanujan_Number, hamiltonian_less_15_1_D_Hardy_Ramanujan_Number)

Top 5 bitstrings:
Bitstring: 0101010101010101, Cost: -8761458.0000, Count: 1
Bitstring: 0000110100011101, Cost: -8761458.0000, Count: 1
Bitstring: 0000101010100001, Cost: -8761458.0000, Count: 1
Bitstring: 0100100101001001, Cost: -8761458.0000, Count: 1
Bitstring: 0110101101101011, Cost: -8761458.0000, Count: 1


In [410]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 16  # 
p = 1  # QAOA depth

hamiltonian_less_15_1_D_Hardy_Ramanujan_Number = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_Hardy_Ramanujan_Number_less_15_1))
final_circuit_less_15_1_D_Hardy_Ramanujan_Number, result_less_15_1_D_Hardy_Ramanujan_Number = qaoa(num_qubits, hamiltonian_less_15_1_D_Hardy_Ramanujan_Number, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.92188465 0.8265557 ]
Minimum expectation value: -151974.3984375


In [415]:
def evaluate_hamiltonian_Hardy_Ramanujan_Number_less_15(z):
    return(-576*z[1]*z[10]*z[11]*z[12]*z[2]*z[3] - 1152*z[1]*z[10]*z[11]*z[12]*z[2]*z[4] + 2160*z[1]*z[10]*z[11]*z[12]*z[2] - 2304*z[1]*z[10]*z[11]*z[12]*z[3]*z[4] + 4320*z[1]*z[10]*z[11]*z[12]*z[3] + 8640*z[1]*z[10]*z[11]*z[12]*z[4] - 11136*z[1]*z[10]*z[11]*z[12] - 72*z[1]*z[10]*z[11]*z[2]*z[3]*z[9] + 1080*z[1]*z[10]*z[11]*z[2]*z[3] - 144*z[1]*z[10]*z[11]*z[2]*z[4]*z[9] + 2160*z[1]*z[10]*z[11]*z[2]*z[4] + 270*z[1]*z[10]*z[11]*z[2]*z[9] - 4050*z[1]*z[10]*z[11]*z[2] - 288*z[1]*z[10]*z[11]*z[3]*z[4]*z[9] + 4320*z[1]*z[10]*z[11]*z[3]*z[4] + 540*z[1]*z[10]*z[11]*z[3]*z[9] - 8100*z[1]*z[10]*z[11]*z[3] + 1080*z[1]*z[10]*z[11]*z[4]*z[9] - 16200*z[1]*z[10]*z[11]*z[4] - 1392*z[1]*z[10]*z[11]*z[9] + 20880*z[1]*z[10]*z[11] - 144*z[1]*z[10]*z[12]*z[2]*z[3]*z[9] + 2160*z[1]*z[10]*z[12]*z[2]*z[3] - 288*z[1]*z[10]*z[12]*z[2]*z[4]*z[9] + 4320*z[1]*z[10]*z[12]*z[2]*z[4] + 540*z[1]*z[10]*z[12]*z[2]*z[9] - 8100*z[1]*z[10]*z[12]*z[2] - 576*z[1]*z[10]*z[12]*z[3]*z[4]*z[9] + 8640*z[1]*z[10]*z[12]*z[3]*z[4] + 1080*z[1]*z[10]*z[12]*z[3]*z[9] - 16200*z[1]*z[10]*z[12]*z[3] + 2160*z[1]*z[10]*z[12]*z[4]*z[9] - 32400*z[1]*z[10]*z[12]*z[4] - 2784*z[1]*z[10]*z[12]*z[9] + 41760*z[1]*z[10]*z[12] + 270*z[1]*z[10]*z[2]*z[3]*z[9] - 2766*z[1]*z[10]*z[2]*z[3] + 540*z[1]*z[10]*z[2]*z[4]*z[9] - 5532*z[1]*z[10]*z[2]*z[4] - 2025*z[1]*z[10]*z[2]*z[9]/2 + 20745*z[1]*z[10]*z[2]/2 + 1080*z[1]*z[10]*z[3]*z[4]*z[9] - 11064*z[1]*z[10]*z[3]*z[4] - 2025*z[1]*z[10]*z[3]*z[9] + 20745*z[1]*z[10]*z[3] - 4050*z[1]*z[10]*z[4]*z[9] + 41490*z[1]*z[10]*z[4] + 5220*z[1]*z[10]*z[9] - 53476*z[1]*z[10] - 288*z[1]*z[11]*z[12]*z[2]*z[3]*z[9] + 4320*z[1]*z[11]*z[12]*z[2]*z[3] - 576*z[1]*z[11]*z[12]*z[2]*z[4]*z[9] + 8640*z[1]*z[11]*z[12]*z[2]*z[4] + 1080*z[1]*z[11]*z[12]*z[2]*z[9] - 16200*z[1]*z[11]*z[12]*z[2] - 1152*z[1]*z[11]*z[12]*z[3]*z[4]*z[9] + 17280*z[1]*z[11]*z[12]*z[3]*z[4] + 2160*z[1]*z[11]*z[12]*z[3]*z[9] - 32400*z[1]*z[11]*z[12]*z[3] + 4320*z[1]*z[11]*z[12]*z[4]*z[9] - 64800*z[1]*z[11]*z[12]*z[4] - 5568*z[1]*z[11]*z[12]*z[9] + 83520*z[1]*z[11]*z[12] + 540*z[1]*z[11]*z[2]*z[3]*z[9] - 5388*z[1]*z[11]*z[2]*z[3] + 1080*z[1]*z[11]*z[2]*z[4]*z[9] - 10776*z[1]*z[11]*z[2]*z[4] - 2025*z[1]*z[11]*z[2]*z[9] + 20205*z[1]*z[11]*z[2] + 2160*z[1]*z[11]*z[3]*z[4]*z[9] - 21552*z[1]*z[11]*z[3]*z[4] - 4050*z[1]*z[11]*z[3]*z[9] + 40410*z[1]*z[11]*z[3] - 8100*z[1]*z[11]*z[4]*z[9] + 80820*z[1]*z[11]*z[4] + 10440*z[1]*z[11]*z[9] - 104168*z[1]*z[11] + 1080*z[1]*z[12]*z[2]*z[3]*z[9] - 9624*z[1]*z[12]*z[2]*z[3] + 2160*z[1]*z[12]*z[2]*z[4]*z[9] - 19248*z[1]*z[12]*z[2]*z[4] - 4050*z[1]*z[12]*z[2]*z[9] + 36090*z[1]*z[12]*z[2] + 4320*z[1]*z[12]*z[3]*z[4]*z[9] - 38496*z[1]*z[12]*z[3]*z[4] - 8100*z[1]*z[12]*z[3]*z[9] + 72180*z[1]*z[12]*z[3] - 16200*z[1]*z[12]*z[4]*z[9] + 144360*z[1]*z[12]*z[4] + 20880*z[1]*z[12]*z[9] - 186064*z[1]*z[12] - 72*z[1]*z[13]*z[14]*z[15]*z[2]*z[3] - 144*z[1]*z[13]*z[14]*z[15]*z[2]*z[4] + 270*z[1]*z[13]*z[14]*z[15]*z[2] - 288*z[1]*z[13]*z[14]*z[15]*z[3]*z[4] + 540*z[1]*z[13]*z[14]*z[15]*z[3] + 1080*z[1]*z[13]*z[14]*z[15]*z[4] - 1392*z[1]*z[13]*z[14]*z[15] - 144*z[1]*z[13]*z[14]*z[16]*z[2]*z[3] - 288*z[1]*z[13]*z[14]*z[16]*z[2]*z[4] + 540*z[1]*z[13]*z[14]*z[16]*z[2] - 576*z[1]*z[13]*z[14]*z[16]*z[3]*z[4] + 1080*z[1]*z[13]*z[14]*z[16]*z[3] + 2160*z[1]*z[13]*z[14]*z[16]*z[4] - 2784*z[1]*z[13]*z[14]*z[16] + 270*z[1]*z[13]*z[14]*z[2]*z[3] + 540*z[1]*z[13]*z[14]*z[2]*z[4] - 2025*z[1]*z[13]*z[14]*z[2]/2 + 1080*z[1]*z[13]*z[14]*z[3]*z[4] - 2025*z[1]*z[13]*z[14]*z[3] - 4050*z[1]*z[13]*z[14]*z[4] + 5220*z[1]*z[13]*z[14] - 288*z[1]*z[13]*z[15]*z[16]*z[2]*z[3] - 576*z[1]*z[13]*z[15]*z[16]*z[2]*z[4] + 1080*z[1]*z[13]*z[15]*z[16]*z[2] - 1152*z[1]*z[13]*z[15]*z[16]*z[3]*z[4] + 2160*z[1]*z[13]*z[15]*z[16]*z[3] + 4320*z[1]*z[13]*z[15]*z[16]*z[4] - 5568*z[1]*z[13]*z[15]*z[16] + 540*z[1]*z[13]*z[15]*z[2]*z[3] + 1080*z[1]*z[13]*z[15]*z[2]*z[4] - 2025*z[1]*z[13]*z[15]*z[2] + 2160*z[1]*z[13]*z[15]*z[3]*z[4] - 4050*z[1]*z[13]*z[15]*z[3] - 8100*z[1]*z[13]*z[15]*z[4] + 10440*z[1]*z[13]*z[15] + 1080*z[1]*z[13]*z[16]*z[2]*z[3] + 2160*z[1]*z[13]*z[16]*z[2]*z[4] - 4050*z[1]*z[13]*z[16]*z[2] + 4320*z[1]*z[13]*z[16]*z[3]*z[4] - 8100*z[1]*z[13]*z[16]*z[3] - 16200*z[1]*z[13]*z[16]*z[4] + 20880*z[1]*z[13]*z[16] - 1392*z[1]*z[13]*z[2]*z[3] - 2784*z[1]*z[13]*z[2]*z[4] + 5220*z[1]*z[13]*z[2] - 5568*z[1]*z[13]*z[3]*z[4] + 10440*z[1]*z[13]*z[3] + 20880*z[1]*z[13]*z[4] - 26912*z[1]*z[13] - 576*z[1]*z[14]*z[15]*z[16]*z[2]*z[3] - 1152*z[1]*z[14]*z[15]*z[16]*z[2]*z[4] + 2160*z[1]*z[14]*z[15]*z[16]*z[2] - 2304*z[1]*z[14]*z[15]*z[16]*z[3]*z[4] + 4320*z[1]*z[14]*z[15]*z[16]*z[3] + 8640*z[1]*z[14]*z[15]*z[16]*z[4] - 11136*z[1]*z[14]*z[15]*z[16] + 1080*z[1]*z[14]*z[15]*z[2]*z[3] + 2160*z[1]*z[14]*z[15]*z[2]*z[4] - 4050*z[1]*z[14]*z[15]*z[2] + 4320*z[1]*z[14]*z[15]*z[3]*z[4] - 8100*z[1]*z[14]*z[15]*z[3] - 16200*z[1]*z[14]*z[15]*z[4] + 20880*z[1]*z[14]*z[15] + 2160*z[1]*z[14]*z[16]*z[2]*z[3] + 4320*z[1]*z[14]*z[16]*z[2]*z[4] - 8100*z[1]*z[14]*z[16]*z[2] + 8640*z[1]*z[14]*z[16]*z[3]*z[4] - 16200*z[1]*z[14]*z[16]*z[3] - 32400*z[1]*z[14]*z[16]*z[4] + 41760*z[1]*z[14]*z[16] - 2766*z[1]*z[14]*z[2]*z[3] - 5532*z[1]*z[14]*z[2]*z[4] + 20745*z[1]*z[14]*z[2]/2 - 11064*z[1]*z[14]*z[3]*z[4] + 20745*z[1]*z[14]*z[3] + 41490*z[1]*z[14]*z[4] - 53476*z[1]*z[14] + 4320*z[1]*z[15]*z[16]*z[2]*z[3] + 8640*z[1]*z[15]*z[16]*z[2]*z[4] - 16200*z[1]*z[15]*z[16]*z[2] + 17280*z[1]*z[15]*z[16]*z[3]*z[4] - 32400*z[1]*z[15]*z[16]*z[3] - 64800*z[1]*z[15]*z[16]*z[4] + 83520*z[1]*z[15]*z[16] - 5388*z[1]*z[15]*z[2]*z[3] - 10776*z[1]*z[15]*z[2]*z[4] + 20205*z[1]*z[15]*z[2] - 21552*z[1]*z[15]*z[3]*z[4] + 40410*z[1]*z[15]*z[3] + 80820*z[1]*z[15]*z[4] - 104168*z[1]*z[15] - 9624*z[1]*z[16]*z[2]*z[3] - 19248*z[1]*z[16]*z[2]*z[4] + 36090*z[1]*z[16]*z[2] - 38496*z[1]*z[16]*z[3]*z[4] + 72180*z[1]*z[16]*z[3] + 144360*z[1]*z[16]*z[4] - 186064*z[1]*z[16] + 91200*z[1]*z[2]*z[3]*z[4] + 72*z[1]*z[2]*z[3]*z[5]*z[6]*z[7] + 144*z[1]*z[2]*z[3]*z[5]*z[6]*z[8] - 270*z[1]*z[2]*z[3]*z[5]*z[6] + 288*z[1]*z[2]*z[3]*z[5]*z[7]*z[8] - 540*z[1]*z[2]*z[3]*z[5]*z[7] - 1080*z[1]*z[2]*z[3]*z[5]*z[8] + 1392*z[1]*z[2]*z[3]*z[5] + 576*z[1]*z[2]*z[3]*z[6]*z[7]*z[8] - 1080*z[1]*z[2]*z[3]*z[6]*z[7] - 2160*z[1]*z[2]*z[3]*z[6]*z[8] + 2766*z[1]*z[2]*z[3]*z[6] - 4320*z[1]*z[2]*z[3]*z[7]*z[8] + 5388*z[1]*z[2]*z[3]*z[7] + 9624*z[1]*z[2]*z[3]*z[8] - 1392*z[1]*z[2]*z[3]*z[9] - 87750*z[1]*z[2]*z[3] + 144*z[1]*z[2]*z[4]*z[5]*z[6]*z[7] + 288*z[1]*z[2]*z[4]*z[5]*z[6]*z[8] - 540*z[1]*z[2]*z[4]*z[5]*z[6] + 576*z[1]*z[2]*z[4]*z[5]*z[7]*z[8] - 1080*z[1]*z[2]*z[4]*z[5]*z[7] - 2160*z[1]*z[2]*z[4]*z[5]*z[8] + 2784*z[1]*z[2]*z[4]*z[5] + 1152*z[1]*z[2]*z[4]*z[6]*z[7]*z[8] - 2160*z[1]*z[2]*z[4]*z[6]*z[7] - 4320*z[1]*z[2]*z[4]*z[6]*z[8] + 5532*z[1]*z[2]*z[4]*z[6] - 8640*z[1]*z[2]*z[4]*z[7]*z[8] + 10776*z[1]*z[2]*z[4]*z[7] + 19248*z[1]*z[2]*z[4]*z[8] - 2784*z[1]*z[2]*z[4]*z[9] - 132300*z[1]*z[2]*z[4] - 270*z[1]*z[2]*z[5]*z[6]*z[7] - 540*z[1]*z[2]*z[5]*z[6]*z[8] + 2025*z[1]*z[2]*z[5]*z[6]/2 - 1080*z[1]*z[2]*z[5]*z[7]*z[8] + 2025*z[1]*z[2]*z[5]*z[7] + 4050*z[1]*z[2]*z[5]*z[8] - 5220*z[1]*z[2]*z[5] - 2160*z[1]*z[2]*z[6]*z[7]*z[8] + 4050*z[1]*z[2]*z[6]*z[7] + 8100*z[1]*z[2]*z[6]*z[8] - 20745*z[1]*z[2]*z[6]/2 + 16200*z[1]*z[2]*z[7]*z[8] - 20205*z[1]*z[2]*z[7] - 36090*z[1]*z[2]*z[8] + 5220*z[1]*z[2]*z[9] + 120916*z[1]*z[2] + 288*z[1]*z[3]*z[4]*z[5]*z[6]*z[7] + 576*z[1]*z[3]*z[4]*z[5]*z[6]*z[8] - 1080*z[1]*z[3]*z[4]*z[5]*z[6] + 1152*z[1]*z[3]*z[4]*z[5]*z[7]*z[8] - 2160*z[1]*z[3]*z[4]*z[5]*z[7] - 4320*z[1]*z[3]*z[4]*z[5]*z[8] + 5568*z[1]*z[3]*z[4]*z[5] + 2304*z[1]*z[3]*z[4]*z[6]*z[7]*z[8] - 4320*z[1]*z[3]*z[4]*z[6]*z[7] - 8640*z[1]*z[3]*z[4]*z[6]*z[8] + 11064*z[1]*z[3]*z[4]*z[6] - 17280*z[1]*z[3]*z[4]*z[7]*z[8] + 21552*z[1]*z[3]*z[4]*z[7] + 38496*z[1]*z[3]*z[4]*z[8] - 5568*z[1]*z[3]*z[4]*z[9] - 243000*z[1]*z[3]*z[4] - 540*z[1]*z[3]*z[5]*z[6]*z[7] - 1080*z[1]*z[3]*z[5]*z[6]*z[8] + 2025*z[1]*z[3]*z[5]*z[6] - 2160*z[1]*z[3]*z[5]*z[7]*z[8] + 4050*z[1]*z[3]*z[5]*z[7] + 8100*z[1]*z[3]*z[5]*z[8] - 10440*z[1]*z[3]*z[5] - 4320*z[1]*z[3]*z[6]*z[7]*z[8] + 8100*z[1]*z[3]*z[6]*z[7] + 16200*z[1]*z[3]*z[6]*z[8] - 20745*z[1]*z[3]*z[6] + 32400*z[1]*z[3]*z[7]*z[8] - 40410*z[1]*z[3]*z[7] - 72180*z[1]*z[3]*z[8] + 10440*z[1]*z[3]*z[9] + 215432*z[1]*z[3] - 1080*z[1]*z[4]*z[5]*z[6]*z[7] - 2160*z[1]*z[4]*z[5]*z[6]*z[8] + 4050*z[1]*z[4]*z[5]*z[6] - 4320*z[1]*z[4]*z[5]*z[7]*z[8] + 8100*z[1]*z[4]*z[5]*z[7] + 16200*z[1]*z[4]*z[5]*z[8] - 20880*z[1]*z[4]*z[5] - 8640*z[1]*z[4]*z[6]*z[7]*z[8] + 16200*z[1]*z[4]*z[6]*z[7] + 32400*z[1]*z[4]*z[6]*z[8] - 41490*z[1]*z[4]*z[6] + 64800*z[1]*z[4]*z[7]*z[8] - 80820*z[1]*z[4]*z[7] - 144360*z[1]*z[4]*z[8] + 20880*z[1]*z[4]*z[9] + 254224*z[1]*z[4] + 1392*z[1]*z[5]*z[6]*z[7] + 2784*z[1]*z[5]*z[6]*z[8] - 5220*z[1]*z[5]*z[6] + 5568*z[1]*z[5]*z[7]*z[8] - 10440*z[1]*z[5]*z[7] - 20880*z[1]*z[5]*z[8] + 26912*z[1]*z[5] + 11136*z[1]*z[6]*z[7]*z[8] - 20880*z[1]*z[6]*z[7] - 41760*z[1]*z[6]*z[8] + 53476*z[1]*z[6] - 83520*z[1]*z[7]*z[8] + 104168*z[1]*z[7] + 186064*z[1]*z[8] - 26912*z[1]*z[9] - 435645*z[1]/2 + 576*z[10]*z[11]*z[12]*z[13]*z[14]*z[15] + 1152*z[10]*z[11]*z[12]*z[13]*z[14]*z[16] - 2160*z[10]*z[11]*z[12]*z[13]*z[14] + 2304*z[10]*z[11]*z[12]*z[13]*z[15]*z[16] - 4320*z[10]*z[11]*z[12]*z[13]*z[15] - 8640*z[10]*z[11]*z[12]*z[13]*z[16] + 11136*z[10]*z[11]*z[12]*z[13] + 4608*z[10]*z[11]*z[12]*z[14]*z[15]*z[16] - 8640*z[10]*z[11]*z[12]*z[14]*z[15] - 17280*z[10]*z[11]*z[12]*z[14]*z[16] + 22128*z[10]*z[11]*z[12]*z[14] - 34560*z[10]*z[11]*z[12]*z[15]*z[16] + 43104*z[10]*z[11]*z[12]*z[15] + 76992*z[10]*z[11]*z[12]*z[16] - 4608*z[10]*z[11]*z[12]*z[2]*z[3]*z[4] + 8640*z[10]*z[11]*z[12]*z[2]*z[3] + 17280*z[10]*z[11]*z[12]*z[2]*z[4] - 22128*z[10]*z[11]*z[12]*z[2] + 34560*z[10]*z[11]*z[12]*z[3]*z[4] - 43104*z[10]*z[11]*z[12]*z[3] - 76992*z[10]*z[11]*z[12]*z[4] - 576*z[10]*z[11]*z[12]*z[5]*z[6]*z[7] - 1152*z[10]*z[11]*z[12]*z[5]*z[6]*z[8] + 2160*z[10]*z[11]*z[12]*z[5]*z[6] - 2304*z[10]*z[11]*z[12]*z[5]*z[7]*z[8] + 4320*z[10]*z[11]*z[12]*z[5]*z[7] + 8640*z[10]*z[11]*z[12]*z[5]*z[8] - 11136*z[10]*z[11]*z[12]*z[5] - 4608*z[10]*z[11]*z[12]*z[6]*z[7]*z[8] + 8640*z[10]*z[11]*z[12]*z[6]*z[7] + 17280*z[10]*z[11]*z[12]*z[6]*z[8] - 22128*z[10]*z[11]*z[12]*z[6] + 34560*z[10]*z[11]*z[12]*z[7]*z[8] - 43104*z[10]*z[11]*z[12]*z[7] - 76992*z[10]*z[11]*z[12]*z[8] + 91200*z[10]*z[11]*z[12]*z[9] - 475200*z[10]*z[11]*z[12] + 72*z[10]*z[11]*z[13]*z[14]*z[15]*z[9] - 1080*z[10]*z[11]*z[13]*z[14]*z[15] + 144*z[10]*z[11]*z[13]*z[14]*z[16]*z[9] - 2160*z[10]*z[11]*z[13]*z[14]*z[16] - 270*z[10]*z[11]*z[13]*z[14]*z[9] + 4050*z[10]*z[11]*z[13]*z[14] + 288*z[10]*z[11]*z[13]*z[15]*z[16]*z[9] - 4320*z[10]*z[11]*z[13]*z[15]*z[16] - 540*z[10]*z[11]*z[13]*z[15]*z[9] + 8100*z[10]*z[11]*z[13]*z[15] - 1080*z[10]*z[11]*z[13]*z[16]*z[9] + 16200*z[10]*z[11]*z[13]*z[16] + 1392*z[10]*z[11]*z[13]*z[9] - 20880*z[10]*z[11]*z[13] + 576*z[10]*z[11]*z[14]*z[15]*z[16]*z[9] - 8640*z[10]*z[11]*z[14]*z[15]*z[16] - 1080*z[10]*z[11]*z[14]*z[15]*z[9] + 16200*z[10]*z[11]*z[14]*z[15] - 2160*z[10]*z[11]*z[14]*z[16]*z[9] + 32400*z[10]*z[11]*z[14]*z[16] + 2766*z[10]*z[11]*z[14]*z[9] - 41490*z[10]*z[11]*z[14] - 4320*z[10]*z[11]*z[15]*z[16]*z[9] + 64800*z[10]*z[11]*z[15]*z[16] + 5388*z[10]*z[11]*z[15]*z[9] - 80820*z[10]*z[11]*z[15] + 9624*z[10]*z[11]*z[16]*z[9] - 144360*z[10]*z[11]*z[16] - 576*z[10]*z[11]*z[2]*z[3]*z[4]*z[9] + 8640*z[10]*z[11]*z[2]*z[3]*z[4] + 1080*z[10]*z[11]*z[2]*z[3]*z[9] - 16200*z[10]*z[11]*z[2]*z[3] + 2160*z[10]*z[11]*z[2]*z[4]*z[9] - 32400*z[10]*z[11]*z[2]*z[4] - 2766*z[10]*z[11]*z[2]*z[9] + 41490*z[10]*z[11]*z[2] + 4320*z[10]*z[11]*z[3]*z[4]*z[9] - 64800*z[10]*z[11]*z[3]*z[4] - 5388*z[10]*z[11]*z[3]*z[9] + 80820*z[10]*z[11]*z[3] - 9624*z[10]*z[11]*z[4]*z[9] + 144360*z[10]*z[11]*z[4] - 72*z[10]*z[11]*z[5]*z[6]*z[7]*z[9] + 1080*z[10]*z[11]*z[5]*z[6]*z[7] - 144*z[10]*z[11]*z[5]*z[6]*z[8]*z[9] + 2160*z[10]*z[11]*z[5]*z[6]*z[8] + 270*z[10]*z[11]*z[5]*z[6]*z[9] - 4050*z[10]*z[11]*z[5]*z[6] - 288*z[10]*z[11]*z[5]*z[7]*z[8]*z[9] + 4320*z[10]*z[11]*z[5]*z[7]*z[8] + 540*z[10]*z[11]*z[5]*z[7]*z[9] - 8100*z[10]*z[11]*z[5]*z[7] + 1080*z[10]*z[11]*z[5]*z[8]*z[9] - 16200*z[10]*z[11]*z[5]*z[8] - 1392*z[10]*z[11]*z[5]*z[9] + 20880*z[10]*z[11]*z[5] - 576*z[10]*z[11]*z[6]*z[7]*z[8]*z[9] + 8640*z[10]*z[11]*z[6]*z[7]*z[8] + 1080*z[10]*z[11]*z[6]*z[7]*z[9] - 16200*z[10]*z[11]*z[6]*z[7] + 2160*z[10]*z[11]*z[6]*z[8]*z[9] - 32400*z[10]*z[11]*z[6]*z[8] - 2766*z[10]*z[11]*z[6]*z[9] + 41490*z[10]*z[11]*z[6] + 4320*z[10]*z[11]*z[7]*z[8]*z[9] - 64800*z[10]*z[11]*z[7]*z[8] - 5388*z[10]*z[11]*z[7]*z[9] + 80820*z[10]*z[11]*z[7] - 9624*z[10]*z[11]*z[8]*z[9] + 144360*z[10]*z[11]*z[8] - 87750*z[10]*z[11]*z[9] + 417574*z[10]*z[11] + 144*z[10]*z[12]*z[13]*z[14]*z[15]*z[9] - 2160*z[10]*z[12]*z[13]*z[14]*z[15] + 288*z[10]*z[12]*z[13]*z[14]*z[16]*z[9] - 4320*z[10]*z[12]*z[13]*z[14]*z[16] - 540*z[10]*z[12]*z[13]*z[14]*z[9] + 8100*z[10]*z[12]*z[13]*z[14] + 576*z[10]*z[12]*z[13]*z[15]*z[16]*z[9] - 8640*z[10]*z[12]*z[13]*z[15]*z[16] - 1080*z[10]*z[12]*z[13]*z[15]*z[9] + 16200*z[10]*z[12]*z[13]*z[15] - 2160*z[10]*z[12]*z[13]*z[16]*z[9] + 32400*z[10]*z[12]*z[13]*z[16] + 2784*z[10]*z[12]*z[13]*z[9] - 41760*z[10]*z[12]*z[13] + 1152*z[10]*z[12]*z[14]*z[15]*z[16]*z[9] - 17280*z[10]*z[12]*z[14]*z[15]*z[16] - 2160*z[10]*z[12]*z[14]*z[15]*z[9] + 32400*z[10]*z[12]*z[14]*z[15] - 4320*z[10]*z[12]*z[14]*z[16]*z[9] + 64800*z[10]*z[12]*z[14]*z[16] + 5532*z[10]*z[12]*z[14]*z[9] - 82980*z[10]*z[12]*z[14] - 8640*z[10]*z[12]*z[15]*z[16]*z[9] + 129600*z[10]*z[12]*z[15]*z[16] + 10776*z[10]*z[12]*z[15]*z[9] - 161640*z[10]*z[12]*z[15] + 19248*z[10]*z[12]*z[16]*z[9] - 288720*z[10]*z[12]*z[16] - 1152*z[10]*z[12]*z[2]*z[3]*z[4]*z[9] + 17280*z[10]*z[12]*z[2]*z[3]*z[4] + 2160*z[10]*z[12]*z[2]*z[3]*z[9] - 32400*z[10]*z[12]*z[2]*z[3] + 4320*z[10]*z[12]*z[2]*z[4]*z[9] - 64800*z[10]*z[12]*z[2]*z[4] - 5532*z[10]*z[12]*z[2]*z[9] + 82980*z[10]*z[12]*z[2] + 8640*z[10]*z[12]*z[3]*z[4]*z[9] - 129600*z[10]*z[12]*z[3]*z[4] - 10776*z[10]*z[12]*z[3]*z[9] + 161640*z[10]*z[12]*z[3] - 19248*z[10]*z[12]*z[4]*z[9] + 288720*z[10]*z[12]*z[4] - 144*z[10]*z[12]*z[5]*z[6]*z[7]*z[9] + 2160*z[10]*z[12]*z[5]*z[6]*z[7] - 288*z[10]*z[12]*z[5]*z[6]*z[8]*z[9] + 4320*z[10]*z[12]*z[5]*z[6]*z[8] + 540*z[10]*z[12]*z[5]*z[6]*z[9] - 8100*z[10]*z[12]*z[5]*z[6] - 576*z[10]*z[12]*z[5]*z[7]*z[8]*z[9] + 8640*z[10]*z[12]*z[5]*z[7]*z[8] + 1080*z[10]*z[12]*z[5]*z[7]*z[9] - 16200*z[10]*z[12]*z[5]*z[7] + 2160*z[10]*z[12]*z[5]*z[8]*z[9] - 32400*z[10]*z[12]*z[5]*z[8] - 2784*z[10]*z[12]*z[5]*z[9] + 41760*z[10]*z[12]*z[5] - 1152*z[10]*z[12]*z[6]*z[7]*z[8]*z[9] + 17280*z[10]*z[12]*z[6]*z[7]*z[8] + 2160*z[10]*z[12]*z[6]*z[7]*z[9] - 32400*z[10]*z[12]*z[6]*z[7] + 4320*z[10]*z[12]*z[6]*z[8]*z[9] - 64800*z[10]*z[12]*z[6]*z[8] - 5532*z[10]*z[12]*z[6]*z[9] + 82980*z[10]*z[12]*z[6] + 8640*z[10]*z[12]*z[7]*z[8]*z[9] - 129600*z[10]*z[12]*z[7]*z[8] - 10776*z[10]*z[12]*z[7]*z[9] + 161640*z[10]*z[12]*z[7] - 19248*z[10]*z[12]*z[8]*z[9] + 288720*z[10]*z[12]*z[8] - 132300*z[10]*z[12]*z[9] + 484748*z[10]*z[12] - 270*z[10]*z[13]*z[14]*z[15]*z[9] + 2766*z[10]*z[13]*z[14]*z[15] - 540*z[10]*z[13]*z[14]*z[16]*z[9] + 5532*z[10]*z[13]*z[14]*z[16] + 2025*z[10]*z[13]*z[14]*z[9]/2 - 20745*z[10]*z[13]*z[14]/2 - 1080*z[10]*z[13]*z[15]*z[16]*z[9] + 11064*z[10]*z[13]*z[15]*z[16] + 2025*z[10]*z[13]*z[15]*z[9] - 20745*z[10]*z[13]*z[15] + 4050*z[10]*z[13]*z[16]*z[9] - 41490*z[10]*z[13]*z[16] - 5220*z[10]*z[13]*z[9] + 53476*z[10]*z[13] - 2160*z[10]*z[14]*z[15]*z[16]*z[9] + 22128*z[10]*z[14]*z[15]*z[16] + 4050*z[10]*z[14]*z[15]*z[9] - 41490*z[10]*z[14]*z[15] + 8100*z[10]*z[14]*z[16]*z[9] - 82980*z[10]*z[14]*z[16] - 20745*z[10]*z[14]*z[9]/2 + 212521*z[10]*z[14]/2 + 16200*z[10]*z[15]*z[16]*z[9] - 165960*z[10]*z[15]*z[16] - 20205*z[10]*z[15]*z[9] + 206989*z[10]*z[15] - 36090*z[10]*z[16]*z[9] + 369722*z[10]*z[16] + 2160*z[10]*z[2]*z[3]*z[4]*z[9] - 22128*z[10]*z[2]*z[3]*z[4] - 4050*z[10]*z[2]*z[3]*z[9] + 41490*z[10]*z[2]*z[3] - 8100*z[10]*z[2]*z[4]*z[9] + 82980*z[10]*z[2]*z[4] + 20745*z[10]*z[2]*z[9]/2 - 212521*z[10]*z[2]/2 - 16200*z[10]*z[3]*z[4]*z[9] + 165960*z[10]*z[3]*z[4] + 20205*z[10]*z[3]*z[9] - 206989*z[10]*z[3] + 36090*z[10]*z[4]*z[9] - 369722*z[10]*z[4] + 270*z[10]*z[5]*z[6]*z[7]*z[9] - 2766*z[10]*z[5]*z[6]*z[7] + 540*z[10]*z[5]*z[6]*z[8]*z[9] - 5532*z[10]*z[5]*z[6]*z[8] - 2025*z[10]*z[5]*z[6]*z[9]/2 + 20745*z[10]*z[5]*z[6]/2 + 1080*z[10]*z[5]*z[7]*z[8]*z[9] - 11064*z[10]*z[5]*z[7]*z[8] - 2025*z[10]*z[5]*z[7]*z[9] + 20745*z[10]*z[5]*z[7] - 4050*z[10]*z[5]*z[8]*z[9] + 41490*z[10]*z[5]*z[8] + 5220*z[10]*z[5]*z[9] - 53476*z[10]*z[5] + 2160*z[10]*z[6]*z[7]*z[8]*z[9] - 22128*z[10]*z[6]*z[7]*z[8] - 4050*z[10]*z[6]*z[7]*z[9] + 41490*z[10]*z[6]*z[7] - 8100*z[10]*z[6]*z[8]*z[9] + 82980*z[10]*z[6]*z[8] + 20745*z[10]*z[6]*z[9]/2 - 212521*z[10]*z[6]/2 - 16200*z[10]*z[7]*z[8]*z[9] + 165960*z[10]*z[7]*z[8] + 20205*z[10]*z[7]*z[9] - 206989*z[10]*z[7] + 36090*z[10]*z[8]*z[9] - 369722*z[10]*z[8] + 120916*z[10]*z[9] - 412020*z[10] + 288*z[11]*z[12]*z[13]*z[14]*z[15]*z[9] - 4320*z[11]*z[12]*z[13]*z[14]*z[15] + 576*z[11]*z[12]*z[13]*z[14]*z[16]*z[9] - 8640*z[11]*z[12]*z[13]*z[14]*z[16] - 1080*z[11]*z[12]*z[13]*z[14]*z[9] + 16200*z[11]*z[12]*z[13]*z[14] + 1152*z[11]*z[12]*z[13]*z[15]*z[16]*z[9] - 17280*z[11]*z[12]*z[13]*z[15]*z[16] - 2160*z[11]*z[12]*z[13]*z[15]*z[9] + 32400*z[11]*z[12]*z[13]*z[15] - 4320*z[11]*z[12]*z[13]*z[16]*z[9] + 64800*z[11]*z[12]*z[13]*z[16] + 5568*z[11]*z[12]*z[13]*z[9] - 83520*z[11]*z[12]*z[13] + 2304*z[11]*z[12]*z[14]*z[15]*z[16]*z[9] - 34560*z[11]*z[12]*z[14]*z[15]*z[16] - 4320*z[11]*z[12]*z[14]*z[15]*z[9] + 64800*z[11]*z[12]*z[14]*z[15] - 8640*z[11]*z[12]*z[14]*z[16]*z[9] + 129600*z[11]*z[12]*z[14]*z[16] + 11064*z[11]*z[12]*z[14]*z[9] - 165960*z[11]*z[12]*z[14] - 17280*z[11]*z[12]*z[15]*z[16]*z[9] + 259200*z[11]*z[12]*z[15]*z[16] + 21552*z[11]*z[12]*z[15]*z[9] - 323280*z[11]*z[12]*z[15] + 38496*z[11]*z[12]*z[16]*z[9] - 577440*z[11]*z[12]*z[16] - 2304*z[11]*z[12]*z[2]*z[3]*z[4]*z[9] + 34560*z[11]*z[12]*z[2]*z[3]*z[4] + 4320*z[11]*z[12]*z[2]*z[3]*z[9] - 64800*z[11]*z[12]*z[2]*z[3] + 8640*z[11]*z[12]*z[2]*z[4]*z[9] - 129600*z[11]*z[12]*z[2]*z[4] - 11064*z[11]*z[12]*z[2]*z[9] + 165960*z[11]*z[12]*z[2] + 17280*z[11]*z[12]*z[3]*z[4]*z[9] - 259200*z[11]*z[12]*z[3]*z[4] - 21552*z[11]*z[12]*z[3]*z[9] + 323280*z[11]*z[12]*z[3] - 38496*z[11]*z[12]*z[4]*z[9] + 577440*z[11]*z[12]*z[4] - 288*z[11]*z[12]*z[5]*z[6]*z[7]*z[9] + 4320*z[11]*z[12]*z[5]*z[6]*z[7] - 576*z[11]*z[12]*z[5]*z[6]*z[8]*z[9] + 8640*z[11]*z[12]*z[5]*z[6]*z[8] + 1080*z[11]*z[12]*z[5]*z[6]*z[9] - 16200*z[11]*z[12]*z[5]*z[6] - 1152*z[11]*z[12]*z[5]*z[7]*z[8]*z[9] + 17280*z[11]*z[12]*z[5]*z[7]*z[8] + 2160*z[11]*z[12]*z[5]*z[7]*z[9] - 32400*z[11]*z[12]*z[5]*z[7] + 4320*z[11]*z[12]*z[5]*z[8]*z[9] - 64800*z[11]*z[12]*z[5]*z[8] - 5568*z[11]*z[12]*z[5]*z[9] + 83520*z[11]*z[12]*z[5] - 2304*z[11]*z[12]*z[6]*z[7]*z[8]*z[9] + 34560*z[11]*z[12]*z[6]*z[7]*z[8] + 4320*z[11]*z[12]*z[6]*z[7]*z[9] - 64800*z[11]*z[12]*z[6]*z[7] + 8640*z[11]*z[12]*z[6]*z[8]*z[9] - 129600*z[11]*z[12]*z[6]*z[8] - 11064*z[11]*z[12]*z[6]*z[9] + 165960*z[11]*z[12]*z[6] + 17280*z[11]*z[12]*z[7]*z[8]*z[9] - 259200*z[11]*z[12]*z[7]*z[8] - 21552*z[11]*z[12]*z[7]*z[9] + 323280*z[11]*z[12]*z[7] - 38496*z[11]*z[12]*z[8]*z[9] + 577440*z[11]*z[12]*z[8] - 243000*z[11]*z[12]*z[9] + 788536*z[11]*z[12] - 540*z[11]*z[13]*z[14]*z[15]*z[9] + 5388*z[11]*z[13]*z[14]*z[15] - 1080*z[11]*z[13]*z[14]*z[16]*z[9] + 10776*z[11]*z[13]*z[14]*z[16] + 2025*z[11]*z[13]*z[14]*z[9] - 20205*z[11]*z[13]*z[14] - 2160*z[11]*z[13]*z[15]*z[16]*z[9] + 21552*z[11]*z[13]*z[15]*z[16] + 4050*z[11]*z[13]*z[15]*z[9] - 40410*z[11]*z[13]*z[15] + 8100*z[11]*z[13]*z[16]*z[9] - 80820*z[11]*z[13]*z[16] - 10440*z[11]*z[13]*z[9] + 104168*z[11]*z[13] - 4320*z[11]*z[14]*z[15]*z[16]*z[9] + 43104*z[11]*z[14]*z[15]*z[16] + 8100*z[11]*z[14]*z[15]*z[9] - 80820*z[11]*z[14]*z[15] + 16200*z[11]*z[14]*z[16]*z[9] - 161640*z[11]*z[14]*z[16] - 20745*z[11]*z[14]*z[9] + 206989*z[11]*z[14] + 32400*z[11]*z[15]*z[16]*z[9] - 323280*z[11]*z[15]*z[16] - 40410*z[11]*z[15]*z[9] + 403202*z[11]*z[15] - 72180*z[11]*z[16]*z[9] + 720196*z[11]*z[16] + 4320*z[11]*z[2]*z[3]*z[4]*z[9] - 43104*z[11]*z[2]*z[3]*z[4] - 8100*z[11]*z[2]*z[3]*z[9] + 80820*z[11]*z[2]*z[3] - 16200*z[11]*z[2]*z[4]*z[9] + 161640*z[11]*z[2]*z[4] + 20745*z[11]*z[2]*z[9] - 206989*z[11]*z[2] - 32400*z[11]*z[3]*z[4]*z[9] + 323280*z[11]*z[3]*z[4] + 40410*z[11]*z[3]*z[9] - 403202*z[11]*z[3] + 72180*z[11]*z[4]*z[9] - 720196*z[11]*z[4] + 540*z[11]*z[5]*z[6]*z[7]*z[9] - 5388*z[11]*z[5]*z[6]*z[7] + 1080*z[11]*z[5]*z[6]*z[8]*z[9] - 10776*z[11]*z[5]*z[6]*z[8] - 2025*z[11]*z[5]*z[6]*z[9] + 20205*z[11]*z[5]*z[6] + 2160*z[11]*z[5]*z[7]*z[8]*z[9] - 21552*z[11]*z[5]*z[7]*z[8] - 4050*z[11]*z[5]*z[7]*z[9] + 40410*z[11]*z[5]*z[7] - 8100*z[11]*z[5]*z[8]*z[9] + 80820*z[11]*z[5]*z[8] + 10440*z[11]*z[5]*z[9] - 104168*z[11]*z[5] + 4320*z[11]*z[6]*z[7]*z[8]*z[9] - 43104*z[11]*z[6]*z[7]*z[8] - 8100*z[11]*z[6]*z[7]*z[9] + 80820*z[11]*z[6]*z[7] - 16200*z[11]*z[6]*z[8]*z[9] + 161640*z[11]*z[6]*z[8] + 20745*z[11]*z[6]*z[9] - 206989*z[11]*z[6] - 32400*z[11]*z[7]*z[8]*z[9] + 323280*z[11]*z[7]*z[8] + 40410*z[11]*z[7]*z[9] - 403202*z[11]*z[7] + 72180*z[11]*z[8]*z[9] - 720196*z[11]*z[8] + 215432*z[11]*z[9] - 651240*z[11] - 1080*z[12]*z[13]*z[14]*z[15]*z[9] + 9624*z[12]*z[13]*z[14]*z[15] - 2160*z[12]*z[13]*z[14]*z[16]*z[9] + 19248*z[12]*z[13]*z[14]*z[16] + 4050*z[12]*z[13]*z[14]*z[9] - 36090*z[12]*z[13]*z[14] - 4320*z[12]*z[13]*z[15]*z[16]*z[9] + 38496*z[12]*z[13]*z[15]*z[16] + 8100*z[12]*z[13]*z[15]*z[9] - 72180*z[12]*z[13]*z[15] + 16200*z[12]*z[13]*z[16]*z[9] - 144360*z[12]*z[13]*z[16] - 20880*z[12]*z[13]*z[9] + 186064*z[12]*z[13] - 8640*z[12]*z[14]*z[15]*z[16]*z[9] + 76992*z[12]*z[14]*z[15]*z[16] + 16200*z[12]*z[14]*z[15]*z[9] - 144360*z[12]*z[14]*z[15] + 32400*z[12]*z[14]*z[16]*z[9] - 288720*z[12]*z[14]*z[16] - 41490*z[12]*z[14]*z[9] + 369722*z[12]*z[14] + 64800*z[12]*z[15]*z[16]*z[9] - 577440*z[12]*z[15]*z[16] - 80820*z[12]*z[15]*z[9] + 720196*z[12]*z[15] - 144360*z[12]*z[16]*z[9] + 1286408*z[12]*z[16] + 8640*z[12]*z[2]*z[3]*z[4]*z[9] - 76992*z[12]*z[2]*z[3]*z[4] - 16200*z[12]*z[2]*z[3]*z[9] + 144360*z[12]*z[2]*z[3] - 32400*z[12]*z[2]*z[4]*z[9] + 288720*z[12]*z[2]*z[4] + 41490*z[12]*z[2]*z[9] - 369722*z[12]*z[2] - 64800*z[12]*z[3]*z[4]*z[9] + 577440*z[12]*z[3]*z[4] + 80820*z[12]*z[3]*z[9] - 720196*z[12]*z[3] + 144360*z[12]*z[4]*z[9] - 1286408*z[12]*z[4] + 1080*z[12]*z[5]*z[6]*z[7]*z[9] - 9624*z[12]*z[5]*z[6]*z[7] + 2160*z[12]*z[5]*z[6]*z[8]*z[9] - 19248*z[12]*z[5]*z[6]*z[8] - 4050*z[12]*z[5]*z[6]*z[9] + 36090*z[12]*z[5]*z[6] + 4320*z[12]*z[5]*z[7]*z[8]*z[9] - 38496*z[12]*z[5]*z[7]*z[8] - 8100*z[12]*z[5]*z[7]*z[9] + 72180*z[12]*z[5]*z[7] - 16200*z[12]*z[5]*z[8]*z[9] + 144360*z[12]*z[5]*z[8] + 20880*z[12]*z[5]*z[9] - 186064*z[12]*z[5] + 8640*z[12]*z[6]*z[7]*z[8]*z[9] - 76992*z[12]*z[6]*z[7]*z[8] - 16200*z[12]*z[6]*z[7]*z[9] + 144360*z[12]*z[6]*z[7] - 32400*z[12]*z[6]*z[8]*z[9] + 288720*z[12]*z[6]*z[8] + 41490*z[12]*z[6]*z[9] - 369722*z[12]*z[6] - 64800*z[12]*z[7]*z[8]*z[9] + 577440*z[12]*z[7]*z[8] + 80820*z[12]*z[7]*z[9] - 720196*z[12]*z[7] + 144360*z[12]*z[8]*z[9] - 1286408*z[12]*z[8] + 254224*z[12]*z[9] - 438480*z[12] + 91200*z[13]*z[14]*z[15]*z[16] - 576*z[13]*z[14]*z[15]*z[2]*z[3]*z[4] + 1080*z[13]*z[14]*z[15]*z[2]*z[3] + 2160*z[13]*z[14]*z[15]*z[2]*z[4] - 2766*z[13]*z[14]*z[15]*z[2] + 4320*z[13]*z[14]*z[15]*z[3]*z[4] - 5388*z[13]*z[14]*z[15]*z[3] - 9624*z[13]*z[14]*z[15]*z[4] - 72*z[13]*z[14]*z[15]*z[5]*z[6]*z[7] - 144*z[13]*z[14]*z[15]*z[5]*z[6]*z[8] + 270*z[13]*z[14]*z[15]*z[5]*z[6] - 288*z[13]*z[14]*z[15]*z[5]*z[7]*z[8] + 540*z[13]*z[14]*z[15]*z[5]*z[7] + 1080*z[13]*z[14]*z[15]*z[5]*z[8] - 1392*z[13]*z[14]*z[15]*z[5] - 576*z[13]*z[14]*z[15]*z[6]*z[7]*z[8] + 1080*z[13]*z[14]*z[15]*z[6]*z[7] + 2160*z[13]*z[14]*z[15]*z[6]*z[8] - 2766*z[13]*z[14]*z[15]*z[6] + 4320*z[13]*z[14]*z[15]*z[7]*z[8] - 5388*z[13]*z[14]*z[15]*z[7] - 9624*z[13]*z[14]*z[15]*z[8] + 1392*z[13]*z[14]*z[15]*z[9] - 87750*z[13]*z[14]*z[15] - 1152*z[13]*z[14]*z[16]*z[2]*z[3]*z[4] + 2160*z[13]*z[14]*z[16]*z[2]*z[3] + 4320*z[13]*z[14]*z[16]*z[2]*z[4] - 5532*z[13]*z[14]*z[16]*z[2] + 8640*z[13]*z[14]*z[16]*z[3]*z[4] - 10776*z[13]*z[14]*z[16]*z[3] - 19248*z[13]*z[14]*z[16]*z[4] - 144*z[13]*z[14]*z[16]*z[5]*z[6]*z[7] - 288*z[13]*z[14]*z[16]*z[5]*z[6]*z[8] + 540*z[13]*z[14]*z[16]*z[5]*z[6] - 576*z[13]*z[14]*z[16]*z[5]*z[7]*z[8] + 1080*z[13]*z[14]*z[16]*z[5]*z[7] + 2160*z[13]*z[14]*z[16]*z[5]*z[8] - 2784*z[13]*z[14]*z[16]*z[5] - 1152*z[13]*z[14]*z[16]*z[6]*z[7]*z[8] + 2160*z[13]*z[14]*z[16]*z[6]*z[7] + 4320*z[13]*z[14]*z[16]*z[6]*z[8] - 5532*z[13]*z[14]*z[16]*z[6] + 8640*z[13]*z[14]*z[16]*z[7]*z[8] - 10776*z[13]*z[14]*z[16]*z[7] - 19248*z[13]*z[14]*z[16]*z[8] + 2784*z[13]*z[14]*z[16]*z[9] - 132300*z[13]*z[14]*z[16] + 2160*z[13]*z[14]*z[2]*z[3]*z[4] - 4050*z[13]*z[14]*z[2]*z[3] - 8100*z[13]*z[14]*z[2]*z[4] + 20745*z[13]*z[14]*z[2]/2 - 16200*z[13]*z[14]*z[3]*z[4] + 20205*z[13]*z[14]*z[3] + 36090*z[13]*z[14]*z[4] + 270*z[13]*z[14]*z[5]*z[6]*z[7] + 540*z[13]*z[14]*z[5]*z[6]*z[8] - 2025*z[13]*z[14]*z[5]*z[6]/2 + 1080*z[13]*z[14]*z[5]*z[7]*z[8] - 2025*z[13]*z[14]*z[5]*z[7] - 4050*z[13]*z[14]*z[5]*z[8] + 5220*z[13]*z[14]*z[5] + 2160*z[13]*z[14]*z[6]*z[7]*z[8] - 4050*z[13]*z[14]*z[6]*z[7] - 8100*z[13]*z[14]*z[6]*z[8] + 20745*z[13]*z[14]*z[6]/2 - 16200*z[13]*z[14]*z[7]*z[8] + 20205*z[13]*z[14]*z[7] + 36090*z[13]*z[14]*z[8] - 5220*z[13]*z[14]*z[9] + 120916*z[13]*z[14] - 2304*z[13]*z[15]*z[16]*z[2]*z[3]*z[4] + 4320*z[13]*z[15]*z[16]*z[2]*z[3] + 8640*z[13]*z[15]*z[16]*z[2]*z[4] - 11064*z[13]*z[15]*z[16]*z[2] + 17280*z[13]*z[15]*z[16]*z[3]*z[4] - 21552*z[13]*z[15]*z[16]*z[3] - 38496*z[13]*z[15]*z[16]*z[4] - 288*z[13]*z[15]*z[16]*z[5]*z[6]*z[7] - 576*z[13]*z[15]*z[16]*z[5]*z[6]*z[8] + 1080*z[13]*z[15]*z[16]*z[5]*z[6] - 1152*z[13]*z[15]*z[16]*z[5]*z[7]*z[8] + 2160*z[13]*z[15]*z[16]*z[5]*z[7] + 4320*z[13]*z[15]*z[16]*z[5]*z[8] - 5568*z[13]*z[15]*z[16]*z[5] - 2304*z[13]*z[15]*z[16]*z[6]*z[7]*z[8] + 4320*z[13]*z[15]*z[16]*z[6]*z[7] + 8640*z[13]*z[15]*z[16]*z[6]*z[8] - 11064*z[13]*z[15]*z[16]*z[6] + 17280*z[13]*z[15]*z[16]*z[7]*z[8] - 21552*z[13]*z[15]*z[16]*z[7] - 38496*z[13]*z[15]*z[16]*z[8] + 5568*z[13]*z[15]*z[16]*z[9] - 243000*z[13]*z[15]*z[16] + 4320*z[13]*z[15]*z[2]*z[3]*z[4] - 8100*z[13]*z[15]*z[2]*z[3] - 16200*z[13]*z[15]*z[2]*z[4] + 20745*z[13]*z[15]*z[2] - 32400*z[13]*z[15]*z[3]*z[4] + 40410*z[13]*z[15]*z[3] + 72180*z[13]*z[15]*z[4] + 540*z[13]*z[15]*z[5]*z[6]*z[7] + 1080*z[13]*z[15]*z[5]*z[6]*z[8] - 2025*z[13]*z[15]*z[5]*z[6] + 2160*z[13]*z[15]*z[5]*z[7]*z[8] - 4050*z[13]*z[15]*z[5]*z[7] - 8100*z[13]*z[15]*z[5]*z[8] + 10440*z[13]*z[15]*z[5] + 4320*z[13]*z[15]*z[6]*z[7]*z[8] - 8100*z[13]*z[15]*z[6]*z[7] - 16200*z[13]*z[15]*z[6]*z[8] + 20745*z[13]*z[15]*z[6] - 32400*z[13]*z[15]*z[7]*z[8] + 40410*z[13]*z[15]*z[7] + 72180*z[13]*z[15]*z[8] - 10440*z[13]*z[15]*z[9] + 215432*z[13]*z[15] + 8640*z[13]*z[16]*z[2]*z[3]*z[4] - 16200*z[13]*z[16]*z[2]*z[3] - 32400*z[13]*z[16]*z[2]*z[4] + 41490*z[13]*z[16]*z[2] - 64800*z[13]*z[16]*z[3]*z[4] + 80820*z[13]*z[16]*z[3] + 144360*z[13]*z[16]*z[4] + 1080*z[13]*z[16]*z[5]*z[6]*z[7] + 2160*z[13]*z[16]*z[5]*z[6]*z[8] - 4050*z[13]*z[16]*z[5]*z[6] + 4320*z[13]*z[16]*z[5]*z[7]*z[8] - 8100*z[13]*z[16]*z[5]*z[7] - 16200*z[13]*z[16]*z[5]*z[8] + 20880*z[13]*z[16]*z[5] + 8640*z[13]*z[16]*z[6]*z[7]*z[8] - 16200*z[13]*z[16]*z[6]*z[7] - 32400*z[13]*z[16]*z[6]*z[8] + 41490*z[13]*z[16]*z[6] - 64800*z[13]*z[16]*z[7]*z[8] + 80820*z[13]*z[16]*z[7] + 144360*z[13]*z[16]*z[8] - 20880*z[13]*z[16]*z[9] + 254224*z[13]*z[16] - 11136*z[13]*z[2]*z[3]*z[4] + 20880*z[13]*z[2]*z[3] + 41760*z[13]*z[2]*z[4] - 53476*z[13]*z[2] + 83520*z[13]*z[3]*z[4] - 104168*z[13]*z[3] - 186064*z[13]*z[4] - 1392*z[13]*z[5]*z[6]*z[7] - 2784*z[13]*z[5]*z[6]*z[8] + 5220*z[13]*z[5]*z[6] - 5568*z[13]*z[5]*z[7]*z[8] + 10440*z[13]*z[5]*z[7] + 20880*z[13]*z[5]*z[8] - 26912*z[13]*z[5] - 11136*z[13]*z[6]*z[7]*z[8] + 20880*z[13]*z[6]*z[7] + 41760*z[13]*z[6]*z[8] - 53476*z[13]*z[6] + 83520*z[13]*z[7]*z[8] - 104168*z[13]*z[7] - 186064*z[13]*z[8] + 26912*z[13]*z[9] - 435645*z[13]/2 - 4608*z[14]*z[15]*z[16]*z[2]*z[3]*z[4] + 8640*z[14]*z[15]*z[16]*z[2]*z[3] + 17280*z[14]*z[15]*z[16]*z[2]*z[4] - 22128*z[14]*z[15]*z[16]*z[2] + 34560*z[14]*z[15]*z[16]*z[3]*z[4] - 43104*z[14]*z[15]*z[16]*z[3] - 76992*z[14]*z[15]*z[16]*z[4] - 576*z[14]*z[15]*z[16]*z[5]*z[6]*z[7] - 1152*z[14]*z[15]*z[16]*z[5]*z[6]*z[8] + 2160*z[14]*z[15]*z[16]*z[5]*z[6] - 2304*z[14]*z[15]*z[16]*z[5]*z[7]*z[8] + 4320*z[14]*z[15]*z[16]*z[5]*z[7] + 8640*z[14]*z[15]*z[16]*z[5]*z[8] - 11136*z[14]*z[15]*z[16]*z[5] - 4608*z[14]*z[15]*z[16]*z[6]*z[7]*z[8] + 8640*z[14]*z[15]*z[16]*z[6]*z[7] + 17280*z[14]*z[15]*z[16]*z[6]*z[8] - 22128*z[14]*z[15]*z[16]*z[6] + 34560*z[14]*z[15]*z[16]*z[7]*z[8] - 43104*z[14]*z[15]*z[16]*z[7] - 76992*z[14]*z[15]*z[16]*z[8] + 11136*z[14]*z[15]*z[16]*z[9] - 475200*z[14]*z[15]*z[16] + 8640*z[14]*z[15]*z[2]*z[3]*z[4] - 16200*z[14]*z[15]*z[2]*z[3] - 32400*z[14]*z[15]*z[2]*z[4] + 41490*z[14]*z[15]*z[2] - 64800*z[14]*z[15]*z[3]*z[4] + 80820*z[14]*z[15]*z[3] + 144360*z[14]*z[15]*z[4] + 1080*z[14]*z[15]*z[5]*z[6]*z[7] + 2160*z[14]*z[15]*z[5]*z[6]*z[8] - 4050*z[14]*z[15]*z[5]*z[6] + 4320*z[14]*z[15]*z[5]*z[7]*z[8] - 8100*z[14]*z[15]*z[5]*z[7] - 16200*z[14]*z[15]*z[5]*z[8] + 20880*z[14]*z[15]*z[5] + 8640*z[14]*z[15]*z[6]*z[7]*z[8] - 16200*z[14]*z[15]*z[6]*z[7] - 32400*z[14]*z[15]*z[6]*z[8] + 41490*z[14]*z[15]*z[6] - 64800*z[14]*z[15]*z[7]*z[8] + 80820*z[14]*z[15]*z[7] + 144360*z[14]*z[15]*z[8] - 20880*z[14]*z[15]*z[9] + 417574*z[14]*z[15] + 17280*z[14]*z[16]*z[2]*z[3]*z[4] - 32400*z[14]*z[16]*z[2]*z[3] - 64800*z[14]*z[16]*z[2]*z[4] + 82980*z[14]*z[16]*z[2] - 129600*z[14]*z[16]*z[3]*z[4] + 161640*z[14]*z[16]*z[3] + 288720*z[14]*z[16]*z[4] + 2160*z[14]*z[16]*z[5]*z[6]*z[7] + 4320*z[14]*z[16]*z[5]*z[6]*z[8] - 8100*z[14]*z[16]*z[5]*z[6] + 8640*z[14]*z[16]*z[5]*z[7]*z[8] - 16200*z[14]*z[16]*z[5]*z[7] - 32400*z[14]*z[16]*z[5]*z[8] + 41760*z[14]*z[16]*z[5] + 17280*z[14]*z[16]*z[6]*z[7]*z[8] - 32400*z[14]*z[16]*z[6]*z[7] - 64800*z[14]*z[16]*z[6]*z[8] + 82980*z[14]*z[16]*z[6] - 129600*z[14]*z[16]*z[7]*z[8] + 161640*z[14]*z[16]*z[7] + 288720*z[14]*z[16]*z[8] - 41760*z[14]*z[16]*z[9] + 484748*z[14]*z[16] - 22128*z[14]*z[2]*z[3]*z[4] + 41490*z[14]*z[2]*z[3] + 82980*z[14]*z[2]*z[4] - 212521*z[14]*z[2]/2 + 165960*z[14]*z[3]*z[4] - 206989*z[14]*z[3] - 369722*z[14]*z[4] - 2766*z[14]*z[5]*z[6]*z[7] - 5532*z[14]*z[5]*z[6]*z[8] + 20745*z[14]*z[5]*z[6]/2 - 11064*z[14]*z[5]*z[7]*z[8] + 20745*z[14]*z[5]*z[7] + 41490*z[14]*z[5]*z[8] - 53476*z[14]*z[5] - 22128*z[14]*z[6]*z[7]*z[8] + 41490*z[14]*z[6]*z[7] + 82980*z[14]*z[6]*z[8] - 212521*z[14]*z[6]/2 + 165960*z[14]*z[7]*z[8] - 206989*z[14]*z[7] - 369722*z[14]*z[8] + 53476*z[14]*z[9] - 412020*z[14] + 34560*z[15]*z[16]*z[2]*z[3]*z[4] - 64800*z[15]*z[16]*z[2]*z[3] - 129600*z[15]*z[16]*z[2]*z[4] + 165960*z[15]*z[16]*z[2] - 259200*z[15]*z[16]*z[3]*z[4] + 323280*z[15]*z[16]*z[3] + 577440*z[15]*z[16]*z[4] + 4320*z[15]*z[16]*z[5]*z[6]*z[7] + 8640*z[15]*z[16]*z[5]*z[6]*z[8] - 16200*z[15]*z[16]*z[5]*z[6] + 17280*z[15]*z[16]*z[5]*z[7]*z[8] - 32400*z[15]*z[16]*z[5]*z[7] - 64800*z[15]*z[16]*z[5]*z[8] + 83520*z[15]*z[16]*z[5] + 34560*z[15]*z[16]*z[6]*z[7]*z[8] - 64800*z[15]*z[16]*z[6]*z[7] - 129600*z[15]*z[16]*z[6]*z[8] + 165960*z[15]*z[16]*z[6] - 259200*z[15]*z[16]*z[7]*z[8] + 323280*z[15]*z[16]*z[7] + 577440*z[15]*z[16]*z[8] - 83520*z[15]*z[16]*z[9] + 788536*z[15]*z[16] - 43104*z[15]*z[2]*z[3]*z[4] + 80820*z[15]*z[2]*z[3] + 161640*z[15]*z[2]*z[4] - 206989*z[15]*z[2] + 323280*z[15]*z[3]*z[4] - 403202*z[15]*z[3] - 720196*z[15]*z[4] - 5388*z[15]*z[5]*z[6]*z[7] - 10776*z[15]*z[5]*z[6]*z[8] + 20205*z[15]*z[5]*z[6] - 21552*z[15]*z[5]*z[7]*z[8] + 40410*z[15]*z[5]*z[7] + 80820*z[15]*z[5]*z[8] - 104168*z[15]*z[5] - 43104*z[15]*z[6]*z[7]*z[8] + 80820*z[15]*z[6]*z[7] + 161640*z[15]*z[6]*z[8] - 206989*z[15]*z[6] + 323280*z[15]*z[7]*z[8] - 403202*z[15]*z[7] - 720196*z[15]*z[8] + 104168*z[15]*z[9] - 651240*z[15] - 76992*z[16]*z[2]*z[3]*z[4] + 144360*z[16]*z[2]*z[3] + 288720*z[16]*z[2]*z[4] - 369722*z[16]*z[2] + 577440*z[16]*z[3]*z[4] - 720196*z[16]*z[3] - 1286408*z[16]*z[4] - 9624*z[16]*z[5]*z[6]*z[7] - 19248*z[16]*z[5]*z[6]*z[8] + 36090*z[16]*z[5]*z[6] - 38496*z[16]*z[5]*z[7]*z[8] + 72180*z[16]*z[5]*z[7] + 144360*z[16]*z[5]*z[8] - 186064*z[16]*z[5] - 76992*z[16]*z[6]*z[7]*z[8] + 144360*z[16]*z[6]*z[7] + 288720*z[16]*z[6]*z[8] - 369722*z[16]*z[6] + 577440*z[16]*z[7]*z[8] - 720196*z[16]*z[7] - 1286408*z[16]*z[8] + 186064*z[16]*z[9] - 438480*z[16] + 576*z[2]*z[3]*z[4]*z[5]*z[6]*z[7] + 1152*z[2]*z[3]*z[4]*z[5]*z[6]*z[8] - 2160*z[2]*z[3]*z[4]*z[5]*z[6] + 2304*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] - 4320*z[2]*z[3]*z[4]*z[5]*z[7] - 8640*z[2]*z[3]*z[4]*z[5]*z[8] + 11136*z[2]*z[3]*z[4]*z[5] + 4608*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] - 8640*z[2]*z[3]*z[4]*z[6]*z[7] - 17280*z[2]*z[3]*z[4]*z[6]*z[8] + 22128*z[2]*z[3]*z[4]*z[6] - 34560*z[2]*z[3]*z[4]*z[7]*z[8] + 43104*z[2]*z[3]*z[4]*z[7] + 76992*z[2]*z[3]*z[4]*z[8] - 11136*z[2]*z[3]*z[4]*z[9] - 475200*z[2]*z[3]*z[4] - 1080*z[2]*z[3]*z[5]*z[6]*z[7] - 2160*z[2]*z[3]*z[5]*z[6]*z[8] + 4050*z[2]*z[3]*z[5]*z[6] - 4320*z[2]*z[3]*z[5]*z[7]*z[8] + 8100*z[2]*z[3]*z[5]*z[7] + 16200*z[2]*z[3]*z[5]*z[8] - 20880*z[2]*z[3]*z[5] - 8640*z[2]*z[3]*z[6]*z[7]*z[8] + 16200*z[2]*z[3]*z[6]*z[7] + 32400*z[2]*z[3]*z[6]*z[8] - 41490*z[2]*z[3]*z[6] + 64800*z[2]*z[3]*z[7]*z[8] - 80820*z[2]*z[3]*z[7] - 144360*z[2]*z[3]*z[8] + 20880*z[2]*z[3]*z[9] + 417574*z[2]*z[3] - 2160*z[2]*z[4]*z[5]*z[6]*z[7] - 4320*z[2]*z[4]*z[5]*z[6]*z[8] + 8100*z[2]*z[4]*z[5]*z[6] - 8640*z[2]*z[4]*z[5]*z[7]*z[8] + 16200*z[2]*z[4]*z[5]*z[7] + 32400*z[2]*z[4]*z[5]*z[8] - 41760*z[2]*z[4]*z[5] - 17280*z[2]*z[4]*z[6]*z[7]*z[8] + 32400*z[2]*z[4]*z[6]*z[7] + 64800*z[2]*z[4]*z[6]*z[8] - 82980*z[2]*z[4]*z[6] + 129600*z[2]*z[4]*z[7]*z[8] - 161640*z[2]*z[4]*z[7] - 288720*z[2]*z[4]*z[8] + 41760*z[2]*z[4]*z[9] + 484748*z[2]*z[4] + 2766*z[2]*z[5]*z[6]*z[7] + 5532*z[2]*z[5]*z[6]*z[8] - 20745*z[2]*z[5]*z[6]/2 + 11064*z[2]*z[5]*z[7]*z[8] - 20745*z[2]*z[5]*z[7] - 41490*z[2]*z[5]*z[8] + 53476*z[2]*z[5] + 22128*z[2]*z[6]*z[7]*z[8] - 41490*z[2]*z[6]*z[7] - 82980*z[2]*z[6]*z[8] + 212521*z[2]*z[6]/2 - 165960*z[2]*z[7]*z[8] + 206989*z[2]*z[7] + 369722*z[2]*z[8] - 53476*z[2]*z[9] - 412020*z[2] - 4320*z[3]*z[4]*z[5]*z[6]*z[7] - 8640*z[3]*z[4]*z[5]*z[6]*z[8] + 16200*z[3]*z[4]*z[5]*z[6] - 17280*z[3]*z[4]*z[5]*z[7]*z[8] + 32400*z[3]*z[4]*z[5]*z[7] + 64800*z[3]*z[4]*z[5]*z[8] - 83520*z[3]*z[4]*z[5] - 34560*z[3]*z[4]*z[6]*z[7]*z[8] + 64800*z[3]*z[4]*z[6]*z[7] + 129600*z[3]*z[4]*z[6]*z[8] - 165960*z[3]*z[4]*z[6] + 259200*z[3]*z[4]*z[7]*z[8] - 323280*z[3]*z[4]*z[7] - 577440*z[3]*z[4]*z[8] + 83520*z[3]*z[4]*z[9] + 788536*z[3]*z[4] + 5388*z[3]*z[5]*z[6]*z[7] + 10776*z[3]*z[5]*z[6]*z[8] - 20205*z[3]*z[5]*z[6] + 21552*z[3]*z[5]*z[7]*z[8] - 40410*z[3]*z[5]*z[7] - 80820*z[3]*z[5]*z[8] + 104168*z[3]*z[5] + 43104*z[3]*z[6]*z[7]*z[8] - 80820*z[3]*z[6]*z[7] - 161640*z[3]*z[6]*z[8] + 206989*z[3]*z[6] - 323280*z[3]*z[7]*z[8] + 403202*z[3]*z[7] + 720196*z[3]*z[8] - 104168*z[3]*z[9] - 651240*z[3] + 9624*z[4]*z[5]*z[6]*z[7] + 19248*z[4]*z[5]*z[6]*z[8] - 36090*z[4]*z[5]*z[6] + 38496*z[4]*z[5]*z[7]*z[8] - 72180*z[4]*z[5]*z[7] - 144360*z[4]*z[5]*z[8] + 186064*z[4]*z[5] + 76992*z[4]*z[6]*z[7]*z[8] - 144360*z[4]*z[6]*z[7] - 288720*z[4]*z[6]*z[8] + 369722*z[4]*z[6] - 577440*z[4]*z[7]*z[8] + 720196*z[4]*z[7] + 1286408*z[4]*z[8] - 186064*z[4]*z[9] - 438480*z[4] + 91200*z[5]*z[6]*z[7]*z[8] - 1392*z[5]*z[6]*z[7]*z[9] - 87750*z[5]*z[6]*z[7] - 2784*z[5]*z[6]*z[8]*z[9] - 132300*z[5]*z[6]*z[8] + 5220*z[5]*z[6]*z[9] + 120916*z[5]*z[6] - 5568*z[5]*z[7]*z[8]*z[9] - 243000*z[5]*z[7]*z[8] + 10440*z[5]*z[7]*z[9] + 215432*z[5]*z[7] + 20880*z[5]*z[8]*z[9] + 254224*z[5]*z[8] - 26912*z[5]*z[9] - 435645*z[5]/2 - 11136*z[6]*z[7]*z[8]*z[9] - 475200*z[6]*z[7]*z[8] + 20880*z[6]*z[7]*z[9] + 417574*z[6]*z[7] + 41760*z[6]*z[8]*z[9] + 484748*z[6]*z[8] - 53476*z[6]*z[9] - 412020*z[6] + 83520*z[7]*z[8]*z[9] + 788536*z[7]*z[8] - 104168*z[7]*z[9] - 651240*z[7] - 186064*z[8]*z[9] - 438480*z[8] - 435645*z[9]/2 + 4380730)

In [416]:
bitstring_to_pm1(top_solutions_less_15_1_D_Hardy_Ramanujan_Number, evaluate_hamiltonian_Hardy_Ramanujan_Number_less_15)

["Bitstring: ('0101010101010101', -8761458.0, 1), Evaluated cost: 0.0",
 "Bitstring: ('0101010101010101', -8761458.0, 1), Evaluated cost: 1.0",
 "Bitstring: ('0101010101010101', -8761458.0, 1), Evaluated cost: 1.0",
 "Bitstring: ('0101010101010101', -8761458.0, 1), Evaluated cost: 0.0",
 "Bitstring: ('0101010101010101', -8761458.0, 1), Evaluated cost: 0.0"]

# Diophantine Equation $x^4+y^4 = z^2-1$ it is known that no integer solutions exist with $ 0 < y < 7.9 \cdot 10^7$.

In [417]:
def D_Equation_2(x,y,z):
    return(x**4+y**4-z**2+1)

In [418]:
D_Equation_2(x[1],x[2],x[3])

x1**4 + x2**4 - x3**2 + 1

In [421]:
paso1_D_Equation_2_less_15 = substitute_with_global_binary_symbols(D_Equation_2(x[1],x[2],x[3])**2, 4, base_name="b")
paso2_D_Equation_2_less_15 = remove_variable_exponents(paso1_D_Equation_2_less_15)
paso3_D_Equation_2_less_15 = substitute_with_spin_variables(paso2_D_Equation_2_less_15)

In [423]:
print(paso3_D_Equation_2_less_15)

-768*z_1*z_10*z_11*z_2*z_3*z_4 + 1440*z_1*z_10*z_11*z_2*z_3 + 2880*z_1*z_10*z_11*z_2*z_4 - 3680*z_1*z_10*z_11*z_2 + 5760*z_1*z_10*z_11*z_3*z_4 - 7168*z_1*z_10*z_11*z_3 - 12800*z_1*z_10*z_11*z_4 + 14340*z_1*z_10*z_11 - 1536*z_1*z_10*z_12*z_2*z_3*z_4 + 2880*z_1*z_10*z_12*z_2*z_3 + 5760*z_1*z_10*z_12*z_2*z_4 - 7360*z_1*z_10*z_12*z_2 + 11520*z_1*z_10*z_12*z_3*z_4 - 14336*z_1*z_10*z_12*z_3 - 25600*z_1*z_10*z_12*z_4 + 28680*z_1*z_10*z_12 - 192*z_1*z_10*z_2*z_3*z_4*z_9 + 2880*z_1*z_10*z_2*z_3*z_4 + 360*z_1*z_10*z_2*z_3*z_9 - 5400*z_1*z_10*z_2*z_3 + 720*z_1*z_10*z_2*z_4*z_9 - 10800*z_1*z_10*z_2*z_4 - 920*z_1*z_10*z_2*z_9 + 13800*z_1*z_10*z_2 + 1440*z_1*z_10*z_3*z_4*z_9 - 21600*z_1*z_10*z_3*z_4 - 1792*z_1*z_10*z_3*z_9 + 26880*z_1*z_10*z_3 - 3200*z_1*z_10*z_4*z_9 + 48000*z_1*z_10*z_4 + 3585*z_1*z_10*z_9 - 53775*z_1*z_10 - 3072*z_1*z_11*z_12*z_2*z_3*z_4 + 5760*z_1*z_11*z_12*z_2*z_3 + 11520*z_1*z_11*z_12*z_2*z_4 - 14720*z_1*z_11*z_12*z_2 + 23040*z_1*z_11*z_12*z_3*z_4 - 28672*z_1*z_11*z_12*z_3 - 51

In [426]:
def evaluate_hamiltonian_D_Equation_2_less_15(z):
    return(-768*z[1]*z[10]*z[11]*z[2]*z[3]*z[4] + 1440*z[1]*z[10]*z[11]*z[2]*z[3] + 2880*z[1]*z[10]*z[11]*z[2]*z[4] - 3680*z[1]*z[10]*z[11]*z[2] + 5760*z[1]*z[10]*z[11]*z[3]*z[4] - 7168*z[1]*z[10]*z[11]*z[3] - 12800*z[1]*z[10]*z[11]*z[4] + 14340*z[1]*z[10]*z[11] - 1536*z[1]*z[10]*z[12]*z[2]*z[3]*z[4] + 2880*z[1]*z[10]*z[12]*z[2]*z[3] + 5760*z[1]*z[10]*z[12]*z[2]*z[4] - 7360*z[1]*z[10]*z[12]*z[2] + 11520*z[1]*z[10]*z[12]*z[3]*z[4] - 14336*z[1]*z[10]*z[12]*z[3] - 25600*z[1]*z[10]*z[12]*z[4] + 28680*z[1]*z[10]*z[12] - 192*z[1]*z[10]*z[2]*z[3]*z[4]*z[9] + 2880*z[1]*z[10]*z[2]*z[3]*z[4] + 360*z[1]*z[10]*z[2]*z[3]*z[9] - 5400*z[1]*z[10]*z[2]*z[3] + 720*z[1]*z[10]*z[2]*z[4]*z[9] - 10800*z[1]*z[10]*z[2]*z[4] - 920*z[1]*z[10]*z[2]*z[9] + 13800*z[1]*z[10]*z[2] + 1440*z[1]*z[10]*z[3]*z[4]*z[9] - 21600*z[1]*z[10]*z[3]*z[4] - 1792*z[1]*z[10]*z[3]*z[9] + 26880*z[1]*z[10]*z[3] - 3200*z[1]*z[10]*z[4]*z[9] + 48000*z[1]*z[10]*z[4] + 3585*z[1]*z[10]*z[9] - 53775*z[1]*z[10] - 3072*z[1]*z[11]*z[12]*z[2]*z[3]*z[4] + 5760*z[1]*z[11]*z[12]*z[2]*z[3] + 11520*z[1]*z[11]*z[12]*z[2]*z[4] - 14720*z[1]*z[11]*z[12]*z[2] + 23040*z[1]*z[11]*z[12]*z[3]*z[4] - 28672*z[1]*z[11]*z[12]*z[3] - 51200*z[1]*z[11]*z[12]*z[4] + 57360*z[1]*z[11]*z[12] - 384*z[1]*z[11]*z[2]*z[3]*z[4]*z[9] + 5760*z[1]*z[11]*z[2]*z[3]*z[4] + 720*z[1]*z[11]*z[2]*z[3]*z[9] - 10800*z[1]*z[11]*z[2]*z[3] + 1440*z[1]*z[11]*z[2]*z[4]*z[9] - 21600*z[1]*z[11]*z[2]*z[4] - 1840*z[1]*z[11]*z[2]*z[9] + 27600*z[1]*z[11]*z[2] + 2880*z[1]*z[11]*z[3]*z[4]*z[9] - 43200*z[1]*z[11]*z[3]*z[4] - 3584*z[1]*z[11]*z[3]*z[9] + 53760*z[1]*z[11]*z[3] - 6400*z[1]*z[11]*z[4]*z[9] + 96000*z[1]*z[11]*z[4] + 7170*z[1]*z[11]*z[9] - 107550*z[1]*z[11] - 768*z[1]*z[12]*z[2]*z[3]*z[4]*z[9] + 11520*z[1]*z[12]*z[2]*z[3]*z[4] + 1440*z[1]*z[12]*z[2]*z[3]*z[9] - 21600*z[1]*z[12]*z[2]*z[3] + 2880*z[1]*z[12]*z[2]*z[4]*z[9] - 43200*z[1]*z[12]*z[2]*z[4] - 3680*z[1]*z[12]*z[2]*z[9] + 55200*z[1]*z[12]*z[2] + 5760*z[1]*z[12]*z[3]*z[4]*z[9] - 86400*z[1]*z[12]*z[3]*z[4] - 7168*z[1]*z[12]*z[3]*z[9] + 107520*z[1]*z[12]*z[3] - 12800*z[1]*z[12]*z[4]*z[9] + 192000*z[1]*z[12]*z[4] + 14340*z[1]*z[12]*z[9] - 215100*z[1]*z[12] + 18432*z[1]*z[2]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] - 34560*z[1]*z[2]*z[3]*z[4]*z[5]*z[6]*z[7] - 69120*z[1]*z[2]*z[3]*z[4]*z[5]*z[6]*z[8] + 88320*z[1]*z[2]*z[3]*z[4]*z[5]*z[6] - 138240*z[1]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] + 172032*z[1]*z[2]*z[3]*z[4]*z[5]*z[7] + 307200*z[1]*z[2]*z[3]*z[4]*z[5]*z[8] - 344160*z[1]*z[2]*z[3]*z[4]*z[5] - 276480*z[1]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] + 341760*z[1]*z[2]*z[3]*z[4]*z[6]*z[7] + 609792*z[1]*z[2]*z[3]*z[4]*z[6]*z[8] - 679680*z[1]*z[2]*z[3]*z[4]*z[6] + 1182720*z[1]*z[2]*z[3]*z[4]*z[7]*z[8] - 1290240*z[1]*z[2]*z[3]*z[4]*z[7] - 2027520*z[1]*z[2]*z[3]*z[4]*z[8] + 1440*z[1]*z[2]*z[3]*z[4]*z[9] + 40219392*z[1]*z[2]*z[3]*z[4] - 34560*z[1]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] + 64800*z[1]*z[2]*z[3]*z[5]*z[6]*z[7] + 129600*z[1]*z[2]*z[3]*z[5]*z[6]*z[8] - 165600*z[1]*z[2]*z[3]*z[5]*z[6] + 259200*z[1]*z[2]*z[3]*z[5]*z[7]*z[8] - 322560*z[1]*z[2]*z[3]*z[5]*z[7] - 576000*z[1]*z[2]*z[3]*z[5]*z[8] + 645300*z[1]*z[2]*z[3]*z[5] + 518400*z[1]*z[2]*z[3]*z[6]*z[7]*z[8] - 640800*z[1]*z[2]*z[3]*z[6]*z[7] - 1143360*z[1]*z[2]*z[3]*z[6]*z[8] + 1274400*z[1]*z[2]*z[3]*z[6] - 2217600*z[1]*z[2]*z[3]*z[7]*z[8] + 2419200*z[1]*z[2]*z[3]*z[7] + 3801600*z[1]*z[2]*z[3]*z[8] - 2700*z[1]*z[2]*z[3]*z[9] - 42548040*z[1]*z[2]*z[3] - 69120*z[1]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] + 129600*z[1]*z[2]*z[4]*z[5]*z[6]*z[7] + 259200*z[1]*z[2]*z[4]*z[5]*z[6]*z[8] - 331200*z[1]*z[2]*z[4]*z[5]*z[6] + 518400*z[1]*z[2]*z[4]*z[5]*z[7]*z[8] - 645120*z[1]*z[2]*z[4]*z[5]*z[7] - 1152000*z[1]*z[2]*z[4]*z[5]*z[8] + 1290600*z[1]*z[2]*z[4]*z[5] + 1036800*z[1]*z[2]*z[4]*z[6]*z[7]*z[8] - 1281600*z[1]*z[2]*z[4]*z[6]*z[7] - 2286720*z[1]*z[2]*z[4]*z[6]*z[8] + 2548800*z[1]*z[2]*z[4]*z[6] - 4435200*z[1]*z[2]*z[4]*z[7]*z[8] + 4838400*z[1]*z[2]*z[4]*z[7] + 7603200*z[1]*z[2]*z[4]*z[8] - 5400*z[1]*z[2]*z[4]*z[9] - 57073680*z[1]*z[2]*z[4] + 88320*z[1]*z[2]*z[5]*z[6]*z[7]*z[8] - 165600*z[1]*z[2]*z[5]*z[6]*z[7] - 331200*z[1]*z[2]*z[5]*z[6]*z[8] + 423200*z[1]*z[2]*z[5]*z[6] - 662400*z[1]*z[2]*z[5]*z[7]*z[8] + 824320*z[1]*z[2]*z[5]*z[7] + 1472000*z[1]*z[2]*z[5]*z[8] - 1649100*z[1]*z[2]*z[5] - 1324800*z[1]*z[2]*z[6]*z[7]*z[8] + 1637600*z[1]*z[2]*z[6]*z[7] + 2921920*z[1]*z[2]*z[6]*z[8] - 3256800*z[1]*z[2]*z[6] + 5667200*z[1]*z[2]*z[7]*z[8] - 6182400*z[1]*z[2]*z[7] - 9715200*z[1]*z[2]*z[8] + 6900*z[1]*z[2]*z[9] + 59758080*z[1]*z[2] - 138240*z[1]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 259200*z[1]*z[3]*z[4]*z[5]*z[6]*z[7] + 518400*z[1]*z[3]*z[4]*z[5]*z[6]*z[8] - 662400*z[1]*z[3]*z[4]*z[5]*z[6] + 1036800*z[1]*z[3]*z[4]*z[5]*z[7]*z[8] - 1290240*z[1]*z[3]*z[4]*z[5]*z[7] - 2304000*z[1]*z[3]*z[4]*z[5]*z[8] + 2581200*z[1]*z[3]*z[4]*z[5] + 2073600*z[1]*z[3]*z[4]*z[6]*z[7]*z[8] - 2563200*z[1]*z[3]*z[4]*z[6]*z[7] - 4573440*z[1]*z[3]*z[4]*z[6]*z[8] + 5097600*z[1]*z[3]*z[4]*z[6] - 8870400*z[1]*z[3]*z[4]*z[7]*z[8] + 9676800*z[1]*z[3]*z[4]*z[7] + 15206400*z[1]*z[3]*z[4]*z[8] - 10800*z[1]*z[3]*z[4]*z[9] - 98926560*z[1]*z[3]*z[4] + 172032*z[1]*z[3]*z[5]*z[6]*z[7]*z[8] - 322560*z[1]*z[3]*z[5]*z[6]*z[7] - 645120*z[1]*z[3]*z[5]*z[6]*z[8] + 824320*z[1]*z[3]*z[5]*z[6] - 1290240*z[1]*z[3]*z[5]*z[7]*z[8] + 1605632*z[1]*z[3]*z[5]*z[7] + 2867200*z[1]*z[3]*z[5]*z[8] - 3212160*z[1]*z[3]*z[5] - 2580480*z[1]*z[3]*z[6]*z[7]*z[8] + 3189760*z[1]*z[3]*z[6]*z[7] + 5691392*z[1]*z[3]*z[6]*z[8] - 6343680*z[1]*z[3]*z[6] + 11038720*z[1]*z[3]*z[7]*z[8] - 12042240*z[1]*z[3]*z[7] - 18923520*z[1]*z[3]*z[8] + 13440*z[1]*z[3]*z[9] + 103372992*z[1]*z[3] + 307200*z[1]*z[4]*z[5]*z[6]*z[7]*z[8] - 576000*z[1]*z[4]*z[5]*z[6]*z[7] - 1152000*z[1]*z[4]*z[5]*z[6]*z[8] + 1472000*z[1]*z[4]*z[5]*z[6] - 2304000*z[1]*z[4]*z[5]*z[7]*z[8] + 2867200*z[1]*z[4]*z[5]*z[7] + 5120000*z[1]*z[4]*z[5]*z[8] - 5736000*z[1]*z[4]*z[5] - 4608000*z[1]*z[4]*z[6]*z[7]*z[8] + 5696000*z[1]*z[4]*z[6]*z[7] + 10163200*z[1]*z[4]*z[6]*z[8] - 11328000*z[1]*z[4]*z[6] + 19712000*z[1]*z[4]*z[7]*z[8] - 21504000*z[1]*z[4]*z[7] - 33792000*z[1]*z[4]*z[8] + 24000*z[1]*z[4]*z[9] + 135984000*z[1]*z[4] - 344160*z[1]*z[5]*z[6]*z[7]*z[8] + 645300*z[1]*z[5]*z[6]*z[7] + 1290600*z[1]*z[5]*z[6]*z[8] - 1649100*z[1]*z[5]*z[6] + 2581200*z[1]*z[5]*z[7]*z[8] - 3212160*z[1]*z[5]*z[7] - 5736000*z[1]*z[5]*z[8] + 12852225*z[1]*z[5]/2 + 5162400*z[1]*z[6]*z[7]*z[8] - 6381300*z[1]*z[6]*z[7] - 11385960*z[1]*z[6]*z[8] + 12690900*z[1]*z[6] - 22083600*z[1]*z[7]*z[8] + 24091200*z[1]*z[7] + 37857600*z[1]*z[8] - 53775*z[1]*z[9]/2 - 281594505*z[1]/2 + 96*z[10]*z[11]*z[12]*z[9] - 1440*z[10]*z[11]*z[12] + 11520*z[10]*z[11]*z[2]*z[3]*z[4] - 14240*z[10]*z[11]*z[2]*z[3] - 25408*z[10]*z[11]*z[2]*z[4] + 28320*z[10]*z[11]*z[2] - 49280*z[10]*z[11]*z[3]*z[4] + 53760*z[10]*z[11]*z[3] + 84480*z[10]*z[11]*z[4] - 768*z[10]*z[11]*z[5]*z[6]*z[7]*z[8] + 1440*z[10]*z[11]*z[5]*z[6]*z[7] + 2880*z[10]*z[11]*z[5]*z[6]*z[8] - 3680*z[10]*z[11]*z[5]*z[6] + 5760*z[10]*z[11]*z[5]*z[7]*z[8] - 7168*z[10]*z[11]*z[5]*z[7] - 12800*z[10]*z[11]*z[5]*z[8] + 14340*z[10]*z[11]*z[5] + 11520*z[10]*z[11]*z[6]*z[7]*z[8] - 14240*z[10]*z[11]*z[6]*z[7] - 25408*z[10]*z[11]*z[6]*z[8] + 28320*z[10]*z[11]*z[6] - 49280*z[10]*z[11]*z[7]*z[8] + 53760*z[10]*z[11]*z[7] + 84480*z[10]*z[11]*z[8] - 180*z[10]*z[11]*z[9] - 176540*z[10]*z[11] + 23040*z[10]*z[12]*z[2]*z[3]*z[4] - 28480*z[10]*z[12]*z[2]*z[3] - 50816*z[10]*z[12]*z[2]*z[4] + 56640*z[10]*z[12]*z[2] - 98560*z[10]*z[12]*z[3]*z[4] + 107520*z[10]*z[12]*z[3] + 168960*z[10]*z[12]*z[4] - 1536*z[10]*z[12]*z[5]*z[6]*z[7]*z[8] + 2880*z[10]*z[12]*z[5]*z[6]*z[7] + 5760*z[10]*z[12]*z[5]*z[6]*z[8] - 7360*z[10]*z[12]*z[5]*z[6] + 11520*z[10]*z[12]*z[5]*z[7]*z[8] - 14336*z[10]*z[12]*z[5]*z[7] - 25600*z[10]*z[12]*z[5]*z[8] + 28680*z[10]*z[12]*z[5] + 23040*z[10]*z[12]*z[6]*z[7]*z[8] - 28480*z[10]*z[12]*z[6]*z[7] - 50816*z[10]*z[12]*z[6]*z[8] + 56640*z[10]*z[12]*z[6] - 98560*z[10]*z[12]*z[7]*z[8] + 107520*z[10]*z[12]*z[7] + 168960*z[10]*z[12]*z[8] - 360*z[10]*z[12]*z[9] - 353464*z[10]*z[12] + 2880*z[10]*z[2]*z[3]*z[4]*z[9] - 43200*z[10]*z[2]*z[3]*z[4] - 3560*z[10]*z[2]*z[3]*z[9] + 53400*z[10]*z[2]*z[3] - 6352*z[10]*z[2]*z[4]*z[9] + 95280*z[10]*z[2]*z[4] + 7080*z[10]*z[2]*z[9] - 106200*z[10]*z[2] - 12320*z[10]*z[3]*z[4]*z[9] + 184800*z[10]*z[3]*z[4] + 13440*z[10]*z[3]*z[9] - 201600*z[10]*z[3] + 21120*z[10]*z[4]*z[9] - 316800*z[10]*z[4] - 192*z[10]*z[5]*z[6]*z[7]*z[8]*z[9] + 2880*z[10]*z[5]*z[6]*z[7]*z[8] + 360*z[10]*z[5]*z[6]*z[7]*z[9] - 5400*z[10]*z[5]*z[6]*z[7] + 720*z[10]*z[5]*z[6]*z[8]*z[9] - 10800*z[10]*z[5]*z[6]*z[8] - 920*z[10]*z[5]*z[6]*z[9] + 13800*z[10]*z[5]*z[6] + 1440*z[10]*z[5]*z[7]*z[8]*z[9] - 21600*z[10]*z[5]*z[7]*z[8] - 1792*z[10]*z[5]*z[7]*z[9] + 26880*z[10]*z[5]*z[7] - 3200*z[10]*z[5]*z[8]*z[9] + 48000*z[10]*z[5]*z[8] + 3585*z[10]*z[5]*z[9] - 53775*z[10]*z[5] + 2880*z[10]*z[6]*z[7]*z[8]*z[9] - 43200*z[10]*z[6]*z[7]*z[8] - 3560*z[10]*z[6]*z[7]*z[9] + 53400*z[10]*z[6]*z[7] - 6352*z[10]*z[6]*z[8]*z[9] + 95280*z[10]*z[6]*z[8] + 7080*z[10]*z[6]*z[9] - 106200*z[10]*z[6] - 12320*z[10]*z[7]*z[8]*z[9] + 184800*z[10]*z[7]*z[8] + 13440*z[10]*z[7]*z[9] - 201600*z[10]*z[7] + 21120*z[10]*z[8]*z[9] - 316800*z[10]*z[8] - 44120*z[10]*z[9] + 665160*z[10] + 46080*z[11]*z[12]*z[2]*z[3]*z[4] - 56960*z[11]*z[12]*z[2]*z[3] - 101632*z[11]*z[12]*z[2]*z[4] + 113280*z[11]*z[12]*z[2] - 197120*z[11]*z[12]*z[3]*z[4] + 215040*z[11]*z[12]*z[3] + 337920*z[11]*z[12]*z[4] - 3072*z[11]*z[12]*z[5]*z[6]*z[7]*z[8] + 5760*z[11]*z[12]*z[5]*z[6]*z[7] + 11520*z[11]*z[12]*z[5]*z[6]*z[8] - 14720*z[11]*z[12]*z[5]*z[6] + 23040*z[11]*z[12]*z[5]*z[7]*z[8] - 28672*z[11]*z[12]*z[5]*z[7] - 51200*z[11]*z[12]*z[5]*z[8] + 57360*z[11]*z[12]*z[5] + 46080*z[11]*z[12]*z[6]*z[7]*z[8] - 56960*z[11]*z[12]*z[6]*z[7] - 101632*z[11]*z[12]*z[6]*z[8] + 113280*z[11]*z[12]*z[6] - 197120*z[11]*z[12]*z[7]*z[8] + 215040*z[11]*z[12]*z[7] + 337920*z[11]*z[12]*z[8] - 720*z[11]*z[12]*z[9] - 707120*z[11]*z[12] + 5760*z[11]*z[2]*z[3]*z[4]*z[9] - 86400*z[11]*z[2]*z[3]*z[4] - 7120*z[11]*z[2]*z[3]*z[9] + 106800*z[11]*z[2]*z[3] - 12704*z[11]*z[2]*z[4]*z[9] + 190560*z[11]*z[2]*z[4] + 14160*z[11]*z[2]*z[9] - 212400*z[11]*z[2] - 24640*z[11]*z[3]*z[4]*z[9] + 369600*z[11]*z[3]*z[4] + 26880*z[11]*z[3]*z[9] - 403200*z[11]*z[3] + 42240*z[11]*z[4]*z[9] - 633600*z[11]*z[4] - 384*z[11]*z[5]*z[6]*z[7]*z[8]*z[9] + 5760*z[11]*z[5]*z[6]*z[7]*z[8] + 720*z[11]*z[5]*z[6]*z[7]*z[9] - 10800*z[11]*z[5]*z[6]*z[7] + 1440*z[11]*z[5]*z[6]*z[8]*z[9] - 21600*z[11]*z[5]*z[6]*z[8] - 1840*z[11]*z[5]*z[6]*z[9] + 27600*z[11]*z[5]*z[6] + 2880*z[11]*z[5]*z[7]*z[8]*z[9] - 43200*z[11]*z[5]*z[7]*z[8] - 3584*z[11]*z[5]*z[7]*z[9] + 53760*z[11]*z[5]*z[7] - 6400*z[11]*z[5]*z[8]*z[9] + 96000*z[11]*z[5]*z[8] + 7170*z[11]*z[5]*z[9] - 107550*z[11]*z[5] + 5760*z[11]*z[6]*z[7]*z[8]*z[9] - 86400*z[11]*z[6]*z[7]*z[8] - 7120*z[11]*z[6]*z[7]*z[9] + 106800*z[11]*z[6]*z[7] - 12704*z[11]*z[6]*z[8]*z[9] + 190560*z[11]*z[6]*z[8] + 14160*z[11]*z[6]*z[9] - 212400*z[11]*z[6] - 24640*z[11]*z[7]*z[8]*z[9] + 369600*z[11]*z[7]*z[8] + 26880*z[11]*z[7]*z[9] - 403200*z[11]*z[7] + 42240*z[11]*z[8]*z[9] - 633600*z[11]*z[8] - 88264*z[11]*z[9] + 1330680*z[11] + 11520*z[12]*z[2]*z[3]*z[4]*z[9] - 172800*z[12]*z[2]*z[3]*z[4] - 14240*z[12]*z[2]*z[3]*z[9] + 213600*z[12]*z[2]*z[3] - 25408*z[12]*z[2]*z[4]*z[9] + 381120*z[12]*z[2]*z[4] + 28320*z[12]*z[2]*z[9] - 424800*z[12]*z[2] - 49280*z[12]*z[3]*z[4]*z[9] + 739200*z[12]*z[3]*z[4] + 53760*z[12]*z[3]*z[9] - 806400*z[12]*z[3] + 84480*z[12]*z[4]*z[9] - 1267200*z[12]*z[4] - 768*z[12]*z[5]*z[6]*z[7]*z[8]*z[9] + 11520*z[12]*z[5]*z[6]*z[7]*z[8] + 1440*z[12]*z[5]*z[6]*z[7]*z[9] - 21600*z[12]*z[5]*z[6]*z[7] + 2880*z[12]*z[5]*z[6]*z[8]*z[9] - 43200*z[12]*z[5]*z[6]*z[8] - 3680*z[12]*z[5]*z[6]*z[9] + 55200*z[12]*z[5]*z[6] + 5760*z[12]*z[5]*z[7]*z[8]*z[9] - 86400*z[12]*z[5]*z[7]*z[8] - 7168*z[12]*z[5]*z[7]*z[9] + 107520*z[12]*z[5]*z[7] - 12800*z[12]*z[5]*z[8]*z[9] + 192000*z[12]*z[5]*z[8] + 14340*z[12]*z[5]*z[9] - 215100*z[12]*z[5] + 11520*z[12]*z[6]*z[7]*z[8]*z[9] - 172800*z[12]*z[6]*z[7]*z[8] - 14240*z[12]*z[6]*z[7]*z[9] + 213600*z[12]*z[6]*z[7] - 25408*z[12]*z[6]*z[8]*z[9] + 381120*z[12]*z[6]*z[8] + 28320*z[12]*z[6]*z[9] - 424800*z[12]*z[6] - 49280*z[12]*z[7]*z[8]*z[9] + 739200*z[12]*z[7]*z[8] + 53760*z[12]*z[7]*z[9] - 806400*z[12]*z[7] + 84480*z[12]*z[8]*z[9] - 1267200*z[12]*z[8] - 176720*z[12]*z[9] + 2664240*z[12] - 276480*z[2]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 518400*z[2]*z[3]*z[4]*z[5]*z[6]*z[7] + 1036800*z[2]*z[3]*z[4]*z[5]*z[6]*z[8] - 1324800*z[2]*z[3]*z[4]*z[5]*z[6] + 2073600*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] - 2580480*z[2]*z[3]*z[4]*z[5]*z[7] - 4608000*z[2]*z[3]*z[4]*z[5]*z[8] + 5162400*z[2]*z[3]*z[4]*z[5] + 4147200*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] - 5126400*z[2]*z[3]*z[4]*z[6]*z[7] - 9146880*z[2]*z[3]*z[4]*z[6]*z[8] + 10195200*z[2]*z[3]*z[4]*z[6] - 17740800*z[2]*z[3]*z[4]*z[7]*z[8] + 19353600*z[2]*z[3]*z[4]*z[7] + 30412800*z[2]*z[3]*z[4]*z[8] - 21600*z[2]*z[3]*z[4]*z[9] - 190091520*z[2]*z[3]*z[4] + 341760*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] - 640800*z[2]*z[3]*z[5]*z[6]*z[7] - 1281600*z[2]*z[3]*z[5]*z[6]*z[8] + 1637600*z[2]*z[3]*z[5]*z[6] - 2563200*z[2]*z[3]*z[5]*z[7]*z[8] + 3189760*z[2]*z[3]*z[5]*z[7] + 5696000*z[2]*z[3]*z[5]*z[8] - 6381300*z[2]*z[3]*z[5] - 5126400*z[2]*z[3]*z[6]*z[7]*z[8] + 6336800*z[2]*z[3]*z[6]*z[7] + 11306560*z[2]*z[3]*z[6]*z[8] - 12602400*z[2]*z[3]*z[6] + 21929600*z[2]*z[3]*z[7]*z[8] - 23923200*z[2]*z[3]*z[7] - 37593600*z[2]*z[3]*z[8] + 26700*z[2]*z[3]*z[9] + 198490440*z[2]*z[3] + 609792*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] - 1143360*z[2]*z[4]*z[5]*z[6]*z[7] - 2286720*z[2]*z[4]*z[5]*z[6]*z[8] + 2921920*z[2]*z[4]*z[5]*z[6] - 4573440*z[2]*z[4]*z[5]*z[7]*z[8] + 5691392*z[2]*z[4]*z[5]*z[7] + 10163200*z[2]*z[4]*z[5]*z[8] - 11385960*z[2]*z[4]*z[5] - 9146880*z[2]*z[4]*z[6]*z[7]*z[8] + 11306560*z[2]*z[4]*z[6]*z[7] + 20173952*z[2]*z[4]*z[6]*z[8] - 22486080*z[2]*z[4]*z[6] + 39128320*z[2]*z[4]*z[7]*z[8] - 42685440*z[2]*z[4]*z[7] - 67077120*z[2]*z[4]*z[8] + 47640*z[2]*z[4]*z[9] + 260335632*z[2]*z[4] - 679680*z[2]*z[5]*z[6]*z[7]*z[8] + 1274400*z[2]*z[5]*z[6]*z[7] + 2548800*z[2]*z[5]*z[6]*z[8] - 3256800*z[2]*z[5]*z[6] + 5097600*z[2]*z[5]*z[7]*z[8] - 6343680*z[2]*z[5]*z[7] - 11328000*z[2]*z[5]*z[8] + 12690900*z[2]*z[5] + 10195200*z[2]*z[6]*z[7]*z[8] - 12602400*z[2]*z[6]*z[7] - 22486080*z[2]*z[6]*z[8] + 25063200*z[2]*z[6] - 43612800*z[2]*z[7]*z[8] + 47577600*z[2]*z[7] + 74764800*z[2]*z[8] - 53100*z[2]*z[9] - 269267520*z[2] + 1182720*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] - 2217600*z[3]*z[4]*z[5]*z[6]*z[7] - 4435200*z[3]*z[4]*z[5]*z[6]*z[8] + 5667200*z[3]*z[4]*z[5]*z[6] - 8870400*z[3]*z[4]*z[5]*z[7]*z[8] + 11038720*z[3]*z[4]*z[5]*z[7] + 19712000*z[3]*z[4]*z[5]*z[8] - 22083600*z[3]*z[4]*z[5] - 17740800*z[3]*z[4]*z[6]*z[7]*z[8] + 21929600*z[3]*z[4]*z[6]*z[7] + 39128320*z[3]*z[4]*z[6]*z[8] - 43612800*z[3]*z[4]*z[6] + 75891200*z[3]*z[4]*z[7]*z[8] - 82790400*z[3]*z[4]*z[7] - 130099200*z[3]*z[4]*z[8] + 92400*z[3]*z[4]*z[9] + 442752480*z[3]*z[4] - 1290240*z[3]*z[5]*z[6]*z[7]*z[8] + 2419200*z[3]*z[5]*z[6]*z[7] + 4838400*z[3]*z[5]*z[6]*z[8] - 6182400*z[3]*z[5]*z[6] + 9676800*z[3]*z[5]*z[7]*z[8] - 12042240*z[3]*z[5]*z[7] - 21504000*z[3]*z[5]*z[8] + 24091200*z[3]*z[5] + 19353600*z[3]*z[6]*z[7]*z[8] - 23923200*z[3]*z[6]*z[7] - 42685440*z[3]*z[6]*z[8] + 47577600*z[3]*z[6] - 82790400*z[3]*z[7]*z[8] + 90316800*z[3]*z[7] + 141926400*z[3]*z[8] - 100800*z[3]*z[9] - 456135360*z[3] - 2027520*z[4]*z[5]*z[6]*z[7]*z[8] + 3801600*z[4]*z[5]*z[6]*z[7] + 7603200*z[4]*z[5]*z[6]*z[8] - 9715200*z[4]*z[5]*z[6] + 15206400*z[4]*z[5]*z[7]*z[8] - 18923520*z[4]*z[5]*z[7] - 33792000*z[4]*z[5]*z[8] + 37857600*z[4]*z[5] + 30412800*z[4]*z[6]*z[7]*z[8] - 37593600*z[4]*z[6]*z[7] - 67077120*z[4]*z[6]*z[8] + 74764800*z[4]*z[6] - 130099200*z[4]*z[7]*z[8] + 141926400*z[4]*z[7] + 223027200*z[4]*z[8] - 158400*z[4]*z[9] - 586922880*z[4] + 1440*z[5]*z[6]*z[7]*z[8]*z[9] + 40219392*z[5]*z[6]*z[7]*z[8] - 2700*z[5]*z[6]*z[7]*z[9] - 42548040*z[5]*z[6]*z[7] - 5400*z[5]*z[6]*z[8]*z[9] - 57073680*z[5]*z[6]*z[8] + 6900*z[5]*z[6]*z[9] + 59758080*z[5]*z[6] - 10800*z[5]*z[7]*z[8]*z[9] - 98926560*z[5]*z[7]*z[8] + 13440*z[5]*z[7]*z[9] + 103372992*z[5]*z[7] + 24000*z[5]*z[8]*z[9] + 135984000*z[5]*z[8] - 53775*z[5]*z[9]/2 - 281594505*z[5]/2 - 21600*z[6]*z[7]*z[8]*z[9] - 190091520*z[6]*z[7]*z[8] + 26700*z[6]*z[7]*z[9] + 198490440*z[6]*z[7] + 47640*z[6]*z[8]*z[9] + 260335632*z[6]*z[8] - 53100*z[6]*z[9] - 269267520*z[6] + 92400*z[7]*z[8]*z[9] + 442752480*z[7]*z[8] - 100800*z[7]*z[9] - 456135360*z[7] - 158400*z[8]*z[9] - 586922880*z[8] + 665115*z[9]/2 + 953310823)

In [425]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 12  # 
p = 1  # QAOA depth

hamiltonian_less_15_D_Equation_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_Equation_2_less_15))
final_circuit_less_15_D_Equation_2, result_less_15_D_Equation_2 = qaoa(num_qubits, hamiltonian_less_15_D_Equation_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [2.77083758 1.1574045 ]
Minimum expectation value: 27940346.86328125


In [427]:
top_solutions_less_15_D_Equation_2_less_15 = find_best_bitstrings(final_circuit_less_15_D_Equation_2, hamiltonian_less_15_D_Equation_2)

Top 5 bitstrings:
Bitstring: 100100010011, Cost: -1906621642.0000, Count: 3
Bitstring: 000000000001, Cost: -1906621642.0000, Count: 1
Bitstring: 010000010001, Cost: -1906621480.0000, Count: 1
Bitstring: 101000110001, Cost: -1906621360.0000, Count: 1
Bitstring: 100100100011, Cost: -1906621356.0000, Count: 2


In [428]:
bitstring_to_pm1(top_solutions_less_15_D_Equation_2_less_15, evaluate_hamiltonian_D_Equation_2_less_15)

["Bitstring: ('100100010011', -1906621642.0, 3), Evaluated cost: 4.0",
 "Bitstring: ('100100010011', -1906621642.0, 3), Evaluated cost: 4.0",
 "Bitstring: ('100100010011', -1906621642.0, 3), Evaluated cost: 169.0",
 "Bitstring: ('100100010011', -1906621642.0, 3), Evaluated cost: 289.0",
 "Bitstring: ('100100010011', -1906621642.0, 3), Evaluated cost: 289.0"]

In [429]:
paso1_D_Equation_2_less_31 = substitute_with_global_binary_symbols(D_Equation_2(x[1],x[2],x[3])**2, 5, base_name="b")
paso2_D_Equation_2_less_31 = remove_variable_exponents(paso1_D_Equation_2_less_31)
paso3_D_Equation_2_less_31 = substitute_with_spin_variables(paso2_D_Equation_2_less_31)

In [430]:
print(paso3_D_Equation_2_less_31)

36864*z_1*z_10*z_2*z_3*z_4*z_6*z_7*z_8 + 73728*z_1*z_10*z_2*z_3*z_4*z_6*z_7*z_9 - 285696*z_1*z_10*z_2*z_3*z_4*z_6*z_7 + 147456*z_1*z_10*z_2*z_3*z_4*z_6*z_8*z_9 - 571392*z_1*z_10*z_2*z_3*z_4*z_6*z_8 - 1142784*z_1*z_10*z_2*z_3*z_4*z_6*z_9 + 2605056*z_1*z_10*z_2*z_3*z_4*z_6 + 294912*z_1*z_10*z_2*z_3*z_4*z_7*z_8*z_9 - 1142784*z_1*z_10*z_2*z_3*z_4*z_7*z_8 - 2285568*z_1*z_10*z_2*z_3*z_4*z_7*z_9 + 5200896*z_1*z_10*z_2*z_3*z_4*z_7 - 4571136*z_1*z_10*z_2*z_3*z_4*z_8*z_9 + 10328064*z_1*z_10*z_2*z_3*z_4*z_8 + 20066304*z_1*z_10*z_2*z_3*z_4*z_9 - 35045376*z_1*z_10*z_2*z_3*z_4 + 73728*z_1*z_10*z_2*z_3*z_5*z_6*z_7*z_8 + 147456*z_1*z_10*z_2*z_3*z_5*z_6*z_7*z_9 - 571392*z_1*z_10*z_2*z_3*z_5*z_6*z_7 + 294912*z_1*z_10*z_2*z_3*z_5*z_6*z_8*z_9 - 1142784*z_1*z_10*z_2*z_3*z_5*z_6*z_8 - 2285568*z_1*z_10*z_2*z_3*z_5*z_6*z_9 + 5210112*z_1*z_10*z_2*z_3*z_5*z_6 + 589824*z_1*z_10*z_2*z_3*z_5*z_7*z_8*z_9 - 2285568*z_1*z_10*z_2*z_3*z_5*z_7*z_8 - 4571136*z_1*z_10*z_2*z_3*z_5*z_7*z_9 + 10401792*z_1*z_10*z_2*z_3*z_5*z_

In [432]:
def evaluate_hamiltonian_D_Equation_2_less_31(z):
    return(36864*z[1]*z[10]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] + 73728*z[1]*z[10]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] - 285696*z[1]*z[10]*z[2]*z[3]*z[4]*z[6]*z[7] + 147456*z[1]*z[10]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] - 571392*z[1]*z[10]*z[2]*z[3]*z[4]*z[6]*z[8] - 1142784*z[1]*z[10]*z[2]*z[3]*z[4]*z[6]*z[9] + 2605056*z[1]*z[10]*z[2]*z[3]*z[4]*z[6] + 294912*z[1]*z[10]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] - 1142784*z[1]*z[10]*z[2]*z[3]*z[4]*z[7]*z[8] - 2285568*z[1]*z[10]*z[2]*z[3]*z[4]*z[7]*z[9] + 5200896*z[1]*z[10]*z[2]*z[3]*z[4]*z[7] - 4571136*z[1]*z[10]*z[2]*z[3]*z[4]*z[8]*z[9] + 10328064*z[1]*z[10]*z[2]*z[3]*z[4]*z[8] + 20066304*z[1]*z[10]*z[2]*z[3]*z[4]*z[9] - 35045376*z[1]*z[10]*z[2]*z[3]*z[4] + 73728*z[1]*z[10]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] + 147456*z[1]*z[10]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] - 571392*z[1]*z[10]*z[2]*z[3]*z[5]*z[6]*z[7] + 294912*z[1]*z[10]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] - 1142784*z[1]*z[10]*z[2]*z[3]*z[5]*z[6]*z[8] - 2285568*z[1]*z[10]*z[2]*z[3]*z[5]*z[6]*z[9] + 5210112*z[1]*z[10]*z[2]*z[3]*z[5]*z[6] + 589824*z[1]*z[10]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] - 2285568*z[1]*z[10]*z[2]*z[3]*z[5]*z[7]*z[8] - 4571136*z[1]*z[10]*z[2]*z[3]*z[5]*z[7]*z[9] + 10401792*z[1]*z[10]*z[2]*z[3]*z[5]*z[7] - 9142272*z[1]*z[10]*z[2]*z[3]*z[5]*z[8]*z[9] + 20656128*z[1]*z[10]*z[2]*z[3]*z[5]*z[8] + 40132608*z[1]*z[10]*z[2]*z[3]*z[5]*z[9] - 70090752*z[1]*z[10]*z[2]*z[3]*z[5] - 142848*z[1]*z[10]*z[2]*z[3]*z[6]*z[7]*z[8] - 285696*z[1]*z[10]*z[2]*z[3]*z[6]*z[7]*z[9] + 1107072*z[1]*z[10]*z[2]*z[3]*z[6]*z[7] - 571392*z[1]*z[10]*z[2]*z[3]*z[6]*z[8]*z[9] + 2214144*z[1]*z[10]*z[2]*z[3]*z[6]*z[8] + 4428288*z[1]*z[10]*z[2]*z[3]*z[6]*z[9] - 10094592*z[1]*z[10]*z[2]*z[3]*z[6] - 1142784*z[1]*z[10]*z[2]*z[3]*z[7]*z[8]*z[9] + 4428288*z[1]*z[10]*z[2]*z[3]*z[7]*z[8] + 8856576*z[1]*z[10]*z[2]*z[3]*z[7]*z[9] - 20153472*z[1]*z[10]*z[2]*z[3]*z[7] + 17713152*z[1]*z[10]*z[2]*z[3]*z[8]*z[9] - 40021248*z[1]*z[10]*z[2]*z[3]*z[8] - 77756928*z[1]*z[10]*z[2]*z[3]*z[9] + 135800832*z[1]*z[10]*z[2]*z[3] + 147456*z[1]*z[10]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] + 294912*z[1]*z[10]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] - 1142784*z[1]*z[10]*z[2]*z[4]*z[5]*z[6]*z[7] + 589824*z[1]*z[10]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] - 2285568*z[1]*z[10]*z[2]*z[4]*z[5]*z[6]*z[8] - 4571136*z[1]*z[10]*z[2]*z[4]*z[5]*z[6]*z[9] + 10420224*z[1]*z[10]*z[2]*z[4]*z[5]*z[6] + 1179648*z[1]*z[10]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] - 4571136*z[1]*z[10]*z[2]*z[4]*z[5]*z[7]*z[8] - 9142272*z[1]*z[10]*z[2]*z[4]*z[5]*z[7]*z[9] + 20803584*z[1]*z[10]*z[2]*z[4]*z[5]*z[7] - 18284544*z[1]*z[10]*z[2]*z[4]*z[5]*z[8]*z[9] + 41312256*z[1]*z[10]*z[2]*z[4]*z[5]*z[8] + 80265216*z[1]*z[10]*z[2]*z[4]*z[5]*z[9] - 140181504*z[1]*z[10]*z[2]*z[4]*z[5] - 285696*z[1]*z[10]*z[2]*z[4]*z[6]*z[7]*z[8] - 571392*z[1]*z[10]*z[2]*z[4]*z[6]*z[7]*z[9] + 2214144*z[1]*z[10]*z[2]*z[4]*z[6]*z[7] - 1142784*z[1]*z[10]*z[2]*z[4]*z[6]*z[8]*z[9] + 4428288*z[1]*z[10]*z[2]*z[4]*z[6]*z[8] + 8856576*z[1]*z[10]*z[2]*z[4]*z[6]*z[9] - 20189184*z[1]*z[10]*z[2]*z[4]*z[6] - 2285568*z[1]*z[10]*z[2]*z[4]*z[7]*z[8]*z[9] + 8856576*z[1]*z[10]*z[2]*z[4]*z[7]*z[8] + 17713152*z[1]*z[10]*z[2]*z[4]*z[7]*z[9] - 40306944*z[1]*z[10]*z[2]*z[4]*z[7] + 35426304*z[1]*z[10]*z[2]*z[4]*z[8]*z[9] - 80042496*z[1]*z[10]*z[2]*z[4]*z[8] - 155513856*z[1]*z[10]*z[2]*z[4]*z[9] + 271601664*z[1]*z[10]*z[2]*z[4] - 571392*z[1]*z[10]*z[2]*z[5]*z[6]*z[7]*z[8] - 1142784*z[1]*z[10]*z[2]*z[5]*z[6]*z[7]*z[9] + 4428288*z[1]*z[10]*z[2]*z[5]*z[6]*z[7] - 2285568*z[1]*z[10]*z[2]*z[5]*z[6]*z[8]*z[9] + 8856576*z[1]*z[10]*z[2]*z[5]*z[6]*z[8] + 17713152*z[1]*z[10]*z[2]*z[5]*z[6]*z[9] - 40378368*z[1]*z[10]*z[2]*z[5]*z[6] - 4571136*z[1]*z[10]*z[2]*z[5]*z[7]*z[8]*z[9] + 17713152*z[1]*z[10]*z[2]*z[5]*z[7]*z[8] + 35426304*z[1]*z[10]*z[2]*z[5]*z[7]*z[9] - 80613888*z[1]*z[10]*z[2]*z[5]*z[7] + 70852608*z[1]*z[10]*z[2]*z[5]*z[8]*z[9] - 160084992*z[1]*z[10]*z[2]*z[5]*z[8] - 311027712*z[1]*z[10]*z[2]*z[5]*z[9] + 543203328*z[1]*z[10]*z[2]*z[5] + 748032*z[1]*z[10]*z[2]*z[6]*z[7]*z[8] + 1496064*z[1]*z[10]*z[2]*z[6]*z[7]*z[9] - 5797248*z[1]*z[10]*z[2]*z[6]*z[7] + 2992128*z[1]*z[10]*z[2]*z[6]*z[8]*z[9] - 11594496*z[1]*z[10]*z[2]*z[6]*z[8] - 23188992*z[1]*z[10]*z[2]*z[6]*z[9] + 52860928*z[1]*z[10]*z[2]*z[6] + 5984256*z[1]*z[10]*z[2]*z[7]*z[8]*z[9] - 23188992*z[1]*z[10]*z[2]*z[7]*z[8] - 46377984*z[1]*z[10]*z[2]*z[7]*z[9] + 105534848*z[1]*z[10]*z[2]*z[7] - 92755968*z[1]*z[10]*z[2]*z[8]*z[9] + 209573632*z[1]*z[10]*z[2]*z[8] + 407178752*z[1]*z[10]*z[2]*z[9] - 711129088*z[1]*z[10]*z[2] + 294912*z[1]*z[10]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 589824*z[1]*z[10]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] - 2285568*z[1]*z[10]*z[3]*z[4]*z[5]*z[6]*z[7] + 1179648*z[1]*z[10]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] - 4571136*z[1]*z[10]*z[3]*z[4]*z[5]*z[6]*z[8] - 9142272*z[1]*z[10]*z[3]*z[4]*z[5]*z[6]*z[9] + 20840448*z[1]*z[10]*z[3]*z[4]*z[5]*z[6] + 2359296*z[1]*z[10]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 9142272*z[1]*z[10]*z[3]*z[4]*z[5]*z[7]*z[8] - 18284544*z[1]*z[10]*z[3]*z[4]*z[5]*z[7]*z[9] + 41607168*z[1]*z[10]*z[3]*z[4]*z[5]*z[7] - 36569088*z[1]*z[10]*z[3]*z[4]*z[5]*z[8]*z[9] + 82624512*z[1]*z[10]*z[3]*z[4]*z[5]*z[8] + 160530432*z[1]*z[10]*z[3]*z[4]*z[5]*z[9] - 280363008*z[1]*z[10]*z[3]*z[4]*z[5] - 571392*z[1]*z[10]*z[3]*z[4]*z[6]*z[7]*z[8] - 1142784*z[1]*z[10]*z[3]*z[4]*z[6]*z[7]*z[9] + 4428288*z[1]*z[10]*z[3]*z[4]*z[6]*z[7] - 2285568*z[1]*z[10]*z[3]*z[4]*z[6]*z[8]*z[9] + 8856576*z[1]*z[10]*z[3]*z[4]*z[6]*z[8] + 17713152*z[1]*z[10]*z[3]*z[4]*z[6]*z[9] - 40378368*z[1]*z[10]*z[3]*z[4]*z[6] - 4571136*z[1]*z[10]*z[3]*z[4]*z[7]*z[8]*z[9] + 17713152*z[1]*z[10]*z[3]*z[4]*z[7]*z[8] + 35426304*z[1]*z[10]*z[3]*z[4]*z[7]*z[9] - 80613888*z[1]*z[10]*z[3]*z[4]*z[7] + 70852608*z[1]*z[10]*z[3]*z[4]*z[8]*z[9] - 160084992*z[1]*z[10]*z[3]*z[4]*z[8] - 311027712*z[1]*z[10]*z[3]*z[4]*z[9] + 543203328*z[1]*z[10]*z[3]*z[4] - 1142784*z[1]*z[10]*z[3]*z[5]*z[6]*z[7]*z[8] - 2285568*z[1]*z[10]*z[3]*z[5]*z[6]*z[7]*z[9] + 8856576*z[1]*z[10]*z[3]*z[5]*z[6]*z[7] - 4571136*z[1]*z[10]*z[3]*z[5]*z[6]*z[8]*z[9] + 17713152*z[1]*z[10]*z[3]*z[5]*z[6]*z[8] + 35426304*z[1]*z[10]*z[3]*z[5]*z[6]*z[9] - 80756736*z[1]*z[10]*z[3]*z[5]*z[6] - 9142272*z[1]*z[10]*z[3]*z[5]*z[7]*z[8]*z[9] + 35426304*z[1]*z[10]*z[3]*z[5]*z[7]*z[8] + 70852608*z[1]*z[10]*z[3]*z[5]*z[7]*z[9] - 161227776*z[1]*z[10]*z[3]*z[5]*z[7] + 141705216*z[1]*z[10]*z[3]*z[5]*z[8]*z[9] - 320169984*z[1]*z[10]*z[3]*z[5]*z[8] - 622055424*z[1]*z[10]*z[3]*z[5]*z[9] + 1086406656*z[1]*z[10]*z[3]*z[5] + 1486848*z[1]*z[10]*z[3]*z[6]*z[7]*z[8] + 2973696*z[1]*z[10]*z[3]*z[6]*z[7]*z[9] - 11523072*z[1]*z[10]*z[3]*z[6]*z[7] + 5947392*z[1]*z[10]*z[3]*z[6]*z[8]*z[9] - 23046144*z[1]*z[10]*z[3]*z[6]*z[8] - 46092288*z[1]*z[10]*z[3]*z[6]*z[9] + 105070592*z[1]*z[10]*z[3]*z[6] + 11894784*z[1]*z[10]*z[3]*z[7]*z[8]*z[9] - 46092288*z[1]*z[10]*z[3]*z[7]*z[8] - 92184576*z[1]*z[10]*z[3]*z[7]*z[9] + 209769472*z[1]*z[10]*z[3]*z[7] - 184369152*z[1]*z[10]*z[3]*z[8]*z[9] + 416565248*z[1]*z[10]*z[3]*z[8] + 809340928*z[1]*z[10]*z[3]*z[9] - 1413496832*z[1]*z[10]*z[3] - 2285568*z[1]*z[10]*z[4]*z[5]*z[6]*z[7]*z[8] - 4571136*z[1]*z[10]*z[4]*z[5]*z[6]*z[7]*z[9] + 17713152*z[1]*z[10]*z[4]*z[5]*z[6]*z[7] - 9142272*z[1]*z[10]*z[4]*z[5]*z[6]*z[8]*z[9] + 35426304*z[1]*z[10]*z[4]*z[5]*z[6]*z[8] + 70852608*z[1]*z[10]*z[4]*z[5]*z[6]*z[9] - 161513472*z[1]*z[10]*z[4]*z[5]*z[6] - 18284544*z[1]*z[10]*z[4]*z[5]*z[7]*z[8]*z[9] + 70852608*z[1]*z[10]*z[4]*z[5]*z[7]*z[8] + 141705216*z[1]*z[10]*z[4]*z[5]*z[7]*z[9] - 322455552*z[1]*z[10]*z[4]*z[5]*z[7] + 283410432*z[1]*z[10]*z[4]*z[5]*z[8]*z[9] - 640339968*z[1]*z[10]*z[4]*z[5]*z[8] - 1244110848*z[1]*z[10]*z[4]*z[5]*z[9] + 2172813312*z[1]*z[10]*z[4]*z[5] + 2899968*z[1]*z[10]*z[4]*z[6]*z[7]*z[8] + 5799936*z[1]*z[10]*z[4]*z[6]*z[7]*z[9] - 22474752*z[1]*z[10]*z[4]*z[6]*z[7] + 11599872*z[1]*z[10]*z[4]*z[6]*z[8]*z[9] - 44949504*z[1]*z[10]*z[4]*z[6]*z[8] - 89899008*z[1]*z[10]*z[4]*z[6]*z[9] + 204931072*z[1]*z[10]*z[4]*z[6] + 23199744*z[1]*z[10]*z[4]*z[7]*z[8]*z[9] - 89899008*z[1]*z[10]*z[4]*z[7]*z[8] - 179798016*z[1]*z[10]*z[4]*z[7]*z[9] + 409137152*z[1]*z[10]*z[4]*z[7] - 359596032*z[1]*z[10]*z[4]*z[8]*z[9] + 812474368*z[1]*z[10]*z[4]*z[8] + 1578549248*z[1]*z[10]*z[4]*z[9] - 2756902912*z[1]*z[10]*z[4] + 5210112*z[1]*z[10]*z[5]*z[6]*z[7]*z[8] + 10420224*z[1]*z[10]*z[5]*z[6]*z[7]*z[9] - 40378368*z[1]*z[10]*z[5]*z[6]*z[7] + 20840448*z[1]*z[10]*z[5]*z[6]*z[8]*z[9] - 80756736*z[1]*z[10]*z[5]*z[6]*z[8] - 161513472*z[1]*z[10]*z[5]*z[6]*z[9] + 368181248*z[1]*z[10]*z[5]*z[6] + 41680896*z[1]*z[10]*z[5]*z[7]*z[8]*z[9] - 161513472*z[1]*z[10]*z[5]*z[7]*z[8] - 323026944*z[1]*z[10]*z[5]*z[7]*z[9] + 735059968*z[1]*z[10]*z[5]*z[7] - 646053888*z[1]*z[10]*z[5]*z[8]*z[9] + 1459699712*z[1]*z[10]*z[5]*z[8] + 2836037632*z[1]*z[10]*z[5]*z[9] - 4953079808*z[1]*z[10]*z[5] - 5898432*z[1]*z[10]*z[6]*z[7]*z[8] - 11796864*z[1]*z[10]*z[6]*z[7]*z[9] + 45712848*z[1]*z[10]*z[6]*z[7] - 23593728*z[1]*z[10]*z[6]*z[8]*z[9] + 91425696*z[1]*z[10]*z[6]*z[8] + 182851392*z[1]*z[10]*z[6]*z[9] - 416822528*z[1]*z[10]*z[6] - 47187456*z[1]*z[10]*z[7]*z[8]*z[9] + 182851392*z[1]*z[10]*z[7]*z[8] + 365702784*z[1]*z[10]*z[7]*z[9] - 832170448*z[1]*z[10]*z[7] + 731405568*z[1]*z[10]*z[8]*z[9] - 1652544032*z[1]*z[10]*z[8] - 3210713152*z[1]*z[10]*z[9] + 5607442688*z[1]*z[10] - 192*z[1]*z[11]*z[12]*z[2]*z[3]*z[4] - 384*z[1]*z[11]*z[12]*z[2]*z[3]*z[5] + 744*z[1]*z[11]*z[12]*z[2]*z[3] - 768*z[1]*z[11]*z[12]*z[2]*z[4]*z[5] + 1488*z[1]*z[11]*z[12]*z[2]*z[4] + 2976*z[1]*z[11]*z[12]*z[2]*z[5] - 3896*z[1]*z[11]*z[12]*z[2] - 1536*z[1]*z[11]*z[12]*z[3]*z[4]*z[5] + 2976*z[1]*z[11]*z[12]*z[3]*z[4] + 5952*z[1]*z[11]*z[12]*z[3]*z[5] - 7744*z[1]*z[11]*z[12]*z[3] + 11904*z[1]*z[11]*z[12]*z[4]*z[5] - 15104*z[1]*z[11]*z[12]*z[4] - 27136*z[1]*z[11]*z[12]*z[5] + 30721*z[1]*z[11]*z[12] - 384*z[1]*z[11]*z[13]*z[2]*z[3]*z[4] - 768*z[1]*z[11]*z[13]*z[2]*z[3]*z[5] + 1488*z[1]*z[11]*z[13]*z[2]*z[3] - 1536*z[1]*z[11]*z[13]*z[2]*z[4]*z[5] + 2976*z[1]*z[11]*z[13]*z[2]*z[4] + 5952*z[1]*z[11]*z[13]*z[2]*z[5] - 7792*z[1]*z[11]*z[13]*z[2] - 3072*z[1]*z[11]*z[13]*z[3]*z[4]*z[5] + 5952*z[1]*z[11]*z[13]*z[3]*z[4] + 11904*z[1]*z[11]*z[13]*z[3]*z[5] - 15488*z[1]*z[11]*z[13]*z[3] + 23808*z[1]*z[11]*z[13]*z[4]*z[5] - 30208*z[1]*z[11]*z[13]*z[4] - 54272*z[1]*z[11]*z[13]*z[5] + 61442*z[1]*z[11]*z[13] - 768*z[1]*z[11]*z[14]*z[2]*z[3]*z[4] - 1536*z[1]*z[11]*z[14]*z[2]*z[3]*z[5] + 2976*z[1]*z[11]*z[14]*z[2]*z[3] - 3072*z[1]*z[11]*z[14]*z[2]*z[4]*z[5] + 5952*z[1]*z[11]*z[14]*z[2]*z[4] + 11904*z[1]*z[11]*z[14]*z[2]*z[5] - 15584*z[1]*z[11]*z[14]*z[2] - 6144*z[1]*z[11]*z[14]*z[3]*z[4]*z[5] + 11904*z[1]*z[11]*z[14]*z[3]*z[4] + 23808*z[1]*z[11]*z[14]*z[3]*z[5] - 30976*z[1]*z[11]*z[14]*z[3] + 47616*z[1]*z[11]*z[14]*z[4]*z[5] - 60416*z[1]*z[11]*z[14]*z[4] - 108544*z[1]*z[11]*z[14]*z[5] + 122884*z[1]*z[11]*z[14] - 1536*z[1]*z[11]*z[15]*z[2]*z[3]*z[4] - 3072*z[1]*z[11]*z[15]*z[2]*z[3]*z[5] + 5952*z[1]*z[11]*z[15]*z[2]*z[3] - 6144*z[1]*z[11]*z[15]*z[2]*z[4]*z[5] + 11904*z[1]*z[11]*z[15]*z[2]*z[4] + 23808*z[1]*z[11]*z[15]*z[2]*z[5] - 31168*z[1]*z[11]*z[15]*z[2] - 12288*z[1]*z[11]*z[15]*z[3]*z[4]*z[5] + 23808*z[1]*z[11]*z[15]*z[3]*z[4] + 47616*z[1]*z[11]*z[15]*z[3]*z[5] - 61952*z[1]*z[11]*z[15]*z[3] + 95232*z[1]*z[11]*z[15]*z[4]*z[5] - 120832*z[1]*z[11]*z[15]*z[4] - 217088*z[1]*z[11]*z[15]*z[5] + 245768*z[1]*z[11]*z[15] + 2976*z[1]*z[11]*z[2]*z[3]*z[4] + 5952*z[1]*z[11]*z[2]*z[3]*z[5] - 11532*z[1]*z[11]*z[2]*z[3] + 11904*z[1]*z[11]*z[2]*z[4]*z[5] - 23064*z[1]*z[11]*z[2]*z[4] - 46128*z[1]*z[11]*z[2]*z[5] + 60388*z[1]*z[11]*z[2] + 23808*z[1]*z[11]*z[3]*z[4]*z[5] - 46128*z[1]*z[11]*z[3]*z[4] - 92256*z[1]*z[11]*z[3]*z[5] + 120032*z[1]*z[11]*z[3] - 184512*z[1]*z[11]*z[4]*z[5] + 234112*z[1]*z[11]*z[4] + 420608*z[1]*z[11]*z[5] - 952351*z[1]*z[11]/2 - 768*z[1]*z[12]*z[13]*z[2]*z[3]*z[4] - 1536*z[1]*z[12]*z[13]*z[2]*z[3]*z[5] + 2976*z[1]*z[12]*z[13]*z[2]*z[3] - 3072*z[1]*z[12]*z[13]*z[2]*z[4]*z[5] + 5952*z[1]*z[12]*z[13]*z[2]*z[4] + 11904*z[1]*z[12]*z[13]*z[2]*z[5] - 15584*z[1]*z[12]*z[13]*z[2] - 6144*z[1]*z[12]*z[13]*z[3]*z[4]*z[5] + 11904*z[1]*z[12]*z[13]*z[3]*z[4] + 23808*z[1]*z[12]*z[13]*z[3]*z[5] - 30976*z[1]*z[12]*z[13]*z[3] + 47616*z[1]*z[12]*z[13]*z[4]*z[5] - 60416*z[1]*z[12]*z[13]*z[4] - 108544*z[1]*z[12]*z[13]*z[5] + 122884*z[1]*z[12]*z[13] - 1536*z[1]*z[12]*z[14]*z[2]*z[3]*z[4] - 3072*z[1]*z[12]*z[14]*z[2]*z[3]*z[5] + 5952*z[1]*z[12]*z[14]*z[2]*z[3] - 6144*z[1]*z[12]*z[14]*z[2]*z[4]*z[5] + 11904*z[1]*z[12]*z[14]*z[2]*z[4] + 23808*z[1]*z[12]*z[14]*z[2]*z[5] - 31168*z[1]*z[12]*z[14]*z[2] - 12288*z[1]*z[12]*z[14]*z[3]*z[4]*z[5] + 23808*z[1]*z[12]*z[14]*z[3]*z[4] + 47616*z[1]*z[12]*z[14]*z[3]*z[5] - 61952*z[1]*z[12]*z[14]*z[3] + 95232*z[1]*z[12]*z[14]*z[4]*z[5] - 120832*z[1]*z[12]*z[14]*z[4] - 217088*z[1]*z[12]*z[14]*z[5] + 245768*z[1]*z[12]*z[14] - 3072*z[1]*z[12]*z[15]*z[2]*z[3]*z[4] - 6144*z[1]*z[12]*z[15]*z[2]*z[3]*z[5] + 11904*z[1]*z[12]*z[15]*z[2]*z[3] - 12288*z[1]*z[12]*z[15]*z[2]*z[4]*z[5] + 23808*z[1]*z[12]*z[15]*z[2]*z[4] + 47616*z[1]*z[12]*z[15]*z[2]*z[5] - 62336*z[1]*z[12]*z[15]*z[2] - 24576*z[1]*z[12]*z[15]*z[3]*z[4]*z[5] + 47616*z[1]*z[12]*z[15]*z[3]*z[4] + 95232*z[1]*z[12]*z[15]*z[3]*z[5] - 123904*z[1]*z[12]*z[15]*z[3] + 190464*z[1]*z[12]*z[15]*z[4]*z[5] - 241664*z[1]*z[12]*z[15]*z[4] - 434176*z[1]*z[12]*z[15]*z[5] + 491536*z[1]*z[12]*z[15] + 5952*z[1]*z[12]*z[2]*z[3]*z[4] + 11904*z[1]*z[12]*z[2]*z[3]*z[5] - 23064*z[1]*z[12]*z[2]*z[3] + 23808*z[1]*z[12]*z[2]*z[4]*z[5] - 46128*z[1]*z[12]*z[2]*z[4] - 92256*z[1]*z[12]*z[2]*z[5] + 120776*z[1]*z[12]*z[2] + 47616*z[1]*z[12]*z[3]*z[4]*z[5] - 92256*z[1]*z[12]*z[3]*z[4] - 184512*z[1]*z[12]*z[3]*z[5] + 240064*z[1]*z[12]*z[3] - 369024*z[1]*z[12]*z[4]*z[5] + 468224*z[1]*z[12]*z[4] + 841216*z[1]*z[12]*z[5] - 952351*z[1]*z[12] - 3072*z[1]*z[13]*z[14]*z[2]*z[3]*z[4] - 6144*z[1]*z[13]*z[14]*z[2]*z[3]*z[5] + 11904*z[1]*z[13]*z[14]*z[2]*z[3] - 12288*z[1]*z[13]*z[14]*z[2]*z[4]*z[5] + 23808*z[1]*z[13]*z[14]*z[2]*z[4] + 47616*z[1]*z[13]*z[14]*z[2]*z[5] - 62336*z[1]*z[13]*z[14]*z[2] - 24576*z[1]*z[13]*z[14]*z[3]*z[4]*z[5] + 47616*z[1]*z[13]*z[14]*z[3]*z[4] + 95232*z[1]*z[13]*z[14]*z[3]*z[5] - 123904*z[1]*z[13]*z[14]*z[3] + 190464*z[1]*z[13]*z[14]*z[4]*z[5] - 241664*z[1]*z[13]*z[14]*z[4] - 434176*z[1]*z[13]*z[14]*z[5] + 491536*z[1]*z[13]*z[14] - 6144*z[1]*z[13]*z[15]*z[2]*z[3]*z[4] - 12288*z[1]*z[13]*z[15]*z[2]*z[3]*z[5] + 23808*z[1]*z[13]*z[15]*z[2]*z[3] - 24576*z[1]*z[13]*z[15]*z[2]*z[4]*z[5] + 47616*z[1]*z[13]*z[15]*z[2]*z[4] + 95232*z[1]*z[13]*z[15]*z[2]*z[5] - 124672*z[1]*z[13]*z[15]*z[2] - 49152*z[1]*z[13]*z[15]*z[3]*z[4]*z[5] + 95232*z[1]*z[13]*z[15]*z[3]*z[4] + 190464*z[1]*z[13]*z[15]*z[3]*z[5] - 247808*z[1]*z[13]*z[15]*z[3] + 380928*z[1]*z[13]*z[15]*z[4]*z[5] - 483328*z[1]*z[13]*z[15]*z[4] - 868352*z[1]*z[13]*z[15]*z[5] + 983072*z[1]*z[13]*z[15] + 11904*z[1]*z[13]*z[2]*z[3]*z[4] + 23808*z[1]*z[13]*z[2]*z[3]*z[5] - 46128*z[1]*z[13]*z[2]*z[3] + 47616*z[1]*z[13]*z[2]*z[4]*z[5] - 92256*z[1]*z[13]*z[2]*z[4] - 184512*z[1]*z[13]*z[2]*z[5] + 241552*z[1]*z[13]*z[2] + 95232*z[1]*z[13]*z[3]*z[4]*z[5] - 184512*z[1]*z[13]*z[3]*z[4] - 369024*z[1]*z[13]*z[3]*z[5] + 480128*z[1]*z[13]*z[3] - 738048*z[1]*z[13]*z[4]*z[5] + 936448*z[1]*z[13]*z[4] + 1682432*z[1]*z[13]*z[5] - 1904702*z[1]*z[13] - 12288*z[1]*z[14]*z[15]*z[2]*z[3]*z[4] - 24576*z[1]*z[14]*z[15]*z[2]*z[3]*z[5] + 47616*z[1]*z[14]*z[15]*z[2]*z[3] - 49152*z[1]*z[14]*z[15]*z[2]*z[4]*z[5] + 95232*z[1]*z[14]*z[15]*z[2]*z[4] + 190464*z[1]*z[14]*z[15]*z[2]*z[5] - 249344*z[1]*z[14]*z[15]*z[2] - 98304*z[1]*z[14]*z[15]*z[3]*z[4]*z[5] + 190464*z[1]*z[14]*z[15]*z[3]*z[4] + 380928*z[1]*z[14]*z[15]*z[3]*z[5] - 495616*z[1]*z[14]*z[15]*z[3] + 761856*z[1]*z[14]*z[15]*z[4]*z[5] - 966656*z[1]*z[14]*z[15]*z[4] - 1736704*z[1]*z[14]*z[15]*z[5] + 1966144*z[1]*z[14]*z[15] + 23808*z[1]*z[14]*z[2]*z[3]*z[4] + 47616*z[1]*z[14]*z[2]*z[3]*z[5] - 92256*z[1]*z[14]*z[2]*z[3] + 95232*z[1]*z[14]*z[2]*z[4]*z[5] - 184512*z[1]*z[14]*z[2]*z[4] - 369024*z[1]*z[14]*z[2]*z[5] + 483104*z[1]*z[14]*z[2] + 190464*z[1]*z[14]*z[3]*z[4]*z[5] - 369024*z[1]*z[14]*z[3]*z[4] - 738048*z[1]*z[14]*z[3]*z[5] + 960256*z[1]*z[14]*z[3] - 1476096*z[1]*z[14]*z[4]*z[5] + 1872896*z[1]*z[14]*z[4] + 3364864*z[1]*z[14]*z[5] - 3809404*z[1]*z[14] + 47616*z[1]*z[15]*z[2]*z[3]*z[4] + 95232*z[1]*z[15]*z[2]*z[3]*z[5] - 184512*z[1]*z[15]*z[2]*z[3] + 190464*z[1]*z[15]*z[2]*z[4]*z[5] - 369024*z[1]*z[15]*z[2]*z[4] - 738048*z[1]*z[15]*z[2]*z[5] + 966208*z[1]*z[15]*z[2] + 380928*z[1]*z[15]*z[3]*z[4]*z[5] - 738048*z[1]*z[15]*z[3]*z[4] - 1476096*z[1]*z[15]*z[3]*z[5] + 1920512*z[1]*z[15]*z[3] - 2952192*z[1]*z[15]*z[4]*z[5] + 3745792*z[1]*z[15]*z[4] + 6729728*z[1]*z[15]*z[5] - 7618808*z[1]*z[15] - 1084930560*z[1]*z[2]*z[3]*z[4]*z[5] + 18432*z[1]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 71424*z[1]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] - 142848*z[1]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] + 374016*z[1]*z[2]*z[3]*z[4]*z[6]*z[7] - 285696*z[1]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] + 743424*z[1]*z[2]*z[3]*z[4]*z[6]*z[8] + 1449984*z[1]*z[2]*z[3]*z[4]*z[6]*z[9] - 2949216*z[1]*z[2]*z[3]*z[4]*z[6] - 571392*z[1]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] + 1484544*z[1]*z[2]*z[3]*z[4]*z[7]*z[8] + 2895360*z[1]*z[2]*z[3]*z[4]*z[7]*z[9] - 5880576*z[1]*z[2]*z[3]*z[4]*z[7] + 5753856*z[1]*z[2]*z[3]*z[4]*z[8]*z[9] - 11618304*z[1]*z[2]*z[3]*z[4]*z[8] - 22093824*z[1]*z[2]*z[3]*z[4]*z[9] + 1160147712*z[1]*z[2]*z[3]*z[4] + 36864*z[1]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 142848*z[1]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] - 285696*z[1]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] + 748032*z[1]*z[2]*z[3]*z[5]*z[6]*z[7] - 571392*z[1]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] + 1486848*z[1]*z[2]*z[3]*z[5]*z[6]*z[8] + 2899968*z[1]*z[2]*z[3]*z[5]*z[6]*z[9] - 5898432*z[1]*z[2]*z[3]*z[5]*z[6] - 1142784*z[1]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] + 2969088*z[1]*z[2]*z[3]*z[5]*z[7]*z[8] + 5790720*z[1]*z[2]*z[3]*z[5]*z[7]*z[9] - 11761152*z[1]*z[2]*z[3]*z[5]*z[7] + 11507712*z[1]*z[2]*z[3]*z[5]*z[8]*z[9] - 23236608*z[1]*z[2]*z[3]*z[5]*z[8] - 44187648*z[1]*z[2]*z[3]*z[5]*z[9] + 1654531584*z[1]*z[2]*z[3]*z[5] - 71424*z[1]*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] + 276768*z[1]*z[2]*z[3]*z[6]*z[7]*z[8] + 553536*z[1]*z[2]*z[3]*z[6]*z[7]*z[9] - 1449312*z[1]*z[2]*z[3]*z[6]*z[7] + 1107072*z[1]*z[2]*z[3]*z[6]*z[8]*z[9] - 2880768*z[1]*z[2]*z[3]*z[6]*z[8] - 5618688*z[1]*z[2]*z[3]*z[6]*z[9] + 11428212*z[1]*z[2]*z[3]*z[6] + 2214144*z[1]*z[2]*z[3]*z[7]*z[8]*z[9] - 5752608*z[1]*z[2]*z[3]*z[7]*z[8] - 11219520*z[1]*z[2]*z[3]*z[7]*z[9] + 22787232*z[1]*z[2]*z[3]*z[7] - 22296192*z[1]*z[2]*z[3]*z[8]*z[9] + 45020928*z[1]*z[2]*z[3]*z[8] + 85613568*z[1]*z[2]*z[3]*z[9] - 1762700424*z[1]*z[2]*z[3] + 73728*z[1]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 285696*z[1]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] - 571392*z[1]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] + 1496064*z[1]*z[2]*z[4]*z[5]*z[6]*z[7] - 1142784*z[1]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] + 2973696*z[1]*z[2]*z[4]*z[5]*z[6]*z[8] + 5799936*z[1]*z[2]*z[4]*z[5]*z[6]*z[9] - 11796864*z[1]*z[2]*z[4]*z[5]*z[6] - 2285568*z[1]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] + 5938176*z[1]*z[2]*z[4]*z[5]*z[7]*z[8] + 11581440*z[1]*z[2]*z[4]*z[5]*z[7]*z[9] - 23522304*z[1]*z[2]*z[4]*z[5]*z[7] + 23015424*z[1]*z[2]*z[4]*z[5]*z[8]*z[9] - 46473216*z[1]*z[2]*z[4]*z[5]*z[8] - 88375296*z[1]*z[2]*z[4]*z[5]*z[9] + 2965859328*z[1]*z[2]*z[4]*z[5] - 142848*z[1]*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] + 553536*z[1]*z[2]*z[4]*z[6]*z[7]*z[8] + 1107072*z[1]*z[2]*z[4]*z[6]*z[7]*z[9] - 2898624*z[1]*z[2]*z[4]*z[6]*z[7] + 2214144*z[1]*z[2]*z[4]*z[6]*z[8]*z[9] - 5761536*z[1]*z[2]*z[4]*z[6]*z[8] - 11237376*z[1]*z[2]*z[4]*z[6]*z[9] + 22856424*z[1]*z[2]*z[4]*z[6] + 4428288*z[1]*z[2]*z[4]*z[7]*z[8]*z[9] - 11505216*z[1]*z[2]*z[4]*z[7]*z[8] - 22439040*z[1]*z[2]*z[4]*z[7]*z[9] + 45574464*z[1]*z[2]*z[4]*z[7] - 44592384*z[1]*z[2]*z[4]*z[8]*z[9] + 90041856*z[1]*z[2]*z[4]*z[8] + 171227136*z[1]*z[2]*z[4]*z[9] - 3154174608*z[1]*z[2]*z[4] - 285696*z[1]*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] + 1107072*z[1]*z[2]*z[5]*z[6]*z[7]*z[8] + 2214144*z[1]*z[2]*z[5]*z[6]*z[7]*z[9] - 5797248*z[1]*z[2]*z[5]*z[6]*z[7] + 4428288*z[1]*z[2]*z[5]*z[6]*z[8]*z[9] - 11523072*z[1]*z[2]*z[5]*z[6]*z[8] - 22474752*z[1]*z[2]*z[5]*z[6]*z[9] + 45712848*z[1]*z[2]*z[5]*z[6] + 8856576*z[1]*z[2]*z[5]*z[7]*z[8]*z[9] - 23010432*z[1]*z[2]*z[5]*z[7]*z[8] - 44878080*z[1]*z[2]*z[5]*z[7]*z[9] + 91148928*z[1]*z[2]*z[5]*z[7] - 89184768*z[1]*z[2]*z[5]*z[8]*z[9] + 180083712*z[1]*z[2]*z[5]*z[8] + 342454272*z[1]*z[2]*z[5]*z[9] - 4298477856*z[1]*z[2]*z[5] + 374016*z[1]*z[2]*z[6]*z[7]*z[8]*z[9] - 1449312*z[1]*z[2]*z[6]*z[7]*z[8] - 2898624*z[1]*z[2]*z[6]*z[7]*z[9] + 7589408*z[1]*z[2]*z[6]*z[7] - 5797248*z[1]*z[2]*z[6]*z[8]*z[9] + 15085312*z[1]*z[2]*z[6]*z[8] + 29422592*z[1]*z[2]*z[6]*z[9] - 59844508*z[1]*z[2]*z[6] - 11594496*z[1]*z[2]*z[7]*z[8]*z[9] + 30123872*z[1]*z[2]*z[7]*z[8] + 58751680*z[1]*z[2]*z[7]*z[9] - 119326688*z[1]*z[2]*z[7] + 116755328*z[1]*z[2]*z[8]*z[9] - 235754752*z[1]*z[2]*z[8] - 448320512*z[1]*z[2]*z[9] + 4525933536*z[1]*z[2] + 147456*z[1]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 571392*z[1]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] - 1142784*z[1]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] + 2992128*z[1]*z[3]*z[4]*z[5]*z[6]*z[7] - 2285568*z[1]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] + 5947392*z[1]*z[3]*z[4]*z[5]*z[6]*z[8] + 11599872*z[1]*z[3]*z[4]*z[5]*z[6]*z[9] - 23593728*z[1]*z[3]*z[4]*z[5]*z[6] - 4571136*z[1]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] + 11876352*z[1]*z[3]*z[4]*z[5]*z[7]*z[8] + 23162880*z[1]*z[3]*z[4]*z[5]*z[7]*z[9] - 47044608*z[1]*z[3]*z[4]*z[5]*z[7] + 46030848*z[1]*z[3]*z[4]*z[5]*z[8]*z[9] - 92946432*z[1]*z[3]*z[4]*z[5]*z[8] - 176750592*z[1]*z[3]*z[4]*z[5]*z[9] + 5758826496*z[1]*z[3]*z[4]*z[5] - 285696*z[1]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] + 1107072*z[1]*z[3]*z[4]*z[6]*z[7]*z[8] + 2214144*z[1]*z[3]*z[4]*z[6]*z[7]*z[9] - 5797248*z[1]*z[3]*z[4]*z[6]*z[7] + 4428288*z[1]*z[3]*z[4]*z[6]*z[8]*z[9] - 11523072*z[1]*z[3]*z[4]*z[6]*z[8] - 22474752*z[1]*z[3]*z[4]*z[6]*z[9] + 45712848*z[1]*z[3]*z[4]*z[6] + 8856576*z[1]*z[3]*z[4]*z[7]*z[8]*z[9] - 23010432*z[1]*z[3]*z[4]*z[7]*z[8] - 44878080*z[1]*z[3]*z[4]*z[7]*z[9] + 91148928*z[1]*z[3]*z[4]*z[7] - 89184768*z[1]*z[3]*z[4]*z[8]*z[9] + 180083712*z[1]*z[3]*z[4]*z[8] + 342454272*z[1]*z[3]*z[4]*z[9] - 6120236256*z[1]*z[3]*z[4] - 571392*z[1]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] + 2214144*z[1]*z[3]*z[5]*z[6]*z[7]*z[8] + 4428288*z[1]*z[3]*z[5]*z[6]*z[7]*z[9] - 11594496*z[1]*z[3]*z[5]*z[6]*z[7] + 8856576*z[1]*z[3]*z[5]*z[6]*z[8]*z[9] - 23046144*z[1]*z[3]*z[5]*z[6]*z[8] - 44949504*z[1]*z[3]*z[5]*z[6]*z[9] + 91425696*z[1]*z[3]*z[5]*z[6] + 17713152*z[1]*z[3]*z[5]*z[7]*z[8]*z[9] - 46020864*z[1]*z[3]*z[5]*z[7]*z[8] - 89756160*z[1]*z[3]*z[5]*z[7]*z[9] + 182297856*z[1]*z[3]*z[5]*z[7] - 178369536*z[1]*z[3]*z[5]*z[8]*z[9] + 360167424*z[1]*z[3]*z[5]*z[8] + 684908544*z[1]*z[3]*z[5]*z[9] - 8300724672*z[1]*z[3]*z[5] + 743424*z[1]*z[3]*z[6]*z[7]*z[8]*z[9] - 2880768*z[1]*z[3]*z[6]*z[7]*z[8] - 5761536*z[1]*z[3]*z[6]*z[7]*z[9] + 15085312*z[1]*z[3]*z[6]*z[7] - 11523072*z[1]*z[3]*z[6]*z[8]*z[9] + 29984768*z[1]*z[3]*z[6]*z[8] + 58482688*z[1]*z[3]*z[6]*z[9] - 118951712*z[1]*z[3]*z[6] - 23046144*z[1]*z[3]*z[7]*z[8]*z[9] + 59876608*z[1]*z[3]*z[7]*z[8] + 116779520*z[1]*z[3]*z[7]*z[9] - 237183232*z[1]*z[3]*z[7] + 232072192*z[1]*z[3]*z[8]*z[9] - 468604928*z[1]*z[3]*z[8] - 891117568*z[1]*z[3]*z[9] + 8730743424*z[1]*z[3] - 1142784*z[1]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 4428288*z[1]*z[4]*z[5]*z[6]*z[7]*z[8] + 8856576*z[1]*z[4]*z[5]*z[6]*z[7]*z[9] - 23188992*z[1]*z[4]*z[5]*z[6]*z[7] + 17713152*z[1]*z[4]*z[5]*z[6]*z[8]*z[9] - 46092288*z[1]*z[4]*z[5]*z[6]*z[8] - 89899008*z[1]*z[4]*z[5]*z[6]*z[9] + 182851392*z[1]*z[4]*z[5]*z[6] + 35426304*z[1]*z[4]*z[5]*z[7]*z[8]*z[9] - 92041728*z[1]*z[4]*z[5]*z[7]*z[8] - 179512320*z[1]*z[4]*z[5]*z[7]*z[9] + 364595712*z[1]*z[4]*z[5]*z[7] - 356739072*z[1]*z[4]*z[5]*z[8]*z[9] + 720334848*z[1]*z[4]*z[5]*z[8] + 1369817088*z[1]*z[4]*z[5]*z[9] - 14471585664*z[1]*z[4]*z[5] + 1449984*z[1]*z[4]*z[6]*z[7]*z[8]*z[9] - 5618688*z[1]*z[4]*z[6]*z[7]*z[8] - 11237376*z[1]*z[4]*z[6]*z[7]*z[9] + 29422592*z[1]*z[4]*z[6]*z[7] - 22474752*z[1]*z[4]*z[6]*z[8]*z[9] + 58482688*z[1]*z[4]*z[6]*z[8] + 114065408*z[1]*z[4]*z[6]*z[9] - 232004992*z[1]*z[4]*z[6] - 44949504*z[1]*z[4]*z[7]*z[8]*z[9] + 116784128*z[1]*z[4]*z[7]*z[8] + 227768320*z[1]*z[4]*z[7]*z[9] - 462605312*z[1]*z[4]*z[7] + 452636672*z[1]*z[4]*z[8]*z[9] - 913973248*z[1]*z[4]*z[8] - 1738047488*z[1]*z[4]*z[9] + 15190865664*z[1]*z[4] + 2605056*z[1]*z[5]*z[6]*z[7]*z[8]*z[9] - 10094592*z[1]*z[5]*z[6]*z[7]*z[8] - 20189184*z[1]*z[5]*z[6]*z[7]*z[9] + 52860928*z[1]*z[5]*z[6]*z[7] - 40378368*z[1]*z[5]*z[6]*z[8]*z[9] + 105070592*z[1]*z[5]*z[6]*z[8] + 204931072*z[1]*z[5]*z[6]*z[9] - 416822528*z[1]*z[5]*z[6] - 80756736*z[1]*z[5]*z[7]*z[8]*z[9] + 209815552*z[1]*z[5]*z[7]*z[8] + 409210880*z[1]*z[5]*z[7]*z[9] - 831121408*z[1]*z[5]*z[7] + 813211648*z[1]*z[5]*z[8]*z[9] - 1642053632*z[1]*z[5]*z[8] - 3122593792*z[1]*z[5]*z[9] + 20185284096*z[1]*z[5] - 2949216*z[1]*z[6]*z[7]*z[8]*z[9] + 11428212*z[1]*z[6]*z[7]*z[8] + 22856424*z[1]*z[6]*z[7]*z[9] - 59844508*z[1]*z[6]*z[7] + 45712848*z[1]*z[6]*z[8]*z[9] - 118951712*z[1]*z[6]*z[8] - 232004992*z[1]*z[6]*z[9] + 943779841*z[1]*z[6]/2 + 91425696*z[1]*z[7]*z[8]*z[9] - 237534772*z[1]*z[7]*z[8] - 463272680*z[1]*z[7]*z[9] + 940922788*z[1]*z[7] - 920646928*z[1]*z[8]*z[9] + 1858989152*z[1]*z[8] + 3535126912*z[1]*z[9] - 41959110297*z[1]/2 - 384*z[10]*z[11]*z[12]*z[6]*z[7]*z[8] - 768*z[10]*z[11]*z[12]*z[6]*z[7]*z[9] + 2976*z[10]*z[11]*z[12]*z[6]*z[7] - 1536*z[10]*z[11]*z[12]*z[6]*z[8]*z[9] + 5952*z[10]*z[11]*z[12]*z[6]*z[8] + 11904*z[10]*z[11]*z[12]*z[6]*z[9] - 27136*z[10]*z[11]*z[12]*z[6] - 3072*z[10]*z[11]*z[12]*z[7]*z[8]*z[9] + 11904*z[10]*z[11]*z[12]*z[7]*z[8] + 23808*z[10]*z[11]*z[12]*z[7]*z[9] - 54176*z[10]*z[11]*z[12]*z[7] + 47616*z[10]*z[11]*z[12]*z[8]*z[9] - 107584*z[10]*z[11]*z[12]*z[8] - 209024*z[10]*z[11]*z[12]*z[9] + 365056*z[10]*z[11]*z[12] - 768*z[10]*z[11]*z[13]*z[6]*z[7]*z[8] - 1536*z[10]*z[11]*z[13]*z[6]*z[7]*z[9] + 5952*z[10]*z[11]*z[13]*z[6]*z[7] - 3072*z[10]*z[11]*z[13]*z[6]*z[8]*z[9] + 11904*z[10]*z[11]*z[13]*z[6]*z[8] + 23808*z[10]*z[11]*z[13]*z[6]*z[9] - 54272*z[10]*z[11]*z[13]*z[6] - 6144*z[10]*z[11]*z[13]*z[7]*z[8]*z[9] + 23808*z[10]*z[11]*z[13]*z[7]*z[8] + 47616*z[10]*z[11]*z[13]*z[7]*z[9] - 108352*z[10]*z[11]*z[13]*z[7] + 95232*z[10]*z[11]*z[13]*z[8]*z[9] - 215168*z[10]*z[11]*z[13]*z[8] - 418048*z[10]*z[11]*z[13]*z[9] + 730112*z[10]*z[11]*z[13] - 1536*z[10]*z[11]*z[14]*z[6]*z[7]*z[8] - 3072*z[10]*z[11]*z[14]*z[6]*z[7]*z[9] + 11904*z[10]*z[11]*z[14]*z[6]*z[7] - 6144*z[10]*z[11]*z[14]*z[6]*z[8]*z[9] + 23808*z[10]*z[11]*z[14]*z[6]*z[8] + 47616*z[10]*z[11]*z[14]*z[6]*z[9] - 108544*z[10]*z[11]*z[14]*z[6] - 12288*z[10]*z[11]*z[14]*z[7]*z[8]*z[9] + 47616*z[10]*z[11]*z[14]*z[7]*z[8] + 95232*z[10]*z[11]*z[14]*z[7]*z[9] - 216704*z[10]*z[11]*z[14]*z[7] + 190464*z[10]*z[11]*z[14]*z[8]*z[9] - 430336*z[10]*z[11]*z[14]*z[8] - 836096*z[10]*z[11]*z[14]*z[9] + 1460224*z[10]*z[11]*z[14] - 3072*z[10]*z[11]*z[15]*z[6]*z[7]*z[8] - 6144*z[10]*z[11]*z[15]*z[6]*z[7]*z[9] + 23808*z[10]*z[11]*z[15]*z[6]*z[7] - 12288*z[10]*z[11]*z[15]*z[6]*z[8]*z[9] + 47616*z[10]*z[11]*z[15]*z[6]*z[8] + 95232*z[10]*z[11]*z[15]*z[6]*z[9] - 217088*z[10]*z[11]*z[15]*z[6] - 24576*z[10]*z[11]*z[15]*z[7]*z[8]*z[9] + 95232*z[10]*z[11]*z[15]*z[7]*z[8] + 190464*z[10]*z[11]*z[15]*z[7]*z[9] - 433408*z[10]*z[11]*z[15]*z[7] + 380928*z[10]*z[11]*z[15]*z[8]*z[9] - 860672*z[10]*z[11]*z[15]*z[8] - 1672192*z[10]*z[11]*z[15]*z[9] + 2920448*z[10]*z[11]*z[15] + 5952*z[10]*z[11]*z[6]*z[7]*z[8] + 11904*z[10]*z[11]*z[6]*z[7]*z[9] - 46128*z[10]*z[11]*z[6]*z[7] + 23808*z[10]*z[11]*z[6]*z[8]*z[9] - 92256*z[10]*z[11]*z[6]*z[8] - 184512*z[10]*z[11]*z[6]*z[9] + 420608*z[10]*z[11]*z[6] + 47616*z[10]*z[11]*z[7]*z[8]*z[9] - 184512*z[10]*z[11]*z[7]*z[8] - 369024*z[10]*z[11]*z[7]*z[9] + 839728*z[10]*z[11]*z[7] - 738048*z[10]*z[11]*z[8]*z[9] + 1667552*z[10]*z[11]*z[8] + 3239872*z[10]*z[11]*z[9] - 5658368*z[10]*z[11] - 1536*z[10]*z[12]*z[13]*z[6]*z[7]*z[8] - 3072*z[10]*z[12]*z[13]*z[6]*z[7]*z[9] + 11904*z[10]*z[12]*z[13]*z[6]*z[7] - 6144*z[10]*z[12]*z[13]*z[6]*z[8]*z[9] + 23808*z[10]*z[12]*z[13]*z[6]*z[8] + 47616*z[10]*z[12]*z[13]*z[6]*z[9] - 108544*z[10]*z[12]*z[13]*z[6] - 12288*z[10]*z[12]*z[13]*z[7]*z[8]*z[9] + 47616*z[10]*z[12]*z[13]*z[7]*z[8] + 95232*z[10]*z[12]*z[13]*z[7]*z[9] - 216704*z[10]*z[12]*z[13]*z[7] + 190464*z[10]*z[12]*z[13]*z[8]*z[9] - 430336*z[10]*z[12]*z[13]*z[8] - 836096*z[10]*z[12]*z[13]*z[9] + 1460224*z[10]*z[12]*z[13] - 3072*z[10]*z[12]*z[14]*z[6]*z[7]*z[8] - 6144*z[10]*z[12]*z[14]*z[6]*z[7]*z[9] + 23808*z[10]*z[12]*z[14]*z[6]*z[7] - 12288*z[10]*z[12]*z[14]*z[6]*z[8]*z[9] + 47616*z[10]*z[12]*z[14]*z[6]*z[8] + 95232*z[10]*z[12]*z[14]*z[6]*z[9] - 217088*z[10]*z[12]*z[14]*z[6] - 24576*z[10]*z[12]*z[14]*z[7]*z[8]*z[9] + 95232*z[10]*z[12]*z[14]*z[7]*z[8] + 190464*z[10]*z[12]*z[14]*z[7]*z[9] - 433408*z[10]*z[12]*z[14]*z[7] + 380928*z[10]*z[12]*z[14]*z[8]*z[9] - 860672*z[10]*z[12]*z[14]*z[8] - 1672192*z[10]*z[12]*z[14]*z[9] + 2920448*z[10]*z[12]*z[14] - 6144*z[10]*z[12]*z[15]*z[6]*z[7]*z[8] - 12288*z[10]*z[12]*z[15]*z[6]*z[7]*z[9] + 47616*z[10]*z[12]*z[15]*z[6]*z[7] - 24576*z[10]*z[12]*z[15]*z[6]*z[8]*z[9] + 95232*z[10]*z[12]*z[15]*z[6]*z[8] + 190464*z[10]*z[12]*z[15]*z[6]*z[9] - 434176*z[10]*z[12]*z[15]*z[6] - 49152*z[10]*z[12]*z[15]*z[7]*z[8]*z[9] + 190464*z[10]*z[12]*z[15]*z[7]*z[8] + 380928*z[10]*z[12]*z[15]*z[7]*z[9] - 866816*z[10]*z[12]*z[15]*z[7] + 761856*z[10]*z[12]*z[15]*z[8]*z[9] - 1721344*z[10]*z[12]*z[15]*z[8] - 3344384*z[10]*z[12]*z[15]*z[9] + 5840896*z[10]*z[12]*z[15] + 11904*z[10]*z[12]*z[6]*z[7]*z[8] + 23808*z[10]*z[12]*z[6]*z[7]*z[9] - 92256*z[10]*z[12]*z[6]*z[7] + 47616*z[10]*z[12]*z[6]*z[8]*z[9] - 184512*z[10]*z[12]*z[6]*z[8] - 369024*z[10]*z[12]*z[6]*z[9] + 841216*z[10]*z[12]*z[6] + 95232*z[10]*z[12]*z[7]*z[8]*z[9] - 369024*z[10]*z[12]*z[7]*z[8] - 738048*z[10]*z[12]*z[7]*z[9] + 1679456*z[10]*z[12]*z[7] - 1476096*z[10]*z[12]*z[8]*z[9] + 3335104*z[10]*z[12]*z[8] + 6479744*z[10]*z[12]*z[9] - 11316736*z[10]*z[12] - 6144*z[10]*z[13]*z[14]*z[6]*z[7]*z[8] - 12288*z[10]*z[13]*z[14]*z[6]*z[7]*z[9] + 47616*z[10]*z[13]*z[14]*z[6]*z[7] - 24576*z[10]*z[13]*z[14]*z[6]*z[8]*z[9] + 95232*z[10]*z[13]*z[14]*z[6]*z[8] + 190464*z[10]*z[13]*z[14]*z[6]*z[9] - 434176*z[10]*z[13]*z[14]*z[6] - 49152*z[10]*z[13]*z[14]*z[7]*z[8]*z[9] + 190464*z[10]*z[13]*z[14]*z[7]*z[8] + 380928*z[10]*z[13]*z[14]*z[7]*z[9] - 866816*z[10]*z[13]*z[14]*z[7] + 761856*z[10]*z[13]*z[14]*z[8]*z[9] - 1721344*z[10]*z[13]*z[14]*z[8] - 3344384*z[10]*z[13]*z[14]*z[9] + 5840896*z[10]*z[13]*z[14] - 12288*z[10]*z[13]*z[15]*z[6]*z[7]*z[8] - 24576*z[10]*z[13]*z[15]*z[6]*z[7]*z[9] + 95232*z[10]*z[13]*z[15]*z[6]*z[7] - 49152*z[10]*z[13]*z[15]*z[6]*z[8]*z[9] + 190464*z[10]*z[13]*z[15]*z[6]*z[8] + 380928*z[10]*z[13]*z[15]*z[6]*z[9] - 868352*z[10]*z[13]*z[15]*z[6] - 98304*z[10]*z[13]*z[15]*z[7]*z[8]*z[9] + 380928*z[10]*z[13]*z[15]*z[7]*z[8] + 761856*z[10]*z[13]*z[15]*z[7]*z[9] - 1733632*z[10]*z[13]*z[15]*z[7] + 1523712*z[10]*z[13]*z[15]*z[8]*z[9] - 3442688*z[10]*z[13]*z[15]*z[8] - 6688768*z[10]*z[13]*z[15]*z[9] + 11681792*z[10]*z[13]*z[15] + 23808*z[10]*z[13]*z[6]*z[7]*z[8] + 47616*z[10]*z[13]*z[6]*z[7]*z[9] - 184512*z[10]*z[13]*z[6]*z[7] + 95232*z[10]*z[13]*z[6]*z[8]*z[9] - 369024*z[10]*z[13]*z[6]*z[8] - 738048*z[10]*z[13]*z[6]*z[9] + 1682432*z[10]*z[13]*z[6] + 190464*z[10]*z[13]*z[7]*z[8]*z[9] - 738048*z[10]*z[13]*z[7]*z[8] - 1476096*z[10]*z[13]*z[7]*z[9] + 3358912*z[10]*z[13]*z[7] - 2952192*z[10]*z[13]*z[8]*z[9] + 6670208*z[10]*z[13]*z[8] + 12959488*z[10]*z[13]*z[9] - 22633472*z[10]*z[13] - 24576*z[10]*z[14]*z[15]*z[6]*z[7]*z[8] - 49152*z[10]*z[14]*z[15]*z[6]*z[7]*z[9] + 190464*z[10]*z[14]*z[15]*z[6]*z[7] - 98304*z[10]*z[14]*z[15]*z[6]*z[8]*z[9] + 380928*z[10]*z[14]*z[15]*z[6]*z[8] + 761856*z[10]*z[14]*z[15]*z[6]*z[9] - 1736704*z[10]*z[14]*z[15]*z[6] - 196608*z[10]*z[14]*z[15]*z[7]*z[8]*z[9] + 761856*z[10]*z[14]*z[15]*z[7]*z[8] + 1523712*z[10]*z[14]*z[15]*z[7]*z[9] - 3467264*z[10]*z[14]*z[15]*z[7] + 3047424*z[10]*z[14]*z[15]*z[8]*z[9] - 6885376*z[10]*z[14]*z[15]*z[8] - 13377536*z[10]*z[14]*z[15]*z[9] + 23363584*z[10]*z[14]*z[15] + 47616*z[10]*z[14]*z[6]*z[7]*z[8] + 95232*z[10]*z[14]*z[6]*z[7]*z[9] - 369024*z[10]*z[14]*z[6]*z[7] + 190464*z[10]*z[14]*z[6]*z[8]*z[9] - 738048*z[10]*z[14]*z[6]*z[8] - 1476096*z[10]*z[14]*z[6]*z[9] + 3364864*z[10]*z[14]*z[6] + 380928*z[10]*z[14]*z[7]*z[8]*z[9] - 1476096*z[10]*z[14]*z[7]*z[8] - 2952192*z[10]*z[14]*z[7]*z[9] + 6717824*z[10]*z[14]*z[7] - 5904384*z[10]*z[14]*z[8]*z[9] + 13340416*z[10]*z[14]*z[8] + 25918976*z[10]*z[14]*z[9] - 45266944*z[10]*z[14] + 95232*z[10]*z[15]*z[6]*z[7]*z[8] + 190464*z[10]*z[15]*z[6]*z[7]*z[9] - 738048*z[10]*z[15]*z[6]*z[7] + 380928*z[10]*z[15]*z[6]*z[8]*z[9] - 1476096*z[10]*z[15]*z[6]*z[8] - 2952192*z[10]*z[15]*z[6]*z[9] + 6729728*z[10]*z[15]*z[6] + 761856*z[10]*z[15]*z[7]*z[8]*z[9] - 2952192*z[10]*z[15]*z[7]*z[8] - 5904384*z[10]*z[15]*z[7]*z[9] + 13435648*z[10]*z[15]*z[7] - 11808768*z[10]*z[15]*z[8]*z[9] + 26680832*z[10]*z[15]*z[8] + 51837952*z[10]*z[15]*z[9] - 90533888*z[10]*z[15] + 589824*z[10]*z[2]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 1179648*z[10]*z[2]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] - 4571136*z[10]*z[2]*z[3]*z[4]*z[5]*z[6]*z[7] + 2359296*z[10]*z[2]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] - 9142272*z[10]*z[2]*z[3]*z[4]*z[5]*z[6]*z[8] - 18284544*z[10]*z[2]*z[3]*z[4]*z[5]*z[6]*z[9] + 41680896*z[10]*z[2]*z[3]*z[4]*z[5]*z[6] + 4718592*z[10]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 18284544*z[10]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] - 36569088*z[10]*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] + 83214336*z[10]*z[2]*z[3]*z[4]*z[5]*z[7] - 73138176*z[10]*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] + 165249024*z[10]*z[2]*z[3]*z[4]*z[5]*z[8] + 321060864*z[10]*z[2]*z[3]*z[4]*z[5]*z[9] - 560726016*z[10]*z[2]*z[3]*z[4]*z[5] - 1142784*z[10]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] - 2285568*z[10]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] + 8856576*z[10]*z[2]*z[3]*z[4]*z[6]*z[7] - 4571136*z[10]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] + 17713152*z[10]*z[2]*z[3]*z[4]*z[6]*z[8] + 35426304*z[10]*z[2]*z[3]*z[4]*z[6]*z[9] - 80756736*z[10]*z[2]*z[3]*z[4]*z[6] - 9142272*z[10]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] + 35426304*z[10]*z[2]*z[3]*z[4]*z[7]*z[8] + 70852608*z[10]*z[2]*z[3]*z[4]*z[7]*z[9] - 161227776*z[10]*z[2]*z[3]*z[4]*z[7] + 141705216*z[10]*z[2]*z[3]*z[4]*z[8]*z[9] - 320169984*z[10]*z[2]*z[3]*z[4]*z[8] - 622055424*z[10]*z[2]*z[3]*z[4]*z[9] + 1086406656*z[10]*z[2]*z[3]*z[4] - 2285568*z[10]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] - 4571136*z[10]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] + 17713152*z[10]*z[2]*z[3]*z[5]*z[6]*z[7] - 9142272*z[10]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] + 35426304*z[10]*z[2]*z[3]*z[5]*z[6]*z[8] + 70852608*z[10]*z[2]*z[3]*z[5]*z[6]*z[9] - 161513472*z[10]*z[2]*z[3]*z[5]*z[6] - 18284544*z[10]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] + 70852608*z[10]*z[2]*z[3]*z[5]*z[7]*z[8] + 141705216*z[10]*z[2]*z[3]*z[5]*z[7]*z[9] - 322455552*z[10]*z[2]*z[3]*z[5]*z[7] + 283410432*z[10]*z[2]*z[3]*z[5]*z[8]*z[9] - 640339968*z[10]*z[2]*z[3]*z[5]*z[8] - 1244110848*z[10]*z[2]*z[3]*z[5]*z[9] + 2172813312*z[10]*z[2]*z[3]*z[5] + 2969088*z[10]*z[2]*z[3]*z[6]*z[7]*z[8] + 5938176*z[10]*z[2]*z[3]*z[6]*z[7]*z[9] - 23010432*z[10]*z[2]*z[3]*z[6]*z[7] + 11876352*z[10]*z[2]*z[3]*z[6]*z[8]*z[9] - 46020864*z[10]*z[2]*z[3]*z[6]*z[8] - 92041728*z[10]*z[2]*z[3]*z[6]*z[9] + 209815552*z[10]*z[2]*z[3]*z[6] + 23752704*z[10]*z[2]*z[3]*z[7]*z[8]*z[9] - 92041728*z[10]*z[2]*z[3]*z[7]*z[8] - 184083456*z[10]*z[2]*z[3]*z[7]*z[9] + 418888832*z[10]*z[2]*z[3]*z[7] - 368166912*z[10]*z[2]*z[3]*z[8]*z[9] + 831839488*z[10]*z[2]*z[3]*z[8] + 1616173568*z[10]*z[2]*z[3]*z[9] - 2822612992*z[10]*z[2]*z[3] - 4571136*z[10]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] - 9142272*z[10]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] + 35426304*z[10]*z[2]*z[4]*z[5]*z[6]*z[7] - 18284544*z[10]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] + 70852608*z[10]*z[2]*z[4]*z[5]*z[6]*z[8] + 141705216*z[10]*z[2]*z[4]*z[5]*z[6]*z[9] - 323026944*z[10]*z[2]*z[4]*z[5]*z[6] - 36569088*z[10]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] + 141705216*z[10]*z[2]*z[4]*z[5]*z[7]*z[8] + 283410432*z[10]*z[2]*z[4]*z[5]*z[7]*z[9] - 644911104*z[10]*z[2]*z[4]*z[5]*z[7] + 566820864*z[10]*z[2]*z[4]*z[5]*z[8]*z[9] - 1280679936*z[10]*z[2]*z[4]*z[5]*z[8] - 2488221696*z[10]*z[2]*z[4]*z[5]*z[9] + 4345626624*z[10]*z[2]*z[4]*z[5] + 5790720*z[10]*z[2]*z[4]*z[6]*z[7]*z[8] + 11581440*z[10]*z[2]*z[4]*z[6]*z[7]*z[9] - 44878080*z[10]*z[2]*z[4]*z[6]*z[7] + 23162880*z[10]*z[2]*z[4]*z[6]*z[8]*z[9] - 89756160*z[10]*z[2]*z[4]*z[6]*z[8] - 179512320*z[10]*z[2]*z[4]*z[6]*z[9] + 409210880*z[10]*z[2]*z[4]*z[6] + 46325760*z[10]*z[2]*z[4]*z[7]*z[8]*z[9] - 179512320*z[10]*z[2]*z[4]*z[7]*z[8] - 359024640*z[10]*z[2]*z[4]*z[7]*z[9] + 816974080*z[10]*z[2]*z[4]*z[7] - 718049280*z[10]*z[2]*z[4]*z[8]*z[9] + 1622366720*z[10]*z[2]*z[4]*z[8] + 3152081920*z[10]*z[2]*z[4]*z[9] - 5505044480*z[10]*z[2]*z[4] + 10401792*z[10]*z[2]*z[5]*z[6]*z[7]*z[8] + 20803584*z[10]*z[2]*z[5]*z[6]*z[7]*z[9] - 80613888*z[10]*z[2]*z[5]*z[6]*z[7] + 41607168*z[10]*z[2]*z[5]*z[6]*z[8]*z[9] - 161227776*z[10]*z[2]*z[5]*z[6]*z[8] - 322455552*z[10]*z[2]*z[5]*z[6]*z[9] + 735059968*z[10]*z[2]*z[5]*z[6] + 83214336*z[10]*z[2]*z[5]*z[7]*z[8]*z[9] - 322455552*z[10]*z[2]*z[5]*z[7]*z[8] - 644911104*z[10]*z[2]*z[5]*z[7]*z[9] + 1467519488*z[10]*z[2]*z[5]*z[7] - 1289822208*z[10]*z[2]*z[5]*z[8]*z[9] + 2914235392*z[10]*z[2]*z[5]*z[8] + 5662042112*z[10]*z[2]*z[5]*z[9] - 9888636928*z[10]*z[2]*z[5] - 11761152*z[10]*z[2]*z[6]*z[7]*z[8] - 23522304*z[10]*z[2]*z[6]*z[7]*z[9] + 91148928*z[10]*z[2]*z[6]*z[7] - 47044608*z[10]*z[2]*z[6]*z[8]*z[9] + 182297856*z[10]*z[2]*z[6]*z[8] + 364595712*z[10]*z[2]*z[6]*z[9] - 831121408*z[10]*z[2]*z[6] - 94089216*z[10]*z[2]*z[7]*z[8]*z[9] + 364595712*z[10]*z[2]*z[7]*z[8] + 729191424*z[10]*z[2]*z[7]*z[9] - 1659302528*z[10]*z[2]*z[7] + 1458382848*z[10]*z[2]*z[8]*z[9] - 3295082752*z[10]*z[2]*z[8] - 6401987072*z[10]*z[2]*z[9] + 11180935168*z[10]*z[2] - 9142272*z[10]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] - 18284544*z[10]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] + 70852608*z[10]*z[3]*z[4]*z[5]*z[6]*z[7] - 36569088*z[10]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] + 141705216*z[10]*z[3]*z[4]*z[5]*z[6]*z[8] + 283410432*z[10]*z[3]*z[4]*z[5]*z[6]*z[9] - 646053888*z[10]*z[3]*z[4]*z[5]*z[6] - 73138176*z[10]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] + 283410432*z[10]*z[3]*z[4]*z[5]*z[7]*z[8] + 566820864*z[10]*z[3]*z[4]*z[5]*z[7]*z[9] - 1289822208*z[10]*z[3]*z[4]*z[5]*z[7] + 1133641728*z[10]*z[3]*z[4]*z[5]*z[8]*z[9] - 2561359872*z[10]*z[3]*z[4]*z[5]*z[8] - 4976443392*z[10]*z[3]*z[4]*z[5]*z[9] + 8691253248*z[10]*z[3]*z[4]*z[5] + 11507712*z[10]*z[3]*z[4]*z[6]*z[7]*z[8] + 23015424*z[10]*z[3]*z[4]*z[6]*z[7]*z[9] - 89184768*z[10]*z[3]*z[4]*z[6]*z[7] + 46030848*z[10]*z[3]*z[4]*z[6]*z[8]*z[9] - 178369536*z[10]*z[3]*z[4]*z[6]*z[8] - 356739072*z[10]*z[3]*z[4]*z[6]*z[9] + 813211648*z[10]*z[3]*z[4]*z[6] + 92061696*z[10]*z[3]*z[4]*z[7]*z[8]*z[9] - 356739072*z[10]*z[3]*z[4]*z[7]*z[8] - 713478144*z[10]*z[3]*z[4]*z[7]*z[9] + 1623546368*z[10]*z[3]*z[4]*z[7] - 1426956288*z[10]*z[3]*z[4]*z[8]*z[9] + 3224077312*z[10]*z[3]*z[4]*z[8] + 6264031232*z[10]*z[3]*z[4]*z[9] - 10939998208*z[10]*z[3]*z[4] + 20656128*z[10]*z[3]*z[5]*z[6]*z[7]*z[8] + 41312256*z[10]*z[3]*z[5]*z[6]*z[7]*z[9] - 160084992*z[10]*z[3]*z[5]*z[6]*z[7] + 82624512*z[10]*z[3]*z[5]*z[6]*z[8]*z[9] - 320169984*z[10]*z[3]*z[5]*z[6]*z[8] - 640339968*z[10]*z[3]*z[5]*z[6]*z[9] + 1459699712*z[10]*z[3]*z[5]*z[6] + 165249024*z[10]*z[3]*z[5]*z[7]*z[8]*z[9] - 640339968*z[10]*z[3]*z[5]*z[7]*z[8] - 1280679936*z[10]*z[3]*z[5]*z[7]*z[9] + 2914235392*z[10]*z[3]*z[5]*z[7] - 2561359872*z[10]*z[3]*z[5]*z[8]*z[9] + 5787158528*z[10]*z[3]*z[5]*z[8] + 11243819008*z[10]*z[3]*z[5]*z[9] - 19637092352*z[10]*z[3]*z[5] - 23236608*z[10]*z[3]*z[6]*z[7]*z[8] - 46473216*z[10]*z[3]*z[6]*z[7]*z[9] + 180083712*z[10]*z[3]*z[6]*z[7] - 92946432*z[10]*z[3]*z[6]*z[8]*z[9] + 360167424*z[10]*z[3]*z[6]*z[8] + 720334848*z[10]*z[3]*z[6]*z[9] - 1642053632*z[10]*z[3]*z[6] - 185892864*z[10]*z[3]*z[7]*z[8]*z[9] + 720334848*z[10]*z[3]*z[7]*z[8] + 1440669696*z[10]*z[3]*z[7]*z[9] - 3278298112*z[10]*z[3]*z[7] + 2881339392*z[10]*z[3]*z[8]*z[9] - 6510123008*z[10]*z[3]*z[8] - 12648460288*z[10]*z[3]*z[9] + 22090268672*z[10]*z[3] + 40132608*z[10]*z[4]*z[5]*z[6]*z[7]*z[8] + 80265216*z[10]*z[4]*z[5]*z[6]*z[7]*z[9] - 311027712*z[10]*z[4]*z[5]*z[6]*z[7] + 160530432*z[10]*z[4]*z[5]*z[6]*z[8]*z[9] - 622055424*z[10]*z[4]*z[5]*z[6]*z[8] - 1244110848*z[10]*z[4]*z[5]*z[6]*z[9] + 2836037632*z[10]*z[4]*z[5]*z[6] + 321060864*z[10]*z[4]*z[5]*z[7]*z[8]*z[9] - 1244110848*z[10]*z[4]*z[5]*z[7]*z[8] - 2488221696*z[10]*z[4]*z[5]*z[7]*z[9] + 5662042112*z[10]*z[4]*z[5]*z[7] - 4976443392*z[10]*z[4]*z[5]*z[8]*z[9] + 11243819008*z[10]*z[4]*z[5]*z[8] + 21845516288*z[10]*z[4]*z[5]*z[9] - 38152732672*z[10]*z[4]*z[5] - 44187648*z[10]*z[4]*z[6]*z[7]*z[8] - 88375296*z[10]*z[4]*z[6]*z[7]*z[9] + 342454272*z[10]*z[4]*z[6]*z[7] - 176750592*z[10]*z[4]*z[6]*z[8]*z[9] + 684908544*z[10]*z[4]*z[6]*z[8] + 1369817088*z[10]*z[4]*z[6]*z[9] - 3122593792*z[10]*z[4]*z[6] - 353501184*z[10]*z[4]*z[7]*z[8]*z[9] + 1369817088*z[10]*z[4]*z[7]*z[8] + 2739634176*z[10]*z[4]*z[7]*z[9] - 6234140672*z[10]*z[4]*z[7] + 5479268352*z[10]*z[4]*z[8]*z[9] - 12379906048*z[10]*z[4]*z[8] - 24052809728*z[10]*z[4]*z[9] + 42007724032*z[10]*z[4] - 70090752*z[10]*z[5]*z[6]*z[7]*z[8] - 140181504*z[10]*z[5]*z[6]*z[7]*z[9] + 543203328*z[10]*z[5]*z[6]*z[7] - 280363008*z[10]*z[5]*z[6]*z[8]*z[9] + 1086406656*z[10]*z[5]*z[6]*z[8] + 2172813312*z[10]*z[5]*z[6]*z[9] - 4953079808*z[10]*z[5]*z[6] - 560726016*z[10]*z[5]*z[7]*z[8]*z[9] + 2172813312*z[10]*z[5]*z[7]*z[8] + 4345626624*z[10]*z[5]*z[7]*z[9] - 9888636928*z[10]*z[5]*z[7] + 8691253248*z[10]*z[5]*z[8]*z[9] - 19637092352*z[10]*z[5]*z[8] - 38152732672*z[10]*z[5]*z[9] + 66632941568*z[10]*z[5] - 1084930560*z[10]*z[6]*z[7]*z[8]*z[9] + 1654531584*z[10]*z[6]*z[7]*z[8] + 2965859328*z[10]*z[6]*z[7]*z[9] - 4298477856*z[10]*z[6]*z[7] + 5758826496*z[10]*z[6]*z[8]*z[9] - 8300724672*z[10]*z[6]*z[8] - 14471585664*z[10]*z[6]*z[9] + 20185284096*z[10]*z[6] + 11431045632*z[10]*z[7]*z[8]*z[9] - 16452708864*z[10]*z[7]*z[8] - 28665689088*z[10]*z[7]*z[9] + 39942893856*z[10]*z[7] - 55171516416*z[10]*z[8]*z[9] + 76599182784*z[10]*z[8] + 131021391744*z[10]*z[9] - 175932120576*z[10] + 96*z[11]*z[12]*z[13]*z[14] + 192*z[11]*z[12]*z[13]*z[15] - 372*z[11]*z[12]*z[13] + 384*z[11]*z[12]*z[14]*z[15] - 744*z[11]*z[12]*z[14] - 1488*z[11]*z[12]*z[15] - 3072*z[11]*z[12]*z[2]*z[3]*z[4]*z[5] + 5952*z[11]*z[12]*z[2]*z[3]*z[4] + 11904*z[11]*z[12]*z[2]*z[3]*z[5] - 15464*z[11]*z[12]*z[2]*z[3] + 23808*z[11]*z[12]*z[2]*z[4]*z[5] - 30160*z[11]*z[12]*z[2]*z[4] - 54176*z[11]*z[12]*z[2]*z[5] + 61256*z[11]*z[12]*z[2] + 47616*z[11]*z[12]*z[3]*z[4]*z[5] - 59936*z[11]*z[12]*z[3]*z[4] - 107584*z[11]*z[12]*z[3]*z[5] + 121024*z[11]*z[12]*z[3] - 209024*z[11]*z[12]*z[4]*z[5] + 230144*z[11]*z[12]*z[4] + 365056*z[11]*z[12]*z[5] - 192*z[11]*z[12]*z[6]*z[7]*z[8]*z[9] + 744*z[11]*z[12]*z[6]*z[7]*z[8] + 1488*z[11]*z[12]*z[6]*z[7]*z[9] - 3896*z[11]*z[12]*z[6]*z[7] + 2976*z[11]*z[12]*z[6]*z[8]*z[9] - 7744*z[11]*z[12]*z[6]*z[8] - 15104*z[11]*z[12]*z[6]*z[9] + 30721*z[11]*z[12]*z[6] + 5952*z[11]*z[12]*z[7]*z[8]*z[9] - 15464*z[11]*z[12]*z[7]*z[8] - 30160*z[11]*z[12]*z[7]*z[9] + 61256*z[11]*z[12]*z[7] - 59936*z[11]*z[12]*z[8]*z[9] + 121024*z[11]*z[12]*z[8] + 230144*z[11]*z[12]*z[9] - 772744*z[11]*z[12] + 768*z[11]*z[13]*z[14]*z[15] - 1488*z[11]*z[13]*z[14] - 2976*z[11]*z[13]*z[15] - 6144*z[11]*z[13]*z[2]*z[3]*z[4]*z[5] + 11904*z[11]*z[13]*z[2]*z[3]*z[4] + 23808*z[11]*z[13]*z[2]*z[3]*z[5] - 30928*z[11]*z[13]*z[2]*z[3] + 47616*z[11]*z[13]*z[2]*z[4]*z[5] - 60320*z[11]*z[13]*z[2]*z[4] - 108352*z[11]*z[13]*z[2]*z[5] + 122512*z[11]*z[13]*z[2] + 95232*z[11]*z[13]*z[3]*z[4]*z[5] - 119872*z[11]*z[13]*z[3]*z[4] - 215168*z[11]*z[13]*z[3]*z[5] + 242048*z[11]*z[13]*z[3] - 418048*z[11]*z[13]*z[4]*z[5] + 460288*z[11]*z[13]*z[4] + 730112*z[11]*z[13]*z[5] - 384*z[11]*z[13]*z[6]*z[7]*z[8]*z[9] + 1488*z[11]*z[13]*z[6]*z[7]*z[8] + 2976*z[11]*z[13]*z[6]*z[7]*z[9] - 7792*z[11]*z[13]*z[6]*z[7] + 5952*z[11]*z[13]*z[6]*z[8]*z[9] - 15488*z[11]*z[13]*z[6]*z[8] - 30208*z[11]*z[13]*z[6]*z[9] + 61442*z[11]*z[13]*z[6] + 11904*z[11]*z[13]*z[7]*z[8]*z[9] - 30928*z[11]*z[13]*z[7]*z[8] - 60320*z[11]*z[13]*z[7]*z[9] + 122512*z[11]*z[13]*z[7] - 119872*z[11]*z[13]*z[8]*z[9] + 242048*z[11]*z[13]*z[8] + 460288*z[11]*z[13]*z[9] - 1545512*z[11]*z[13] - 5952*z[11]*z[14]*z[15] - 12288*z[11]*z[14]*z[2]*z[3]*z[4]*z[5] + 23808*z[11]*z[14]*z[2]*z[3]*z[4] + 47616*z[11]*z[14]*z[2]*z[3]*z[5] - 61856*z[11]*z[14]*z[2]*z[3] + 95232*z[11]*z[14]*z[2]*z[4]*z[5] - 120640*z[11]*z[14]*z[2]*z[4] - 216704*z[11]*z[14]*z[2]*z[5] + 245024*z[11]*z[14]*z[2] + 190464*z[11]*z[14]*z[3]*z[4]*z[5] - 239744*z[11]*z[14]*z[3]*z[4] - 430336*z[11]*z[14]*z[3]*z[5] + 484096*z[11]*z[14]*z[3] - 836096*z[11]*z[14]*z[4]*z[5] + 920576*z[11]*z[14]*z[4] + 1460224*z[11]*z[14]*z[5] - 768*z[11]*z[14]*z[6]*z[7]*z[8]*z[9] + 2976*z[11]*z[14]*z[6]*z[7]*z[8] + 5952*z[11]*z[14]*z[6]*z[7]*z[9] - 15584*z[11]*z[14]*z[6]*z[7] + 11904*z[11]*z[14]*z[6]*z[8]*z[9] - 30976*z[11]*z[14]*z[6]*z[8] - 60416*z[11]*z[14]*z[6]*z[9] + 122884*z[11]*z[14]*z[6] + 23808*z[11]*z[14]*z[7]*z[8]*z[9] - 61856*z[11]*z[14]*z[7]*z[8] - 120640*z[11]*z[14]*z[7]*z[9] + 245024*z[11]*z[14]*z[7] - 239744*z[11]*z[14]*z[8]*z[9] + 484096*z[11]*z[14]*z[8] + 920576*z[11]*z[14]*z[9] - 3091216*z[11]*z[14] - 24576*z[11]*z[15]*z[2]*z[3]*z[4]*z[5] + 47616*z[11]*z[15]*z[2]*z[3]*z[4] + 95232*z[11]*z[15]*z[2]*z[3]*z[5] - 123712*z[11]*z[15]*z[2]*z[3] + 190464*z[11]*z[15]*z[2]*z[4]*z[5] - 241280*z[11]*z[15]*z[2]*z[4] - 433408*z[11]*z[15]*z[2]*z[5] + 490048*z[11]*z[15]*z[2] + 380928*z[11]*z[15]*z[3]*z[4]*z[5] - 479488*z[11]*z[15]*z[3]*z[4] - 860672*z[11]*z[15]*z[3]*z[5] + 968192*z[11]*z[15]*z[3] - 1672192*z[11]*z[15]*z[4]*z[5] + 1841152*z[11]*z[15]*z[4] + 2920448*z[11]*z[15]*z[5] - 1536*z[11]*z[15]*z[6]*z[7]*z[8]*z[9] + 5952*z[11]*z[15]*z[6]*z[7]*z[8] + 11904*z[11]*z[15]*z[6]*z[7]*z[9] - 31168*z[11]*z[15]*z[6]*z[7] + 23808*z[11]*z[15]*z[6]*z[8]*z[9] - 61952*z[11]*z[15]*z[6]*z[8] - 120832*z[11]*z[15]*z[6]*z[9] + 245768*z[11]*z[15]*z[6] + 47616*z[11]*z[15]*z[7]*z[8]*z[9] - 123712*z[11]*z[15]*z[7]*z[8] - 241280*z[11]*z[15]*z[7]*z[9] + 490048*z[11]*z[15]*z[7] - 479488*z[11]*z[15]*z[8]*z[9] + 968192*z[11]*z[15]*z[8] + 1841152*z[11]*z[15]*z[9] - 6183968*z[11]*z[15] + 47616*z[11]*z[2]*z[3]*z[4]*z[5] - 92256*z[11]*z[2]*z[3]*z[4] - 184512*z[11]*z[2]*z[3]*z[5] + 239692*z[11]*z[2]*z[3] - 369024*z[11]*z[2]*z[4]*z[5] + 467480*z[11]*z[2]*z[4] + 839728*z[11]*z[2]*z[5] - 949468*z[11]*z[2] - 738048*z[11]*z[3]*z[4]*z[5] + 929008*z[11]*z[3]*z[4] + 1667552*z[11]*z[3]*z[5] - 1875872*z[11]*z[3] + 3239872*z[11]*z[4]*z[5] - 3567232*z[11]*z[4] - 5658368*z[11]*z[5] + 2976*z[11]*z[6]*z[7]*z[8]*z[9] - 11532*z[11]*z[6]*z[7]*z[8] - 23064*z[11]*z[6]*z[7]*z[9] + 60388*z[11]*z[6]*z[7] - 46128*z[11]*z[6]*z[8]*z[9] + 120032*z[11]*z[6]*z[8] + 234112*z[11]*z[6]*z[9] - 952351*z[11]*z[6]/2 - 92256*z[11]*z[7]*z[8]*z[9] + 239692*z[11]*z[7]*z[8] + 467480*z[11]*z[7]*z[9] - 949468*z[11]*z[7] + 929008*z[11]*z[8]*z[9] - 1875872*z[11]*z[8] - 3567232*z[11]*z[9] + 23984731*z[11]/2 + 1536*z[12]*z[13]*z[14]*z[15] - 2976*z[12]*z[13]*z[14] - 5952*z[12]*z[13]*z[15] - 12288*z[12]*z[13]*z[2]*z[3]*z[4]*z[5] + 23808*z[12]*z[13]*z[2]*z[3]*z[4] + 47616*z[12]*z[13]*z[2]*z[3]*z[5] - 61856*z[12]*z[13]*z[2]*z[3] + 95232*z[12]*z[13]*z[2]*z[4]*z[5] - 120640*z[12]*z[13]*z[2]*z[4] - 216704*z[12]*z[13]*z[2]*z[5] + 245024*z[12]*z[13]*z[2] + 190464*z[12]*z[13]*z[3]*z[4]*z[5] - 239744*z[12]*z[13]*z[3]*z[4] - 430336*z[12]*z[13]*z[3]*z[5] + 484096*z[12]*z[13]*z[3] - 836096*z[12]*z[13]*z[4]*z[5] + 920576*z[12]*z[13]*z[4] + 1460224*z[12]*z[13]*z[5] - 768*z[12]*z[13]*z[6]*z[7]*z[8]*z[9] + 2976*z[12]*z[13]*z[6]*z[7]*z[8] + 5952*z[12]*z[13]*z[6]*z[7]*z[9] - 15584*z[12]*z[13]*z[6]*z[7] + 11904*z[12]*z[13]*z[6]*z[8]*z[9] - 30976*z[12]*z[13]*z[6]*z[8] - 60416*z[12]*z[13]*z[6]*z[9] + 122884*z[12]*z[13]*z[6] + 23808*z[12]*z[13]*z[7]*z[8]*z[9] - 61856*z[12]*z[13]*z[7]*z[8] - 120640*z[12]*z[13]*z[7]*z[9] + 245024*z[12]*z[13]*z[7] - 239744*z[12]*z[13]*z[8]*z[9] + 484096*z[12]*z[13]*z[8] + 920576*z[12]*z[13]*z[9] - 3091036*z[12]*z[13] - 11904*z[12]*z[14]*z[15] - 24576*z[12]*z[14]*z[2]*z[3]*z[4]*z[5] + 47616*z[12]*z[14]*z[2]*z[3]*z[4] + 95232*z[12]*z[14]*z[2]*z[3]*z[5] - 123712*z[12]*z[14]*z[2]*z[3] + 190464*z[12]*z[14]*z[2]*z[4]*z[5] - 241280*z[12]*z[14]*z[2]*z[4] - 433408*z[12]*z[14]*z[2]*z[5] + 490048*z[12]*z[14]*z[2] + 380928*z[12]*z[14]*z[3]*z[4]*z[5] - 479488*z[12]*z[14]*z[3]*z[4] - 860672*z[12]*z[14]*z[3]*z[5] + 968192*z[12]*z[14]*z[3] - 1672192*z[12]*z[14]*z[4]*z[5] + 1841152*z[12]*z[14]*z[4] + 2920448*z[12]*z[14]*z[5] - 1536*z[12]*z[14]*z[6]*z[7]*z[8]*z[9] + 5952*z[12]*z[14]*z[6]*z[7]*z[8] + 11904*z[12]*z[14]*z[6]*z[7]*z[9] - 31168*z[12]*z[14]*z[6]*z[7] + 23808*z[12]*z[14]*z[6]*z[8]*z[9] - 61952*z[12]*z[14]*z[6]*z[8] - 120832*z[12]*z[14]*z[6]*z[9] + 245768*z[12]*z[14]*z[6] + 47616*z[12]*z[14]*z[7]*z[8]*z[9] - 123712*z[12]*z[14]*z[7]*z[8] - 241280*z[12]*z[14]*z[7]*z[9] + 490048*z[12]*z[14]*z[7] - 479488*z[12]*z[14]*z[8]*z[9] + 968192*z[12]*z[14]*z[8] + 1841152*z[12]*z[14]*z[9] - 6182456*z[12]*z[14] - 49152*z[12]*z[15]*z[2]*z[3]*z[4]*z[5] + 95232*z[12]*z[15]*z[2]*z[3]*z[4] + 190464*z[12]*z[15]*z[2]*z[3]*z[5] - 247424*z[12]*z[15]*z[2]*z[3] + 380928*z[12]*z[15]*z[2]*z[4]*z[5] - 482560*z[12]*z[15]*z[2]*z[4] - 866816*z[12]*z[15]*z[2]*z[5] + 980096*z[12]*z[15]*z[2] + 761856*z[12]*z[15]*z[3]*z[4]*z[5] - 958976*z[12]*z[15]*z[3]*z[4] - 1721344*z[12]*z[15]*z[3]*z[5] + 1936384*z[12]*z[15]*z[3] - 3344384*z[12]*z[15]*z[4]*z[5] + 3682304*z[12]*z[15]*z[4] + 5840896*z[12]*z[15]*z[5] - 3072*z[12]*z[15]*z[6]*z[7]*z[8]*z[9] + 11904*z[12]*z[15]*z[6]*z[7]*z[8] + 23808*z[12]*z[15]*z[6]*z[7]*z[9] - 62336*z[12]*z[15]*z[6]*z[7] + 47616*z[12]*z[15]*z[6]*z[8]*z[9] - 123904*z[12]*z[15]*z[6]*z[8] - 241664*z[12]*z[15]*z[6]*z[9] + 491536*z[12]*z[15]*z[6] + 95232*z[12]*z[15]*z[7]*z[8]*z[9] - 247424*z[12]*z[15]*z[7]*z[8] - 482560*z[12]*z[15]*z[7]*z[9] + 980096*z[12]*z[15]*z[7] - 958976*z[12]*z[15]*z[8]*z[9] + 1936384*z[12]*z[15]*z[8] + 3682304*z[12]*z[15]*z[9] - 12367984*z[12]*z[15] + 95232*z[12]*z[2]*z[3]*z[4]*z[5] - 184512*z[12]*z[2]*z[3]*z[4] - 369024*z[12]*z[2]*z[3]*z[5] + 479384*z[12]*z[2]*z[3] - 738048*z[12]*z[2]*z[4]*z[5] + 934960*z[12]*z[2]*z[4] + 1679456*z[12]*z[2]*z[5] - 1898936*z[12]*z[2] - 1476096*z[12]*z[3]*z[4]*z[5] + 1858016*z[12]*z[3]*z[4] + 3335104*z[12]*z[3]*z[5] - 3751744*z[12]*z[3] + 6479744*z[12]*z[4]*z[5] - 7134464*z[12]*z[4] - 11316736*z[12]*z[5] + 5952*z[12]*z[6]*z[7]*z[8]*z[9] - 23064*z[12]*z[6]*z[7]*z[8] - 46128*z[12]*z[6]*z[7]*z[9] + 120776*z[12]*z[6]*z[7] - 92256*z[12]*z[6]*z[8]*z[9] + 240064*z[12]*z[6]*z[8] + 468224*z[12]*z[6]*z[9] - 952351*z[12]*z[6] - 184512*z[12]*z[7]*z[8]*z[9] + 479384*z[12]*z[7]*z[8] + 934960*z[12]*z[7]*z[9] - 1898936*z[12]*z[7] + 1858016*z[12]*z[8]*z[9] - 3751744*z[12]*z[8] - 7134464*z[12]*z[9] + 23984824*z[12] - 23808*z[13]*z[14]*z[15] - 49152*z[13]*z[14]*z[2]*z[3]*z[4]*z[5] + 95232*z[13]*z[14]*z[2]*z[3]*z[4] + 190464*z[13]*z[14]*z[2]*z[3]*z[5] - 247424*z[13]*z[14]*z[2]*z[3] + 380928*z[13]*z[14]*z[2]*z[4]*z[5] - 482560*z[13]*z[14]*z[2]*z[4] - 866816*z[13]*z[14]*z[2]*z[5] + 980096*z[13]*z[14]*z[2] + 761856*z[13]*z[14]*z[3]*z[4]*z[5] - 958976*z[13]*z[14]*z[3]*z[4] - 1721344*z[13]*z[14]*z[3]*z[5] + 1936384*z[13]*z[14]*z[3] - 3344384*z[13]*z[14]*z[4]*z[5] + 3682304*z[13]*z[14]*z[4] + 5840896*z[13]*z[14]*z[5] - 3072*z[13]*z[14]*z[6]*z[7]*z[8]*z[9] + 11904*z[13]*z[14]*z[6]*z[7]*z[8] + 23808*z[13]*z[14]*z[6]*z[7]*z[9] - 62336*z[13]*z[14]*z[6]*z[7] + 47616*z[13]*z[14]*z[6]*z[8]*z[9] - 123904*z[13]*z[14]*z[6]*z[8] - 241664*z[13]*z[14]*z[6]*z[9] + 491536*z[13]*z[14]*z[6] + 95232*z[13]*z[14]*z[7]*z[8]*z[9] - 247424*z[13]*z[14]*z[7]*z[8] - 482560*z[13]*z[14]*z[7]*z[9] + 980096*z[13]*z[14]*z[7] - 958976*z[13]*z[14]*z[8]*z[9] + 1936384*z[13]*z[14]*z[8] + 3682304*z[13]*z[14]*z[9] - 12365104*z[13]*z[14] - 98304*z[13]*z[15]*z[2]*z[3]*z[4]*z[5] + 190464*z[13]*z[15]*z[2]*z[3]*z[4] + 380928*z[13]*z[15]*z[2]*z[3]*z[5] - 494848*z[13]*z[15]*z[2]*z[3] + 761856*z[13]*z[15]*z[2]*z[4]*z[5] - 965120*z[13]*z[15]*z[2]*z[4] - 1733632*z[13]*z[15]*z[2]*z[5] + 1960192*z[13]*z[15]*z[2] + 1523712*z[13]*z[15]*z[3]*z[4]*z[5] - 1917952*z[13]*z[15]*z[3]*z[4] - 3442688*z[13]*z[15]*z[3]*z[5] + 3872768*z[13]*z[15]*z[3] - 6688768*z[13]*z[15]*z[4]*z[5] + 7364608*z[13]*z[15]*z[4] + 11681792*z[13]*z[15]*z[5] - 6144*z[13]*z[15]*z[6]*z[7]*z[8]*z[9] + 23808*z[13]*z[15]*z[6]*z[7]*z[8] + 47616*z[13]*z[15]*z[6]*z[7]*z[9] - 124672*z[13]*z[15]*z[6]*z[7] + 95232*z[13]*z[15]*z[6]*z[8]*z[9] - 247808*z[13]*z[15]*z[6]*z[8] - 483328*z[13]*z[15]*z[6]*z[9] + 983072*z[13]*z[15]*z[6] + 190464*z[13]*z[15]*z[7]*z[8]*z[9] - 494848*z[13]*z[15]*z[7]*z[8] - 965120*z[13]*z[15]*z[7]*z[9] + 1960192*z[13]*z[15]*z[7] - 1917952*z[13]*z[15]*z[8]*z[9] + 3872768*z[13]*z[15]*z[8] + 7364608*z[13]*z[15]*z[9] - 24736352*z[13]*z[15] + 190464*z[13]*z[2]*z[3]*z[4]*z[5] - 369024*z[13]*z[2]*z[3]*z[4] - 738048*z[13]*z[2]*z[3]*z[5] + 958768*z[13]*z[2]*z[3] - 1476096*z[13]*z[2]*z[4]*z[5] + 1869920*z[13]*z[2]*z[4] + 3358912*z[13]*z[2]*z[5] - 3797872*z[13]*z[2] - 2952192*z[13]*z[3]*z[4]*z[5] + 3716032*z[13]*z[3]*z[4] + 6670208*z[13]*z[3]*z[5] - 7503488*z[13]*z[3] + 12959488*z[13]*z[4]*z[5] - 14268928*z[13]*z[4] - 22633472*z[13]*z[5] + 11904*z[13]*z[6]*z[7]*z[8]*z[9] - 46128*z[13]*z[6]*z[7]*z[8] - 92256*z[13]*z[6]*z[7]*z[9] + 241552*z[13]*z[6]*z[7] - 184512*z[13]*z[6]*z[8]*z[9] + 480128*z[13]*z[6]*z[8] + 936448*z[13]*z[6]*z[9] - 1904702*z[13]*z[6] - 369024*z[13]*z[7]*z[8]*z[9] + 958768*z[13]*z[7]*z[8] + 1869920*z[13]*z[7]*z[9] - 3797872*z[13]*z[7] + 3716032*z[13]*z[8]*z[9] - 7503488*z[13]*z[8] - 14268928*z[13]*z[9] + 47970392*z[13] - 196608*z[14]*z[15]*z[2]*z[3]*z[4]*z[5] + 380928*z[14]*z[15]*z[2]*z[3]*z[4] + 761856*z[14]*z[15]*z[2]*z[3]*z[5] - 989696*z[14]*z[15]*z[2]*z[3] + 1523712*z[14]*z[15]*z[2]*z[4]*z[5] - 1930240*z[14]*z[15]*z[2]*z[4] - 3467264*z[14]*z[15]*z[2]*z[5] + 3920384*z[14]*z[15]*z[2] + 3047424*z[14]*z[15]*z[3]*z[4]*z[5] - 3835904*z[14]*z[15]*z[3]*z[4] - 6885376*z[14]*z[15]*z[3]*z[5] + 7745536*z[14]*z[15]*z[3] - 13377536*z[14]*z[15]*z[4]*z[5] + 14729216*z[14]*z[15]*z[4] + 23363584*z[14]*z[15]*z[5] - 12288*z[14]*z[15]*z[6]*z[7]*z[8]*z[9] + 47616*z[14]*z[15]*z[6]*z[7]*z[8] + 95232*z[14]*z[15]*z[6]*z[7]*z[9] - 249344*z[14]*z[15]*z[6]*z[7] + 190464*z[14]*z[15]*z[6]*z[8]*z[9] - 495616*z[14]*z[15]*z[6]*z[8] - 966656*z[14]*z[15]*z[6]*z[9] + 1966144*z[14]*z[15]*z[6] + 380928*z[14]*z[15]*z[7]*z[8]*z[9] - 989696*z[14]*z[15]*z[7]*z[8] - 1930240*z[14]*z[15]*z[7]*z[9] + 3920384*z[14]*z[15]*z[7] - 3835904*z[14]*z[15]*z[8]*z[9] + 7745536*z[14]*z[15]*z[8] + 14729216*z[14]*z[15]*z[9] - 49475776*z[14]*z[15] + 380928*z[14]*z[2]*z[3]*z[4]*z[5] - 738048*z[14]*z[2]*z[3]*z[4] - 1476096*z[14]*z[2]*z[3]*z[5] + 1917536*z[14]*z[2]*z[3] - 2952192*z[14]*z[2]*z[4]*z[5] + 3739840*z[14]*z[2]*z[4] + 6717824*z[14]*z[2]*z[5] - 7595744*z[14]*z[2] - 5904384*z[14]*z[3]*z[4]*z[5] + 7432064*z[14]*z[3]*z[4] + 13340416*z[14]*z[3]*z[5] - 15006976*z[14]*z[3] + 25918976*z[14]*z[4]*z[5] - 28537856*z[14]*z[4] - 45266944*z[14]*z[5] + 23808*z[14]*z[6]*z[7]*z[8]*z[9] - 92256*z[14]*z[6]*z[7]*z[8] - 184512*z[14]*z[6]*z[7]*z[9] + 483104*z[14]*z[6]*z[7] - 369024*z[14]*z[6]*z[8]*z[9] + 960256*z[14]*z[6]*z[8] + 1872896*z[14]*z[6]*z[9] - 3809404*z[14]*z[6] - 738048*z[14]*z[7]*z[8]*z[9] + 1917536*z[14]*z[7]*z[8] + 3739840*z[14]*z[7]*z[9] - 7595744*z[14]*z[7] + 7432064*z[14]*z[8]*z[9] - 15006976*z[14]*z[8] - 28537856*z[14]*z[9] + 95946736*z[14] + 761856*z[15]*z[2]*z[3]*z[4]*z[5] - 1476096*z[15]*z[2]*z[3]*z[4] - 2952192*z[15]*z[2]*z[3]*z[5] + 3835072*z[15]*z[2]*z[3] - 5904384*z[15]*z[2]*z[4]*z[5] + 7479680*z[15]*z[2]*z[4] + 13435648*z[15]*z[2]*z[5] - 15191488*z[15]*z[2] - 11808768*z[15]*z[3]*z[4]*z[5] + 14864128*z[15]*z[3]*z[4] + 26680832*z[15]*z[3]*z[5] - 30013952*z[15]*z[3] + 51837952*z[15]*z[4]*z[5] - 57075712*z[15]*z[4] - 90533888*z[15]*z[5] + 47616*z[15]*z[6]*z[7]*z[8]*z[9] - 184512*z[15]*z[6]*z[7]*z[8] - 369024*z[15]*z[6]*z[7]*z[9] + 966208*z[15]*z[6]*z[7] - 738048*z[15]*z[6]*z[8]*z[9] + 1920512*z[15]*z[6]*z[8] + 3745792*z[15]*z[6]*z[9] - 7618808*z[15]*z[6] - 1476096*z[15]*z[7]*z[8]*z[9] + 3835072*z[15]*z[7]*z[8] + 7479680*z[15]*z[7]*z[9] - 15191488*z[15]*z[7] + 14864128*z[15]*z[8]*z[9] - 30013952*z[15]*z[8] - 57075712*z[15]*z[9] + 191941088*z[15] + 294912*z[2]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 1142784*z[2]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] - 2285568*z[2]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] + 5984256*z[2]*z[3]*z[4]*z[5]*z[6]*z[7] - 4571136*z[2]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] + 11894784*z[2]*z[3]*z[4]*z[5]*z[6]*z[8] + 23199744*z[2]*z[3]*z[4]*z[5]*z[6]*z[9] - 47187456*z[2]*z[3]*z[4]*z[5]*z[6] - 9142272*z[2]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] + 23752704*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] + 46325760*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] - 94089216*z[2]*z[3]*z[4]*z[5]*z[7] + 92061696*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] - 185892864*z[2]*z[3]*z[4]*z[5]*z[8] - 353501184*z[2]*z[3]*z[4]*z[5]*z[9] + 11431045632*z[2]*z[3]*z[4]*z[5] - 571392*z[2]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] + 2214144*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] + 4428288*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] - 11594496*z[2]*z[3]*z[4]*z[6]*z[7] + 8856576*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] - 23046144*z[2]*z[3]*z[4]*z[6]*z[8] - 44949504*z[2]*z[3]*z[4]*z[6]*z[9] + 91425696*z[2]*z[3]*z[4]*z[6] + 17713152*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] - 46020864*z[2]*z[3]*z[4]*z[7]*z[8] - 89756160*z[2]*z[3]*z[4]*z[7]*z[9] + 182297856*z[2]*z[3]*z[4]*z[7] - 178369536*z[2]*z[3]*z[4]*z[8]*z[9] + 360167424*z[2]*z[3]*z[4]*z[8] + 684908544*z[2]*z[3]*z[4]*z[9] - 12146103552*z[2]*z[3]*z[4] - 1142784*z[2]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] + 4428288*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] + 8856576*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] - 23188992*z[2]*z[3]*z[5]*z[6]*z[7] + 17713152*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] - 46092288*z[2]*z[3]*z[5]*z[6]*z[8] - 89899008*z[2]*z[3]*z[5]*z[6]*z[9] + 182851392*z[2]*z[3]*z[5]*z[6] + 35426304*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] - 92041728*z[2]*z[3]*z[5]*z[7]*z[8] - 179512320*z[2]*z[3]*z[5]*z[7]*z[9] + 364595712*z[2]*z[3]*z[5]*z[7] - 356739072*z[2]*z[3]*z[5]*z[8]*z[9] + 720334848*z[2]*z[3]*z[5]*z[8] + 1369817088*z[2]*z[3]*z[5]*z[9] - 16452708864*z[2]*z[3]*z[5] + 1484544*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] - 5752608*z[2]*z[3]*z[6]*z[7]*z[8] - 11505216*z[2]*z[3]*z[6]*z[7]*z[9] + 30123872*z[2]*z[3]*z[6]*z[7] - 23010432*z[2]*z[3]*z[6]*z[8]*z[9] + 59876608*z[2]*z[3]*z[6]*z[8] + 116784128*z[2]*z[3]*z[6]*z[9] - 237534772*z[2]*z[3]*z[6] - 46020864*z[2]*z[3]*z[7]*z[8]*z[9] + 119567648*z[2]*z[3]*z[7]*z[8] + 233197120*z[2]*z[3]*z[7]*z[9] - 473631392*z[2]*z[3]*z[7] + 463425152*z[2]*z[3]*z[8]*z[9] - 935757568*z[2]*z[3]*z[8] - 1779473408*z[2]*z[3]*z[9] + 17300116104*z[2]*z[3] - 2285568*z[2]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 8856576*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] + 17713152*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] - 46377984*z[2]*z[4]*z[5]*z[6]*z[7] + 35426304*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] - 92184576*z[2]*z[4]*z[5]*z[6]*z[8] - 179798016*z[2]*z[4]*z[5]*z[6]*z[9] + 365702784*z[2]*z[4]*z[5]*z[6] + 70852608*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] - 184083456*z[2]*z[4]*z[5]*z[7]*z[8] - 359024640*z[2]*z[4]*z[5]*z[7]*z[9] + 729191424*z[2]*z[4]*z[5]*z[7] - 713478144*z[2]*z[4]*z[5]*z[8]*z[9] + 1440669696*z[2]*z[4]*z[5]*z[8] + 2739634176*z[2]*z[4]*z[5]*z[9] - 28665689088*z[2]*z[4]*z[5] + 2895360*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] - 11219520*z[2]*z[4]*z[6]*z[7]*z[8] - 22439040*z[2]*z[4]*z[6]*z[7]*z[9] + 58751680*z[2]*z[4]*z[6]*z[7] - 44878080*z[2]*z[4]*z[6]*z[8]*z[9] + 116779520*z[2]*z[4]*z[6]*z[8] + 227768320*z[2]*z[4]*z[6]*z[9] - 463272680*z[2]*z[4]*z[6] - 89756160*z[2]*z[4]*z[7]*z[8]*z[9] + 233197120*z[2]*z[4]*z[7]*z[8] + 454812800*z[2]*z[4]*z[7]*z[9] - 923740480*z[2]*z[4]*z[7] + 903834880*z[2]*z[4]*z[8]*z[9] - 1825041920*z[2]*z[4]*z[8] - 3470571520*z[2]*z[4]*z[9] + 30083867280*z[2]*z[4] + 5200896*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] - 20153472*z[2]*z[5]*z[6]*z[7]*z[8] - 40306944*z[2]*z[5]*z[6]*z[7]*z[9] + 105534848*z[2]*z[5]*z[6]*z[7] - 80613888*z[2]*z[5]*z[6]*z[8]*z[9] + 209769472*z[2]*z[5]*z[6]*z[8] + 409137152*z[2]*z[5]*z[6]*z[9] - 832170448*z[2]*z[5]*z[6] - 161227776*z[2]*z[5]*z[7]*z[8]*z[9] + 418888832*z[2]*z[5]*z[7]*z[8] + 816974080*z[2]*z[5]*z[7]*z[9] - 1659302528*z[2]*z[5]*z[7] + 1623546368*z[2]*z[5]*z[8]*z[9] - 3278298112*z[2]*z[5]*z[8] - 6234140672*z[2]*z[5]*z[9] + 39942893856*z[2]*z[5] - 5880576*z[2]*z[6]*z[7]*z[8]*z[9] + 22787232*z[2]*z[6]*z[7]*z[8] + 45574464*z[2]*z[6]*z[7]*z[9] - 119326688*z[2]*z[6]*z[7] + 91148928*z[2]*z[6]*z[8]*z[9] - 237183232*z[2]*z[6]*z[8] - 462605312*z[2]*z[6]*z[9] + 940922788*z[2]*z[6] + 182297856*z[2]*z[7]*z[8]*z[9] - 473631392*z[2]*z[7]*z[8] - 923740480*z[2]*z[7]*z[9] + 1876148768*z[2]*z[7] - 1835719808*z[2]*z[8]*z[9] + 3706723072*z[2]*z[8] + 7048850432*z[2]*z[9] - 41502703776*z[2] - 4571136*z[3]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 17713152*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 35426304*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] - 92755968*z[3]*z[4]*z[5]*z[6]*z[7] + 70852608*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] - 184369152*z[3]*z[4]*z[5]*z[6]*z[8] - 359596032*z[3]*z[4]*z[5]*z[6]*z[9] + 731405568*z[3]*z[4]*z[5]*z[6] + 141705216*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 368166912*z[3]*z[4]*z[5]*z[7]*z[8] - 718049280*z[3]*z[4]*z[5]*z[7]*z[9] + 1458382848*z[3]*z[4]*z[5]*z[7] - 1426956288*z[3]*z[4]*z[5]*z[8]*z[9] + 2881339392*z[3]*z[4]*z[5]*z[8] + 5479268352*z[3]*z[4]*z[5]*z[9] - 55171516416*z[3]*z[4]*z[5] + 5753856*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 22296192*z[3]*z[4]*z[6]*z[7]*z[8] - 44592384*z[3]*z[4]*z[6]*z[7]*z[9] + 116755328*z[3]*z[4]*z[6]*z[7] - 89184768*z[3]*z[4]*z[6]*z[8]*z[9] + 232072192*z[3]*z[4]*z[6]*z[8] + 452636672*z[3]*z[4]*z[6]*z[9] - 920646928*z[3]*z[4]*z[6] - 178369536*z[3]*z[4]*z[7]*z[8]*z[9] + 463425152*z[3]*z[4]*z[7]*z[8] + 903834880*z[3]*z[4]*z[7]*z[9] - 1835719808*z[3]*z[4]*z[7] + 1796162048*z[3]*z[4]*z[8]*z[9] - 3626847232*z[3]*z[4]*z[8] - 6896955392*z[3]*z[4]*z[9] + 57859958496*z[3]*z[4] + 10328064*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 40021248*z[3]*z[5]*z[6]*z[7]*z[8] - 80042496*z[3]*z[5]*z[6]*z[7]*z[9] + 209573632*z[3]*z[5]*z[6]*z[7] - 160084992*z[3]*z[5]*z[6]*z[8]*z[9] + 416565248*z[3]*z[5]*z[6]*z[8] + 812474368*z[3]*z[5]*z[6]*z[9] - 1652544032*z[3]*z[5]*z[6] - 320169984*z[3]*z[5]*z[7]*z[8]*z[9] + 831839488*z[3]*z[5]*z[7]*z[8] + 1622366720*z[3]*z[5]*z[7]*z[9] - 3295082752*z[3]*z[5]*z[7] + 3224077312*z[3]*z[5]*z[8]*z[9] - 6510123008*z[3]*z[5]*z[8] - 12379906048*z[3]*z[5]*z[9] + 76599182784*z[3]*z[5] - 11618304*z[3]*z[6]*z[7]*z[8]*z[9] + 45020928*z[3]*z[6]*z[7]*z[8] + 90041856*z[3]*z[6]*z[7]*z[9] - 235754752*z[3]*z[6]*z[7] + 180083712*z[3]*z[6]*z[8]*z[9] - 468604928*z[3]*z[6]*z[8] - 913973248*z[3]*z[6]*z[9] + 1858989152*z[3]*z[6] + 360167424*z[3]*z[7]*z[8]*z[9] - 935757568*z[3]*z[7]*z[8] - 1825041920*z[3]*z[7]*z[9] + 3706723072*z[3]*z[7] - 3626847232*z[3]*z[8]*z[9] + 7323404288*z[3]*z[8] + 13926473728*z[3]*z[9] - 79505161344*z[3] + 20066304*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 77756928*z[4]*z[5]*z[6]*z[7]*z[8] - 155513856*z[4]*z[5]*z[6]*z[7]*z[9] + 407178752*z[4]*z[5]*z[6]*z[7] - 311027712*z[4]*z[5]*z[6]*z[8]*z[9] + 809340928*z[4]*z[5]*z[6]*z[8] + 1578549248*z[4]*z[5]*z[6]*z[9] - 3210713152*z[4]*z[5]*z[6] - 622055424*z[4]*z[5]*z[7]*z[8]*z[9] + 1616173568*z[4]*z[5]*z[7]*z[8] + 3152081920*z[4]*z[5]*z[7]*z[9] - 6401987072*z[4]*z[5]*z[7] + 6264031232*z[4]*z[5]*z[8]*z[9] - 12648460288*z[4]*z[5]*z[8] - 24052809728*z[4]*z[5]*z[9] + 131021391744*z[4]*z[5] - 22093824*z[4]*z[6]*z[7]*z[8]*z[9] + 85613568*z[4]*z[6]*z[7]*z[8] + 171227136*z[4]*z[6]*z[7]*z[9] - 448320512*z[4]*z[6]*z[7] + 342454272*z[4]*z[6]*z[8]*z[9] - 891117568*z[4]*z[6]*z[8] - 1738047488*z[4]*z[6]*z[9] + 3535126912*z[4]*z[6] + 684908544*z[4]*z[7]*z[8]*z[9] - 1779473408*z[4]*z[7]*z[8] - 3470571520*z[4]*z[7]*z[9] + 7048850432*z[4]*z[7] - 6896955392*z[4]*z[8]*z[9] + 13926473728*z[4]*z[8] + 26483130368*z[4]*z[9] - 135458068224*z[4] - 35045376*z[5]*z[6]*z[7]*z[8]*z[9] + 135800832*z[5]*z[6]*z[7]*z[8] + 271601664*z[5]*z[6]*z[7]*z[9] - 711129088*z[5]*z[6]*z[7] + 543203328*z[5]*z[6]*z[8]*z[9] - 1413496832*z[5]*z[6]*z[8] - 2756902912*z[5]*z[6]*z[9] + 5607442688*z[5]*z[6] + 1086406656*z[5]*z[7]*z[8]*z[9] - 2822612992*z[5]*z[7]*z[8] - 5505044480*z[5]*z[7]*z[9] + 11180935168*z[5]*z[7] - 10939998208*z[5]*z[8]*z[9] + 22090268672*z[5]*z[8] + 42007724032*z[5]*z[9] - 175932120576*z[5] + 1160147712*z[6]*z[7]*z[8]*z[9] - 1762700424*z[6]*z[7]*z[8] - 3154174608*z[6]*z[7]*z[9] + 4525933536*z[6]*z[7] - 6120236256*z[6]*z[8]*z[9] + 8730743424*z[6]*z[8] + 15190865664*z[6]*z[9] - 41959110297*z[6]/2 - 12146103552*z[7]*z[8]*z[9] + 17300116104*z[7]*z[8] + 30083867280*z[7]*z[9] - 41502703776*z[7] + 57859958496*z[8]*z[9] - 79505161344*z[8] - 135458068224*z[9] + 286173737095)

In [431]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 15  # 
p = 1  # QAOA depth

hamiltonian_less_31_D_Equation_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_Equation_2_less_31))
final_circuit_less_31_D_Equation_2, result_less_31_D_Equation_2 = qaoa(num_qubits, hamiltonian_less_31_D_Equation_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [-0.64240723  1.15676042]
Minimum expectation value: 11987865399.648438


In [433]:
top_solutions_less_31_D_Equation_2 = find_best_bitstrings(final_circuit_less_31_D_Equation_2, hamiltonian_less_31_D_Equation_2)

Top 5 bitstrings:
Bitstring: 111100010000101, Cost: -572347473866.0000, Count: 1
Bitstring: 110000000000101, Cost: -572347471690.0000, Count: 1
Bitstring: 110100000000101, Cost: -572347471690.0000, Count: 1
Bitstring: 110000000100101, Cost: -572347471592.0000, Count: 1
Bitstring: 110000010100001, Cost: -572347471592.0000, Count: 1


In [434]:
bitstring_to_pm1(top_solutions_less_31_D_Equation_2, evaluate_hamiltonian_D_Equation_2_less_31)

["Bitstring: ('111100010000101', -572347473866.0, 1), Evaluated cost: 324.0",
 "Bitstring: ('111100010000101', -572347473866.0, 1), Evaluated cost: 2500.0",
 "Bitstring: ('111100010000101', -572347473866.0, 1), Evaluated cost: 2500.0",
 "Bitstring: ('111100010000101', -572347473866.0, 1), Evaluated cost: 2601.0",
 "Bitstring: ('111100010000101', -572347473866.0, 1), Evaluated cost: 2601.0"]

In [435]:
paso1_D_Equation_2_less_63 = substitute_with_global_binary_symbols(D_Equation_2(x[1],x[2],x[3])**2, 6, base_name="b")
paso2_D_Equation_2_less_63 = remove_variable_exponents(paso1_D_Equation_2_less_63)
paso3_D_Equation_2_less_63 = substitute_with_spin_variables(paso2_D_Equation_2_less_63)

In [436]:
print(paso3_D_Equation_2_less_63)

1179648*z_1*z_10*z_11*z_12*z_2*z_3*z_4*z_7 + 2359296*z_1*z_10*z_11*z_12*z_2*z_3*z_4*z_8 + 4718592*z_1*z_10*z_11*z_12*z_2*z_3*z_4*z_9 - 74317824*z_1*z_10*z_11*z_12*z_2*z_3*z_4 + 2359296*z_1*z_10*z_11*z_12*z_2*z_3*z_5*z_7 + 4718592*z_1*z_10*z_11*z_12*z_2*z_3*z_5*z_8 + 9437184*z_1*z_10*z_11*z_12*z_2*z_3*z_5*z_9 - 148635648*z_1*z_10*z_11*z_12*z_2*z_3*z_5 + 4718592*z_1*z_10*z_11*z_12*z_2*z_3*z_6*z_7 + 9437184*z_1*z_10*z_11*z_12*z_2*z_3*z_6*z_8 + 18874368*z_1*z_10*z_11*z_12*z_2*z_3*z_6*z_9 - 297271296*z_1*z_10*z_11*z_12*z_2*z_3*z_6 - 9289728*z_1*z_10*z_11*z_12*z_2*z_3*z_7 - 18579456*z_1*z_10*z_11*z_12*z_2*z_3*z_8 - 37158912*z_1*z_10*z_11*z_12*z_2*z_3*z_9 + 585252864*z_1*z_10*z_11*z_12*z_2*z_3 + 4718592*z_1*z_10*z_11*z_12*z_2*z_4*z_5*z_7 + 9437184*z_1*z_10*z_11*z_12*z_2*z_4*z_5*z_8 + 18874368*z_1*z_10*z_11*z_12*z_2*z_4*z_5*z_9 - 297271296*z_1*z_10*z_11*z_12*z_2*z_4*z_5 + 9437184*z_1*z_10*z_11*z_12*z_2*z_4*z_6*z_7 + 18874368*z_1*z_10*z_11*z_12*z_2*z_4*z_6*z_8 + 37748736*z_1*z_10*z_11*z_12*z_2*

In [437]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 18  # 
p = 1  # QAOA depth

hamiltonian_less_63_D_Equation_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_Equation_2_less_63))
final_circuit_less_63_D_Equation_2, result_less_63_D_Equation_2 = qaoa(num_qubits, hamiltonian_less_63_D_Equation_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.52972348 1.20726987]
Minimum expectation value: -4408857783990.867


In [ ]:
def evaluate_hamiltonian_D_Equation_2_less_63(z):
    return(1179648*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[7] + 2359296*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[8] + 4718592*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[9] - 74317824*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[4] + 2359296*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[7] + 4718592*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[8] + 9437184*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[9] - 148635648*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[5] + 4718592*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[6]*z[7] + 9437184*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[6]*z[8] + 18874368*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[6]*z[9] - 297271296*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[6] - 9289728*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[7] - 18579456*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[8] - 37158912*z[1]*z[10]*z[11]*z[12]*z[2]*z[3]*z[9] + 585252864*z[1]*z[10]*z[11]*z[12]*z[2]*z[3] + 4718592*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[7] + 9437184*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[8] + 18874368*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[9] - 297271296*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[5] + 9437184*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[6]*z[7] + 18874368*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[6]*z[8] + 37748736*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[6]*z[9] - 594542592*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[6] - 18579456*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[7] - 37158912*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[8] - 74317824*z[1]*z[10]*z[11]*z[12]*z[2]*z[4]*z[9] + 1170505728*z[1]*z[10]*z[11]*z[12]*z[2]*z[4] + 18874368*z[1]*z[10]*z[11]*z[12]*z[2]*z[5]*z[6]*z[7] + 37748736*z[1]*z[10]*z[11]*z[12]*z[2]*z[5]*z[6]*z[8] + 75497472*z[1]*z[10]*z[11]*z[12]*z[2]*z[5]*z[6]*z[9] - 1189085184*z[1]*z[10]*z[11]*z[12]*z[2]*z[5]*z[6] - 37158912*z[1]*z[10]*z[11]*z[12]*z[2]*z[5]*z[7] - 74317824*z[1]*z[10]*z[11]*z[12]*z[2]*z[5]*z[8] - 148635648*z[1]*z[10]*z[11]*z[12]*z[2]*z[5]*z[9] + 2341011456*z[1]*z[10]*z[11]*z[12]*z[2]*z[5] - 74317824*z[1]*z[10]*z[11]*z[12]*z[2]*z[6]*z[7] - 148635648*z[1]*z[10]*z[11]*z[12]*z[2]*z[6]*z[8] - 297271296*z[1]*z[10]*z[11]*z[12]*z[2]*z[6]*z[9] + 4682022912*z[1]*z[10]*z[11]*z[12]*z[2]*z[6] + 98254848*z[1]*z[10]*z[11]*z[12]*z[2]*z[7] + 196509696*z[1]*z[10]*z[11]*z[12]*z[2]*z[8] + 393019392*z[1]*z[10]*z[11]*z[12]*z[2]*z[9] - 6190055424*z[1]*z[10]*z[11]*z[12]*z[2] + 9437184*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[7] + 18874368*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[8] + 37748736*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[9] - 594542592*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[5] + 18874368*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[6]*z[7] + 37748736*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[6]*z[8] + 75497472*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[6]*z[9] - 1189085184*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[6] - 37158912*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[7] - 74317824*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[8] - 148635648*z[1]*z[10]*z[11]*z[12]*z[3]*z[4]*z[9] + 2341011456*z[1]*z[10]*z[11]*z[12]*z[3]*z[4] + 37748736*z[1]*z[10]*z[11]*z[12]*z[3]*z[5]*z[6]*z[7] + 75497472*z[1]*z[10]*z[11]*z[12]*z[3]*z[5]*z[6]*z[8] + 150994944*z[1]*z[10]*z[11]*z[12]*z[3]*z[5]*z[6]*z[9] - 2378170368*z[1]*z[10]*z[11]*z[12]*z[3]*z[5]*z[6] - 74317824*z[1]*z[10]*z[11]*z[12]*z[3]*z[5]*z[7] - 148635648*z[1]*z[10]*z[11]*z[12]*z[3]*z[5]*z[8] - 297271296*z[1]*z[10]*z[11]*z[12]*z[3]*z[5]*z[9] + 4682022912*z[1]*z[10]*z[11]*z[12]*z[3]*z[5] - 148635648*z[1]*z[10]*z[11]*z[12]*z[3]*z[6]*z[7] - 297271296*z[1]*z[10]*z[11]*z[12]*z[3]*z[6]*z[8] - 594542592*z[1]*z[10]*z[11]*z[12]*z[3]*z[6]*z[9] + 9364045824*z[1]*z[10]*z[11]*z[12]*z[3]*z[6] + 196214784*z[1]*z[10]*z[11]*z[12]*z[3]*z[7] + 392429568*z[1]*z[10]*z[11]*z[12]*z[3]*z[8] + 784859136*z[1]*z[10]*z[11]*z[12]*z[3]*z[9] - 12361531392*z[1]*z[10]*z[11]*z[12]*z[3] + 75497472*z[1]*z[10]*z[11]*z[12]*z[4]*z[5]*z[6]*z[7] + 150994944*z[1]*z[10]*z[11]*z[12]*z[4]*z[5]*z[6]*z[8] + 301989888*z[1]*z[10]*z[11]*z[12]*z[4]*z[5]*z[6]*z[9] - 4756340736*z[1]*z[10]*z[11]*z[12]*z[4]*z[5]*z[6] - 148635648*z[1]*z[10]*z[11]*z[12]*z[4]*z[5]*z[7] - 297271296*z[1]*z[10]*z[11]*z[12]*z[4]*z[5]*z[8] - 594542592*z[1]*z[10]*z[11]*z[12]*z[4]*z[5]*z[9] + 9364045824*z[1]*z[10]*z[11]*z[12]*z[4]*z[5] - 297271296*z[1]*z[10]*z[11]*z[12]*z[4]*z[6]*z[7] - 594542592*z[1]*z[10]*z[11]*z[12]*z[4]*z[6]*z[8] - 1189085184*z[1]*z[10]*z[11]*z[12]*z[4]*z[6]*z[9] + 18728091648*z[1]*z[10]*z[11]*z[12]*z[4]*z[6] + 390070272*z[1]*z[10]*z[11]*z[12]*z[4]*z[7] + 780140544*z[1]*z[10]*z[11]*z[12]*z[4]*z[8] + 1560281088*z[1]*z[10]*z[11]*z[12]*z[4]*z[9] - 24574427136*z[1]*z[10]*z[11]*z[12]*z[4] - 594542592*z[1]*z[10]*z[11]*z[12]*z[5]*z[6]*z[7] - 1189085184*z[1]*z[10]*z[11]*z[12]*z[5]*z[6]*z[8] - 2378170368*z[1]*z[10]*z[11]*z[12]*z[5]*z[6]*z[9] + 37456183296*z[1]*z[10]*z[11]*z[12]*z[5]*z[6] + 761266176*z[1]*z[10]*z[11]*z[12]*z[5]*z[7] + 1522532352*z[1]*z[10]*z[11]*z[12]*z[5]*z[8] + 3045064704*z[1]*z[10]*z[11]*z[12]*z[5]*z[9] - 47959769088*z[1]*z[10]*z[11]*z[12]*z[5] + 1371537408*z[1]*z[10]*z[11]*z[12]*z[6]*z[7] + 2743074816*z[1]*z[10]*z[11]*z[12]*z[6]*z[8] + 5486149632*z[1]*z[10]*z[11]*z[12]*z[6]*z[9] - 86406856704*z[1]*z[10]*z[11]*z[12]*z[6] - 1560287232*z[1]*z[10]*z[11]*z[12]*z[7] - 3120574464*z[1]*z[10]*z[11]*z[12]*z[8] - 6241148928*z[1]*z[10]*z[11]*z[12]*z[9] + 98298095616*z[1]*z[10]*z[11]*z[12] + 73728*z[1]*z[10]*z[11]*z[2]*z[3]*z[4]*z[7]*z[8] + 147456*z[1]*z[10]*z[11]*z[2]*z[3]*z[4]*z[7]*z[9] - 2322432*z[1]*z[10]*z[11]*z[2]*z[3]*z[4]*z[7] + 294912*z[1]*z[10]*z[11]*z[2]*z[3]*z[4]*z[8]*z[9] - 4644864*z[1]*z[10]*z[11]*z[2]*z[3]*z[4]*z[8] - 9289728*z[1]*z[10]*z[11]*z[2]*z[3]*z[4]*z[9] + 94384128*z[1]*z[10]*z[11]*z[2]*z[3]*z[4] + 147456*z[1]*z[10]*z[11]*z[2]*z[3]*z[5]*z[7]*z[8] + 294912*z[1]*z[10]*z[11]*z[2]*z[3]*z[5]*z[7]*z[9] - 4644864*z[1]*z[10]*z[11]*z[2]*z[3]*z[5]*z[7] + 589824*z[1]*z[10]*z[11]*z[2]*z[3]*z[5]*z[8]*z[9] - 9289728*z[1]*z[10]*z[11]*z[2]*z[3]*z[5]*z[8] - 18579456*z[1]*z[10]*z[11]*z[2]*z[3]*z[5]*z[9] + 188768256*z[1]*z[10]*z[11]*z[2]*z[3]*z[5] + 294912*z[1]*z[10]*z[11]*z[2]*z[3]*z[6]*z[7]*z[8] + 589824*z[1]*z[10]*z[11]*z[2]*z[3]*z[6]*z[7]*z[9] - 9289728*z[1]*z[10]*z[11]*z[2]*z[3]*z[6]*z[7] + 1179648*z[1]*z[10]*z[11]*z[2]*z[3]*z[6]*z[8]*z[9] - 18579456*z[1]*z[10]*z[11]*z[2]*z[3]*z[6]*z[8] - 37158912*z[1]*z[10]*z[11]*z[2]*z[3]*z[6]*z[9] + 377536512*z[1]*z[10]*z[11]*z[2]*z[3]*z[6] - 580608*z[1]*z[10]*z[11]*z[2]*z[3]*z[7]*z[8] - 1161216*z[1]*z[10]*z[11]*z[2]*z[3]*z[7]*z[9] + 18289152*z[1]*z[10]*z[11]*z[2]*z[3]*z[7] - 2322432*z[1]*z[10]*z[11]*z[2]*z[3]*z[8]*z[9] + 36578304*z[1]*z[10]*z[11]*z[2]*z[3]*z[8] + 73156608*z[1]*z[10]*z[11]*z[2]*z[3]*z[9] - 743275008*z[1]*z[10]*z[11]*z[2]*z[3] + 294912*z[1]*z[10]*z[11]*z[2]*z[4]*z[5]*z[7]*z[8] + 589824*z[1]*z[10]*z[11]*z[2]*z[4]*z[5]*z[7]*z[9] - 9289728*z[1]*z[10]*z[11]*z[2]*z[4]*z[5]*z[7] + 1179648*z[1]*z[10]*z[11]*z[2]*z[4]*z[5]*z[8]*z[9] - 18579456*z[1]*z[10]*z[11]*z[2]*z[4]*z[5]*z[8] - 37158912*z[1]*z[10]*z[11]*z[2]*z[4]*z[5]*z[9] + 377536512*z[1]*z[10]*z[11]*z[2]*z[4]*z[5] + 589824*z[1]*z[10]*z[11]*z[2]*z[4]*z[6]*z[7]*z[8] + 1179648*z[1]*z[10]*z[11]*z[2]*z[4]*z[6]*z[7]*z[9] - 18579456*z[1]*z[10]*z[11]*z[2]*z[4]*z[6]*z[7] + 2359296*z[1]*z[10]*z[11]*z[2]*z[4]*z[6]*z[8]*z[9] - 37158912*z[1]*z[10]*z[11]*z[2]*z[4]*z[6]*z[8] - 74317824*z[1]*z[10]*z[11]*z[2]*z[4]*z[6]*z[9] + 755073024*z[1]*z[10]*z[11]*z[2]*z[4]*z[6] - 1161216*z[1]*z[10]*z[11]*z[2]*z[4]*z[7]*z[8] - 2322432*z[1]*z[10]*z[11]*z[2]*z[4]*z[7]*z[9] + 36578304*z[1]*z[10]*z[11]*z[2]*z[4]*z[7] - 4644864*z[1]*z[10]*z[11]*z[2]*z[4]*z[8]*z[9] + 73156608*z[1]*z[10]*z[11]*z[2]*z[4]*z[8] + 146313216*z[1]*z[10]*z[11]*z[2]*z[4]*z[9] - 1486550016*z[1]*z[10]*z[11]*z[2]*z[4] + 1179648*z[1]*z[10]*z[11]*z[2]*z[5]*z[6]*z[7]*z[8] + 2359296*z[1]*z[10]*z[11]*z[2]*z[5]*z[6]*z[7]*z[9] - 37158912*z[1]*z[10]*z[11]*z[2]*z[5]*z[6]*z[7] + 4718592*z[1]*z[10]*z[11]*z[2]*z[5]*z[6]*z[8]*z[9] - 74317824*z[1]*z[10]*z[11]*z[2]*z[5]*z[6]*z[8] - 148635648*z[1]*z[10]*z[11]*z[2]*z[5]*z[6]*z[9] + 1510146048*z[1]*z[10]*z[11]*z[2]*z[5]*z[6] - 2322432*z[1]*z[10]*z[11]*z[2]*z[5]*z[7]*z[8] - 4644864*z[1]*z[10]*z[11]*z[2]*z[5]*z[7]*z[9] + 73156608*z[1]*z[10]*z[11]*z[2]*z[5]*z[7] - 9289728*z[1]*z[10]*z[11]*z[2]*z[5]*z[8]*z[9] + 146313216*z[1]*z[10]*z[11]*z[2]*z[5]*z[8] + 292626432*z[1]*z[10]*z[11]*z[2]*z[5]*z[9] - 2973100032*z[1]*z[10]*z[11]*z[2]*z[5] - 4644864*z[1]*z[10]*z[11]*z[2]*z[6]*z[7]*z[8] - 9289728*z[1]*z[10]*z[11]*z[2]*z[6]*z[7]*z[9] + 146313216*z[1]*z[10]*z[11]*z[2]*z[6]*z[7] - 18579456*z[1]*z[10]*z[11]*z[2]*z[6]*z[8]*z[9] + 292626432*z[1]*z[10]*z[11]*z[2]*z[6]*z[8] + 585252864*z[1]*z[10]*z[11]*z[2]*z[6]*z[9] - 5946200064*z[1]*z[10]*z[11]*z[2]*z[6] + 6140928*z[1]*z[10]*z[11]*z[2]*z[7]*z[8] + 12281856*z[1]*z[10]*z[11]*z[2]*z[7]*z[9] - 193439232*z[1]*z[10]*z[11]*z[2]*z[7] + 24563712*z[1]*z[10]*z[11]*z[2]*z[8]*z[9] - 386878464*z[1]*z[10]*z[11]*z[2]*z[8] - 773756928*z[1]*z[10]*z[11]*z[2]*z[9] + 7861411328*z[1]*z[10]*z[11]*z[2] + 589824*z[1]*z[10]*z[11]*z[3]*z[4]*z[5]*z[7]*z[8] + 1179648*z[1]*z[10]*z[11]*z[3]*z[4]*z[5]*z[7]*z[9] - 18579456*z[1]*z[10]*z[11]*z[3]*z[4]*z[5]*z[7] + 2359296*z[1]*z[10]*z[11]*z[3]*z[4]*z[5]*z[8]*z[9] - 37158912*z[1]*z[10]*z[11]*z[3]*z[4]*z[5]*z[8] - 74317824*z[1]*z[10]*z[11]*z[3]*z[4]*z[5]*z[9] + 755073024*z[1]*z[10]*z[11]*z[3]*z[4]*z[5] + 1179648*z[1]*z[10]*z[11]*z[3]*z[4]*z[6]*z[7]*z[8] + 2359296*z[1]*z[10]*z[11]*z[3]*z[4]*z[6]*z[7]*z[9] - 37158912*z[1]*z[10]*z[11]*z[3]*z[4]*z[6]*z[7] + 4718592*z[1]*z[10]*z[11]*z[3]*z[4]*z[6]*z[8]*z[9] - 74317824*z[1]*z[10]*z[11]*z[3]*z[4]*z[6]*z[8] - 148635648*z[1]*z[10]*z[11]*z[3]*z[4]*z[6]*z[9] + 1510146048*z[1]*z[10]*z[11]*z[3]*z[4]*z[6] - 2322432*z[1]*z[10]*z[11]*z[3]*z[4]*z[7]*z[8] - 4644864*z[1]*z[10]*z[11]*z[3]*z[4]*z[7]*z[9] + 73156608*z[1]*z[10]*z[11]*z[3]*z[4]*z[7] - 9289728*z[1]*z[10]*z[11]*z[3]*z[4]*z[8]*z[9] + 146313216*z[1]*z[10]*z[11]*z[3]*z[4]*z[8] + 292626432*z[1]*z[10]*z[11]*z[3]*z[4]*z[9] - 2973100032*z[1]*z[10]*z[11]*z[3]*z[4] + 2359296*z[1]*z[10]*z[11]*z[3]*z[5]*z[6]*z[7]*z[8] + 4718592*z[1]*z[10]*z[11]*z[3]*z[5]*z[6]*z[7]*z[9] - 74317824*z[1]*z[10]*z[11]*z[3]*z[5]*z[6]*z[7] + 9437184*z[1]*z[10]*z[11]*z[3]*z[5]*z[6]*z[8]*z[9] - 148635648*z[1]*z[10]*z[11]*z[3]*z[5]*z[6]*z[8] - 297271296*z[1]*z[10]*z[11]*z[3]*z[5]*z[6]*z[9] + 3020292096*z[1]*z[10]*z[11]*z[3]*z[5]*z[6] - 4644864*z[1]*z[10]*z[11]*z[3]*z[5]*z[7]*z[8] - 9289728*z[1]*z[10]*z[11]*z[3]*z[5]*z[7]*z[9] + 146313216*z[1]*z[10]*z[11]*z[3]*z[5]*z[7] - 18579456*z[1]*z[10]*z[11]*z[3]*z[5]*z[8]*z[9] + 292626432*z[1]*z[10]*z[11]*z[3]*z[5]*z[8] + 585252864*z[1]*z[10]*z[11]*z[3]*z[5]*z[9] - 5946200064*z[1]*z[10]*z[11]*z[3]*z[5] - 9289728*z[1]*z[10]*z[11]*z[3]*z[6]*z[7]*z[8] - 18579456*z[1]*z[10]*z[11]*z[3]*z[6]*z[7]*z[9] + 292626432*z[1]*z[10]*z[11]*z[3]*z[6]*z[7] - 37158912*z[1]*z[10]*z[11]*z[3]*z[6]*z[8]*z[9] + 585252864*z[1]*z[10]*z[11]*z[3]*z[6]*z[8] + 1170505728*z[1]*z[10]*z[11]*z[3]*z[6]*z[9] - 11892400128*z[1]*z[10]*z[11]*z[3]*z[6] + 12263424*z[1]*z[10]*z[11]*z[3]*z[7]*z[8] + 24526848*z[1]*z[10]*z[11]*z[3]*z[7]*z[9] - 386297856*z[1]*z[10]*z[11]*z[3]*z[7] + 49053696*z[1]*z[10]*z[11]*z[3]*z[8]*z[9] - 772595712*z[1]*z[10]*z[11]*z[3]*z[8] - 1545191424*z[1]*z[10]*z[11]*z[3]*z[9] + 15699226624*z[1]*z[10]*z[11]*z[3] + 4718592*z[1]*z[10]*z[11]*z[4]*z[5]*z[6]*z[7]*z[8] + 9437184*z[1]*z[10]*z[11]*z[4]*z[5]*z[6]*z[7]*z[9] - 148635648*z[1]*z[10]*z[11]*z[4]*z[5]*z[6]*z[7] + 18874368*z[1]*z[10]*z[11]*z[4]*z[5]*z[6]*z[8]*z[9] - 297271296*z[1]*z[10]*z[11]*z[4]*z[5]*z[6]*z[8] - 594542592*z[1]*z[10]*z[11]*z[4]*z[5]*z[6]*z[9] + 6040584192*z[1]*z[10]*z[11]*z[4]*z[5]*z[6] - 9289728*z[1]*z[10]*z[11]*z[4]*z[5]*z[7]*z[8] - 18579456*z[1]*z[10]*z[11]*z[4]*z[5]*z[7]*z[9] + 292626432*z[1]*z[10]*z[11]*z[4]*z[5]*z[7] - 37158912*z[1]*z[10]*z[11]*z[4]*z[5]*z[8]*z[9] + 585252864*z[1]*z[10]*z[11]*z[4]*z[5]*z[8] + 1170505728*z[1]*z[10]*z[11]*z[4]*z[5]*z[9] - 11892400128*z[1]*z[10]*z[11]*z[4]*z[5] - 18579456*z[1]*z[10]*z[11]*z[4]*z[6]*z[7]*z[8] - 37158912*z[1]*z[10]*z[11]*z[4]*z[6]*z[7]*z[9] + 585252864*z[1]*z[10]*z[11]*z[4]*z[6]*z[7] - 74317824*z[1]*z[10]*z[11]*z[4]*z[6]*z[8]*z[9] + 1170505728*z[1]*z[10]*z[11]*z[4]*z[6]*z[8] + 2341011456*z[1]*z[10]*z[11]*z[4]*z[6]*z[9] - 23784800256*z[1]*z[10]*z[11]*z[4]*z[6] + 24379392*z[1]*z[10]*z[11]*z[4]*z[7]*z[8] + 48758784*z[1]*z[10]*z[11]*z[4]*z[7]*z[9] - 767950848*z[1]*z[10]*z[11]*z[4]*z[7] + 97517568*z[1]*z[10]*z[11]*z[4]*z[8]*z[9] - 1535901696*z[1]*z[10]*z[11]*z[4]*z[8] - 3071803392*z[1]*z[10]*z[11]*z[4]*z[9] + 31209684992*z[1]*z[10]*z[11]*z[4] - 37158912*z[1]*z[10]*z[11]*z[5]*z[6]*z[7]*z[8] - 74317824*z[1]*z[10]*z[11]*z[5]*z[6]*z[7]*z[9] + 1170505728*z[1]*z[10]*z[11]*z[5]*z[6]*z[7] - 148635648*z[1]*z[10]*z[11]*z[5]*z[6]*z[8]*z[9] + 2341011456*z[1]*z[10]*z[11]*z[5]*z[6]*z[8] + 4682022912*z[1]*z[10]*z[11]*z[5]*z[6]*z[9] - 47569600512*z[1]*z[10]*z[11]*z[5]*z[6] + 47579136*z[1]*z[10]*z[11]*z[5]*z[7]*z[8] + 95158272*z[1]*z[10]*z[11]*z[5]*z[7]*z[9] - 1498742784*z[1]*z[10]*z[11]*z[5]*z[7] + 190316544*z[1]*z[10]*z[11]*z[5]*z[8]*z[9] - 2997485568*z[1]*z[10]*z[11]*z[5]*z[8] - 5994971136*z[1]*z[10]*z[11]*z[5]*z[9] + 60909223936*z[1]*z[10]*z[11]*z[5] + 85721088*z[1]*z[10]*z[11]*z[6]*z[7]*z[8] + 171442176*z[1]*z[10]*z[11]*z[6]*z[7]*z[9] - 2700214272*z[1]*z[10]*z[11]*z[6]*z[7] + 342884352*z[1]*z[10]*z[11]*z[6]*z[8]*z[9] - 5400428544*z[1]*z[10]*z[11]*z[6]*z[8] - 10800857088*z[1]*z[10]*z[11]*z[6]*z[9] + 109737279488*z[1]*z[10]*z[11]*z[6] - 97517952*z[1]*z[10]*z[11]*z[7]*z[8] - 195035904*z[1]*z[10]*z[11]*z[7]*z[9] + 3071815488*z[1]*z[10]*z[11]*z[7] - 390071808*z[1]*z[10]*z[11]*z[8]*z[9] + 6143630976*z[1]*z[10]*z[11]*z[8] + 12287261952*z[1]*z[10]*z[11]*z[9] - 124839231552*z[1]*z[10]*z[11] + 147456*z[1]*z[10]*z[12]*z[2]*z[3]*z[4]*z[7]*z[8] + 294912*z[1]*z[10]*z[12]*z[2]*z[3]*z[4]*z[7]*z[9] - 4644864*z[1]*z[10]*z[12]*z[2]*z[3]*z[4]*z[7] + 589824*z[1]*z[10]*z[12]*z[2]*z[3]*z[4]*z[8]*z[9] - 9289728*z[1]*z[10]*z[12]*z[2]*z[3]*z[4]*z[8] - 18579456*z[1]*z[10]*z[12]*z[2]*z[3]*z[4]*z[9] + 169893888*z[1]*z[10]*z[12]*z[2]*z[3]*z[4] + 294912*z[1]*z[10]*z[12]*z[2]*z[3]*z[5]*z[7]*z[8] + 589824*z[1]*z[10]*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] - 9289728*z[1]*z[10]*z[12]*z[2]*z[3]*z[5]*z[7] + 1179648*z[1]*z[10]*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] - 18579456*z[1]*z[10]*z[12]*z[2]*z[3]*z[5]*z[8] - 37158912*z[1]*z[10]*z[12]*z[2]*z[3]*z[5]*z[9] + 339787776*z[1]*z[10]*z[12]*z[2]*z[3]*z[5] + 589824*z[1]*z[10]*z[12]*z[2]*z[3]*z[6]*z[7]*z[8] + 1179648*z[1]*z[10]*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] - 18579456*z[1]*z[10]*z[12]*z[2]*z[3]*z[6]*z[7] + 2359296*z[1]*z[10]*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] - 37158912*z[1]*z[10]*z[12]*z[2]*z[3]*z[6]*z[8] - 74317824*z[1]*z[10]*z[12]*z[2]*z[3]*z[6]*z[9] + 679575552*z[1]*z[10]*z[12]*z[2]*z[3]*z[6] - 1161216*z[1]*z[10]*z[12]*z[2]*z[3]*z[7]*z[8] - 2322432*z[1]*z[10]*z[12]*z[2]*z[3]*z[7]*z[9] + 36578304*z[1]*z[10]*z[12]*z[2]*z[3]*z[7] - 4644864*z[1]*z[10]*z[12]*z[2]*z[3]*z[8]*z[9] + 73156608*z[1]*z[10]*z[12]*z[2]*z[3]*z[8] + 146313216*z[1]*z[10]*z[12]*z[2]*z[3]*z[9] - 1337914368*z[1]*z[10]*z[12]*z[2]*z[3] + 589824*z[1]*z[10]*z[12]*z[2]*z[4]*z[5]*z[7]*z[8] + 1179648*z[1]*z[10]*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] - 18579456*z[1]*z[10]*z[12]*z[2]*z[4]*z[5]*z[7] + 2359296*z[1]*z[10]*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] - 37158912*z[1]*z[10]*z[12]*z[2]*z[4]*z[5]*z[8] - 74317824*z[1]*z[10]*z[12]*z[2]*z[4]*z[5]*z[9] + 679575552*z[1]*z[10]*z[12]*z[2]*z[4]*z[5] + 1179648*z[1]*z[10]*z[12]*z[2]*z[4]*z[6]*z[7]*z[8] + 2359296*z[1]*z[10]*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] - 37158912*z[1]*z[10]*z[12]*z[2]*z[4]*z[6]*z[7] + 4718592*z[1]*z[10]*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] - 74317824*z[1]*z[10]*z[12]*z[2]*z[4]*z[6]*z[8] - 148635648*z[1]*z[10]*z[12]*z[2]*z[4]*z[6]*z[9] + 1359151104*z[1]*z[10]*z[12]*z[2]*z[4]*z[6] - 2322432*z[1]*z[10]*z[12]*z[2]*z[4]*z[7]*z[8] - 4644864*z[1]*z[10]*z[12]*z[2]*z[4]*z[7]*z[9] + 73156608*z[1]*z[10]*z[12]*z[2]*z[4]*z[7] - 9289728*z[1]*z[10]*z[12]*z[2]*z[4]*z[8]*z[9] + 146313216*z[1]*z[10]*z[12]*z[2]*z[4]*z[8] + 292626432*z[1]*z[10]*z[12]*z[2]*z[4]*z[9] - 2675828736*z[1]*z[10]*z[12]*z[2]*z[4] + 2359296*z[1]*z[10]*z[12]*z[2]*z[5]*z[6]*z[7]*z[8] + 4718592*z[1]*z[10]*z[12]*z[2]*z[5]*z[6]*z[7]*z[9] - 74317824*z[1]*z[10]*z[12]*z[2]*z[5]*z[6]*z[7] + 9437184*z[1]*z[10]*z[12]*z[2]*z[5]*z[6]*z[8]*z[9] - 148635648*z[1]*z[10]*z[12]*z[2]*z[5]*z[6]*z[8] - 297271296*z[1]*z[10]*z[12]*z[2]*z[5]*z[6]*z[9] + 2718302208*z[1]*z[10]*z[12]*z[2]*z[5]*z[6] - 4644864*z[1]*z[10]*z[12]*z[2]*z[5]*z[7]*z[8] - 9289728*z[1]*z[10]*z[12]*z[2]*z[5]*z[7]*z[9] + 146313216*z[1]*z[10]*z[12]*z[2]*z[5]*z[7] - 18579456*z[1]*z[10]*z[12]*z[2]*z[5]*z[8]*z[9] + 292626432*z[1]*z[10]*z[12]*z[2]*z[5]*z[8] + 585252864*z[1]*z[10]*z[12]*z[2]*z[5]*z[9] - 5351657472*z[1]*z[10]*z[12]*z[2]*z[5] - 9289728*z[1]*z[10]*z[12]*z[2]*z[6]*z[7]*z[8] - 18579456*z[1]*z[10]*z[12]*z[2]*z[6]*z[7]*z[9] + 292626432*z[1]*z[10]*z[12]*z[2]*z[6]*z[7] - 37158912*z[1]*z[10]*z[12]*z[2]*z[6]*z[8]*z[9] + 585252864*z[1]*z[10]*z[12]*z[2]*z[6]*z[8] + 1170505728*z[1]*z[10]*z[12]*z[2]*z[6]*z[9] - 10703314944*z[1]*z[10]*z[12]*z[2]*z[6] + 12281856*z[1]*z[10]*z[12]*z[2]*z[7]*z[8] + 24563712*z[1]*z[10]*z[12]*z[2]*z[7]*z[9] - 386878464*z[1]*z[10]*z[12]*z[2]*z[7] + 49127424*z[1]*z[10]*z[12]*z[2]*z[8]*z[9] - 773756928*z[1]*z[10]*z[12]*z[2]*z[8] - 1547513856*z[1]*z[10]*z[12]*z[2]*z[9] + 14150745088*z[1]*z[10]*z[12]*z[2] + 1179648*z[1]*z[10]*z[12]*z[3]*z[4]*z[5]*z[7]*z[8] + 2359296*z[1]*z[10]*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] - 37158912*z[1]*z[10]*z[12]*z[3]*z[4]*z[5]*z[7] + 4718592*z[1]*z[10]*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] - 74317824*z[1]*z[10]*z[12]*z[3]*z[4]*z[5]*z[8] - 148635648*z[1]*z[10]*z[12]*z[3]*z[4]*z[5]*z[9] + 1359151104*z[1]*z[10]*z[12]*z[3]*z[4]*z[5] + 2359296*z[1]*z[10]*z[12]*z[3]*z[4]*z[6]*z[7]*z[8] + 4718592*z[1]*z[10]*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] - 74317824*z[1]*z[10]*z[12]*z[3]*z[4]*z[6]*z[7] + 9437184*z[1]*z[10]*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] - 148635648*z[1]*z[10]*z[12]*z[3]*z[4]*z[6]*z[8] - 297271296*z[1]*z[10]*z[12]*z[3]*z[4]*z[6]*z[9] + 2718302208*z[1]*z[10]*z[12]*z[3]*z[4]*z[6] - 4644864*z[1]*z[10]*z[12]*z[3]*z[4]*z[7]*z[8] - 9289728*z[1]*z[10]*z[12]*z[3]*z[4]*z[7]*z[9] + 146313216*z[1]*z[10]*z[12]*z[3]*z[4]*z[7] - 18579456*z[1]*z[10]*z[12]*z[3]*z[4]*z[8]*z[9] + 292626432*z[1]*z[10]*z[12]*z[3]*z[4]*z[8] + 585252864*z[1]*z[10]*z[12]*z[3]*z[4]*z[9] - 5351657472*z[1]*z[10]*z[12]*z[3]*z[4] + 4718592*z[1]*z[10]*z[12]*z[3]*z[5]*z[6]*z[7]*z[8] + 9437184*z[1]*z[10]*z[12]*z[3]*z[5]*z[6]*z[7]*z[9] - 148635648*z[1]*z[10]*z[12]*z[3]*z[5]*z[6]*z[7] + 18874368*z[1]*z[10]*z[12]*z[3]*z[5]*z[6]*z[8]*z[9] - 297271296*z[1]*z[10]*z[12]*z[3]*z[5]*z[6]*z[8] - 594542592*z[1]*z[10]*z[12]*z[3]*z[5]*z[6]*z[9] + 5436604416*z[1]*z[10]*z[12]*z[3]*z[5]*z[6] - 9289728*z[1]*z[10]*z[12]*z[3]*z[5]*z[7]*z[8] - 18579456*z[1]*z[10]*z[12]*z[3]*z[5]*z[7]*z[9] + 292626432*z[1]*z[10]*z[12]*z[3]*z[5]*z[7] - 37158912*z[1]*z[10]*z[12]*z[3]*z[5]*z[8]*z[9] + 585252864*z[1]*z[10]*z[12]*z[3]*z[5]*z[8] + 1170505728*z[1]*z[10]*z[12]*z[3]*z[5]*z[9] - 10703314944*z[1]*z[10]*z[12]*z[3]*z[5] - 18579456*z[1]*z[10]*z[12]*z[3]*z[6]*z[7]*z[8] - 37158912*z[1]*z[10]*z[12]*z[3]*z[6]*z[7]*z[9] + 585252864*z[1]*z[10]*z[12]*z[3]*z[6]*z[7] - 74317824*z[1]*z[10]*z[12]*z[3]*z[6]*z[8]*z[9] + 1170505728*z[1]*z[10]*z[12]*z[3]*z[6]*z[8] + 2341011456*z[1]*z[10]*z[12]*z[3]*z[6]*z[9] - 21406629888*z[1]*z[10]*z[12]*z[3]*z[6] + 24526848*z[1]*z[10]*z[12]*z[3]*z[7]*z[8] + 49053696*z[1]*z[10]*z[12]*z[3]*z[7]*z[9] - 772595712*z[1]*z[10]*z[12]*z[3]*z[7] + 98107392*z[1]*z[10]*z[12]*z[3]*z[8]*z[9] - 1545191424*z[1]*z[10]*z[12]*z[3]*z[8] - 3090382848*z[1]*z[10]*z[12]*z[3]*z[9] + 28259016704*z[1]*z[10]*z[12]*z[3] + 9437184*z[1]*z[10]*z[12]*z[4]*z[5]*z[6]*z[7]*z[8] + 18874368*z[1]*z[10]*z[12]*z[4]*z[5]*z[6]*z[7]*z[9] - 297271296*z[1]*z[10]*z[12]*z[4]*z[5]*z[6]*z[7] + 37748736*z[1]*z[10]*z[12]*z[4]*z[5]*z[6]*z[8]*z[9] - 594542592*z[1]*z[10]*z[12]*z[4]*z[5]*z[6]*z[8] - 1189085184*z[1]*z[10]*z[12]*z[4]*z[5]*z[6]*z[9] + 10873208832*z[1]*z[10]*z[12]*z[4]*z[5]*z[6] - 18579456*z[1]*z[10]*z[12]*z[4]*z[5]*z[7]*z[8] - 37158912*z[1]*z[10]*z[12]*z[4]*z[5]*z[7]*z[9] + 585252864*z[1]*z[10]*z[12]*z[4]*z[5]*z[7] - 74317824*z[1]*z[10]*z[12]*z[4]*z[5]*z[8]*z[9] + 1170505728*z[1]*z[10]*z[12]*z[4]*z[5]*z[8] + 2341011456*z[1]*z[10]*z[12]*z[4]*z[5]*z[9] - 21406629888*z[1]*z[10]*z[12]*z[4]*z[5] - 37158912*z[1]*z[10]*z[12]*z[4]*z[6]*z[7]*z[8] - 74317824*z[1]*z[10]*z[12]*z[4]*z[6]*z[7]*z[9] + 1170505728*z[1]*z[10]*z[12]*z[4]*z[6]*z[7] - 148635648*z[1]*z[10]*z[12]*z[4]*z[6]*z[8]*z[9] + 2341011456*z[1]*z[10]*z[12]*z[4]*z[6]*z[8] + 4682022912*z[1]*z[10]*z[12]*z[4]*z[6]*z[9] - 42813259776*z[1]*z[10]*z[12]*z[4]*z[6] + 48758784*z[1]*z[10]*z[12]*z[4]*z[7]*z[8] + 97517568*z[1]*z[10]*z[12]*z[4]*z[7]*z[9] - 1535901696*z[1]*z[10]*z[12]*z[4]*z[7] + 195035136*z[1]*z[10]*z[12]*z[4]*z[8]*z[9] - 3071803392*z[1]*z[10]*z[12]*z[4]*z[8] - 6143606784*z[1]*z[10]*z[12]*z[4]*z[9] + 56178245632*z[1]*z[10]*z[12]*z[4] - 74317824*z[1]*z[10]*z[12]*z[5]*z[6]*z[7]*z[8] - 148635648*z[1]*z[10]*z[12]*z[5]*z[6]*z[7]*z[9] + 2341011456*z[1]*z[10]*z[12]*z[5]*z[6]*z[7] - 297271296*z[1]*z[10]*z[12]*z[5]*z[6]*z[8]*z[9] + 4682022912*z[1]*z[10]*z[12]*z[5]*z[6]*z[8] + 9364045824*z[1]*z[10]*z[12]*z[5]*z[6]*z[9] - 85626519552*z[1]*z[10]*z[12]*z[5]*z[6] + 95158272*z[1]*z[10]*z[12]*z[5]*z[7]*z[8] + 190316544*z[1]*z[10]*z[12]*z[5]*z[7]*z[9] - 2997485568*z[1]*z[10]*z[12]*z[5]*z[7] + 380633088*z[1]*z[10]*z[12]*z[5]*z[8]*z[9] - 5994971136*z[1]*z[10]*z[12]*z[5]*z[8] - 11989942272*z[1]*z[10]*z[12]*z[5]*z[9] + 109638189056*z[1]*z[10]*z[12]*z[5] + 171442176*z[1]*z[10]*z[12]*z[6]*z[7]*z[8] + 342884352*z[1]*z[10]*z[12]*z[6]*z[7]*z[9] - 5400428544*z[1]*z[10]*z[12]*z[6]*z[7] + 685768704*z[1]*z[10]*z[12]*z[6]*z[8]*z[9] - 10800857088*z[1]*z[10]*z[12]*z[6]*z[8] - 21601714176*z[1]*z[10]*z[12]*z[6]*z[9] + 197529960448*z[1]*z[10]*z[12]*z[6] - 195035904*z[1]*z[10]*z[12]*z[7]*z[8] - 390071808*z[1]*z[10]*z[12]*z[7]*z[9] + 6143630976*z[1]*z[10]*z[12]*z[7] - 780143616*z[1]*z[10]*z[12]*z[8]*z[9] + 12287261952*z[1]*z[10]*z[12]*z[8] + 24574523904*z[1]*z[10]*z[12]*z[9] - 224713867392*z[1]*z[10]*z[12] + 18432*z[1]*z[10]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] - 290304*z[1]*z[10]*z[2]*z[3]*z[4]*z[7]*z[8] - 580608*z[1]*z[10]*z[2]*z[3]*z[4]*z[7]*z[9] + 6094848*z[1]*z[10]*z[2]*z[3]*z[4]*z[7] - 1161216*z[1]*z[10]*z[2]*z[3]*z[4]*z[8]*z[9] + 12185088*z[1]*z[10]*z[2]*z[3]*z[4]*z[8] + 24333312*z[1]*z[10]*z[2]*z[3]*z[4]*z[9] - 191987712*z[1]*z[10]*z[2]*z[3]*z[4] + 36864*z[1]*z[10]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] - 580608*z[1]*z[10]*z[2]*z[3]*z[5]*z[7]*z[8] - 1161216*z[1]*z[10]*z[2]*z[3]*z[5]*z[7]*z[9] + 12189696*z[1]*z[10]*z[2]*z[3]*z[5]*z[7] - 2322432*z[1]*z[10]*z[2]*z[3]*z[5]*z[8]*z[9] + 24370176*z[1]*z[10]*z[2]*z[3]*z[5]*z[8] + 48666624*z[1]*z[10]*z[2]*z[3]*z[5]*z[9] - 383975424*z[1]*z[10]*z[2]*z[3]*z[5] + 73728*z[1]*z[10]*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] - 1161216*z[1]*z[10]*z[2]*z[3]*z[6]*z[7]*z[8] - 2322432*z[1]*z[10]*z[2]*z[3]*z[6]*z[7]*z[9] + 24379392*z[1]*z[10]*z[2]*z[3]*z[6]*z[7] - 4644864*z[1]*z[10]*z[2]*z[3]*z[6]*z[8]*z[9] + 48740352*z[1]*z[10]*z[2]*z[3]*z[6]*z[8] + 97333248*z[1]*z[10]*z[2]*z[3]*z[6]*z[9] - 767950848*z[1]*z[10]*z[2]*z[3]*z[6] - 145152*z[1]*z[10]*z[2]*z[3]*z[7]*z[8]*z[9] + 2286144*z[1]*z[10]*z[2]*z[3]*z[7]*z[8] + 4572288*z[1]*z[10]*z[2]*z[3]*z[7]*z[9] - 47996928*z[1]*z[10]*z[2]*z[3]*z[7] + 9144576*z[1]*z[10]*z[2]*z[3]*z[8]*z[9] - 95957568*z[1]*z[10]*z[2]*z[3]*z[8] - 191624832*z[1]*z[10]*z[2]*z[3]*z[9] + 1511903232*z[1]*z[10]*z[2]*z[3] + 73728*z[1]*z[10]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] - 1161216*z[1]*z[10]*z[2]*z[4]*z[5]*z[7]*z[8] - 2322432*z[1]*z[10]*z[2]*z[4]*z[5]*z[7]*z[9] + 24379392*z[1]*z[10]*z[2]*z[4]*z[5]*z[7] - 4644864*z[1]*z[10]*z[2]*z[4]*z[5]*z[8]*z[9] + 48740352*z[1]*z[10]*z[2]*z[4]*z[5]*z[8] + 97333248*z[1]*z[10]*z[2]*z[4]*z[5]*z[9] - 767950848*z[1]*z[10]*z[2]*z[4]*z[5] + 147456*z[1]*z[10]*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] - 2322432*z[1]*z[10]*z[2]*z[4]*z[6]*z[7]*z[8] - 4644864*z[1]*z[10]*z[2]*z[4]*z[6]*z[7]*z[9] + 48758784*z[1]*z[10]*z[2]*z[4]*z[6]*z[7] - 9289728*z[1]*z[10]*z[2]*z[4]*z[6]*z[8]*z[9] + 97480704*z[1]*z[10]*z[2]*z[4]*z[6]*z[8] + 194666496*z[1]*z[10]*z[2]*z[4]*z[6]*z[9] - 1535901696*z[1]*z[10]*z[2]*z[4]*z[6] - 290304*z[1]*z[10]*z[2]*z[4]*z[7]*z[8]*z[9] + 4572288*z[1]*z[10]*z[2]*z[4]*z[7]*z[8] + 9144576*z[1]*z[10]*z[2]*z[4]*z[7]*z[9] - 95993856*z[1]*z[10]*z[2]*z[4]*z[7] + 18289152*z[1]*z[10]*z[2]*z[4]*z[8]*z[9] - 191915136*z[1]*z[10]*z[2]*z[4]*z[8] - 383249664*z[1]*z[10]*z[2]*z[4]*z[9] + 3023806464*z[1]*z[10]*z[2]*z[4] + 294912*z[1]*z[10]*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] - 4644864*z[1]*z[10]*z[2]*z[5]*z[6]*z[7]*z[8] - 9289728*z[1]*z[10]*z[2]*z[5]*z[6]*z[7]*z[9] + 97517568*z[1]*z[10]*z[2]*z[5]*z[6]*z[7] - 18579456*z[1]*z[10]*z[2]*z[5]*z[6]*z[8]*z[9] + 194961408*z[1]*z[10]*z[2]*z[5]*z[6]*z[8] + 389332992*z[1]*z[10]*z[2]*z[5]*z[6]*z[9] - 3071803392*z[1]*z[10]*z[2]*z[5]*z[6] - 580608*z[1]*z[10]*z[2]*z[5]*z[7]*z[8]*z[9] + 9144576*z[1]*z[10]*z[2]*z[5]*z[7]*z[8] + 18289152*z[1]*z[10]*z[2]*z[5]*z[7]*z[9] - 191987712*z[1]*z[10]*z[2]*z[5]*z[7] + 36578304*z[1]*z[10]*z[2]*z[5]*z[8]*z[9] - 383830272*z[1]*z[10]*z[2]*z[5]*z[8] - 766499328*z[1]*z[10]*z[2]*z[5]*z[9] + 6047612928*z[1]*z[10]*z[2]*z[5] - 1161216*z[1]*z[10]*z[2]*z[6]*z[7]*z[8]*z[9] + 18289152*z[1]*z[10]*z[2]*z[6]*z[7]*z[8] + 36578304*z[1]*z[10]*z[2]*z[6]*z[7]*z[9] - 383975424*z[1]*z[10]*z[2]*z[6]*z[7] + 73156608*z[1]*z[10]*z[2]*z[6]*z[8]*z[9] - 767660544*z[1]*z[10]*z[2]*z[6]*z[8] - 1532998656*z[1]*z[10]*z[2]*z[6]*z[9] + 12095225856*z[1]*z[10]*z[2]*z[6] + 1535232*z[1]*z[10]*z[2]*z[7]*z[8]*z[9] - 24179904*z[1]*z[10]*z[2]*z[7]*z[8] - 48359808*z[1]*z[10]*z[2]*z[7]*z[9] + 507650048*z[1]*z[10]*z[2]*z[7] - 96719616*z[1]*z[10]*z[2]*z[8]*z[9] + 1014916288*z[1]*z[10]*z[2]*z[8] + 2026762112*z[1]*z[10]*z[2]*z[9] - 15990976512*z[1]*z[10]*z[2] + 147456*z[1]*z[10]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 2322432*z[1]*z[10]*z[3]*z[4]*z[5]*z[7]*z[8] - 4644864*z[1]*z[10]*z[3]*z[4]*z[5]*z[7]*z[9] + 48758784*z[1]*z[10]*z[3]*z[4]*z[5]*z[7] - 9289728*z[1]*z[10]*z[3]*z[4]*z[5]*z[8]*z[9] + 97480704*z[1]*z[10]*z[3]*z[4]*z[5]*z[8] + 194666496*z[1]*z[10]*z[3]*z[4]*z[5]*z[9] - 1535901696*z[1]*z[10]*z[3]*z[4]*z[5] + 294912*z[1]*z[10]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 4644864*z[1]*z[10]*z[3]*z[4]*z[6]*z[7]*z[8] - 9289728*z[1]*z[10]*z[3]*z[4]*z[6]*z[7]*z[9] + 97517568*z[1]*z[10]*z[3]*z[4]*z[6]*z[7] - 18579456*z[1]*z[10]*z[3]*z[4]*z[6]*z[8]*z[9] + 194961408*z[1]*z[10]*z[3]*z[4]*z[6]*z[8] + 389332992*z[1]*z[10]*z[3]*z[4]*z[6]*z[9] - 3071803392*z[1]*z[10]*z[3]*z[4]*z[6] - 580608*z[1]*z[10]*z[3]*z[4]*z[7]*z[8]*z[9] + 9144576*z[1]*z[10]*z[3]*z[4]*z[7]*z[8] + 18289152*z[1]*z[10]*z[3]*z[4]*z[7]*z[9] - 191987712*z[1]*z[10]*z[3]*z[4]*z[7] + 36578304*z[1]*z[10]*z[3]*z[4]*z[8]*z[9] - 383830272*z[1]*z[10]*z[3]*z[4]*z[8] - 766499328*z[1]*z[10]*z[3]*z[4]*z[9] + 6047612928*z[1]*z[10]*z[3]*z[4] + 589824*z[1]*z[10]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 9289728*z[1]*z[10]*z[3]*z[5]*z[6]*z[7]*z[8] - 18579456*z[1]*z[10]*z[3]*z[5]*z[6]*z[7]*z[9] + 195035136*z[1]*z[10]*z[3]*z[5]*z[6]*z[7] - 37158912*z[1]*z[10]*z[3]*z[5]*z[6]*z[8]*z[9] + 389922816*z[1]*z[10]*z[3]*z[5]*z[6]*z[8] + 778665984*z[1]*z[10]*z[3]*z[5]*z[6]*z[9] - 6143606784*z[1]*z[10]*z[3]*z[5]*z[6] - 1161216*z[1]*z[10]*z[3]*z[5]*z[7]*z[8]*z[9] + 18289152*z[1]*z[10]*z[3]*z[5]*z[7]*z[8] + 36578304*z[1]*z[10]*z[3]*z[5]*z[7]*z[9] - 383975424*z[1]*z[10]*z[3]*z[5]*z[7] + 73156608*z[1]*z[10]*z[3]*z[5]*z[8]*z[9] - 767660544*z[1]*z[10]*z[3]*z[5]*z[8] - 1532998656*z[1]*z[10]*z[3]*z[5]*z[9] + 12095225856*z[1]*z[10]*z[3]*z[5] - 2322432*z[1]*z[10]*z[3]*z[6]*z[7]*z[8]*z[9] + 36578304*z[1]*z[10]*z[3]*z[6]*z[7]*z[8] + 73156608*z[1]*z[10]*z[3]*z[6]*z[7]*z[9] - 767950848*z[1]*z[10]*z[3]*z[6]*z[7] + 146313216*z[1]*z[10]*z[3]*z[6]*z[8]*z[9] - 1535321088*z[1]*z[10]*z[3]*z[6]*z[8] - 3065997312*z[1]*z[10]*z[3]*z[6]*z[9] + 24190451712*z[1]*z[10]*z[3]*z[6] + 3065856*z[1]*z[10]*z[3]*z[7]*z[8]*z[9] - 48287232*z[1]*z[10]*z[3]*z[7]*z[8] - 96574464*z[1]*z[10]*z[3]*z[7]*z[9] + 1013776384*z[1]*z[10]*z[3]*z[7] - 193148928*z[1]*z[10]*z[3]*z[8]*z[9] + 2026786304*z[1]*z[10]*z[3]*z[8] + 4047440896*z[1]*z[10]*z[3]*z[9] - 31933956096*z[1]*z[10]*z[3] + 1179648*z[1]*z[10]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 18579456*z[1]*z[10]*z[4]*z[5]*z[6]*z[7]*z[8] - 37158912*z[1]*z[10]*z[4]*z[5]*z[6]*z[7]*z[9] + 390070272*z[1]*z[10]*z[4]*z[5]*z[6]*z[7] - 74317824*z[1]*z[10]*z[4]*z[5]*z[6]*z[8]*z[9] + 779845632*z[1]*z[10]*z[4]*z[5]*z[6]*z[8] + 1557331968*z[1]*z[10]*z[4]*z[5]*z[6]*z[9] - 12287213568*z[1]*z[10]*z[4]*z[5]*z[6] - 2322432*z[1]*z[10]*z[4]*z[5]*z[7]*z[8]*z[9] + 36578304*z[1]*z[10]*z[4]*z[5]*z[7]*z[8] + 73156608*z[1]*z[10]*z[4]*z[5]*z[7]*z[9] - 767950848*z[1]*z[10]*z[4]*z[5]*z[7] + 146313216*z[1]*z[10]*z[4]*z[5]*z[8]*z[9] - 1535321088*z[1]*z[10]*z[4]*z[5]*z[8] - 3065997312*z[1]*z[10]*z[4]*z[5]*z[9] + 24190451712*z[1]*z[10]*z[4]*z[5] - 4644864*z[1]*z[10]*z[4]*z[6]*z[7]*z[8]*z[9] + 73156608*z[1]*z[10]*z[4]*z[6]*z[7]*z[8] + 146313216*z[1]*z[10]*z[4]*z[6]*z[7]*z[9] - 1535901696*z[1]*z[10]*z[4]*z[6]*z[7] + 292626432*z[1]*z[10]*z[4]*z[6]*z[8]*z[9] - 3070642176*z[1]*z[10]*z[4]*z[6]*z[8] - 6131994624*z[1]*z[10]*z[4]*z[6]*z[9] + 48380903424*z[1]*z[10]*z[4]*z[6] + 6094848*z[1]*z[10]*z[4]*z[7]*z[8]*z[9] - 95993856*z[1]*z[10]*z[4]*z[7]*z[8] - 191987712*z[1]*z[10]*z[4]*z[7]*z[9] + 2015363072*z[1]*z[10]*z[4]*z[7] - 383975424*z[1]*z[10]*z[4]*z[8]*z[9] + 4029202432*z[1]*z[10]*z[4]*z[8] + 8046215168*z[1]*z[10]*z[4]*z[9] - 63483936768*z[1]*z[10]*z[4] - 9289728*z[1]*z[10]*z[5]*z[6]*z[7]*z[8]*z[9] + 146313216*z[1]*z[10]*z[5]*z[6]*z[7]*z[8] + 292626432*z[1]*z[10]*z[5]*z[6]*z[7]*z[9] - 3071803392*z[1]*z[10]*z[5]*z[6]*z[7] + 585252864*z[1]*z[10]*z[5]*z[6]*z[8]*z[9] - 6141284352*z[1]*z[10]*z[5]*z[6]*z[8] - 12263989248*z[1]*z[10]*z[5]*z[6]*z[9] + 96761806848*z[1]*z[10]*z[5]*z[6] + 11894784*z[1]*z[10]*z[5]*z[7]*z[8]*z[9] - 187342848*z[1]*z[10]*z[5]*z[7]*z[8] - 374685696*z[1]*z[10]*z[5]*z[7]*z[9] + 3933208576*z[1]*z[10]*z[5]*z[7] - 749371392*z[1]*z[10]*z[5]*z[8]*z[9] + 7863443456*z[1]*z[10]*z[5]*z[8] + 15703097344*z[1]*z[10]*z[5]*z[9] - 123896070144*z[1]*z[10]*z[5] + 21430272*z[1]*z[10]*z[6]*z[7]*z[8]*z[9] - 337526784*z[1]*z[10]*z[6]*z[7]*z[8] - 675053568*z[1]*z[10]*z[6]*z[7]*z[9] + 7086276608*z[1]*z[10]*z[6]*z[7] - 1350107136*z[1]*z[10]*z[6]*z[8]*z[9] + 14167195648*z[1]*z[10]*z[6]*z[8] + 28291530752*z[1]*z[10]*z[6]*z[9] - 223217713152*z[1]*z[10]*z[6] - 24379488*z[1]*z[10]*z[7]*z[8]*z[9] + 383976936*z[1]*z[10]*z[7]*z[8] + 767953872*z[1]*z[10]*z[7]*z[9] - 8061484032*z[1]*z[10]*z[7] + 1535907744*z[1]*z[10]*z[8]*z[9] - 16116873192*z[1]*z[10]*z[8] - 32184987408*z[1]*z[10]*z[9] + 253936747008*z[1]*z[10] + 294912*z[1]*z[11]*z[12]*z[2]*z[3]*z[4]*z[7]*z[8] + 589824*z[1]*z[11]*z[12]*z[2]*z[3]*z[4]*z[7]*z[9] - 9289728*z[1]*z[11]*z[12]*z[2]*z[3]*z[4]*z[7] + 1179648*z[1]*z[11]*z[12]*z[2]*z[3]*z[4]*z[8]*z[9] - 18579456*z[1]*z[11]*z[12]*z[2]*z[3]*z[4]*z[8] - 37158912*z[1]*z[11]*z[12]*z[2]*z[3]*z[4]*z[9] + 330350592*z[1]*z[11]*z[12]*z[2]*z[3]*z[4] + 589824*z[1]*z[11]*z[12]*z[2]*z[3]*z[5]*z[7]*z[8] + 1179648*z[1]*z[11]*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] - 18579456*z[1]*z[11]*z[12]*z[2]*z[3]*z[5]*z[7] + 2359296*z[1]*z[11]*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] - 37158912*z[1]*z[11]*z[12]*z[2]*z[3]*z[5]*z[8] - 74317824*z[1]*z[11]*z[12]*z[2]*z[3]*z[5]*z[9] + 660701184*z[1]*z[11]*z[12]*z[2]*z[3]*z[5] + 1179648*z[1]*z[11]*z[12]*z[2]*z[3]*z[6]*z[7]*z[8] + 2359296*z[1]*z[11]*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] - 37158912*z[1]*z[11]*z[12]*z[2]*z[3]*z[6]*z[7] + 4718592*z[1]*z[11]*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] - 74317824*z[1]*z[11]*z[12]*z[2]*z[3]*z[6]*z[8] - 148635648*z[1]*z[11]*z[12]*z[2]*z[3]*z[6]*z[9] + 1321402368*z[1]*z[11]*z[12]*z[2]*z[3]*z[6] - 2322432*z[1]*z[11]*z[12]*z[2]*z[3]*z[7]*z[8] - 4644864*z[1]*z[11]*z[12]*z[2]*z[3]*z[7]*z[9] + 73156608*z[1]*z[11]*z[12]*z[2]*z[3]*z[7] - 9289728*z[1]*z[11]*z[12]*z[2]*z[3]*z[8]*z[9] + 146313216*z[1]*z[11]*z[12]*z[2]*z[3]*z[8] + 292626432*z[1]*z[11]*z[12]*z[2]*z[3]*z[9] - 2601510912*z[1]*z[11]*z[12]*z[2]*z[3] + 1179648*z[1]*z[11]*z[12]*z[2]*z[4]*z[5]*z[7]*z[8] + 2359296*z[1]*z[11]*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] - 37158912*z[1]*z[11]*z[12]*z[2]*z[4]*z[5]*z[7] + 4718592*z[1]*z[11]*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] - 74317824*z[1]*z[11]*z[12]*z[2]*z[4]*z[5]*z[8] - 148635648*z[1]*z[11]*z[12]*z[2]*z[4]*z[5]*z[9] + 1321402368*z[1]*z[11]*z[12]*z[2]*z[4]*z[5] + 2359296*z[1]*z[11]*z[12]*z[2]*z[4]*z[6]*z[7]*z[8] + 4718592*z[1]*z[11]*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] - 74317824*z[1]*z[11]*z[12]*z[2]*z[4]*z[6]*z[7] + 9437184*z[1]*z[11]*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] - 148635648*z[1]*z[11]*z[12]*z[2]*z[4]*z[6]*z[8] - 297271296*z[1]*z[11]*z[12]*z[2]*z[4]*z[6]*z[9] + 2642804736*z[1]*z[11]*z[12]*z[2]*z[4]*z[6] - 4644864*z[1]*z[11]*z[12]*z[2]*z[4]*z[7]*z[8] - 9289728*z[1]*z[11]*z[12]*z[2]*z[4]*z[7]*z[9] + 146313216*z[1]*z[11]*z[12]*z[2]*z[4]*z[7] - 18579456*z[1]*z[11]*z[12]*z[2]*z[4]*z[8]*z[9] + 292626432*z[1]*z[11]*z[12]*z[2]*z[4]*z[8] + 585252864*z[1]*z[11]*z[12]*z[2]*z[4]*z[9] - 5203021824*z[1]*z[11]*z[12]*z[2]*z[4] + 4718592*z[1]*z[11]*z[12]*z[2]*z[5]*z[6]*z[7]*z[8] + 9437184*z[1]*z[11]*z[12]*z[2]*z[5]*z[6]*z[7]*z[9] - 148635648*z[1]*z[11]*z[12]*z[2]*z[5]*z[6]*z[7] + 18874368*z[1]*z[11]*z[12]*z[2]*z[5]*z[6]*z[8]*z[9] - 297271296*z[1]*z[11]*z[12]*z[2]*z[5]*z[6]*z[8] - 594542592*z[1]*z[11]*z[12]*z[2]*z[5]*z[6]*z[9] + 5285609472*z[1]*z[11]*z[12]*z[2]*z[5]*z[6] - 9289728*z[1]*z[11]*z[12]*z[2]*z[5]*z[7]*z[8] - 18579456*z[1]*z[11]*z[12]*z[2]*z[5]*z[7]*z[9] + 292626432*z[1]*z[11]*z[12]*z[2]*z[5]*z[7] - 37158912*z[1]*z[11]*z[12]*z[2]*z[5]*z[8]*z[9] + 585252864*z[1]*z[11]*z[12]*z[2]*z[5]*z[8] + 1170505728*z[1]*z[11]*z[12]*z[2]*z[5]*z[9] - 10406043648*z[1]*z[11]*z[12]*z[2]*z[5] - 18579456*z[1]*z[11]*z[12]*z[2]*z[6]*z[7]*z[8] - 37158912*z[1]*z[11]*z[12]*z[2]*z[6]*z[7]*z[9] + 585252864*z[1]*z[11]*z[12]*z[2]*z[6]*z[7] - 74317824*z[1]*z[11]*z[12]*z[2]*z[6]*z[8]*z[9] + 1170505728*z[1]*z[11]*z[12]*z[2]*z[6]*z[8] + 2341011456*z[1]*z[11]*z[12]*z[2]*z[6]*z[9] - 20812087296*z[1]*z[11]*z[12]*z[2]*z[6] + 24563712*z[1]*z[11]*z[12]*z[2]*z[7]*z[8] + 49127424*z[1]*z[11]*z[12]*z[2]*z[7]*z[9] - 773756928*z[1]*z[11]*z[12]*z[2]*z[7] + 98254848*z[1]*z[11]*z[12]*z[2]*z[8]*z[9] - 1547513856*z[1]*z[11]*z[12]*z[2]*z[8] - 3095027712*z[1]*z[11]*z[12]*z[2]*z[9] + 27515451392*z[1]*z[11]*z[12]*z[2] + 2359296*z[1]*z[11]*z[12]*z[3]*z[4]*z[5]*z[7]*z[8] + 4718592*z[1]*z[11]*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] - 74317824*z[1]*z[11]*z[12]*z[3]*z[4]*z[5]*z[7] + 9437184*z[1]*z[11]*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] - 148635648*z[1]*z[11]*z[12]*z[3]*z[4]*z[5]*z[8] - 297271296*z[1]*z[11]*z[12]*z[3]*z[4]*z[5]*z[9] + 2642804736*z[1]*z[11]*z[12]*z[3]*z[4]*z[5] + 4718592*z[1]*z[11]*z[12]*z[3]*z[4]*z[6]*z[7]*z[8] + 9437184*z[1]*z[11]*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] - 148635648*z[1]*z[11]*z[12]*z[3]*z[4]*z[6]*z[7] + 18874368*z[1]*z[11]*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] - 297271296*z[1]*z[11]*z[12]*z[3]*z[4]*z[6]*z[8] - 594542592*z[1]*z[11]*z[12]*z[3]*z[4]*z[6]*z[9] + 5285609472*z[1]*z[11]*z[12]*z[3]*z[4]*z[6] - 9289728*z[1]*z[11]*z[12]*z[3]*z[4]*z[7]*z[8] - 18579456*z[1]*z[11]*z[12]*z[3]*z[4]*z[7]*z[9] + 292626432*z[1]*z[11]*z[12]*z[3]*z[4]*z[7] - 37158912*z[1]*z[11]*z[12]*z[3]*z[4]*z[8]*z[9] + 585252864*z[1]*z[11]*z[12]*z[3]*z[4]*z[8] + 1170505728*z[1]*z[11]*z[12]*z[3]*z[4]*z[9] - 10406043648*z[1]*z[11]*z[12]*z[3]*z[4] + 9437184*z[1]*z[11]*z[12]*z[3]*z[5]*z[6]*z[7]*z[8] + 18874368*z[1]*z[11]*z[12]*z[3]*z[5]*z[6]*z[7]*z[9] - 297271296*z[1]*z[11]*z[12]*z[3]*z[5]*z[6]*z[7] + 37748736*z[1]*z[11]*z[12]*z[3]*z[5]*z[6]*z[8]*z[9] - 594542592*z[1]*z[11]*z[12]*z[3]*z[5]*z[6]*z[8] - 1189085184*z[1]*z[11]*z[12]*z[3]*z[5]*z[6]*z[9] + 10571218944*z[1]*z[11]*z[12]*z[3]*z[5]*z[6] - 18579456*z[1]*z[11]*z[12]*z[3]*z[5]*z[7]*z[8] - 37158912*z[1]*z[11]*z[12]*z[3]*z[5]*z[7]*z[9] + 585252864*z[1]*z[11]*z[12]*z[3]*z[5]*z[7] - 74317824*z[1]*z[11]*z[12]*z[3]*z[5]*z[8]*z[9] + 1170505728*z[1]*z[11]*z[12]*z[3]*z[5]*z[8] + 2341011456*z[1]*z[11]*z[12]*z[3]*z[5]*z[9] - 20812087296*z[1]*z[11]*z[12]*z[3]*z[5] - 37158912*z[1]*z[11]*z[12]*z[3]*z[6]*z[7]*z[8] - 74317824*z[1]*z[11]*z[12]*z[3]*z[6]*z[7]*z[9] + 1170505728*z[1]*z[11]*z[12]*z[3]*z[6]*z[7] - 148635648*z[1]*z[11]*z[12]*z[3]*z[6]*z[8]*z[9] + 2341011456*z[1]*z[11]*z[12]*z[3]*z[6]*z[8] + 4682022912*z[1]*z[11]*z[12]*z[3]*z[6]*z[9] - 41624174592*z[1]*z[11]*z[12]*z[3]*z[6] + 49053696*z[1]*z[11]*z[12]*z[3]*z[7]*z[8] + 98107392*z[1]*z[11]*z[12]*z[3]*z[7]*z[9] - 1545191424*z[1]*z[11]*z[12]*z[3]*z[7] + 196214784*z[1]*z[11]*z[12]*z[3]*z[8]*z[9] - 3090382848*z[1]*z[11]*z[12]*z[3]*z[8] - 6180765696*z[1]*z[11]*z[12]*z[3]*z[9] + 54948315136*z[1]*z[11]*z[12]*z[3] + 18874368*z[1]*z[11]*z[12]*z[4]*z[5]*z[6]*z[7]*z[8] + 37748736*z[1]*z[11]*z[12]*z[4]*z[5]*z[6]*z[7]*z[9] - 594542592*z[1]*z[11]*z[12]*z[4]*z[5]*z[6]*z[7] + 75497472*z[1]*z[11]*z[12]*z[4]*z[5]*z[6]*z[8]*z[9] - 1189085184*z[1]*z[11]*z[12]*z[4]*z[5]*z[6]*z[8] - 2378170368*z[1]*z[11]*z[12]*z[4]*z[5]*z[6]*z[9] + 21142437888*z[1]*z[11]*z[12]*z[4]*z[5]*z[6] - 37158912*z[1]*z[11]*z[12]*z[4]*z[5]*z[7]*z[8] - 74317824*z[1]*z[11]*z[12]*z[4]*z[5]*z[7]*z[9] + 1170505728*z[1]*z[11]*z[12]*z[4]*z[5]*z[7] - 148635648*z[1]*z[11]*z[12]*z[4]*z[5]*z[8]*z[9] + 2341011456*z[1]*z[11]*z[12]*z[4]*z[5]*z[8] + 4682022912*z[1]*z[11]*z[12]*z[4]*z[5]*z[9] - 41624174592*z[1]*z[11]*z[12]*z[4]*z[5] - 74317824*z[1]*z[11]*z[12]*z[4]*z[6]*z[7]*z[8] - 148635648*z[1]*z[11]*z[12]*z[4]*z[6]*z[7]*z[9] + 2341011456*z[1]*z[11]*z[12]*z[4]*z[6]*z[7] - 297271296*z[1]*z[11]*z[12]*z[4]*z[6]*z[8]*z[9] + 4682022912*z[1]*z[11]*z[12]*z[4]*z[6]*z[8] + 9364045824*z[1]*z[11]*z[12]*z[4]*z[6]*z[9] - 83248349184*z[1]*z[11]*z[12]*z[4]*z[6] + 97517568*z[1]*z[11]*z[12]*z[4]*z[7]*z[8] + 195035136*z[1]*z[11]*z[12]*z[4]*z[7]*z[9] - 3071803392*z[1]*z[11]*z[12]*z[4]*z[7] + 390070272*z[1]*z[11]*z[12]*z[4]*z[8]*z[9] - 6143606784*z[1]*z[11]*z[12]*z[4]*z[8] - 12287213568*z[1]*z[11]*z[12]*z[4]*z[9] + 109235929088*z[1]*z[11]*z[12]*z[4] - 148635648*z[1]*z[11]*z[12]*z[5]*z[6]*z[7]*z[8] - 297271296*z[1]*z[11]*z[12]*z[5]*z[6]*z[7]*z[9] + 4682022912*z[1]*z[11]*z[12]*z[5]*z[6]*z[7] - 594542592*z[1]*z[11]*z[12]*z[5]*z[6]*z[8]*z[9] + 9364045824*z[1]*z[11]*z[12]*z[5]*z[6]*z[8] + 18728091648*z[1]*z[11]*z[12]*z[5]*z[6]*z[9] - 166496698368*z[1]*z[11]*z[12]*z[5]*z[6] + 190316544*z[1]*z[11]*z[12]*z[5]*z[7]*z[8] + 380633088*z[1]*z[11]*z[12]*z[5]*z[7]*z[9] - 5994971136*z[1]*z[11]*z[12]*z[5]*z[7] + 761266176*z[1]*z[11]*z[12]*z[5]*z[8]*z[9] - 11989942272*z[1]*z[11]*z[12]*z[5]*z[8] - 23979884544*z[1]*z[11]*z[12]*z[5]*z[9] + 213186248704*z[1]*z[11]*z[12]*z[5] + 342884352*z[1]*z[11]*z[12]*z[6]*z[7]*z[8] + 685768704*z[1]*z[11]*z[12]*z[6]*z[7]*z[9] - 10800857088*z[1]*z[11]*z[12]*z[6]*z[7] + 1371537408*z[1]*z[11]*z[12]*z[6]*z[8]*z[9] - 21601714176*z[1]*z[11]*z[12]*z[6]*z[8] - 43203428352*z[1]*z[11]*z[12]*z[6]*z[9] + 384087621632*z[1]*z[11]*z[12]*z[6] - 390071808*z[1]*z[11]*z[12]*z[7]*z[8] - 780143616*z[1]*z[11]*z[12]*z[7]*z[9] + 12287261952*z[1]*z[11]*z[12]*z[7] - 1560287232*z[1]*z[11]*z[12]*z[8]*z[9] + 24574523904*z[1]*z[11]*z[12]*z[8] + 49149047808*z[1]*z[11]*z[12]*z[9] - 436945436928*z[1]*z[11]*z[12] + 36864*z[1]*z[11]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] - 580608*z[1]*z[11]*z[2]*z[3]*z[4]*z[7]*z[8] - 1161216*z[1]*z[11]*z[2]*z[3]*z[4]*z[7]*z[9] + 11894784*z[1]*z[11]*z[2]*z[3]*z[4]*z[7] - 2322432*z[1]*z[11]*z[2]*z[3]*z[4]*z[8]*z[9] + 23780352*z[1]*z[11]*z[2]*z[3]*z[4]*z[8] + 47486976*z[1]*z[11]*z[2]*z[3]*z[4]*z[9] - 365395968*z[1]*z[11]*z[2]*z[3]*z[4] + 73728*z[1]*z[11]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] - 1161216*z[1]*z[11]*z[2]*z[3]*z[5]*z[7]*z[8] - 2322432*z[1]*z[11]*z[2]*z[3]*z[5]*z[7]*z[9] + 23789568*z[1]*z[11]*z[2]*z[3]*z[5]*z[7] - 4644864*z[1]*z[11]*z[2]*z[3]*z[5]*z[8]*z[9] + 47560704*z[1]*z[11]*z[2]*z[3]*z[5]*z[8] + 94973952*z[1]*z[11]*z[2]*z[3]*z[5]*z[9] - 730791936*z[1]*z[11]*z[2]*z[3]*z[5] + 147456*z[1]*z[11]*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] - 2322432*z[1]*z[11]*z[2]*z[3]*z[6]*z[7]*z[8] - 4644864*z[1]*z[11]*z[2]*z[3]*z[6]*z[7]*z[9] + 47579136*z[1]*z[11]*z[2]*z[3]*z[6]*z[7] - 9289728*z[1]*z[11]*z[2]*z[3]*z[6]*z[8]*z[9] + 95121408*z[1]*z[11]*z[2]*z[3]*z[6]*z[8] + 189947904*z[1]*z[11]*z[2]*z[3]*z[6]*z[9] - 1461583872*z[1]*z[11]*z[2]*z[3]*z[6] - 290304*z[1]*z[11]*z[2]*z[3]*z[7]*z[8]*z[9] + 4572288*z[1]*z[11]*z[2]*z[3]*z[7]*z[8] + 9144576*z[1]*z[11]*z[2]*z[3]*z[7]*z[9] - 93671424*z[1]*z[11]*z[2]*z[3]*z[7] + 18289152*z[1]*z[11]*z[2]*z[3]*z[8]*z[9] - 187270272*z[1]*z[11]*z[2]*z[3]*z[8] - 373959936*z[1]*z[11]*z[2]*z[3]*z[9] + 2877493248*z[1]*z[11]*z[2]*z[3] + 147456*z[1]*z[11]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] - 2322432*z[1]*z[11]*z[2]*z[4]*z[5]*z[7]*z[8] - 4644864*z[1]*z[11]*z[2]*z[4]*z[5]*z[7]*z[9] + 47579136*z[1]*z[11]*z[2]*z[4]*z[5]*z[7] - 9289728*z[1]*z[11]*z[2]*z[4]*z[5]*z[8]*z[9] + 95121408*z[1]*z[11]*z[2]*z[4]*z[5]*z[8] + 189947904*z[1]*z[11]*z[2]*z[4]*z[5]*z[9] - 1461583872*z[1]*z[11]*z[2]*z[4]*z[5] + 294912*z[1]*z[11]*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] - 4644864*z[1]*z[11]*z[2]*z[4]*z[6]*z[7]*z[8] - 9289728*z[1]*z[11]*z[2]*z[4]*z[6]*z[7]*z[9] + 95158272*z[1]*z[11]*z[2]*z[4]*z[6]*z[7] - 18579456*z[1]*z[11]*z[2]*z[4]*z[6]*z[8]*z[9] + 190242816*z[1]*z[11]*z[2]*z[4]*z[6]*z[8] + 379895808*z[1]*z[11]*z[2]*z[4]*z[6]*z[9] - 2923167744*z[1]*z[11]*z[2]*z[4]*z[6] - 580608*z[1]*z[11]*z[2]*z[4]*z[7]*z[8]*z[9] + 9144576*z[1]*z[11]*z[2]*z[4]*z[7]*z[8] + 18289152*z[1]*z[11]*z[2]*z[4]*z[7]*z[9] - 187342848*z[1]*z[11]*z[2]*z[4]*z[7] + 36578304*z[1]*z[11]*z[2]*z[4]*z[8]*z[9] - 374540544*z[1]*z[11]*z[2]*z[4]*z[8] - 747919872*z[1]*z[11]*z[2]*z[4]*z[9] + 5754986496*z[1]*z[11]*z[2]*z[4] + 589824*z[1]*z[11]*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] - 9289728*z[1]*z[11]*z[2]*z[5]*z[6]*z[7]*z[8] - 18579456*z[1]*z[11]*z[2]*z[5]*z[6]*z[7]*z[9] + 190316544*z[1]*z[11]*z[2]*z[5]*z[6]*z[7] - 37158912*z[1]*z[11]*z[2]*z[5]*z[6]*z[8]*z[9] + 380485632*z[1]*z[11]*z[2]*z[5]*z[6]*z[8] + 759791616*z[1]*z[11]*z[2]*z[5]*z[6]*z[9] - 5846335488*z[1]*z[11]*z[2]*z[5]*z[6] - 1161216*z[1]*z[11]*z[2]*z[5]*z[7]*z[8]*z[9] + 18289152*z[1]*z[11]*z[2]*z[5]*z[7]*z[8] + 36578304*z[1]*z[11]*z[2]*z[5]*z[7]*z[9] - 374685696*z[1]*z[11]*z[2]*z[5]*z[7] + 73156608*z[1]*z[11]*z[2]*z[5]*z[8]*z[9] - 749081088*z[1]*z[11]*z[2]*z[5]*z[8] - 1495839744*z[1]*z[11]*z[2]*z[5]*z[9] + 11509972992*z[1]*z[11]*z[2]*z[5] - 2322432*z[1]*z[11]*z[2]*z[6]*z[7]*z[8]*z[9] + 36578304*z[1]*z[11]*z[2]*z[6]*z[7]*z[8] + 73156608*z[1]*z[11]*z[2]*z[6]*z[7]*z[9] - 749371392*z[1]*z[11]*z[2]*z[6]*z[7] + 146313216*z[1]*z[11]*z[2]*z[6]*z[8]*z[9] - 1498162176*z[1]*z[11]*z[2]*z[6]*z[8] - 2991679488*z[1]*z[11]*z[2]*z[6]*z[9] + 23019945984*z[1]*z[11]*z[2]*z[6] + 3070464*z[1]*z[11]*z[2]*z[7]*z[8]*z[9] - 48359808*z[1]*z[11]*z[2]*z[7]*z[8] - 96719616*z[1]*z[11]*z[2]*z[7]*z[9] + 990736384*z[1]*z[11]*z[2]*z[7] - 193439232*z[1]*z[11]*z[2]*z[8]*z[9] + 1980705152*z[1]*z[11]*z[2]*z[8] + 3955269376*z[1]*z[11]*z[2]*z[9] - 30434439168*z[1]*z[11]*z[2] + 294912*z[1]*z[11]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 4644864*z[1]*z[11]*z[3]*z[4]*z[5]*z[7]*z[8] - 9289728*z[1]*z[11]*z[3]*z[4]*z[5]*z[7]*z[9] + 95158272*z[1]*z[11]*z[3]*z[4]*z[5]*z[7] - 18579456*z[1]*z[11]*z[3]*z[4]*z[5]*z[8]*z[9] + 190242816*z[1]*z[11]*z[3]*z[4]*z[5]*z[8] + 379895808*z[1]*z[11]*z[3]*z[4]*z[5]*z[9] - 2923167744*z[1]*z[11]*z[3]*z[4]*z[5] + 589824*z[1]*z[11]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 9289728*z[1]*z[11]*z[3]*z[4]*z[6]*z[7]*z[8] - 18579456*z[1]*z[11]*z[3]*z[4]*z[6]*z[7]*z[9] + 190316544*z[1]*z[11]*z[3]*z[4]*z[6]*z[7] - 37158912*z[1]*z[11]*z[3]*z[4]*z[6]*z[8]*z[9] + 380485632*z[1]*z[11]*z[3]*z[4]*z[6]*z[8] + 759791616*z[1]*z[11]*z[3]*z[4]*z[6]*z[9] - 5846335488*z[1]*z[11]*z[3]*z[4]*z[6] - 1161216*z[1]*z[11]*z[3]*z[4]*z[7]*z[8]*z[9] + 18289152*z[1]*z[11]*z[3]*z[4]*z[7]*z[8] + 36578304*z[1]*z[11]*z[3]*z[4]*z[7]*z[9] - 374685696*z[1]*z[11]*z[3]*z[4]*z[7] + 73156608*z[1]*z[11]*z[3]*z[4]*z[8]*z[9] - 749081088*z[1]*z[11]*z[3]*z[4]*z[8] - 1495839744*z[1]*z[11]*z[3]*z[4]*z[9] + 11509972992*z[1]*z[11]*z[3]*z[4] + 1179648*z[1]*z[11]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 18579456*z[1]*z[11]*z[3]*z[5]*z[6]*z[7]*z[8] - 37158912*z[1]*z[11]*z[3]*z[5]*z[6]*z[7]*z[9] + 380633088*z[1]*z[11]*z[3]*z[5]*z[6]*z[7] - 74317824*z[1]*z[11]*z[3]*z[5]*z[6]*z[8]*z[9] + 760971264*z[1]*z[11]*z[3]*z[5]*z[6]*z[8] + 1519583232*z[1]*z[11]*z[3]*z[5]*z[6]*z[9] - 11692670976*z[1]*z[11]*z[3]*z[5]*z[6] - 2322432*z[1]*z[11]*z[3]*z[5]*z[7]*z[8]*z[9] + 36578304*z[1]*z[11]*z[3]*z[5]*z[7]*z[8] + 73156608*z[1]*z[11]*z[3]*z[5]*z[7]*z[9] - 749371392*z[1]*z[11]*z[3]*z[5]*z[7] + 146313216*z[1]*z[11]*z[3]*z[5]*z[8]*z[9] - 1498162176*z[1]*z[11]*z[3]*z[5]*z[8] - 2991679488*z[1]*z[11]*z[3]*z[5]*z[9] + 23019945984*z[1]*z[11]*z[3]*z[5] - 4644864*z[1]*z[11]*z[3]*z[6]*z[7]*z[8]*z[9] + 73156608*z[1]*z[11]*z[3]*z[6]*z[7]*z[8] + 146313216*z[1]*z[11]*z[3]*z[6]*z[7]*z[9] - 1498742784*z[1]*z[11]*z[3]*z[6]*z[7] + 292626432*z[1]*z[11]*z[3]*z[6]*z[8]*z[9] - 2996324352*z[1]*z[11]*z[3]*z[6]*z[8] - 5983358976*z[1]*z[11]*z[3]*z[6]*z[9] + 46039891968*z[1]*z[11]*z[3]*z[6] + 6131712*z[1]*z[11]*z[3]*z[7]*z[8]*z[9] - 96574464*z[1]*z[11]*z[3]*z[7]*z[8] - 193148928*z[1]*z[11]*z[3]*z[7]*z[9] + 1978499072*z[1]*z[11]*z[3]*z[7] - 386297856*z[1]*z[11]*z[3]*z[8]*z[9] + 3955465216*z[1]*z[11]*z[3]*z[8] + 7898667008*z[1]*z[11]*z[3]*z[9] - 60777529344*z[1]*z[11]*z[3] + 2359296*z[1]*z[11]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 37158912*z[1]*z[11]*z[4]*z[5]*z[6]*z[7]*z[8] - 74317824*z[1]*z[11]*z[4]*z[5]*z[6]*z[7]*z[9] + 761266176*z[1]*z[11]*z[4]*z[5]*z[6]*z[7] - 148635648*z[1]*z[11]*z[4]*z[5]*z[6]*z[8]*z[9] + 1521942528*z[1]*z[11]*z[4]*z[5]*z[6]*z[8] + 3039166464*z[1]*z[11]*z[4]*z[5]*z[6]*z[9] - 23385341952*z[1]*z[11]*z[4]*z[5]*z[6] - 4644864*z[1]*z[11]*z[4]*z[5]*z[7]*z[8]*z[9] + 73156608*z[1]*z[11]*z[4]*z[5]*z[7]*z[8] + 146313216*z[1]*z[11]*z[4]*z[5]*z[7]*z[9] - 1498742784*z[1]*z[11]*z[4]*z[5]*z[7] + 292626432*z[1]*z[11]*z[4]*z[5]*z[8]*z[9] - 2996324352*z[1]*z[11]*z[4]*z[5]*z[8] - 5983358976*z[1]*z[11]*z[4]*z[5]*z[9] + 46039891968*z[1]*z[11]*z[4]*z[5] - 9289728*z[1]*z[11]*z[4]*z[6]*z[7]*z[8]*z[9] + 146313216*z[1]*z[11]*z[4]*z[6]*z[7]*z[8] + 292626432*z[1]*z[11]*z[4]*z[6]*z[7]*z[9] - 2997485568*z[1]*z[11]*z[4]*z[6]*z[7] + 585252864*z[1]*z[11]*z[4]*z[6]*z[8]*z[9] - 5992648704*z[1]*z[11]*z[4]*z[6]*z[8] - 11966717952*z[1]*z[11]*z[4]*z[6]*z[9] + 92079783936*z[1]*z[11]*z[4]*z[6] + 12189696*z[1]*z[11]*z[4]*z[7]*z[8]*z[9] - 191987712*z[1]*z[11]*z[4]*z[7]*z[8] - 383975424*z[1]*z[11]*z[4]*z[7]*z[9] + 3933208576*z[1]*z[11]*z[4]*z[7] - 767950848*z[1]*z[11]*z[4]*z[8]*z[9] + 7863369728*z[1]*z[11]*z[4]*z[8] + 15702360064*z[1]*z[11]*z[4]*z[9] - 120824266752*z[1]*z[11]*z[4] - 18579456*z[1]*z[11]*z[5]*z[6]*z[7]*z[8]*z[9] + 292626432*z[1]*z[11]*z[5]*z[6]*z[7]*z[8] + 585252864*z[1]*z[11]*z[5]*z[6]*z[7]*z[9] - 5994971136*z[1]*z[11]*z[5]*z[6]*z[7] + 1170505728*z[1]*z[11]*z[5]*z[6]*z[8]*z[9] - 11985297408*z[1]*z[11]*z[5]*z[6]*z[8] - 23933435904*z[1]*z[11]*z[5]*z[6]*z[9] + 184159567872*z[1]*z[11]*z[5]*z[6] + 23789568*z[1]*z[11]*z[5]*z[7]*z[8]*z[9] - 374685696*z[1]*z[11]*z[5]*z[7]*z[8] - 749371392*z[1]*z[11]*z[5]*z[7]*z[9] + 7676100608*z[1]*z[11]*z[5]*z[7] - 1498742784*z[1]*z[11]*z[5]*z[8]*z[9] + 15346253824*z[1]*z[11]*z[5]*z[8] + 30644928512*z[1]*z[11]*z[5]*z[9] - 235802198016*z[1]*z[11]*z[5] + 42860544*z[1]*z[11]*z[6]*z[7]*z[8]*z[9] - 675053568*z[1]*z[11]*z[6]*z[7]*z[8] - 1350107136*z[1]*z[11]*z[6]*z[7]*z[9] + 13829668864*z[1]*z[11]*z[6]*z[7] - 2700214272*z[1]*z[11]*z[6]*z[8]*z[9] + 27648622592*z[1]*z[11]*z[6]*z[8] + 55211524096*z[1]*z[11]*z[6]*z[9] - 424833712128*z[1]*z[11]*z[6] - 48758976*z[1]*z[11]*z[7]*z[8]*z[9] + 767953872*z[1]*z[11]*z[7]*z[8] + 1535907744*z[1]*z[11]*z[7]*z[9] - 15732896256*z[1]*z[11]*z[7] + 3071815488*z[1]*z[11]*z[8]*z[9] - 31453602768*z[1]*z[11]*z[8] - 62809687584*z[1]*z[11]*z[9] + 483298970112*z[1]*z[11] + 73728*z[1]*z[12]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] - 1161216*z[1]*z[12]*z[2]*z[3]*z[4]*z[7]*z[8] - 2322432*z[1]*z[12]*z[2]*z[3]*z[4]*z[7]*z[9] + 21430272*z[1]*z[12]*z[2]*z[3]*z[4]*z[7] - 4644864*z[1]*z[12]*z[2]*z[3]*z[4]*z[8]*z[9] + 42842112*z[1]*z[12]*z[2]*z[3]*z[4]*z[8] + 85536768*z[1]*z[12]*z[2]*z[3]*z[4]*z[9] - 582156288*z[1]*z[12]*z[2]*z[3]*z[4] + 147456*z[1]*z[12]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] - 2322432*z[1]*z[12]*z[2]*z[3]*z[5]*z[7]*z[8] - 4644864*z[1]*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] + 42860544*z[1]*z[12]*z[2]*z[3]*z[5]*z[7] - 9289728*z[1]*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] + 85684224*z[1]*z[12]*z[2]*z[3]*z[5]*z[8] + 171073536*z[1]*z[12]*z[2]*z[3]*z[5]*z[9] - 1164312576*z[1]*z[12]*z[2]*z[3]*z[5] + 294912*z[1]*z[12]*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] - 4644864*z[1]*z[12]*z[2]*z[3]*z[6]*z[7]*z[8] - 9289728*z[1]*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] + 85721088*z[1]*z[12]*z[2]*z[3]*z[6]*z[7] - 18579456*z[1]*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] + 171368448*z[1]*z[12]*z[2]*z[3]*z[6]*z[8] + 342147072*z[1]*z[12]*z[2]*z[3]*z[6]*z[9] - 2328625152*z[1]*z[12]*z[2]*z[3]*z[6] - 580608*z[1]*z[12]*z[2]*z[3]*z[7]*z[8]*z[9] + 9144576*z[1]*z[12]*z[2]*z[3]*z[7]*z[8] + 18289152*z[1]*z[12]*z[2]*z[3]*z[7]*z[9] - 168763392*z[1]*z[12]*z[2]*z[3]*z[7] + 36578304*z[1]*z[12]*z[2]*z[3]*z[8]*z[9] - 337381632*z[1]*z[12]*z[2]*z[3]*z[8] - 673602048*z[1]*z[12]*z[2]*z[3]*z[9] + 4584480768*z[1]*z[12]*z[2]*z[3] + 294912*z[1]*z[12]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] - 4644864*z[1]*z[12]*z[2]*z[4]*z[5]*z[7]*z[8] - 9289728*z[1]*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] + 85721088*z[1]*z[12]*z[2]*z[4]*z[5]*z[7] - 18579456*z[1]*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] + 171368448*z[1]*z[12]*z[2]*z[4]*z[5]*z[8] + 342147072*z[1]*z[12]*z[2]*z[4]*z[5]*z[9] - 2328625152*z[1]*z[12]*z[2]*z[4]*z[5] + 589824*z[1]*z[12]*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] - 9289728*z[1]*z[12]*z[2]*z[4]*z[6]*z[7]*z[8] - 18579456*z[1]*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] + 171442176*z[1]*z[12]*z[2]*z[4]*z[6]*z[7] - 37158912*z[1]*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] + 342736896*z[1]*z[12]*z[2]*z[4]*z[6]*z[8] + 684294144*z[1]*z[12]*z[2]*z[4]*z[6]*z[9] - 4657250304*z[1]*z[12]*z[2]*z[4]*z[6] - 1161216*z[1]*z[12]*z[2]*z[4]*z[7]*z[8]*z[9] + 18289152*z[1]*z[12]*z[2]*z[4]*z[7]*z[8] + 36578304*z[1]*z[12]*z[2]*z[4]*z[7]*z[9] - 337526784*z[1]*z[12]*z[2]*z[4]*z[7] + 73156608*z[1]*z[12]*z[2]*z[4]*z[8]*z[9] - 674763264*z[1]*z[12]*z[2]*z[4]*z[8] - 1347204096*z[1]*z[12]*z[2]*z[4]*z[9] + 9168961536*z[1]*z[12]*z[2]*z[4] + 1179648*z[1]*z[12]*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] - 18579456*z[1]*z[12]*z[2]*z[5]*z[6]*z[7]*z[8] - 37158912*z[1]*z[12]*z[2]*z[5]*z[6]*z[7]*z[9] + 342884352*z[1]*z[12]*z[2]*z[5]*z[6]*z[7] - 74317824*z[1]*z[12]*z[2]*z[5]*z[6]*z[8]*z[9] + 685473792*z[1]*z[12]*z[2]*z[5]*z[6]*z[8] + 1368588288*z[1]*z[12]*z[2]*z[5]*z[6]*z[9] - 9314500608*z[1]*z[12]*z[2]*z[5]*z[6] - 2322432*z[1]*z[12]*z[2]*z[5]*z[7]*z[8]*z[9] + 36578304*z[1]*z[12]*z[2]*z[5]*z[7]*z[8] + 73156608*z[1]*z[12]*z[2]*z[5]*z[7]*z[9] - 675053568*z[1]*z[12]*z[2]*z[5]*z[7] + 146313216*z[1]*z[12]*z[2]*z[5]*z[8]*z[9] - 1349526528*z[1]*z[12]*z[2]*z[5]*z[8] - 2694408192*z[1]*z[12]*z[2]*z[5]*z[9] + 18337923072*z[1]*z[12]*z[2]*z[5] - 4644864*z[1]*z[12]*z[2]*z[6]*z[7]*z[8]*z[9] + 73156608*z[1]*z[12]*z[2]*z[6]*z[7]*z[8] + 146313216*z[1]*z[12]*z[2]*z[6]*z[7]*z[9] - 1350107136*z[1]*z[12]*z[2]*z[6]*z[7] + 292626432*z[1]*z[12]*z[2]*z[6]*z[8]*z[9] - 2699053056*z[1]*z[12]*z[2]*z[6]*z[8] - 5388816384*z[1]*z[12]*z[2]*z[6]*z[9] + 36675846144*z[1]*z[12]*z[2]*z[6] + 6140928*z[1]*z[12]*z[2]*z[7]*z[8]*z[9] - 96719616*z[1]*z[12]*z[2]*z[7]*z[8] - 193439232*z[1]*z[12]*z[2]*z[7]*z[9] + 1784963072*z[1]*z[12]*z[2]*z[7] - 386878464*z[1]*z[12]*z[2]*z[8]*z[9] + 3568390912*z[1]*z[12]*z[2]*z[8] + 7124499968*z[1]*z[12]*z[2]*z[9] - 48488767488*z[1]*z[12]*z[2] + 589824*z[1]*z[12]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 9289728*z[1]*z[12]*z[3]*z[4]*z[5]*z[7]*z[8] - 18579456*z[1]*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] + 171442176*z[1]*z[12]*z[3]*z[4]*z[5]*z[7] - 37158912*z[1]*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] + 342736896*z[1]*z[12]*z[3]*z[4]*z[5]*z[8] + 684294144*z[1]*z[12]*z[3]*z[4]*z[5]*z[9] - 4657250304*z[1]*z[12]*z[3]*z[4]*z[5] + 1179648*z[1]*z[12]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 18579456*z[1]*z[12]*z[3]*z[4]*z[6]*z[7]*z[8] - 37158912*z[1]*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] + 342884352*z[1]*z[12]*z[3]*z[4]*z[6]*z[7] - 74317824*z[1]*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] + 685473792*z[1]*z[12]*z[3]*z[4]*z[6]*z[8] + 1368588288*z[1]*z[12]*z[3]*z[4]*z[6]*z[9] - 9314500608*z[1]*z[12]*z[3]*z[4]*z[6] - 2322432*z[1]*z[12]*z[3]*z[4]*z[7]*z[8]*z[9] + 36578304*z[1]*z[12]*z[3]*z[4]*z[7]*z[8] + 73156608*z[1]*z[12]*z[3]*z[4]*z[7]*z[9] - 675053568*z[1]*z[12]*z[3]*z[4]*z[7] + 146313216*z[1]*z[12]*z[3]*z[4]*z[8]*z[9] - 1349526528*z[1]*z[12]*z[3]*z[4]*z[8] - 2694408192*z[1]*z[12]*z[3]*z[4]*z[9] + 18337923072*z[1]*z[12]*z[3]*z[4] + 2359296*z[1]*z[12]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 37158912*z[1]*z[12]*z[3]*z[5]*z[6]*z[7]*z[8] - 74317824*z[1]*z[12]*z[3]*z[5]*z[6]*z[7]*z[9] + 685768704*z[1]*z[12]*z[3]*z[5]*z[6]*z[7] - 148635648*z[1]*z[12]*z[3]*z[5]*z[6]*z[8]*z[9] + 1370947584*z[1]*z[12]*z[3]*z[5]*z[6]*z[8] + 2737176576*z[1]*z[12]*z[3]*z[5]*z[6]*z[9] - 18629001216*z[1]*z[12]*z[3]*z[5]*z[6] - 4644864*z[1]*z[12]*z[3]*z[5]*z[7]*z[8]*z[9] + 73156608*z[1]*z[12]*z[3]*z[5]*z[7]*z[8] + 146313216*z[1]*z[12]*z[3]*z[5]*z[7]*z[9] - 1350107136*z[1]*z[12]*z[3]*z[5]*z[7] + 292626432*z[1]*z[12]*z[3]*z[5]*z[8]*z[9] - 2699053056*z[1]*z[12]*z[3]*z[5]*z[8] - 5388816384*z[1]*z[12]*z[3]*z[5]*z[9] + 36675846144*z[1]*z[12]*z[3]*z[5] - 9289728*z[1]*z[12]*z[3]*z[6]*z[7]*z[8]*z[9] + 146313216*z[1]*z[12]*z[3]*z[6]*z[7]*z[8] + 292626432*z[1]*z[12]*z[3]*z[6]*z[7]*z[9] - 2700214272*z[1]*z[12]*z[3]*z[6]*z[7] + 585252864*z[1]*z[12]*z[3]*z[6]*z[8]*z[9] - 5398106112*z[1]*z[12]*z[3]*z[6]*z[8] - 10777632768*z[1]*z[12]*z[3]*z[6]*z[9] + 73351692288*z[1]*z[12]*z[3]*z[6] + 12263424*z[1]*z[12]*z[3]*z[7]*z[8]*z[9] - 193148928*z[1]*z[12]*z[3]*z[7]*z[8] - 386297856*z[1]*z[12]*z[3]*z[7]*z[9] + 3564568576*z[1]*z[12]*z[3]*z[7] - 772595712*z[1]*z[12]*z[3]*z[8]*z[9] + 7126071296*z[1]*z[12]*z[3]*z[8] + 14227615744*z[1]*z[12]*z[3]*z[9] - 96831995904*z[1]*z[12]*z[3] + 4718592*z[1]*z[12]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 74317824*z[1]*z[12]*z[4]*z[5]*z[6]*z[7]*z[8] - 148635648*z[1]*z[12]*z[4]*z[5]*z[6]*z[7]*z[9] + 1371537408*z[1]*z[12]*z[4]*z[5]*z[6]*z[7] - 297271296*z[1]*z[12]*z[4]*z[5]*z[6]*z[8]*z[9] + 2741895168*z[1]*z[12]*z[4]*z[5]*z[6]*z[8] + 5474353152*z[1]*z[12]*z[4]*z[5]*z[6]*z[9] - 37258002432*z[1]*z[12]*z[4]*z[5]*z[6] - 9289728*z[1]*z[12]*z[4]*z[5]*z[7]*z[8]*z[9] + 146313216*z[1]*z[12]*z[4]*z[5]*z[7]*z[8] + 292626432*z[1]*z[12]*z[4]*z[5]*z[7]*z[9] - 2700214272*z[1]*z[12]*z[4]*z[5]*z[7] + 585252864*z[1]*z[12]*z[4]*z[5]*z[8]*z[9] - 5398106112*z[1]*z[12]*z[4]*z[5]*z[8] - 10777632768*z[1]*z[12]*z[4]*z[5]*z[9] + 73351692288*z[1]*z[12]*z[4]*z[5] - 18579456*z[1]*z[12]*z[4]*z[6]*z[7]*z[8]*z[9] + 292626432*z[1]*z[12]*z[4]*z[6]*z[7]*z[8] + 585252864*z[1]*z[12]*z[4]*z[6]*z[7]*z[9] - 5400428544*z[1]*z[12]*z[4]*z[6]*z[7] + 1170505728*z[1]*z[12]*z[4]*z[6]*z[8]*z[9] - 10796212224*z[1]*z[12]*z[4]*z[6]*z[8] - 21555265536*z[1]*z[12]*z[4]*z[6]*z[9] + 146703384576*z[1]*z[12]*z[4]*z[6] + 24379392*z[1]*z[12]*z[4]*z[7]*z[8]*z[9] - 383975424*z[1]*z[12]*z[4]*z[7]*z[8] - 767950848*z[1]*z[12]*z[4]*z[7]*z[9] + 7086276608*z[1]*z[12]*z[4]*z[7] - 1535901696*z[1]*z[12]*z[4]*z[8]*z[9] + 14166458368*z[1]*z[12]*z[4]*z[8] + 28284157952*z[1]*z[12]*z[4]*z[9] - 192499679232*z[1]*z[12]*z[4] - 37158912*z[1]*z[12]*z[5]*z[6]*z[7]*z[8]*z[9] + 585252864*z[1]*z[12]*z[5]*z[6]*z[7]*z[8] + 1170505728*z[1]*z[12]*z[5]*z[6]*z[7]*z[9] - 10800857088*z[1]*z[12]*z[5]*z[6]*z[7] + 2341011456*z[1]*z[12]*z[5]*z[6]*z[8]*z[9] - 21592424448*z[1]*z[12]*z[5]*z[6]*z[8] - 43110531072*z[1]*z[12]*z[5]*z[6]*z[9] + 293406769152*z[1]*z[12]*z[5]*z[6] + 47579136*z[1]*z[12]*z[5]*z[7]*z[8]*z[9] - 749371392*z[1]*z[12]*z[5]*z[7]*z[8] - 1498742784*z[1]*z[12]*z[5]*z[7]*z[9] + 13829668864*z[1]*z[12]*z[5]*z[7] - 2997485568*z[1]*z[12]*z[5]*z[8]*z[9] + 27647442944*z[1]*z[12]*z[5]*z[8] + 55199727616*z[1]*z[12]*z[5]*z[9] - 375684857856*z[1]*z[12]*z[5] + 85721088*z[1]*z[12]*z[6]*z[7]*z[8]*z[9] - 1350107136*z[1]*z[12]*z[6]*z[7]*z[8] - 2700214272*z[1]*z[12]*z[6]*z[7]*z[9] + 24916262912*z[1]*z[12]*z[6]*z[7] - 5400428544*z[1]*z[12]*z[6]*z[8]*z[9] + 49811095552*z[1]*z[12]*z[6]*z[8] + 99450748928*z[1]*z[12]*z[6]*z[9] - 676853710848*z[1]*z[12]*z[6] - 97517952*z[1]*z[12]*z[7]*z[8]*z[9] + 1535907744*z[1]*z[12]*z[7]*z[8] + 3071815488*z[1]*z[12]*z[7]*z[9] - 28345218048*z[1]*z[12]*z[7] + 6143630976*z[1]*z[12]*z[8]*z[9] - 56666056608*z[1]*z[12]*z[8] - 113137077312*z[1]*z[12]*z[9] + 770001748992*z[1]*z[12] - 192*z[1]*z[13]*z[14]*z[2]*z[3]*z[4] - 384*z[1]*z[13]*z[14]*z[2]*z[3]*z[5] - 768*z[1]*z[13]*z[14]*z[2]*z[3]*z[6] + 1512*z[1]*z[13]*z[14]*z[2]*z[3] - 768*z[1]*z[13]*z[14]*z[2]*z[4]*z[5] - 1536*z[1]*z[13]*z[14]*z[2]*z[4]*z[6] + 3024*z[1]*z[13]*z[14]*z[2]*z[4] - 3072*z[1]*z[13]*z[14]*z[2]*z[5]*z[6] + 6048*z[1]*z[13]*z[14]*z[2]*z[5] + 12096*z[1]*z[13]*z[14]*z[2]*z[6] - 15992*z[1]*z[13]*z[14]*z[2] - 1536*z[1]*z[13]*z[14]*z[3]*z[4]*z[5] - 3072*z[1]*z[13]*z[14]*z[3]*z[4]*z[6] + 6048*z[1]*z[13]*z[14]*z[3]*z[4] - 6144*z[1]*z[13]*z[14]*z[3]*z[5]*z[6] + 12096*z[1]*z[13]*z[14]*z[3]*z[5] + 24192*z[1]*z[13]*z[14]*z[3]*z[6] - 31936*z[1]*z[13]*z[14]*z[3] - 12288*z[1]*z[13]*z[14]*z[4]*z[5]*z[6] + 24192*z[1]*z[13]*z[14]*z[4]*z[5] + 48384*z[1]*z[13]*z[14]*z[4]*z[6] - 63488*z[1]*z[13]*z[14]*z[4] + 96768*z[1]*z[13]*z[14]*z[5]*z[6] - 123904*z[1]*z[13]*z[14]*z[5] - 223232*z[1]*z[13]*z[14]*z[6] + 253953*z[1]*z[13]*z[14] - 384*z[1]*z[13]*z[15]*z[2]*z[3]*z[4] - 768*z[1]*z[13]*z[15]*z[2]*z[3]*z[5] - 1536*z[1]*z[13]*z[15]*z[2]*z[3]*z[6] + 3024*z[1]*z[13]*z[15]*z[2]*z[3] - 1536*z[1]*z[13]*z[15]*z[2]*z[4]*z[5] - 3072*z[1]*z[13]*z[15]*z[2]*z[4]*z[6] + 6048*z[1]*z[13]*z[15]*z[2]*z[4] - 6144*z[1]*z[13]*z[15]*z[2]*z[5]*z[6] + 12096*z[1]*z[13]*z[15]*z[2]*z[5] + 24192*z[1]*z[13]*z[15]*z[2]*z[6] - 31984*z[1]*z[13]*z[15]*z[2] - 3072*z[1]*z[13]*z[15]*z[3]*z[4]*z[5] - 6144*z[1]*z[13]*z[15]*z[3]*z[4]*z[6] + 12096*z[1]*z[13]*z[15]*z[3]*z[4] - 12288*z[1]*z[13]*z[15]*z[3]*z[5]*z[6] + 24192*z[1]*z[13]*z[15]*z[3]*z[5] + 48384*z[1]*z[13]*z[15]*z[3]*z[6] - 63872*z[1]*z[13]*z[15]*z[3] - 24576*z[1]*z[13]*z[15]*z[4]*z[5]*z[6] + 48384*z[1]*z[13]*z[15]*z[4]*z[5] + 96768*z[1]*z[13]*z[15]*z[4]*z[6] - 126976*z[1]*z[13]*z[15]*z[4] + 193536*z[1]*z[13]*z[15]*z[5]*z[6] - 247808*z[1]*z[13]*z[15]*z[5] - 446464*z[1]*z[13]*z[15]*z[6] + 507906*z[1]*z[13]*z[15] - 768*z[1]*z[13]*z[16]*z[2]*z[3]*z[4] - 1536*z[1]*z[13]*z[16]*z[2]*z[3]*z[5] - 3072*z[1]*z[13]*z[16]*z[2]*z[3]*z[6] + 6048*z[1]*z[13]*z[16]*z[2]*z[3] - 3072*z[1]*z[13]*z[16]*z[2]*z[4]*z[5] - 6144*z[1]*z[13]*z[16]*z[2]*z[4]*z[6] + 12096*z[1]*z[13]*z[16]*z[2]*z[4] - 12288*z[1]*z[13]*z[16]*z[2]*z[5]*z[6] + 24192*z[1]*z[13]*z[16]*z[2]*z[5] + 48384*z[1]*z[13]*z[16]*z[2]*z[6] - 63968*z[1]*z[13]*z[16]*z[2] - 6144*z[1]*z[13]*z[16]*z[3]*z[4]*z[5] - 12288*z[1]*z[13]*z[16]*z[3]*z[4]*z[6] + 24192*z[1]*z[13]*z[16]*z[3]*z[4] - 24576*z[1]*z[13]*z[16]*z[3]*z[5]*z[6] + 48384*z[1]*z[13]*z[16]*z[3]*z[5] + 96768*z[1]*z[13]*z[16]*z[3]*z[6] - 127744*z[1]*z[13]*z[16]*z[3] - 49152*z[1]*z[13]*z[16]*z[4]*z[5]*z[6] + 96768*z[1]*z[13]*z[16]*z[4]*z[5] + 193536*z[1]*z[13]*z[16]*z[4]*z[6] - 253952*z[1]*z[13]*z[16]*z[4] + 387072*z[1]*z[13]*z[16]*z[5]*z[6] - 495616*z[1]*z[13]*z[16]*z[5] - 892928*z[1]*z[13]*z[16]*z[6] + 1015812*z[1]*z[13]*z[16] - 1536*z[1]*z[13]*z[17]*z[2]*z[3]*z[4] - 3072*z[1]*z[13]*z[17]*z[2]*z[3]*z[5] - 6144*z[1]*z[13]*z[17]*z[2]*z[3]*z[6] + 12096*z[1]*z[13]*z[17]*z[2]*z[3] - 6144*z[1]*z[13]*z[17]*z[2]*z[4]*z[5] - 12288*z[1]*z[13]*z[17]*z[2]*z[4]*z[6] + 24192*z[1]*z[13]*z[17]*z[2]*z[4] - 24576*z[1]*z[13]*z[17]*z[2]*z[5]*z[6] + 48384*z[1]*z[13]*z[17]*z[2]*z[5] + 96768*z[1]*z[13]*z[17]*z[2]*z[6] - 127936*z[1]*z[13]*z[17]*z[2] - 12288*z[1]*z[13]*z[17]*z[3]*z[4]*z[5] - 24576*z[1]*z[13]*z[17]*z[3]*z[4]*z[6] + 48384*z[1]*z[13]*z[17]*z[3]*z[4] - 49152*z[1]*z[13]*z[17]*z[3]*z[5]*z[6] + 96768*z[1]*z[13]*z[17]*z[3]*z[5] + 193536*z[1]*z[13]*z[17]*z[3]*z[6] - 255488*z[1]*z[13]*z[17]*z[3] - 98304*z[1]*z[13]*z[17]*z[4]*z[5]*z[6] + 193536*z[1]*z[13]*z[17]*z[4]*z[5] + 387072*z[1]*z[13]*z[17]*z[4]*z[6] - 507904*z[1]*z[13]*z[17]*z[4] + 774144*z[1]*z[13]*z[17]*z[5]*z[6] - 991232*z[1]*z[13]*z[17]*z[5] - 1785856*z[1]*z[13]*z[17]*z[6] + 2031624*z[1]*z[13]*z[17] - 3072*z[1]*z[13]*z[18]*z[2]*z[3]*z[4] - 6144*z[1]*z[13]*z[18]*z[2]*z[3]*z[5] - 12288*z[1]*z[13]*z[18]*z[2]*z[3]*z[6] + 24192*z[1]*z[13]*z[18]*z[2]*z[3] - 12288*z[1]*z[13]*z[18]*z[2]*z[4]*z[5] - 24576*z[1]*z[13]*z[18]*z[2]*z[4]*z[6] + 48384*z[1]*z[13]*z[18]*z[2]*z[4] - 49152*z[1]*z[13]*z[18]*z[2]*z[5]*z[6] + 96768*z[1]*z[13]*z[18]*z[2]*z[5] + 193536*z[1]*z[13]*z[18]*z[2]*z[6] - 255872*z[1]*z[13]*z[18]*z[2] - 24576*z[1]*z[13]*z[18]*z[3]*z[4]*z[5] - 49152*z[1]*z[13]*z[18]*z[3]*z[4]*z[6] + 96768*z[1]*z[13]*z[18]*z[3]*z[4] - 98304*z[1]*z[13]*z[18]*z[3]*z[5]*z[6] + 193536*z[1]*z[13]*z[18]*z[3]*z[5] + 387072*z[1]*z[13]*z[18]*z[3]*z[6] - 510976*z[1]*z[13]*z[18]*z[3] - 196608*z[1]*z[13]*z[18]*z[4]*z[5]*z[6] + 387072*z[1]*z[13]*z[18]*z[4]*z[5] + 774144*z[1]*z[13]*z[18]*z[4]*z[6] - 1015808*z[1]*z[13]*z[18]*z[4] + 1548288*z[1]*z[13]*z[18]*z[5]*z[6] - 1982464*z[1]*z[13]*z[18]*z[5] - 3571712*z[1]*z[13]*z[18]*z[6] + 4063248*z[1]*z[13]*z[18] + 6048*z[1]*z[13]*z[2]*z[3]*z[4] + 12096*z[1]*z[13]*z[2]*z[3]*z[5] + 24192*z[1]*z[13]*z[2]*z[3]*z[6] - 47628*z[1]*z[13]*z[2]*z[3] + 24192*z[1]*z[13]*z[2]*z[4]*z[5] + 48384*z[1]*z[13]*z[2]*z[4]*z[6] - 95256*z[1]*z[13]*z[2]*z[4] + 96768*z[1]*z[13]*z[2]*z[5]*z[6] - 190512*z[1]*z[13]*z[2]*z[5] - 381024*z[1]*z[13]*z[2]*z[6] + 503748*z[1]*z[13]*z[2] + 48384*z[1]*z[13]*z[3]*z[4]*z[5] + 96768*z[1]*z[13]*z[3]*z[4]*z[6] - 190512*z[1]*z[13]*z[3]*z[4] + 193536*z[1]*z[13]*z[3]*z[5]*z[6] - 381024*z[1]*z[13]*z[3]*z[5] - 762048*z[1]*z[13]*z[3]*z[6] + 1005984*z[1]*z[13]*z[3] + 387072*z[1]*z[13]*z[4]*z[5]*z[6] - 762048*z[1]*z[13]*z[4]*z[5] - 1524096*z[1]*z[13]*z[4]*z[6] + 1999872*z[1]*z[13]*z[4] - 3048192*z[1]*z[13]*z[5]*z[6] + 3902976*z[1]*z[13]*z[5] + 7031808*z[1]*z[13]*z[6] - 15999039*z[1]*z[13]/2 - 768*z[1]*z[14]*z[15]*z[2]*z[3]*z[4] - 1536*z[1]*z[14]*z[15]*z[2]*z[3]*z[5] - 3072*z[1]*z[14]*z[15]*z[2]*z[3]*z[6] + 6048*z[1]*z[14]*z[15]*z[2]*z[3] - 3072*z[1]*z[14]*z[15]*z[2]*z[4]*z[5] - 6144*z[1]*z[14]*z[15]*z[2]*z[4]*z[6] + 12096*z[1]*z[14]*z[15]*z[2]*z[4] - 12288*z[1]*z[14]*z[15]*z[2]*z[5]*z[6] + 24192*z[1]*z[14]*z[15]*z[2]*z[5] + 48384*z[1]*z[14]*z[15]*z[2]*z[6] - 63968*z[1]*z[14]*z[15]*z[2] - 6144*z[1]*z[14]*z[15]*z[3]*z[4]*z[5] - 12288*z[1]*z[14]*z[15]*z[3]*z[4]*z[6] + 24192*z[1]*z[14]*z[15]*z[3]*z[4] - 24576*z[1]*z[14]*z[15]*z[3]*z[5]*z[6] + 48384*z[1]*z[14]*z[15]*z[3]*z[5] + 96768*z[1]*z[14]*z[15]*z[3]*z[6] - 127744*z[1]*z[14]*z[15]*z[3] - 49152*z[1]*z[14]*z[15]*z[4]*z[5]*z[6] + 96768*z[1]*z[14]*z[15]*z[4]*z[5] + 193536*z[1]*z[14]*z[15]*z[4]*z[6] - 253952*z[1]*z[14]*z[15]*z[4] + 387072*z[1]*z[14]*z[15]*z[5]*z[6] - 495616*z[1]*z[14]*z[15]*z[5] - 892928*z[1]*z[14]*z[15]*z[6] + 1015812*z[1]*z[14]*z[15] - 1536*z[1]*z[14]*z[16]*z[2]*z[3]*z[4] - 3072*z[1]*z[14]*z[16]*z[2]*z[3]*z[5] - 6144*z[1]*z[14]*z[16]*z[2]*z[3]*z[6] + 12096*z[1]*z[14]*z[16]*z[2]*z[3] - 6144*z[1]*z[14]*z[16]*z[2]*z[4]*z[5] - 12288*z[1]*z[14]*z[16]*z[2]*z[4]*z[6] + 24192*z[1]*z[14]*z[16]*z[2]*z[4] - 24576*z[1]*z[14]*z[16]*z[2]*z[5]*z[6] + 48384*z[1]*z[14]*z[16]*z[2]*z[5] + 96768*z[1]*z[14]*z[16]*z[2]*z[6] - 127936*z[1]*z[14]*z[16]*z[2] - 12288*z[1]*z[14]*z[16]*z[3]*z[4]*z[5] - 24576*z[1]*z[14]*z[16]*z[3]*z[4]*z[6] + 48384*z[1]*z[14]*z[16]*z[3]*z[4] - 49152*z[1]*z[14]*z[16]*z[3]*z[5]*z[6] + 96768*z[1]*z[14]*z[16]*z[3]*z[5] + 193536*z[1]*z[14]*z[16]*z[3]*z[6] - 255488*z[1]*z[14]*z[16]*z[3] - 98304*z[1]*z[14]*z[16]*z[4]*z[5]*z[6] + 193536*z[1]*z[14]*z[16]*z[4]*z[5] + 387072*z[1]*z[14]*z[16]*z[4]*z[6] - 507904*z[1]*z[14]*z[16]*z[4] + 774144*z[1]*z[14]*z[16]*z[5]*z[6] - 991232*z[1]*z[14]*z[16]*z[5] - 1785856*z[1]*z[14]*z[16]*z[6] + 2031624*z[1]*z[14]*z[16] - 3072*z[1]*z[14]*z[17]*z[2]*z[3]*z[4] - 6144*z[1]*z[14]*z[17]*z[2]*z[3]*z[5] - 12288*z[1]*z[14]*z[17]*z[2]*z[3]*z[6] + 24192*z[1]*z[14]*z[17]*z[2]*z[3] - 12288*z[1]*z[14]*z[17]*z[2]*z[4]*z[5] - 24576*z[1]*z[14]*z[17]*z[2]*z[4]*z[6] + 48384*z[1]*z[14]*z[17]*z[2]*z[4] - 49152*z[1]*z[14]*z[17]*z[2]*z[5]*z[6] + 96768*z[1]*z[14]*z[17]*z[2]*z[5] + 193536*z[1]*z[14]*z[17]*z[2]*z[6] - 255872*z[1]*z[14]*z[17]*z[2] - 24576*z[1]*z[14]*z[17]*z[3]*z[4]*z[5] - 49152*z[1]*z[14]*z[17]*z[3]*z[4]*z[6] + 96768*z[1]*z[14]*z[17]*z[3]*z[4] - 98304*z[1]*z[14]*z[17]*z[3]*z[5]*z[6] + 193536*z[1]*z[14]*z[17]*z[3]*z[5] + 387072*z[1]*z[14]*z[17]*z[3]*z[6] - 510976*z[1]*z[14]*z[17]*z[3] - 196608*z[1]*z[14]*z[17]*z[4]*z[5]*z[6] + 387072*z[1]*z[14]*z[17]*z[4]*z[5] + 774144*z[1]*z[14]*z[17]*z[4]*z[6] - 1015808*z[1]*z[14]*z[17]*z[4] + 1548288*z[1]*z[14]*z[17]*z[5]*z[6] - 1982464*z[1]*z[14]*z[17]*z[5] - 3571712*z[1]*z[14]*z[17]*z[6] + 4063248*z[1]*z[14]*z[17] - 6144*z[1]*z[14]*z[18]*z[2]*z[3]*z[4] - 12288*z[1]*z[14]*z[18]*z[2]*z[3]*z[5] - 24576*z[1]*z[14]*z[18]*z[2]*z[3]*z[6] + 48384*z[1]*z[14]*z[18]*z[2]*z[3] - 24576*z[1]*z[14]*z[18]*z[2]*z[4]*z[5] - 49152*z[1]*z[14]*z[18]*z[2]*z[4]*z[6] + 96768*z[1]*z[14]*z[18]*z[2]*z[4] - 98304*z[1]*z[14]*z[18]*z[2]*z[5]*z[6] + 193536*z[1]*z[14]*z[18]*z[2]*z[5] + 387072*z[1]*z[14]*z[18]*z[2]*z[6] - 511744*z[1]*z[14]*z[18]*z[2] - 49152*z[1]*z[14]*z[18]*z[3]*z[4]*z[5] - 98304*z[1]*z[14]*z[18]*z[3]*z[4]*z[6] + 193536*z[1]*z[14]*z[18]*z[3]*z[4] - 196608*z[1]*z[14]*z[18]*z[3]*z[5]*z[6] + 387072*z[1]*z[14]*z[18]*z[3]*z[5] + 774144*z[1]*z[14]*z[18]*z[3]*z[6] - 1021952*z[1]*z[14]*z[18]*z[3] - 393216*z[1]*z[14]*z[18]*z[4]*z[5]*z[6] + 774144*z[1]*z[14]*z[18]*z[4]*z[5] + 1548288*z[1]*z[14]*z[18]*z[4]*z[6] - 2031616*z[1]*z[14]*z[18]*z[4] + 3096576*z[1]*z[14]*z[18]*z[5]*z[6] - 3964928*z[1]*z[14]*z[18]*z[5] - 7143424*z[1]*z[14]*z[18]*z[6] + 8126496*z[1]*z[14]*z[18] + 12096*z[1]*z[14]*z[2]*z[3]*z[4] + 24192*z[1]*z[14]*z[2]*z[3]*z[5] + 48384*z[1]*z[14]*z[2]*z[3]*z[6] - 95256*z[1]*z[14]*z[2]*z[3] + 48384*z[1]*z[14]*z[2]*z[4]*z[5] + 96768*z[1]*z[14]*z[2]*z[4]*z[6] - 190512*z[1]*z[14]*z[2]*z[4] + 193536*z[1]*z[14]*z[2]*z[5]*z[6] - 381024*z[1]*z[14]*z[2]*z[5] - 762048*z[1]*z[14]*z[2]*z[6] + 1007496*z[1]*z[14]*z[2] + 96768*z[1]*z[14]*z[3]*z[4]*z[5] + 193536*z[1]*z[14]*z[3]*z[4]*z[6] - 381024*z[1]*z[14]*z[3]*z[4] + 387072*z[1]*z[14]*z[3]*z[5]*z[6] - 762048*z[1]*z[14]*z[3]*z[5] - 1524096*z[1]*z[14]*z[3]*z[6] + 2011968*z[1]*z[14]*z[3] + 774144*z[1]*z[14]*z[4]*z[5]*z[6] - 1524096*z[1]*z[14]*z[4]*z[5] - 3048192*z[1]*z[14]*z[4]*z[6] + 3999744*z[1]*z[14]*z[4] - 6096384*z[1]*z[14]*z[5]*z[6] + 7805952*z[1]*z[14]*z[5] + 14063616*z[1]*z[14]*z[6] - 15999039*z[1]*z[14] - 3072*z[1]*z[15]*z[16]*z[2]*z[3]*z[4] - 6144*z[1]*z[15]*z[16]*z[2]*z[3]*z[5] - 12288*z[1]*z[15]*z[16]*z[2]*z[3]*z[6] + 24192*z[1]*z[15]*z[16]*z[2]*z[3] - 12288*z[1]*z[15]*z[16]*z[2]*z[4]*z[5] - 24576*z[1]*z[15]*z[16]*z[2]*z[4]*z[6] + 48384*z[1]*z[15]*z[16]*z[2]*z[4] - 49152*z[1]*z[15]*z[16]*z[2]*z[5]*z[6] + 96768*z[1]*z[15]*z[16]*z[2]*z[5] + 193536*z[1]*z[15]*z[16]*z[2]*z[6] - 255872*z[1]*z[15]*z[16]*z[2] - 24576*z[1]*z[15]*z[16]*z[3]*z[4]*z[5] - 49152*z[1]*z[15]*z[16]*z[3]*z[4]*z[6] + 96768*z[1]*z[15]*z[16]*z[3]*z[4] - 98304*z[1]*z[15]*z[16]*z[3]*z[5]*z[6] + 193536*z[1]*z[15]*z[16]*z[3]*z[5] + 387072*z[1]*z[15]*z[16]*z[3]*z[6] - 510976*z[1]*z[15]*z[16]*z[3] - 196608*z[1]*z[15]*z[16]*z[4]*z[5]*z[6] + 387072*z[1]*z[15]*z[16]*z[4]*z[5] + 774144*z[1]*z[15]*z[16]*z[4]*z[6] - 1015808*z[1]*z[15]*z[16]*z[4] + 1548288*z[1]*z[15]*z[16]*z[5]*z[6] - 1982464*z[1]*z[15]*z[16]*z[5] - 3571712*z[1]*z[15]*z[16]*z[6] + 4063248*z[1]*z[15]*z[16] - 6144*z[1]*z[15]*z[17]*z[2]*z[3]*z[4] - 12288*z[1]*z[15]*z[17]*z[2]*z[3]*z[5] - 24576*z[1]*z[15]*z[17]*z[2]*z[3]*z[6] + 48384*z[1]*z[15]*z[17]*z[2]*z[3] - 24576*z[1]*z[15]*z[17]*z[2]*z[4]*z[5] - 49152*z[1]*z[15]*z[17]*z[2]*z[4]*z[6] + 96768*z[1]*z[15]*z[17]*z[2]*z[4] - 98304*z[1]*z[15]*z[17]*z[2]*z[5]*z[6] + 193536*z[1]*z[15]*z[17]*z[2]*z[5] + 387072*z[1]*z[15]*z[17]*z[2]*z[6] - 511744*z[1]*z[15]*z[17]*z[2] - 49152*z[1]*z[15]*z[17]*z[3]*z[4]*z[5] - 98304*z[1]*z[15]*z[17]*z[3]*z[4]*z[6] + 193536*z[1]*z[15]*z[17]*z[3]*z[4] - 196608*z[1]*z[15]*z[17]*z[3]*z[5]*z[6] + 387072*z[1]*z[15]*z[17]*z[3]*z[5] + 774144*z[1]*z[15]*z[17]*z[3]*z[6] - 1021952*z[1]*z[15]*z[17]*z[3] - 393216*z[1]*z[15]*z[17]*z[4]*z[5]*z[6] + 774144*z[1]*z[15]*z[17]*z[4]*z[5] + 1548288*z[1]*z[15]*z[17]*z[4]*z[6] - 2031616*z[1]*z[15]*z[17]*z[4] + 3096576*z[1]*z[15]*z[17]*z[5]*z[6] - 3964928*z[1]*z[15]*z[17]*z[5] - 7143424*z[1]*z[15]*z[17]*z[6] + 8126496*z[1]*z[15]*z[17] - 12288*z[1]*z[15]*z[18]*z[2]*z[3]*z[4] - 24576*z[1]*z[15]*z[18]*z[2]*z[3]*z[5] - 49152*z[1]*z[15]*z[18]*z[2]*z[3]*z[6] + 96768*z[1]*z[15]*z[18]*z[2]*z[3] - 49152*z[1]*z[15]*z[18]*z[2]*z[4]*z[5] - 98304*z[1]*z[15]*z[18]*z[2]*z[4]*z[6] + 193536*z[1]*z[15]*z[18]*z[2]*z[4] - 196608*z[1]*z[15]*z[18]*z[2]*z[5]*z[6] + 387072*z[1]*z[15]*z[18]*z[2]*z[5] + 774144*z[1]*z[15]*z[18]*z[2]*z[6] - 1023488*z[1]*z[15]*z[18]*z[2] - 98304*z[1]*z[15]*z[18]*z[3]*z[4]*z[5] - 196608*z[1]*z[15]*z[18]*z[3]*z[4]*z[6] + 387072*z[1]*z[15]*z[18]*z[3]*z[4] - 393216*z[1]*z[15]*z[18]*z[3]*z[5]*z[6] + 774144*z[1]*z[15]*z[18]*z[3]*z[5] + 1548288*z[1]*z[15]*z[18]*z[3]*z[6] - 2043904*z[1]*z[15]*z[18]*z[3] - 786432*z[1]*z[15]*z[18]*z[4]*z[5]*z[6] + 1548288*z[1]*z[15]*z[18]*z[4]*z[5] + 3096576*z[1]*z[15]*z[18]*z[4]*z[6] - 4063232*z[1]*z[15]*z[18]*z[4] + 6193152*z[1]*z[15]*z[18]*z[5]*z[6] - 7929856*z[1]*z[15]*z[18]*z[5] - 14286848*z[1]*z[15]*z[18]*z[6] + 16252992*z[1]*z[15]*z[18] + 24192*z[1]*z[15]*z[2]*z[3]*z[4] + 48384*z[1]*z[15]*z[2]*z[3]*z[5] + 96768*z[1]*z[15]*z[2]*z[3]*z[6] - 190512*z[1]*z[15]*z[2]*z[3] + 96768*z[1]*z[15]*z[2]*z[4]*z[5] + 193536*z[1]*z[15]*z[2]*z[4]*z[6] - 381024*z[1]*z[15]*z[2]*z[4] + 387072*z[1]*z[15]*z[2]*z[5]*z[6] - 762048*z[1]*z[15]*z[2]*z[5] - 1524096*z[1]*z[15]*z[2]*z[6] + 2014992*z[1]*z[15]*z[2] + 193536*z[1]*z[15]*z[3]*z[4]*z[5] + 387072*z[1]*z[15]*z[3]*z[4]*z[6] - 762048*z[1]*z[15]*z[3]*z[4] + 774144*z[1]*z[15]*z[3]*z[5]*z[6] - 1524096*z[1]*z[15]*z[3]*z[5] - 3048192*z[1]*z[15]*z[3]*z[6] + 4023936*z[1]*z[15]*z[3] + 1548288*z[1]*z[15]*z[4]*z[5]*z[6] - 3048192*z[1]*z[15]*z[4]*z[5] - 6096384*z[1]*z[15]*z[4]*z[6] + 7999488*z[1]*z[15]*z[4] - 12192768*z[1]*z[15]*z[5]*z[6] + 15611904*z[1]*z[15]*z[5] + 28127232*z[1]*z[15]*z[6] - 31998078*z[1]*z[15] - 12288*z[1]*z[16]*z[17]*z[2]*z[3]*z[4] - 24576*z[1]*z[16]*z[17]*z[2]*z[3]*z[5] - 49152*z[1]*z[16]*z[17]*z[2]*z[3]*z[6] + 96768*z[1]*z[16]*z[17]*z[2]*z[3] - 49152*z[1]*z[16]*z[17]*z[2]*z[4]*z[5] - 98304*z[1]*z[16]*z[17]*z[2]*z[4]*z[6] + 193536*z[1]*z[16]*z[17]*z[2]*z[4] - 196608*z[1]*z[16]*z[17]*z[2]*z[5]*z[6] + 387072*z[1]*z[16]*z[17]*z[2]*z[5] + 774144*z[1]*z[16]*z[17]*z[2]*z[6] - 1023488*z[1]*z[16]*z[17]*z[2] - 98304*z[1]*z[16]*z[17]*z[3]*z[4]*z[5] - 196608*z[1]*z[16]*z[17]*z[3]*z[4]*z[6] + 387072*z[1]*z[16]*z[17]*z[3]*z[4] - 393216*z[1]*z[16]*z[17]*z[3]*z[5]*z[6] + 774144*z[1]*z[16]*z[17]*z[3]*z[5] + 1548288*z[1]*z[16]*z[17]*z[3]*z[6] - 2043904*z[1]*z[16]*z[17]*z[3] - 786432*z[1]*z[16]*z[17]*z[4]*z[5]*z[6] + 1548288*z[1]*z[16]*z[17]*z[4]*z[5] + 3096576*z[1]*z[16]*z[17]*z[4]*z[6] - 4063232*z[1]*z[16]*z[17]*z[4] + 6193152*z[1]*z[16]*z[17]*z[5]*z[6] - 7929856*z[1]*z[16]*z[17]*z[5] - 14286848*z[1]*z[16]*z[17]*z[6] + 16252992*z[1]*z[16]*z[17] - 24576*z[1]*z[16]*z[18]*z[2]*z[3]*z[4] - 49152*z[1]*z[16]*z[18]*z[2]*z[3]*z[5] - 98304*z[1]*z[16]*z[18]*z[2]*z[3]*z[6] + 193536*z[1]*z[16]*z[18]*z[2]*z[3] - 98304*z[1]*z[16]*z[18]*z[2]*z[4]*z[5] - 196608*z[1]*z[16]*z[18]*z[2]*z[4]*z[6] + 387072*z[1]*z[16]*z[18]*z[2]*z[4] - 393216*z[1]*z[16]*z[18]*z[2]*z[5]*z[6] + 774144*z[1]*z[16]*z[18]*z[2]*z[5] + 1548288*z[1]*z[16]*z[18]*z[2]*z[6] - 2046976*z[1]*z[16]*z[18]*z[2] - 196608*z[1]*z[16]*z[18]*z[3]*z[4]*z[5] - 393216*z[1]*z[16]*z[18]*z[3]*z[4]*z[6] + 774144*z[1]*z[16]*z[18]*z[3]*z[4] - 786432*z[1]*z[16]*z[18]*z[3]*z[5]*z[6] + 1548288*z[1]*z[16]*z[18]*z[3]*z[5] + 3096576*z[1]*z[16]*z[18]*z[3]*z[6] - 4087808*z[1]*z[16]*z[18]*z[3] - 1572864*z[1]*z[16]*z[18]*z[4]*z[5]*z[6] + 3096576*z[1]*z[16]*z[18]*z[4]*z[5] + 6193152*z[1]*z[16]*z[18]*z[4]*z[6] - 8126464*z[1]*z[16]*z[18]*z[4] + 12386304*z[1]*z[16]*z[18]*z[5]*z[6] - 15859712*z[1]*z[16]*z[18]*z[5] - 28573696*z[1]*z[16]*z[18]*z[6] + 32505984*z[1]*z[16]*z[18] + 48384*z[1]*z[16]*z[2]*z[3]*z[4] + 96768*z[1]*z[16]*z[2]*z[3]*z[5] + 193536*z[1]*z[16]*z[2]*z[3]*z[6] - 381024*z[1]*z[16]*z[2]*z[3] + 193536*z[1]*z[16]*z[2]*z[4]*z[5] + 387072*z[1]*z[16]*z[2]*z[4]*z[6] - 762048*z[1]*z[16]*z[2]*z[4] + 774144*z[1]*z[16]*z[2]*z[5]*z[6] - 1524096*z[1]*z[16]*z[2]*z[5] - 3048192*z[1]*z[16]*z[2]*z[6] + 4029984*z[1]*z[16]*z[2] + 387072*z[1]*z[16]*z[3]*z[4]*z[5] + 774144*z[1]*z[16]*z[3]*z[4]*z[6] - 1524096*z[1]*z[16]*z[3]*z[4] + 1548288*z[1]*z[16]*z[3]*z[5]*z[6] - 3048192*z[1]*z[16]*z[3]*z[5] - 6096384*z[1]*z[16]*z[3]*z[6] + 8047872*z[1]*z[16]*z[3] + 3096576*z[1]*z[16]*z[4]*z[5]*z[6] - 6096384*z[1]*z[16]*z[4]*z[5] - 12192768*z[1]*z[16]*z[4]*z[6] + 15998976*z[1]*z[16]*z[4] - 24385536*z[1]*z[16]*z[5]*z[6] + 31223808*z[1]*z[16]*z[5] + 56254464*z[1]*z[16]*z[6] - 63996156*z[1]*z[16] - 49152*z[1]*z[17]*z[18]*z[2]*z[3]*z[4] - 98304*z[1]*z[17]*z[18]*z[2]*z[3]*z[5] - 196608*z[1]*z[17]*z[18]*z[2]*z[3]*z[6] + 387072*z[1]*z[17]*z[18]*z[2]*z[3] - 196608*z[1]*z[17]*z[18]*z[2]*z[4]*z[5] - 393216*z[1]*z[17]*z[18]*z[2]*z[4]*z[6] + 774144*z[1]*z[17]*z[18]*z[2]*z[4] - 786432*z[1]*z[17]*z[18]*z[2]*z[5]*z[6] + 1548288*z[1]*z[17]*z[18]*z[2]*z[5] + 3096576*z[1]*z[17]*z[18]*z[2]*z[6] - 4093952*z[1]*z[17]*z[18]*z[2] - 393216*z[1]*z[17]*z[18]*z[3]*z[4]*z[5] - 786432*z[1]*z[17]*z[18]*z[3]*z[4]*z[6] + 1548288*z[1]*z[17]*z[18]*z[3]*z[4] - 1572864*z[1]*z[17]*z[18]*z[3]*z[5]*z[6] + 3096576*z[1]*z[17]*z[18]*z[3]*z[5] + 6193152*z[1]*z[17]*z[18]*z[3]*z[6] - 8175616*z[1]*z[17]*z[18]*z[3] - 3145728*z[1]*z[17]*z[18]*z[4]*z[5]*z[6] + 6193152*z[1]*z[17]*z[18]*z[4]*z[5] + 12386304*z[1]*z[17]*z[18]*z[4]*z[6] - 16252928*z[1]*z[17]*z[18]*z[4] + 24772608*z[1]*z[17]*z[18]*z[5]*z[6] - 31719424*z[1]*z[17]*z[18]*z[5] - 57147392*z[1]*z[17]*z[18]*z[6] + 65011968*z[1]*z[17]*z[18] + 96768*z[1]*z[17]*z[2]*z[3]*z[4] + 193536*z[1]*z[17]*z[2]*z[3]*z[5] + 387072*z[1]*z[17]*z[2]*z[3]*z[6] - 762048*z[1]*z[17]*z[2]*z[3] + 387072*z[1]*z[17]*z[2]*z[4]*z[5] + 774144*z[1]*z[17]*z[2]*z[4]*z[6] - 1524096*z[1]*z[17]*z[2]*z[4] + 1548288*z[1]*z[17]*z[2]*z[5]*z[6] - 3048192*z[1]*z[17]*z[2]*z[5] - 6096384*z[1]*z[17]*z[2]*z[6] + 8059968*z[1]*z[17]*z[2] + 774144*z[1]*z[17]*z[3]*z[4]*z[5] + 1548288*z[1]*z[17]*z[3]*z[4]*z[6] - 3048192*z[1]*z[17]*z[3]*z[4] + 3096576*z[1]*z[17]*z[3]*z[5]*z[6] - 6096384*z[1]*z[17]*z[3]*z[5] - 12192768*z[1]*z[17]*z[3]*z[6] + 16095744*z[1]*z[17]*z[3] + 6193152*z[1]*z[17]*z[4]*z[5]*z[6] - 12192768*z[1]*z[17]*z[4]*z[5] - 24385536*z[1]*z[17]*z[4]*z[6] + 31997952*z[1]*z[17]*z[4] - 48771072*z[1]*z[17]*z[5]*z[6] + 62447616*z[1]*z[17]*z[5] + 112508928*z[1]*z[17]*z[6] - 127992312*z[1]*z[17] + 193536*z[1]*z[18]*z[2]*z[3]*z[4] + 387072*z[1]*z[18]*z[2]*z[3]*z[5] + 774144*z[1]*z[18]*z[2]*z[3]*z[6] - 1524096*z[1]*z[18]*z[2]*z[3] + 774144*z[1]*z[18]*z[2]*z[4]*z[5] + 1548288*z[1]*z[18]*z[2]*z[4]*z[6] - 3048192*z[1]*z[18]*z[2]*z[4] + 3096576*z[1]*z[18]*z[2]*z[5]*z[6] - 6096384*z[1]*z[18]*z[2]*z[5] - 12192768*z[1]*z[18]*z[2]*z[6] + 16119936*z[1]*z[18]*z[2] + 1548288*z[1]*z[18]*z[3]*z[4]*z[5] + 3096576*z[1]*z[18]*z[3]*z[4]*z[6] - 6096384*z[1]*z[18]*z[3]*z[4] + 6193152*z[1]*z[18]*z[3]*z[5]*z[6] - 12192768*z[1]*z[18]*z[3]*z[5] - 24385536*z[1]*z[18]*z[3]*z[6] + 32191488*z[1]*z[18]*z[3] + 12386304*z[1]*z[18]*z[4]*z[5]*z[6] - 24385536*z[1]*z[18]*z[4]*z[5] - 48771072*z[1]*z[18]*z[4]*z[6] + 63995904*z[1]*z[18]*z[4] - 97542144*z[1]*z[18]*z[5]*z[6] + 124895232*z[1]*z[18]*z[5] + 225017856*z[1]*z[18]*z[6] - 255984624*z[1]*z[18] + 11416043520*z[1]*z[2]*z[3]*z[4]*z[5]*z[6] - 12500974080*z[1]*z[2]*z[3]*z[4]*z[5] - 19799700480*z[1]*z[2]*z[3]*z[4]*z[6] - 145152*z[1]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] + 1535232*z[1]*z[2]*z[3]*z[4]*z[7]*z[8] + 3065856*z[1]*z[2]*z[3]*z[4]*z[7]*z[9] - 24379488*z[1]*z[2]*z[3]*z[4]*z[7] + 6129408*z[1]*z[2]*z[3]*z[4]*z[8]*z[9] - 48722688*z[1]*z[2]*z[3]*z[4]*z[8] - 97155072*z[1]*z[2]*z[3]*z[4]*z[9] + 21541810944*z[1]*z[2]*z[3]*z[4] - 36998277120*z[1]*z[2]*z[3]*z[5]*z[6] - 290304*z[1]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] + 3070464*z[1]*z[2]*z[3]*z[5]*z[7]*z[8] + 6131712*z[1]*z[2]*z[3]*z[5]*z[7]*z[9] - 48758976*z[1]*z[2]*z[3]*z[5]*z[7] + 12258816*z[1]*z[2]*z[3]*z[5]*z[8]*z[9] - 97445376*z[1]*z[2]*z[3]*z[5]*z[8] - 194310144*z[1]*z[2]*z[3]*z[5]*z[9] + 39816734208*z[1]*z[2]*z[3]*z[5] - 580608*z[1]*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] + 6140928*z[1]*z[2]*z[3]*z[6]*z[7]*z[8] + 12263424*z[1]*z[2]*z[3]*z[6]*z[7]*z[9] - 97517952*z[1]*z[2]*z[3]*z[6]*z[7] + 24517632*z[1]*z[2]*z[3]*z[6]*z[8]*z[9] - 194890752*z[1]*z[2]*z[3]*z[6]*z[8] - 388620288*z[1]*z[2]*z[3]*z[6]*z[9] + 57461984256*z[1]*z[2]*z[3]*z[6] + 1143072*z[1]*z[2]*z[3]*z[7]*z[8]*z[9] - 12089952*z[1]*z[2]*z[3]*z[7]*z[8] - 24143616*z[1]*z[2]*z[3]*z[7]*z[9] + 191988468*z[1]*z[2]*z[3]*z[7] - 48269088*z[1]*z[2]*z[3]*z[8]*z[9] + 383691168*z[1]*z[2]*z[3]*z[8] + 765096192*z[1]*z[2]*z[3]*z[9] - 61479790344*z[1]*z[2]*z[3] - 72695992320*z[1]*z[2]*z[4]*z[5]*z[6] - 580608*z[1]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] + 6140928*z[1]*z[2]*z[4]*z[5]*z[7]*z[8] + 12263424*z[1]*z[2]*z[4]*z[5]*z[7]*z[9] - 97517952*z[1]*z[2]*z[4]*z[5]*z[7] + 24517632*z[1]*z[2]*z[4]*z[5]*z[8]*z[9] - 194890752*z[1]*z[2]*z[4]*z[5]*z[8] - 388620288*z[1]*z[2]*z[4]*z[5]*z[9] + 77989702656*z[1]*z[2]*z[4]*z[5] - 1161216*z[1]*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] + 12281856*z[1]*z[2]*z[4]*z[6]*z[7]*z[8] + 24526848*z[1]*z[2]*z[4]*z[6]*z[7]*z[9] - 195035904*z[1]*z[2]*z[4]*z[6]*z[7] + 49035264*z[1]*z[2]*z[4]*z[6]*z[8]*z[9] - 389781504*z[1]*z[2]*z[4]*z[6]*z[8] - 777240576*z[1]*z[2]*z[4]*z[6]*z[9] + 111966738432*z[1]*z[2]*z[4]*z[6] + 2286144*z[1]*z[2]*z[4]*z[7]*z[8]*z[9] - 24179904*z[1]*z[2]*z[4]*z[7]*z[8] - 48287232*z[1]*z[2]*z[4]*z[7]*z[9] + 383976936*z[1]*z[2]*z[4]*z[7] - 96538176*z[1]*z[2]*z[4]*z[8]*z[9] + 767382336*z[1]*z[2]*z[4]*z[8] + 1530192384*z[1]*z[2]*z[4]*z[9] - 119631124368*z[1]*z[2]*z[4] - 2322432*z[1]*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] + 24563712*z[1]*z[2]*z[5]*z[6]*z[7]*z[8] + 49053696*z[1]*z[2]*z[5]*z[6]*z[7]*z[9] - 390071808*z[1]*z[2]*z[5]*z[6]*z[7] + 98070528*z[1]*z[2]*z[5]*z[6]*z[8]*z[9] - 779563008*z[1]*z[2]*z[5]*z[6]*z[8] - 1554481152*z[1]*z[2]*z[5]*z[6]*z[9] + 201266540544*z[1]*z[2]*z[5]*z[6] + 4572288*z[1]*z[2]*z[5]*z[7]*z[8]*z[9] - 48359808*z[1]*z[2]*z[5]*z[7]*z[8] - 96574464*z[1]*z[2]*z[5]*z[7]*z[9] + 767953872*z[1]*z[2]*z[5]*z[7] - 193076352*z[1]*z[2]*z[5]*z[8]*z[9] + 1534764672*z[1]*z[2]*z[5]*z[8] + 3060384768*z[1]*z[2]*z[5]*z[9] - 214585441056*z[1]*z[2]*z[5] + 9144576*z[1]*z[2]*z[6]*z[7]*z[8]*z[9] - 96719616*z[1]*z[2]*z[6]*z[7]*z[8] - 193148928*z[1]*z[2]*z[6]*z[7]*z[9] + 1535907744*z[1]*z[2]*z[6]*z[7] - 386152704*z[1]*z[2]*z[6]*z[8]*z[9] + 3069529344*z[1]*z[2]*z[6]*z[8] + 6120769536*z[1]*z[2]*z[6]*z[9] - 294183392832*z[1]*z[2]*z[6] - 12089952*z[1]*z[2]*z[7]*z[8]*z[9] + 127872032*z[1]*z[2]*z[7]*z[8] + 255360256*z[1]*z[2]*z[7]*z[9] - 2030608188*z[1]*z[2]*z[7] + 510528608*z[1]*z[2]*z[8]*z[9] - 4058193888*z[1]*z[2]*z[8] - 8092207872*z[1]*z[2]*z[9] + 310518320544*z[1]*z[2] - 144741703680*z[1]*z[3]*z[4]*z[5]*z[6] - 1161216*z[1]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] + 12281856*z[1]*z[3]*z[4]*z[5]*z[7]*z[8] + 24526848*z[1]*z[3]*z[4]*z[5]*z[7]*z[9] - 195035904*z[1]*z[3]*z[4]*z[5]*z[7] + 49035264*z[1]*z[3]*z[4]*z[5]*z[8]*z[9] - 389781504*z[1]*z[3]*z[4]*z[5]*z[8] - 777240576*z[1]*z[3]*z[4]*z[5]*z[9] + 155156232192*z[1]*z[3]*z[4]*z[5] - 2322432*z[1]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] + 24563712*z[1]*z[3]*z[4]*z[6]*z[7]*z[8] + 49053696*z[1]*z[3]*z[4]*z[6]*z[7]*z[9] - 390071808*z[1]*z[3]*z[4]*z[6]*z[7] + 98070528*z[1]*z[3]*z[4]*z[6]*z[8]*z[9] - 779563008*z[1]*z[3]*z[4]*z[6]*z[8] - 1554481152*z[1]*z[3]*z[4]*z[6]*z[9] + 222452281344*z[1]*z[3]*z[4]*z[6] + 4572288*z[1]*z[3]*z[4]*z[7]*z[8]*z[9] - 48359808*z[1]*z[3]*z[4]*z[7]*z[8] - 96574464*z[1]*z[3]*z[4]*z[7]*z[9] + 767953872*z[1]*z[3]*z[4]*z[7] - 193076352*z[1]*z[3]*z[4]*z[8]*z[9] + 1534764672*z[1]*z[3]*z[4]*z[8] + 3060384768*z[1]*z[3]*z[4]*z[9] - 237592940256*z[1]*z[3]*z[4] - 4644864*z[1]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] + 49127424*z[1]*z[3]*z[5]*z[6]*z[7]*z[8] + 98107392*z[1]*z[3]*z[5]*z[6]*z[7]*z[9] - 780143616*z[1]*z[3]*z[5]*z[6]*z[7] + 196141056*z[1]*z[3]*z[5]*z[6]*z[8]*z[9] - 1559126016*z[1]*z[3]*z[5]*z[6]*z[8] - 3108962304*z[1]*z[3]*z[5]*z[6]*z[9] + 399653265408*z[1]*z[3]*z[5]*z[6] + 9144576*z[1]*z[3]*z[5]*z[7]*z[8]*z[9] - 96719616*z[1]*z[3]*z[5]*z[7]*z[8] - 193148928*z[1]*z[3]*z[5]*z[7]*z[9] + 1535907744*z[1]*z[3]*z[5]*z[7] - 386152704*z[1]*z[3]*z[5]*z[8]*z[9] + 3069529344*z[1]*z[3]*z[5]*z[8] + 6120769536*z[1]*z[3]*z[5]*z[9] - 425994835392*z[1]*z[3]*z[5] + 18289152*z[1]*z[3]*z[6]*z[7]*z[8]*z[9] - 193439232*z[1]*z[3]*z[6]*z[7]*z[8] - 386297856*z[1]*z[3]*z[6]*z[7]*z[9] + 3071815488*z[1]*z[3]*z[6]*z[7] - 772305408*z[1]*z[3]*z[6]*z[8]*z[9] + 6139058688*z[1]*z[3]*z[6]*z[8] + 12241539072*z[1]*z[3]*z[6]*z[9] - 583315254144*z[1]*z[3]*z[6] - 24143616*z[1]*z[3]*z[7]*z[8]*z[9] + 255360256*z[1]*z[3]*z[7]*z[8] + 509954048*z[1]*z[3]*z[7]*z[9] - 4055121504*z[1]*z[3]*z[7] + 1019524864*z[1]*z[3]*z[8]*z[9] - 8104207104*z[1]*z[3]*z[8] - 16160126976*z[1]*z[3]*z[9] + 615518495232*z[1]*z[3] - 9289728*z[1]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 98254848*z[1]*z[4]*z[5]*z[6]*z[7]*z[8] + 196214784*z[1]*z[4]*z[5]*z[6]*z[7]*z[9] - 1560287232*z[1]*z[4]*z[5]*z[6]*z[7] + 392282112*z[1]*z[4]*z[5]*z[6]*z[8]*z[9] - 3118252032*z[1]*z[4]*z[5]*z[6]*z[8] - 6217924608*z[1]*z[4]*z[5]*z[6]*z[9] + 776515731456*z[1]*z[4]*z[5]*z[6] + 18289152*z[1]*z[4]*z[5]*z[7]*z[8]*z[9] - 193439232*z[1]*z[4]*z[5]*z[7]*z[8] - 386297856*z[1]*z[4]*z[5]*z[7]*z[9] + 3071815488*z[1]*z[4]*z[5]*z[7] - 772305408*z[1]*z[4]*z[5]*z[8]*z[9] + 6139058688*z[1]*z[4]*z[5]*z[8] + 12241539072*z[1]*z[4]*z[5]*z[9] - 827069007744*z[1]*z[4]*z[5] + 36578304*z[1]*z[4]*z[6]*z[7]*z[8]*z[9] - 386878464*z[1]*z[4]*z[6]*z[7]*z[8] - 772595712*z[1]*z[4]*z[6]*z[7]*z[9] + 6143630976*z[1]*z[4]*z[6]*z[7] - 1544610816*z[1]*z[4]*z[6]*z[8]*z[9] + 12278117376*z[1]*z[4]*z[6]*z[8] + 24483078144*z[1]*z[4]*z[6]*z[9] - 1127193677568*z[1]*z[4]*z[6] - 47996928*z[1]*z[4]*z[7]*z[8]*z[9] + 507650048*z[1]*z[4]*z[7]*z[8] + 1013776384*z[1]*z[4]*z[7]*z[9] - 8061484032*z[1]*z[4]*z[7] + 2026790912*z[1]*z[4]*z[8]*z[9] - 16110968832*z[1]*z[4]*z[8] - 32125943808*z[1]*z[4]*z[9] + 1188165613056*z[1]*z[4] + 73156608*z[1]*z[5]*z[6]*z[7]*z[8]*z[9] - 773756928*z[1]*z[5]*z[6]*z[7]*z[8] - 1545191424*z[1]*z[5]*z[6]*z[7]*z[9] + 12287261952*z[1]*z[5]*z[6]*z[7] - 3089221632*z[1]*z[5]*z[6]*z[8]*z[9] + 24556234752*z[1]*z[5]*z[6]*z[8] + 48966156288*z[1]*z[5]*z[6]*z[9] - 1970106195456*z[1]*z[5]*z[6] - 93671424*z[1]*z[5]*z[7]*z[8]*z[9] + 990736384*z[1]*z[5]*z[7]*z[8] + 1978499072*z[1]*z[5]*z[7]*z[9] - 15732896256*z[1]*z[5]*z[7] + 3955511296*z[1]*z[5]*z[8]*z[9] - 31442374656*z[1]*z[5]*z[8] - 62697406464*z[1]*z[5]*z[9] + 2072542215168*z[1]*z[5] - 168763392*z[1]*z[6]*z[7]*z[8]*z[9] + 1784963072*z[1]*z[6]*z[7]*z[8] + 3564568576*z[1]*z[6]*z[7]*z[9] - 28345218048*z[1]*z[6]*z[7] + 7126458368*z[1]*z[6]*z[8]*z[9] - 56648245248*z[1]*z[6]*z[8] - 112958963712*z[1]*z[6]*z[9] + 2766452127744*z[1]*z[6] + 191988468*z[1]*z[7]*z[8]*z[9] - 2030608188*z[1]*z[7]*z[8] - 4055121504*z[1]*z[7]*z[9] + 64492126209*z[1]*z[7]/2 - 8107195572*z[1]*z[8]*z[9] + 64444129092*z[1]*z[8] + 128504281248*z[1]*z[9] - 5761097508537*z[1]/2 - 12288*z[10]*z[11]*z[12]*z[13]*z[14]*z[7] - 24576*z[10]*z[11]*z[12]*z[13]*z[14]*z[8] - 49152*z[10]*z[11]*z[12]*z[13]*z[14]*z[9] + 774144*z[10]*z[11]*z[12]*z[13]*z[14] - 24576*z[10]*z[11]*z[12]*z[13]*z[15]*z[7] - 49152*z[10]*z[11]*z[12]*z[13]*z[15]*z[8] - 98304*z[10]*z[11]*z[12]*z[13]*z[15]*z[9] + 1548288*z[10]*z[11]*z[12]*z[13]*z[15] - 49152*z[10]*z[11]*z[12]*z[13]*z[16]*z[7] - 98304*z[10]*z[11]*z[12]*z[13]*z[16]*z[8] - 196608*z[10]*z[11]*z[12]*z[13]*z[16]*z[9] + 3096576*z[10]*z[11]*z[12]*z[13]*z[16] - 98304*z[10]*z[11]*z[12]*z[13]*z[17]*z[7] - 196608*z[10]*z[11]*z[12]*z[13]*z[17]*z[8] - 393216*z[10]*z[11]*z[12]*z[13]*z[17]*z[9] + 6193152*z[10]*z[11]*z[12]*z[13]*z[17] - 196608*z[10]*z[11]*z[12]*z[13]*z[18]*z[7] - 393216*z[10]*z[11]*z[12]*z[13]*z[18]*z[8] - 786432*z[10]*z[11]*z[12]*z[13]*z[18]*z[9] + 12386304*z[10]*z[11]*z[12]*z[13]*z[18] + 387072*z[10]*z[11]*z[12]*z[13]*z[7] + 774144*z[10]*z[11]*z[12]*z[13]*z[8] + 1548288*z[10]*z[11]*z[12]*z[13]*z[9] - 24385536*z[10]*z[11]*z[12]*z[13] - 49152*z[10]*z[11]*z[12]*z[14]*z[15]*z[7] - 98304*z[10]*z[11]*z[12]*z[14]*z[15]*z[8] - 196608*z[10]*z[11]*z[12]*z[14]*z[15]*z[9] + 3096576*z[10]*z[11]*z[12]*z[14]*z[15] - 98304*z[10]*z[11]*z[12]*z[14]*z[16]*z[7] - 196608*z[10]*z[11]*z[12]*z[14]*z[16]*z[8] - 393216*z[10]*z[11]*z[12]*z[14]*z[16]*z[9] + 6193152*z[10]*z[11]*z[12]*z[14]*z[16] - 196608*z[10]*z[11]*z[12]*z[14]*z[17]*z[7] - 393216*z[10]*z[11]*z[12]*z[14]*z[17]*z[8] - 786432*z[10]*z[11]*z[12]*z[14]*z[17]*z[9] + 12386304*z[10]*z[11]*z[12]*z[14]*z[17] - 393216*z[10]*z[11]*z[12]*z[14]*z[18]*z[7] - 786432*z[10]*z[11]*z[12]*z[14]*z[18]*z[8] - 1572864*z[10]*z[11]*z[12]*z[14]*z[18]*z[9] + 24772608*z[10]*z[11]*z[12]*z[14]*z[18] + 774144*z[10]*z[11]*z[12]*z[14]*z[7] + 1548288*z[10]*z[11]*z[12]*z[14]*z[8] + 3096576*z[10]*z[11]*z[12]*z[14]*z[9] - 48771072*z[10]*z[11]*z[12]*z[14] - 196608*z[10]*z[11]*z[12]*z[15]*z[16]*z[7] - 393216*z[10]*z[11]*z[12]*z[15]*z[16]*z[8] - 786432*z[10]*z[11]*z[12]*z[15]*z[16]*z[9] + 12386304*z[10]*z[11]*z[12]*z[15]*z[16] - 393216*z[10]*z[11]*z[12]*z[15]*z[17]*z[7] - 786432*z[10]*z[11]*z[12]*z[15]*z[17]*z[8] - 1572864*z[10]*z[11]*z[12]*z[15]*z[17]*z[9] + 24772608*z[10]*z[11]*z[12]*z[15]*z[17] - 786432*z[10]*z[11]*z[12]*z[15]*z[18]*z[7] - 1572864*z[10]*z[11]*z[12]*z[15]*z[18]*z[8] - 3145728*z[10]*z[11]*z[12]*z[15]*z[18]*z[9] + 49545216*z[10]*z[11]*z[12]*z[15]*z[18] + 1548288*z[10]*z[11]*z[12]*z[15]*z[7] + 3096576*z[10]*z[11]*z[12]*z[15]*z[8] + 6193152*z[10]*z[11]*z[12]*z[15]*z[9] - 97542144*z[10]*z[11]*z[12]*z[15] - 786432*z[10]*z[11]*z[12]*z[16]*z[17]*z[7] - 1572864*z[10]*z[11]*z[12]*z[16]*z[17]*z[8] - 3145728*z[10]*z[11]*z[12]*z[16]*z[17]*z[9] + 49545216*z[10]*z[11]*z[12]*z[16]*z[17] - 1572864*z[10]*z[11]*z[12]*z[16]*z[18]*z[7] - 3145728*z[10]*z[11]*z[12]*z[16]*z[18]*z[8] - 6291456*z[10]*z[11]*z[12]*z[16]*z[18]*z[9] + 99090432*z[10]*z[11]*z[12]*z[16]*z[18] + 3096576*z[10]*z[11]*z[12]*z[16]*z[7] + 6193152*z[10]*z[11]*z[12]*z[16]*z[8] + 12386304*z[10]*z[11]*z[12]*z[16]*z[9] - 195084288*z[10]*z[11]*z[12]*z[16] - 3145728*z[10]*z[11]*z[12]*z[17]*z[18]*z[7] - 6291456*z[10]*z[11]*z[12]*z[17]*z[18]*z[8] - 12582912*z[10]*z[11]*z[12]*z[17]*z[18]*z[9] + 198180864*z[10]*z[11]*z[12]*z[17]*z[18] + 6193152*z[10]*z[11]*z[12]*z[17]*z[7] + 12386304*z[10]*z[11]*z[12]*z[17]*z[8] + 24772608*z[10]*z[11]*z[12]*z[17]*z[9] - 390168576*z[10]*z[11]*z[12]*z[17] + 12386304*z[10]*z[11]*z[12]*z[18]*z[7] + 24772608*z[10]*z[11]*z[12]*z[18]*z[8] + 49545216*z[10]*z[11]*z[12]*z[18]*z[9] - 780337152*z[10]*z[11]*z[12]*z[18] + 18874368*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[7] + 37748736*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[8] + 75497472*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[9] - 1189085184*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[5] + 37748736*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[7] + 75497472*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[8] + 150994944*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[9] - 2378170368*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[6] - 74317824*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[7] - 148635648*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[8] - 297271296*z[10]*z[11]*z[12]*z[2]*z[3]*z[4]*z[9] + 4682022912*z[10]*z[11]*z[12]*z[2]*z[3]*z[4] + 75497472*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[7] + 150994944*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[8] + 301989888*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[9] - 4756340736*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[6] - 148635648*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[7] - 297271296*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[8] - 594542592*z[10]*z[11]*z[12]*z[2]*z[3]*z[5]*z[9] + 9364045824*z[10]*z[11]*z[12]*z[2]*z[3]*z[5] - 297271296*z[10]*z[11]*z[12]*z[2]*z[3]*z[6]*z[7] - 594542592*z[10]*z[11]*z[12]*z[2]*z[3]*z[6]*z[8] - 1189085184*z[10]*z[11]*z[12]*z[2]*z[3]*z[6]*z[9] + 18728091648*z[10]*z[11]*z[12]*z[2]*z[3]*z[6] + 392282112*z[10]*z[11]*z[12]*z[2]*z[3]*z[7] + 784564224*z[10]*z[11]*z[12]*z[2]*z[3]*z[8] + 1569128448*z[10]*z[11]*z[12]*z[2]*z[3]*z[9] - 24713773056*z[10]*z[11]*z[12]*z[2]*z[3] + 150994944*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[7] + 301989888*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[8] + 603979776*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[9] - 9512681472*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[6] - 297271296*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[7] - 594542592*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[8] - 1189085184*z[10]*z[11]*z[12]*z[2]*z[4]*z[5]*z[9] + 18728091648*z[10]*z[11]*z[12]*z[2]*z[4]*z[5] - 594542592*z[10]*z[11]*z[12]*z[2]*z[4]*z[6]*z[7] - 1189085184*z[10]*z[11]*z[12]*z[2]*z[4]*z[6]*z[8] - 2378170368*z[10]*z[11]*z[12]*z[2]*z[4]*z[6]*z[9] + 37456183296*z[10]*z[11]*z[12]*z[2]*z[4]*z[6] + 779845632*z[10]*z[11]*z[12]*z[2]*z[4]*z[7] + 1559691264*z[10]*z[11]*z[12]*z[2]*z[4]*z[8] + 3119382528*z[10]*z[11]*z[12]*z[2]*z[4]*z[9] - 49130274816*z[10]*z[11]*z[12]*z[2]*z[4] - 1189085184*z[10]*z[11]*z[12]*z[2]*z[5]*z[6]*z[7] - 2378170368*z[10]*z[11]*z[12]*z[2]*z[5]*z[6]*z[8] - 4756340736*z[10]*z[11]*z[12]*z[2]*z[5]*z[6]*z[9] + 74912366592*z[10]*z[11]*z[12]*z[2]*z[5]*z[6] + 1521942528*z[10]*z[11]*z[12]*z[2]*z[5]*z[7] + 3043885056*z[10]*z[11]*z[12]*z[2]*z[5]*z[8] + 6087770112*z[10]*z[11]*z[12]*z[2]*z[5]*z[9] - 95882379264*z[10]*z[11]*z[12]*z[2]*z[5] + 2741895168*z[10]*z[11]*z[12]*z[2]*z[6]*z[7] + 5483790336*z[10]*z[11]*z[12]*z[2]*z[6]*z[8] + 10967580672*z[10]*z[11]*z[12]*z[2]*z[6]*z[9] - 172739395584*z[10]*z[11]*z[12]*z[2]*z[6] - 3118252032*z[10]*z[11]*z[12]*z[2]*z[7] - 6236504064*z[10]*z[11]*z[12]*z[2]*z[8] - 12473008128*z[10]*z[11]*z[12]*z[2]*z[9] + 196449878016*z[10]*z[11]*z[12]*z[2] + 301989888*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[7] + 603979776*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[8] + 1207959552*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[9] - 19025362944*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[6] - 594542592*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[7] - 1189085184*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[8] - 2378170368*z[10]*z[11]*z[12]*z[3]*z[4]*z[5]*z[9] + 37456183296*z[10]*z[11]*z[12]*z[3]*z[4]*z[5] - 1189085184*z[10]*z[11]*z[12]*z[3]*z[4]*z[6]*z[7] - 2378170368*z[10]*z[11]*z[12]*z[3]*z[4]*z[6]*z[8] - 4756340736*z[10]*z[11]*z[12]*z[3]*z[4]*z[6]*z[9] + 74912366592*z[10]*z[11]*z[12]*z[3]*z[4]*z[6] + 1557331968*z[10]*z[11]*z[12]*z[3]*z[4]*z[7] + 3114663936*z[10]*z[11]*z[12]*z[3]*z[4]*z[8] + 6229327872*z[10]*z[11]*z[12]*z[3]*z[4]*z[9] - 98111913984*z[10]*z[11]*z[12]*z[3]*z[4] - 2378170368*z[10]*z[11]*z[12]*z[3]*z[5]*z[6]*z[7] - 4756340736*z[10]*z[11]*z[12]*z[3]*z[5]*z[6]*z[8] - 9512681472*z[10]*z[11]*z[12]*z[3]*z[5]*z[6]*z[9] + 149824733184*z[10]*z[11]*z[12]*z[3]*z[5]*z[6] + 3039166464*z[10]*z[11]*z[12]*z[3]*z[5]*z[7] + 6078332928*z[10]*z[11]*z[12]*z[3]*z[5]*z[8] + 12156665856*z[10]*z[11]*z[12]*z[3]*z[5]*z[9] - 191467487232*z[10]*z[11]*z[12]*z[3]*z[5] + 5474353152*z[10]*z[11]*z[12]*z[3]*z[6]*z[7] + 10948706304*z[10]*z[11]*z[12]*z[3]*z[6]*z[8] + 21897412608*z[10]*z[11]*z[12]*z[3]*z[6]*z[9] - 344884248576*z[10]*z[11]*z[12]*z[3]*z[6] - 6217924608*z[10]*z[11]*z[12]*z[3]*z[7] - 12435849216*z[10]*z[11]*z[12]*z[3]*z[8] - 24871698432*z[10]*z[11]*z[12]*z[3]*z[9] + 391729250304*z[10]*z[11]*z[12]*z[3] - 4756340736*z[10]*z[11]*z[12]*z[4]*z[5]*z[6]*z[7] - 9512681472*z[10]*z[11]*z[12]*z[4]*z[5]*z[6]*z[8] - 19025362944*z[10]*z[11]*z[12]*z[4]*z[5]*z[6]*z[9] + 299649466368*z[10]*z[11]*z[12]*z[4]*z[5]*z[6] + 6040584192*z[10]*z[11]*z[12]*z[4]*z[5]*z[7] + 12081168384*z[10]*z[11]*z[12]*z[4]*z[5]*z[8] + 24162336768*z[10]*z[11]*z[12]*z[4]*z[5]*z[9] - 380556804096*z[10]*z[11]*z[12]*z[4]*z[5] + 10873208832*z[10]*z[11]*z[12]*z[4]*z[6]*z[7] + 21746417664*z[10]*z[11]*z[12]*z[4]*z[6]*z[8] + 43492835328*z[10]*z[11]*z[12]*z[4]*z[6]*z[9] - 685012156416*z[10]*z[11]*z[12]*z[4]*z[6] - 12287213568*z[10]*z[11]*z[12]*z[4]*z[7] - 24574427136*z[10]*z[11]*z[12]*z[4]*z[8] - 49148854272*z[10]*z[11]*z[12]*z[4]*z[9] + 774094454784*z[10]*z[11]*z[12]*z[4] + 21142437888*z[10]*z[11]*z[12]*z[5]*z[6]*z[7] + 42284875776*z[10]*z[11]*z[12]*z[5]*z[6]*z[8] + 84569751552*z[10]*z[11]*z[12]*z[5]*z[6]*z[9] - 1331973586944*z[10]*z[11]*z[12]*z[5]*z[6] - 23385341952*z[10]*z[11]*z[12]*z[5]*z[7] - 46770683904*z[10]*z[11]*z[12]*z[5]*z[8] - 93541367808*z[10]*z[11]*z[12]*z[5]*z[9] + 1473276542976*z[10]*z[11]*z[12]*z[5] - 37258002432*z[10]*z[11]*z[12]*z[6]*z[7] - 74516004864*z[10]*z[11]*z[12]*z[6]*z[8] - 149032009728*z[10]*z[11]*z[12]*z[6]*z[9] + 2347254153216*z[10]*z[11]*z[12]*z[6] + 11416043520*z[10]*z[11]*z[12]*z[7]*z[8]*z[9] - 72695992320*z[10]*z[11]*z[12]*z[7]*z[8] - 144741703680*z[10]*z[11]*z[12]*z[7]*z[9] + 776515731456*z[10]*z[11]*z[12]*z[7] - 289158266880*z[10]*z[11]*z[12]*z[8]*z[9] + 1550171000832*z[10]*z[11]*z[12]*z[8] + 3077520236544*z[10]*z[11]*z[12]*z[9] - 15000102125568*z[10]*z[11]*z[12] - 768*z[10]*z[11]*z[13]*z[14]*z[7]*z[8] - 1536*z[10]*z[11]*z[13]*z[14]*z[7]*z[9] + 24192*z[10]*z[11]*z[13]*z[14]*z[7] - 3072*z[10]*z[11]*z[13]*z[14]*z[8]*z[9] + 48384*z[10]*z[11]*z[13]*z[14]*z[8] + 96768*z[10]*z[11]*z[13]*z[14]*z[9] - 983168*z[10]*z[11]*z[13]*z[14] - 1536*z[10]*z[11]*z[13]*z[15]*z[7]*z[8] - 3072*z[10]*z[11]*z[13]*z[15]*z[7]*z[9] + 48384*z[10]*z[11]*z[13]*z[15]*z[7] - 6144*z[10]*z[11]*z[13]*z[15]*z[8]*z[9] + 96768*z[10]*z[11]*z[13]*z[15]*z[8] + 193536*z[10]*z[11]*z[13]*z[15]*z[9] - 1966336*z[10]*z[11]*z[13]*z[15] - 3072*z[10]*z[11]*z[13]*z[16]*z[7]*z[8] - 6144*z[10]*z[11]*z[13]*z[16]*z[7]*z[9] + 96768*z[10]*z[11]*z[13]*z[16]*z[7] - 12288*z[10]*z[11]*z[13]*z[16]*z[8]*z[9] + 193536*z[10]*z[11]*z[13]*z[16]*z[8] + 387072*z[10]*z[11]*z[13]*z[16]*z[9] - 3932672*z[10]*z[11]*z[13]*z[16] - 6144*z[10]*z[11]*z[13]*z[17]*z[7]*z[8] - 12288*z[10]*z[11]*z[13]*z[17]*z[7]*z[9] + 193536*z[10]*z[11]*z[13]*z[17]*z[7] - 24576*z[10]*z[11]*z[13]*z[17]*z[8]*z[9] + 387072*z[10]*z[11]*z[13]*z[17]*z[8] + 774144*z[10]*z[11]*z[13]*z[17]*z[9] - 7865344*z[10]*z[11]*z[13]*z[17] - 12288*z[10]*z[11]*z[13]*z[18]*z[7]*z[8] - 24576*z[10]*z[11]*z[13]*z[18]*z[7]*z[9] + 387072*z[10]*z[11]*z[13]*z[18]*z[7] - 49152*z[10]*z[11]*z[13]*z[18]*z[8]*z[9] + 774144*z[10]*z[11]*z[13]*z[18]*z[8] + 1548288*z[10]*z[11]*z[13]*z[18]*z[9] - 15730688*z[10]*z[11]*z[13]*z[18] + 24192*z[10]*z[11]*z[13]*z[7]*z[8] + 48384*z[10]*z[11]*z[13]*z[7]*z[9] - 762048*z[10]*z[11]*z[13]*z[7] + 96768*z[10]*z[11]*z[13]*z[8]*z[9] - 1524096*z[10]*z[11]*z[13]*z[8] - 3048192*z[10]*z[11]*z[13]*z[9] + 30969792*z[10]*z[11]*z[13] - 3072*z[10]*z[11]*z[14]*z[15]*z[7]*z[8] - 6144*z[10]*z[11]*z[14]*z[15]*z[7]*z[9] + 96768*z[10]*z[11]*z[14]*z[15]*z[7] - 12288*z[10]*z[11]*z[14]*z[15]*z[8]*z[9] + 193536*z[10]*z[11]*z[14]*z[15]*z[8] + 387072*z[10]*z[11]*z[14]*z[15]*z[9] - 3932672*z[10]*z[11]*z[14]*z[15] - 6144*z[10]*z[11]*z[14]*z[16]*z[7]*z[8] - 12288*z[10]*z[11]*z[14]*z[16]*z[7]*z[9] + 193536*z[10]*z[11]*z[14]*z[16]*z[7] - 24576*z[10]*z[11]*z[14]*z[16]*z[8]*z[9] + 387072*z[10]*z[11]*z[14]*z[16]*z[8] + 774144*z[10]*z[11]*z[14]*z[16]*z[9] - 7865344*z[10]*z[11]*z[14]*z[16] - 12288*z[10]*z[11]*z[14]*z[17]*z[7]*z[8] - 24576*z[10]*z[11]*z[14]*z[17]*z[7]*z[9] + 387072*z[10]*z[11]*z[14]*z[17]*z[7] - 49152*z[10]*z[11]*z[14]*z[17]*z[8]*z[9] + 774144*z[10]*z[11]*z[14]*z[17]*z[8] + 1548288*z[10]*z[11]*z[14]*z[17]*z[9] - 15730688*z[10]*z[11]*z[14]*z[17] - 24576*z[10]*z[11]*z[14]*z[18]*z[7]*z[8] - 49152*z[10]*z[11]*z[14]*z[18]*z[7]*z[9] + 774144*z[10]*z[11]*z[14]*z[18]*z[7] - 98304*z[10]*z[11]*z[14]*z[18]*z[8]*z[9] + 1548288*z[10]*z[11]*z[14]*z[18]*z[8] + 3096576*z[10]*z[11]*z[14]*z[18]*z[9] - 31461376*z[10]*z[11]*z[14]*z[18] + 48384*z[10]*z[11]*z[14]*z[7]*z[8] + 96768*z[10]*z[11]*z[14]*z[7]*z[9] - 1524096*z[10]*z[11]*z[14]*z[7] + 193536*z[10]*z[11]*z[14]*z[8]*z[9] - 3048192*z[10]*z[11]*z[14]*z[8] - 6096384*z[10]*z[11]*z[14]*z[9] + 61939584*z[10]*z[11]*z[14] - 12288*z[10]*z[11]*z[15]*z[16]*z[7]*z[8] - 24576*z[10]*z[11]*z[15]*z[16]*z[7]*z[9] + 387072*z[10]*z[11]*z[15]*z[16]*z[7] - 49152*z[10]*z[11]*z[15]*z[16]*z[8]*z[9] + 774144*z[10]*z[11]*z[15]*z[16]*z[8] + 1548288*z[10]*z[11]*z[15]*z[16]*z[9] - 15730688*z[10]*z[11]*z[15]*z[16] - 24576*z[10]*z[11]*z[15]*z[17]*z[7]*z[8] - 49152*z[10]*z[11]*z[15]*z[17]*z[7]*z[9] + 774144*z[10]*z[11]*z[15]*z[17]*z[7] - 98304*z[10]*z[11]*z[15]*z[17]*z[8]*z[9] + 1548288*z[10]*z[11]*z[15]*z[17]*z[8] + 3096576*z[10]*z[11]*z[15]*z[17]*z[9] - 31461376*z[10]*z[11]*z[15]*z[17] - 49152*z[10]*z[11]*z[15]*z[18]*z[7]*z[8] - 98304*z[10]*z[11]*z[15]*z[18]*z[7]*z[9] + 1548288*z[10]*z[11]*z[15]*z[18]*z[7] - 196608*z[10]*z[11]*z[15]*z[18]*z[8]*z[9] + 3096576*z[10]*z[11]*z[15]*z[18]*z[8] + 6193152*z[10]*z[11]*z[15]*z[18]*z[9] - 62922752*z[10]*z[11]*z[15]*z[18] + 96768*z[10]*z[11]*z[15]*z[7]*z[8] + 193536*z[10]*z[11]*z[15]*z[7]*z[9] - 3048192*z[10]*z[11]*z[15]*z[7] + 387072*z[10]*z[11]*z[15]*z[8]*z[9] - 6096384*z[10]*z[11]*z[15]*z[8] - 12192768*z[10]*z[11]*z[15]*z[9] + 123879168*z[10]*z[11]*z[15] - 49152*z[10]*z[11]*z[16]*z[17]*z[7]*z[8] - 98304*z[10]*z[11]*z[16]*z[17]*z[7]*z[9] + 1548288*z[10]*z[11]*z[16]*z[17]*z[7] - 196608*z[10]*z[11]*z[16]*z[17]*z[8]*z[9] + 3096576*z[10]*z[11]*z[16]*z[17]*z[8] + 6193152*z[10]*z[11]*z[16]*z[17]*z[9] - 62922752*z[10]*z[11]*z[16]*z[17] - 98304*z[10]*z[11]*z[16]*z[18]*z[7]*z[8] - 196608*z[10]*z[11]*z[16]*z[18]*z[7]*z[9] + 3096576*z[10]*z[11]*z[16]*z[18]*z[7] - 393216*z[10]*z[11]*z[16]*z[18]*z[8]*z[9] + 6193152*z[10]*z[11]*z[16]*z[18]*z[8] + 12386304*z[10]*z[11]*z[16]*z[18]*z[9] - 125845504*z[10]*z[11]*z[16]*z[18] + 193536*z[10]*z[11]*z[16]*z[7]*z[8] + 387072*z[10]*z[11]*z[16]*z[7]*z[9] - 6096384*z[10]*z[11]*z[16]*z[7] + 774144*z[10]*z[11]*z[16]*z[8]*z[9] - 12192768*z[10]*z[11]*z[16]*z[8] - 24385536*z[10]*z[11]*z[16]*z[9] + 247758336*z[10]*z[11]*z[16] - 196608*z[10]*z[11]*z[17]*z[18]*z[7]*z[8] - 393216*z[10]*z[11]*z[17]*z[18]*z[7]*z[9] + 6193152*z[10]*z[11]*z[17]*z[18]*z[7] - 786432*z[10]*z[11]*z[17]*z[18]*z[8]*z[9] + 12386304*z[10]*z[11]*z[17]*z[18]*z[8] + 24772608*z[10]*z[11]*z[17]*z[18]*z[9] - 251691008*z[10]*z[11]*z[17]*z[18] + 387072*z[10]*z[11]*z[17]*z[7]*z[8] + 774144*z[10]*z[11]*z[17]*z[7]*z[9] - 12192768*z[10]*z[11]*z[17]*z[7] + 1548288*z[10]*z[11]*z[17]*z[8]*z[9] - 24385536*z[10]*z[11]*z[17]*z[8] - 48771072*z[10]*z[11]*z[17]*z[9] + 495516672*z[10]*z[11]*z[17] + 774144*z[10]*z[11]*z[18]*z[7]*z[8] + 1548288*z[10]*z[11]*z[18]*z[7]*z[9] - 24385536*z[10]*z[11]*z[18]*z[7] + 3096576*z[10]*z[11]*z[18]*z[8]*z[9] - 48771072*z[10]*z[11]*z[18]*z[8] - 97542144*z[10]*z[11]*z[18]*z[9] + 991033344*z[10]*z[11]*z[18] + 1179648*z[10]*z[11]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] + 2359296*z[10]*z[11]*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] - 37158912*z[10]*z[11]*z[2]*z[3]*z[4]*z[5]*z[7] + 4718592*z[10]*z[11]*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] - 74317824*z[10]*z[11]*z[2]*z[3]*z[4]*z[5]*z[8] - 148635648*z[10]*z[11]*z[2]*z[3]*z[4]*z[5]*z[9] + 1510146048*z[10]*z[11]*z[2]*z[3]*z[4]*z[5] + 2359296*z[10]*z[11]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] + 4718592*z[10]*z[11]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] - 74317824*z[10]*z[11]*z[2]*z[3]*z[4]*z[6]*z[7] + 9437184*z[10]*z[11]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] - 148635648*z[10]*z[11]*z[2]*z[3]*z[4]*z[6]*z[8] - 297271296*z[10]*z[11]*z[2]*z[3]*z[4]*z[6]*z[9] + 3020292096*z[10]*z[11]*z[2]*z[3]*z[4]*z[6] - 4644864*z[10]*z[11]*z[2]*z[3]*z[4]*z[7]*z[8] - 9289728*z[10]*z[11]*z[2]*z[3]*z[4]*z[7]*z[9] + 146313216*z[10]*z[11]*z[2]*z[3]*z[4]*z[7] - 18579456*z[10]*z[11]*z[2]*z[3]*z[4]*z[8]*z[9] + 292626432*z[10]*z[11]*z[2]*z[3]*z[4]*z[8] + 585252864*z[10]*z[11]*z[2]*z[3]*z[4]*z[9] - 5946200064*z[10]*z[11]*z[2]*z[3]*z[4] + 4718592*z[10]*z[11]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] + 9437184*z[10]*z[11]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] - 148635648*z[10]*z[11]*z[2]*z[3]*z[5]*z[6]*z[7] + 18874368*z[10]*z[11]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] - 297271296*z[10]*z[11]*z[2]*z[3]*z[5]*z[6]*z[8] - 594542592*z[10]*z[11]*z[2]*z[3]*z[5]*z[6]*z[9] + 6040584192*z[10]*z[11]*z[2]*z[3]*z[5]*z[6] - 9289728*z[10]*z[11]*z[2]*z[3]*z[5]*z[7]*z[8] - 18579456*z[10]*z[11]*z[2]*z[3]*z[5]*z[7]*z[9] + 292626432*z[10]*z[11]*z[2]*z[3]*z[5]*z[7] - 37158912*z[10]*z[11]*z[2]*z[3]*z[5]*z[8]*z[9] + 585252864*z[10]*z[11]*z[2]*z[3]*z[5]*z[8] + 1170505728*z[10]*z[11]*z[2]*z[3]*z[5]*z[9] - 11892400128*z[10]*z[11]*z[2]*z[3]*z[5] - 18579456*z[10]*z[11]*z[2]*z[3]*z[6]*z[7]*z[8] - 37158912*z[10]*z[11]*z[2]*z[3]*z[6]*z[7]*z[9] + 585252864*z[10]*z[11]*z[2]*z[3]*z[6]*z[7] - 74317824*z[10]*z[11]*z[2]*z[3]*z[6]*z[8]*z[9] + 1170505728*z[10]*z[11]*z[2]*z[3]*z[6]*z[8] + 2341011456*z[10]*z[11]*z[2]*z[3]*z[6]*z[9] - 23784800256*z[10]*z[11]*z[2]*z[3]*z[6] + 24517632*z[10]*z[11]*z[2]*z[3]*z[7]*z[8] + 49035264*z[10]*z[11]*z[2]*z[3]*z[7]*z[9] - 772305408*z[10]*z[11]*z[2]*z[3]*z[7] + 98070528*z[10]*z[11]*z[2]*z[3]*z[8]*z[9] - 1544610816*z[10]*z[11]*z[2]*z[3]*z[8] - 3089221632*z[10]*z[11]*z[2]*z[3]*z[9] + 31386655232*z[10]*z[11]*z[2]*z[3] + 9437184*z[10]*z[11]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] + 18874368*z[10]*z[11]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] - 297271296*z[10]*z[11]*z[2]*z[4]*z[5]*z[6]*z[7] + 37748736*z[10]*z[11]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] - 594542592*z[10]*z[11]*z[2]*z[4]*z[5]*z[6]*z[8] - 1189085184*z[10]*z[11]*z[2]*z[4]*z[5]*z[6]*z[9] + 12081168384*z[10]*z[11]*z[2]*z[4]*z[5]*z[6] - 18579456*z[10]*z[11]*z[2]*z[4]*z[5]*z[7]*z[8] - 37158912*z[10]*z[11]*z[2]*z[4]*z[5]*z[7]*z[9] + 585252864*z[10]*z[11]*z[2]*z[4]*z[5]*z[7] - 74317824*z[10]*z[11]*z[2]*z[4]*z[5]*z[8]*z[9] + 1170505728*z[10]*z[11]*z[2]*z[4]*z[5]*z[8] + 2341011456*z[10]*z[11]*z[2]*z[4]*z[5]*z[9] - 23784800256*z[10]*z[11]*z[2]*z[4]*z[5] - 37158912*z[10]*z[11]*z[2]*z[4]*z[6]*z[7]*z[8] - 74317824*z[10]*z[11]*z[2]*z[4]*z[6]*z[7]*z[9] + 1170505728*z[10]*z[11]*z[2]*z[4]*z[6]*z[7] - 148635648*z[10]*z[11]*z[2]*z[4]*z[6]*z[8]*z[9] + 2341011456*z[10]*z[11]*z[2]*z[4]*z[6]*z[8] + 4682022912*z[10]*z[11]*z[2]*z[4]*z[6]*z[9] - 47569600512*z[10]*z[11]*z[2]*z[4]*z[6] + 48740352*z[10]*z[11]*z[2]*z[4]*z[7]*z[8] + 97480704*z[10]*z[11]*z[2]*z[4]*z[7]*z[9] - 1535321088*z[10]*z[11]*z[2]*z[4]*z[7] + 194961408*z[10]*z[11]*z[2]*z[4]*z[8]*z[9] - 3070642176*z[10]*z[11]*z[2]*z[4]*z[8] - 6141284352*z[10]*z[11]*z[2]*z[4]*z[9] + 62395773952*z[10]*z[11]*z[2]*z[4] - 74317824*z[10]*z[11]*z[2]*z[5]*z[6]*z[7]*z[8] - 148635648*z[10]*z[11]*z[2]*z[5]*z[6]*z[7]*z[9] + 2341011456*z[10]*z[11]*z[2]*z[5]*z[6]*z[7] - 297271296*z[10]*z[11]*z[2]*z[5]*z[6]*z[8]*z[9] + 4682022912*z[10]*z[11]*z[2]*z[5]*z[6]*z[8] + 9364045824*z[10]*z[11]*z[2]*z[5]*z[6]*z[9] - 95139201024*z[10]*z[11]*z[2]*z[5]*z[6] + 95121408*z[10]*z[11]*z[2]*z[5]*z[7]*z[8] + 190242816*z[10]*z[11]*z[2]*z[5]*z[7]*z[9] - 2996324352*z[10]*z[11]*z[2]*z[5]*z[7] + 380485632*z[10]*z[11]*z[2]*z[5]*z[8]*z[9] - 5992648704*z[10]*z[11]*z[2]*z[5]*z[8] - 11985297408*z[10]*z[11]*z[2]*z[5]*z[9] + 121771255808*z[10]*z[11]*z[2]*z[5] + 171368448*z[10]*z[11]*z[2]*z[6]*z[7]*z[8] + 342736896*z[10]*z[11]*z[2]*z[6]*z[7]*z[9] - 5398106112*z[10]*z[11]*z[2]*z[6]*z[7] + 685473792*z[10]*z[11]*z[2]*z[6]*z[8]*z[9] - 10796212224*z[10]*z[11]*z[2]*z[6]*z[8] - 21592424448*z[10]*z[11]*z[2]*z[6]*z[9] + 219380174848*z[10]*z[11]*z[2]*z[6] - 194890752*z[10]*z[11]*z[2]*z[7]*z[8] - 389781504*z[10]*z[11]*z[2]*z[7]*z[9] + 6139058688*z[10]*z[11]*z[2]*z[7] - 779563008*z[10]*z[11]*z[2]*z[8]*z[9] + 12278117376*z[10]*z[11]*z[2]*z[8] + 24556234752*z[10]*z[11]*z[2]*z[9] - 249492644352*z[10]*z[11]*z[2] + 18874368*z[10]*z[11]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 37748736*z[10]*z[11]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] - 594542592*z[10]*z[11]*z[3]*z[4]*z[5]*z[6]*z[7] + 75497472*z[10]*z[11]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] - 1189085184*z[10]*z[11]*z[3]*z[4]*z[5]*z[6]*z[8] - 2378170368*z[10]*z[11]*z[3]*z[4]*z[5]*z[6]*z[9] + 24162336768*z[10]*z[11]*z[3]*z[4]*z[5]*z[6] - 37158912*z[10]*z[11]*z[3]*z[4]*z[5]*z[7]*z[8] - 74317824*z[10]*z[11]*z[3]*z[4]*z[5]*z[7]*z[9] + 1170505728*z[10]*z[11]*z[3]*z[4]*z[5]*z[7] - 148635648*z[10]*z[11]*z[3]*z[4]*z[5]*z[8]*z[9] + 2341011456*z[10]*z[11]*z[3]*z[4]*z[5]*z[8] + 4682022912*z[10]*z[11]*z[3]*z[4]*z[5]*z[9] - 47569600512*z[10]*z[11]*z[3]*z[4]*z[5] - 74317824*z[10]*z[11]*z[3]*z[4]*z[6]*z[7]*z[8] - 148635648*z[10]*z[11]*z[3]*z[4]*z[6]*z[7]*z[9] + 2341011456*z[10]*z[11]*z[3]*z[4]*z[6]*z[7] - 297271296*z[10]*z[11]*z[3]*z[4]*z[6]*z[8]*z[9] + 4682022912*z[10]*z[11]*z[3]*z[4]*z[6]*z[8] + 9364045824*z[10]*z[11]*z[3]*z[4]*z[6]*z[9] - 95139201024*z[10]*z[11]*z[3]*z[4]*z[6] + 97333248*z[10]*z[11]*z[3]*z[4]*z[7]*z[8] + 194666496*z[10]*z[11]*z[3]*z[4]*z[7]*z[9] - 3065997312*z[10]*z[11]*z[3]*z[4]*z[7] + 389332992*z[10]*z[11]*z[3]*z[4]*z[8]*z[9] - 6131994624*z[10]*z[11]*z[3]*z[4]*z[8] - 12263989248*z[10]*z[11]*z[3]*z[4]*z[9] + 124602779648*z[10]*z[11]*z[3]*z[4] - 148635648*z[10]*z[11]*z[3]*z[5]*z[6]*z[7]*z[8] - 297271296*z[10]*z[11]*z[3]*z[5]*z[6]*z[7]*z[9] + 4682022912*z[10]*z[11]*z[3]*z[5]*z[6]*z[7] - 594542592*z[10]*z[11]*z[3]*z[5]*z[6]*z[8]*z[9] + 9364045824*z[10]*z[11]*z[3]*z[5]*z[6]*z[8] + 18728091648*z[10]*z[11]*z[3]*z[5]*z[6]*z[9] - 190278402048*z[10]*z[11]*z[3]*z[5]*z[6] + 189947904*z[10]*z[11]*z[3]*z[5]*z[7]*z[8] + 379895808*z[10]*z[11]*z[3]*z[5]*z[7]*z[9] - 5983358976*z[10]*z[11]*z[3]*z[5]*z[7] + 759791616*z[10]*z[11]*z[3]*z[5]*z[8]*z[9] - 11966717952*z[10]*z[11]*z[3]*z[5]*z[8] - 23933435904*z[10]*z[11]*z[3]*z[5]*z[9] + 243164975104*z[10]*z[11]*z[3]*z[5] + 342147072*z[10]*z[11]*z[3]*z[6]*z[7]*z[8] + 684294144*z[10]*z[11]*z[3]*z[6]*z[7]*z[9] - 10777632768*z[10]*z[11]*z[3]*z[6]*z[7] + 1368588288*z[10]*z[11]*z[3]*z[6]*z[8]*z[9] - 21555265536*z[10]*z[11]*z[3]*z[6]*z[8] - 43110531072*z[10]*z[11]*z[3]*z[6]*z[9] + 438005276672*z[10]*z[11]*z[3]*z[6] - 388620288*z[10]*z[11]*z[3]*z[7]*z[8] - 777240576*z[10]*z[11]*z[3]*z[7]*z[9] + 12241539072*z[10]*z[11]*z[3]*z[7] - 1554481152*z[10]*z[11]*z[3]*z[8]*z[9] + 24483078144*z[10]*z[11]*z[3]*z[8] + 48966156288*z[10]*z[11]*z[3]*z[9] - 497498738688*z[10]*z[11]*z[3] - 297271296*z[10]*z[11]*z[4]*z[5]*z[6]*z[7]*z[8] - 594542592*z[10]*z[11]*z[4]*z[5]*z[6]*z[7]*z[9] + 9364045824*z[10]*z[11]*z[4]*z[5]*z[6]*z[7] - 1189085184*z[10]*z[11]*z[4]*z[5]*z[6]*z[8]*z[9] + 18728091648*z[10]*z[11]*z[4]*z[5]*z[6]*z[8] + 37456183296*z[10]*z[11]*z[4]*z[5]*z[6]*z[9] - 380556804096*z[10]*z[11]*z[4]*z[5]*z[6] + 377536512*z[10]*z[11]*z[4]*z[5]*z[7]*z[8] + 755073024*z[10]*z[11]*z[4]*z[5]*z[7]*z[9] - 11892400128*z[10]*z[11]*z[4]*z[5]*z[7] + 1510146048*z[10]*z[11]*z[4]*z[5]*z[8]*z[9] - 23784800256*z[10]*z[11]*z[4]*z[5]*z[8] - 47569600512*z[10]*z[11]*z[4]*z[5]*z[9] + 483309658112*z[10]*z[11]*z[4]*z[5] + 679575552*z[10]*z[11]*z[4]*z[6]*z[7]*z[8] + 1359151104*z[10]*z[11]*z[4]*z[6]*z[7]*z[9] - 21406629888*z[10]*z[11]*z[4]*z[6]*z[7] + 2718302208*z[10]*z[11]*z[4]*z[6]*z[8]*z[9] - 42813259776*z[10]*z[11]*z[4]*z[6]*z[8] - 85626519552*z[10]*z[11]*z[4]*z[6]*z[9] + 869969969152*z[10]*z[11]*z[4]*z[6] - 767950848*z[10]*z[11]*z[4]*z[7]*z[8] - 1535901696*z[10]*z[11]*z[4]*z[7]*z[9] + 24190451712*z[10]*z[11]*z[4]*z[7] - 3071803392*z[10]*z[11]*z[4]*z[8]*z[9] + 48380903424*z[10]*z[11]*z[4]*z[8] + 96761806848*z[10]*z[11]*z[4]*z[9] - 983105077248*z[10]*z[11]*z[4] + 1321402368*z[10]*z[11]*z[5]*z[6]*z[7]*z[8] + 2642804736*z[10]*z[11]*z[5]*z[6]*z[7]*z[9] - 41624174592*z[10]*z[11]*z[5]*z[6]*z[7] + 5285609472*z[10]*z[11]*z[5]*z[6]*z[8]*z[9] - 83248349184*z[10]*z[11]*z[5]*z[6]*z[8] - 166496698368*z[10]*z[11]*z[5]*z[6]*z[9] + 1691615264768*z[10]*z[11]*z[5]*z[6] - 1461583872*z[10]*z[11]*z[5]*z[7]*z[8] - 2923167744*z[10]*z[11]*z[5]*z[7]*z[9] + 46039891968*z[10]*z[11]*z[5]*z[7] - 5846335488*z[10]*z[11]*z[5]*z[8]*z[9] + 92079783936*z[10]*z[11]*z[5]*z[8] + 184159567872*z[10]*z[11]*z[5]*z[9] - 1871070953472*z[10]*z[11]*z[5] - 2328625152*z[10]*z[11]*z[6]*z[7]*z[8] - 4657250304*z[10]*z[11]*z[6]*z[7]*z[9] + 73351692288*z[10]*z[11]*z[6]*z[7] - 9314500608*z[10]*z[11]*z[6]*z[8]*z[9] + 146703384576*z[10]*z[11]*z[6]*z[8] + 293406769152*z[10]*z[11]*z[6]*z[9] - 2981028298752*z[10]*z[11]*z[6] - 12500974080*z[10]*z[11]*z[7]*z[8]*z[9] + 77989702656*z[10]*z[11]*z[7]*z[8] + 155156232192*z[10]*z[11]*z[7]*z[9] - 827069007744*z[10]*z[11]*z[7] + 309900716544*z[10]*z[11]*z[8]*z[9] - 1651000071168*z[10]*z[11]*z[8] - 3277018515456*z[10]*z[11]*z[9] + 15764686966656*z[10]*z[11] - 1536*z[10]*z[12]*z[13]*z[14]*z[7]*z[8] - 3072*z[10]*z[12]*z[13]*z[14]*z[7]*z[9] + 48384*z[10]*z[12]*z[13]*z[14]*z[7] - 6144*z[10]*z[12]*z[13]*z[14]*z[8]*z[9] + 96768*z[10]*z[12]*z[13]*z[14]*z[8] + 193536*z[10]*z[12]*z[13]*z[14]*z[9] - 1769728*z[10]*z[12]*z[13]*z[14] - 3072*z[10]*z[12]*z[13]*z[15]*z[7]*z[8] - 6144*z[10]*z[12]*z[13]*z[15]*z[7]*z[9] + 96768*z[10]*z[12]*z[13]*z[15]*z[7] - 12288*z[10]*z[12]*z[13]*z[15]*z[8]*z[9] + 193536*z[10]*z[12]*z[13]*z[15]*z[8] + 387072*z[10]*z[12]*z[13]*z[15]*z[9] - 3539456*z[10]*z[12]*z[13]*z[15] - 6144*z[10]*z[12]*z[13]*z[16]*z[7]*z[8] - 12288*z[10]*z[12]*z[13]*z[16]*z[7]*z[9] + 193536*z[10]*z[12]*z[13]*z[16]*z[7] - 24576*z[10]*z[12]*z[13]*z[16]*z[8]*z[9] + 387072*z[10]*z[12]*z[13]*z[16]*z[8] + 774144*z[10]*z[12]*z[13]*z[16]*z[9] - 7078912*z[10]*z[12]*z[13]*z[16] - 12288*z[10]*z[12]*z[13]*z[17]*z[7]*z[8] - 24576*z[10]*z[12]*z[13]*z[17]*z[7]*z[9] + 387072*z[10]*z[12]*z[13]*z[17]*z[7] - 49152*z[10]*z[12]*z[13]*z[17]*z[8]*z[9] + 774144*z[10]*z[12]*z[13]*z[17]*z[8] + 1548288*z[10]*z[12]*z[13]*z[17]*z[9] - 14157824*z[10]*z[12]*z[13]*z[17] - 24576*z[10]*z[12]*z[13]*z[18]*z[7]*z[8] - 49152*z[10]*z[12]*z[13]*z[18]*z[7]*z[9] + 774144*z[10]*z[12]*z[13]*z[18]*z[7] - 98304*z[10]*z[12]*z[13]*z[18]*z[8]*z[9] + 1548288*z[10]*z[12]*z[13]*z[18]*z[8] + 3096576*z[10]*z[12]*z[13]*z[18]*z[9] - 28315648*z[10]*z[12]*z[13]*z[18] + 48384*z[10]*z[12]*z[13]*z[7]*z[8] + 96768*z[10]*z[12]*z[13]*z[7]*z[9] - 1524096*z[10]*z[12]*z[13]*z[7] + 193536*z[10]*z[12]*z[13]*z[8]*z[9] - 3048192*z[10]*z[12]*z[13]*z[8] - 6096384*z[10]*z[12]*z[13]*z[9] + 55746432*z[10]*z[12]*z[13] - 6144*z[10]*z[12]*z[14]*z[15]*z[7]*z[8] - 12288*z[10]*z[12]*z[14]*z[15]*z[7]*z[9] + 193536*z[10]*z[12]*z[14]*z[15]*z[7] - 24576*z[10]*z[12]*z[14]*z[15]*z[8]*z[9] + 387072*z[10]*z[12]*z[14]*z[15]*z[8] + 774144*z[10]*z[12]*z[14]*z[15]*z[9] - 7078912*z[10]*z[12]*z[14]*z[15] - 12288*z[10]*z[12]*z[14]*z[16]*z[7]*z[8] - 24576*z[10]*z[12]*z[14]*z[16]*z[7]*z[9] + 387072*z[10]*z[12]*z[14]*z[16]*z[7] - 49152*z[10]*z[12]*z[14]*z[16]*z[8]*z[9] + 774144*z[10]*z[12]*z[14]*z[16]*z[8] + 1548288*z[10]*z[12]*z[14]*z[16]*z[9] - 14157824*z[10]*z[12]*z[14]*z[16] - 24576*z[10]*z[12]*z[14]*z[17]*z[7]*z[8] - 49152*z[10]*z[12]*z[14]*z[17]*z[7]*z[9] + 774144*z[10]*z[12]*z[14]*z[17]*z[7] - 98304*z[10]*z[12]*z[14]*z[17]*z[8]*z[9] + 1548288*z[10]*z[12]*z[14]*z[17]*z[8] + 3096576*z[10]*z[12]*z[14]*z[17]*z[9] - 28315648*z[10]*z[12]*z[14]*z[17] - 49152*z[10]*z[12]*z[14]*z[18]*z[7]*z[8] - 98304*z[10]*z[12]*z[14]*z[18]*z[7]*z[9] + 1548288*z[10]*z[12]*z[14]*z[18]*z[7] - 196608*z[10]*z[12]*z[14]*z[18]*z[8]*z[9] + 3096576*z[10]*z[12]*z[14]*z[18]*z[8] + 6193152*z[10]*z[12]*z[14]*z[18]*z[9] - 56631296*z[10]*z[12]*z[14]*z[18] + 96768*z[10]*z[12]*z[14]*z[7]*z[8] + 193536*z[10]*z[12]*z[14]*z[7]*z[9] - 3048192*z[10]*z[12]*z[14]*z[7] + 387072*z[10]*z[12]*z[14]*z[8]*z[9] - 6096384*z[10]*z[12]*z[14]*z[8] - 12192768*z[10]*z[12]*z[14]*z[9] + 111492864*z[10]*z[12]*z[14] - 24576*z[10]*z[12]*z[15]*z[16]*z[7]*z[8] - 49152*z[10]*z[12]*z[15]*z[16]*z[7]*z[9] + 774144*z[10]*z[12]*z[15]*z[16]*z[7] - 98304*z[10]*z[12]*z[15]*z[16]*z[8]*z[9] + 1548288*z[10]*z[12]*z[15]*z[16]*z[8] + 3096576*z[10]*z[12]*z[15]*z[16]*z[9] - 28315648*z[10]*z[12]*z[15]*z[16] - 49152*z[10]*z[12]*z[15]*z[17]*z[7]*z[8] - 98304*z[10]*z[12]*z[15]*z[17]*z[7]*z[9] + 1548288*z[10]*z[12]*z[15]*z[17]*z[7] - 196608*z[10]*z[12]*z[15]*z[17]*z[8]*z[9] + 3096576*z[10]*z[12]*z[15]*z[17]*z[8] + 6193152*z[10]*z[12]*z[15]*z[17]*z[9] - 56631296*z[10]*z[12]*z[15]*z[17] - 98304*z[10]*z[12]*z[15]*z[18]*z[7]*z[8] - 196608*z[10]*z[12]*z[15]*z[18]*z[7]*z[9] + 3096576*z[10]*z[12]*z[15]*z[18]*z[7] - 393216*z[10]*z[12]*z[15]*z[18]*z[8]*z[9] + 6193152*z[10]*z[12]*z[15]*z[18]*z[8] + 12386304*z[10]*z[12]*z[15]*z[18]*z[9] - 113262592*z[10]*z[12]*z[15]*z[18] + 193536*z[10]*z[12]*z[15]*z[7]*z[8] + 387072*z[10]*z[12]*z[15]*z[7]*z[9] - 6096384*z[10]*z[12]*z[15]*z[7] + 774144*z[10]*z[12]*z[15]*z[8]*z[9] - 12192768*z[10]*z[12]*z[15]*z[8] - 24385536*z[10]*z[12]*z[15]*z[9] + 222985728*z[10]*z[12]*z[15] - 98304*z[10]*z[12]*z[16]*z[17]*z[7]*z[8] - 196608*z[10]*z[12]*z[16]*z[17]*z[7]*z[9] + 3096576*z[10]*z[12]*z[16]*z[17]*z[7] - 393216*z[10]*z[12]*z[16]*z[17]*z[8]*z[9] + 6193152*z[10]*z[12]*z[16]*z[17]*z[8] + 12386304*z[10]*z[12]*z[16]*z[17]*z[9] - 113262592*z[10]*z[12]*z[16]*z[17] - 196608*z[10]*z[12]*z[16]*z[18]*z[7]*z[8] - 393216*z[10]*z[12]*z[16]*z[18]*z[7]*z[9] + 6193152*z[10]*z[12]*z[16]*z[18]*z[7] - 786432*z[10]*z[12]*z[16]*z[18]*z[8]*z[9] + 12386304*z[10]*z[12]*z[16]*z[18]*z[8] + 24772608*z[10]*z[12]*z[16]*z[18]*z[9] - 226525184*z[10]*z[12]*z[16]*z[18] + 387072*z[10]*z[12]*z[16]*z[7]*z[8] + 774144*z[10]*z[12]*z[16]*z[7]*z[9] - 12192768*z[10]*z[12]*z[16]*z[7] + 1548288*z[10]*z[12]*z[16]*z[8]*z[9] - 24385536*z[10]*z[12]*z[16]*z[8] - 48771072*z[10]*z[12]*z[16]*z[9] + 445971456*z[10]*z[12]*z[16] - 393216*z[10]*z[12]*z[17]*z[18]*z[7]*z[8] - 786432*z[10]*z[12]*z[17]*z[18]*z[7]*z[9] + 12386304*z[10]*z[12]*z[17]*z[18]*z[7] - 1572864*z[10]*z[12]*z[17]*z[18]*z[8]*z[9] + 24772608*z[10]*z[12]*z[17]*z[18]*z[8] + 49545216*z[10]*z[12]*z[17]*z[18]*z[9] - 453050368*z[10]*z[12]*z[17]*z[18] + 774144*z[10]*z[12]*z[17]*z[7]*z[8] + 1548288*z[10]*z[12]*z[17]*z[7]*z[9] - 24385536*z[10]*z[12]*z[17]*z[7] + 3096576*z[10]*z[12]*z[17]*z[8]*z[9] - 48771072*z[10]*z[12]*z[17]*z[8] - 97542144*z[10]*z[12]*z[17]*z[9] + 891942912*z[10]*z[12]*z[17] + 1548288*z[10]*z[12]*z[18]*z[7]*z[8] + 3096576*z[10]*z[12]*z[18]*z[7]*z[9] - 48771072*z[10]*z[12]*z[18]*z[7] + 6193152*z[10]*z[12]*z[18]*z[8]*z[9] - 97542144*z[10]*z[12]*z[18]*z[8] - 195084288*z[10]*z[12]*z[18]*z[9] + 1783885824*z[10]*z[12]*z[18] + 2359296*z[10]*z[12]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] + 4718592*z[10]*z[12]*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] - 74317824*z[10]*z[12]*z[2]*z[3]*z[4]*z[5]*z[7] + 9437184*z[10]*z[12]*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] - 148635648*z[10]*z[12]*z[2]*z[3]*z[4]*z[5]*z[8] - 297271296*z[10]*z[12]*z[2]*z[3]*z[4]*z[5]*z[9] + 2718302208*z[10]*z[12]*z[2]*z[3]*z[4]*z[5] + 4718592*z[10]*z[12]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] + 9437184*z[10]*z[12]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] - 148635648*z[10]*z[12]*z[2]*z[3]*z[4]*z[6]*z[7] + 18874368*z[10]*z[12]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] - 297271296*z[10]*z[12]*z[2]*z[3]*z[4]*z[6]*z[8] - 594542592*z[10]*z[12]*z[2]*z[3]*z[4]*z[6]*z[9] + 5436604416*z[10]*z[12]*z[2]*z[3]*z[4]*z[6] - 9289728*z[10]*z[12]*z[2]*z[3]*z[4]*z[7]*z[8] - 18579456*z[10]*z[12]*z[2]*z[3]*z[4]*z[7]*z[9] + 292626432*z[10]*z[12]*z[2]*z[3]*z[4]*z[7] - 37158912*z[10]*z[12]*z[2]*z[3]*z[4]*z[8]*z[9] + 585252864*z[10]*z[12]*z[2]*z[3]*z[4]*z[8] + 1170505728*z[10]*z[12]*z[2]*z[3]*z[4]*z[9] - 10703314944*z[10]*z[12]*z[2]*z[3]*z[4] + 9437184*z[10]*z[12]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] + 18874368*z[10]*z[12]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] - 297271296*z[10]*z[12]*z[2]*z[3]*z[5]*z[6]*z[7] + 37748736*z[10]*z[12]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] - 594542592*z[10]*z[12]*z[2]*z[3]*z[5]*z[6]*z[8] - 1189085184*z[10]*z[12]*z[2]*z[3]*z[5]*z[6]*z[9] + 10873208832*z[10]*z[12]*z[2]*z[3]*z[5]*z[6] - 18579456*z[10]*z[12]*z[2]*z[3]*z[5]*z[7]*z[8] - 37158912*z[10]*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] + 585252864*z[10]*z[12]*z[2]*z[3]*z[5]*z[7] - 74317824*z[10]*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] + 1170505728*z[10]*z[12]*z[2]*z[3]*z[5]*z[8] + 2341011456*z[10]*z[12]*z[2]*z[3]*z[5]*z[9] - 21406629888*z[10]*z[12]*z[2]*z[3]*z[5] - 37158912*z[10]*z[12]*z[2]*z[3]*z[6]*z[7]*z[8] - 74317824*z[10]*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] + 1170505728*z[10]*z[12]*z[2]*z[3]*z[6]*z[7] - 148635648*z[10]*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] + 2341011456*z[10]*z[12]*z[2]*z[3]*z[6]*z[8] + 4682022912*z[10]*z[12]*z[2]*z[3]*z[6]*z[9] - 42813259776*z[10]*z[12]*z[2]*z[3]*z[6] + 49035264*z[10]*z[12]*z[2]*z[3]*z[7]*z[8] + 98070528*z[10]*z[12]*z[2]*z[3]*z[7]*z[9] - 1544610816*z[10]*z[12]*z[2]*z[3]*z[7] + 196141056*z[10]*z[12]*z[2]*z[3]*z[8]*z[9] - 3089221632*z[10]*z[12]*z[2]*z[3]*z[8] - 6178443264*z[10]*z[12]*z[2]*z[3]*z[9] + 56496796672*z[10]*z[12]*z[2]*z[3] + 18874368*z[10]*z[12]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] + 37748736*z[10]*z[12]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] - 594542592*z[10]*z[12]*z[2]*z[4]*z[5]*z[6]*z[7] + 75497472*z[10]*z[12]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] - 1189085184*z[10]*z[12]*z[2]*z[4]*z[5]*z[6]*z[8] - 2378170368*z[10]*z[12]*z[2]*z[4]*z[5]*z[6]*z[9] + 21746417664*z[10]*z[12]*z[2]*z[4]*z[5]*z[6] - 37158912*z[10]*z[12]*z[2]*z[4]*z[5]*z[7]*z[8] - 74317824*z[10]*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] + 1170505728*z[10]*z[12]*z[2]*z[4]*z[5]*z[7] - 148635648*z[10]*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] + 2341011456*z[10]*z[12]*z[2]*z[4]*z[5]*z[8] + 4682022912*z[10]*z[12]*z[2]*z[4]*z[5]*z[9] - 42813259776*z[10]*z[12]*z[2]*z[4]*z[5] - 74317824*z[10]*z[12]*z[2]*z[4]*z[6]*z[7]*z[8] - 148635648*z[10]*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] + 2341011456*z[10]*z[12]*z[2]*z[4]*z[6]*z[7] - 297271296*z[10]*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] + 4682022912*z[10]*z[12]*z[2]*z[4]*z[6]*z[8] + 9364045824*z[10]*z[12]*z[2]*z[4]*z[6]*z[9] - 85626519552*z[10]*z[12]*z[2]*z[4]*z[6] + 97480704*z[10]*z[12]*z[2]*z[4]*z[7]*z[8] + 194961408*z[10]*z[12]*z[2]*z[4]*z[7]*z[9] - 3070642176*z[10]*z[12]*z[2]*z[4]*z[7] + 389922816*z[10]*z[12]*z[2]*z[4]*z[8]*z[9] - 6141284352*z[10]*z[12]*z[2]*z[4]*z[8] - 12282568704*z[10]*z[12]*z[2]*z[4]*z[9] + 112314017792*z[10]*z[12]*z[2]*z[4] - 148635648*z[10]*z[12]*z[2]*z[5]*z[6]*z[7]*z[8] - 297271296*z[10]*z[12]*z[2]*z[5]*z[6]*z[7]*z[9] + 4682022912*z[10]*z[12]*z[2]*z[5]*z[6]*z[7] - 594542592*z[10]*z[12]*z[2]*z[5]*z[6]*z[8]*z[9] + 9364045824*z[10]*z[12]*z[2]*z[5]*z[6]*z[8] + 18728091648*z[10]*z[12]*z[2]*z[5]*z[6]*z[9] - 171253039104*z[10]*z[12]*z[2]*z[5]*z[6] + 190242816*z[10]*z[12]*z[2]*z[5]*z[7]*z[8] + 380485632*z[10]*z[12]*z[2]*z[5]*z[7]*z[9] - 5992648704*z[10]*z[12]*z[2]*z[5]*z[7] + 760971264*z[10]*z[12]*z[2]*z[5]*z[8]*z[9] - 11985297408*z[10]*z[12]*z[2]*z[5]*z[8] - 23970594816*z[10]*z[12]*z[2]*z[5]*z[9] + 219191431168*z[10]*z[12]*z[2]*z[5] + 342736896*z[10]*z[12]*z[2]*z[6]*z[7]*z[8] + 685473792*z[10]*z[12]*z[2]*z[6]*z[7]*z[9] - 10796212224*z[10]*z[12]*z[2]*z[6]*z[7] + 1370947584*z[10]*z[12]*z[2]*z[6]*z[8]*z[9] - 21592424448*z[10]*z[12]*z[2]*z[6]*z[8] - 43184848896*z[10]*z[12]*z[2]*z[6]*z[9] + 394890027008*z[10]*z[12]*z[2]*z[6] - 389781504*z[10]*z[12]*z[2]*z[7]*z[8] - 779563008*z[10]*z[12]*z[2]*z[7]*z[9] + 12278117376*z[10]*z[12]*z[2]*z[7] - 1559126016*z[10]*z[12]*z[2]*z[8]*z[9] + 24556234752*z[10]*z[12]*z[2]*z[8] + 49112469504*z[10]*z[12]*z[2]*z[9] - 449093256192*z[10]*z[12]*z[2] + 37748736*z[10]*z[12]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 75497472*z[10]*z[12]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] - 1189085184*z[10]*z[12]*z[3]*z[4]*z[5]*z[6]*z[7] + 150994944*z[10]*z[12]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] - 2378170368*z[10]*z[12]*z[3]*z[4]*z[5]*z[6]*z[8] - 4756340736*z[10]*z[12]*z[3]*z[4]*z[5]*z[6]*z[9] + 43492835328*z[10]*z[12]*z[3]*z[4]*z[5]*z[6] - 74317824*z[10]*z[12]*z[3]*z[4]*z[5]*z[7]*z[8] - 148635648*z[10]*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] + 2341011456*z[10]*z[12]*z[3]*z[4]*z[5]*z[7] - 297271296*z[10]*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] + 4682022912*z[10]*z[12]*z[3]*z[4]*z[5]*z[8] + 9364045824*z[10]*z[12]*z[3]*z[4]*z[5]*z[9] - 85626519552*z[10]*z[12]*z[3]*z[4]*z[5] - 148635648*z[10]*z[12]*z[3]*z[4]*z[6]*z[7]*z[8] - 297271296*z[10]*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] + 4682022912*z[10]*z[12]*z[3]*z[4]*z[6]*z[7] - 594542592*z[10]*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] + 9364045824*z[10]*z[12]*z[3]*z[4]*z[6]*z[8] + 18728091648*z[10]*z[12]*z[3]*z[4]*z[6]*z[9] - 171253039104*z[10]*z[12]*z[3]*z[4]*z[6] + 194666496*z[10]*z[12]*z[3]*z[4]*z[7]*z[8] + 389332992*z[10]*z[12]*z[3]*z[4]*z[7]*z[9] - 6131994624*z[10]*z[12]*z[3]*z[4]*z[7] + 778665984*z[10]*z[12]*z[3]*z[4]*z[8]*z[9] - 12263989248*z[10]*z[12]*z[3]*z[4]*z[8] - 24527978496*z[10]*z[12]*z[3]*z[4]*z[9] + 224288247808*z[10]*z[12]*z[3]*z[4] - 297271296*z[10]*z[12]*z[3]*z[5]*z[6]*z[7]*z[8] - 594542592*z[10]*z[12]*z[3]*z[5]*z[6]*z[7]*z[9] + 9364045824*z[10]*z[12]*z[3]*z[5]*z[6]*z[7] - 1189085184*z[10]*z[12]*z[3]*z[5]*z[6]*z[8]*z[9] + 18728091648*z[10]*z[12]*z[3]*z[5]*z[6]*z[8] + 37456183296*z[10]*z[12]*z[3]*z[5]*z[6]*z[9] - 342506078208*z[10]*z[12]*z[3]*z[5]*z[6] + 379895808*z[10]*z[12]*z[3]*z[5]*z[7]*z[8] + 759791616*z[10]*z[12]*z[3]*z[5]*z[7]*z[9] - 11966717952*z[10]*z[12]*z[3]*z[5]*z[7] + 1519583232*z[10]*z[12]*z[3]*z[5]*z[8]*z[9] - 23933435904*z[10]*z[12]*z[3]*z[5]*z[8] - 47866871808*z[10]*z[12]*z[3]*z[5]*z[9] + 437703286784*z[10]*z[12]*z[3]*z[5] + 684294144*z[10]*z[12]*z[3]*z[6]*z[7]*z[8] + 1368588288*z[10]*z[12]*z[3]*z[6]*z[7]*z[9] - 21555265536*z[10]*z[12]*z[3]*z[6]*z[7] + 2737176576*z[10]*z[12]*z[3]*z[6]*z[8]*z[9] - 43110531072*z[10]*z[12]*z[3]*z[6]*z[8] - 86221062144*z[10]*z[12]*z[3]*z[6]*z[9] + 788420902912*z[10]*z[12]*z[3]*z[6] - 777240576*z[10]*z[12]*z[3]*z[7]*z[8] - 1554481152*z[10]*z[12]*z[3]*z[7]*z[9] + 24483078144*z[10]*z[12]*z[3]*z[7] - 3108962304*z[10]*z[12]*z[3]*z[8]*z[9] + 48966156288*z[10]*z[12]*z[3]*z[8] + 97932312576*z[10]*z[12]*z[3]*z[9] - 895510683648*z[10]*z[12]*z[3] - 594542592*z[10]*z[12]*z[4]*z[5]*z[6]*z[7]*z[8] - 1189085184*z[10]*z[12]*z[4]*z[5]*z[6]*z[7]*z[9] + 18728091648*z[10]*z[12]*z[4]*z[5]*z[6]*z[7] - 2378170368*z[10]*z[12]*z[4]*z[5]*z[6]*z[8]*z[9] + 37456183296*z[10]*z[12]*z[4]*z[5]*z[6]*z[8] + 74912366592*z[10]*z[12]*z[4]*z[5]*z[6]*z[9] - 685012156416*z[10]*z[12]*z[4]*z[5]*z[6] + 755073024*z[10]*z[12]*z[4]*z[5]*z[7]*z[8] + 1510146048*z[10]*z[12]*z[4]*z[5]*z[7]*z[9] - 23784800256*z[10]*z[12]*z[4]*z[5]*z[7] + 3020292096*z[10]*z[12]*z[4]*z[5]*z[8]*z[9] - 47569600512*z[10]*z[12]*z[4]*z[5]*z[8] - 95139201024*z[10]*z[12]*z[4]*z[5]*z[9] + 869969969152*z[10]*z[12]*z[4]*z[5] + 1359151104*z[10]*z[12]*z[4]*z[6]*z[7]*z[8] + 2718302208*z[10]*z[12]*z[4]*z[6]*z[7]*z[9] - 42813259776*z[10]*z[12]*z[4]*z[6]*z[7] + 5436604416*z[10]*z[12]*z[4]*z[6]*z[8]*z[9] - 85626519552*z[10]*z[12]*z[4]*z[6]*z[8] - 171253039104*z[10]*z[12]*z[4]*z[6]*z[9] + 1565968596992*z[10]*z[12]*z[4]*z[6] - 1535901696*z[10]*z[12]*z[4]*z[7]*z[8] - 3071803392*z[10]*z[12]*z[4]*z[7]*z[9] + 48380903424*z[10]*z[12]*z[4]*z[7] - 6143606784*z[10]*z[12]*z[4]*z[8]*z[9] + 96761806848*z[10]*z[12]*z[4]*z[8] + 193523613696*z[10]*z[12]*z[4]*z[9] - 1769614737408*z[10]*z[12]*z[4] + 2642804736*z[10]*z[12]*z[5]*z[6]*z[7]*z[8] + 5285609472*z[10]*z[12]*z[5]*z[6]*z[7]*z[9] - 83248349184*z[10]*z[12]*z[5]*z[6]*z[7] + 10571218944*z[10]*z[12]*z[5]*z[6]*z[8]*z[9] - 166496698368*z[10]*z[12]*z[5]*z[6]*z[8] - 332993396736*z[10]*z[12]*z[5]*z[6]*z[9] + 3044951523328*z[10]*z[12]*z[5]*z[6] - 2923167744*z[10]*z[12]*z[5]*z[7]*z[8] - 5846335488*z[10]*z[12]*z[5]*z[7]*z[9] + 92079783936*z[10]*z[12]*z[5]*z[7] - 11692670976*z[10]*z[12]*z[5]*z[8]*z[9] + 184159567872*z[10]*z[12]*z[5]*z[8] + 368319135744*z[10]*z[12]*z[5]*z[9] - 3367976435712*z[10]*z[12]*z[5] - 4657250304*z[10]*z[12]*z[6]*z[7]*z[8] - 9314500608*z[10]*z[12]*z[6]*z[7]*z[9] + 146703384576*z[10]*z[12]*z[6]*z[7] - 18629001216*z[10]*z[12]*z[6]*z[8]*z[9] + 293406769152*z[10]*z[12]*z[6]*z[8] + 586813538304*z[10]*z[12]*z[6]*z[9] - 5365928558592*z[10]*z[12]*z[6] - 19799700480*z[10]*z[12]*z[7]*z[8]*z[9] + 111966738432*z[10]*z[12]*z[7]*z[8] + 222452281344*z[10]*z[12]*z[7]*z[9] - 1127193677568*z[10]*z[12]*z[7] + 444163642368*z[10]*z[12]*z[8]*z[9] - 2249412028416*z[10]*z[12]*z[8] - 4459265298432*z[10]*z[12]*z[9] + 20961798233856*z[10]*z[12] - 192*z[10]*z[13]*z[14]*z[7]*z[8]*z[9] + 3024*z[10]*z[13]*z[14]*z[7]*z[8] + 6048*z[10]*z[13]*z[14]*z[7]*z[9] - 63488*z[10]*z[13]*z[14]*z[7] + 12096*z[10]*z[13]*z[14]*z[8]*z[9] - 126928*z[10]*z[13]*z[14]*z[8] - 253472*z[10]*z[13]*z[14]*z[9] + 1999872*z[10]*z[13]*z[14] - 384*z[10]*z[13]*z[15]*z[7]*z[8]*z[9] + 6048*z[10]*z[13]*z[15]*z[7]*z[8] + 12096*z[10]*z[13]*z[15]*z[7]*z[9] - 126976*z[10]*z[13]*z[15]*z[7] + 24192*z[10]*z[13]*z[15]*z[8]*z[9] - 253856*z[10]*z[13]*z[15]*z[8] - 506944*z[10]*z[13]*z[15]*z[9] + 3999744*z[10]*z[13]*z[15] - 768*z[10]*z[13]*z[16]*z[7]*z[8]*z[9] + 12096*z[10]*z[13]*z[16]*z[7]*z[8] + 24192*z[10]*z[13]*z[16]*z[7]*z[9] - 253952*z[10]*z[13]*z[16]*z[7] + 48384*z[10]*z[13]*z[16]*z[8]*z[9] - 507712*z[10]*z[13]*z[16]*z[8] - 1013888*z[10]*z[13]*z[16]*z[9] + 7999488*z[10]*z[13]*z[16] - 1536*z[10]*z[13]*z[17]*z[7]*z[8]*z[9] + 24192*z[10]*z[13]*z[17]*z[7]*z[8] + 48384*z[10]*z[13]*z[17]*z[7]*z[9] - 507904*z[10]*z[13]*z[17]*z[7] + 96768*z[10]*z[13]*z[17]*z[8]*z[9] - 1015424*z[10]*z[13]*z[17]*z[8] - 2027776*z[10]*z[13]*z[17]*z[9] + 15998976*z[10]*z[13]*z[17] - 3072*z[10]*z[13]*z[18]*z[7]*z[8]*z[9] + 48384*z[10]*z[13]*z[18]*z[7]*z[8] + 96768*z[10]*z[13]*z[18]*z[7]*z[9] - 1015808*z[10]*z[13]*z[18]*z[7] + 193536*z[10]*z[13]*z[18]*z[8]*z[9] - 2030848*z[10]*z[13]*z[18]*z[8] - 4055552*z[10]*z[13]*z[18]*z[9] + 31997952*z[10]*z[13]*z[18] + 6048*z[10]*z[13]*z[7]*z[8]*z[9] - 95256*z[10]*z[13]*z[7]*z[8] - 190512*z[10]*z[13]*z[7]*z[9] + 1999872*z[10]*z[13]*z[7] - 381024*z[10]*z[13]*z[8]*z[9] + 3998232*z[10]*z[13]*z[8] + 7984368*z[10]*z[13]*z[9] - 62995968*z[10]*z[13] - 768*z[10]*z[14]*z[15]*z[7]*z[8]*z[9] + 12096*z[10]*z[14]*z[15]*z[7]*z[8] + 24192*z[10]*z[14]*z[15]*z[7]*z[9] - 253952*z[10]*z[14]*z[15]*z[7] + 48384*z[10]*z[14]*z[15]*z[8]*z[9] - 507712*z[10]*z[14]*z[15]*z[8] - 1013888*z[10]*z[14]*z[15]*z[9] + 7999488*z[10]*z[14]*z[15] - 1536*z[10]*z[14]*z[16]*z[7]*z[8]*z[9] + 24192*z[10]*z[14]*z[16]*z[7]*z[8] + 48384*z[10]*z[14]*z[16]*z[7]*z[9] - 507904*z[10]*z[14]*z[16]*z[7] + 96768*z[10]*z[14]*z[16]*z[8]*z[9] - 1015424*z[10]*z[14]*z[16]*z[8] - 2027776*z[10]*z[14]*z[16]*z[9] + 15998976*z[10]*z[14]*z[16] - 3072*z[10]*z[14]*z[17]*z[7]*z[8]*z[9] + 48384*z[10]*z[14]*z[17]*z[7]*z[8] + 96768*z[10]*z[14]*z[17]*z[7]*z[9] - 1015808*z[10]*z[14]*z[17]*z[7] + 193536*z[10]*z[14]*z[17]*z[8]*z[9] - 2030848*z[10]*z[14]*z[17]*z[8] - 4055552*z[10]*z[14]*z[17]*z[9] + 31997952*z[10]*z[14]*z[17] - 6144*z[10]*z[14]*z[18]*z[7]*z[8]*z[9] + 96768*z[10]*z[14]*z[18]*z[7]*z[8] + 193536*z[10]*z[14]*z[18]*z[7]*z[9] - 2031616*z[10]*z[14]*z[18]*z[7] + 387072*z[10]*z[14]*z[18]*z[8]*z[9] - 4061696*z[10]*z[14]*z[18]*z[8] - 8111104*z[10]*z[14]*z[18]*z[9] + 63995904*z[10]*z[14]*z[18] + 12096*z[10]*z[14]*z[7]*z[8]*z[9] - 190512*z[10]*z[14]*z[7]*z[8] - 381024*z[10]*z[14]*z[7]*z[9] + 3999744*z[10]*z[14]*z[7] - 762048*z[10]*z[14]*z[8]*z[9] + 7996464*z[10]*z[14]*z[8] + 15968736*z[10]*z[14]*z[9] - 125991936*z[10]*z[14] - 3072*z[10]*z[15]*z[16]*z[7]*z[8]*z[9] + 48384*z[10]*z[15]*z[16]*z[7]*z[8] + 96768*z[10]*z[15]*z[16]*z[7]*z[9] - 1015808*z[10]*z[15]*z[16]*z[7] + 193536*z[10]*z[15]*z[16]*z[8]*z[9] - 2030848*z[10]*z[15]*z[16]*z[8] - 4055552*z[10]*z[15]*z[16]*z[9] + 31997952*z[10]*z[15]*z[16] - 6144*z[10]*z[15]*z[17]*z[7]*z[8]*z[9] + 96768*z[10]*z[15]*z[17]*z[7]*z[8] + 193536*z[10]*z[15]*z[17]*z[7]*z[9] - 2031616*z[10]*z[15]*z[17]*z[7] + 387072*z[10]*z[15]*z[17]*z[8]*z[9] - 4061696*z[10]*z[15]*z[17]*z[8] - 8111104*z[10]*z[15]*z[17]*z[9] + 63995904*z[10]*z[15]*z[17] - 12288*z[10]*z[15]*z[18]*z[7]*z[8]*z[9] + 193536*z[10]*z[15]*z[18]*z[7]*z[8] + 387072*z[10]*z[15]*z[18]*z[7]*z[9] - 4063232*z[10]*z[15]*z[18]*z[7] + 774144*z[10]*z[15]*z[18]*z[8]*z[9] - 8123392*z[10]*z[15]*z[18]*z[8] - 16222208*z[10]*z[15]*z[18]*z[9] + 127991808*z[10]*z[15]*z[18] + 24192*z[10]*z[15]*z[7]*z[8]*z[9] - 381024*z[10]*z[15]*z[7]*z[8] - 762048*z[10]*z[15]*z[7]*z[9] + 7999488*z[10]*z[15]*z[7] - 1524096*z[10]*z[15]*z[8]*z[9] + 15992928*z[10]*z[15]*z[8] + 31937472*z[10]*z[15]*z[9] - 251983872*z[10]*z[15] - 12288*z[10]*z[16]*z[17]*z[7]*z[8]*z[9] + 193536*z[10]*z[16]*z[17]*z[7]*z[8] + 387072*z[10]*z[16]*z[17]*z[7]*z[9] - 4063232*z[10]*z[16]*z[17]*z[7] + 774144*z[10]*z[16]*z[17]*z[8]*z[9] - 8123392*z[10]*z[16]*z[17]*z[8] - 16222208*z[10]*z[16]*z[17]*z[9] + 127991808*z[10]*z[16]*z[17] - 24576*z[10]*z[16]*z[18]*z[7]*z[8]*z[9] + 387072*z[10]*z[16]*z[18]*z[7]*z[8] + 774144*z[10]*z[16]*z[18]*z[7]*z[9] - 8126464*z[10]*z[16]*z[18]*z[7] + 1548288*z[10]*z[16]*z[18]*z[8]*z[9] - 16246784*z[10]*z[16]*z[18]*z[8] - 32444416*z[10]*z[16]*z[18]*z[9] + 255983616*z[10]*z[16]*z[18] + 48384*z[10]*z[16]*z[7]*z[8]*z[9] - 762048*z[10]*z[16]*z[7]*z[8] - 1524096*z[10]*z[16]*z[7]*z[9] + 15998976*z[10]*z[16]*z[7] - 3048192*z[10]*z[16]*z[8]*z[9] + 31985856*z[10]*z[16]*z[8] + 63874944*z[10]*z[16]*z[9] - 503967744*z[10]*z[16] - 49152*z[10]*z[17]*z[18]*z[7]*z[8]*z[9] + 774144*z[10]*z[17]*z[18]*z[7]*z[8] + 1548288*z[10]*z[17]*z[18]*z[7]*z[9] - 16252928*z[10]*z[17]*z[18]*z[7] + 3096576*z[10]*z[17]*z[18]*z[8]*z[9] - 32493568*z[10]*z[17]*z[18]*z[8] - 64888832*z[10]*z[17]*z[18]*z[9] + 511967232*z[10]*z[17]*z[18] + 96768*z[10]*z[17]*z[7]*z[8]*z[9] - 1524096*z[10]*z[17]*z[7]*z[8] - 3048192*z[10]*z[17]*z[7]*z[9] + 31997952*z[10]*z[17]*z[7] - 6096384*z[10]*z[17]*z[8]*z[9] + 63971712*z[10]*z[17]*z[8] + 127749888*z[10]*z[17]*z[9] - 1007935488*z[10]*z[17] + 193536*z[10]*z[18]*z[7]*z[8]*z[9] - 3048192*z[10]*z[18]*z[7]*z[8] - 6096384*z[10]*z[18]*z[7]*z[9] + 63995904*z[10]*z[18]*z[7] - 12192768*z[10]*z[18]*z[8]*z[9] + 127943424*z[10]*z[18]*z[8] + 255499776*z[10]*z[18]*z[9] - 2015870976*z[10]*z[18] + 294912*z[10]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 4644864*z[10]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] - 9289728*z[10]*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] + 97517568*z[10]*z[2]*z[3]*z[4]*z[5]*z[7] - 18579456*z[10]*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] + 194961408*z[10]*z[2]*z[3]*z[4]*z[5]*z[8] + 389332992*z[10]*z[2]*z[3]*z[4]*z[5]*z[9] - 3071803392*z[10]*z[2]*z[3]*z[4]*z[5] + 589824*z[10]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 9289728*z[10]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] - 18579456*z[10]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] + 195035136*z[10]*z[2]*z[3]*z[4]*z[6]*z[7] - 37158912*z[10]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] + 389922816*z[10]*z[2]*z[3]*z[4]*z[6]*z[8] + 778665984*z[10]*z[2]*z[3]*z[4]*z[6]*z[9] - 6143606784*z[10]*z[2]*z[3]*z[4]*z[6] - 1161216*z[10]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] + 18289152*z[10]*z[2]*z[3]*z[4]*z[7]*z[8] + 36578304*z[10]*z[2]*z[3]*z[4]*z[7]*z[9] - 383975424*z[10]*z[2]*z[3]*z[4]*z[7] + 73156608*z[10]*z[2]*z[3]*z[4]*z[8]*z[9] - 767660544*z[10]*z[2]*z[3]*z[4]*z[8] - 1532998656*z[10]*z[2]*z[3]*z[4]*z[9] + 12095225856*z[10]*z[2]*z[3]*z[4] + 1179648*z[10]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 18579456*z[10]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] - 37158912*z[10]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] + 390070272*z[10]*z[2]*z[3]*z[5]*z[6]*z[7] - 74317824*z[10]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] + 779845632*z[10]*z[2]*z[3]*z[5]*z[6]*z[8] + 1557331968*z[10]*z[2]*z[3]*z[5]*z[6]*z[9] - 12287213568*z[10]*z[2]*z[3]*z[5]*z[6] - 2322432*z[10]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] + 36578304*z[10]*z[2]*z[3]*z[5]*z[7]*z[8] + 73156608*z[10]*z[2]*z[3]*z[5]*z[7]*z[9] - 767950848*z[10]*z[2]*z[3]*z[5]*z[7] + 146313216*z[10]*z[2]*z[3]*z[5]*z[8]*z[9] - 1535321088*z[10]*z[2]*z[3]*z[5]*z[8] - 3065997312*z[10]*z[2]*z[3]*z[5]*z[9] + 24190451712*z[10]*z[2]*z[3]*z[5] - 4644864*z[10]*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] + 73156608*z[10]*z[2]*z[3]*z[6]*z[7]*z[8] + 146313216*z[10]*z[2]*z[3]*z[6]*z[7]*z[9] - 1535901696*z[10]*z[2]*z[3]*z[6]*z[7] + 292626432*z[10]*z[2]*z[3]*z[6]*z[8]*z[9] - 3070642176*z[10]*z[2]*z[3]*z[6]*z[8] - 6131994624*z[10]*z[2]*z[3]*z[6]*z[9] + 48380903424*z[10]*z[2]*z[3]*z[6] + 6129408*z[10]*z[2]*z[3]*z[7]*z[8]*z[9] - 96538176*z[10]*z[2]*z[3]*z[7]*z[8] - 193076352*z[10]*z[2]*z[3]*z[7]*z[9] + 2026790912*z[10]*z[2]*z[3]*z[7] - 386152704*z[10]*z[2]*z[3]*z[8]*z[9] + 4052049472*z[10]*z[2]*z[3]*z[8] + 8091840128*z[10]*z[2]*z[3]*z[9] - 63843913728*z[10]*z[2]*z[3] + 2359296*z[10]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 37158912*z[10]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] - 74317824*z[10]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] + 780140544*z[10]*z[2]*z[4]*z[5]*z[6]*z[7] - 148635648*z[10]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] + 1559691264*z[10]*z[2]*z[4]*z[5]*z[6]*z[8] + 3114663936*z[10]*z[2]*z[4]*z[5]*z[6]*z[9] - 24574427136*z[10]*z[2]*z[4]*z[5]*z[6] - 4644864*z[10]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] + 73156608*z[10]*z[2]*z[4]*z[5]*z[7]*z[8] + 146313216*z[10]*z[2]*z[4]*z[5]*z[7]*z[9] - 1535901696*z[10]*z[2]*z[4]*z[5]*z[7] + 292626432*z[10]*z[2]*z[4]*z[5]*z[8]*z[9] - 3070642176*z[10]*z[2]*z[4]*z[5]*z[8] - 6131994624*z[10]*z[2]*z[4]*z[5]*z[9] + 48380903424*z[10]*z[2]*z[4]*z[5] - 9289728*z[10]*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] + 146313216*z[10]*z[2]*z[4]*z[6]*z[7]*z[8] + 292626432*z[10]*z[2]*z[4]*z[6]*z[7]*z[9] - 3071803392*z[10]*z[2]*z[4]*z[6]*z[7] + 585252864*z[10]*z[2]*z[4]*z[6]*z[8]*z[9] - 6141284352*z[10]*z[2]*z[4]*z[6]*z[8] - 12263989248*z[10]*z[2]*z[4]*z[6]*z[9] + 96761806848*z[10]*z[2]*z[4]*z[6] + 12185088*z[10]*z[2]*z[4]*z[7]*z[8]*z[9] - 191915136*z[10]*z[2]*z[4]*z[7]*z[8] - 383830272*z[10]*z[2]*z[4]*z[7]*z[9] + 4029202432*z[10]*z[2]*z[4]*z[7] - 767660544*z[10]*z[2]*z[4]*z[8]*z[9] + 8055358592*z[10]*z[2]*z[4]*z[8] + 16086347008*z[10]*z[2]*z[4]*z[9] - 126919876608*z[10]*z[2]*z[4] - 18579456*z[10]*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] + 292626432*z[10]*z[2]*z[5]*z[6]*z[7]*z[8] + 585252864*z[10]*z[2]*z[5]*z[6]*z[7]*z[9] - 6143606784*z[10]*z[2]*z[5]*z[6]*z[7] + 1170505728*z[10]*z[2]*z[5]*z[6]*z[8]*z[9] - 12282568704*z[10]*z[2]*z[5]*z[6]*z[8] - 24527978496*z[10]*z[2]*z[5]*z[6]*z[9] + 193523613696*z[10]*z[2]*z[5]*z[6] + 23780352*z[10]*z[2]*z[5]*z[7]*z[8]*z[9] - 374540544*z[10]*z[2]*z[5]*z[7]*z[8] - 749081088*z[10]*z[2]*z[5]*z[7]*z[9] + 7863369728*z[10]*z[2]*z[5]*z[7] - 1498162176*z[10]*z[2]*z[5]*z[8]*z[9] + 15720794368*z[10]*z[2]*z[5]*z[8] + 31394028032*z[10]*z[2]*z[5]*z[9] - 247696146432*z[10]*z[2]*z[5] + 42842112*z[10]*z[2]*z[6]*z[7]*z[8]*z[9] - 674763264*z[10]*z[2]*z[6]*z[7]*z[8] - 1349526528*z[10]*z[2]*z[6]*z[7]*z[9] + 14166458368*z[10]*z[2]*z[6]*z[7] - 2699053056*z[10]*z[2]*z[6]*z[8]*z[9] + 28322206208*z[10]*z[2]*z[6]*z[8] + 56558728192*z[10]*z[2]*z[6]*z[9] - 446243438592*z[10]*z[2]*z[6] - 48722688*z[10]*z[2]*z[7]*z[8]*z[9] + 767382336*z[10]*z[2]*z[7]*z[8] + 1534764672*z[10]*z[2]*z[7]*z[9] - 16110968832*z[10]*z[2]*z[7] + 3069529344*z[10]*z[2]*z[8]*z[9] - 32209756992*z[10]*z[2]*z[8] - 64322068608*z[10]*z[2]*z[9] + 507495518208*z[10]*z[2] + 4718592*z[10]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 74317824*z[10]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] - 148635648*z[10]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] + 1560281088*z[10]*z[3]*z[4]*z[5]*z[6]*z[7] - 297271296*z[10]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] + 3119382528*z[10]*z[3]*z[4]*z[5]*z[6]*z[8] + 6229327872*z[10]*z[3]*z[4]*z[5]*z[6]*z[9] - 49148854272*z[10]*z[3]*z[4]*z[5]*z[6] - 9289728*z[10]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] + 146313216*z[10]*z[3]*z[4]*z[5]*z[7]*z[8] + 292626432*z[10]*z[3]*z[4]*z[5]*z[7]*z[9] - 3071803392*z[10]*z[3]*z[4]*z[5]*z[7] + 585252864*z[10]*z[3]*z[4]*z[5]*z[8]*z[9] - 6141284352*z[10]*z[3]*z[4]*z[5]*z[8] - 12263989248*z[10]*z[3]*z[4]*z[5]*z[9] + 96761806848*z[10]*z[3]*z[4]*z[5] - 18579456*z[10]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] + 292626432*z[10]*z[3]*z[4]*z[6]*z[7]*z[8] + 585252864*z[10]*z[3]*z[4]*z[6]*z[7]*z[9] - 6143606784*z[10]*z[3]*z[4]*z[6]*z[7] + 1170505728*z[10]*z[3]*z[4]*z[6]*z[8]*z[9] - 12282568704*z[10]*z[3]*z[4]*z[6]*z[8] - 24527978496*z[10]*z[3]*z[4]*z[6]*z[9] + 193523613696*z[10]*z[3]*z[4]*z[6] + 24333312*z[10]*z[3]*z[4]*z[7]*z[8]*z[9] - 383249664*z[10]*z[3]*z[4]*z[7]*z[8] - 766499328*z[10]*z[3]*z[4]*z[7]*z[9] + 8046215168*z[10]*z[3]*z[4]*z[7] - 1532998656*z[10]*z[3]*z[4]*z[8]*z[9] + 16086347008*z[10]*z[3]*z[4]*z[8] + 32124027392*z[10]*z[3]*z[4]*z[9] - 253455777792*z[10]*z[3]*z[4] - 37158912*z[10]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] + 585252864*z[10]*z[3]*z[5]*z[6]*z[7]*z[8] + 1170505728*z[10]*z[3]*z[5]*z[6]*z[7]*z[9] - 12287213568*z[10]*z[3]*z[5]*z[6]*z[7] + 2341011456*z[10]*z[3]*z[5]*z[6]*z[8]*z[9] - 24565137408*z[10]*z[3]*z[5]*z[6]*z[8] - 49055956992*z[10]*z[3]*z[5]*z[6]*z[9] + 387047227392*z[10]*z[3]*z[5]*z[6] + 47486976*z[10]*z[3]*z[5]*z[7]*z[8]*z[9] - 747919872*z[10]*z[3]*z[5]*z[7]*z[8] - 1495839744*z[10]*z[3]*z[5]*z[7]*z[9] + 15702360064*z[10]*z[3]*z[5]*z[7] - 2991679488*z[10]*z[3]*z[5]*z[8]*z[9] + 31392848384*z[10]*z[3]*z[5]*z[8] + 62690722816*z[10]*z[3]*z[5]*z[9] - 494624342016*z[10]*z[3]*z[5] + 85536768*z[10]*z[3]*z[6]*z[7]*z[8]*z[9] - 1347204096*z[10]*z[3]*z[6]*z[7]*z[8] - 2694408192*z[10]*z[3]*z[6]*z[7]*z[9] + 28284157952*z[10]*z[3]*z[6]*z[7] - 5388816384*z[10]*z[3]*z[6]*z[8]*z[9] + 56546931712*z[10]*z[3]*z[6]*z[8] + 112922789888*z[10]*z[3]*z[6]*z[9] - 890950975488*z[10]*z[3]*z[6] - 97155072*z[10]*z[3]*z[7]*z[8]*z[9] + 1530192384*z[10]*z[3]*z[7]*z[8] + 3060384768*z[10]*z[3]*z[7]*z[9] - 32125943808*z[10]*z[3]*z[7] + 6120769536*z[10]*z[3]*z[8]*z[9] - 64227598848*z[10]*z[3]*z[8] - 128260887552*z[10]*z[3]*z[9] + 1011967229952*z[10]*z[3] - 74317824*z[10]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 1170505728*z[10]*z[4]*z[5]*z[6]*z[7]*z[8] + 2341011456*z[10]*z[4]*z[5]*z[6]*z[7]*z[9] - 24574427136*z[10]*z[4]*z[5]*z[6]*z[7] + 4682022912*z[10]*z[4]*z[5]*z[6]*z[8]*z[9] - 49130274816*z[10]*z[4]*z[5]*z[6]*z[8] - 98111913984*z[10]*z[4]*z[5]*z[6]*z[9] + 774094454784*z[10]*z[4]*z[5]*z[6] + 94384128*z[10]*z[4]*z[5]*z[7]*z[8]*z[9] - 1486550016*z[10]*z[4]*z[5]*z[7]*z[8] - 2973100032*z[10]*z[4]*z[5]*z[7]*z[9] + 31209684992*z[10]*z[4]*z[5]*z[7] - 5946200064*z[10]*z[4]*z[5]*z[8]*z[9] + 62395773952*z[10]*z[4]*z[5]*z[8] + 124602779648*z[10]*z[4]*z[5]*z[9] - 983105077248*z[10]*z[4]*z[5] + 169893888*z[10]*z[4]*z[6]*z[7]*z[8]*z[9] - 2675828736*z[10]*z[4]*z[6]*z[7]*z[8] - 5351657472*z[10]*z[4]*z[6]*z[7]*z[9] + 56178245632*z[10]*z[4]*z[6]*z[7] - 10703314944*z[10]*z[4]*z[6]*z[8]*z[9] + 112314017792*z[10]*z[4]*z[6]*z[8] + 224288247808*z[10]*z[4]*z[6]*z[9] - 1769614737408*z[10]*z[4]*z[6] - 191987712*z[10]*z[4]*z[7]*z[8]*z[9] + 3023806464*z[10]*z[4]*z[7]*z[8] + 6047612928*z[10]*z[4]*z[7]*z[9] - 63483936768*z[10]*z[4]*z[7] + 12095225856*z[10]*z[4]*z[8]*z[9] - 126919876608*z[10]*z[4]*z[8] - 253455777792*z[10]*z[4]*z[9] + 1999744008192*z[10]*z[4] + 330350592*z[10]*z[5]*z[6]*z[7]*z[8]*z[9] - 5203021824*z[10]*z[5]*z[6]*z[7]*z[8] - 10406043648*z[10]*z[5]*z[6]*z[7]*z[9] + 109235929088*z[10]*z[5]*z[6]*z[7] - 20812087296*z[10]*z[5]*z[6]*z[8]*z[9] + 218389270528*z[10]*z[5]*z[6]*z[8] + 436117839872*z[10]*z[5]*z[6]*z[9] - 3440931766272*z[10]*z[5]*z[6] - 365395968*z[10]*z[5]*z[7]*z[8]*z[9] + 5754986496*z[10]*z[5]*z[7]*z[8] + 11509972992*z[10]*z[5]*z[7]*z[9] - 120824266752*z[10]*z[5]*z[7] + 23019945984*z[10]*z[5]*z[8]*z[9] - 241557184512*z[10]*z[5]*z[8] - 482383577088*z[10]*z[5]*z[9] + 3805964402688*z[10]*z[5] - 582156288*z[10]*z[6]*z[7]*z[8]*z[9] + 9168961536*z[10]*z[6]*z[7]*z[8] + 18337923072*z[10]*z[6]*z[7]*z[9] - 192499679232*z[10]*z[6]*z[7] + 36675846144*z[10]*z[6]*z[8]*z[9] - 384853819392*z[10]*z[6]*z[8] - 768543326208*z[10]*z[6]*z[9] + 6063739895808*z[10]*z[6] + 21541810944*z[10]*z[7]*z[8]*z[9] - 119631124368*z[10]*z[7]*z[8] - 237592940256*z[10]*z[7]*z[9] + 1188165613056*z[10]*z[7] - 474350591232*z[10]*z[8]*z[9] + 2370912544656*z[10]*z[8] + 4698794629344*z[10]*z[9] - 21794835654144*z[10] - 3072*z[11]*z[12]*z[13]*z[14]*z[7]*z[8] - 6144*z[11]*z[12]*z[13]*z[14]*z[7]*z[9] + 96768*z[11]*z[12]*z[13]*z[14]*z[7] - 12288*z[11]*z[12]*z[13]*z[14]*z[8]*z[9] + 193536*z[11]*z[12]*z[13]*z[14]*z[8] + 387072*z[11]*z[12]*z[13]*z[14]*z[9] - 3441152*z[11]*z[12]*z[13]*z[14] - 6144*z[11]*z[12]*z[13]*z[15]*z[7]*z[8] - 12288*z[11]*z[12]*z[13]*z[15]*z[7]*z[9] + 193536*z[11]*z[12]*z[13]*z[15]*z[7] - 24576*z[11]*z[12]*z[13]*z[15]*z[8]*z[9] + 387072*z[11]*z[12]*z[13]*z[15]*z[8] + 774144*z[11]*z[12]*z[13]*z[15]*z[9] - 6882304*z[11]*z[12]*z[13]*z[15] - 12288*z[11]*z[12]*z[13]*z[16]*z[7]*z[8] - 24576*z[11]*z[12]*z[13]*z[16]*z[7]*z[9] + 387072*z[11]*z[12]*z[13]*z[16]*z[7] - 49152*z[11]*z[12]*z[13]*z[16]*z[8]*z[9] + 774144*z[11]*z[12]*z[13]*z[16]*z[8] + 1548288*z[11]*z[12]*z[13]*z[16]*z[9] - 13764608*z[11]*z[12]*z[13]*z[16] - 24576*z[11]*z[12]*z[13]*z[17]*z[7]*z[8] - 49152*z[11]*z[12]*z[13]*z[17]*z[7]*z[9] + 774144*z[11]*z[12]*z[13]*z[17]*z[7] - 98304*z[11]*z[12]*z[13]*z[17]*z[8]*z[9] + 1548288*z[11]*z[12]*z[13]*z[17]*z[8] + 3096576*z[11]*z[12]*z[13]*z[17]*z[9] - 27529216*z[11]*z[12]*z[13]*z[17] - 49152*z[11]*z[12]*z[13]*z[18]*z[7]*z[8] - 98304*z[11]*z[12]*z[13]*z[18]*z[7]*z[9] + 1548288*z[11]*z[12]*z[13]*z[18]*z[7] - 196608*z[11]*z[12]*z[13]*z[18]*z[8]*z[9] + 3096576*z[11]*z[12]*z[13]*z[18]*z[8] + 6193152*z[11]*z[12]*z[13]*z[18]*z[9] - 55058432*z[11]*z[12]*z[13]*z[18] + 96768*z[11]*z[12]*z[13]*z[7]*z[8] + 193536*z[11]*z[12]*z[13]*z[7]*z[9] - 3048192*z[11]*z[12]*z[13]*z[7] + 387072*z[11]*z[12]*z[13]*z[8]*z[9] - 6096384*z[11]*z[12]*z[13]*z[8] - 12192768*z[11]*z[12]*z[13]*z[9] + 108396288*z[11]*z[12]*z[13] - 12288*z[11]*z[12]*z[14]*z[15]*z[7]*z[8] - 24576*z[11]*z[12]*z[14]*z[15]*z[7]*z[9] + 387072*z[11]*z[12]*z[14]*z[15]*z[7] - 49152*z[11]*z[12]*z[14]*z[15]*z[8]*z[9] + 774144*z[11]*z[12]*z[14]*z[15]*z[8] + 1548288*z[11]*z[12]*z[14]*z[15]*z[9] - 13764608*z[11]*z[12]*z[14]*z[15] - 24576*z[11]*z[12]*z[14]*z[16]*z[7]*z[8] - 49152*z[11]*z[12]*z[14]*z[16]*z[7]*z[9] + 774144*z[11]*z[12]*z[14]*z[16]*z[7] - 98304*z[11]*z[12]*z[14]*z[16]*z[8]*z[9] + 1548288*z[11]*z[12]*z[14]*z[16]*z[8] + 3096576*z[11]*z[12]*z[14]*z[16]*z[9] - 27529216*z[11]*z[12]*z[14]*z[16] - 49152*z[11]*z[12]*z[14]*z[17]*z[7]*z[8] - 98304*z[11]*z[12]*z[14]*z[17]*z[7]*z[9] + 1548288*z[11]*z[12]*z[14]*z[17]*z[7] - 196608*z[11]*z[12]*z[14]*z[17]*z[8]*z[9] + 3096576*z[11]*z[12]*z[14]*z[17]*z[8] + 6193152*z[11]*z[12]*z[14]*z[17]*z[9] - 55058432*z[11]*z[12]*z[14]*z[17] - 98304*z[11]*z[12]*z[14]*z[18]*z[7]*z[8] - 196608*z[11]*z[12]*z[14]*z[18]*z[7]*z[9] + 3096576*z[11]*z[12]*z[14]*z[18]*z[7] - 393216*z[11]*z[12]*z[14]*z[18]*z[8]*z[9] + 6193152*z[11]*z[12]*z[14]*z[18]*z[8] + 12386304*z[11]*z[12]*z[14]*z[18]*z[9] - 110116864*z[11]*z[12]*z[14]*z[18] + 193536*z[11]*z[12]*z[14]*z[7]*z[8] + 387072*z[11]*z[12]*z[14]*z[7]*z[9] - 6096384*z[11]*z[12]*z[14]*z[7] + 774144*z[11]*z[12]*z[14]*z[8]*z[9] - 12192768*z[11]*z[12]*z[14]*z[8] - 24385536*z[11]*z[12]*z[14]*z[9] + 216792576*z[11]*z[12]*z[14] - 49152*z[11]*z[12]*z[15]*z[16]*z[7]*z[8] - 98304*z[11]*z[12]*z[15]*z[16]*z[7]*z[9] + 1548288*z[11]*z[12]*z[15]*z[16]*z[7] - 196608*z[11]*z[12]*z[15]*z[16]*z[8]*z[9] + 3096576*z[11]*z[12]*z[15]*z[16]*z[8] + 6193152*z[11]*z[12]*z[15]*z[16]*z[9] - 55058432*z[11]*z[12]*z[15]*z[16] - 98304*z[11]*z[12]*z[15]*z[17]*z[7]*z[8] - 196608*z[11]*z[12]*z[15]*z[17]*z[7]*z[9] + 3096576*z[11]*z[12]*z[15]*z[17]*z[7] - 393216*z[11]*z[12]*z[15]*z[17]*z[8]*z[9] + 6193152*z[11]*z[12]*z[15]*z[17]*z[8] + 12386304*z[11]*z[12]*z[15]*z[17]*z[9] - 110116864*z[11]*z[12]*z[15]*z[17] - 196608*z[11]*z[12]*z[15]*z[18]*z[7]*z[8] - 393216*z[11]*z[12]*z[15]*z[18]*z[7]*z[9] + 6193152*z[11]*z[12]*z[15]*z[18]*z[7] - 786432*z[11]*z[12]*z[15]*z[18]*z[8]*z[9] + 12386304*z[11]*z[12]*z[15]*z[18]*z[8] + 24772608*z[11]*z[12]*z[15]*z[18]*z[9] - 220233728*z[11]*z[12]*z[15]*z[18] + 387072*z[11]*z[12]*z[15]*z[7]*z[8] + 774144*z[11]*z[12]*z[15]*z[7]*z[9] - 12192768*z[11]*z[12]*z[15]*z[7] + 1548288*z[11]*z[12]*z[15]*z[8]*z[9] - 24385536*z[11]*z[12]*z[15]*z[8] - 48771072*z[11]*z[12]*z[15]*z[9] + 433585152*z[11]*z[12]*z[15] - 196608*z[11]*z[12]*z[16]*z[17]*z[7]*z[8] - 393216*z[11]*z[12]*z[16]*z[17]*z[7]*z[9] + 6193152*z[11]*z[12]*z[16]*z[17]*z[7] - 786432*z[11]*z[12]*z[16]*z[17]*z[8]*z[9] + 12386304*z[11]*z[12]*z[16]*z[17]*z[8] + 24772608*z[11]*z[12]*z[16]*z[17]*z[9] - 220233728*z[11]*z[12]*z[16]*z[17] - 393216*z[11]*z[12]*z[16]*z[18]*z[7]*z[8] - 786432*z[11]*z[12]*z[16]*z[18]*z[7]*z[9] + 12386304*z[11]*z[12]*z[16]*z[18]*z[7] - 1572864*z[11]*z[12]*z[16]*z[18]*z[8]*z[9] + 24772608*z[11]*z[12]*z[16]*z[18]*z[8] + 49545216*z[11]*z[12]*z[16]*z[18]*z[9] - 440467456*z[11]*z[12]*z[16]*z[18] + 774144*z[11]*z[12]*z[16]*z[7]*z[8] + 1548288*z[11]*z[12]*z[16]*z[7]*z[9] - 24385536*z[11]*z[12]*z[16]*z[7] + 3096576*z[11]*z[12]*z[16]*z[8]*z[9] - 48771072*z[11]*z[12]*z[16]*z[8] - 97542144*z[11]*z[12]*z[16]*z[9] + 867170304*z[11]*z[12]*z[16] - 786432*z[11]*z[12]*z[17]*z[18]*z[7]*z[8] - 1572864*z[11]*z[12]*z[17]*z[18]*z[7]*z[9] + 24772608*z[11]*z[12]*z[17]*z[18]*z[7] - 3145728*z[11]*z[12]*z[17]*z[18]*z[8]*z[9] + 49545216*z[11]*z[12]*z[17]*z[18]*z[8] + 99090432*z[11]*z[12]*z[17]*z[18]*z[9] - 880934912*z[11]*z[12]*z[17]*z[18] + 1548288*z[11]*z[12]*z[17]*z[7]*z[8] + 3096576*z[11]*z[12]*z[17]*z[7]*z[9] - 48771072*z[11]*z[12]*z[17]*z[7] + 6193152*z[11]*z[12]*z[17]*z[8]*z[9] - 97542144*z[11]*z[12]*z[17]*z[8] - 195084288*z[11]*z[12]*z[17]*z[9] + 1734340608*z[11]*z[12]*z[17] + 3096576*z[11]*z[12]*z[18]*z[7]*z[8] + 6193152*z[11]*z[12]*z[18]*z[7]*z[9] - 97542144*z[11]*z[12]*z[18]*z[7] + 12386304*z[11]*z[12]*z[18]*z[8]*z[9] - 195084288*z[11]*z[12]*z[18]*z[8] - 390168576*z[11]*z[12]*z[18]*z[9] + 3468681216*z[11]*z[12]*z[18] + 4718592*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] + 9437184*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] - 148635648*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[7] + 18874368*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] - 297271296*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[8] - 594542592*z[11]*z[12]*z[2]*z[3]*z[4]*z[5]*z[9] + 5285609472*z[11]*z[12]*z[2]*z[3]*z[4]*z[5] + 9437184*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] + 18874368*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] - 297271296*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[7] + 37748736*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] - 594542592*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[8] - 1189085184*z[11]*z[12]*z[2]*z[3]*z[4]*z[6]*z[9] + 10571218944*z[11]*z[12]*z[2]*z[3]*z[4]*z[6] - 18579456*z[11]*z[12]*z[2]*z[3]*z[4]*z[7]*z[8] - 37158912*z[11]*z[12]*z[2]*z[3]*z[4]*z[7]*z[9] + 585252864*z[11]*z[12]*z[2]*z[3]*z[4]*z[7] - 74317824*z[11]*z[12]*z[2]*z[3]*z[4]*z[8]*z[9] + 1170505728*z[11]*z[12]*z[2]*z[3]*z[4]*z[8] + 2341011456*z[11]*z[12]*z[2]*z[3]*z[4]*z[9] - 20812087296*z[11]*z[12]*z[2]*z[3]*z[4] + 18874368*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] + 37748736*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] - 594542592*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[7] + 75497472*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] - 1189085184*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[8] - 2378170368*z[11]*z[12]*z[2]*z[3]*z[5]*z[6]*z[9] + 21142437888*z[11]*z[12]*z[2]*z[3]*z[5]*z[6] - 37158912*z[11]*z[12]*z[2]*z[3]*z[5]*z[7]*z[8] - 74317824*z[11]*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] + 1170505728*z[11]*z[12]*z[2]*z[3]*z[5]*z[7] - 148635648*z[11]*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] + 2341011456*z[11]*z[12]*z[2]*z[3]*z[5]*z[8] + 4682022912*z[11]*z[12]*z[2]*z[3]*z[5]*z[9] - 41624174592*z[11]*z[12]*z[2]*z[3]*z[5] - 74317824*z[11]*z[12]*z[2]*z[3]*z[6]*z[7]*z[8] - 148635648*z[11]*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] + 2341011456*z[11]*z[12]*z[2]*z[3]*z[6]*z[7] - 297271296*z[11]*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] + 4682022912*z[11]*z[12]*z[2]*z[3]*z[6]*z[8] + 9364045824*z[11]*z[12]*z[2]*z[3]*z[6]*z[9] - 83248349184*z[11]*z[12]*z[2]*z[3]*z[6] + 98070528*z[11]*z[12]*z[2]*z[3]*z[7]*z[8] + 196141056*z[11]*z[12]*z[2]*z[3]*z[7]*z[9] - 3089221632*z[11]*z[12]*z[2]*z[3]*z[7] + 392282112*z[11]*z[12]*z[2]*z[3]*z[8]*z[9] - 6178443264*z[11]*z[12]*z[2]*z[3]*z[8] - 12356886528*z[11]*z[12]*z[2]*z[3]*z[9] + 109855336448*z[11]*z[12]*z[2]*z[3] + 37748736*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] + 75497472*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] - 1189085184*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[7] + 150994944*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] - 2378170368*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[8] - 4756340736*z[11]*z[12]*z[2]*z[4]*z[5]*z[6]*z[9] + 42284875776*z[11]*z[12]*z[2]*z[4]*z[5]*z[6] - 74317824*z[11]*z[12]*z[2]*z[4]*z[5]*z[7]*z[8] - 148635648*z[11]*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] + 2341011456*z[11]*z[12]*z[2]*z[4]*z[5]*z[7] - 297271296*z[11]*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] + 4682022912*z[11]*z[12]*z[2]*z[4]*z[5]*z[8] + 9364045824*z[11]*z[12]*z[2]*z[4]*z[5]*z[9] - 83248349184*z[11]*z[12]*z[2]*z[4]*z[5] - 148635648*z[11]*z[12]*z[2]*z[4]*z[6]*z[7]*z[8] - 297271296*z[11]*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] + 4682022912*z[11]*z[12]*z[2]*z[4]*z[6]*z[7] - 594542592*z[11]*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] + 9364045824*z[11]*z[12]*z[2]*z[4]*z[6]*z[8] + 18728091648*z[11]*z[12]*z[2]*z[4]*z[6]*z[9] - 166496698368*z[11]*z[12]*z[2]*z[4]*z[6] + 194961408*z[11]*z[12]*z[2]*z[4]*z[7]*z[8] + 389922816*z[11]*z[12]*z[2]*z[4]*z[7]*z[9] - 6141284352*z[11]*z[12]*z[2]*z[4]*z[7] + 779845632*z[11]*z[12]*z[2]*z[4]*z[8]*z[9] - 12282568704*z[11]*z[12]*z[2]*z[4]*z[8] - 24565137408*z[11]*z[12]*z[2]*z[4]*z[9] + 218389270528*z[11]*z[12]*z[2]*z[4] - 297271296*z[11]*z[12]*z[2]*z[5]*z[6]*z[7]*z[8] - 594542592*z[11]*z[12]*z[2]*z[5]*z[6]*z[7]*z[9] + 9364045824*z[11]*z[12]*z[2]*z[5]*z[6]*z[7] - 1189085184*z[11]*z[12]*z[2]*z[5]*z[6]*z[8]*z[9] + 18728091648*z[11]*z[12]*z[2]*z[5]*z[6]*z[8] + 37456183296*z[11]*z[12]*z[2]*z[5]*z[6]*z[9] - 332993396736*z[11]*z[12]*z[2]*z[5]*z[6] + 380485632*z[11]*z[12]*z[2]*z[5]*z[7]*z[8] + 760971264*z[11]*z[12]*z[2]*z[5]*z[7]*z[9] - 11985297408*z[11]*z[12]*z[2]*z[5]*z[7] + 1521942528*z[11]*z[12]*z[2]*z[5]*z[8]*z[9] - 23970594816*z[11]*z[12]*z[2]*z[5]*z[8] - 47941189632*z[11]*z[12]*z[2]*z[5]*z[9] + 426207322112*z[11]*z[12]*z[2]*z[5] + 685473792*z[11]*z[12]*z[2]*z[6]*z[7]*z[8] + 1370947584*z[11]*z[12]*z[2]*z[6]*z[7]*z[9] - 21592424448*z[11]*z[12]*z[2]*z[6]*z[7] + 2741895168*z[11]*z[12]*z[2]*z[6]*z[8]*z[9] - 43184848896*z[11]*z[12]*z[2]*z[6]*z[8] - 86369697792*z[11]*z[12]*z[2]*z[6]*z[9] + 767844892672*z[11]*z[12]*z[2]*z[6] - 779563008*z[11]*z[12]*z[2]*z[7]*z[8] - 1559126016*z[11]*z[12]*z[2]*z[7]*z[9] + 24556234752*z[11]*z[12]*z[2]*z[7] - 3118252032*z[11]*z[12]*z[2]*z[8]*z[9] + 49112469504*z[11]*z[12]*z[2]*z[8] + 98224939008*z[11]*z[12]*z[2]*z[9] - 873240496128*z[11]*z[12]*z[2] + 75497472*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 150994944*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] - 2378170368*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[7] + 301989888*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] - 4756340736*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[8] - 9512681472*z[11]*z[12]*z[3]*z[4]*z[5]*z[6]*z[9] + 84569751552*z[11]*z[12]*z[3]*z[4]*z[5]*z[6] - 148635648*z[11]*z[12]*z[3]*z[4]*z[5]*z[7]*z[8] - 297271296*z[11]*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] + 4682022912*z[11]*z[12]*z[3]*z[4]*z[5]*z[7] - 594542592*z[11]*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] + 9364045824*z[11]*z[12]*z[3]*z[4]*z[5]*z[8] + 18728091648*z[11]*z[12]*z[3]*z[4]*z[5]*z[9] - 166496698368*z[11]*z[12]*z[3]*z[4]*z[5] - 297271296*z[11]*z[12]*z[3]*z[4]*z[6]*z[7]*z[8] - 594542592*z[11]*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] + 9364045824*z[11]*z[12]*z[3]*z[4]*z[6]*z[7] - 1189085184*z[11]*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] + 18728091648*z[11]*z[12]*z[3]*z[4]*z[6]*z[8] + 37456183296*z[11]*z[12]*z[3]*z[4]*z[6]*z[9] - 332993396736*z[11]*z[12]*z[3]*z[4]*z[6] + 389332992*z[11]*z[12]*z[3]*z[4]*z[7]*z[8] + 778665984*z[11]*z[12]*z[3]*z[4]*z[7]*z[9] - 12263989248*z[11]*z[12]*z[3]*z[4]*z[7] + 1557331968*z[11]*z[12]*z[3]*z[4]*z[8]*z[9] - 24527978496*z[11]*z[12]*z[3]*z[4]*z[8] - 49055956992*z[11]*z[12]*z[3]*z[4]*z[9] + 436117839872*z[11]*z[12]*z[3]*z[4] - 594542592*z[11]*z[12]*z[3]*z[5]*z[6]*z[7]*z[8] - 1189085184*z[11]*z[12]*z[3]*z[5]*z[6]*z[7]*z[9] + 18728091648*z[11]*z[12]*z[3]*z[5]*z[6]*z[7] - 2378170368*z[11]*z[12]*z[3]*z[5]*z[6]*z[8]*z[9] + 37456183296*z[11]*z[12]*z[3]*z[5]*z[6]*z[8] + 74912366592*z[11]*z[12]*z[3]*z[5]*z[6]*z[9] - 665986793472*z[11]*z[12]*z[3]*z[5]*z[6] + 759791616*z[11]*z[12]*z[3]*z[5]*z[7]*z[8] + 1519583232*z[11]*z[12]*z[3]*z[5]*z[7]*z[9] - 23933435904*z[11]*z[12]*z[3]*z[5]*z[7] + 3039166464*z[11]*z[12]*z[3]*z[5]*z[8]*z[9] - 47866871808*z[11]*z[12]*z[3]*z[5]*z[8] - 95733743616*z[11]*z[12]*z[3]*z[5]*z[9] + 851093241856*z[11]*z[12]*z[3]*z[5] + 1368588288*z[11]*z[12]*z[3]*z[6]*z[7]*z[8] + 2737176576*z[11]*z[12]*z[3]*z[6]*z[7]*z[9] - 43110531072*z[11]*z[12]*z[3]*z[6]*z[7] + 5474353152*z[11]*z[12]*z[3]*z[6]*z[8]*z[9] - 86221062144*z[11]*z[12]*z[3]*z[6]*z[8] - 172442124288*z[11]*z[12]*z[3]*z[6]*z[9] + 1533046980608*z[11]*z[12]*z[3]*z[6] - 1554481152*z[11]*z[12]*z[3]*z[7]*z[8] - 3108962304*z[11]*z[12]*z[3]*z[7]*z[9] + 48966156288*z[11]*z[12]*z[3]*z[7] - 6217924608*z[11]*z[12]*z[3]*z[8]*z[9] + 97932312576*z[11]*z[12]*z[3]*z[8] + 195864625152*z[11]*z[12]*z[3]*z[9] - 1741277970432*z[11]*z[12]*z[3] - 1189085184*z[11]*z[12]*z[4]*z[5]*z[6]*z[7]*z[8] - 2378170368*z[11]*z[12]*z[4]*z[5]*z[6]*z[7]*z[9] + 37456183296*z[11]*z[12]*z[4]*z[5]*z[6]*z[7] - 4756340736*z[11]*z[12]*z[4]*z[5]*z[6]*z[8]*z[9] + 74912366592*z[11]*z[12]*z[4]*z[5]*z[6]*z[8] + 149824733184*z[11]*z[12]*z[4]*z[5]*z[6]*z[9] - 1331973586944*z[11]*z[12]*z[4]*z[5]*z[6] + 1510146048*z[11]*z[12]*z[4]*z[5]*z[7]*z[8] + 3020292096*z[11]*z[12]*z[4]*z[5]*z[7]*z[9] - 47569600512*z[11]*z[12]*z[4]*z[5]*z[7] + 6040584192*z[11]*z[12]*z[4]*z[5]*z[8]*z[9] - 95139201024*z[11]*z[12]*z[4]*z[5]*z[8] - 190278402048*z[11]*z[12]*z[4]*z[5]*z[9] + 1691615264768*z[11]*z[12]*z[4]*z[5] + 2718302208*z[11]*z[12]*z[4]*z[6]*z[7]*z[8] + 5436604416*z[11]*z[12]*z[4]*z[6]*z[7]*z[9] - 85626519552*z[11]*z[12]*z[4]*z[6]*z[7] + 10873208832*z[11]*z[12]*z[4]*z[6]*z[8]*z[9] - 171253039104*z[11]*z[12]*z[4]*z[6]*z[8] - 342506078208*z[11]*z[12]*z[4]*z[6]*z[9] + 3044951523328*z[11]*z[12]*z[4]*z[6] - 3071803392*z[11]*z[12]*z[4]*z[7]*z[8] - 6143606784*z[11]*z[12]*z[4]*z[7]*z[9] + 96761806848*z[11]*z[12]*z[4]*z[7] - 12287213568*z[11]*z[12]*z[4]*z[8]*z[9] + 193523613696*z[11]*z[12]*z[4]*z[8] + 387047227392*z[11]*z[12]*z[4]*z[9] - 3440931766272*z[11]*z[12]*z[4] + 5285609472*z[11]*z[12]*z[5]*z[6]*z[7]*z[8] + 10571218944*z[11]*z[12]*z[5]*z[6]*z[7]*z[9] - 166496698368*z[11]*z[12]*z[5]*z[6]*z[7] + 21142437888*z[11]*z[12]*z[5]*z[6]*z[8]*z[9] - 332993396736*z[11]*z[12]*z[5]*z[6]*z[8] - 665986793472*z[11]*z[12]*z[5]*z[6]*z[9] + 5920763543552*z[11]*z[12]*z[5]*z[6] - 5846335488*z[11]*z[12]*z[5]*z[7]*z[8] - 11692670976*z[11]*z[12]*z[5]*z[7]*z[9] + 184159567872*z[11]*z[12]*z[5]*z[7] - 23385341952*z[11]*z[12]*z[5]*z[8]*z[9] + 368319135744*z[11]*z[12]*z[5]*z[8] + 736638271488*z[11]*z[12]*z[5]*z[9] - 6548870135808*z[11]*z[12]*z[5] - 9314500608*z[11]*z[12]*z[6]*z[7]*z[8] - 18629001216*z[11]*z[12]*z[6]*z[7]*z[9] + 293406769152*z[11]*z[12]*z[6]*z[7] - 37258002432*z[11]*z[12]*z[6]*z[8]*z[9] + 586813538304*z[11]*z[12]*z[6]*z[8] + 1173627076608*z[11]*z[12]*z[6]*z[9] - 10433793097728*z[11]*z[12]*z[6] - 36998277120*z[11]*z[12]*z[7]*z[8]*z[9] + 201266540544*z[11]*z[12]*z[7]*z[8] + 399653265408*z[11]*z[12]*z[7]*z[9] - 1970106195456*z[11]*z[12]*z[7] + 797865977856*z[11]*z[12]*z[8]*z[9] - 3930912018432*z[11]*z[12]*z[8] - 7787908767744*z[11]*z[12]*z[9] + 35949148306944*z[11]*z[12] - 384*z[11]*z[13]*z[14]*z[7]*z[8]*z[9] + 6048*z[11]*z[13]*z[14]*z[7]*z[8] + 12096*z[11]*z[13]*z[14]*z[7]*z[9] - 123904*z[11]*z[13]*z[14]*z[7] + 24192*z[11]*z[13]*z[14]*z[8]*z[9] - 247712*z[11]*z[13]*z[14]*z[8] - 494656*z[11]*z[13]*z[14]*z[9] + 3806208*z[11]*z[13]*z[14] - 768*z[11]*z[13]*z[15]*z[7]*z[8]*z[9] + 12096*z[11]*z[13]*z[15]*z[7]*z[8] + 24192*z[11]*z[13]*z[15]*z[7]*z[9] - 247808*z[11]*z[13]*z[15]*z[7] + 48384*z[11]*z[13]*z[15]*z[8]*z[9] - 495424*z[11]*z[13]*z[15]*z[8] - 989312*z[11]*z[13]*z[15]*z[9] + 7612416*z[11]*z[13]*z[15] - 1536*z[11]*z[13]*z[16]*z[7]*z[8]*z[9] + 24192*z[11]*z[13]*z[16]*z[7]*z[8] + 48384*z[11]*z[13]*z[16]*z[7]*z[9] - 495616*z[11]*z[13]*z[16]*z[7] + 96768*z[11]*z[13]*z[16]*z[8]*z[9] - 990848*z[11]*z[13]*z[16]*z[8] - 1978624*z[11]*z[13]*z[16]*z[9] + 15224832*z[11]*z[13]*z[16] - 3072*z[11]*z[13]*z[17]*z[7]*z[8]*z[9] + 48384*z[11]*z[13]*z[17]*z[7]*z[8] + 96768*z[11]*z[13]*z[17]*z[7]*z[9] - 991232*z[11]*z[13]*z[17]*z[7] + 193536*z[11]*z[13]*z[17]*z[8]*z[9] - 1981696*z[11]*z[13]*z[17]*z[8] - 3957248*z[11]*z[13]*z[17]*z[9] + 30449664*z[11]*z[13]*z[17] - 6144*z[11]*z[13]*z[18]*z[7]*z[8]*z[9] + 96768*z[11]*z[13]*z[18]*z[7]*z[8] + 193536*z[11]*z[13]*z[18]*z[7]*z[9] - 1982464*z[11]*z[13]*z[18]*z[7] + 387072*z[11]*z[13]*z[18]*z[8]*z[9] - 3963392*z[11]*z[13]*z[18]*z[8] - 7914496*z[11]*z[13]*z[18]*z[9] + 60899328*z[11]*z[13]*z[18] + 12096*z[11]*z[13]*z[7]*z[8]*z[9] - 190512*z[11]*z[13]*z[7]*z[8] - 381024*z[11]*z[13]*z[7]*z[9] + 3902976*z[11]*z[13]*z[7] - 762048*z[11]*z[13]*z[8]*z[9] + 7802928*z[11]*z[13]*z[8] + 15581664*z[11]*z[13]*z[9] - 119895552*z[11]*z[13] - 1536*z[11]*z[14]*z[15]*z[7]*z[8]*z[9] + 24192*z[11]*z[14]*z[15]*z[7]*z[8] + 48384*z[11]*z[14]*z[15]*z[7]*z[9] - 495616*z[11]*z[14]*z[15]*z[7] + 96768*z[11]*z[14]*z[15]*z[8]*z[9] - 990848*z[11]*z[14]*z[15]*z[8] - 1978624*z[11]*z[14]*z[15]*z[9] + 15224832*z[11]*z[14]*z[15] - 3072*z[11]*z[14]*z[16]*z[7]*z[8]*z[9] + 48384*z[11]*z[14]*z[16]*z[7]*z[8] + 96768*z[11]*z[14]*z[16]*z[7]*z[9] - 991232*z[11]*z[14]*z[16]*z[7] + 193536*z[11]*z[14]*z[16]*z[8]*z[9] - 1981696*z[11]*z[14]*z[16]*z[8] - 3957248*z[11]*z[14]*z[16]*z[9] + 30449664*z[11]*z[14]*z[16] - 6144*z[11]*z[14]*z[17]*z[7]*z[8]*z[9] + 96768*z[11]*z[14]*z[17]*z[7]*z[8] + 193536*z[11]*z[14]*z[17]*z[7]*z[9] - 1982464*z[11]*z[14]*z[17]*z[7] + 387072*z[11]*z[14]*z[17]*z[8]*z[9] - 3963392*z[11]*z[14]*z[17]*z[8] - 7914496*z[11]*z[14]*z[17]*z[9] + 60899328*z[11]*z[14]*z[17] - 12288*z[11]*z[14]*z[18]*z[7]*z[8]*z[9] + 193536*z[11]*z[14]*z[18]*z[7]*z[8] + 387072*z[11]*z[14]*z[18]*z[7]*z[9] - 3964928*z[11]*z[14]*z[18]*z[7] + 774144*z[11]*z[14]*z[18]*z[8]*z[9] - 7926784*z[11]*z[14]*z[18]*z[8] - 15828992*z[11]*z[14]*z[18]*z[9] + 121798656*z[11]*z[14]*z[18] + 24192*z[11]*z[14]*z[7]*z[8]*z[9] - 381024*z[11]*z[14]*z[7]*z[8] - 762048*z[11]*z[14]*z[7]*z[9] + 7805952*z[11]*z[14]*z[7] - 1524096*z[11]*z[14]*z[8]*z[9] + 15605856*z[11]*z[14]*z[8] + 31163328*z[11]*z[14]*z[9] - 239791104*z[11]*z[14] - 6144*z[11]*z[15]*z[16]*z[7]*z[8]*z[9] + 96768*z[11]*z[15]*z[16]*z[7]*z[8] + 193536*z[11]*z[15]*z[16]*z[7]*z[9] - 1982464*z[11]*z[15]*z[16]*z[7] + 387072*z[11]*z[15]*z[16]*z[8]*z[9] - 3963392*z[11]*z[15]*z[16]*z[8] - 7914496*z[11]*z[15]*z[16]*z[9] + 60899328*z[11]*z[15]*z[16] - 12288*z[11]*z[15]*z[17]*z[7]*z[8]*z[9] + 193536*z[11]*z[15]*z[17]*z[7]*z[8] + 387072*z[11]*z[15]*z[17]*z[7]*z[9] - 3964928*z[11]*z[15]*z[17]*z[7] + 774144*z[11]*z[15]*z[17]*z[8]*z[9] - 7926784*z[11]*z[15]*z[17]*z[8] - 15828992*z[11]*z[15]*z[17]*z[9] + 121798656*z[11]*z[15]*z[17] - 24576*z[11]*z[15]*z[18]*z[7]*z[8]*z[9] + 387072*z[11]*z[15]*z[18]*z[7]*z[8] + 774144*z[11]*z[15]*z[18]*z[7]*z[9] - 7929856*z[11]*z[15]*z[18]*z[7] + 1548288*z[11]*z[15]*z[18]*z[8]*z[9] - 15853568*z[11]*z[15]*z[18]*z[8] - 31657984*z[11]*z[15]*z[18]*z[9] + 243597312*z[11]*z[15]*z[18] + 48384*z[11]*z[15]*z[7]*z[8]*z[9] - 762048*z[11]*z[15]*z[7]*z[8] - 1524096*z[11]*z[15]*z[7]*z[9] + 15611904*z[11]*z[15]*z[7] - 3048192*z[11]*z[15]*z[8]*z[9] + 31211712*z[11]*z[15]*z[8] + 62326656*z[11]*z[15]*z[9] - 479582208*z[11]*z[15] - 24576*z[11]*z[16]*z[17]*z[7]*z[8]*z[9] + 387072*z[11]*z[16]*z[17]*z[7]*z[8] + 774144*z[11]*z[16]*z[17]*z[7]*z[9] - 7929856*z[11]*z[16]*z[17]*z[7] + 1548288*z[11]*z[16]*z[17]*z[8]*z[9] - 15853568*z[11]*z[16]*z[17]*z[8] - 31657984*z[11]*z[16]*z[17]*z[9] + 243597312*z[11]*z[16]*z[17] - 49152*z[11]*z[16]*z[18]*z[7]*z[8]*z[9] + 774144*z[11]*z[16]*z[18]*z[7]*z[8] + 1548288*z[11]*z[16]*z[18]*z[7]*z[9] - 15859712*z[11]*z[16]*z[18]*z[7] + 3096576*z[11]*z[16]*z[18]*z[8]*z[9] - 31707136*z[11]*z[16]*z[18]*z[8] - 63315968*z[11]*z[16]*z[18]*z[9] + 487194624*z[11]*z[16]*z[18] + 96768*z[11]*z[16]*z[7]*z[8]*z[9] - 1524096*z[11]*z[16]*z[7]*z[8] - 3048192*z[11]*z[16]*z[7]*z[9] + 31223808*z[11]*z[16]*z[7] - 6096384*z[11]*z[16]*z[8]*z[9] + 62423424*z[11]*z[16]*z[8] + 124653312*z[11]*z[16]*z[9] - 959164416*z[11]*z[16] - 98304*z[11]*z[17]*z[18]*z[7]*z[8]*z[9] + 1548288*z[11]*z[17]*z[18]*z[7]*z[8] + 3096576*z[11]*z[17]*z[18]*z[7]*z[9] - 31719424*z[11]*z[17]*z[18]*z[7] + 6193152*z[11]*z[17]*z[18]*z[8]*z[9] - 63414272*z[11]*z[17]*z[18]*z[8] - 126631936*z[11]*z[17]*z[18]*z[9] + 974389248*z[11]*z[17]*z[18] + 193536*z[11]*z[17]*z[7]*z[8]*z[9] - 3048192*z[11]*z[17]*z[7]*z[8] - 6096384*z[11]*z[17]*z[7]*z[9] + 62447616*z[11]*z[17]*z[7] - 12192768*z[11]*z[17]*z[8]*z[9] + 124846848*z[11]*z[17]*z[8] + 249306624*z[11]*z[17]*z[9] - 1918328832*z[11]*z[17] + 387072*z[11]*z[18]*z[7]*z[8]*z[9] - 6096384*z[11]*z[18]*z[7]*z[8] - 12192768*z[11]*z[18]*z[7]*z[9] + 124895232*z[11]*z[18]*z[7] - 24385536*z[11]*z[18]*z[8]*z[9] + 249693696*z[11]*z[18]*z[8] + 498613248*z[11]*z[18]*z[9] - 3836657664*z[11]*z[18] + 589824*z[11]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 9289728*z[11]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] - 18579456*z[11]*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] + 190316544*z[11]*z[2]*z[3]*z[4]*z[5]*z[7] - 37158912*z[11]*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] + 380485632*z[11]*z[2]*z[3]*z[4]*z[5]*z[8] + 759791616*z[11]*z[2]*z[3]*z[4]*z[5]*z[9] - 5846335488*z[11]*z[2]*z[3]*z[4]*z[5] + 1179648*z[11]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 18579456*z[11]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] - 37158912*z[11]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] + 380633088*z[11]*z[2]*z[3]*z[4]*z[6]*z[7] - 74317824*z[11]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] + 760971264*z[11]*z[2]*z[3]*z[4]*z[6]*z[8] + 1519583232*z[11]*z[2]*z[3]*z[4]*z[6]*z[9] - 11692670976*z[11]*z[2]*z[3]*z[4]*z[6] - 2322432*z[11]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] + 36578304*z[11]*z[2]*z[3]*z[4]*z[7]*z[8] + 73156608*z[11]*z[2]*z[3]*z[4]*z[7]*z[9] - 749371392*z[11]*z[2]*z[3]*z[4]*z[7] + 146313216*z[11]*z[2]*z[3]*z[4]*z[8]*z[9] - 1498162176*z[11]*z[2]*z[3]*z[4]*z[8] - 2991679488*z[11]*z[2]*z[3]*z[4]*z[9] + 23019945984*z[11]*z[2]*z[3]*z[4] + 2359296*z[11]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 37158912*z[11]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] - 74317824*z[11]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] + 761266176*z[11]*z[2]*z[3]*z[5]*z[6]*z[7] - 148635648*z[11]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] + 1521942528*z[11]*z[2]*z[3]*z[5]*z[6]*z[8] + 3039166464*z[11]*z[2]*z[3]*z[5]*z[6]*z[9] - 23385341952*z[11]*z[2]*z[3]*z[5]*z[6] - 4644864*z[11]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] + 73156608*z[11]*z[2]*z[3]*z[5]*z[7]*z[8] + 146313216*z[11]*z[2]*z[3]*z[5]*z[7]*z[9] - 1498742784*z[11]*z[2]*z[3]*z[5]*z[7] + 292626432*z[11]*z[2]*z[3]*z[5]*z[8]*z[9] - 2996324352*z[11]*z[2]*z[3]*z[5]*z[8] - 5983358976*z[11]*z[2]*z[3]*z[5]*z[9] + 46039891968*z[11]*z[2]*z[3]*z[5] - 9289728*z[11]*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] + 146313216*z[11]*z[2]*z[3]*z[6]*z[7]*z[8] + 292626432*z[11]*z[2]*z[3]*z[6]*z[7]*z[9] - 2997485568*z[11]*z[2]*z[3]*z[6]*z[7] + 585252864*z[11]*z[2]*z[3]*z[6]*z[8]*z[9] - 5992648704*z[11]*z[2]*z[3]*z[6]*z[8] - 11966717952*z[11]*z[2]*z[3]*z[6]*z[9] + 92079783936*z[11]*z[2]*z[3]*z[6] + 12258816*z[11]*z[2]*z[3]*z[7]*z[8]*z[9] - 193076352*z[11]*z[2]*z[3]*z[7]*z[8] - 386152704*z[11]*z[2]*z[3]*z[7]*z[9] + 3955511296*z[11]*z[2]*z[3]*z[7] - 772305408*z[11]*z[2]*z[3]*z[8]*z[9] + 7907957888*z[11]*z[2]*z[3]*z[8] + 15791398144*z[11]*z[2]*z[3]*z[9] - 121509384192*z[11]*z[2]*z[3] + 4718592*z[11]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 74317824*z[11]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] - 148635648*z[11]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] + 1522532352*z[11]*z[2]*z[4]*z[5]*z[6]*z[7] - 297271296*z[11]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] + 3043885056*z[11]*z[2]*z[4]*z[5]*z[6]*z[8] + 6078332928*z[11]*z[2]*z[4]*z[5]*z[6]*z[9] - 46770683904*z[11]*z[2]*z[4]*z[5]*z[6] - 9289728*z[11]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] + 146313216*z[11]*z[2]*z[4]*z[5]*z[7]*z[8] + 292626432*z[11]*z[2]*z[4]*z[5]*z[7]*z[9] - 2997485568*z[11]*z[2]*z[4]*z[5]*z[7] + 585252864*z[11]*z[2]*z[4]*z[5]*z[8]*z[9] - 5992648704*z[11]*z[2]*z[4]*z[5]*z[8] - 11966717952*z[11]*z[2]*z[4]*z[5]*z[9] + 92079783936*z[11]*z[2]*z[4]*z[5] - 18579456*z[11]*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] + 292626432*z[11]*z[2]*z[4]*z[6]*z[7]*z[8] + 585252864*z[11]*z[2]*z[4]*z[6]*z[7]*z[9] - 5994971136*z[11]*z[2]*z[4]*z[6]*z[7] + 1170505728*z[11]*z[2]*z[4]*z[6]*z[8]*z[9] - 11985297408*z[11]*z[2]*z[4]*z[6]*z[8] - 23933435904*z[11]*z[2]*z[4]*z[6]*z[9] + 184159567872*z[11]*z[2]*z[4]*z[6] + 24370176*z[11]*z[2]*z[4]*z[7]*z[8]*z[9] - 383830272*z[11]*z[2]*z[4]*z[7]*z[8] - 767660544*z[11]*z[2]*z[4]*z[7]*z[9] + 7863443456*z[11]*z[2]*z[4]*z[7] - 1535321088*z[11]*z[2]*z[4]*z[8]*z[9] + 15720794368*z[11]*z[2]*z[4]*z[8] + 31392848384*z[11]*z[2]*z[4]*z[9] - 241557184512*z[11]*z[2]*z[4] - 37158912*z[11]*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] + 585252864*z[11]*z[2]*z[5]*z[6]*z[7]*z[8] + 1170505728*z[11]*z[2]*z[5]*z[6]*z[7]*z[9] - 11989942272*z[11]*z[2]*z[5]*z[6]*z[7] + 2341011456*z[11]*z[2]*z[5]*z[6]*z[8]*z[9] - 23970594816*z[11]*z[2]*z[5]*z[6]*z[8] - 47866871808*z[11]*z[2]*z[5]*z[6]*z[9] + 368319135744*z[11]*z[2]*z[5]*z[6] + 47560704*z[11]*z[2]*z[5]*z[7]*z[8]*z[9] - 749081088*z[11]*z[2]*z[5]*z[7]*z[8] - 1498162176*z[11]*z[2]*z[5]*z[7]*z[9] + 15346253824*z[11]*z[2]*z[5]*z[7] - 2996324352*z[11]*z[2]*z[5]*z[8]*z[9] + 30680617472*z[11]*z[2]*z[5]*z[8] + 61266113536*z[11]*z[2]*z[5]*z[9] - 471421698048*z[11]*z[2]*z[5] + 85684224*z[11]*z[2]*z[6]*z[7]*z[8]*z[9] - 1349526528*z[11]*z[2]*z[6]*z[7]*z[8] - 2699053056*z[11]*z[2]*z[6]*z[7]*z[9] + 27647442944*z[11]*z[2]*z[6]*z[7] - 5398106112*z[11]*z[2]*z[6]*z[8]*z[9] + 55273464832*z[11]*z[2]*z[6]*z[8] + 110375561216*z[11]*z[2]*z[6]*z[9] - 849302028288*z[11]*z[2]*z[6] - 97445376*z[11]*z[2]*z[7]*z[8]*z[9] + 1534764672*z[11]*z[2]*z[7]*z[8] + 3069529344*z[11]*z[2]*z[7]*z[9] - 31442374656*z[11]*z[2]*z[7] + 6139058688*z[11]*z[2]*z[8]*z[9] - 62860387968*z[11]*z[2]*z[8] - 125525885184*z[11]*z[2]*z[9] + 965878566912*z[11]*z[2] + 9437184*z[11]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 148635648*z[11]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] - 297271296*z[11]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] + 3045064704*z[11]*z[3]*z[4]*z[5]*z[6]*z[7] - 594542592*z[11]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] + 6087770112*z[11]*z[3]*z[4]*z[5]*z[6]*z[8] + 12156665856*z[11]*z[3]*z[4]*z[5]*z[6]*z[9] - 93541367808*z[11]*z[3]*z[4]*z[5]*z[6] - 18579456*z[11]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] + 292626432*z[11]*z[3]*z[4]*z[5]*z[7]*z[8] + 585252864*z[11]*z[3]*z[4]*z[5]*z[7]*z[9] - 5994971136*z[11]*z[3]*z[4]*z[5]*z[7] + 1170505728*z[11]*z[3]*z[4]*z[5]*z[8]*z[9] - 11985297408*z[11]*z[3]*z[4]*z[5]*z[8] - 23933435904*z[11]*z[3]*z[4]*z[5]*z[9] + 184159567872*z[11]*z[3]*z[4]*z[5] - 37158912*z[11]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] + 585252864*z[11]*z[3]*z[4]*z[6]*z[7]*z[8] + 1170505728*z[11]*z[3]*z[4]*z[6]*z[7]*z[9] - 11989942272*z[11]*z[3]*z[4]*z[6]*z[7] + 2341011456*z[11]*z[3]*z[4]*z[6]*z[8]*z[9] - 23970594816*z[11]*z[3]*z[4]*z[6]*z[8] - 47866871808*z[11]*z[3]*z[4]*z[6]*z[9] + 368319135744*z[11]*z[3]*z[4]*z[6] + 48666624*z[11]*z[3]*z[4]*z[7]*z[8]*z[9] - 766499328*z[11]*z[3]*z[4]*z[7]*z[8] - 1532998656*z[11]*z[3]*z[4]*z[7]*z[9] + 15703097344*z[11]*z[3]*z[4]*z[7] - 3065997312*z[11]*z[3]*z[4]*z[8]*z[9] + 31394028032*z[11]*z[3]*z[4]*z[8] + 62690722816*z[11]*z[3]*z[4]*z[9] - 482383577088*z[11]*z[3]*z[4] - 74317824*z[11]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] + 1170505728*z[11]*z[3]*z[5]*z[6]*z[7]*z[8] + 2341011456*z[11]*z[3]*z[5]*z[6]*z[7]*z[9] - 23979884544*z[11]*z[3]*z[5]*z[6]*z[7] + 4682022912*z[11]*z[3]*z[5]*z[6]*z[8]*z[9] - 47941189632*z[11]*z[3]*z[5]*z[6]*z[8] - 95733743616*z[11]*z[3]*z[5]*z[6]*z[9] + 736638271488*z[11]*z[3]*z[5]*z[6] + 94973952*z[11]*z[3]*z[5]*z[7]*z[8]*z[9] - 1495839744*z[11]*z[3]*z[5]*z[7]*z[8] - 2991679488*z[11]*z[3]*z[5]*z[7]*z[9] + 30644928512*z[11]*z[3]*z[5]*z[7] - 5983358976*z[11]*z[3]*z[5]*z[8]*z[9] + 61266113536*z[11]*z[3]*z[5]*z[8] + 122342279168*z[11]*z[3]*z[5]*z[9] - 941381812224*z[11]*z[3]*z[5] + 171073536*z[11]*z[3]*z[6]*z[7]*z[8]*z[9] - 2694408192*z[11]*z[3]*z[6]*z[7]*z[8] - 5388816384*z[11]*z[3]*z[6]*z[7]*z[9] + 55199727616*z[11]*z[3]*z[6]*z[7] - 10777632768*z[11]*z[3]*z[6]*z[8]*z[9] + 110356686848*z[11]*z[3]*z[6]*z[8] + 220371226624*z[11]*z[3]*z[6]*z[9] - 1695680888832*z[11]*z[3]*z[6] - 194310144*z[11]*z[3]*z[7]*z[8]*z[9] + 3060384768*z[11]*z[3]*z[7]*z[8] + 6120769536*z[11]*z[3]*z[7]*z[9] - 62697406464*z[11]*z[3]*z[7] + 12241539072*z[11]*z[3]*z[8]*z[9] - 125346235392*z[11]*z[3]*z[8] - 250303850496*z[11]*z[3]*z[9] + 1926002147328*z[11]*z[3] - 148635648*z[11]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 2341011456*z[11]*z[4]*z[5]*z[6]*z[7]*z[8] + 4682022912*z[11]*z[4]*z[5]*z[6]*z[7]*z[9] - 47959769088*z[11]*z[4]*z[5]*z[6]*z[7] + 9364045824*z[11]*z[4]*z[5]*z[6]*z[8]*z[9] - 95882379264*z[11]*z[4]*z[5]*z[6]*z[8] - 191467487232*z[11]*z[4]*z[5]*z[6]*z[9] + 1473276542976*z[11]*z[4]*z[5]*z[6] + 188768256*z[11]*z[4]*z[5]*z[7]*z[8]*z[9] - 2973100032*z[11]*z[4]*z[5]*z[7]*z[8] - 5946200064*z[11]*z[4]*z[5]*z[7]*z[9] + 60909223936*z[11]*z[4]*z[5]*z[7] - 11892400128*z[11]*z[4]*z[5]*z[8]*z[9] + 121771255808*z[11]*z[4]*z[5]*z[8] + 243164975104*z[11]*z[4]*z[5]*z[9] - 1871070953472*z[11]*z[4]*z[5] + 339787776*z[11]*z[4]*z[6]*z[7]*z[8]*z[9] - 5351657472*z[11]*z[4]*z[6]*z[7]*z[8] - 10703314944*z[11]*z[4]*z[6]*z[7]*z[9] + 109638189056*z[11]*z[4]*z[6]*z[7] - 21406629888*z[11]*z[4]*z[6]*z[8]*z[9] + 219191431168*z[11]*z[4]*z[6]*z[8] + 437703286784*z[11]*z[4]*z[6]*z[9] - 3367976435712*z[11]*z[4]*z[6] - 383975424*z[11]*z[4]*z[7]*z[8]*z[9] + 6047612928*z[11]*z[4]*z[7]*z[8] + 12095225856*z[11]*z[4]*z[7]*z[9] - 123896070144*z[11]*z[4]*z[7] + 24190451712*z[11]*z[4]*z[8]*z[9] - 247696146432*z[11]*z[4]*z[8] - 494624342016*z[11]*z[4]*z[9] + 3805964402688*z[11]*z[4] + 660701184*z[11]*z[5]*z[6]*z[7]*z[8]*z[9] - 10406043648*z[11]*z[5]*z[6]*z[7]*z[8] - 20812087296*z[11]*z[5]*z[6]*z[7]*z[9] + 213186248704*z[11]*z[5]*z[6]*z[7] - 41624174592*z[11]*z[5]*z[6]*z[8]*z[9] + 426207322112*z[11]*z[5]*z[6]*z[8] + 851093241856*z[11]*z[5]*z[6]*z[9] - 6548870135808*z[11]*z[5]*z[6] - 730791936*z[11]*z[5]*z[7]*z[8]*z[9] + 11509972992*z[11]*z[5]*z[7]*z[8] + 23019945984*z[11]*z[5]*z[7]*z[9] - 235802198016*z[11]*z[5]*z[7] + 46039891968*z[11]*z[5]*z[8]*z[9] - 471421698048*z[11]*z[5]*z[8] - 941381812224*z[11]*z[5]*z[9] + 7243609669632*z[11]*z[5] - 1164312576*z[11]*z[6]*z[7]*z[8]*z[9] + 18337923072*z[11]*z[6]*z[7]*z[8] + 36675846144*z[11]*z[6]*z[7]*z[9] - 375684857856*z[11]*z[6]*z[7] + 73351692288*z[11]*z[6]*z[8]*z[9] - 751078637568*z[11]*z[6]*z[8] - 1499828649984*z[11]*z[6]*z[9] + 11540666253312*z[11]*z[6] + 39816734208*z[11]*z[7]*z[8]*z[9] - 214585441056*z[11]*z[7]*z[8] - 425994835392*z[11]*z[7]*z[9] + 2072542215168*z[11]*z[7] - 850400377344*z[11]*z[8]*z[9] + 4135065402144*z[11]*z[8] + 8190601079232*z[11]*z[9] - 37231585606656*z[11] - 768*z[12]*z[13]*z[14]*z[7]*z[8]*z[9] + 12096*z[12]*z[13]*z[14]*z[7]*z[8] + 24192*z[12]*z[13]*z[14]*z[7]*z[9] - 223232*z[12]*z[13]*z[14]*z[7] + 48384*z[12]*z[13]*z[14]*z[8]*z[9] - 446272*z[12]*z[13]*z[14]*z[8] - 891008*z[12]*z[13]*z[14]*z[9] + 6064128*z[12]*z[13]*z[14] - 1536*z[12]*z[13]*z[15]*z[7]*z[8]*z[9] + 24192*z[12]*z[13]*z[15]*z[7]*z[8] + 48384*z[12]*z[13]*z[15]*z[7]*z[9] - 446464*z[12]*z[13]*z[15]*z[7] + 96768*z[12]*z[13]*z[15]*z[8]*z[9] - 892544*z[12]*z[13]*z[15]*z[8] - 1782016*z[12]*z[13]*z[15]*z[9] + 12128256*z[12]*z[13]*z[15] - 3072*z[12]*z[13]*z[16]*z[7]*z[8]*z[9] + 48384*z[12]*z[13]*z[16]*z[7]*z[8] + 96768*z[12]*z[13]*z[16]*z[7]*z[9] - 892928*z[12]*z[13]*z[16]*z[7] + 193536*z[12]*z[13]*z[16]*z[8]*z[9] - 1785088*z[12]*z[13]*z[16]*z[8] - 3564032*z[12]*z[13]*z[16]*z[9] + 24256512*z[12]*z[13]*z[16] - 6144*z[12]*z[13]*z[17]*z[7]*z[8]*z[9] + 96768*z[12]*z[13]*z[17]*z[7]*z[8] + 193536*z[12]*z[13]*z[17]*z[7]*z[9] - 1785856*z[12]*z[13]*z[17]*z[7] + 387072*z[12]*z[13]*z[17]*z[8]*z[9] - 3570176*z[12]*z[13]*z[17]*z[8] - 7128064*z[12]*z[13]*z[17]*z[9] + 48513024*z[12]*z[13]*z[17] - 12288*z[12]*z[13]*z[18]*z[7]*z[8]*z[9] + 193536*z[12]*z[13]*z[18]*z[7]*z[8] + 387072*z[12]*z[13]*z[18]*z[7]*z[9] - 3571712*z[12]*z[13]*z[18]*z[7] + 774144*z[12]*z[13]*z[18]*z[8]*z[9] - 7140352*z[12]*z[13]*z[18]*z[8] - 14256128*z[12]*z[13]*z[18]*z[9] + 97026048*z[12]*z[13]*z[18] + 24192*z[12]*z[13]*z[7]*z[8]*z[9] - 381024*z[12]*z[13]*z[7]*z[8] - 762048*z[12]*z[13]*z[7]*z[9] + 7031808*z[12]*z[13]*z[7] - 1524096*z[12]*z[13]*z[8]*z[9] + 14057568*z[12]*z[13]*z[8] + 28066752*z[12]*z[13]*z[9] - 191020032*z[12]*z[13] - 3072*z[12]*z[14]*z[15]*z[7]*z[8]*z[9] + 48384*z[12]*z[14]*z[15]*z[7]*z[8] + 96768*z[12]*z[14]*z[15]*z[7]*z[9] - 892928*z[12]*z[14]*z[15]*z[7] + 193536*z[12]*z[14]*z[15]*z[8]*z[9] - 1785088*z[12]*z[14]*z[15]*z[8] - 3564032*z[12]*z[14]*z[15]*z[9] + 24256512*z[12]*z[14]*z[15] - 6144*z[12]*z[14]*z[16]*z[7]*z[8]*z[9] + 96768*z[12]*z[14]*z[16]*z[7]*z[8] + 193536*z[12]*z[14]*z[16]*z[7]*z[9] - 1785856*z[12]*z[14]*z[16]*z[7] + 387072*z[12]*z[14]*z[16]*z[8]*z[9] - 3570176*z[12]*z[14]*z[16]*z[8] - 7128064*z[12]*z[14]*z[16]*z[9] + 48513024*z[12]*z[14]*z[16] - 12288*z[12]*z[14]*z[17]*z[7]*z[8]*z[9] + 193536*z[12]*z[14]*z[17]*z[7]*z[8] + 387072*z[12]*z[14]*z[17]*z[7]*z[9] - 3571712*z[12]*z[14]*z[17]*z[7] + 774144*z[12]*z[14]*z[17]*z[8]*z[9] - 7140352*z[12]*z[14]*z[17]*z[8] - 14256128*z[12]*z[14]*z[17]*z[9] + 97026048*z[12]*z[14]*z[17] - 24576*z[12]*z[14]*z[18]*z[7]*z[8]*z[9] + 387072*z[12]*z[14]*z[18]*z[7]*z[8] + 774144*z[12]*z[14]*z[18]*z[7]*z[9] - 7143424*z[12]*z[14]*z[18]*z[7] + 1548288*z[12]*z[14]*z[18]*z[8]*z[9] - 14280704*z[12]*z[14]*z[18]*z[8] - 28512256*z[12]*z[14]*z[18]*z[9] + 194052096*z[12]*z[14]*z[18] + 48384*z[12]*z[14]*z[7]*z[8]*z[9] - 762048*z[12]*z[14]*z[7]*z[8] - 1524096*z[12]*z[14]*z[7]*z[9] + 14063616*z[12]*z[14]*z[7] - 3048192*z[12]*z[14]*z[8]*z[9] + 28115136*z[12]*z[14]*z[8] + 56133504*z[12]*z[14]*z[9] - 382040064*z[12]*z[14] - 12288*z[12]*z[15]*z[16]*z[7]*z[8]*z[9] + 193536*z[12]*z[15]*z[16]*z[7]*z[8] + 387072*z[12]*z[15]*z[16]*z[7]*z[9] - 3571712*z[12]*z[15]*z[16]*z[7] + 774144*z[12]*z[15]*z[16]*z[8]*z[9] - 7140352*z[12]*z[15]*z[16]*z[8] - 14256128*z[12]*z[15]*z[16]*z[9] + 97026048*z[12]*z[15]*z[16] - 24576*z[12]*z[15]*z[17]*z[7]*z[8]*z[9] + 387072*z[12]*z[15]*z[17]*z[7]*z[8] + 774144*z[12]*z[15]*z[17]*z[7]*z[9] - 7143424*z[12]*z[15]*z[17]*z[7] + 1548288*z[12]*z[15]*z[17]*z[8]*z[9] - 14280704*z[12]*z[15]*z[17]*z[8] - 28512256*z[12]*z[15]*z[17]*z[9] + 194052096*z[12]*z[15]*z[17] - 49152*z[12]*z[15]*z[18]*z[7]*z[8]*z[9] + 774144*z[12]*z[15]*z[18]*z[7]*z[8] + 1548288*z[12]*z[15]*z[18]*z[7]*z[9] - 14286848*z[12]*z[15]*z[18]*z[7] + 3096576*z[12]*z[15]*z[18]*z[8]*z[9] - 28561408*z[12]*z[15]*z[18]*z[8] - 57024512*z[12]*z[15]*z[18]*z[9] + 388104192*z[12]*z[15]*z[18] + 96768*z[12]*z[15]*z[7]*z[8]*z[9] - 1524096*z[12]*z[15]*z[7]*z[8] - 3048192*z[12]*z[15]*z[7]*z[9] + 28127232*z[12]*z[15]*z[7] - 6096384*z[12]*z[15]*z[8]*z[9] + 56230272*z[12]*z[15]*z[8] + 112267008*z[12]*z[15]*z[9] - 764080128*z[12]*z[15] - 49152*z[12]*z[16]*z[17]*z[7]*z[8]*z[9] + 774144*z[12]*z[16]*z[17]*z[7]*z[8] + 1548288*z[12]*z[16]*z[17]*z[7]*z[9] - 14286848*z[12]*z[16]*z[17]*z[7] + 3096576*z[12]*z[16]*z[17]*z[8]*z[9] - 28561408*z[12]*z[16]*z[17]*z[8] - 57024512*z[12]*z[16]*z[17]*z[9] + 388104192*z[12]*z[16]*z[17] - 98304*z[12]*z[16]*z[18]*z[7]*z[8]*z[9] + 1548288*z[12]*z[16]*z[18]*z[7]*z[8] + 3096576*z[12]*z[16]*z[18]*z[7]*z[9] - 28573696*z[12]*z[16]*z[18]*z[7] + 6193152*z[12]*z[16]*z[18]*z[8]*z[9] - 57122816*z[12]*z[16]*z[18]*z[8] - 114049024*z[12]*z[16]*z[18]*z[9] + 776208384*z[12]*z[16]*z[18] + 193536*z[12]*z[16]*z[7]*z[8]*z[9] - 3048192*z[12]*z[16]*z[7]*z[8] - 6096384*z[12]*z[16]*z[7]*z[9] + 56254464*z[12]*z[16]*z[7] - 12192768*z[12]*z[16]*z[8]*z[9] + 112460544*z[12]*z[16]*z[8] + 224534016*z[12]*z[16]*z[9] - 1528160256*z[12]*z[16] - 196608*z[12]*z[17]*z[18]*z[7]*z[8]*z[9] + 3096576*z[12]*z[17]*z[18]*z[7]*z[8] + 6193152*z[12]*z[17]*z[18]*z[7]*z[9] - 57147392*z[12]*z[17]*z[18]*z[7] + 12386304*z[12]*z[17]*z[18]*z[8]*z[9] - 114245632*z[12]*z[17]*z[18]*z[8] - 228098048*z[12]*z[17]*z[18]*z[9] + 1552416768*z[12]*z[17]*z[18] + 387072*z[12]*z[17]*z[7]*z[8]*z[9] - 6096384*z[12]*z[17]*z[7]*z[8] - 12192768*z[12]*z[17]*z[7]*z[9] + 112508928*z[12]*z[17]*z[7] - 24385536*z[12]*z[17]*z[8]*z[9] + 224921088*z[12]*z[17]*z[8] + 449068032*z[12]*z[17]*z[9] - 3056320512*z[12]*z[17] + 774144*z[12]*z[18]*z[7]*z[8]*z[9] - 12192768*z[12]*z[18]*z[7]*z[8] - 24385536*z[12]*z[18]*z[7]*z[9] + 225017856*z[12]*z[18]*z[7] - 48771072*z[12]*z[18]*z[8]*z[9] + 449842176*z[12]*z[18]*z[8] + 898136064*z[12]*z[18]*z[9] - 6112641024*z[12]*z[18] + 1179648*z[12]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 18579456*z[12]*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] - 37158912*z[12]*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] + 342884352*z[12]*z[2]*z[3]*z[4]*z[5]*z[7] - 74317824*z[12]*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] + 685473792*z[12]*z[2]*z[3]*z[4]*z[5]*z[8] + 1368588288*z[12]*z[2]*z[3]*z[4]*z[5]*z[9] - 9314500608*z[12]*z[2]*z[3]*z[4]*z[5] + 2359296*z[12]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 37158912*z[12]*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] - 74317824*z[12]*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] + 685768704*z[12]*z[2]*z[3]*z[4]*z[6]*z[7] - 148635648*z[12]*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] + 1370947584*z[12]*z[2]*z[3]*z[4]*z[6]*z[8] + 2737176576*z[12]*z[2]*z[3]*z[4]*z[6]*z[9] - 18629001216*z[12]*z[2]*z[3]*z[4]*z[6] - 4644864*z[12]*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] + 73156608*z[12]*z[2]*z[3]*z[4]*z[7]*z[8] + 146313216*z[12]*z[2]*z[3]*z[4]*z[7]*z[9] - 1350107136*z[12]*z[2]*z[3]*z[4]*z[7] + 292626432*z[12]*z[2]*z[3]*z[4]*z[8]*z[9] - 2699053056*z[12]*z[2]*z[3]*z[4]*z[8] - 5388816384*z[12]*z[2]*z[3]*z[4]*z[9] + 36675846144*z[12]*z[2]*z[3]*z[4] + 4718592*z[12]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 74317824*z[12]*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] - 148635648*z[12]*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] + 1371537408*z[12]*z[2]*z[3]*z[5]*z[6]*z[7] - 297271296*z[12]*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] + 2741895168*z[12]*z[2]*z[3]*z[5]*z[6]*z[8] + 5474353152*z[12]*z[2]*z[3]*z[5]*z[6]*z[9] - 37258002432*z[12]*z[2]*z[3]*z[5]*z[6] - 9289728*z[12]*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] + 146313216*z[12]*z[2]*z[3]*z[5]*z[7]*z[8] + 292626432*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] - 2700214272*z[12]*z[2]*z[3]*z[5]*z[7] + 585252864*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] - 5398106112*z[12]*z[2]*z[3]*z[5]*z[8] - 10777632768*z[12]*z[2]*z[3]*z[5]*z[9] + 73351692288*z[12]*z[2]*z[3]*z[5] - 18579456*z[12]*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] + 292626432*z[12]*z[2]*z[3]*z[6]*z[7]*z[8] + 585252864*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] - 5400428544*z[12]*z[2]*z[3]*z[6]*z[7] + 1170505728*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] - 10796212224*z[12]*z[2]*z[3]*z[6]*z[8] - 21555265536*z[12]*z[2]*z[3]*z[6]*z[9] + 146703384576*z[12]*z[2]*z[3]*z[6] + 24517632*z[12]*z[2]*z[3]*z[7]*z[8]*z[9] - 386152704*z[12]*z[2]*z[3]*z[7]*z[8] - 772305408*z[12]*z[2]*z[3]*z[7]*z[9] + 7126458368*z[12]*z[2]*z[3]*z[7] - 1544610816*z[12]*z[2]*z[3]*z[8]*z[9] + 14246787328*z[12]*z[2]*z[3]*z[8] + 28444539392*z[12]*z[2]*z[3]*z[9] - 193591222272*z[12]*z[2]*z[3] + 9437184*z[12]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 148635648*z[12]*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] - 297271296*z[12]*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] + 2743074816*z[12]*z[2]*z[4]*z[5]*z[6]*z[7] - 594542592*z[12]*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] + 5483790336*z[12]*z[2]*z[4]*z[5]*z[6]*z[8] + 10948706304*z[12]*z[2]*z[4]*z[5]*z[6]*z[9] - 74516004864*z[12]*z[2]*z[4]*z[5]*z[6] - 18579456*z[12]*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] + 292626432*z[12]*z[2]*z[4]*z[5]*z[7]*z[8] + 585252864*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] - 5400428544*z[12]*z[2]*z[4]*z[5]*z[7] + 1170505728*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] - 10796212224*z[12]*z[2]*z[4]*z[5]*z[8] - 21555265536*z[12]*z[2]*z[4]*z[5]*z[9] + 146703384576*z[12]*z[2]*z[4]*z[5] - 37158912*z[12]*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] + 585252864*z[12]*z[2]*z[4]*z[6]*z[7]*z[8] + 1170505728*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] - 10800857088*z[12]*z[2]*z[4]*z[6]*z[7] + 2341011456*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] - 21592424448*z[12]*z[2]*z[4]*z[6]*z[8] - 43110531072*z[12]*z[2]*z[4]*z[6]*z[9] + 293406769152*z[12]*z[2]*z[4]*z[6] + 48740352*z[12]*z[2]*z[4]*z[7]*z[8]*z[9] - 767660544*z[12]*z[2]*z[4]*z[7]*z[8] - 1535321088*z[12]*z[2]*z[4]*z[7]*z[9] + 14167195648*z[12]*z[2]*z[4]*z[7] - 3070642176*z[12]*z[2]*z[4]*z[8]*z[9] + 28322206208*z[12]*z[2]*z[4]*z[8] + 56546931712*z[12]*z[2]*z[4]*z[9] - 384853819392*z[12]*z[2]*z[4] - 74317824*z[12]*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] + 1170505728*z[12]*z[2]*z[5]*z[6]*z[7]*z[8] + 2341011456*z[12]*z[2]*z[5]*z[6]*z[7]*z[9] - 21601714176*z[12]*z[2]*z[5]*z[6]*z[7] + 4682022912*z[12]*z[2]*z[5]*z[6]*z[8]*z[9] - 43184848896*z[12]*z[2]*z[5]*z[6]*z[8] - 86221062144*z[12]*z[2]*z[5]*z[6]*z[9] + 586813538304*z[12]*z[2]*z[5]*z[6] + 95121408*z[12]*z[2]*z[5]*z[7]*z[8]*z[9] - 1498162176*z[12]*z[2]*z[5]*z[7]*z[8] - 2996324352*z[12]*z[2]*z[5]*z[7]*z[9] + 27648622592*z[12]*z[2]*z[5]*z[7] - 5992648704*z[12]*z[2]*z[5]*z[8]*z[9] + 55273464832*z[12]*z[2]*z[5]*z[8] + 110356686848*z[12]*z[2]*z[5]*z[9] - 751078637568*z[12]*z[2]*z[5] + 171368448*z[12]*z[2]*z[6]*z[7]*z[8]*z[9] - 2699053056*z[12]*z[2]*z[6]*z[7]*z[8] - 5398106112*z[12]*z[2]*z[6]*z[7]*z[9] + 49811095552*z[12]*z[2]*z[6]*z[7] - 10796212224*z[12]*z[2]*z[6]*z[8]*z[9] + 99579348992*z[12]*z[2]*z[6]*z[8] + 198815961088*z[12]*z[2]*z[6]*z[9] - 1353125265408*z[12]*z[2]*z[6] - 194890752*z[12]*z[2]*z[7]*z[8]*z[9] + 3069529344*z[12]*z[2]*z[7]*z[8] + 6139058688*z[12]*z[2]*z[7]*z[9] - 56648245248*z[12]*z[2]*z[7] + 12278117376*z[12]*z[2]*z[8]*z[9] - 113247767808*z[12]*z[2]*z[8] - 226105754112*z[12]*z[2]*z[9] + 1538857377792*z[12]*z[2] + 18874368*z[12]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 297271296*z[12]*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] - 594542592*z[12]*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] + 5486149632*z[12]*z[3]*z[4]*z[5]*z[6]*z[7] - 1189085184*z[12]*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] + 10967580672*z[12]*z[3]*z[4]*z[5]*z[6]*z[8] + 21897412608*z[12]*z[3]*z[4]*z[5]*z[6]*z[9] - 149032009728*z[12]*z[3]*z[4]*z[5]*z[6] - 37158912*z[12]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] + 585252864*z[12]*z[3]*z[4]*z[5]*z[7]*z[8] + 1170505728*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] - 10800857088*z[12]*z[3]*z[4]*z[5]*z[7] + 2341011456*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] - 21592424448*z[12]*z[3]*z[4]*z[5]*z[8] - 43110531072*z[12]*z[3]*z[4]*z[5]*z[9] + 293406769152*z[12]*z[3]*z[4]*z[5] - 74317824*z[12]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] + 1170505728*z[12]*z[3]*z[4]*z[6]*z[7]*z[8] + 2341011456*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] - 21601714176*z[12]*z[3]*z[4]*z[6]*z[7] + 4682022912*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] - 43184848896*z[12]*z[3]*z[4]*z[6]*z[8] - 86221062144*z[12]*z[3]*z[4]*z[6]*z[9] + 586813538304*z[12]*z[3]*z[4]*z[6] + 97333248*z[12]*z[3]*z[4]*z[7]*z[8]*z[9] - 1532998656*z[12]*z[3]*z[4]*z[7]*z[8] - 3065997312*z[12]*z[3]*z[4]*z[7]*z[9] + 28291530752*z[12]*z[3]*z[4]*z[7] - 6131994624*z[12]*z[3]*z[4]*z[8]*z[9] + 56558728192*z[12]*z[3]*z[4]*z[8] + 112922789888*z[12]*z[3]*z[4]*z[9] - 768543326208*z[12]*z[3]*z[4] - 148635648*z[12]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] + 2341011456*z[12]*z[3]*z[5]*z[6]*z[7]*z[8] + 4682022912*z[12]*z[3]*z[5]*z[6]*z[7]*z[9] - 43203428352*z[12]*z[3]*z[5]*z[6]*z[7] + 9364045824*z[12]*z[3]*z[5]*z[6]*z[8]*z[9] - 86369697792*z[12]*z[3]*z[5]*z[6]*z[8] - 172442124288*z[12]*z[3]*z[5]*z[6]*z[9] + 1173627076608*z[12]*z[3]*z[5]*z[6] + 189947904*z[12]*z[3]*z[5]*z[7]*z[8]*z[9] - 2991679488*z[12]*z[3]*z[5]*z[7]*z[8] - 5983358976*z[12]*z[3]*z[5]*z[7]*z[9] + 55211524096*z[12]*z[3]*z[5]*z[7] - 11966717952*z[12]*z[3]*z[5]*z[8]*z[9] + 110375561216*z[12]*z[3]*z[5]*z[8] + 220371226624*z[12]*z[3]*z[5]*z[9] - 1499828649984*z[12]*z[3]*z[5] + 342147072*z[12]*z[3]*z[6]*z[7]*z[8]*z[9] - 5388816384*z[12]*z[3]*z[6]*z[7]*z[8] - 10777632768*z[12]*z[3]*z[6]*z[7]*z[9] + 99450748928*z[12]*z[3]*z[6]*z[7] - 21555265536*z[12]*z[3]*z[6]*z[8]*z[9] + 198815961088*z[12]*z[3]*z[6]*z[8] + 396947628032*z[12]*z[3]*z[6]*z[9] - 2701593280512*z[12]*z[3]*z[6] - 388620288*z[12]*z[3]*z[7]*z[8]*z[9] + 6120769536*z[12]*z[3]*z[7]*z[8] + 12241539072*z[12]*z[3]*z[7]*z[9] - 112958963712*z[12]*z[3]*z[7] + 24483078144*z[12]*z[3]*z[8]*z[9] - 225820772352*z[12]*z[3]*z[8] - 450864304128*z[12]*z[3]*z[9] + 3068545794048*z[12]*z[3] - 297271296*z[12]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 4682022912*z[12]*z[4]*z[5]*z[6]*z[7]*z[8] + 9364045824*z[12]*z[4]*z[5]*z[6]*z[7]*z[9] - 86406856704*z[12]*z[4]*z[5]*z[6]*z[7] + 18728091648*z[12]*z[4]*z[5]*z[6]*z[8]*z[9] - 172739395584*z[12]*z[4]*z[5]*z[6]*z[8] - 344884248576*z[12]*z[4]*z[5]*z[6]*z[9] + 2347254153216*z[12]*z[4]*z[5]*z[6] + 377536512*z[12]*z[4]*z[5]*z[7]*z[8]*z[9] - 5946200064*z[12]*z[4]*z[5]*z[7]*z[8] - 11892400128*z[12]*z[4]*z[5]*z[7]*z[9] + 109737279488*z[12]*z[4]*z[5]*z[7] - 23784800256*z[12]*z[4]*z[5]*z[8]*z[9] + 219380174848*z[12]*z[4]*z[5]*z[8] + 438005276672*z[12]*z[4]*z[5]*z[9] - 2981028298752*z[12]*z[4]*z[5] + 679575552*z[12]*z[4]*z[6]*z[7]*z[8]*z[9] - 10703314944*z[12]*z[4]*z[6]*z[7]*z[8] - 21406629888*z[12]*z[4]*z[6]*z[7]*z[9] + 197529960448*z[12]*z[4]*z[6]*z[7] - 42813259776*z[12]*z[4]*z[6]*z[8]*z[9] + 394890027008*z[12]*z[4]*z[6]*z[8] + 788420902912*z[12]*z[4]*z[6]*z[9] - 5365928558592*z[12]*z[4]*z[6] - 767950848*z[12]*z[4]*z[7]*z[8]*z[9] + 12095225856*z[12]*z[4]*z[7]*z[8] + 24190451712*z[12]*z[4]*z[7]*z[9] - 223217713152*z[12]*z[4]*z[7] + 48380903424*z[12]*z[4]*z[8]*z[9] - 446243438592*z[12]*z[4]*z[8] - 890950975488*z[12]*z[4]*z[9] + 6063739895808*z[12]*z[4] + 1321402368*z[12]*z[5]*z[6]*z[7]*z[8]*z[9] - 20812087296*z[12]*z[5]*z[6]*z[7]*z[8] - 41624174592*z[12]*z[5]*z[6]*z[7]*z[9] + 384087621632*z[12]*z[5]*z[6]*z[7] - 83248349184*z[12]*z[5]*z[6]*z[8]*z[9] + 767844892672*z[12]*z[5]*z[6]*z[8] + 1533046980608*z[12]*z[5]*z[6]*z[9] - 10433793097728*z[12]*z[5]*z[6] - 1461583872*z[12]*z[5]*z[7]*z[8]*z[9] + 23019945984*z[12]*z[5]*z[7]*z[8] + 46039891968*z[12]*z[5]*z[7]*z[9] - 424833712128*z[12]*z[5]*z[7] + 92079783936*z[12]*z[5]*z[8]*z[9] - 849302028288*z[12]*z[5]*z[8] - 1695680888832*z[12]*z[5]*z[9] + 11540666253312*z[12]*z[5] - 2328625152*z[12]*z[6]*z[7]*z[8]*z[9] + 36675846144*z[12]*z[6]*z[7]*z[8] + 73351692288*z[12]*z[6]*z[7]*z[9] - 676853710848*z[12]*z[6]*z[7] + 146703384576*z[12]*z[6]*z[8]*z[9] - 1353125265408*z[12]*z[6]*z[8] - 2701593280512*z[12]*z[6]*z[9] + 18386824200192*z[12]*z[6] + 57461984256*z[12]*z[7]*z[8]*z[9] - 294183392832*z[12]*z[7]*z[8] - 583315254144*z[12]*z[7]*z[9] + 2766452127744*z[12]*z[7] - 1164102202368*z[12]*z[8]*z[9] + 5518421972544*z[12]*z[8] + 10922106819456*z[12]*z[9] - 48569274464256*z[12] + 96*z[13]*z[14]*z[15]*z[16] + 192*z[13]*z[14]*z[15]*z[17] + 384*z[13]*z[14]*z[15]*z[18] - 756*z[13]*z[14]*z[15] + 384*z[13]*z[14]*z[16]*z[17] + 768*z[13]*z[14]*z[16]*z[18] - 1512*z[13]*z[14]*z[16] + 1536*z[13]*z[14]*z[17]*z[18] - 3024*z[13]*z[14]*z[17] - 6048*z[13]*z[14]*z[18] - 3072*z[13]*z[14]*z[2]*z[3]*z[4]*z[5] - 6144*z[13]*z[14]*z[2]*z[3]*z[4]*z[6] + 12096*z[13]*z[14]*z[2]*z[3]*z[4] - 12288*z[13]*z[14]*z[2]*z[3]*z[5]*z[6] + 24192*z[13]*z[14]*z[2]*z[3]*z[5] + 48384*z[13]*z[14]*z[2]*z[3]*z[6] - 63848*z[13]*z[14]*z[2]*z[3] - 24576*z[13]*z[14]*z[2]*z[4]*z[5]*z[6] + 48384*z[13]*z[14]*z[2]*z[4]*z[5] + 96768*z[13]*z[14]*z[2]*z[4]*z[6] - 126928*z[13]*z[14]*z[2]*z[4] + 193536*z[13]*z[14]*z[2]*z[5]*z[6] - 247712*z[13]*z[14]*z[2]*z[5] - 446272*z[13]*z[14]*z[2]*z[6] + 507528*z[13]*z[14]*z[2] - 49152*z[13]*z[14]*z[3]*z[4]*z[5]*z[6] + 96768*z[13]*z[14]*z[3]*z[4]*z[5] + 193536*z[13]*z[14]*z[3]*z[4]*z[6] - 253472*z[13]*z[14]*z[3]*z[4] + 387072*z[13]*z[14]*z[3]*z[5]*z[6] - 494656*z[13]*z[14]*z[3]*z[5] - 891008*z[13]*z[14]*z[3]*z[6] + 1012032*z[13]*z[14]*z[3] + 774144*z[13]*z[14]*z[4]*z[5]*z[6] - 983168*z[13]*z[14]*z[4]*z[5] - 1769728*z[13]*z[14]*z[4]*z[6] + 1999872*z[13]*z[14]*z[4] - 3441152*z[13]*z[14]*z[5]*z[6] + 3806208*z[13]*z[14]*z[5] + 6064128*z[13]*z[14]*z[6] + 1512*z[13]*z[14]*z[7]*z[8]*z[9] - 15992*z[13]*z[14]*z[7]*z[8] - 31936*z[13]*z[14]*z[7]*z[9] + 253953*z[13]*z[14]*z[7] - 63848*z[13]*z[14]*z[8]*z[9] + 507528*z[13]*z[14]*z[8] + 1012032*z[13]*z[14]*z[9] - 12894952*z[13]*z[14] + 768*z[13]*z[15]*z[16]*z[17] + 1536*z[13]*z[15]*z[16]*z[18] - 3024*z[13]*z[15]*z[16] + 3072*z[13]*z[15]*z[17]*z[18] - 6048*z[13]*z[15]*z[17] - 12096*z[13]*z[15]*z[18] - 6144*z[13]*z[15]*z[2]*z[3]*z[4]*z[5] - 12288*z[13]*z[15]*z[2]*z[3]*z[4]*z[6] + 24192*z[13]*z[15]*z[2]*z[3]*z[4] - 24576*z[13]*z[15]*z[2]*z[3]*z[5]*z[6] + 48384*z[13]*z[15]*z[2]*z[3]*z[5] + 96768*z[13]*z[15]*z[2]*z[3]*z[6] - 127696*z[13]*z[15]*z[2]*z[3] - 49152*z[13]*z[15]*z[2]*z[4]*z[5]*z[6] + 96768*z[13]*z[15]*z[2]*z[4]*z[5] + 193536*z[13]*z[15]*z[2]*z[4]*z[6] - 253856*z[13]*z[15]*z[2]*z[4] + 387072*z[13]*z[15]*z[2]*z[5]*z[6] - 495424*z[13]*z[15]*z[2]*z[5] - 892544*z[13]*z[15]*z[2]*z[6] + 1015056*z[13]*z[15]*z[2] - 98304*z[13]*z[15]*z[3]*z[4]*z[5]*z[6] + 193536*z[13]*z[15]*z[3]*z[4]*z[5] + 387072*z[13]*z[15]*z[3]*z[4]*z[6] - 506944*z[13]*z[15]*z[3]*z[4] + 774144*z[13]*z[15]*z[3]*z[5]*z[6] - 989312*z[13]*z[15]*z[3]*z[5] - 1782016*z[13]*z[15]*z[3]*z[6] + 2024064*z[13]*z[15]*z[3] + 1548288*z[13]*z[15]*z[4]*z[5]*z[6] - 1966336*z[13]*z[15]*z[4]*z[5] - 3539456*z[13]*z[15]*z[4]*z[6] + 3999744*z[13]*z[15]*z[4] - 6882304*z[13]*z[15]*z[5]*z[6] + 7612416*z[13]*z[15]*z[5] + 12128256*z[13]*z[15]*z[6] + 3024*z[13]*z[15]*z[7]*z[8]*z[9] - 31984*z[13]*z[15]*z[7]*z[8] - 63872*z[13]*z[15]*z[7]*z[9] + 507906*z[13]*z[15]*z[7] - 127696*z[13]*z[15]*z[8]*z[9] + 1015056*z[13]*z[15]*z[8] + 2024064*z[13]*z[15]*z[9] - 25789928*z[13]*z[15] + 6144*z[13]*z[16]*z[17]*z[18] - 12096*z[13]*z[16]*z[17] - 24192*z[13]*z[16]*z[18] - 12288*z[13]*z[16]*z[2]*z[3]*z[4]*z[5] - 24576*z[13]*z[16]*z[2]*z[3]*z[4]*z[6] + 48384*z[13]*z[16]*z[2]*z[3]*z[4] - 49152*z[13]*z[16]*z[2]*z[3]*z[5]*z[6] + 96768*z[13]*z[16]*z[2]*z[3]*z[5] + 193536*z[13]*z[16]*z[2]*z[3]*z[6] - 255392*z[13]*z[16]*z[2]*z[3] - 98304*z[13]*z[16]*z[2]*z[4]*z[5]*z[6] + 193536*z[13]*z[16]*z[2]*z[4]*z[5] + 387072*z[13]*z[16]*z[2]*z[4]*z[6] - 507712*z[13]*z[16]*z[2]*z[4] + 774144*z[13]*z[16]*z[2]*z[5]*z[6] - 990848*z[13]*z[16]*z[2]*z[5] - 1785088*z[13]*z[16]*z[2]*z[6] + 2030112*z[13]*z[16]*z[2] - 196608*z[13]*z[16]*z[3]*z[4]*z[5]*z[6] + 387072*z[13]*z[16]*z[3]*z[4]*z[5] + 774144*z[13]*z[16]*z[3]*z[4]*z[6] - 1013888*z[13]*z[16]*z[3]*z[4] + 1548288*z[13]*z[16]*z[3]*z[5]*z[6] - 1978624*z[13]*z[16]*z[3]*z[5] - 3564032*z[13]*z[16]*z[3]*z[6] + 4048128*z[13]*z[16]*z[3] + 3096576*z[13]*z[16]*z[4]*z[5]*z[6] - 3932672*z[13]*z[16]*z[4]*z[5] - 7078912*z[13]*z[16]*z[4]*z[6] + 7999488*z[13]*z[16]*z[4] - 13764608*z[13]*z[16]*z[5]*z[6] + 15224832*z[13]*z[16]*z[5] + 24256512*z[13]*z[16]*z[6] + 6048*z[13]*z[16]*z[7]*z[8]*z[9] - 63968*z[13]*z[16]*z[7]*z[8] - 127744*z[13]*z[16]*z[7]*z[9] + 1015812*z[13]*z[16]*z[7] - 255392*z[13]*z[16]*z[8]*z[9] + 2030112*z[13]*z[16]*z[8] + 4048128*z[13]*z[16]*z[9] - 51580048*z[13]*z[16] - 48384*z[13]*z[17]*z[18] - 24576*z[13]*z[17]*z[2]*z[3]*z[4]*z[5] - 49152*z[13]*z[17]*z[2]*z[3]*z[4]*z[6] + 96768*z[13]*z[17]*z[2]*z[3]*z[4] - 98304*z[13]*z[17]*z[2]*z[3]*z[5]*z[6] + 193536*z[13]*z[17]*z[2]*z[3]*z[5] + 387072*z[13]*z[17]*z[2]*z[3]*z[6] - 510784*z[13]*z[17]*z[2]*z[3] - 196608*z[13]*z[17]*z[2]*z[4]*z[5]*z[6] + 387072*z[13]*z[17]*z[2]*z[4]*z[5] + 774144*z[13]*z[17]*z[2]*z[4]*z[6] - 1015424*z[13]*z[17]*z[2]*z[4] + 1548288*z[13]*z[17]*z[2]*z[5]*z[6] - 1981696*z[13]*z[17]*z[2]*z[5] - 3570176*z[13]*z[17]*z[2]*z[6] + 4060224*z[13]*z[17]*z[2] - 393216*z[13]*z[17]*z[3]*z[4]*z[5]*z[6] + 774144*z[13]*z[17]*z[3]*z[4]*z[5] + 1548288*z[13]*z[17]*z[3]*z[4]*z[6] - 2027776*z[13]*z[17]*z[3]*z[4] + 3096576*z[13]*z[17]*z[3]*z[5]*z[6] - 3957248*z[13]*z[17]*z[3]*z[5] - 7128064*z[13]*z[17]*z[3]*z[6] + 8096256*z[13]*z[17]*z[3] + 6193152*z[13]*z[17]*z[4]*z[5]*z[6] - 7865344*z[13]*z[17]*z[4]*z[5] - 14157824*z[13]*z[17]*z[4]*z[6] + 15998976*z[13]*z[17]*z[4] - 27529216*z[13]*z[17]*z[5]*z[6] + 30449664*z[13]*z[17]*z[5] + 48513024*z[13]*z[17]*z[6] + 12096*z[13]*z[17]*z[7]*z[8]*z[9] - 127936*z[13]*z[17]*z[7]*z[8] - 255488*z[13]*z[17]*z[7]*z[9] + 2031624*z[13]*z[17]*z[7] - 510784*z[13]*z[17]*z[8]*z[9] + 4060224*z[13]*z[17]*z[8] + 8096256*z[13]*z[17]*z[9] - 103161632*z[13]*z[17] - 49152*z[13]*z[18]*z[2]*z[3]*z[4]*z[5] - 98304*z[13]*z[18]*z[2]*z[3]*z[4]*z[6] + 193536*z[13]*z[18]*z[2]*z[3]*z[4] - 196608*z[13]*z[18]*z[2]*z[3]*z[5]*z[6] + 387072*z[13]*z[18]*z[2]*z[3]*z[5] + 774144*z[13]*z[18]*z[2]*z[3]*z[6] - 1021568*z[13]*z[18]*z[2]*z[3] - 393216*z[13]*z[18]*z[2]*z[4]*z[5]*z[6] + 774144*z[13]*z[18]*z[2]*z[4]*z[5] + 1548288*z[13]*z[18]*z[2]*z[4]*z[6] - 2030848*z[13]*z[18]*z[2]*z[4] + 3096576*z[13]*z[18]*z[2]*z[5]*z[6] - 3963392*z[13]*z[18]*z[2]*z[5] - 7140352*z[13]*z[18]*z[2]*z[6] + 8120448*z[13]*z[18]*z[2] - 786432*z[13]*z[18]*z[3]*z[4]*z[5]*z[6] + 1548288*z[13]*z[18]*z[3]*z[4]*z[5] + 3096576*z[13]*z[18]*z[3]*z[4]*z[6] - 4055552*z[13]*z[18]*z[3]*z[4] + 6193152*z[13]*z[18]*z[3]*z[5]*z[6] - 7914496*z[13]*z[18]*z[3]*z[5] - 14256128*z[13]*z[18]*z[3]*z[6] + 16192512*z[13]*z[18]*z[3] + 12386304*z[13]*z[18]*z[4]*z[5]*z[6] - 15730688*z[13]*z[18]*z[4]*z[5] - 28315648*z[13]*z[18]*z[4]*z[6] + 31997952*z[13]*z[18]*z[4] - 55058432*z[13]*z[18]*z[5]*z[6] + 60899328*z[13]*z[18]*z[5] + 97026048*z[13]*z[18]*z[6] + 24192*z[13]*z[18]*z[7]*z[8]*z[9] - 255872*z[13]*z[18]*z[7]*z[8] - 510976*z[13]*z[18]*z[7]*z[9] + 4063248*z[13]*z[18]*z[7] - 1021568*z[13]*z[18]*z[8]*z[9] + 8120448*z[13]*z[18]*z[8] + 16192512*z[13]*z[18]*z[9] - 206335552*z[13]*z[18] + 96768*z[13]*z[2]*z[3]*z[4]*z[5] + 193536*z[13]*z[2]*z[3]*z[4]*z[6] - 381024*z[13]*z[2]*z[3]*z[4] + 387072*z[13]*z[2]*z[3]*z[5]*z[6] - 762048*z[13]*z[2]*z[3]*z[5] - 1524096*z[13]*z[2]*z[3]*z[6] + 2011212*z[13]*z[2]*z[3] + 774144*z[13]*z[2]*z[4]*z[5]*z[6] - 1524096*z[13]*z[2]*z[4]*z[5] - 3048192*z[13]*z[2]*z[4]*z[6] + 3998232*z[13]*z[2]*z[4] - 6096384*z[13]*z[2]*z[5]*z[6] + 7802928*z[13]*z[2]*z[5] + 14057568*z[13]*z[2]*z[6] - 15987132*z[13]*z[2] + 1548288*z[13]*z[3]*z[4]*z[5]*z[6] - 3048192*z[13]*z[3]*z[4]*z[5] - 6096384*z[13]*z[3]*z[4]*z[6] + 7984368*z[13]*z[3]*z[4] - 12192768*z[13]*z[3]*z[5]*z[6] + 15581664*z[13]*z[3]*z[5] + 28066752*z[13]*z[3]*z[6] - 31879008*z[13]*z[3] - 24385536*z[13]*z[4]*z[5]*z[6] + 30969792*z[13]*z[4]*z[5] + 55746432*z[13]*z[4]*z[6] - 62995968*z[13]*z[4] + 108396288*z[13]*z[5]*z[6] - 119895552*z[13]*z[5] - 191020032*z[13]*z[6] - 47628*z[13]*z[7]*z[8]*z[9] + 503748*z[13]*z[7]*z[8] + 1005984*z[13]*z[7]*z[9] - 15999039*z[13]*z[7]/2 + 2011212*z[13]*z[8]*z[9] - 15987132*z[13]*z[8] - 31879008*z[13]*z[9] + 812631771*z[13]/2 + 1536*z[14]*z[15]*z[16]*z[17] + 3072*z[14]*z[15]*z[16]*z[18] - 6048*z[14]*z[15]*z[16] + 6144*z[14]*z[15]*z[17]*z[18] - 12096*z[14]*z[15]*z[17] - 24192*z[14]*z[15]*z[18] - 12288*z[14]*z[15]*z[2]*z[3]*z[4]*z[5] - 24576*z[14]*z[15]*z[2]*z[3]*z[4]*z[6] + 48384*z[14]*z[15]*z[2]*z[3]*z[4] - 49152*z[14]*z[15]*z[2]*z[3]*z[5]*z[6] + 96768*z[14]*z[15]*z[2]*z[3]*z[5] + 193536*z[14]*z[15]*z[2]*z[3]*z[6] - 255392*z[14]*z[15]*z[2]*z[3] - 98304*z[14]*z[15]*z[2]*z[4]*z[5]*z[6] + 193536*z[14]*z[15]*z[2]*z[4]*z[5] + 387072*z[14]*z[15]*z[2]*z[4]*z[6] - 507712*z[14]*z[15]*z[2]*z[4] + 774144*z[14]*z[15]*z[2]*z[5]*z[6] - 990848*z[14]*z[15]*z[2]*z[5] - 1785088*z[14]*z[15]*z[2]*z[6] + 2030112*z[14]*z[15]*z[2] - 196608*z[14]*z[15]*z[3]*z[4]*z[5]*z[6] + 387072*z[14]*z[15]*z[3]*z[4]*z[5] + 774144*z[14]*z[15]*z[3]*z[4]*z[6] - 1013888*z[14]*z[15]*z[3]*z[4] + 1548288*z[14]*z[15]*z[3]*z[5]*z[6] - 1978624*z[14]*z[15]*z[3]*z[5] - 3564032*z[14]*z[15]*z[3]*z[6] + 4048128*z[14]*z[15]*z[3] + 3096576*z[14]*z[15]*z[4]*z[5]*z[6] - 3932672*z[14]*z[15]*z[4]*z[5] - 7078912*z[14]*z[15]*z[4]*z[6] + 7999488*z[14]*z[15]*z[4] - 13764608*z[14]*z[15]*z[5]*z[6] + 15224832*z[14]*z[15]*z[5] + 24256512*z[14]*z[15]*z[6] + 6048*z[14]*z[15]*z[7]*z[8]*z[9] - 63968*z[14]*z[15]*z[7]*z[8] - 127744*z[14]*z[15]*z[7]*z[9] + 1015812*z[14]*z[15]*z[7] - 255392*z[14]*z[15]*z[8]*z[9] + 2030112*z[14]*z[15]*z[8] + 4048128*z[14]*z[15]*z[9] - 51579868*z[14]*z[15] + 12288*z[14]*z[16]*z[17]*z[18] - 24192*z[14]*z[16]*z[17] - 48384*z[14]*z[16]*z[18] - 24576*z[14]*z[16]*z[2]*z[3]*z[4]*z[5] - 49152*z[14]*z[16]*z[2]*z[3]*z[4]*z[6] + 96768*z[14]*z[16]*z[2]*z[3]*z[4] - 98304*z[14]*z[16]*z[2]*z[3]*z[5]*z[6] + 193536*z[14]*z[16]*z[2]*z[3]*z[5] + 387072*z[14]*z[16]*z[2]*z[3]*z[6] - 510784*z[14]*z[16]*z[2]*z[3] - 196608*z[14]*z[16]*z[2]*z[4]*z[5]*z[6] + 387072*z[14]*z[16]*z[2]*z[4]*z[5] + 774144*z[14]*z[16]*z[2]*z[4]*z[6] - 1015424*z[14]*z[16]*z[2]*z[4] + 1548288*z[14]*z[16]*z[2]*z[5]*z[6] - 1981696*z[14]*z[16]*z[2]*z[5] - 3570176*z[14]*z[16]*z[2]*z[6] + 4060224*z[14]*z[16]*z[2] - 393216*z[14]*z[16]*z[3]*z[4]*z[5]*z[6] + 774144*z[14]*z[16]*z[3]*z[4]*z[5] + 1548288*z[14]*z[16]*z[3]*z[4]*z[6] - 2027776*z[14]*z[16]*z[3]*z[4] + 3096576*z[14]*z[16]*z[3]*z[5]*z[6] - 3957248*z[14]*z[16]*z[3]*z[5] - 7128064*z[14]*z[16]*z[3]*z[6] + 8096256*z[14]*z[16]*z[3] + 6193152*z[14]*z[16]*z[4]*z[5]*z[6] - 7865344*z[14]*z[16]*z[4]*z[5] - 14157824*z[14]*z[16]*z[4]*z[6] + 15998976*z[14]*z[16]*z[4] - 27529216*z[14]*z[16]*z[5]*z[6] + 30449664*z[14]*z[16]*z[5] + 48513024*z[14]*z[16]*z[6] + 12096*z[14]*z[16]*z[7]*z[8]*z[9] - 127936*z[14]*z[16]*z[7]*z[8] - 255488*z[14]*z[16]*z[7]*z[9] + 2031624*z[14]*z[16]*z[7] - 510784*z[14]*z[16]*z[8]*z[9] + 4060224*z[14]*z[16]*z[8] + 8096256*z[14]*z[16]*z[9] - 103160120*z[14]*z[16] - 96768*z[14]*z[17]*z[18] - 49152*z[14]*z[17]*z[2]*z[3]*z[4]*z[5] - 98304*z[14]*z[17]*z[2]*z[3]*z[4]*z[6] + 193536*z[14]*z[17]*z[2]*z[3]*z[4] - 196608*z[14]*z[17]*z[2]*z[3]*z[5]*z[6] + 387072*z[14]*z[17]*z[2]*z[3]*z[5] + 774144*z[14]*z[17]*z[2]*z[3]*z[6] - 1021568*z[14]*z[17]*z[2]*z[3] - 393216*z[14]*z[17]*z[2]*z[4]*z[5]*z[6] + 774144*z[14]*z[17]*z[2]*z[4]*z[5] + 1548288*z[14]*z[17]*z[2]*z[4]*z[6] - 2030848*z[14]*z[17]*z[2]*z[4] + 3096576*z[14]*z[17]*z[2]*z[5]*z[6] - 3963392*z[14]*z[17]*z[2]*z[5] - 7140352*z[14]*z[17]*z[2]*z[6] + 8120448*z[14]*z[17]*z[2] - 786432*z[14]*z[17]*z[3]*z[4]*z[5]*z[6] + 1548288*z[14]*z[17]*z[3]*z[4]*z[5] + 3096576*z[14]*z[17]*z[3]*z[4]*z[6] - 4055552*z[14]*z[17]*z[3]*z[4] + 6193152*z[14]*z[17]*z[3]*z[5]*z[6] - 7914496*z[14]*z[17]*z[3]*z[5] - 14256128*z[14]*z[17]*z[3]*z[6] + 16192512*z[14]*z[17]*z[3] + 12386304*z[14]*z[17]*z[4]*z[5]*z[6] - 15730688*z[14]*z[17]*z[4]*z[5] - 28315648*z[14]*z[17]*z[4]*z[6] + 31997952*z[14]*z[17]*z[4] - 55058432*z[14]*z[17]*z[5]*z[6] + 60899328*z[14]*z[17]*z[5] + 97026048*z[14]*z[17]*z[6] + 24192*z[14]*z[17]*z[7]*z[8]*z[9] - 255872*z[14]*z[17]*z[7]*z[8] - 510976*z[14]*z[17]*z[7]*z[9] + 4063248*z[14]*z[17]*z[7] - 1021568*z[14]*z[17]*z[8]*z[9] + 8120448*z[14]*z[17]*z[8] + 16192512*z[14]*z[17]*z[9] - 206323312*z[14]*z[17] - 98304*z[14]*z[18]*z[2]*z[3]*z[4]*z[5] - 196608*z[14]*z[18]*z[2]*z[3]*z[4]*z[6] + 387072*z[14]*z[18]*z[2]*z[3]*z[4] - 393216*z[14]*z[18]*z[2]*z[3]*z[5]*z[6] + 774144*z[14]*z[18]*z[2]*z[3]*z[5] + 1548288*z[14]*z[18]*z[2]*z[3]*z[6] - 2043136*z[14]*z[18]*z[2]*z[3] - 786432*z[14]*z[18]*z[2]*z[4]*z[5]*z[6] + 1548288*z[14]*z[18]*z[2]*z[4]*z[5] + 3096576*z[14]*z[18]*z[2]*z[4]*z[6] - 4061696*z[14]*z[18]*z[2]*z[4] + 6193152*z[14]*z[18]*z[2]*z[5]*z[6] - 7926784*z[14]*z[18]*z[2]*z[5] - 14280704*z[14]*z[18]*z[2]*z[6] + 16240896*z[14]*z[18]*z[2] - 1572864*z[14]*z[18]*z[3]*z[4]*z[5]*z[6] + 3096576*z[14]*z[18]*z[3]*z[4]*z[5] + 6193152*z[14]*z[18]*z[3]*z[4]*z[6] - 8111104*z[14]*z[18]*z[3]*z[4] + 12386304*z[14]*z[18]*z[3]*z[5]*z[6] - 15828992*z[14]*z[18]*z[3]*z[5] - 28512256*z[14]*z[18]*z[3]*z[6] + 32385024*z[14]*z[18]*z[3] + 24772608*z[14]*z[18]*z[4]*z[5]*z[6] - 31461376*z[14]*z[18]*z[4]*z[5] - 56631296*z[14]*z[18]*z[4]*z[6] + 63995904*z[14]*z[18]*z[4] - 110116864*z[14]*z[18]*z[5]*z[6] + 121798656*z[14]*z[18]*z[5] + 194052096*z[14]*z[18]*z[6] + 48384*z[14]*z[18]*z[7]*z[8]*z[9] - 511744*z[14]*z[18]*z[7]*z[8] - 1021952*z[14]*z[18]*z[7]*z[9] + 8126496*z[14]*z[18]*z[7] - 2043136*z[14]*z[18]*z[8]*z[9] + 16240896*z[14]*z[18]*z[8] + 32385024*z[14]*z[18]*z[9] - 412671200*z[14]*z[18] + 193536*z[14]*z[2]*z[3]*z[4]*z[5] + 387072*z[14]*z[2]*z[3]*z[4]*z[6] - 762048*z[14]*z[2]*z[3]*z[4] + 774144*z[14]*z[2]*z[3]*z[5]*z[6] - 1524096*z[14]*z[2]*z[3]*z[5] - 3048192*z[14]*z[2]*z[3]*z[6] + 4022424*z[14]*z[2]*z[3] + 1548288*z[14]*z[2]*z[4]*z[5]*z[6] - 3048192*z[14]*z[2]*z[4]*z[5] - 6096384*z[14]*z[2]*z[4]*z[6] + 7996464*z[14]*z[2]*z[4] - 12192768*z[14]*z[2]*z[5]*z[6] + 15605856*z[14]*z[2]*z[5] + 28115136*z[14]*z[2]*z[6] - 31974264*z[14]*z[2] + 3096576*z[14]*z[3]*z[4]*z[5]*z[6] - 6096384*z[14]*z[3]*z[4]*z[5] - 12192768*z[14]*z[3]*z[4]*z[6] + 15968736*z[14]*z[3]*z[4] - 24385536*z[14]*z[3]*z[5]*z[6] + 31163328*z[14]*z[3]*z[5] + 56133504*z[14]*z[3]*z[6] - 63758016*z[14]*z[3] - 48771072*z[14]*z[4]*z[5]*z[6] + 61939584*z[14]*z[4]*z[5] + 111492864*z[14]*z[4]*z[6] - 125991936*z[14]*z[4] + 216792576*z[14]*z[5]*z[6] - 239791104*z[14]*z[5] - 382040064*z[14]*z[6] - 95256*z[14]*z[7]*z[8]*z[9] + 1007496*z[14]*z[7]*z[8] + 2011968*z[14]*z[7]*z[9] - 15999039*z[14]*z[7] + 4022424*z[14]*z[8]*z[9] - 31974264*z[14]*z[8] - 63758016*z[14]*z[9] + 812631960*z[14] + 24576*z[15]*z[16]*z[17]*z[18] - 48384*z[15]*z[16]*z[17] - 96768*z[15]*z[16]*z[18] - 49152*z[15]*z[16]*z[2]*z[3]*z[4]*z[5] - 98304*z[15]*z[16]*z[2]*z[3]*z[4]*z[6] + 193536*z[15]*z[16]*z[2]*z[3]*z[4] - 196608*z[15]*z[16]*z[2]*z[3]*z[5]*z[6] + 387072*z[15]*z[16]*z[2]*z[3]*z[5] + 774144*z[15]*z[16]*z[2]*z[3]*z[6] - 1021568*z[15]*z[16]*z[2]*z[3] - 393216*z[15]*z[16]*z[2]*z[4]*z[5]*z[6] + 774144*z[15]*z[16]*z[2]*z[4]*z[5] + 1548288*z[15]*z[16]*z[2]*z[4]*z[6] - 2030848*z[15]*z[16]*z[2]*z[4] + 3096576*z[15]*z[16]*z[2]*z[5]*z[6] - 3963392*z[15]*z[16]*z[2]*z[5] - 7140352*z[15]*z[16]*z[2]*z[6] + 8120448*z[15]*z[16]*z[2] - 786432*z[15]*z[16]*z[3]*z[4]*z[5]*z[6] + 1548288*z[15]*z[16]*z[3]*z[4]*z[5] + 3096576*z[15]*z[16]*z[3]*z[4]*z[6] - 4055552*z[15]*z[16]*z[3]*z[4] + 6193152*z[15]*z[16]*z[3]*z[5]*z[6] - 7914496*z[15]*z[16]*z[3]*z[5] - 14256128*z[15]*z[16]*z[3]*z[6] + 16192512*z[15]*z[16]*z[3] + 12386304*z[15]*z[16]*z[4]*z[5]*z[6] - 15730688*z[15]*z[16]*z[4]*z[5] - 28315648*z[15]*z[16]*z[4]*z[6] + 31997952*z[15]*z[16]*z[4] - 55058432*z[15]*z[16]*z[5]*z[6] + 60899328*z[15]*z[16]*z[5] + 97026048*z[15]*z[16]*z[6] + 24192*z[15]*z[16]*z[7]*z[8]*z[9] - 255872*z[15]*z[16]*z[7]*z[8] - 510976*z[15]*z[16]*z[7]*z[9] + 4063248*z[15]*z[16]*z[7] - 1021568*z[15]*z[16]*z[8]*z[9] + 8120448*z[15]*z[16]*z[8] + 16192512*z[15]*z[16]*z[9] - 206320432*z[15]*z[16] - 193536*z[15]*z[17]*z[18] - 98304*z[15]*z[17]*z[2]*z[3]*z[4]*z[5] - 196608*z[15]*z[17]*z[2]*z[3]*z[4]*z[6] + 387072*z[15]*z[17]*z[2]*z[3]*z[4] - 393216*z[15]*z[17]*z[2]*z[3]*z[5]*z[6] + 774144*z[15]*z[17]*z[2]*z[3]*z[5] + 1548288*z[15]*z[17]*z[2]*z[3]*z[6] - 2043136*z[15]*z[17]*z[2]*z[3] - 786432*z[15]*z[17]*z[2]*z[4]*z[5]*z[6] + 1548288*z[15]*z[17]*z[2]*z[4]*z[5] + 3096576*z[15]*z[17]*z[2]*z[4]*z[6] - 4061696*z[15]*z[17]*z[2]*z[4] + 6193152*z[15]*z[17]*z[2]*z[5]*z[6] - 7926784*z[15]*z[17]*z[2]*z[5] - 14280704*z[15]*z[17]*z[2]*z[6] + 16240896*z[15]*z[17]*z[2] - 1572864*z[15]*z[17]*z[3]*z[4]*z[5]*z[6] + 3096576*z[15]*z[17]*z[3]*z[4]*z[5] + 6193152*z[15]*z[17]*z[3]*z[4]*z[6] - 8111104*z[15]*z[17]*z[3]*z[4] + 12386304*z[15]*z[17]*z[3]*z[5]*z[6] - 15828992*z[15]*z[17]*z[3]*z[5] - 28512256*z[15]*z[17]*z[3]*z[6] + 32385024*z[15]*z[17]*z[3] + 24772608*z[15]*z[17]*z[4]*z[5]*z[6] - 31461376*z[15]*z[17]*z[4]*z[5] - 56631296*z[15]*z[17]*z[4]*z[6] + 63995904*z[15]*z[17]*z[4] - 110116864*z[15]*z[17]*z[5]*z[6] + 121798656*z[15]*z[17]*z[5] + 194052096*z[15]*z[17]*z[6] + 48384*z[15]*z[17]*z[7]*z[8]*z[9] - 511744*z[15]*z[17]*z[7]*z[8] - 1021952*z[15]*z[17]*z[7]*z[9] + 8126496*z[15]*z[17]*z[7] - 2043136*z[15]*z[17]*z[8]*z[9] + 16240896*z[15]*z[17]*z[8] + 32385024*z[15]*z[17]*z[9] - 412647008*z[15]*z[17] - 196608*z[15]*z[18]*z[2]*z[3]*z[4]*z[5] - 393216*z[15]*z[18]*z[2]*z[3]*z[4]*z[6] + 774144*z[15]*z[18]*z[2]*z[3]*z[4] - 786432*z[15]*z[18]*z[2]*z[3]*z[5]*z[6] + 1548288*z[15]*z[18]*z[2]*z[3]*z[5] + 3096576*z[15]*z[18]*z[2]*z[3]*z[6] - 4086272*z[15]*z[18]*z[2]*z[3] - 1572864*z[15]*z[18]*z[2]*z[4]*z[5]*z[6] + 3096576*z[15]*z[18]*z[2]*z[4]*z[5] + 6193152*z[15]*z[18]*z[2]*z[4]*z[6] - 8123392*z[15]*z[18]*z[2]*z[4] + 12386304*z[15]*z[18]*z[2]*z[5]*z[6] - 15853568*z[15]*z[18]*z[2]*z[5] - 28561408*z[15]*z[18]*z[2]*z[6] + 32481792*z[15]*z[18]*z[2] - 3145728*z[15]*z[18]*z[3]*z[4]*z[5]*z[6] + 6193152*z[15]*z[18]*z[3]*z[4]*z[5] + 12386304*z[15]*z[18]*z[3]*z[4]*z[6] - 16222208*z[15]*z[18]*z[3]*z[4] + 24772608*z[15]*z[18]*z[3]*z[5]*z[6] - 31657984*z[15]*z[18]*z[3]*z[5] - 57024512*z[15]*z[18]*z[3]*z[6] + 64770048*z[15]*z[18]*z[3] + 49545216*z[15]*z[18]*z[4]*z[5]*z[6] - 62922752*z[15]*z[18]*z[4]*z[5] - 113262592*z[15]*z[18]*z[4]*z[6] + 127991808*z[15]*z[18]*z[4] - 220233728*z[15]*z[18]*z[5]*z[6] + 243597312*z[15]*z[18]*z[5] + 388104192*z[15]*z[18]*z[6] + 96768*z[15]*z[18]*z[7]*z[8]*z[9] - 1023488*z[15]*z[18]*z[7]*z[8] - 2043904*z[15]*z[18]*z[7]*z[9] + 16252992*z[15]*z[18]*z[7] - 4086272*z[15]*z[18]*z[8]*z[9] + 32481792*z[15]*z[18]*z[8] + 64770048*z[15]*z[18]*z[9] - 825343168*z[15]*z[18] + 387072*z[15]*z[2]*z[3]*z[4]*z[5] + 774144*z[15]*z[2]*z[3]*z[4]*z[6] - 1524096*z[15]*z[2]*z[3]*z[4] + 1548288*z[15]*z[2]*z[3]*z[5]*z[6] - 3048192*z[15]*z[2]*z[3]*z[5] - 6096384*z[15]*z[2]*z[3]*z[6] + 8044848*z[15]*z[2]*z[3] + 3096576*z[15]*z[2]*z[4]*z[5]*z[6] - 6096384*z[15]*z[2]*z[4]*z[5] - 12192768*z[15]*z[2]*z[4]*z[6] + 15992928*z[15]*z[2]*z[4] - 24385536*z[15]*z[2]*z[5]*z[6] + 31211712*z[15]*z[2]*z[5] + 56230272*z[15]*z[2]*z[6] - 63948528*z[15]*z[2] + 6193152*z[15]*z[3]*z[4]*z[5]*z[6] - 12192768*z[15]*z[3]*z[4]*z[5] - 24385536*z[15]*z[3]*z[4]*z[6] + 31937472*z[15]*z[3]*z[4] - 48771072*z[15]*z[3]*z[5]*z[6] + 62326656*z[15]*z[3]*z[5] + 112267008*z[15]*z[3]*z[6] - 127516032*z[15]*z[3] - 97542144*z[15]*z[4]*z[5]*z[6] + 123879168*z[15]*z[4]*z[5] + 222985728*z[15]*z[4]*z[6] - 251983872*z[15]*z[4] + 433585152*z[15]*z[5]*z[6] - 479582208*z[15]*z[5] - 764080128*z[15]*z[6] - 190512*z[15]*z[7]*z[8]*z[9] + 2014992*z[15]*z[7]*z[8] + 4023936*z[15]*z[7]*z[9] - 31998078*z[15]*z[7] + 8044848*z[15]*z[8]*z[9] - 63948528*z[15]*z[8] - 127516032*z[15]*z[9] + 1625265432*z[15] - 387072*z[16]*z[17]*z[18] - 196608*z[16]*z[17]*z[2]*z[3]*z[4]*z[5] - 393216*z[16]*z[17]*z[2]*z[3]*z[4]*z[6] + 774144*z[16]*z[17]*z[2]*z[3]*z[4] - 786432*z[16]*z[17]*z[2]*z[3]*z[5]*z[6] + 1548288*z[16]*z[17]*z[2]*z[3]*z[5] + 3096576*z[16]*z[17]*z[2]*z[3]*z[6] - 4086272*z[16]*z[17]*z[2]*z[3] - 1572864*z[16]*z[17]*z[2]*z[4]*z[5]*z[6] + 3096576*z[16]*z[17]*z[2]*z[4]*z[5] + 6193152*z[16]*z[17]*z[2]*z[4]*z[6] - 8123392*z[16]*z[17]*z[2]*z[4] + 12386304*z[16]*z[17]*z[2]*z[5]*z[6] - 15853568*z[16]*z[17]*z[2]*z[5] - 28561408*z[16]*z[17]*z[2]*z[6] + 32481792*z[16]*z[17]*z[2] - 3145728*z[16]*z[17]*z[3]*z[4]*z[5]*z[6] + 6193152*z[16]*z[17]*z[3]*z[4]*z[5] + 12386304*z[16]*z[17]*z[3]*z[4]*z[6] - 16222208*z[16]*z[17]*z[3]*z[4] + 24772608*z[16]*z[17]*z[3]*z[5]*z[6] - 31657984*z[16]*z[17]*z[3]*z[5] - 57024512*z[16]*z[17]*z[3]*z[6] + 64770048*z[16]*z[17]*z[3] + 49545216*z[16]*z[17]*z[4]*z[5]*z[6] - 62922752*z[16]*z[17]*z[4]*z[5] - 113262592*z[16]*z[17]*z[4]*z[6] + 127991808*z[16]*z[17]*z[4] - 220233728*z[16]*z[17]*z[5]*z[6] + 243597312*z[16]*z[17]*z[5] + 388104192*z[16]*z[17]*z[6] + 96768*z[16]*z[17]*z[7]*z[8]*z[9] - 1023488*z[16]*z[17]*z[7]*z[8] - 2043904*z[16]*z[17]*z[7]*z[9] + 16252992*z[16]*z[17]*z[7] - 4086272*z[16]*z[17]*z[8]*z[9] + 32481792*z[16]*z[17]*z[8] + 64770048*z[16]*z[17]*z[9] - 825297088*z[16]*z[17] - 393216*z[16]*z[18]*z[2]*z[3]*z[4]*z[5] - 786432*z[16]*z[18]*z[2]*z[3]*z[4]*z[6] + 1548288*z[16]*z[18]*z[2]*z[3]*z[4] - 1572864*z[16]*z[18]*z[2]*z[3]*z[5]*z[6] + 3096576*z[16]*z[18]*z[2]*z[3]*z[5] + 6193152*z[16]*z[18]*z[2]*z[3]*z[6] - 8172544*z[16]*z[18]*z[2]*z[3] - 3145728*z[16]*z[18]*z[2]*z[4]*z[5]*z[6] + 6193152*z[16]*z[18]*z[2]*z[4]*z[5] + 12386304*z[16]*z[18]*z[2]*z[4]*z[6] - 16246784*z[16]*z[18]*z[2]*z[4] + 24772608*z[16]*z[18]*z[2]*z[5]*z[6] - 31707136*z[16]*z[18]*z[2]*z[5] - 57122816*z[16]*z[18]*z[2]*z[6] + 64963584*z[16]*z[18]*z[2] - 6291456*z[16]*z[18]*z[3]*z[4]*z[5]*z[6] + 12386304*z[16]*z[18]*z[3]*z[4]*z[5] + 24772608*z[16]*z[18]*z[3]*z[4]*z[6] - 32444416*z[16]*z[18]*z[3]*z[4] + 49545216*z[16]*z[18]*z[3]*z[5]*z[6] - 63315968*z[16]*z[18]*z[3]*z[5] - 114049024*z[16]*z[18]*z[3]*z[6] + 129540096*z[16]*z[18]*z[3] + 99090432*z[16]*z[18]*z[4]*z[5]*z[6] - 125845504*z[16]*z[18]*z[4]*z[5] - 226525184*z[16]*z[18]*z[4]*z[6] + 255983616*z[16]*z[18]*z[4] - 440467456*z[16]*z[18]*z[5]*z[6] + 487194624*z[16]*z[18]*z[5] + 776208384*z[16]*z[18]*z[6] + 193536*z[16]*z[18]*z[7]*z[8]*z[9] - 2046976*z[16]*z[18]*z[7]*z[8] - 4087808*z[16]*z[18]*z[7]*z[9] + 32505984*z[16]*z[18]*z[7] - 8172544*z[16]*z[18]*z[8]*z[9] + 64963584*z[16]*z[18]*z[8] + 129540096*z[16]*z[18]*z[9] - 1650692480*z[16]*z[18] + 774144*z[16]*z[2]*z[3]*z[4]*z[5] + 1548288*z[16]*z[2]*z[3]*z[4]*z[6] - 3048192*z[16]*z[2]*z[3]*z[4] + 3096576*z[16]*z[2]*z[3]*z[5]*z[6] - 6096384*z[16]*z[2]*z[3]*z[5] - 12192768*z[16]*z[2]*z[3]*z[6] + 16089696*z[16]*z[2]*z[3] + 6193152*z[16]*z[2]*z[4]*z[5]*z[6] - 12192768*z[16]*z[2]*z[4]*z[5] - 24385536*z[16]*z[2]*z[4]*z[6] + 31985856*z[16]*z[2]*z[4] - 48771072*z[16]*z[2]*z[5]*z[6] + 62423424*z[16]*z[2]*z[5] + 112460544*z[16]*z[2]*z[6] - 127897056*z[16]*z[2] + 12386304*z[16]*z[3]*z[4]*z[5]*z[6] - 24385536*z[16]*z[3]*z[4]*z[5] - 48771072*z[16]*z[3]*z[4]*z[6] + 63874944*z[16]*z[3]*z[4] - 97542144*z[16]*z[3]*z[5]*z[6] + 124653312*z[16]*z[3]*z[5] + 224534016*z[16]*z[3]*z[6] - 255032064*z[16]*z[3] - 195084288*z[16]*z[4]*z[5]*z[6] + 247758336*z[16]*z[4]*z[5] + 445971456*z[16]*z[4]*z[6] - 503967744*z[16]*z[4] + 867170304*z[16]*z[5]*z[6] - 959164416*z[16]*z[5] - 1528160256*z[16]*z[6] - 381024*z[16]*z[7]*z[8]*z[9] + 4029984*z[16]*z[7]*z[8] + 8047872*z[16]*z[7]*z[9] - 63996156*z[16]*z[7] + 16089696*z[16]*z[8]*z[9] - 127897056*z[16]*z[8] - 255032064*z[16]*z[9] + 3250542960*z[16] - 786432*z[17]*z[18]*z[2]*z[3]*z[4]*z[5] - 1572864*z[17]*z[18]*z[2]*z[3]*z[4]*z[6] + 3096576*z[17]*z[18]*z[2]*z[3]*z[4] - 3145728*z[17]*z[18]*z[2]*z[3]*z[5]*z[6] + 6193152*z[17]*z[18]*z[2]*z[3]*z[5] + 12386304*z[17]*z[18]*z[2]*z[3]*z[6] - 16345088*z[17]*z[18]*z[2]*z[3] - 6291456*z[17]*z[18]*z[2]*z[4]*z[5]*z[6] + 12386304*z[17]*z[18]*z[2]*z[4]*z[5] + 24772608*z[17]*z[18]*z[2]*z[4]*z[6] - 32493568*z[17]*z[18]*z[2]*z[4] + 49545216*z[17]*z[18]*z[2]*z[5]*z[6] - 63414272*z[17]*z[18]*z[2]*z[5] - 114245632*z[17]*z[18]*z[2]*z[6] + 129927168*z[17]*z[18]*z[2] - 12582912*z[17]*z[18]*z[3]*z[4]*z[5]*z[6] + 24772608*z[17]*z[18]*z[3]*z[4]*z[5] + 49545216*z[17]*z[18]*z[3]*z[4]*z[6] - 64888832*z[17]*z[18]*z[3]*z[4] + 99090432*z[17]*z[18]*z[3]*z[5]*z[6] - 126631936*z[17]*z[18]*z[3]*z[5] - 228098048*z[17]*z[18]*z[3]*z[6] + 259080192*z[17]*z[18]*z[3] + 198180864*z[17]*z[18]*z[4]*z[5]*z[6] - 251691008*z[17]*z[18]*z[4]*z[5] - 453050368*z[17]*z[18]*z[4]*z[6] + 511967232*z[17]*z[18]*z[4] - 880934912*z[17]*z[18]*z[5]*z[6] + 974389248*z[17]*z[18]*z[5] + 1552416768*z[17]*z[18]*z[6] + 387072*z[17]*z[18]*z[7]*z[8]*z[9] - 4093952*z[17]*z[18]*z[7]*z[8] - 8175616*z[17]*z[18]*z[7]*z[9] + 65011968*z[17]*z[18]*z[7] - 16345088*z[17]*z[18]*z[8]*z[9] + 129927168*z[17]*z[18]*z[8] + 259080192*z[17]*z[18]*z[9] - 3301434112*z[17]*z[18] + 1548288*z[17]*z[2]*z[3]*z[4]*z[5] + 3096576*z[17]*z[2]*z[3]*z[4]*z[6] - 6096384*z[17]*z[2]*z[3]*z[4] + 6193152*z[17]*z[2]*z[3]*z[5]*z[6] - 12192768*z[17]*z[2]*z[3]*z[5] - 24385536*z[17]*z[2]*z[3]*z[6] + 32179392*z[17]*z[2]*z[3] + 12386304*z[17]*z[2]*z[4]*z[5]*z[6] - 24385536*z[17]*z[2]*z[4]*z[5] - 48771072*z[17]*z[2]*z[4]*z[6] + 63971712*z[17]*z[2]*z[4] - 97542144*z[17]*z[2]*z[5]*z[6] + 124846848*z[17]*z[2]*z[5] + 224921088*z[17]*z[2]*z[6] - 255794112*z[17]*z[2] + 24772608*z[17]*z[3]*z[4]*z[5]*z[6] - 48771072*z[17]*z[3]*z[4]*z[5] - 97542144*z[17]*z[3]*z[4]*z[6] + 127749888*z[17]*z[3]*z[4] - 195084288*z[17]*z[3]*z[5]*z[6] + 249306624*z[17]*z[3]*z[5] + 449068032*z[17]*z[3]*z[6] - 510064128*z[17]*z[3] - 390168576*z[17]*z[4]*z[5]*z[6] + 495516672*z[17]*z[4]*z[5] + 891942912*z[17]*z[4]*z[6] - 1007935488*z[17]*z[4] + 1734340608*z[17]*z[5]*z[6] - 1918328832*z[17]*z[5] - 3056320512*z[17]*z[6] - 762048*z[17]*z[7]*z[8]*z[9] + 8059968*z[17]*z[7]*z[8] + 16095744*z[17]*z[7]*z[9] - 127992312*z[17]*z[7] + 32179392*z[17]*z[8]*z[9] - 255794112*z[17]*z[8] - 510064128*z[17]*z[9] + 6501182688*z[17] + 3096576*z[18]*z[2]*z[3]*z[4]*z[5] + 6193152*z[18]*z[2]*z[3]*z[4]*z[6] - 12192768*z[18]*z[2]*z[3]*z[4] + 12386304*z[18]*z[2]*z[3]*z[5]*z[6] - 24385536*z[18]*z[2]*z[3]*z[5] - 48771072*z[18]*z[2]*z[3]*z[6] + 64358784*z[18]*z[2]*z[3] + 24772608*z[18]*z[2]*z[4]*z[5]*z[6] - 48771072*z[18]*z[2]*z[4]*z[5] - 97542144*z[18]*z[2]*z[4]*z[6] + 127943424*z[18]*z[2]*z[4] - 195084288*z[18]*z[2]*z[5]*z[6] + 249693696*z[18]*z[2]*z[5] + 449842176*z[18]*z[2]*z[6] - 511588224*z[18]*z[2] + 49545216*z[18]*z[3]*z[4]*z[5]*z[6] - 97542144*z[18]*z[3]*z[4]*z[5] - 195084288*z[18]*z[3]*z[4]*z[6] + 255499776*z[18]*z[3]*z[4] - 390168576*z[18]*z[3]*z[5]*z[6] + 498613248*z[18]*z[3]*z[5] + 898136064*z[18]*z[3]*z[6] - 1020128256*z[18]*z[3] - 780337152*z[18]*z[4]*z[5]*z[6] + 991033344*z[18]*z[4]*z[5] + 1783885824*z[18]*z[4]*z[6] - 2015870976*z[18]*z[4] + 3468681216*z[18]*z[5]*z[6] - 3836657664*z[18]*z[5] - 6112641024*z[18]*z[6] - 1524096*z[18]*z[7]*z[8]*z[9] + 16119936*z[18]*z[7]*z[8] + 32191488*z[18]*z[7]*z[9] - 255984624*z[18]*z[7] + 64358784*z[18]*z[8]*z[9] - 511588224*z[18]*z[8] - 1020128256*z[18]*z[9] + 13003139520*z[18] - 289158266880*z[2]*z[3]*z[4]*z[5]*z[6] - 2322432*z[2]*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] + 24563712*z[2]*z[3]*z[4]*z[5]*z[7]*z[8] + 49053696*z[2]*z[3]*z[4]*z[5]*z[7]*z[9] - 390071808*z[2]*z[3]*z[4]*z[5]*z[7] + 98070528*z[2]*z[3]*z[4]*z[5]*z[8]*z[9] - 779563008*z[2]*z[3]*z[4]*z[5]*z[8] - 1554481152*z[2]*z[3]*z[4]*z[5]*z[9] + 309900716544*z[2]*z[3]*z[4]*z[5] - 4644864*z[2]*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] + 49127424*z[2]*z[3]*z[4]*z[6]*z[7]*z[8] + 98107392*z[2]*z[3]*z[4]*z[6]*z[7]*z[9] - 780143616*z[2]*z[3]*z[4]*z[6]*z[7] + 196141056*z[2]*z[3]*z[4]*z[6]*z[8]*z[9] - 1559126016*z[2]*z[3]*z[4]*z[6]*z[8] - 3108962304*z[2]*z[3]*z[4]*z[6]*z[9] + 444163642368*z[2]*z[3]*z[4]*z[6] + 9144576*z[2]*z[3]*z[4]*z[7]*z[8]*z[9] - 96719616*z[2]*z[3]*z[4]*z[7]*z[8] - 193148928*z[2]*z[3]*z[4]*z[7]*z[9] + 1535907744*z[2]*z[3]*z[4]*z[7] - 386152704*z[2]*z[3]*z[4]*z[8]*z[9] + 3069529344*z[2]*z[3]*z[4]*z[8] + 6120769536*z[2]*z[3]*z[4]*z[9] - 474350591232*z[2]*z[3]*z[4] - 9289728*z[2]*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] + 98254848*z[2]*z[3]*z[5]*z[6]*z[7]*z[8] + 196214784*z[2]*z[3]*z[5]*z[6]*z[7]*z[9] - 1560287232*z[2]*z[3]*z[5]*z[6]*z[7] + 392282112*z[2]*z[3]*z[5]*z[6]*z[8]*z[9] - 3118252032*z[2]*z[3]*z[5]*z[6]*z[8] - 6217924608*z[2]*z[3]*z[5]*z[6]*z[9] + 797865977856*z[2]*z[3]*z[5]*z[6] + 18289152*z[2]*z[3]*z[5]*z[7]*z[8]*z[9] - 193439232*z[2]*z[3]*z[5]*z[7]*z[8] - 386297856*z[2]*z[3]*z[5]*z[7]*z[9] + 3071815488*z[2]*z[3]*z[5]*z[7] - 772305408*z[2]*z[3]*z[5]*z[8]*z[9] + 6139058688*z[2]*z[3]*z[5]*z[8] + 12241539072*z[2]*z[3]*z[5]*z[9] - 850400377344*z[2]*z[3]*z[5] + 36578304*z[2]*z[3]*z[6]*z[7]*z[8]*z[9] - 386878464*z[2]*z[3]*z[6]*z[7]*z[8] - 772595712*z[2]*z[3]*z[6]*z[7]*z[9] + 6143630976*z[2]*z[3]*z[6]*z[7] - 1544610816*z[2]*z[3]*z[6]*z[8]*z[9] + 12278117376*z[2]*z[3]*z[6]*z[8] + 24483078144*z[2]*z[3]*z[6]*z[9] - 1164102202368*z[2]*z[3]*z[6] - 48269088*z[2]*z[3]*z[7]*z[8]*z[9] + 510528608*z[2]*z[3]*z[7]*z[8] + 1019524864*z[2]*z[3]*z[7]*z[9] - 8107195572*z[2]*z[3]*z[7] + 2038283552*z[2]*z[3]*z[8]*z[9] - 16202323872*z[2]*z[3]*z[8] - 32308109568*z[2]*z[3]*z[9] + 1228274568456*z[2]*z[3] - 18579456*z[2]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 196509696*z[2]*z[4]*z[5]*z[6]*z[7]*z[8] + 392429568*z[2]*z[4]*z[5]*z[6]*z[7]*z[9] - 3120574464*z[2]*z[4]*z[5]*z[6]*z[7] + 784564224*z[2]*z[4]*z[5]*z[6]*z[8]*z[9] - 6236504064*z[2]*z[4]*z[5]*z[6]*z[8] - 12435849216*z[2]*z[4]*z[5]*z[6]*z[9] + 1550171000832*z[2]*z[4]*z[5]*z[6] + 36578304*z[2]*z[4]*z[5]*z[7]*z[8]*z[9] - 386878464*z[2]*z[4]*z[5]*z[7]*z[8] - 772595712*z[2]*z[4]*z[5]*z[7]*z[9] + 6143630976*z[2]*z[4]*z[5]*z[7] - 1544610816*z[2]*z[4]*z[5]*z[8]*z[9] + 12278117376*z[2]*z[4]*z[5]*z[8] + 24483078144*z[2]*z[4]*z[5]*z[9] - 1651000071168*z[2]*z[4]*z[5] + 73156608*z[2]*z[4]*z[6]*z[7]*z[8]*z[9] - 773756928*z[2]*z[4]*z[6]*z[7]*z[8] - 1545191424*z[2]*z[4]*z[6]*z[7]*z[9] + 12287261952*z[2]*z[4]*z[6]*z[7] - 3089221632*z[2]*z[4]*z[6]*z[8]*z[9] + 24556234752*z[2]*z[4]*z[6]*z[8] + 48966156288*z[2]*z[4]*z[6]*z[9] - 2249412028416*z[2]*z[4]*z[6] - 95957568*z[2]*z[4]*z[7]*z[8]*z[9] + 1014916288*z[2]*z[4]*z[7]*z[8] + 2026786304*z[2]*z[4]*z[7]*z[9] - 16116873192*z[2]*z[4]*z[7] + 4052049472*z[2]*z[4]*z[8]*z[9] - 32209756992*z[2]*z[4]*z[8] - 64227598848*z[2]*z[4]*z[9] + 2370912544656*z[2]*z[4] + 146313216*z[2]*z[5]*z[6]*z[7]*z[8]*z[9] - 1547513856*z[2]*z[5]*z[6]*z[7]*z[8] - 3090382848*z[2]*z[5]*z[6]*z[7]*z[9] + 24574523904*z[2]*z[5]*z[6]*z[7] - 6178443264*z[2]*z[5]*z[6]*z[8]*z[9] + 49112469504*z[2]*z[5]*z[6]*z[8] + 97932312576*z[2]*z[5]*z[6]*z[9] - 3930912018432*z[2]*z[5]*z[6] - 187270272*z[2]*z[5]*z[7]*z[8]*z[9] + 1980705152*z[2]*z[5]*z[7]*z[8] + 3955465216*z[2]*z[5]*z[7]*z[9] - 31453602768*z[2]*z[5]*z[7] + 7907957888*z[2]*z[5]*z[8]*z[9] - 62860387968*z[2]*z[5]*z[8] - 125346235392*z[2]*z[5]*z[9] + 4135065402144*z[2]*z[5] - 337381632*z[2]*z[6]*z[7]*z[8]*z[9] + 3568390912*z[2]*z[6]*z[7]*z[8] + 7126071296*z[2]*z[6]*z[7]*z[9] - 56666056608*z[2]*z[6]*z[7] + 14246787328*z[2]*z[6]*z[8]*z[9] - 113247767808*z[2]*z[6]*z[8] - 225820772352*z[2]*z[6]*z[9] + 5518421972544*z[2]*z[6] + 383691168*z[2]*z[7]*z[8]*z[9] - 4058193888*z[2]*z[7]*z[8] - 8104207104*z[2]*z[7]*z[9] + 64444129092*z[2]*z[7] - 16202323872*z[2]*z[8]*z[9] + 128792335392*z[2]*z[8] + 256817288448*z[2]*z[9] - 5745595042656*z[2] - 37158912*z[3]*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] + 393019392*z[3]*z[4]*z[5]*z[6]*z[7]*z[8] + 784859136*z[3]*z[4]*z[5]*z[6]*z[7]*z[9] - 6241148928*z[3]*z[4]*z[5]*z[6]*z[7] + 1569128448*z[3]*z[4]*z[5]*z[6]*z[8]*z[9] - 12473008128*z[3]*z[4]*z[5]*z[6]*z[8] - 24871698432*z[3]*z[4]*z[5]*z[6]*z[9] + 3077520236544*z[3]*z[4]*z[5]*z[6] + 73156608*z[3]*z[4]*z[5]*z[7]*z[8]*z[9] - 773756928*z[3]*z[4]*z[5]*z[7]*z[8] - 1545191424*z[3]*z[4]*z[5]*z[7]*z[9] + 12287261952*z[3]*z[4]*z[5]*z[7] - 3089221632*z[3]*z[4]*z[5]*z[8]*z[9] + 24556234752*z[3]*z[4]*z[5]*z[8] + 48966156288*z[3]*z[4]*z[5]*z[9] - 3277018515456*z[3]*z[4]*z[5] + 146313216*z[3]*z[4]*z[6]*z[7]*z[8]*z[9] - 1547513856*z[3]*z[4]*z[6]*z[7]*z[8] - 3090382848*z[3]*z[4]*z[6]*z[7]*z[9] + 24574523904*z[3]*z[4]*z[6]*z[7] - 6178443264*z[3]*z[4]*z[6]*z[8]*z[9] + 49112469504*z[3]*z[4]*z[6]*z[8] + 97932312576*z[3]*z[4]*z[6]*z[9] - 4459265298432*z[3]*z[4]*z[6] - 191624832*z[3]*z[4]*z[7]*z[8]*z[9] + 2026762112*z[3]*z[4]*z[7]*z[8] + 4047440896*z[3]*z[4]*z[7]*z[9] - 32184987408*z[3]*z[4]*z[7] + 8091840128*z[3]*z[4]*z[8]*z[9] - 64322068608*z[3]*z[4]*z[8] - 128260887552*z[3]*z[4]*z[9] + 4698794629344*z[3]*z[4] + 292626432*z[3]*z[5]*z[6]*z[7]*z[8]*z[9] - 3095027712*z[3]*z[5]*z[6]*z[7]*z[8] - 6180765696*z[3]*z[5]*z[6]*z[7]*z[9] + 49149047808*z[3]*z[5]*z[6]*z[7] - 12356886528*z[3]*z[5]*z[6]*z[8]*z[9] + 98224939008*z[3]*z[5]*z[6]*z[8] + 195864625152*z[3]*z[5]*z[6]*z[9] - 7787908767744*z[3]*z[5]*z[6] - 373959936*z[3]*z[5]*z[7]*z[8]*z[9] + 3955269376*z[3]*z[5]*z[7]*z[8] + 7898667008*z[3]*z[5]*z[7]*z[9] - 62809687584*z[3]*z[5]*z[7] + 15791398144*z[3]*z[5]*z[8]*z[9] - 125525885184*z[3]*z[5]*z[8] - 250303850496*z[3]*z[5]*z[9] + 8190601079232*z[3]*z[5] - 673602048*z[3]*z[6]*z[7]*z[8]*z[9] + 7124499968*z[3]*z[6]*z[7]*z[8] + 14227615744*z[3]*z[6]*z[7]*z[9] - 113137077312*z[3]*z[6]*z[7] + 28444539392*z[3]*z[6]*z[8]*z[9] - 226105754112*z[3]*z[6]*z[8] - 450864304128*z[3]*z[6]*z[9] + 10922106819456*z[3]*z[6] + 765096192*z[3]*z[7]*z[8]*z[9] - 8092207872*z[3]*z[7]*z[8] - 16160126976*z[3]*z[7]*z[9] + 128504281248*z[3]*z[7] - 32308109568*z[3]*z[8]*z[9] + 256817288448*z[3]*z[8] + 512104384512*z[3]*z[9] - 11368442502144*z[3] + 585252864*z[4]*z[5]*z[6]*z[7]*z[8]*z[9] - 6190055424*z[4]*z[5]*z[6]*z[7]*z[8] - 12361531392*z[4]*z[5]*z[6]*z[7]*z[9] + 98298095616*z[4]*z[5]*z[6]*z[7] - 24713773056*z[4]*z[5]*z[6]*z[8]*z[9] + 196449878016*z[4]*z[5]*z[6]*z[8] + 391729250304*z[4]*z[5]*z[6]*z[9] - 15000102125568*z[4]*z[5]*z[6] - 743275008*z[4]*z[5]*z[7]*z[8]*z[9] + 7861411328*z[4]*z[5]*z[7]*z[8] + 15699226624*z[4]*z[5]*z[7]*z[9] - 124839231552*z[4]*z[5]*z[7] + 31386655232*z[4]*z[5]*z[8]*z[9] - 249492644352*z[4]*z[5]*z[8] - 497498738688*z[4]*z[5]*z[9] + 15764686966656*z[4]*z[5] - 1337914368*z[4]*z[6]*z[7]*z[8]*z[9] + 14150745088*z[4]*z[6]*z[7]*z[8] + 28259016704*z[4]*z[6]*z[7]*z[9] - 224713867392*z[4]*z[6]*z[7] + 56496796672*z[4]*z[6]*z[8]*z[9] - 449093256192*z[4]*z[6]*z[8] - 895510683648*z[4]*z[6]*z[9] + 20961798233856*z[4]*z[6] + 1511903232*z[4]*z[7]*z[8]*z[9] - 15990976512*z[4]*z[7]*z[8] - 31933956096*z[4]*z[7]*z[9] + 253936747008*z[4]*z[7] - 63843913728*z[4]*z[8]*z[9] + 507495518208*z[4]*z[8] + 1011967229952*z[4]*z[9] - 21794835654144*z[4] - 2601510912*z[5]*z[6]*z[7]*z[8]*z[9] + 27515451392*z[5]*z[6]*z[7]*z[8] + 54948315136*z[5]*z[6]*z[7]*z[9] - 436945436928*z[5]*z[6]*z[7] + 109855336448*z[5]*z[6]*z[8]*z[9] - 873240496128*z[5]*z[6]*z[8] - 1741277970432*z[5]*z[6]*z[9] + 35949148306944*z[5]*z[6] + 2877493248*z[5]*z[7]*z[8]*z[9] - 30434439168*z[5]*z[7]*z[8] - 60777529344*z[5]*z[7]*z[9] + 483298970112*z[5]*z[7] - 121509384192*z[5]*z[8]*z[9] + 965878566912*z[5]*z[8] + 1926002147328*z[5]*z[9] - 37231585606656*z[5] + 4584480768*z[6]*z[7]*z[8]*z[9] - 48488767488*z[6]*z[7]*z[8] - 96831995904*z[6]*z[7]*z[9] + 770001748992*z[6]*z[7] - 193591222272*z[6]*z[8]*z[9] + 1538857377792*z[6]*z[8] + 3068545794048*z[6]*z[9] - 48569274464256*z[6] - 61479790344*z[7]*z[8]*z[9] + 310518320544*z[7]*z[8] + 615518495232*z[7]*z[9] - 5761097508537*z[7]/2 + 1228274568456*z[8]*z[9] - 5745595042656*z[8] - 11368442502144*z[9] + 79037120607943)

In [440]:
top_solutions_less_63_D_Equation_2 = find_best_bitstrings(final_circuit_less_63_D_Equation_2, hamiltonian_less_63_D_Equation_2)

Top 5 bitstrings:
Bitstring: 110010000000000111, Cost: -158074241206282.0000, Count: 1
Bitstring: 101101000110000101, Cost: -158074241205276.0000, Count: 1
Bitstring: 111111000110000111, Cost: -158074241142444.0000, Count: 1
Bitstring: 011100000100000011, Cost: -158074241016970.0000, Count: 1
Bitstring: 110001000111000101, Cost: -158074240824010.0000, Count: 1


In [441]:
bitstrings_less_63_D_Equation_2 = [
    "110010000000000111",
    "101101000110000101",
    "111111000110000111",
    "011100000100000011",
    "110001000111000101"
]

In [477]:
def group_and_convert(bitstring, group_size=6):
    Decimals = []
    for i in range(len(bitstring)):
        reversed_bits = bitstring[i][::-1]
        #print(reversed_bits)
        chunks = [reversed_bits[i:i+group_size][::1] for i in range(0, len(reversed_bits), group_size)]
        #print(chunks)
        decimals = [int(chunk, 2) for chunk in chunks]
        Decimals.append(decimals)
    return(Decimals)

In [479]:
def evaluate_Diophantine(entrada,poly):
    Cost = []
    for i in range(len(entrada)):
        Cost.append(poly(entrada[i][0],entrada[i][1],entrada[i][2]))
    return(Cost)
    

In [487]:
def poly_less_63_D_Equation_2(x,y,z):
    return(expand(D_Equation_2(x,y,z)**2))

In [488]:
evaluate_Diophantine(group_and_convert(bitstrings_less_63_D_Equation_2),poly_less_63_D_Equation_2)

[96710230866496,
 8350666621504,
 103272422588416,
 28220711908489,
 153593190865984]

In [ ]:
# Apply the function to each bitstring and print results
for b in bitstrings:
    print(f"Bitstring: {b}")
    decimal_values = group_and_convert(b)
    print(f"Grouped decimals: {decimal_values}\n")

In [ ]:
bitstring_to_pm1(top_solutions_less_63_D_Equation_2, evaluate_hamiltonian_D_Equation_2_less_63)

In [419]:
paso1_D_Equation_2_less_250 = substitute_with_global_binary_symbols(D_Equation_2(x[1],x[2],x[3])**2, 8, base_name="b")
paso2_D_Equation_2_less_250 = remove_variable_exponents(paso1_D_Equation_2_less_250)
paso3_D_Equation_2_less_250 = substitute_with_spin_variables(paso2_D_Equation_2_less_250)

In [420]:
print(paso3_D_Equation_2_less_250)

294912*z_1*z_10*z_11*z_12*z_13*z_2*z_3*z_4 + 589824*z_1*z_10*z_11*z_12*z_13*z_2*z_3*z_5 + 1179648*z_1*z_10*z_11*z_12*z_13*z_2*z_3*z_6 + 2359296*z_1*z_10*z_11*z_12*z_13*z_2*z_3*z_7 + 4718592*z_1*z_10*z_11*z_12*z_13*z_2*z_3*z_8 - 9400320*z_1*z_10*z_11*z_12*z_13*z_2*z_3 + 1179648*z_1*z_10*z_11*z_12*z_13*z_2*z_4*z_5 + 2359296*z_1*z_10*z_11*z_12*z_13*z_2*z_4*z_6 + 4718592*z_1*z_10*z_11*z_12*z_13*z_2*z_4*z_7 + 9437184*z_1*z_10*z_11*z_12*z_13*z_2*z_4*z_8 - 18800640*z_1*z_10*z_11*z_12*z_13*z_2*z_4 + 4718592*z_1*z_10*z_11*z_12*z_13*z_2*z_5*z_6 + 9437184*z_1*z_10*z_11*z_12*z_13*z_2*z_5*z_7 + 18874368*z_1*z_10*z_11*z_12*z_13*z_2*z_5*z_8 - 37601280*z_1*z_10*z_11*z_12*z_13*z_2*z_5 + 18874368*z_1*z_10*z_11*z_12*z_13*z_2*z_6*z_7 + 37748736*z_1*z_10*z_11*z_12*z_13*z_2*z_6*z_8 - 75202560*z_1*z_10*z_11*z_12*z_13*z_2*z_6 + 75497472*z_1*z_10*z_11*z_12*z_13*z_2*z_7*z_8 - 150405120*z_1*z_10*z_11*z_12*z_13*z_2*z_7 - 300810240*z_1*z_10*z_11*z_12*z_13*z_2*z_8 + 400281600*z_1*z_10*z_11*z_12*z_13*z_2 + 2359296*z